<a href="https://colab.research.google.com/github/SiemensTestAutothon2019/Root/blob/master/build_nifty500_pit_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Build Your Own NIFTY 500 Point-in-Time (PIT) Dataset

This notebook reproduces the full pipeline described in the repo README, end to end, from public data sources only. Run the cells in order.

**What you'll end up with:** a single parquet file — one row per (company, trading day), with a flag telling you whether that company was *actually* a NIFTY 500 constituent on that exact date.

**Runtime:** the OHLCV download stage (Section 5) is the slow part — expect 30-60+ minutes for ~1,000 companies. Everything before that is fast (a few minutes).

**Requirements:** Python 3.9+, internet access. No API keys needed — everything here uses free, public endpoints.

## 0. Setup

In [1]:
!pip install -q requests pandas pyarrow pypdf yfinance jugaad-data

In [2]:
import os, re, csv, glob, time, html, warnings
from datetime import datetime, timedelta
import requests
import pandas as pd

warnings.filterwarnings("ignore")

DATA_DIR = "nifty500_pit_data"
os.makedirs(f"{DATA_DIR}/press_releases", exist_ok=True)
os.makedirs(f"{DATA_DIR}/press_releases_text", exist_ok=True)
os.makedirs(f"{DATA_DIR}/constituent_snapshots", exist_ok=True)
os.makedirs(f"{DATA_DIR}/ohlcv_yfinance", exist_ok=True)
os.makedirs(f"{DATA_DIR}/ohlcv_bhavcopy", exist_ok=True)

HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
print("Ready.")

Ready.


## 1. Scrape NIFTY 500 index-change press releases

NSE Indices publishes a press release for every constituent change. The full archive (back to 1998) is embedded client-side in one HTML page — a single request gets the whole list, no need to page through hundreds of months.

In [3]:
MEDIA_URL = "https://www.niftyindices.com/media"

resp = requests.get(MEDIA_URL, headers=HEADERS, timeout=30)
html_text = resp.text
print("fetched", len(html_text), "bytes")

start = html_text.find('id="articleList"')
end = html_text.find("</ul>", start)
block = html_text[start:end]

HEADER_ITEM_RE = re.compile(
    r'<li data-publishdate="([^"]+)">\s*<span>([^<]+)</span>\s*<p>\s*<a href="([^"]+)"[^>]*>\s*([^<]+?)\s*</a>',
)
items = HEADER_ITEM_RE.findall(block)
print("total press releases found in archive:", len(items))

fetched 920963 bytes
total press releases found in archive: 1520


In [4]:
def parse_release_date(s):
    try:
        return datetime.strptime(s.strip(), "%B %d, %Y")
    except ValueError:
        return None

def is_broad_500(title):
    return any(p in title for p in ["Nifty 500 Index", "CNX 500 Index", "S&amp;P CNX 500", "S&P CNX 500", "Nifty500 Index"])

def is_generic_multi_index(title):
    t = title.lower()
    return t.startswith("replacements in indices") or t.startswith("change in indices")

CUTOFF = datetime(2010, 1, 1)  # change this if you want a different start date
relevant = []
for month, date_str, url, title in items:
    d = parse_release_date(date_str)
    if d is None or d < CUTOFF:
        continue
    if is_broad_500(title) or is_generic_multi_index(title):
        relevant.append({"date": d.strftime("%Y-%m-%d"), "pdf_url": url.strip(), "title": html.unescape(title.strip())})

relevant.sort(key=lambda r: r["date"])
print("relevant press releases to download:", len(relevant))

relevant press releases to download: 140


In [5]:
# download every relevant press release PDF
for r in relevant:
    fname = f"{DATA_DIR}/press_releases/{r['date']}_{r['pdf_url'].split('/')[-1]}"
    if os.path.exists(fname):
        continue
    full_url = "https://www.niftyindices.com" + r["pdf_url"]
    try:
        resp = requests.get(full_url, headers=HEADERS, timeout=30)
        if resp.status_code == 200 and len(resp.content) > 500:
            with open(fname, "wb") as f:
                f.write(resp.content)
    except Exception as e:
        print("failed:", r["pdf_url"], e)
    time.sleep(0.3)

pdf_files = glob.glob(f"{DATA_DIR}/press_releases/*.pdf")
print("PDFs downloaded:", len(pdf_files))

PDFs downloaded: 138


In [6]:
# extract text from every PDF
from pypdf import PdfReader

for fp in pdf_files:
    base = os.path.splitext(os.path.basename(fp))[0]
    txt_path = f"{DATA_DIR}/press_releases_text/{base}.txt"
    if os.path.exists(txt_path):
        continue
    try:
        reader = PdfReader(fp)
        text = "\n".join(page.extract_text() or "" for page in reader.pages)
        with open(txt_path, "w", encoding="utf-8") as f:
            f.write(text)
    except Exception as e:
        print("failed to parse:", fp, e)

print("text files:", len(glob.glob(f"{DATA_DIR}/press_releases_text/*.txt")))

failed to parse: nifty500_pit_data/press_releases/2012-03-02_ind_prs02032012.pdf Stream has ended unexpectedly
text files: 137


## 2. Parse press releases into add/exclude events

Each press release lists companies being excluded/included under one or more index headers (e.g. `"S&P CNX 500 Index:"`, `"4) Nifty 500 Index"`). We only keep the sections that refer to the *broad* NIFTY 500 index, not thematic sub-indices like "Nifty500 Value 50".

In [7]:
HEADER_RE = re.compile(
    r'(?:^|\n)\s*(?:\(?([a-zA-Z0-9]+)\)?[\.\)]\s*)?((?:NIFTY|Nifty|CNX|S&P\s*CNX)[A-Za-z0-9 &\.\-]{0,40}?)\s*(?:Index)?\s*:?\s*(?=\n)',
    re.MULTILINE
)
ROW_RE = re.compile(r'^\s*(\d+)\s+(.+?)\s+([A-Z0-9&\-\.]{2,20})\s*$')
ROW_RE_NOSYM = re.compile(r'^\s*(\d+)\s+(.+?)\s*$')

def normalize_index_name(raw):
    s = re.sub(r'\s+', ' ', raw).strip().rstrip('.').strip()
    s = re.sub(r'\s+index$', '', s, flags=re.IGNORECASE).strip()
    return s

def is_broad_500_name(name):
    return bool(re.fullmatch(r'(NIFTY|Nifty|S&P\s*CNX|CNX)\s*500', re.sub(r'\s+', ' ', name).strip()))

def extract_rows(block, action, results):
    label_re = re.compile(r'being\s+' + action + r'[a-z]*\s*:?', re.IGNORECASE)
    m = label_re.search(block)
    if not m:
        return
    started = False
    for line in block[m.end():].split('\n'):
        line = line.strip()
        if not line:
            if started: break
            continue
        if re.match(r'^Sr\.?\s*No\.?', line, re.IGNORECASE):
            started = True; continue
        if re.search(r'being\s+(excluded|included)', line, re.IGNORECASE):
            break
        rm = ROW_RE.match(line)
        if rm:
            started = True
            _, company, symbol = rm.groups()
            results.append({"action": action, "company_name": company.strip(), "symbol": symbol.strip()})
            continue
        rm2 = ROW_RE_NOSYM.match(line)
        if rm2 and not re.search(r'being\s+(excluded|included)', line, re.IGNORECASE):
            started = True
            _, company = rm2.groups()
            results.append({"action": action, "company_name": company.strip(), "symbol": ""})
            continue
        if started: break

def parse_press_release(text, is_dedicated500):
    headers = list(HEADER_RE.finditer(text))
    events = []
    for i, h in enumerate(headers):
        if not is_broad_500_name(normalize_index_name(h.group(2))):
            continue
        start = h.end()
        end = headers[i+1].start() if i+1 < len(headers) else len(text)
        block = text[start:end]
        extract_rows(block, "exclude", events)
        extract_rows(block, "include", events)
    if is_dedicated500 and not events:
        extract_rows(text, "exclude", events)
        extract_rows(text, "include", events)
    return events

print("parser defined.")

parser defined.


In [8]:
all_events = []
for r in relevant:
    base = f"{r['date']}_{r['pdf_url'].split('/')[-1]}"
    base = os.path.splitext(base)[0]
    txt_path = f"{DATA_DIR}/press_releases_text/{base}.txt"
    if not os.path.exists(txt_path):
        continue
    text = open(txt_path, encoding="utf-8").read()
    is_dedicated = is_broad_500(r["title"])
    events = parse_press_release(text, is_dedicated)
    for e in events:
        e["press_release_date"] = r["date"]
    all_events.extend(events)

events_df = pd.DataFrame(all_events)
events_df.to_csv(f"{DATA_DIR}/events_raw.csv", index=False)
print("total events parsed:", len(events_df))
events_df.head()

total events parsed: 971


,action,company_name,symbol,press_release_date
0,include,Asian Hotels Ltd.,,2010-02-19
1,include,Peninsula Land Ltd.,,2010-02-19
2,include,HSBC Invest Direct (India) Limited,HINVDIR,2010-07-07
3,include,Tinplate Company of India Ltd.,TINPLATE,2010-07-07
4,exclude,Mid-Day Multimedia Ltd.,MID-DAY,2011-01-18


## 3. Anchor on real historical constituent snapshots (Wayback Machine)

Press releases alone only capture ad-hoc, corporate-action-driven changes — routine market-cap-ranking reconstitution doesn't seem to generate the same kind of itemized announcement. So instead of trusting a single starting list plus event diffs, we pull **multiple real historical snapshots** of NSE's actual published constituent CSV via the Wayback Machine, across both domains that have hosted it over the years.

In [9]:
def wayback_cdx_search(url_pattern, from_year, to_year):
    cdx_url = "https://web.archive.org/cdx/search/cdx"
    params = {
        "url": url_pattern, "output": "json",
        "from": str(from_year), "to": str(to_year), "limit": 300,
        "filter": "statuscode:200",
    }
    r = requests.get(cdx_url, params=params, timeout=30)
    rows = r.json()
    if len(rows) <= 1:
        return []
    header, *data = rows
    return [dict(zip(header, row)) for row in data]

# search across the domains/paths known to have hosted the constituent list over the years
candidates = []
for pattern in [
    "nseindia.com/content/indices/ind_cnx500list.csv",
    "nseindia.com/content/indices/ind_nifty500list.csv",
    "niftyindices.com/IndexConstituent/ind_nifty500list.csv",
]:
    candidates.extend(wayback_cdx_search(pattern, 2010, 2030))

print("total candidate snapshots found:", len(candidates))
for c in candidates[:5]:
    print(c["timestamp"], c["original"])

total candidate snapshots found: 17
20100102131654 http://nseindia.com/content/indices/ind_cnx500list.csv
20140122091641 http://nseindia.com/content/indices/ind_cnx500list.csv
20140709141815 http://nseindia.com/content/indices/ind_cnx500list.csv
20150325112154 http://nseindia.com/content/indices/ind_cnx500list.csv
20200725142023 https://www1.nseindia.com/content/indices/ind_nifty500list.csv


In [10]:
def download_wayback_snapshot(timestamp, original_url, out_path):
    wb_url = f"https://web.archive.org/web/{timestamp}id_/{original_url}"
    r = requests.get(wb_url, headers=HEADERS, timeout=30)
    r.raise_for_status()
    content = r.content
    # some captures are gzip-compressed even after requests' auto-decompress; guard against that
    if content[:2] == b'\x1f\x8b':
        import gzip
        content = gzip.decompress(content)
    with open(out_path, "wb") as f:
        f.write(content)

downloaded = 0
for c in candidates:
    date_str = datetime.strptime(c["timestamp"], "%Y%m%d%H%M%S").strftime("%Y-%m-%d")
    out_path = f"{DATA_DIR}/constituent_snapshots/nifty500_{date_str}.csv"
    if os.path.exists(out_path):
        continue
    try:
        download_wayback_snapshot(c["timestamp"], c["original"], out_path)
        # sanity check: must be a real constituent list (has ISIN column, ~500 rows)
        text = open(out_path, encoding="utf-8", errors="ignore").read()
        if "ISIN" not in text or text.count("\n") < 100:
            os.remove(out_path)
            continue
        downloaded += 1
    except Exception:
        continue
    time.sleep(0.5)

snapshot_files = sorted(glob.glob(f"{DATA_DIR}/constituent_snapshots/*.csv"))
print(f"usable snapshots downloaded: {len(snapshot_files)}")
for f in snapshot_files:
    print(" ", f)

usable snapshots downloaded: 17
  nifty500_pit_data/constituent_snapshots/nifty500_2010-01-02.csv
  nifty500_pit_data/constituent_snapshots/nifty500_2014-01-22.csv
  nifty500_pit_data/constituent_snapshots/nifty500_2014-07-09.csv
  nifty500_pit_data/constituent_snapshots/nifty500_2015-03-25.csv
  nifty500_pit_data/constituent_snapshots/nifty500_2018-10-04.csv
  nifty500_pit_data/constituent_snapshots/nifty500_2019-02-01.csv
  nifty500_pit_data/constituent_snapshots/nifty500_2020-07-25.csv
  nifty500_pit_data/constituent_snapshots/nifty500_2024-02-07.csv
  nifty500_pit_data/constituent_snapshots/nifty500_2025-06-16.csv
  nifty500_pit_data/constituent_snapshots/nifty500_2025-08-15.csv
  nifty500_pit_data/constituent_snapshots/nifty500_2025-08-21.csv
  nifty500_pit_data/constituent_snapshots/nifty500_2026-01-07.csv
  nifty500_pit_data/constituent_snapshots/nifty500_2026-05-02.csv
  nifty500_pit_data/constituent_snapshots/nifty500_2026-08-27.csv
  nifty500_pit_data/constituent_snapshots/ni

> **Note:** the number of usable snapshots you get depends on what the Wayback Machine happened to crawl — it will vary run to run and may be less complete than the 17 snapshots used in the reference build. The more snapshots you have, the tighter your membership-date confidence will be. Feel free to widen the `pattern` list above if you find additional URL variants NSE has used over the years.

## 4. Build the membership interval table

For each pair of consecutive snapshots, diff the ISIN sets to find who entered/exited. Two things to handle carefully:

- **ISIN migrations**: a company can get a new ISIN (e.g. a face-value-split-driven reissue) while staying continuously listed. Naively diffing ISIN sets makes this look like a fake exit+entry. We detect same-symbol/same-company-name pairs across a boundary and treat them as one continuous membership instead.
- **Confidence labels**: every interval's start/end date is labeled by how it was determined — exact (from a press release), tightly-bounded (snapshots close together), or loosely-bounded (snapshots far apart) — so nothing is presented as more precise than it is.

In [11]:
def load_constituent_csv(path):
    lines = open(path, encoding="utf-8-sig", errors="ignore").readlines()
    start = 0
    for i, line in enumerate(lines):
        if line.strip().lower().startswith("company name"):
            start = i
            break
    rows = list(csv.DictReader(lines[start:]))
    out = {}
    for r in rows:
        isin = r.get("ISIN Code", "").strip()
        if isin:
            out[isin] = {"isin": isin, "symbol": r.get("Symbol", "").strip(), "company_name": r.get("Company Name", "").strip()}
    return out

def norm_symbol(s):
    return re.sub(r'\s+', '', s.strip().upper())

def norm_name(s):
    s = s.strip().lower()
    s = re.sub(r'\.', '', s)
    s = re.sub(r'\blimited\b', 'ltd', s)
    s = re.sub(r'\bltd\b\.?', 'ltd', s)
    s = re.sub(r'&', 'and', s)
    return re.sub(r'\s+', ' ', s).strip()

snapshots = []
for f in snapshot_files:
    date_str = os.path.basename(f).replace("nifty500_", "").replace(".csv", "")
    snapshots.append((datetime.strptime(date_str, "%Y-%m-%d"), load_constituent_csv(f)))
snapshots.sort(key=lambda x: x[0])
print("snapshots loaded:", len(snapshots))

snapshots loaded: 17


In [12]:
# index scraped events by normalized symbol for date lookups
events_by_symbol = {}
for _, row in events_df.iterrows():
    sym = norm_symbol(row["symbol"])
    if not sym:
        continue
    d = datetime.strptime(row["press_release_date"], "%Y-%m-%d")
    events_by_symbol.setdefault(sym, []).append((d, row["action"]))

def find_event_date(symbol, action, window_start, window_end):
    matches = [d for d, a in events_by_symbol.get(norm_symbol(symbol), []) if a == action and window_start <= d <= window_end]
    return min(matches) if matches else None

open_intervals = {}
closed_intervals = []
first_date, first_snap = snapshots[0]
for isin, info in first_snap.items():
    open_intervals[isin] = {**info, "start_date": first_date, "start_confidence": "left_censored"}

for i in range(1, len(snapshots)):
    prev_date, prev_snap = snapshots[i-1]
    curr_date, curr_snap = snapshots[i]
    prev_isins, curr_isins = set(prev_snap), set(curr_snap)
    exited, entered = prev_isins - curr_isins, curr_isins - prev_isins
    gap_conf = "snapshot_adjacent" if (curr_date - prev_date).days <= 220 else "long_gap"

    exited_by_key = {}
    for isin in exited:
        info = open_intervals[isin]
        exited_by_key.setdefault((norm_symbol(info["symbol"]), norm_name(info["company_name"])), []).append(isin)

    migrations = []
    for isin in list(entered):
        info = curr_snap[isin]
        key = (norm_symbol(info["symbol"]), norm_name(info["company_name"]))
        cands = exited_by_key.get(key, [])
        if cands:
            old_isin = cands.pop(0)
            migrations.append((old_isin, isin))
            exited.discard(old_isin); entered.discard(isin)

    for old_isin, new_isin in migrations:
        old_info = open_intervals.pop(old_isin)
        closed_intervals.append({**old_info, "end_date": curr_date, "end_confidence": f"isin_migration_to_{new_isin}"})
        open_intervals[new_isin] = {**curr_snap[new_isin], "start_date": old_info["start_date"], "start_confidence": f"isin_migration_from_{old_isin}"}

    for isin in exited:
        info = open_intervals.pop(isin)
        ev = find_event_date(info["symbol"], "exclude", prev_date, curr_date)
        end_date = ev if ev else prev_date + (curr_date - prev_date) / 2
        confidence = "press_release_exact" if ev else f"interpolated_{gap_conf}"
        closed_intervals.append({**info, "end_date": end_date, "end_confidence": confidence})

    for isin in entered:
        info = curr_snap[isin]
        ev = find_event_date(info["symbol"], "include", prev_date, curr_date)
        start_date = ev if ev else prev_date + (curr_date - prev_date) / 2
        confidence = "press_release_exact" if ev else f"interpolated_{gap_conf}"
        open_intervals[isin] = {**info, "start_date": start_date, "start_confidence": confidence}

    for isin in (prev_isins & curr_isins):
        open_intervals[isin]["symbol"] = curr_snap[isin]["symbol"]
        open_intervals[isin]["company_name"] = curr_snap[isin]["company_name"]

for isin, info in open_intervals.items():
    closed_intervals.append({**info, "end_date": None, "end_confidence": "still_active"})

intervals_df = pd.DataFrame(closed_intervals)
intervals_df["start_date"] = pd.to_datetime(intervals_df["start_date"]).dt.strftime("%Y-%m-%d")
intervals_df["end_date"] = pd.to_datetime(intervals_df["end_date"]).dt.strftime("%Y-%m-%d")
intervals_df = intervals_df[["isin","symbol","company_name","start_date","start_confidence","end_date","end_confidence"]]
intervals_df.to_csv(f"{DATA_DIR}/membership_intervals.csv", index=False)

print("total intervals:", len(intervals_df))
print("unique companies ever in NIFTY 500:", intervals_df["isin"].nunique())
print("currently active:", (intervals_df["end_confidence"]=="still_active").sum())

total intervals: 1292
unique companies ever in NIFTY 500: 1207
currently active: 501


## 5. Resolve each company to its current trading symbol

Yahoo Finance keys on the *current* ticker only — historical symbol changes don't matter for the download, only where the company trades today. We build a symbol resolution table by grouping ISIN-migration chains and checking each against NSE's current listed-equity master.

In [14]:

# ============================================================
# SECTION 5 FIX: Resolve current symbols using NSE equity master
# Avoids the failing Wayback CDX request.
# ============================================================

import io
import os
import time
import requests
import pandas as pd

equity_master_urls = [
    "https://archives.nseindia.com/content/equities/EQUITY_L.csv",
    "https://nsearchives.nseindia.com/content/equities/EQUITY_L.csv",
]

equity_headers = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/140.0.0.0 Safari/537.36"
    ),
    "Accept": "text/csv,text/plain,*/*",
    "Referer": "https://www.nseindia.com/",
}

session = requests.Session()
equity_master_df = None
fetch_errors = []

# Try to establish an NSE session first.
try:
    session.get(
        "https://www.nseindia.com/",
        headers=equity_headers,
        timeout=15,
    )
except requests.RequestException as exc:
    print("NSE homepage session warning:", str(exc)[:250])

for equity_url in equity_master_urls:
    try:
        print("\nTrying:", equity_url)

        response = session.get(
            equity_url,
            headers=equity_headers,
            timeout=40,
        )
        response.raise_for_status()

        # Reject HTML error pages masquerading as successful responses.
        if response.content.lstrip().lower().startswith(
            (b"<!doctype html", b"<html")
        ):
            raise ValueError("Received HTML instead of a CSV.")

        candidate_df = pd.read_csv(
            io.BytesIO(response.content),
            encoding="utf-8-sig",
        )
        candidate_df.columns = [
            str(c).strip() for c in candidate_df.columns
        ]

        normalized = {
            "".join(ch for ch in c.upper() if ch.isalnum()): c
            for c in candidate_df.columns
        }

        isin_col = normalized.get("ISINNUMBER")
        symbol_col = normalized.get("SYMBOL")

        if not isin_col or not symbol_col:
            raise ValueError(
                "Expected ISIN NUMBER and SYMBOL columns; "
                f"got {candidate_df.columns.tolist()}"
            )

        candidate_df[isin_col] = (
            candidate_df[isin_col].astype(str).str.strip()
        )
        candidate_df[symbol_col] = (
            candidate_df[symbol_col].astype(str).str.strip()
        )

        candidate_df = candidate_df[
            candidate_df[isin_col].str.fullmatch(
                r"[A-Z]{2}[A-Z0-9]{10}", na=False
            )
            & candidate_df[symbol_col].ne("")
            & candidate_df[symbol_col].str.lower().ne("nan")
        ]

        if candidate_df.empty:
            raise ValueError("No valid ISIN-symbol rows found.")

        equity_master_df = candidate_df
        print("Downloaded and validated NSE equity master.")
        break

    except Exception as exc:
        fetch_errors.append(f"{equity_url}: {exc}")
        print("Download failed:", str(exc)[:300])
        time.sleep(1)

if equity_master_df is None:
    raise RuntimeError(
        "Could not download a valid NSE equity master from either host.\n"
        + "\n".join(fetch_errors)
        + "\n\nStop here and share this error. Do not run the "
          "remaining build cells using an empty symbol mapping."
    )

# Use the same variable expected by the following notebook cells.
isin_to_current_symbol = dict(
    zip(equity_master_df[isin_col], equity_master_df[symbol_col])
)

# Save a local copy for recovery.
os.makedirs(DATA_DIR, exist_ok=True)
master_path = os.path.join(DATA_DIR, "nse_equity_master.csv")
equity_master_df.to_csv(master_path, index=False)

print("\nEQUITY MASTER READY")
print("Valid rows:", len(equity_master_df))
print("Unique ISINs:", len(isin_to_current_symbol))
print("Saved:", master_path)
print("Example mappings:")
print(list(isin_to_current_symbol.items())[:10])



Trying: https://archives.nseindia.com/content/equities/EQUITY_L.csv
Downloaded and validated NSE equity master.

EQUITY MASTER READY
Valid rows: 2613
Unique ISINs: 2613
Saved: nifty500_pit_data/nse_equity_master.csv
Example mappings:
[('INE144J01027', '20MICRONS'), ('INE253B01015', '21STCENMGM'), ('INE466L01038', '360ONE'), ('INE994E01018', '3BBLACKBIO'), ('INE748C01038', '3IINFOLTD'), ('INE470A01017', '3MINDIA'), ('INE105C01023', '3PLAND'), ('INE618L01018', '5PAISA'), ('INE111B01023', '63MOONS'), ('INE619I01012', 'A2ZINFRA')]


In [15]:
# chain ISIN migrations together (same company, ISIN changed over time)
migration_target = {}
for _, row in intervals_df.iterrows():
    m = re.match(r"isin_migration_to_(\S+)", str(row["end_confidence"]))
    if m:
        migration_target[row["isin"]] = m.group(1)

def chain_head(isin):
    seen = set()
    while isin in migration_target and isin not in seen:
        seen.add(isin); isin = migration_target[isin]
    return isin

by_isin = intervals_df.groupby("isin")
chains = {}
for isin in by_isin.groups:
    chains.setdefault(chain_head(isin), []).append(isin)

manifest_rows = []
for head_isin, chain_isins in chains.items():
    recs = pd.concat([by_isin.get_group(i) for i in chain_isins])
    starts = pd.to_datetime(recs["start_date"])
    real_ends = pd.to_datetime(recs[~recs["end_confidence"].str.startswith("isin_migration_to_")]["end_date"])
    still_active = real_ends.isna().any() if len(real_ends) else True
    last_symbol = by_isin.get_group(head_isin).iloc[-1]["symbol"]
    current_symbol = isin_to_current_symbol.get(head_isin, "")

    manifest_rows.append({
        "isin": head_isin,
        "prior_isins_in_chain": ";".join(i for i in chain_isins if i != head_isin),
        "current_symbol": current_symbol or last_symbol,
        "yfinance_ticker": f"{(current_symbol or last_symbol)}.NS",
        "resolved_from_live_master": bool(current_symbol),
        "download_start_date": starts.min().strftime("%Y-%m-%d"),
        "download_end_date": "" if still_active else real_ends.max().strftime("%Y-%m-%d"),
        "still_active": still_active,
    })

manifest_df = pd.DataFrame(manifest_rows)
# drop temporary "DUMMY" demerger placeholder scrips -- not real independently-tradeable securities
manifest_df = manifest_df[~manifest_df["current_symbol"].str.startswith("DUMMY")]
manifest_df.to_csv(f"{DATA_DIR}/download_manifest.csv", index=False)

print("companies to download:", len(manifest_df))
print("resolved to a live current symbol:", manifest_df["resolved_from_live_master"].sum())

companies to download: 1058
resolved to a live current symbol: 859


In [16]:

import pandas as pd

# Confirm the manifest exists
if "manifest_df" not in globals():
    raise NameError(
        "manifest_df is not available. Rerun the preceding "
        "manifest-building cell first."
    )

manifest = manifest_df.copy()

print("Total companies:", len(manifest))
print("\nManifest columns:")
print(manifest.columns.tolist())

def is_blank(series):
    return (
        series.isna()
        | series.astype(str).str.strip().str.lower().isin(
            ["", "nan", "none", "<na>", "nat"]
        )
    )

for col in ["current_symbol", "yfinance_ticker"]:
    if col in manifest.columns:
        missing = is_blank(manifest[col])
        print(f"\n{col}:")
        print("  Resolved:", int((~missing).sum()))
        print("  Missing:", int(missing.sum()))

# Examine records without a usable current symbol.
if "current_symbol" in manifest.columns:
    unresolved = manifest[is_blank(manifest["current_symbol"])]

    useful_columns = [
        "isin",
        "symbol",
        "company_name",
        "current_symbol",
        "yfinance_ticker",
        "download_start_date",
        "download_end_date",
        "prior_isins_in_chain",
    ]

    useful_columns = [
        c for c in useful_columns if c in manifest.columns
    ]

    print("\nUnresolved company sample:")
    display(unresolved[useful_columns].head(25))

    print("\nUnresolved records:", len(unresolved))
else:
    print(
        "\nThe manifest has no current_symbol column. "
        "Share the columns above so we can diagnose the mapping."
    )


Total companies: 1058

Manifest columns:
['isin', 'prior_isins_in_chain', 'current_symbol', 'yfinance_ticker', 'resolved_from_live_master', 'download_start_date', 'download_end_date', 'still_active']

current_symbol:
  Resolved: 1058
  Missing: 0

yfinance_ticker:
  Resolved: 1058
  Missing: 0

Unresolved company sample:


,isin,current_symbol,yfinance_ticker,download_start_date,download_end_date,prior_isins_in_chain



Unresolved records: 0


In [17]:

import pandas as pd

if "manifest_df" not in globals():
    raise NameError(
        "manifest_df is missing. Run the manifest-building cells first."
    )

m = manifest_df.copy()

# 1. Check how symbols were resolved
print("=" * 60)
print("1. SYMBOL RESOLUTION")
print("=" * 60)

if "resolved_from_live_master" in m.columns:
    print(
        m["resolved_from_live_master"]
        .fillna("MISSING")
        .astype(str)
        .value_counts(dropna=False)
        .to_string()
    )
else:
    print("No resolved_from_live_master column found.")

# 2. Inspect records not resolved through the NSE equity master
print("\n" + "=" * 60)
print("2. SYMBOLS NOT MATCHED TO THE LIVE MASTER")
print("=" * 60)

if "resolved_from_live_master" in m.columns:
    flag = (
        m["resolved_from_live_master"]
        .astype(str).str.strip().str.lower()
    )
    not_live = m[~flag.isin(["true", "1", "yes"])]

    print("Records not marked as live-master matches:", len(not_live))

    cols = [
        "isin",
        "prior_isins_in_chain",
        "current_symbol",
        "yfinance_ticker",
        "resolved_from_live_master",
        "download_start_date",
        "download_end_date",
        "still_active",
    ]
    cols = [c for c in cols if c in m.columns]

    display(not_live[cols].head(30))
else:
    print("Cannot classify resolution source without the flag.")

# 3. Check for duplicate ticker mappings
print("\n" + "=" * 60)
print("3. DUPLICATE TICKERS")
print("=" * 60)

print("Manifest rows:", len(m))
print("Unique ISIN rows:", m["isin"].nunique())
print("Unique current symbols:", m["current_symbol"].nunique())
print("Unique Yahoo tickers:", m["yfinance_ticker"].nunique())

duplicates = m[
    m.duplicated("yfinance_ticker", keep=False)
].sort_values("yfinance_ticker")

print("Rows involved in duplicate Yahoo tickers:", len(duplicates))

if not duplicates.empty:
    display(
        duplicates[
            ["isin", "prior_isins_in_chain",
             "current_symbol", "yfinance_ticker"]
        ].head(30)
    )

# 4. Check download date ranges
print("\n" + "=" * 60)
print("4. DOWNLOAD DATE VALIDATION")
print("=" * 60)

for col in ["download_start_date", "download_end_date"]:
    m[col] = pd.to_datetime(m[col], errors="coerce")

print("Missing start dates:", int(m["download_start_date"].isna().sum()))
print("Missing end dates:", int(m["download_end_date"].isna().sum()))
print(
    "End before start:",
    int((m["download_end_date"] < m["download_start_date"]).sum())
)
print("Earliest start:", m["download_start_date"].min())
print("Latest end:", m["download_end_date"].max())

print("\nAUDIT COMPLETE — no prices downloaded.")


1. SYMBOL RESOLUTION
resolved_from_live_master
True     859
False    199

2. SYMBOLS NOT MATCHED TO THE LIVE MASTER
Records not marked as live-master matches: 199


,isin,prior_isins_in_chain,current_symbol,yfinance_ticker,resolved_from_live_master,download_start_date,download_end_date,still_active
10,IN9155A01020,,TATAMTRDVR,TATAMTRDVR.NS,False,2016-12-28,2024-10-11,False
11,INE001A01036,INE001A01028,HDFC,HDFC.NS,False,2010-01-02,2022-05-02,False
21,INE008I01026,,COX&KINGS,COX&KINGS.NS,False,2012-01-12,2018-05-24,False
34,INE011A01019,,ESSAROIL,ESSAROIL.NS,False,2010-01-02,2014-02-27,False
37,INE013A01015,,RELCAPITAL,RELCAPITAL.NS,False,2010-01-02,2019-10-29,False
39,INE015A01028,,RANBAXY,RANBAXY.NS,False,2010-01-02,2014-11-18,False
44,INE018I01017,,MINDTREE,MINDTREE.NS,False,2010-01-02,2022-09-01,False
65,INE031B01015,,AJANTPHARM,AJANTPHARM.NS,False,2010-01-02,2012-01-12,False
70,INE036B01022,,GAEL,GAEL.NS,False,2010-01-02,2012-01-12,False
77,INE039E01020,,ERAINFRA,ERAINFRA.NS,False,2010-01-02,2014-08-20,False



3. DUPLICATE TICKERS
Manifest rows: 1058
Unique ISIN rows: 1058
Unique current symbols: 1045
Unique Yahoo tickers: 1045
Rows involved in duplicate Yahoo tickers: 26


,isin,prior_isins_in_chain,current_symbol,yfinance_ticker
678,INE517H01028,,ADVANTA,ADVANTA.NS
677,INE517H01010,,ADVANTA,ADVANTA.NS
65,INE031B01015,,AJANTPHARM,AJANTPHARM.NS
66,INE031B01049,,AJANTPHARM,AJANTPHARM.NS
694,INE532F01054,,EDELWEISS,EDELWEISS.NS
693,INE532F01047,,EDELWEISS,EDELWEISS.NS
70,INE036B01022,,GAEL,GAEL.NS
71,INE036B01030,,GAEL,GAEL.NS
301,INE176B01026,,HAVELLS,HAVELLS.NS
302,INE176B01034,,HAVELLS,HAVELLS.NS



4. DOWNLOAD DATE VALIDATION
Missing start dates: 0
Missing end dates: 500
End before start: 0
Earliest start: 2010-01-02 00:00:00
Latest end: 2026-09-30 00:00:00

AUDIT COMPLETE — no prices downloaded.


In [18]:

import pandas as pd
from itertools import combinations

m = manifest_df.copy()

# Parse date columns
for col in ["download_start_date", "download_end_date"]:
    m[col] = pd.to_datetime(m[col], errors="coerce")

print("=" * 65)
print("1. OPEN-ENDED DOWNLOAD PERIODS")
print("=" * 65)

print("\nStill-active classification:")
print(m["still_active"].value_counts(dropna=False).to_string())

missing_end = m["download_end_date"].isna()

print("\nMissing end dates:", int(missing_end.sum()))
print("\nMissing end dates by active flag:")
print(
    m.loc[missing_end, "still_active"]
    .value_counts(dropna=False)
    .to_string()
)

print(
    "\nMissing end dates marked inactive:",
    int((missing_end & ~m["still_active"].fillna(False)).sum())
)

# Use today's date only for comparing open-ended intervals.
as_of = pd.Timestamp.today().normalize()
m["_effective_end"] = m["download_end_date"].fillna(as_of)

print("\n" + "=" * 65)
print("2. DUPLICATE TICKER INTERVAL AUDIT")
print("=" * 65)

dup_mask = m.duplicated("yfinance_ticker", keep=False)
dups = m.loc[dup_mask].sort_values(
    ["yfinance_ticker", "download_start_date"]
)

pairs = []

for ticker, group in dups.groupby("yfinance_ticker"):
    records = list(group.iterrows())

    for (idx_a, a), (idx_b, b) in combinations(records, 2):
        if pd.isna(a["download_start_date"]) or pd.isna(
            b["download_start_date"]
        ):
            overlap = None
        else:
            latest_start = max(
                a["download_start_date"],
                b["download_start_date"],
            )
            earliest_end = min(
                a["_effective_end"],
                b["_effective_end"],
            )
            overlap = latest_start <= earliest_end

        pairs.append({
            "Ticker": ticker,
            "ISIN_A": a["isin"],
            "ISIN_B": b["isin"],
            "Start_A": a["download_start_date"],
            "End_A": a["download_end_date"],
            "Start_B": b["download_start_date"],
            "End_B": b["download_end_date"],
            "Prior_ISIN_A": a["prior_isins_in_chain"],
            "Prior_ISIN_B": b["prior_isins_in_chain"],
            "Intervals_Overlap": overlap,
        })

pair_df = pd.DataFrame(pairs)

if pair_df.empty:
    print("No duplicate ticker pairs found.")
else:
    print("Duplicate ticker pairs:", len(pair_df))
    print("\nPotentially overlapping pairs:")
    display(pair_df[pair_df["Intervals_Overlap"] == True])

    print("\nAll duplicate ticker pairs:")
    display(pair_df)

print("\n" + "=" * 65)
print("3. MANIFEST DATE SANITY CHECK")
print("=" * 65)

print(
    "Missing start dates:",
    int(m["download_start_date"].isna().sum())
)
print(
    "End date before start date:",
    int((
        m["download_end_date"].notna()
        & (m["download_end_date"] < m["download_start_date"])
    ).sum())
)

print("\nVALIDATION COMPLETE — no prices downloaded.")


1. OPEN-ENDED DOWNLOAD PERIODS

Still-active classification:
still_active
False    558
True     500

Missing end dates: 500

Missing end dates by active flag:
still_active
True    500

Missing end dates marked inactive: 0

2. DUPLICATE TICKER INTERVAL AUDIT
Duplicate ticker pairs: 13

Potentially overlapping pairs:


,Ticker,ISIN_A,ISIN_B,Start_A,End_A,Start_B,End_B,Prior_ISIN_A,Prior_ISIN_B,Intervals_Overlap
0,ADVANTA.NS,INE517H01010,INE517H01028,2010-01-02,2012-01-12,2012-01-12,2016-04-22,,,True
2,EDELWEISS.NS,INE532F01047,INE532F01054,2010-01-02,2012-01-12,2012-01-12,2023-04-19,,,True
4,HAVELLS.NS,INE176B01026,INE176B01034,2010-01-02,2014-11-15,2014-11-15,NaT,,,True
6,NBCC.NS,INE095N01015,INE095N01031,2014-02-27,2016-12-28,2016-12-28,NaT,,,True
10,TITAN.NS,INE280A01010,INE280A01028,2010-01-02,2012-01-12,2012-01-12,NaT,,,True



All duplicate ticker pairs:


,Ticker,ISIN_A,ISIN_B,Start_A,End_A,Start_B,End_B,Prior_ISIN_A,Prior_ISIN_B,Intervals_Overlap
0,ADVANTA.NS,INE517H01010,INE517H01028,2010-01-02,2012-01-12,2012-01-12,2016-04-22,,,True
1,AJANTPHARM.NS,INE031B01015,INE031B01049,2010-01-02,2012-01-12,2014-08-20,NaT,,,False
2,EDELWEISS.NS,INE532F01047,INE532F01054,2010-01-02,2012-01-12,2012-01-12,2023-04-19,,,True
3,GAEL.NS,INE036B01022,INE036B01030,2010-01-02,2012-01-12,2022-05-02,2025-02-21,,,False
4,HAVELLS.NS,INE176B01026,INE176B01034,2010-01-02,2014-11-15,2014-11-15,NaT,,,True
5,KSB.NS,INE999A01015,INE999A01023,2010-01-02,2025-02-21,2025-08-22,NaT,,,False
6,NBCC.NS,INE095N01015,INE095N01031,2014-02-27,2016-12-28,2016-12-28,NaT,,,True
7,RKFORGE.NS,INE399G01015,INE399G01023,2016-12-28,2019-08-28,2024-10-11,NaT,,,False
8,SAREGAMA.NS,INE979A01017,INE979A01025,2010-01-02,2012-01-12,2022-05-02,2026-09-30,,,False
9,TDPOWERSYS.NS,INE419M01019,INE419M01035,2012-01-12,2016-12-28,2026-09-30,NaT,,,False



3. MANIFEST DATE SANITY CHECK
Missing start dates: 0
End date before start date: 0

VALIDATION COMPLETE — no prices downloaded.


In [19]:

# ============================================================
# PIT DATASET: Correct exact-boundary overlaps
# Run once after manifest_df and intervals_df are built.
# ============================================================

import os
import pandas as pd
from itertools import combinations

if "manifest_df" not in globals() or "intervals_df" not in globals():
    raise NameError(
        "manifest_df and intervals_df must both exist. "
        "Run the preceding PIT-building sections first."
    )

# Work with timestamps for reliable date comparisons.
manifest_df = manifest_df.copy()
intervals_df = intervals_df.copy()

for col in ["download_start_date", "download_end_date"]:
    manifest_df[col] = pd.to_datetime(
        manifest_df[col], errors="coerce"
    ).dt.normalize()

interval_end_dates = pd.to_datetime(
    intervals_df["end_date"], errors="coerce"
).dt.normalize()

# Ensure the end-date column can accept date strings or timestamps.
intervals_df["end_date"] = intervals_df["end_date"].astype(object)

fixes = []

# Find cases where one ticker's old interval ends on the exact
# date its next ISIN interval starts.
for ticker, group in manifest_df.groupby("yfinance_ticker"):
    records = list(group.iterrows())

    for (idx_a, a), (idx_b, b) in combinations(records, 2):
        start_a = a["download_start_date"]
        end_a = a["download_end_date"]
        start_b = b["download_start_date"]
        end_b = b["download_end_date"]

        if pd.notna(end_a) and pd.notna(start_b) and end_a == start_b:
            old_idx, old_row, boundary = idx_a, a, start_b
        elif pd.notna(end_b) and pd.notna(start_a) and end_b == start_a:
            old_idx, old_row, boundary = idx_b, b, start_a
        else:
            continue

        old_isin = old_row["isin"]
        new_end = boundary - pd.Timedelta(days=1)

        fixes.append({
            "ticker": ticker,
            "old_isin": old_isin,
            "boundary": boundary,
            "new_end": new_end,
            "manifest_index": old_idx,
        })

# Apply the changes to both the download manifest and membership
# intervals so Section 6 uses the same corrected boundaries.
applied = []

for fix in fixes:
    old_isin = fix["old_isin"]
    boundary = fix["boundary"]
    new_end = fix["new_end"]

    # Update download range
    manifest_df.loc[
        fix["manifest_index"], "download_end_date"
    ] = new_end

    # Update only intervals for this ISIN ending on the shared date.
    end_dates_now = pd.to_datetime(
        intervals_df["end_date"], errors="coerce"
    ).dt.normalize()

    mask = (
        intervals_df["isin"].astype(str).eq(str(old_isin))
        & end_dates_now.eq(boundary)
    )

    matched = int(mask.sum())

    if matched:
        intervals_df.loc[mask, "end_date"] = new_end.strftime("%Y-%m-%d")

    applied.append({
        "Ticker": fix["ticker"],
        "Old_ISIN": old_isin,
        "Shared_Boundary": boundary.date(),
        "Corrected_End_Date": new_end.date(),
        "Matching_Intervals_Adjusted": matched,
    })

print("Boundary fixes identified:", len(fixes))
display(pd.DataFrame(applied))

if any(item["Matching_Intervals_Adjusted"] == 0 for item in applied):
    print(
        "\nWARNING: At least one manifest boundary did not match "
        "an interval row. Review the output before continuing."
    )
else:
    print("\nAll identified boundary overlaps were adjusted.")

# Save the corrected intermediate tables for recovery and audit.
os.makedirs(DATA_DIR, exist_ok=True)

manifest_df.to_csv(
    os.path.join(DATA_DIR, "pit_manifest_corrected.csv"),
    index=False,
)
intervals_df.to_csv(
    os.path.join(DATA_DIR, "pit_intervals_corrected.csv"),
    index=False,
)

print("\nSaved corrected manifest and interval tables.")
print("Do not rerun the earlier manifest-building cells afterward.")


Boundary fixes identified: 5


,Ticker,Old_ISIN,Shared_Boundary,Corrected_End_Date,Matching_Intervals_Adjusted
0,ADVANTA.NS,INE517H01010,2012-01-12,2012-01-11,1
1,EDELWEISS.NS,INE532F01047,2012-01-12,2012-01-11,1
2,HAVELLS.NS,INE176B01026,2014-11-15,2014-11-14,1
3,NBCC.NS,INE095N01015,2016-12-28,2016-12-27,1
4,TITAN.NS,INE280A01010,2012-01-12,2012-01-11,1



All identified boundary overlaps were adjusted.

Saved corrected manifest and interval tables.
Do not rerun the earlier manifest-building cells afterward.


In [20]:

import os
import zipfile
from google.colab import files

# Save the corrected in-memory tables again
os.makedirs(DATA_DIR, exist_ok=True)

manifest_df.to_csv(
    os.path.join(DATA_DIR, "pit_manifest_corrected.csv"),
    index=False,
)

intervals_df.to_csv(
    os.path.join(DATA_DIR, "pit_intervals_corrected.csv"),
    index=False,
)

checkpoint_path = "/content/PIT_PreDownload_Checkpoint.zip"

checkpoint_files = [
    "pit_manifest_corrected.csv",
    "pit_intervals_corrected.csv",
    "nse_equity_master.csv",
]

with zipfile.ZipFile(
    checkpoint_path, "w", zipfile.ZIP_DEFLATED
) as archive:
    for name in checkpoint_files:
        path = os.path.join(DATA_DIR, name)
        if os.path.exists(path):
            archive.write(path, arcname=name)
            print("Included:", name)
        else:
            print("WARNING — file missing:", path)

print("\nCheckpoint created:", checkpoint_path)
files.download(checkpoint_path)


Included: pit_manifest_corrected.csv
Included: pit_intervals_corrected.csv
Included: nse_equity_master.csv

Checkpoint created: /content/PIT_PreDownload_Checkpoint.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 6. Download OHLCV

Primary source: **yfinance**. For companies with no live ticker to query (delisted/merged/acquired), fall back to NSE's own historical archive via `jugaad-data`, which handles NSE's session/cookie requirements properly.

**This is the slow step.** Expect 30-60+ minutes for the full universe.

In [22]:
import yfinance as yf


def download_yfinance(ticker, start_date, end_date):
    """Download one ticker's OHLCV data safely."""

    try:
        start_ts = pd.to_datetime(start_date, errors="coerce")
        end_ts = pd.to_datetime(end_date, errors="coerce")

        # A missing start date is invalid; a missing end date
        # means the company is active through the latest date.
        if pd.isna(start_ts):
            print(f"Skipping {ticker}: invalid start date.")
            return None

        if pd.isna(end_ts):
            end_exclusive = (
                pd.Timestamp.now().normalize()
                + pd.Timedelta(days=1)
            )
        else:
            end_exclusive = (
                end_ts.normalize() + pd.Timedelta(days=1)
            )

        df = yf.Ticker(str(ticker).strip()).history(
            start=start_ts.strftime("%Y-%m-%d"),
            end=end_exclusive.strftime("%Y-%m-%d"),
            auto_adjust=False,
            actions=True,
        )

        if df is None or df.empty:
            print(f"No Yahoo Finance history for {ticker}.")
            return None

        df = df.reset_index()

        # Normalize the date column name.
        if "Date" not in df.columns:
            if "Datetime" in df.columns:
                df = df.rename(columns={"Datetime": "Date"})
            else:
                print(f"Missing date column for {ticker}.")
                return None

        df = df.rename(columns={
            "Date": "date",
            "Open": "open",
            "High": "high",
            "Low": "low",
            "Close": "close",
            "Volume": "volume",
            "Dividends": "dividends",
            "Stock Splits": "stock_splits",
        })

        # Some symbols may return incomplete column sets.
        for col in [
            "open", "high", "low", "close", "volume",
            "dividends", "stock_splits"
        ]:
            if col not in df.columns:
                df[col] = 0.0 if col in [
                    "dividends", "stock_splits"
                ] else float("nan")

        df = df[
            [
                "date", "open", "high", "low", "close",
                "volume", "dividends", "stock_splits"
            ]
        ]

        # Retain only observations with an actual closing price.
        df = df.dropna(subset=["close"])

        if df.empty:
            return None

        return df

    except Exception as exc:
        print(f"Yahoo Finance download failed for {ticker}: {exc}")
        return None


ok, fail = 0, 0
fail_list = []
for _, row in manifest_df.iterrows():
    out_path = f"{DATA_DIR}/ohlcv_yfinance/{row['isin']}.parquet"
    if os.path.exists(out_path):
        ok += 1; continue
    df = download_yfinance(row["yfinance_ticker"], row["download_start_date"], row["download_end_date"])
    if df is not None and not df.empty:
        df["isin"] = row["isin"]; df["symbol"] = row["current_symbol"]
        df.to_parquet(out_path, index=False)
        ok += 1
    else:
        fail += 1
        fail_list.append(row.to_dict())
    time.sleep(0.3)

print(f"yfinance: ok={ok} fail={fail}")

ERROR:yfinance:$TATAMTRDVR.NS: possibly delisted; no timezone found


No Yahoo Finance history for TATAMTRDVR.NS.


ERROR:yfinance:$HDFC.NS: possibly delisted; no timezone found


No Yahoo Finance history for HDFC.NS.


ERROR:yfinance:$COX&KINGS.NS: possibly delisted; no timezone found


...

No Yahoo Finance history for ELDERPHARM.NS.


ERROR:yfinance:$EQUITAS.NS: possibly delisted; no timezone found


No Yahoo Finance history for EQUITAS.NS.


ERROR:yfinance:$LAKSHMIEFL.NS: possibly delisted; no timezone found


No Yahoo Finance history for LAKSHMIEFL.NS.
yfinance: ok=894 fail=164


In [23]:

# ============================================================
# SECTION 6: NSE historical-data fallback
# Handles missing end dates and preserves successful downloads.
# ============================================================

import os
import time
import pandas as pd
from datetime import datetime
from jugaad_data.nse import stock_df

fallback_dir = os.path.join(DATA_DIR, "ohlcv_bhavcopy")
os.makedirs(fallback_dir, exist_ok=True)

def safe_date(value):
    """Convert a date-like value to datetime.date; return None if missing."""
    if value is None or pd.isna(value):
        return None

    value = str(value).strip()

    if value.lower() in ("", "nan", "nat", "none"):
        return None

    try:
        return pd.to_datetime(value).date()
    except (ValueError, TypeError):
        return None


def download_bhavcopy(symbol, start_date, end_date):
    """Download historical NSE OHLCV for a ticker where available."""

    start = safe_date(start_date)
    end = safe_date(end_date)

    if start is None:
        print(f"Invalid start date for {symbol}; skipping.")
        return None

    # Active companies with no end date are downloaded through today.
    if end is None:
        end = datetime.today().date()

    if end < start:
        print(f"End date precedes start date for {symbol}; skipping.")
        return None

    try:
        df = stock_df(
            symbol=str(symbol).strip(),
            from_date=start,
            to_date=end,
            series="EQ",
        )
    except Exception as exc:
        print(f"NSE fallback failed for {symbol}: {str(exc)[:180]}")
        return None

    if df is None or df.empty:
        return None

    df = df.rename(columns={
        "DATE": "date",
        "OPEN": "open",
        "HIGH": "high",
        "LOW": "low",
        "CLOSE": "close",
        "VOLUME": "volume",
    })

    required = ["date", "open", "high", "low", "close", "volume"]

    if not set(required).issubset(df.columns):
        print(
            f"Unexpected NSE columns for {symbol}: "
            f"{df.columns.tolist()}"
        )
        return None

    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df["date"] = df["date"].dt.normalize()

    for col in ["open", "high", "low", "close", "volume"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna(subset=["date", "open", "high", "low", "close"])

    if df.empty:
        return None

    # Preserve the repository's documented weekend-date correction.
    dow = df["date"].dt.dayofweek
    df.loc[dow == 5, "date"] -= pd.Timedelta(days=1)
    df.loc[dow == 6, "date"] -= pd.Timedelta(days=2)

    df = df.sort_values("volume", ascending=False)
    df = df.drop_duplicates(subset=["date"], keep="first")

    df["dividends"] = 0.0
    df["stock_splits"] = 0.0

    return df[
        [
            "date", "open", "high", "low", "close",
            "volume", "dividends", "stock_splits",
        ]
    ].sort_values("date")


# Use the failure list produced by the Yahoo Finance stage.
if "fail_list" not in globals():
    raise NameError(
        "fail_list is missing. Run the Yahoo Finance download cell first."
    )

bhav_ok = 0
bhav_fail = 0
bhav_fail_list = []

for i, row in enumerate(fail_list, start=1):
    isin = row["isin"]
    symbol = row["current_symbol"]

    out_path = os.path.join(fallback_dir, f"{isin}.parquet")

    # Resume from successful fallback files if rerun.
    if os.path.exists(out_path):
        bhav_ok += 1
        continue

    df = download_bhavcopy(
        symbol,
        row["download_start_date"],
        row["download_end_date"],
    )

    if df is not None and not df.empty:
        df["isin"] = isin
        df["symbol"] = symbol
        df.to_parquet(out_path, index=False)
        bhav_ok += 1
    else:
        bhav_fail += 1
        bhav_fail_list.append({
            "isin": isin,
            "current_symbol": symbol,
            "download_start_date": row["download_start_date"],
            "download_end_date": row["download_end_date"],
        })

    if i % 10 == 0 or i == len(fail_list):
        print(
            f"Processed {i}/{len(fail_list)} | "
            f"fallback successes: {bhav_ok} | "
            f"failures: {bhav_fail}"
        )

    # Avoid hammering the data source.
    time.sleep(0.5)

print("\n========== NSE FALLBACK SUMMARY ==========")
print("Yahoo failures submitted to fallback:", len(fail_list))
print("NSE fallback successes:", bhav_ok)
print("NSE fallback failures:", bhav_fail)

failure_path = os.path.join(DATA_DIR, "nse_fallback_failures.csv")
pd.DataFrame(bhav_fail_list).to_csv(failure_path, index=False)

print("Failed records saved to:", failure_path)


Processed 10/164 | fallback successes: 10 | failures: 0
Processed 20/164 | fallback successes: 20 | failures: 0
NSE fallback failed for MANPASAND: Expecting value: line 2 column 1 (char 1)
Processed 30/164 | fallback successes: 29 | failures: 1
Processed 40/164 | fallback successes: 39 | failures: 1
Processed 50/164 | fallback successes: 49 | failures: 1
Processed 60/164 | fallback successes: 59 | failures: 1
Processed 70/164 | fallback successes: 69 | failures: 1
Processed 80/164 | fallback successes: 79 | failures: 1
Processed 90/164 | fallback successes: 89 | failures: 1
Processed 100/164 | fallback successes: 99 | failures: 1
Processed 110/164 | fallback successes: 109 | failures: 1
Processed 120/164 | fallback successes: 119 | failures: 1
Processed 130/164 | fallback successes: 129 | failures: 1
Processed 140/164 | fallback successes: 139 | failures: 1
Processed 150/164 | fallback successes: 149 | failures: 1
Processed 160/164 | fallback successes: 159 | failures: 1
Processed 164/

## 7. Consolidate into the final PIT parquet

Merge both sources, join against the membership-interval table so every row carries the point-in-time constituent flag, and apply the extra cleanup steps that catch spurious low-volume single-share prints in the fallback source.

In [24]:
all_frames = []
for _, row in manifest_df.iterrows():
    isin = row["isin"]
    chain = [isin] + [x for x in str(row["prior_isins_in_chain"]).split(";") if x]

    yf_path = f"{DATA_DIR}/ohlcv_yfinance/{isin}.parquet"
    bhav_path = f"{DATA_DIR}/ohlcv_bhavcopy/{isin}.parquet"
    if os.path.exists(yf_path):
        df = pd.read_parquet(yf_path); df["data_source"] = "yfinance"
        df["adjustment"] = "split_adjusted_no_dividend"
    elif os.path.exists(bhav_path):
        df = pd.read_parquet(bhav_path); df["data_source"] = "nse_bhavcopy_raw"
        df["adjustment"] = "raw_unadjusted"
        # drop spurious near-zero-volume prints wildly off the stock's trading range
        df = df.sort_values("date").reset_index(drop=True)
        prev_close = df["close"].shift(1)
        pct = (df["close"] - prev_close).abs() / prev_close
        df = df[~((df["volume"] < 200) & (pct > 0.40))].reset_index(drop=True)
    else:
        continue

    df["date"] = pd.to_datetime(df["date"]).dt.tz_localize(None).dt.normalize()
    combined_intervals = []
    for chain_isin in chain:
        sub = intervals_df[intervals_df["isin"] == chain_isin]
        for _, r in sub.iterrows():
            s = pd.Timestamp(r["start_date"])
            e = pd.Timestamp(r["end_date"]) if r["end_date"] and r["end_date"] != "NaT" else pd.Timestamp.today()
            combined_intervals.append((s, e))

    df["is_nifty500_constituent"] = df["date"].apply(lambda d: any(s <= d <= e for s, e in combined_intervals))
    all_frames.append(df[["isin","symbol","date","open","high","low","close","volume","dividends",
                           "stock_splits","is_nifty500_constituent","data_source","adjustment"]])

final_df = pd.concat(all_frames, ignore_index=True)
final_df = final_df[final_df["close"].notna()]  # drop any still-forming/incomplete session rows
final_df = final_df.drop_duplicates(subset=["isin","date"], keep="first")
final_df = final_df.sort_values(["isin","date"]).reset_index(drop=True)

OUT_PATH = f"{DATA_DIR}/nifty500_ohlcv_pit.parquet"
final_df.to_parquet(OUT_PATH, index=False)

print("FINAL DATASET")
print("path:", OUT_PATH)
print("rows:", len(final_df), " companies:", final_df['isin'].nunique())
print("date range:", final_df["date"].min(), "->", final_df["date"].max())

FINAL DATASET
path: nifty500_pit_data/nifty500_ohlcv_pit.parquet
rows: 2124958  companies: 1057
date range: 2010-01-01 00:00:00 -> 2026-10-23 00:00:00


## 8. Validate

A quick sanity pass before trusting the output for backtesting.

In [25]:
df = final_df

print("duplicate (isin,date) rows:", df.duplicated(subset=["isin","date"]).sum())
print("null prices:", df[["open","high","low","close"]].isna().sum().sum())
print("zero/negative prices:", (df[["open","high","low","close"]] <= 0).sum().sum())

bad_high = df["high"] < df[["open","low","close"]].max(axis=1)
bad_low = df["low"] > df[["open","high","close"]].min(axis=1)
print("OHLC internal-consistency violations:", int((bad_high | bad_low).sum()))

# the key sanity check: does the daily constituent count actually land near 500?
for d in pd.date_range(df["date"].min(), df["date"].max(), periods=6):
    d = d.normalize()
    n = df[(df["date"] == d) & (df["is_nifty500_constituent"])].shape[0]
    print(f"{d.date()}:  {n} constituents")

duplicate (isin,date) rows: 0
null prices: 0
zero/negative prices: 0
OHLC internal-consistency violations: 14
2010-01-01:  0 constituents
2013-05-12:  0 constituents
2016-09-21:  325 constituents
2020-02-01:  0 constituents
2023-06-13:  184 constituents
2026-10-23:  0 constituents


In [26]:

# ============================================================
# DATA QUALITY FIX: Recompute point-in-time membership flags
# Does NOT download prices.
# ============================================================

import os
import numpy as np
import pandas as pd

# Confirm that the three required in-memory objects exist.
for name in ["final_df", "manifest_df", "intervals_df"]:
    if name not in globals():
        raise NameError(
            f"{name} is unavailable. This requires the same "
            "PIT notebook session used to build the dataset."
        )

df = final_df.copy()
manifest = manifest_df.copy()
intervals = intervals_df.copy()

df["date"] = pd.to_datetime(
    df["date"], errors="coerce"
).dt.tz_localize(None).dt.normalize()

for col in ["start_date", "end_date"]:
    intervals[col] = pd.to_datetime(
        intervals[col], errors="coerce"
    ).dt.tz_localize(None).dt.normalize()

df["isin"] = df["isin"].astype(str).str.strip()
manifest["isin"] = manifest["isin"].astype(str).str.strip()
intervals["isin"] = intervals["isin"].astype(str).str.strip()

dataset_end = df["date"].max()

if pd.isna(dataset_end):
    raise ValueError("The dataset has no valid dates.")

# Build membership intervals by the current ISIN and its prior ISIN chain.
intervals_by_isin = {
    isin: group
    for isin, group in intervals.groupby("isin")
}

# Start with every row marked as not a constituent.
df["is_nifty500_constituent"] = False

updated_isins = 0
missing_interval_isins = []

for _, row in manifest.iterrows():
    current_isin = row["isin"]

    prior = row.get("prior_isins_in_chain", "")
    prior = "" if pd.isna(prior) else str(prior)

    chain = [current_isin] + [
        x.strip()
        for x in prior.split(";")
        if x.strip() and x.strip().lower() != "nan"
    ]

    company_intervals = []

    for chain_isin in chain:
        group = intervals_by_isin.get(chain_isin)

        if group is None:
            continue

        for _, iv in group.iterrows():
            start = iv["start_date"]
            end = iv["end_date"]

            if pd.isna(start):
                continue

            # Crucial fix: missing end means open-ended membership.
            # Cap it at the latest date available in our dataset.
            if pd.isna(end):
                end = dataset_end
            else:
                end = min(end, dataset_end)

            if end >= start:
                company_intervals.append((start, end))

    if not company_intervals:
        missing_interval_isins.append(current_isin)
        continue

    row_mask = df["isin"].eq(current_isin)
    company_dates = df.loc[row_mask, "date"]

    member = np.zeros(len(company_dates), dtype=bool)

    for start, end in company_intervals:
        member |= (
            company_dates.ge(start)
            & company_dates.le(end)
        ).to_numpy()

    df.loc[row_mask, "is_nifty500_constituent"] = member
    updated_isins += 1

print("Dataset end date:", dataset_end.date())
print("ISINs with usable membership intervals:", updated_isins)
print("ISINs without usable intervals:", len(missing_interval_isins))

if missing_interval_isins:
    print("First missing ISINs:", missing_interval_isins[:20])

# Preserve the original final dataset until the corrected one validates.
PIT_DIR = os.path.dirname(OUT_PATH) if "OUT_PATH" in globals() else "nifty500_pit_data"

if not os.path.isabs(PIT_DIR):
    PIT_DIR = os.path.join("/content", PIT_DIR)

os.makedirs(PIT_DIR, exist_ok=True)

corrected_path = os.path.join(
    PIT_DIR, "nifty500_ohlcv_pit_membership_fixed.parquet"
)

df.to_parquet(corrected_path, index=False)

print("\nSaved corrected candidate:", corrected_path)

# ------------------------------------------------------------
# Validate counts on dates where price records actually exist.
# ------------------------------------------------------------

counts = (
    df[df["is_nifty500_constituent"]]
    .groupby("date")["isin"]
    .nunique()
)

print("\nDaily constituent counts:")
if len(counts):
    display(counts.describe(percentiles=[0.10, 0.25, 0.50, 0.75, 0.90]).to_frame("Count").round(1))
else:
    print("No rows are currently marked as constituents.")

print("\nCounts near previously sampled dates:")
available_dates = pd.DatetimeIndex(
    df["date"].dropna().unique()
).sort_values()

for target_text in [
    "2016-09-21",
    "2023-06-13",
    "2026-10-23",
]:
    target = pd.Timestamp(target_text)

    if len(available_dates) == 0:
        continue

    nearest = available_dates[
        np.abs(available_dates - target).argmin()
    ]

    day = df[df["date"].eq(nearest)]
    true_day = day[day["is_nifty500_constituent"]]

    print(
        f"Target {target.date()}, checked {nearest.date()}: "
        f"{day['isin'].nunique()} companies with prices, "
        f"{true_day['isin'].nunique()} flagged constituents"
    )

# Keep corrected data in memory for subsequent inspection.
final_df = df
print("\nMEMBERSHIP FLAG REPAIR COMPLETE — review counts before use.")


Dataset end date: 2026-10-23
ISINs with usable membership intervals: 1058
ISINs without usable intervals: 0

Saved corrected candidate: /content/nifty500_pit_data/nifty500_ohlcv_pit_membership_fixed.parquet

Daily constituent counts:


,Count
count,4377.0
mean,471.4
std,100.6
min,1.0
10%,477.0
25%,492.0
50%,497.0
75%,500.0
90%,504.0
max,561.0



Counts near previously sampled dates:
Target 2016-09-21, checked 2016-09-21: 518 companies with prices, 512 flagged constituents
Target 2023-06-13, checked 2023-06-13: 530 companies with prices, 504 flagged constituents
Target 2026-10-23, checked 2026-10-23: 499 companies with prices, 499 flagged constituents

MEMBERSHIP FLAG REPAIR COMPLETE — review counts before use.


In [27]:

# ============================================================
# STEP 9: Investigate outlier constituent counts and OHLC rows
# ============================================================

import pandas as pd
import numpy as np

df = final_df.copy()

# Normalize and validate price fields
df["date"] = pd.to_datetime(df["date"], errors="coerce").dt.normalize()

price_cols = ["open", "high", "low", "close"]
for col in price_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# ------------------------------------------------------------
# 1. Inspect dates with the most flagged constituents
# ------------------------------------------------------------

print("=" * 65)
print("1. DATES WITH THE HIGHEST CONSTITUENT COUNTS")
print("=" * 65)

counts = (
    df[df["is_nifty500_constituent"]]
    .groupby("date")["isin"]
    .nunique()
    .sort_values(ascending=False)
)

print("Dates with membership data:", len(counts))
print("Maximum count:", int(counts.max()))
print("Median count:", round(counts.median(), 1))

print("\nTop 15 dates by constituent count:")
display(counts.head(15).rename("Constituent_Count").to_frame())

print("\nDates with fewer than 450 flagged constituents:")
display(
    counts[counts < 450]
    .sort_index()
    .head(20)
    .rename("Constituent_Count")
    .to_frame()
)

# ------------------------------------------------------------
# 2. Check constituent counts by year
# ------------------------------------------------------------

annual = counts.rename("Constituent_Count").to_frame()
annual["Year"] = annual.index.year

print("\n" + "=" * 65)
print("2. CONSTITUENT COUNTS BY YEAR")
print("=" * 65)

display(
    annual.groupby("Year")["Constituent_Count"]
    .agg(["count", "min", "median", "mean", "max"])
    .round(1)
)

# ------------------------------------------------------------
# 3. Investigate OHLC consistency violations
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("3. OHLC VIOLATIONS")
print("=" * 65)

valid_ohlc = df[price_cols].notna().all(axis=1)

bad_high = df["high"] < df[["open", "low", "close"]].max(axis=1)
bad_low = df["low"] > df[["open", "high", "close"]].min(axis=1)

violations = df[valid_ohlc & (bad_high | bad_low)].copy()

print("Rows with complete OHLC:", int(valid_ohlc.sum()))
print("OHLC violations:", len(violations))

columns_to_show = [
    "isin", "symbol", "date",
    "open", "high", "low", "close", "volume",
    "data_source", "adjustment",
]
columns_to_show = [
    c for c in columns_to_show if c in violations.columns
]

if not violations.empty:
    display(
        violations[columns_to_show]
        .sort_values("date")
        .head(30)
    )

    print("\nViolation counts by source/adjustment:")
    group_cols = [
        c for c in ["data_source", "adjustment"]
        if c in violations.columns
    ]
    if group_cols:
        display(
            violations.groupby(group_cols)
            .size()
            .rename("Violation_Count")
            .to_frame()
        )

# ------------------------------------------------------------
# 4. Check price-source and adjustment coverage
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("4. PRICE SOURCE / ADJUSTMENT MIX")
print("=" * 65)

if "adjustment" in df.columns and "data_source" in df.columns:
    display(
        df.groupby(["data_source", "adjustment"])
        .agg(
            Rows=("isin", "size"),
            Companies=("isin", "nunique"),
            First_Date=("date", "min"),
            Last_Date=("date", "max"),
        )
        .reset_index()
    )

print("\nINVESTIGATION COMPLETE — no data was modified.")


1. DATES WITH THE HIGHEST CONSTITUENT COUNTS
Dates with membership data: 4377
Maximum count: 561
Median count: 497.0

Top 15 dates by constituent count:


,Constituent_Count
date,
2012-01-12,561
2016-12-28,526
2026-09-30,526
2025-02-21,525
2022-05-02,522
2016-02-22,519
2026-02-23,519
2019-10-29,516
2024-02-28,514



Dates with fewer than 450 flagged constituents:


,Constituent_Count
date,
2010-01-25,382
2010-01-26,118
2010-02-06,7
2010-02-11,382
2010-02-12,118
2010-02-26,382
2010-03-01,117
2010-03-23,382
2010-03-24,116



2. CONSTITUENT COUNTS BY YEAR


...


Violation counts by source/adjustment:


,,Violation_Count
data_source,adjustment,
yfinance,split_adjusted_no_dividend,14



4. PRICE SOURCE / ADJUSTMENT MIX


,data_source,adjustment,Rows,Companies,First_Date,Last_Date
0,nse_bhavcopy_raw,raw_unadjusted,217132,163,2010-01-01,2026-10-22
1,yfinance,split_adjusted_no_dividend,1907826,894,2010-01-04,2026-10-23



INVESTIGATION COMPLETE — no data was modified.


In [28]:

import pandas as pd
import numpy as np

df = final_df.copy()

df["date"] = pd.to_datetime(df["date"], errors="coerce").dt.normalize()

# ------------------------------------------------------------
# 1. Inspect unusually low/high constituent counts
# ------------------------------------------------------------

flagged = df[df["is_nifty500_constituent"]].copy()

daily = (
    flagged.groupby("date")
    .agg(
        Constituents=("isin", "nunique"),
        Rows=("isin", "size"),
        Sources=("data_source", "nunique"),
    )
    .sort_index()
)

print("=" * 65)
print("1. LOW-COUNT DATES")
print("=" * 65)

display(
    daily[daily["Constituents"] < 450]
    .sort_values("Constituents")
    .head(15)
)

print("\n" + "=" * 65)
print("2. HIGH-COUNT DATES")
print("=" * 65)

display(
    daily[daily["Constituents"] > 510]
    .sort_values("Constituents", ascending=False)
    .head(15)
)

# ------------------------------------------------------------
# 2. Inspect selected dates and their data sources
# ------------------------------------------------------------

focus_dates = [
    "2010-01-25",
    "2010-01-26",
    "2010-02-06",
    "2012-01-12",
    "2016-12-28",
    "2023-06-13",
    "2026-10-23",
]

print("\n" + "=" * 65)
print("3. DETAILED DATE INSPECTION")
print("=" * 65)

for date_text in focus_dates:
    target = pd.Timestamp(date_text)
    day = df[df["date"].eq(target)]
    members = day[day["is_nifty500_constituent"]]

    print(f"\nDate: {date_text} | Weekday: {target.day_name()}")
    print("Total company rows:", day["isin"].nunique())
    print("Flagged constituents:", members["isin"].nunique())

    if not day.empty:
        print("\nRows by source and membership flag:")
        display(
            day.groupby(["data_source", "is_nifty500_constituent"])
            .agg(
                Rows=("isin", "size"),
                Companies=("isin", "nunique"),
            )
            .reset_index()
        )

        print("Sample flagged rows:")
        cols = [
            "isin", "symbol", "date", "close",
            "volume", "data_source", "adjustment",
        ]
        cols = [c for c in cols if c in members.columns]
        display(members[cols].head(12))

# ------------------------------------------------------------
# 3. Inspect OHLC violations without changing prices
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("4. OHLC VIOLATION DETAILS")
print("=" * 65)

for col in ["open", "high", "low", "close"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

complete = df[["open", "high", "low", "close"]].notna().all(axis=1)

bad_high = df["high"] < df[["open", "low", "close"]].max(axis=1)
bad_low = df["low"] > df[["open", "high", "close"]].min(axis=1)

violations = df[complete & (bad_high | bad_low)].copy()

violations["Close_Outside_Range"] = (
    (violations["close"] > violations["high"])
    | (violations["close"] < violations["low"])
)

print("Total violations:", len(violations))
print(
    "Rows where close is outside high-low range:",
    int(violations["Close_Outside_Range"].sum()),
)

cols = [
    "isin", "symbol", "date",
    "open", "high", "low", "close", "volume",
    "data_source", "adjustment", "Close_Outside_Range",
]
cols = [c for c in cols if c in violations.columns]

display(violations[cols].sort_values("date"))

print("\nDIAGNOSTICS COMPLETE — no data was modified.")


1. LOW-COUNT DATES


,Constituents,Rows,Sources
date,,,
2012-01-07,1,1,1
2012-03-03,1,1,1
2012-09-08,1,1,1
2026-10-20,1,1,1
2026-05-28,1,1,1
2026-06-26,1,1,1
2026-10-02,1,1,1
2026-03-03,2,2,1
2026-03-26,2,2,1



2. HIGH-COUNT DATES


,Constituents,Rows,Sources
date,,,
2012-01-12,561,561,2
2016-12-28,526,526,2
2026-09-30,526,526,2
2025-02-21,525,525,2
2022-05-02,522,522,1
2016-02-22,519,519,2
2026-02-23,519,519,2
2019-10-29,516,516,2
2024-02-28,514,514,2



3. DETAILED DATE INSPECTION

Date: 2010-01-25 | Weekday: Monday
Total company rows: 382
Flagged constituents: 382

Rows by source and membership flag:


...

Sample flagged rows:


,isin,symbol,date,close,volume,data_source,adjustment
11323,INE002A01018,RELIANCE,2026-10-23,1190.000000,14905641,yfinance,split_adjusted_no_dividend
13750,INE002L01015,SJVN,2026-10-23,47.400002,5885403,yfinance,split_adjusted_no_dividend
16142,INE002S01010,MGL,2026-10-23,1023.099976,590802,yfinance,split_adjusted_no_dividend
20290,INE003A01024,SIEMENS,2026-10-23,3245.199951,378876,yfinance,split_adjusted_no_dividend
23293,INE006I01046,ASTRAL,2026-10-23,1357.000000,318950,yfinance,split_adjusted_no_dividend
27442,INE007A01025,CRISIL,2026-10-23,4751.100098,66423,yfinance,split_adjusted_no_dividend
33106,INE008A01015,IDBI,2026-10-23,76.440002,9188527,yfinance,split_adjusted_no_dividend
38826,INE009A01021,INFY,2026-10-23,1153.800049,33044499,yfinance,split_adjusted_no_dividend
39882,INE00FF01025,ACUTAAS,2026-10-23,3137.699951,348109,yfinance,split_adjusted_no_dividend
40293,INE00H001014,SWIGGY,2026-10-23,176.000000,137428526,yfinance,split_adjusted_no_dividend



4. OHLC VIOLATION DETAILS
Total violations: 14
Rows where close is outside high-low range: 10


,isin,symbol,date,open,high,low,close,volume,data_source,adjustment,Close_Outside_Range
810394,INE236A01020,HCL-INSYS,2010-01-21,152.863922,146.908173,147.305222,154.154327,291847,yfinance,split_adjusted_no_dividend,True
1518486,INE586B01026,TAJGVK,2010-01-25,150.100006,149.100006,154.000000,155.449997,158126,yfinance,split_adjusted_no_dividend,True
404749,INE107A01015,TNPL,2010-01-27,82.800003,81.949997,83.000000,84.500000,135761,yfinance,split_adjusted_no_dividend,True
810401,INE236A01020,HCL-INSYS,2010-02-02,134.004074,134.004074,134.996704,138.669403,322785,yfinance,split_adjusted_no_dividend,True
586175,INE168A01041,J&KBANK,2010-02-03,60.700001,59.345001,60.000000,60.980000,2550020,yfinance,split_adjusted_no_dividend,True
1196127,INE419M01019,TDPOWERSYS,2012-07-20,28.905001,28.905001,28.905001,28.904999,10,yfinance,split_adjusted_no_dividend,True
1196128,INE419M01019,TDPOWERSYS,2012-07-23,28.900000,29.195000,28.900000,29.195002,3830,yfinance,split_adjusted_no_dividend,True
1747621,INE725E01024,ORISSAMINE,2012-07-25,3049.995117,3636.004883,3030.010010,3636.005127,119620,yfinance,split_adjusted_no_dividend,True
1747647,INE725E01024,ORISSAMINE,2012-09-03,5809.384766,5809.384766,5256.115234,5256.114746,52630,yfinance,split_adjusted_no_dividend,True
1196157,INE419M01019,TDPOWERSYS,2012-09-04,27.200001,27.969999,27.200001,27.970001,710,yfinance,split_adjusted_no_dividend,True



DIAGNOSTICS COMPLETE — no data was modified.


In [29]:

# ============================================================
# STEP 11: Validate PIT data against the Nifty 500 TRI calendar
# ============================================================

import os
import io
import zipfile
import numpy as np
import pandas as pd
from google.colab import files

PIT_DIR = "/content/nifty500_pit_data"
os.makedirs(PIT_DIR, exist_ok=True)

TRI_PATH = os.path.join(PIT_DIR, "nifty500_tri.csv")

# ------------------------------------------------------------
# 1. Import the benchmark file into the PIT notebook
# ------------------------------------------------------------

if not os.path.exists(TRI_PATH):
    print(
        "Upload either nifty500_tri.csv or "
        "Indian_Stock_Algo_PrePIT_Backup.zip from your Mac."
    )
    uploaded = files.upload()

    if not uploaded:
        raise ValueError("No benchmark file was uploaded.")

    tri_data = None

    for filename, content in uploaded.items():
        if filename.lower().endswith(".zip"):
            with zipfile.ZipFile(io.BytesIO(content)) as z:
                matches = [
                    name for name in z.namelist()
                    if os.path.basename(name).lower()
                    == "nifty500_tri.csv"
                ]

                if matches:
                    with z.open(matches[0]) as f:
                        tri_data = pd.read_csv(f)
                    break

        elif os.path.basename(filename).lower() == "nifty500_tri.csv":
            tri_data = pd.read_csv(io.BytesIO(content))
            break

    if tri_data is None:
        raise FileNotFoundError(
            "Could not find nifty500_tri.csv in the uploaded files. "
            "Upload the CSV itself or the earlier project backup ZIP."
        )

    tri_data.to_csv(TRI_PATH, index=False)
    print("Benchmark imported:", TRI_PATH)

# ------------------------------------------------------------
# 2. Load and validate the market calendar
# ------------------------------------------------------------

tri_raw = pd.read_csv(TRI_PATH)
tri_raw.columns = [str(c).strip() for c in tri_raw.columns]

def norm_col(name):
    return "".join(
        ch.lower() for ch in str(name) if ch.isalnum()
    )

date_col = next(
    (c for c in tri_raw.columns if norm_col(c) == "date"),
    None,
)

if date_col is None:
    # Handles a CSV whose first column is the saved index.
    date_col = tri_raw.columns[0]

tri_dates = pd.to_datetime(
    tri_raw[date_col], errors="coerce", dayfirst=True
).dropna()

calendar = pd.DatetimeIndex(
    tri_dates.dt.normalize().unique()
).sort_values()

calendar = calendar[
    (calendar >= pd.Timestamp("2015-01-01"))
    & (calendar <= pd.Timestamp("2026-10-19"))
]

if len(calendar) == 0:
    raise ValueError("No valid trading-calendar dates found.")

calendar_set = set(calendar)

print("Benchmark trading dates:", len(calendar))
print("First date:", calendar.min().date())
print("Last date:", calendar.max().date())

# ------------------------------------------------------------
# 3. Load the corrected PIT candidate
# ------------------------------------------------------------

if "final_df" in globals():
    source_df = final_df.copy()
else:
    corrected_path = os.path.join(
        PIT_DIR, "nifty500_ohlcv_pit_membership_fixed.parquet"
    )

    if not os.path.exists(corrected_path):
        raise FileNotFoundError(
            "Neither final_df nor the corrected Parquet file is "
            "available. Restore the corrected PIT dataset first."
        )

    source_df = pd.read_parquet(corrected_path)

source_df["date"] = pd.to_datetime(
    source_df["date"], errors="coerce"
).dt.normalize()

# The benchmark covers only through 2026-10-19.
period = source_df[
    (source_df["date"] >= pd.Timestamp("2015-01-01"))
    & (source_df["date"] <= calendar.max())
].copy()

on_calendar = period["date"].isin(calendar_set)
off_calendar = period.loc[~on_calendar].copy()
calendar_data = period.loc[on_calendar].copy()

print("\n" + "=" * 65)
print("1. TRADING-CALENDAR ALIGNMENT")
print("=" * 65)
print("PIT rows in validation period:", len(period))
print("Rows on benchmark trading dates:", len(calendar_data))
print("Rows dated outside benchmark calendar:", len(off_calendar))
print("Distinct off-calendar dates:", off_calendar["date"].nunique())

if not off_calendar.empty:
    print("\nMost frequent off-calendar dates:")
    display(
        off_calendar.groupby("date")
        .agg(
            Rows=("isin", "size"),
            Companies=("isin", "nunique"),
            Source_Count=("data_source", "nunique"),
        )
        .sort_values("Rows", ascending=False)
        .head(20)
    )

# ------------------------------------------------------------
# 4. Recheck constituent counts only on valid trading dates
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("2. MEMBERSHIP COUNTS ON VALID TRADING DATES")
print("=" * 65)

members = calendar_data[
    calendar_data["is_nifty500_constituent"].astype(bool)
]

daily_counts = (
    members.groupby("date")["isin"]
    .nunique()
    .reindex(calendar, fill_value=0)
)

print("Trading dates checked:", len(daily_counts))
print("Minimum constituents:", int(daily_counts.min()))
print("Median constituents:", round(daily_counts.median(), 1))
print("Maximum constituents:", int(daily_counts.max()))

print("\nDates with fewer than 450 flagged constituents:")
display(
    daily_counts[daily_counts < 450]
    .rename("Constituents")
    .to_frame()
    .head(25)
)

print("\nDates with more than 515 flagged constituents:")
display(
    daily_counts[daily_counts > 515]
    .rename("Constituents")
    .to_frame()
)

# ------------------------------------------------------------
# 5. Recheck OHLC violations on valid trading dates
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("3. OHLC PRICE QUALITY")
print("=" * 65)

ohlc_cols = ["open", "high", "low", "close"]

for col in ohlc_cols:
    calendar_data[col] = pd.to_numeric(
        calendar_data[col], errors="coerce"
    )

complete = calendar_data[ohlc_cols].notna().all(axis=1)

bad_high = (
    calendar_data["high"]
    < calendar_data[["open", "low", "close"]].max(axis=1)
)

bad_low = (
    calendar_data["low"]
    > calendar_data[["open", "high", "close"]].min(axis=1)
)

violations = calendar_data[
    complete & (bad_high | bad_low)
].copy()

print("Complete OHLC rows:", int(complete.sum()))
print("OHLC violations on calendar dates:", len(violations))

print("\nViolations by source and adjustment:")
display(
    violations.groupby(["data_source", "adjustment"])
    .size()
    .rename("Violation_Count")
    .to_frame()
)

# ------------------------------------------------------------
# 6. Confirm the split-adjustment limitation
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("4. ADJUSTMENT COVERAGE")
print("=" * 65)

display(
    calendar_data.groupby(["data_source", "adjustment"])
    .agg(
        Rows=("isin", "size"),
        Companies=("isin", "nunique"),
    )
    .reset_index()
)

print("\nVALIDATION COMPLETE — original source_df is unchanged.")


Upload either nifty500_tri.csv or Indian_Stock_Algo_PrePIT_Backup.zip from your Mac.


Saving Indian_Stock_Algo_PrePIT_Backup.zip to Indian_Stock_Algo_PrePIT_Backup.zip
Benchmark imported: /content/nifty500_pit_data/nifty500_tri.csv
Benchmark trading dates: 1149
First date: 2015-01-01
Last date: 2026-10-09

1. TRADING-CALENDAR ALIGNMENT
PIT rows in validation period: 1499371
Rows on benchmark trading dates: 410942
Rows dated outside benchmark calendar: 1088429
Distinct off-calendar dates: 2236

Most frequent off-calendar dates:


,Rows,Companies,Source_Count
date,,,
2018-06-20,539,539,2
2022-05-02,539,539,1
2018-08-28,539,539,2
2018-08-27,539,539,2
2018-08-24,538,538,2
2018-07-26,538,538,2
2018-07-27,538,538,2
2018-08-23,538,538,2
2018-07-24,538,538,2



2. MEMBERSHIP COUNTS ON VALID TRADING DATES
Trading dates checked: 1149
Minimum constituents: 0
Median constituents: 492.0
Maximum constituents: 512

Dates with fewer than 450 flagged constituents:


,Constituents
2015-01-04,0
2015-01-10,0
2015-02-01,0
2015-02-07,0
2015-03-06,80
2015-03-07,0
2015-03-08,0
2015-04-02,0
2015-04-03,80
2015-04-05,0


...


3. OHLC PRICE QUALITY
Complete OHLC rows: 410942
OHLC violations on calendar dates: 0

Violations by source and adjustment:


,,Violation_Count
data_source,adjustment,



4. ADJUSTMENT COVERAGE


,data_source,adjustment,Rows,Companies
0,nse_bhavcopy_raw,raw_unadjusted,24151,97
1,yfinance,split_adjusted_no_dividend,386791,824



VALIDATION COMPLETE — original source_df is unchanged.


In [30]:
import os
import pandas as pd

tri_path = "/content/nifty500_pit_data/nifty500_tri.csv"

if not os.path.exists(tri_path):
    raise FileNotFoundError(f"Benchmark CSV not found: {tri_path}")

tri_raw = pd.read_csv(tri_path)

print("=" * 65)
print("1. FILE DETAILS")
print("=" * 65)
print("File size (MB):", round(os.path.getsize(tri_path) / (1024**2), 2))
print("CSV shape:", tri_raw.shape)
print("Columns:", tri_raw.columns.tolist())

print("\nFirst 8 raw rows:")
display(tri_raw.head(8))

print("\nLast 8 raw rows:")
display(tri_raw.tail(8))

date_cols = [
    c for c in tri_raw.columns
    if "date" in str(c).lower()
]

if not date_cols:
    raise ValueError(
        "No column containing 'date' found. "
        "Inspect the columns above and share the output."
    )

date_col = date_cols[0]
print("\nDate column selected:", date_col)

print("\n" + "=" * 65)
print("2. DATE-PARSING CHECK")
print("=" * 65)

for dayfirst in [False, True]:
    parsed = pd.to_datetime(
        tri_raw[date_col],
        errors="coerce",
        dayfirst=dayfirst
    ).dt.normalize()

    valid = parsed.dropna()
    unique_dates = pd.DatetimeIndex(valid.unique()).sort_values()
    weekdays = unique_dates.dayofweek

    print(f"\n dayfirst={dayfirst}")
    print("Unparseable date rows:", int(parsed.isna().sum()))
    print("Valid date rows:", len(valid))
    print("Unique dates:", len(unique_dates))

    if len(unique_dates):
        print("First date:", unique_dates.min().date())
        print("Last date:", unique_dates.max().date())
        print("Saturday/Sunday dates:", int((weekdays >= 5).sum()))

        print("\nUnique dates per year:")
        year_counts = pd.Series(
            unique_dates.year
        ).value_counts().sort_index()
        display(year_counts.to_frame("Unique_Dates"))

        print("Reference-date presence:")
        for d in [
            "2015-01-04",
            "2015-01-05",
            "2018-06-20",
            "2022-05-02",
            "2026-10-09",
            "2026-10-19"
        ]:
            print(
                d, "FOUND" if pd.Timestamp(d) in unique_dates
                else "MISSING"
            )

        print("\nFirst 15 parsed unique dates:")
        print([d.strftime("%Y-%m-%d") for d in unique_dates[:15]])

        print("\nLast 15 parsed unique dates:")
        print([d.strftime("%Y-%m-%d") for d in unique_dates[-15:]])

1. FILE DETAILS
File size (MB): 0.06
CSV shape: (2923, 2)
Columns: ['Date', 'NIFTY_500_TRI']

First 8 raw rows:


,Date,NIFTY_500_TRI
0,2015-01-01,9598.91
1,2015-01-02,9712.57
2,2015-01-05,9699.88
3,2015-01-06,9416.03
4,2015-01-07,9399.29
5,2015-01-08,9555.56
6,2015-01-09,9594.54
7,2015-01-12,9639.88



Last 8 raw rows:


,Date,NIFTY_500_TRI
2915,2026-10-08,34773.21
2916,2026-10-09,35176.15
2917,2026-10-12,35105.56
2918,2026-10-13,35535.73
2919,2026-10-14,35455.99
2920,2026-10-15,35685.80
2921,2026-10-16,35831.71
2922,2026-10-19,35464.13



Date column selected: Date

2. DATE-PARSING CHECK

 dayfirst=False
Unparseable date rows: 0
Valid date rows: 2923
Unique dates: 2923
First date: 2015-01-01
Last date: 2026-10-19
Saturday/Sunday dates: 11

Unique dates per year:


,Unique_Dates
2015,248
2016,247
2017,248
2018,246
2019,245
2020,252
2021,248
2022,248
2023,246
2024,249


Reference-date presence:
2015-01-04 MISSING
2015-01-05 FOUND
2018-06-20 FOUND
2022-05-02 FOUND
2026-10-09 FOUND
2026-10-19 FOUND

First 15 parsed unique dates:
['2015-01-01', '2015-01-02', '2015-01-05', '2015-01-06', '2015-01-07', '2015-01-08', '2015-01-09', '2015-01-12', '2015-01-13', '2015-01-14', '2015-01-15', '2015-01-16', '2015-01-19', '2015-01-20', '2015-01-21']

Last 15 parsed unique dates:
['2026-09-28', '2026-09-29', '2026-09-30', '2026-10-01', '2026-10-05', '2026-10-06', '2026-10-07', '2026-10-08', '2026-10-09', '2026-10-12', '2026-10-13', '2026-10-14', '2026-10-15', '2026-10-16', '2026-10-19']

 dayfirst=True
Unparseable date rows: 1761
Valid date rows: 1162
Unique dates: 1162
First date: 2015-01-01
Last date: 2026-12-10
Saturday/Sunday dates: 319

Unique dates per year:


,Unique_Dates
2015,97
2016,98
2017,98
2018,99
2019,95
2020,97
2021,100
2022,97
2023,97
2024,102


Reference-date presence:
2015-01-04 FOUND
2015-01-05 MISSING
2018-06-20 MISSING
2022-05-02 MISSING
2026-10-09 FOUND
2026-10-19 MISSING

First 15 parsed unique dates:
['2015-01-01', '2015-01-04', '2015-01-06', '2015-01-07', '2015-01-09', '2015-01-10', '2015-01-12', '2015-02-01', '2015-02-02', '2015-02-03', '2015-02-06', '2015-02-07', '2015-02-09', '2015-02-11', '2015-02-12']

Last 15 parsed unique dates:
['2026-10-08', '2026-10-09', '2026-11-02', '2026-11-03', '2026-11-05', '2026-11-06', '2026-11-08', '2026-11-09', '2026-12-01', '2026-12-02', '2026-12-03', '2026-12-05', '2026-12-06', '2026-12-08', '2026-12-10']


In [31]:
import os
import pandas as pd
import numpy as np

DATA_DIR = "/content/nifty500_pit_data"
TRI_PATH = os.path.join(DATA_DIR, "nifty500_tri.csv")
PIT_PATH = os.path.join(
    DATA_DIR, "nifty500_ohlcv_pit_membership_fixed.parquet"
)

# ---------------------------------------------------------
# 1. Load TRI and parse dates explicitly as YYYY-MM-DD
# ---------------------------------------------------------
tri = pd.read_csv(TRI_PATH)
tri["Date"] = pd.to_datetime(
    tri["Date"].astype(str).str.strip(),
    format="%Y-%m-%d",
    errors="coerce"
).dt.normalize()

tri = tri.dropna(subset=["Date"]).copy()

print("=" * 70)
print("1. BENCHMARK CALENDAR")
print("=" * 70)
print("Rows:", len(tri))
print("Unique dates:", tri["Date"].nunique())
print("First date:", tri["Date"].min().date())
print("Last date:", tri["Date"].max().date())

weekend_tri = tri[tri["Date"].dt.dayofweek >= 5]
print("\nWeekend-dated TRI rows:", len(weekend_tri))
if len(weekend_tri):
    display(weekend_tri.sort_values("Date"))

# Use unique weekdays as the trading calendar for this check.
# The weekend dates are excluded rather than treated as sessions.
calendar_dates = pd.DatetimeIndex(
    tri.loc[tri["Date"].dt.dayofweek < 5, "Date"].unique()
).sort_values()

calendar_set = set(calendar_dates)

print("\nWeekday benchmark dates:", len(calendar_dates))
print("First calendar date:", calendar_dates.min().date())
print("Last calendar date:", calendar_dates.max().date())

# ---------------------------------------------------------
# 2. Load the existing membership-fixed PIT dataset
# ---------------------------------------------------------
if "final_df" in globals() and isinstance(final_df, pd.DataFrame):
    pit = final_df.copy()
    print("\nPIT input: in-memory final_df")
elif os.path.exists(PIT_PATH):
    pit = pd.read_parquet(PIT_PATH)
    print("\nPIT input:", PIT_PATH)
else:
    raise FileNotFoundError(
        "Neither final_df nor the membership-fixed PIT parquet was found."
    )

# Locate the date column
date_col = next(
    (c for c in pit.columns if c.lower() == "date"),
    None
)
if date_col is None:
    raise ValueError(f"Could not find PIT date column. Columns: {pit.columns.tolist()}")

pit[date_col] = pd.to_datetime(
    pit[date_col], errors="coerce"
).dt.normalize()

# Identify company and source columns where available
company_col = next(
    (c for c in ["isin", "ISIN", "symbol", "ticker"] if c in pit.columns),
    None
)
source_col = next(
    (c for c in ["data_source", "source"] if c in pit.columns),
    None
)
adjustment_col = next(
    (c for c in ["adjustment", "price_adjustment"] if c in pit.columns),
    None
)

# Identify membership flag column
member_candidates = [
    c for c in pit.columns
    if ("member" in c.lower()) or ("nifty500" in c.lower())
]
member_col = next(
    (
        c for c in member_candidates
        if c.lower() in {
            "is_nifty500", "in_nifty500", "is_nifty500_member",
            "nifty500_member", "is_member", "membership_flag"
        }
    ),
    member_candidates[0] if member_candidates else None
)

print("\nPIT rows:", len(pit))
print("PIT date column:", date_col)
print("Company column:", company_col)
print("Membership column:", member_col)
print("Source column:", source_col)
print("Adjustment column:", adjustment_col)

# ---------------------------------------------------------
# 3. Corrected calendar alignment
# ---------------------------------------------------------
start_date = pd.Timestamp("2015-01-01")
end_date = calendar_dates.max()

validation = pit[
    pit[date_col].between(start_date, end_date)
].copy()

validation["_on_calendar"] = validation[date_col].isin(calendar_set)

on_calendar = validation[validation["_on_calendar"]]
off_calendar = validation[~validation["_on_calendar"]]

print("\n" + "=" * 70)
print("2. CORRECTED TRADING-CALENDAR ALIGNMENT")
print("=" * 70)
print("PIT rows in period:", len(validation))
print("Rows on weekday benchmark dates:", len(on_calendar))
print("Rows outside weekday benchmark calendar:", len(off_calendar))
print("Distinct off-calendar dates:", off_calendar[date_col].nunique())

if len(off_calendar):
    print("\nMost frequent off-calendar dates:")
    grouped = off_calendar.groupby(date_col).agg(
        Rows=(date_col, "size")
    )

    if company_col:
        grouped["Companies"] = off_calendar.groupby(date_col)[company_col].nunique()
    if source_col:
        grouped["Source_Count"] = off_calendar.groupby(date_col)[source_col].nunique()

    display(grouped.sort_values("Rows", ascending=False).head(20))

# ---------------------------------------------------------
# 4. Membership counts on corrected calendar dates
# ---------------------------------------------------------
print("\n" + "=" * 70)
print("3. MEMBERSHIP COUNTS ON CORRECTED CALENDAR")
print("=" * 70)

if member_col is None:
    print("No recognizable membership flag column was found.")
    print("Candidate columns containing 'member' or 'nifty500':",
          member_candidates)
else:
    member_values = pit[member_col]

    if pd.api.types.is_bool_dtype(member_values):
        pit["_member_flag"] = member_values.fillna(False)
    else:
        pit["_member_flag"] = (
            member_values.astype(str).str.strip().str.lower()
            .isin(["true", "1", "yes", "y"])
        )

    member_rows = pit[
        pit[date_col].between(start_date, end_date)
        & pit[date_col].isin(calendar_set)
        & pit["_member_flag"]
    ]

    if company_col:
        daily_counts = member_rows.groupby(date_col)[company_col].nunique()
    else:
        daily_counts = member_rows.groupby(date_col).size()

    daily_counts = daily_counts.reindex(calendar_dates, fill_value=0)

    print("Trading dates checked:", len(daily_counts))
    print("Minimum flagged constituents:", int(daily_counts.min()))
    print("Median flagged constituents:", float(daily_counts.median()))
    print("Maximum flagged constituents:", int(daily_counts.max()))

    print("\nDates with fewer than 450 flagged constituents:")
    low = daily_counts[daily_counts < 450]
    if len(low):
        display(low.head(30).rename("Flagged_Constituents").to_frame())
        print("Total dates below 450:", len(low))
    else:
        print("None")

    print("\nDates with more than 515 flagged constituents:")
    high = daily_counts[daily_counts > 515]
    if len(high):
        display(high.head(30).rename("Flagged_Constituents").to_frame())
    else:
        print("None")

# ---------------------------------------------------------
# 5. Source and adjustment coverage
# ---------------------------------------------------------
print("\n" + "=" * 70)
print("4. SOURCE AND ADJUSTMENT COVERAGE")
print("=" * 70)

group_cols = [
    c for c in [source_col, adjustment_col]
    if c is not None
]

if group_cols:
    coverage = (
        validation.groupby(group_cols)
        .agg(
            Rows=(date_col, "size"),
            Companies=(
                company_col if company_col else date_col, "nunique"
            )
        )
        .reset_index()
    )
    display(coverage)
else:
    print("Source/adjustment columns were not identified.")

print("\nVALIDATION CELL COMPLETE — no PIT rows or prices were modified.")

1. BENCHMARK CALENDAR
Rows: 2923
Unique dates: 2923
First date: 2015-01-01
Last date: 2026-10-19

Weekend-dated TRI rows: 11


,Date,NIFTY_500_TRI
40,2015-02-28,10244.87
451,2016-10-30,10838.16
1189,2019-10-27,14221.03
1257,2020-02-01,14421.13
1454,2020-11-14,15830.35
2195,2023-11-12,27170.84
2243,2024-01-20,30696.16
2272,2024-03-02,32018.73
2322,2024-05-18,33199.10
2500,2025-02-01,34132.24



Weekday benchmark dates: 2912
First calendar date: 2015-01-01
Last calendar date: 2026-10-19

PIT input: in-memory final_df

PIT rows: 2124958
PIT date column: date
Company column: isin
Membership column: is_nifty500_constituent
Source column: data_source
Adjustment column: adjustment

2. CORRECTED TRADING-CALENDAR ALIGNMENT
PIT rows in period: 1502368
Rows on weekday benchmark dates: 1495771
Rows outside weekday benchmark calendar: 6597
Distinct off-calendar dates: 162

Most frequent off-calendar dates:


,Rows,Companies,Source_Count
date,,,
2025-02-01,501,501,1
2020-11-14,495,495,1
2019-10-27,483,483,1
2026-01-15,455,455,2
2015-01-26,81,81,1
2015-02-17,81,81,1
2015-03-06,80,80,1
2015-04-03,80,80,1
2015-04-14,80,80,1



3. MEMBERSHIP COUNTS ON CORRECTED CALENDAR
Trading dates checked: 2912
Minimum flagged constituents: 0
Median flagged constituents: 497.0
Maximum flagged constituents: 526

Dates with fewer than 450 flagged constituents:


...

Total dates below 450: 35

Dates with more than 515 flagged constituents:


,Flagged_Constituents
2016-02-22,519
2016-12-28,526
2019-10-29,516
2022-05-02,522
2025-02-21,525
2026-02-23,519
2026-09-30,526



4. SOURCE AND ADJUSTMENT COVERAGE


,data_source,adjustment,Rows,Companies
0,nse_bhavcopy_raw,raw_unadjusted,88112,97
1,yfinance,split_adjusted_no_dividend,1414256,825



VALIDATION CELL COMPLETE — no PIT rows or prices were modified.


In [32]:
import os
import pandas as pd
import numpy as np

DATA_DIR = "/content/nifty500_pit_data"
TRI_PATH = os.path.join(DATA_DIR, "nifty500_tri.csv")
PIT_PATH = os.path.join(
    DATA_DIR, "nifty500_ohlcv_pit_membership_fixed.parquet"
)

# 1. Load and parse benchmark dates consistently
tri_diag = pd.read_csv(TRI_PATH)
tri_diag["Date"] = pd.to_datetime(
    tri_diag["Date"].astype(str).str.strip(),
    format="%Y-%m-%d",
    errors="coerce"
).dt.normalize()

benchmark_dates = pd.DatetimeIndex(
    tri_diag["Date"].dropna().unique()
).sort_values()
benchmark_set = set(benchmark_dates)

# 2. Use the existing PIT data; no downloads
if "final_df" in globals() and isinstance(final_df, pd.DataFrame):
    pit_diag = final_df.copy()
else:
    pit_diag = pd.read_parquet(PIT_PATH)

pit_diag["date"] = pd.to_datetime(
    pit_diag["date"], errors="coerce"
).dt.normalize()

company_col = "isin"
member_col = "is_nifty500_constituent"
source_col = "data_source"
adjustment_col = "adjustment"

if member_col not in pit_diag.columns:
    raise ValueError(f"Missing membership column: {member_col}")

pit_diag["_member_flag"] = (
    pit_diag[member_col].astype(str).str.strip().str.lower()
    .isin(["true", "1", "yes", "y"])
)

start_date = pd.Timestamp("2015-01-01")
end_date = benchmark_dates.max()

period = pit_diag[
    pit_diag["date"].between(start_date, end_date)
].copy()

period["_on_benchmark"] = period["date"].isin(benchmark_set)

print("=" * 68)
print("1. ALIGNMENT USING ALL BENCHMARK DATES")
print("=" * 68)
print("Benchmark dates:", len(benchmark_dates))
print("Benchmark first/last:",
      benchmark_dates.min().date(), "to", benchmark_dates.max().date())
print("PIT rows in period:", len(period))
print("Rows on benchmark dates:", int(period["_on_benchmark"].sum()))
print("Rows outside benchmark dates:", int((~period["_on_benchmark"]).sum()))
print("Distinct off-benchmark dates:",
      period.loc[~period["_on_benchmark"], "date"].nunique())

off = period[~period["_on_benchmark"]]

if len(off):
    off_summary = off.groupby("date").agg(
        Rows=("date", "size"),
        Companies=(company_col, "nunique"),
        Sources=(source_col, "nunique")
    ).sort_values("Rows", ascending=False)

    print("\nTop 20 dates outside the benchmark calendar:")
    display(off_summary.head(20))

# 3. Inspect specific dates and their neighbouring sessions
target_dates = [
    "2015-01-23",
    "2015-03-05",
    "2015-11-10",
    "2015-11-11",
    "2015-11-12",
    "2016-02-22",
    "2016-12-28",
    "2019-10-27",
    "2020-11-14",
    "2022-05-02",
    "2025-02-01",
    "2025-02-21",
    "2026-01-15",
    "2026-02-23",
    "2026-09-30",
]

print("\n" + "=" * 68)
print("2. RECORD-LEVEL DIAGNOSTICS")
print("=" * 68)

rows = []

for date_str in target_dates:
    d = pd.Timestamp(date_str)
    g = period[period["date"] == d]

    rows.append({
        "Date": date_str,
        "In_Benchmark": d in benchmark_set,
        "Total_Rows": len(g),
        "Unique_ISINs": g[company_col].nunique(),
        "Flagged_Member_ISINs": g.loc[
            g["_member_flag"], company_col
        ].nunique(),
        "Unflagged_ISINs": g.loc[
            ~g["_member_flag"], company_col
        ].nunique(),
        "Raw_NSE_Rows": (
            int((g[source_col] == "nse_bhavcopy_raw").sum())
            if source_col in g.columns else np.nan
        ),
        "Yahoo_Rows": (
            int((g[source_col] == "yfinance").sum())
            if source_col in g.columns else np.nan
        ),
    })

display(pd.DataFrame(rows))

# 4. Inspect actual records for the zero-flag date and the holiday date
print("\n" + "=" * 68)
print("3. SAMPLE RECORDS FOR CRITICAL DATES")
print("=" * 68)

for date_str in ["2015-11-11", "2026-01-15"]:
    d = pd.Timestamp(date_str)
    g = period[period["date"] == d].copy()

    print(f"\nDate: {date_str} | Rows: {len(g)}")

    if len(g):
        show_cols = [
            c for c in [
                "date", "isin", "symbol", "open", "high", "low",
                "close", "volume", "is_nifty500_constituent",
                "data_source", "adjustment"
            ] if c in g.columns
        ]
        display(g[show_cols].head(12))

print("\nDIAGNOSTIC COMPLETE — original PIT data unchanged.")

1. ALIGNMENT USING ALL BENCHMARK DATES
Benchmark dates: 2923
Benchmark first/last: 2015-01-01 to 2026-10-19
PIT rows in period: 1502368
Rows on benchmark dates: 1497250
Rows outside benchmark dates: 5118
Distinct off-benchmark dates: 159

Top 20 dates outside the benchmark calendar:


,Rows,Companies,Sources
date,,,
2026-01-15,455,455,2
2015-02-17,81,81,1
2015-01-26,81,81,1
2015-04-03,80,80,1
2015-04-14,80,80,1
2015-05-01,80,80,1
2015-03-06,80,80,1
2015-09-17,79,79,1
2015-09-25,78,78,1



2. RECORD-LEVEL DIAGNOSTICS


,Date,In_Benchmark,Total_Rows,Unique_ISINs,Flagged_Member_ISINs,Unflagged_ISINs,Raw_NSE_Rows,Yahoo_Rows
0,2015-01-23,True,425,425,420,5,0,425
1,2015-03-05,True,424,424,419,5,0,424
2,2015-11-10,True,508,508,503,5,77,431
3,2015-11-11,True,0,0,0,0,0,0
4,2015-11-12,False,77,77,77,0,77,0
5,2016-02-22,True,523,523,519,4,73,450
6,2016-12-28,True,532,532,526,6,56,476
7,2019-10-27,True,483,483,454,29,0,483
8,2020-11-14,True,495,495,474,21,0,495
9,2022-05-02,True,539,539,522,17,0,539



3. SAMPLE RECORDS FOR CRITICAL DATES

Date: 2015-11-11 | Rows: 0

Date: 2026-01-15 | Rows: 455


,date,isin,symbol,open,high,low,close,volume,is_nifty500_constituent,data_source,adjustment
11134,2026-01-15,INE002A01018,RELIANCE,1458.800049,1458.800049,1458.800049,1458.800049,0,True,yfinance,split_adjusted_no_dividend
13561,2026-01-15,INE002L01015,SJVN,78.709999,78.709999,78.709999,78.709999,0,True,yfinance,split_adjusted_no_dividend
15953,2026-01-15,INE002S01010,MGL,1048.400024,1048.400024,1048.400024,1048.400024,0,True,yfinance,split_adjusted_no_dividend
23104,2026-01-15,INE006I01046,ASTRAL,1462.599976,1462.599976,1462.599976,1462.599976,0,True,yfinance,split_adjusted_no_dividend
27253,2026-01-15,INE007A01025,CRISIL,4767.200195,4767.200195,4767.200195,4767.200195,0,True,yfinance,split_adjusted_no_dividend
32917,2026-01-15,INE008A01015,IDBI,104.849998,104.849998,104.849998,104.849998,0,True,yfinance,split_adjusted_no_dividend
38637,2026-01-15,INE009A01021,INFY,1599.800049,1599.800049,1599.800049,1599.800049,0,True,yfinance,split_adjusted_no_dividend
39544,2026-01-15,INE00E101023,BIKAJI,699.250000,699.250000,699.250000,699.250000,0,True,yfinance,split_adjusted_no_dividend
40104,2026-01-15,INE00H001014,SWIGGY,347.049988,347.049988,347.049988,347.049988,0,True,yfinance,split_adjusted_no_dividend
44275,2026-01-15,INE00R701025,DALBHARAT,2177.300049,2177.300049,2177.300049,2177.300049,0,True,yfinance,split_adjusted_no_dividend



DIAGNOSTIC COMPLETE — original PIT data unchanged.


In [33]:
import pandas as pd
import numpy as np

# ---------------------------------------------------------
# 1. Use the existing benchmark and PIT data
# ---------------------------------------------------------
tri_check = pd.read_csv(
    "/content/nifty500_pit_data/nifty500_tri.csv"
)
tri_check["Date"] = pd.to_datetime(
    tri_check["Date"].astype(str).str.strip(),
    format="%Y-%m-%d",
    errors="coerce"
).dt.normalize()

benchmark_dates = pd.DatetimeIndex(
    tri_check["Date"].dropna().unique()
).sort_values()
benchmark_set = set(benchmark_dates)

if "final_df" in globals() and isinstance(final_df, pd.DataFrame):
    p = final_df.copy()
else:
    p = pd.read_parquet(
        "/content/nifty500_pit_data/"
        "nifty500_ohlcv_pit_membership_fixed.parquet"
    )

p["date"] = pd.to_datetime(p["date"], errors="coerce").dt.normalize()

start_date = pd.Timestamp("2015-01-01")
end_date = benchmark_dates.max()

p = p[p["date"].between(start_date, end_date)].copy()
p["_on_benchmark"] = p["date"].isin(benchmark_set)

source_col = "data_source" if "data_source" in p.columns else None
adjustment_col = "adjustment" if "adjustment" in p.columns else None
isin_col = "isin" if "isin" in p.columns else None

# ---------------------------------------------------------
# 2. Off-calendar records by source
# ---------------------------------------------------------
print("=" * 68)
print("1. OFF-CALENDAR RECORDS BY SOURCE")
print("=" * 68)

off = p[~p["_on_benchmark"]].copy()
print("Off-calendar rows:", len(off))
print("Distinct dates:", off["date"].nunique())

group_cols = [c for c in [source_col, adjustment_col] if c]
if group_cols:
    summary = off.groupby(group_cols).agg(
        Rows=("date", "size"),
        Distinct_Dates=("date", "nunique")
    ).reset_index()
    display(summary)

if source_col and isin_col and len(off):
    pivot = off.pivot_table(
        index="date",
        columns=source_col,
        values=isin_col,
        aggfunc="nunique",
        fill_value=0
    )
    pivot["Total_Companies"] = off.groupby("date")[isin_col].nunique()
    pivot = pivot.sort_values("Total_Companies", ascending=False)

    print("\nTop 25 off-calendar dates: distinct companies by source")
    display(pivot.head(25))

# ---------------------------------------------------------
# 3. Check for zero-volume, flat OHLC records
# ---------------------------------------------------------
print("\n" + "=" * 68)
print("2. ZERO-VOLUME, FLAT-OHLC RECORDS")
print("=" * 68)

ohlc = ["open", "high", "low", "close"]
if all(c in p.columns for c in ohlc) and "volume" in p.columns:
    q = p.copy()

    for c in ohlc + ["volume"]:
        q[c] = pd.to_numeric(q[c], errors="coerce")

    q["_flat_ohlc"] = (
        q[ohlc].notna().all(axis=1)
        & (q[ohlc].max(axis=1) - q[ohlc].min(axis=1) <= 1e-6)
    )

    q["_zero_volume"] = q["volume"].eq(0)
    q["_flat_and_zero_volume"] = q["_flat_ohlc"] & q["_zero_volume"]

    print("All zero-volume rows:", int(q["_zero_volume"].sum()))
    print("All flat-OHLC rows:", int(q["_flat_ohlc"].sum()))
    print(
        "Flat-OHLC AND zero-volume rows:",
        int(q["_flat_and_zero_volume"].sum())
    )

    print("\nFlat, zero-volume rows outside the benchmark calendar:")
    display(
        q.loc[
            (~q["_on_benchmark"]) & q["_flat_and_zero_volume"],
            [
                c for c in [
                    "date", "isin", "symbol", "open", "high",
                    "low", "close", "volume", "data_source"
                ] if c in q.columns
            ]
        ].groupby(
            ["date", "data_source"] if "data_source" in q.columns
            else ["date"],
            dropna=False
        ).agg(
            Rows=("date", "size"),
            Companies=("isin", "nunique") if "isin" in q.columns
            else ("date", "size")
        ).sort_values("Rows", ascending=False).head(25)
    )

# ---------------------------------------------------------
# 4. Compare intervals with actual price coverage
# ---------------------------------------------------------
print("\n" + "=" * 68)
print("3. HISTORICAL MEMBERSHIP INTERVAL CHECK")
print("=" * 68)

if "intervals_df" not in globals() or not isinstance(
    intervals_df, pd.DataFrame
):
    print("intervals_df is not available in this runtime.")
else:
    iv = intervals_df.copy()
    print("Intervals shape:", iv.shape)
    print("Interval columns:", iv.columns.tolist())

    lower_to_original = {
        str(c).strip().lower(): c for c in iv.columns
    }

    def pick_column(aliases, contains_any=()):
        for alias in aliases:
            if alias in lower_to_original:
                return lower_to_original[alias]
        for c in iv.columns:
            lc = str(c).strip().lower()
            if any(token in lc for token in contains_any):
                return c
        return None

    iv_isin = pick_column(
        ["isin", "security_isin", "company_isin"],
        ("isin",)
    )
    iv_start = pick_column(
        [
            "start_date", "valid_from", "effective_from",
            "membership_start", "from_date", "date_from"
        ],
        ("start_date", "valid_from", "effective_from",
         "membership_start", "date_from")
    )
    iv_end = pick_column(
        [
            "end_date", "valid_to", "effective_to",
            "membership_end", "to_date", "date_to"
        ],
        ("end_date", "valid_to", "effective_to",
         "membership_end", "date_to")
    )

    print("ISIN column:", iv_isin)
    print("Start column:", iv_start)
    print("End column:", iv_end)

    if not all([iv_isin, iv_start, iv_end, isin_col]):
        print(
            "Could not automatically identify all interval columns. "
            "Share the column list printed above."
        )
    else:
        iv["_start"] = pd.to_datetime(
            iv[iv_start], errors="coerce"
        ).dt.normalize()
        iv["_end"] = pd.to_datetime(
            iv[iv_end], errors="coerce"
        ).dt.normalize()
        iv["_isin"] = iv[iv_isin].astype(str).str.strip()

        if "dataset_end" in globals():
            interval_dataset_end = pd.to_datetime(
                dataset_end, errors="coerce"
            )
        else:
            interval_dataset_end = p["date"].max()

        # Missing end dates represent ongoing intervals.
        # Membership is inclusive at both endpoints.
        check_dates = [
            "2015-01-23",
            "2015-03-05",
            "2016-12-28",
            "2022-05-02",
            "2025-02-21",
            "2026-02-23",
            "2026-09-30",
        ]

        out = []
        for ds in check_dates:
            d = pd.Timestamp(ds)
            active = iv[
                iv["_start"].notna()
                & (iv["_start"] <= d)
                & (iv["_end"].isna() | (iv["_end"] >= d))
            ]

            expected_isins = set(active["_isin"].unique())

            day = p[p["date"] == d].copy()
            priced = day[
                pd.to_numeric(day["close"], errors="coerce").notna()
            ] if "close" in day.columns else day.iloc[0:0]

            observed_isins = set(
                day[isin_col].astype(str).str.strip().unique()
            )
            priced_isins = set(
                priced[isin_col].astype(str).str.strip().unique()
            )

            out.append({
                "Date": ds,
                "Expected_Interval_ISINs": len(expected_isins),
                "Observed_Price_ISINs": len(priced_isins),
                "Expected_With_Price": len(expected_isins & priced_isins),
                "Expected_Missing_Price": len(expected_isins - priced_isins),
                "Price_ISINs_Not_In_Intervals": len(
                    priced_isins - expected_isins
                ),
            })

        display(pd.DataFrame(out))

print("\nDIAGNOSTICS COMPLETE — no source data or saved files modified.")

1. OFF-CALENDAR RECORDS BY SOURCE
Off-calendar rows: 5118
Distinct dates: 159


,data_source,adjustment,Rows,Distinct_Dates
0,nse_bhavcopy_raw,raw_unadjusted,4665,159
1,yfinance,split_adjusted_no_dividend,453,1



Top 25 off-calendar dates: distinct companies by source


data_source,nse_bhavcopy_raw,yfinance,Total_Companies
date,,,
2026-01-15,2,453,455
2015-02-17,81,0,81
2015-01-26,81,0,81
2015-04-03,80,0,80
2015-04-14,80,0,80
2015-05-01,80,0,80
2015-03-06,80,0,80
2015-09-17,79,0,79
2015-09-25,78,0,78



2. ZERO-VOLUME, FLAT-OHLC RECORDS
All zero-volume rows: 3020
All flat-OHLC rows: 5455
Flat-OHLC AND zero-volume rows: 3010

Flat, zero-volume rows outside the benchmark calendar:


,,Rows,Companies
date,data_source,,
2026-01-15,yfinance,453,453



3. HISTORICAL MEMBERSHIP INTERVAL CHECK
Intervals shape: (1292, 7)
Interval columns: ['isin', 'symbol', 'company_name', 'start_date', 'start_confidence', 'end_date', 'end_confidence']
ISIN column: isin
Start column: start_date
End column: end_date


,Date,Expected_Interval_ISINs,Observed_Price_ISINs,Expected_With_Price,Expected_Missing_Price,Price_ISINs_Not_In_Intervals
0,2015-01-23,560,425,418,142,7
1,2015-03-05,559,424,417,142,7
2,2016-12-28,623,532,524,99,8
3,2022-05-02,594,539,522,72,17
4,2025-02-21,554,535,525,29,10
5,2026-02-23,523,525,519,4,6
6,2026-09-30,528,526,526,2,0



DIAGNOSTICS COMPLETE — no source data or saved files modified.


In [34]:
import pandas as pd
import numpy as np

# Use the existing objects; no downloads and no edits to source data.
if "final_df" not in globals() or "intervals_df" not in globals():
    raise RuntimeError(
        "Please keep the existing PIT notebook runtime open so "
        "final_df and intervals_df remain available."
    )

p = final_df.copy()
iv = intervals_df.copy()

p["date"] = pd.to_datetime(p["date"], errors="coerce").dt.normalize()
iv["start_date"] = pd.to_datetime(
    iv["start_date"], errors="coerce"
).dt.normalize()
iv["end_date"] = pd.to_datetime(
    iv["end_date"], errors="coerce"
).dt.normalize()

p["_member_flag"] = (
    p["is_nifty500_constituent"].astype(str).str.strip().str.lower()
    .isin(["true", "1", "yes", "y"])
)

dataset_end = p["date"].max()

# Treat missing end dates as continuing intervals for inspection only.
iv["_effective_end"] = iv["end_date"].fillna(dataset_end)

print("=" * 68)
print("1. MEMBERSHIP CONFIDENCE VALUES")
print("=" * 68)

for col in ["start_confidence", "end_confidence"]:
    print(f"\n{col} value counts:")
    display(
        iv[col].fillna("<missing>")
        .value_counts(dropna=False)
        .rename_axis(col)
        .to_frame("Rows")
    )

print("\n" + "=" * 68)
print("2. INTERVAL COUNTS VS ACTUAL FLAGS")
print("=" * 68)

check_dates = [
    "2015-01-23",
    "2015-03-05",
    "2016-12-28",
    "2022-05-02",
    "2025-02-21",
    "2026-02-23",
    "2026-09-30",
]

results = []

for date_str in check_dates:
    d = pd.Timestamp(date_str)

    active_intervals = iv[
        iv["start_date"].notna()
        & (iv["start_date"] <= d)
        & (iv["_effective_end"] >= d)
    ].copy()

    interval_isins = set(active_intervals["isin"].astype(str).str.strip())

    day = p[p["date"] == d].copy()
    price_isins = set(day["isin"].astype(str).str.strip())
    flagged_isins = set(
        day.loc[day["_member_flag"], "isin"].astype(str).str.strip()
    )

    results.append({
        "Date": date_str,
        "Active_Interval_ISINs": len(interval_isins),
        "Priced_ISINs": len(price_isins),
        "Flagged_ISINs": len(flagged_isins),
        "Flagged_Also_In_Intervals": len(flagged_isins & interval_isins),
        "Priced_In_Intervals_Not_Flagged": len(
            (price_isins & interval_isins) - flagged_isins
        ),
        "Flagged_Not_In_Intervals": len(flagged_isins - interval_isins),
        "Priced_Not_In_Intervals": len(price_isins - interval_isins),
    })

display(pd.DataFrame(results))

print("\n" + "=" * 68)
print("3. CONFIDENCE BREAKDOWN FOR JAN 2015 AND SEP 2026")
print("=" * 68)

for date_str in ["2015-01-23", "2026-09-30"]:
    d = pd.Timestamp(date_str)

    active = iv[
        iv["start_date"].notna()
        & (iv["start_date"] <= d)
        & (iv["_effective_end"] >= d)
    ].copy()

    print(f"\nActive intervals on {date_str}: {len(active)}")

    confidence_summary = (
        active.groupby(
            ["start_confidence", "end_confidence"],
            dropna=False
        )
        .agg(
            Interval_Rows=("isin", "size"),
            Unique_ISINs=("isin", "nunique")
        )
        .sort_values("Unique_ISINs", ascending=False)
        .reset_index()
    )
    display(confidence_summary)

    print("Example active intervals:")
    display(
        active[
            [
                c for c in [
                    "isin", "symbol", "company_name",
                    "start_date", "start_confidence",
                    "end_date", "end_confidence"
                ] if c in active.columns
            ]
        ].sort_values(["start_date", "symbol"]).head(20)
    )

print("\nDIAGNOSTIC COMPLETE — no source data or saved files modified.")

1. MEMBERSHIP CONFIDENCE VALUES

start_confidence value counts:


,Rows
start_confidence,
left_censored,500
interpolated_long_gap,306
press_release_exact,289
interpolated_snapshot_adjacent,58
isin_migration_from_INE122H01019,1
...,...
isin_migration_from_INE814H01011,1
isin_migration_from_INE258A01016,1
isin_migration_from_INE531F01015,1



end_confidence value counts:


,Rows
end_confidence,
still_active,501
press_release_exact,325
interpolated_long_gap,275
interpolated_snapshot_adjacent,52
isin_migration_to_INE389H01022,1
...,...
isin_migration_to_INE745G01043,1
isin_migration_to_INE258A01024,1
isin_migration_to_INE531F01023,1



2. INTERVAL COUNTS VS ACTUAL FLAGS


...


Active intervals on 2026-09-30: 528


,start_confidence,end_confidence,Interval_Rows,Unique_ISINs
0,press_release_exact,still_active,152,152
1,interpolated_long_gap,still_active,106,106
2,left_censored,still_active,105,105
3,interpolated_snapshot_adjacent,still_active,44,44
4,interpolated_long_gap,interpolated_snapshot_adjacent,9,9
...,...,...,...,...
98,isin_migration_from_INE885A01024,still_active,1,1
99,isin_migration_from_INE967H01017,still_active,1,1
100,isin_migration_from_INE947Q01010,still_active,1,1
101,isin_migration_from_INE935N01012,still_active,1,1


Example active intervals:


,isin,symbol,company_name,start_date,start_confidence,end_date,end_confidence
778,INE470A01017,3MINDIA,3M India Ltd.,2010-01-02,left_censored,2026-09-30,interpolated_snapshot_adjacent
793,INE769A01020,AARTIIND,Aarti Industries Ltd.,2010-01-02,left_censored,NaT,still_active
791,INE117A01022,ABB,ABB India Ltd.,2010-01-02,left_censored,NaT,still_active
792,INE012A01025,ACC,ACC Ltd.,2010-01-02,left_censored,NaT,still_active
794,INE079A01024,AMBUJACEM,Ambuja Cements Ltd.,2010-01-02,left_censored,NaT,still_active
918,INE437A01024,APOLLOHOSP,Apollo Hospitals Enterprise Ltd.,2010-01-02,isin_migration_from_INE437A01016,NaT,still_active
795,INE438A01022,APOLLOTYRE,Apollo Tyres Ltd.,2010-01-02,left_censored,NaT,still_active
901,INE885A01032,ARE&M,Amara Raja Energy & Mobility Ltd.,2010-01-02,isin_migration_from_INE885A01024,NaT,still_active
796,INE208A01029,ASHOKLEY,Ashok Leyland Ltd.,2010-01-02,left_censored,NaT,still_active
908,INE021A01026,ASIANPAINT,Asian Paints Ltd.,2010-01-02,isin_migration_from_INE021A01018,NaT,still_active



DIAGNOSTIC COMPLETE — no source data or saved files modified.


In [35]:
import pandas as pd
import numpy as np

iv = intervals_df.copy()
p = final_df.copy()

iv["start_date"] = pd.to_datetime(
    iv["start_date"], errors="coerce"
).dt.normalize()
iv["end_date"] = pd.to_datetime(
    iv["end_date"], errors="coerce"
).dt.normalize()

p["date"] = pd.to_datetime(
    p["date"], errors="coerce"
).dt.normalize()

dataset_end = p["date"].max()
iv["_effective_end"] = iv["end_date"].fillna(dataset_end)

check_dates = [
    "2015-01-23",
    "2016-12-28",
    "2022-05-02",
    "2025-02-21",
    "2026-02-23",
    "2026-09-30",
]

print("=" * 70)
print("1. ACTIVE INTERVALS: ROWS VS DISTINCT COMPANIES")
print("=" * 70)

summary_rows = []

for date_str in check_dates:
    d = pd.Timestamp(date_str)

    active = iv[
        iv["start_date"].notna()
        & (iv["start_date"] <= d)
        & (iv["_effective_end"] >= d)
    ].copy()

    day = p[p["date"] == d].copy()

    summary_rows.append({
        "Date": date_str,
        "Active_Interval_Rows": len(active),
        "Active_Unique_ISINs": active["isin"].nunique(),
        "Active_Unique_Symbols": active["symbol"].nunique(),
        "Active_Company_Names": active["company_name"].nunique(),
        "Priced_Unique_ISINs": day["isin"].nunique(),
        "Priced_Unique_Symbols": day["symbol"].nunique(),
        "Flagged_Unique_ISINs": day.loc[
            day["is_nifty500_constituent"].astype(str).str.lower()
            .isin(["true", "1", "yes"]),
            "isin"
        ].nunique(),
    })

display(pd.DataFrame(summary_rows))

print("\n" + "=" * 70)
print("2. DUPLICATE ACTIVE SYMBOLS")
print("=" * 70)

for date_str in ["2015-01-23", "2022-05-02", "2026-09-30"]:
    d = pd.Timestamp(date_str)

    active = iv[
        iv["start_date"].notna()
        & (iv["start_date"] <= d)
        & (iv["_effective_end"] >= d)
    ].copy()

    by_symbol = (
        active.groupby("symbol", dropna=False)
        .agg(
            Active_ISINs=("isin", "nunique"),
            Interval_Rows=("isin", "size"),
            Company_Names=("company_name", "nunique"),
        )
        .reset_index()
    )

    duplicates = by_symbol[by_symbol["Active_ISINs"] > 1]

    print(f"\n{date_str}: {len(duplicates)} symbols with multiple active ISINs")
    if len(duplicates):
        display(
            duplicates.sort_values(
                ["Active_ISINs", "Interval_Rows"], ascending=False
            ).head(25)
        )

print("\n" + "=" * 70)
print("3. CONFIDENCE QUALITY OVER TIME")
print("=" * 70)

for date_str in ["2015-01-23", "2022-05-02", "2026-09-30"]:
    d = pd.Timestamp(date_str)

    active = iv[
        iv["start_date"].notna()
        & (iv["start_date"] <= d)
        & (iv["_effective_end"] >= d)
    ]

    print(f"\nActive interval start confidence on {date_str}:")
    display(
        active["start_confidence"]
        .value_counts(dropna=False)
        .rename_axis("Start_Confidence")
        .to_frame("Intervals")
    )

print("\nAUDIT COMPLETE — no source data or saved files modified.")

1. ACTIVE INTERVALS: ROWS VS DISTINCT COMPANIES


,Date,Active_Interval_Rows,Active_Unique_ISINs,Active_Unique_Symbols,Active_Company_Names,Priced_Unique_ISINs,Priced_Unique_Symbols,Flagged_Unique_ISINs
0,2015-01-23,560,560,504,505,425,425,420
1,2016-12-28,623,623,569,569,532,532,526
2,2022-05-02,594,594,549,550,539,539,522
3,2025-02-21,554,554,530,530,535,535,525
4,2026-02-23,523,523,521,521,525,525,519
5,2026-09-30,528,528,528,528,526,526,526



2. DUPLICATE ACTIVE SYMBOLS

2015-01-23: 55 symbols with multiple active ISINs


,symbol,Active_ISINs,Interval_Rows,Company_Names
45,BAJFINANCE,3,3,1
38,AXISBANK,2,2,1
42,BAJAJFINSV,2,2,1
50,BANKBARODA,2,2,1
53,BATAINDIA,2,2,1
55,BEL,2,2,1
56,BEML,2,2,1
57,BERGEPAINT,2,2,1
74,BRITANNIA,2,2,1
77,CANBK,2,2,1



2022-05-02: 44 symbols with multiple active ISINs


...


Active interval start confidence on 2022-05-02:


,Intervals
Start_Confidence,
interpolated_long_gap,179
left_censored,163
press_release_exact,133
interpolated_snapshot_adjacent,4
isin_migration_from_INE571A01012,1
...,...
isin_migration_from_INE745G01035,1
isin_migration_from_INE814H01011,1
isin_migration_from_INE258A01016,1



Active interval start confidence on 2026-09-30:


,Intervals
Start_Confidence,
press_release_exact,160
interpolated_long_gap,115
left_censored,111
interpolated_snapshot_adjacent,47
isin_migration_from_INE806T01012,1
...,...
isin_migration_from_INE814H01011,1
isin_migration_from_INE258A01016,1
isin_migration_from_INE531F01015,1



AUDIT COMPLETE — no source data or saved files modified.


In [36]:
import pandas as pd
import numpy as np

iv = intervals_df.copy()
p = final_df.copy()

iv["start_date"] = pd.to_datetime(
    iv["start_date"], errors="coerce"
).dt.normalize()
iv["end_date"] = pd.to_datetime(
    iv["end_date"], errors="coerce"
).dt.normalize()

p["date"] = pd.to_datetime(
    p["date"], errors="coerce"
).dt.normalize()

p["_member_flag"] = (
    p["is_nifty500_constituent"].astype(str).str.strip().str.lower()
    .isin(["true", "1", "yes", "y"])
)

dataset_end = p["date"].max()
iv["_effective_end"] = iv["end_date"].fillna(dataset_end)

def active_intervals_on(d):
    return iv[
        iv["start_date"].notna()
        & (iv["start_date"] <= d)
        & (iv["_effective_end"] >= d)
    ].copy()

def prices_on(d):
    return p[p["date"] == d].copy()

# ---------------------------------------------------------
# 1. Daily counts around the boundary
# ---------------------------------------------------------
print("=" * 70)
print("1. DAILY MEMBERSHIP AND PRICE COUNTS")
print("=" * 70)

daily_rows = []

for d in pd.date_range("2026-09-20", "2026-10-23", freq="D"):
    active = active_intervals_on(d)
    prices = prices_on(d)

    daily_rows.append({
        "Date": d.date(),
        "Active_Interval_Rows": len(active),
        "Active_Interval_ISINs": active["isin"].nunique(),
        "Active_Interval_Symbols": active["symbol"].nunique(),
        "Price_Rows": len(prices),
        "Priced_ISINs": prices["isin"].nunique(),
        "Flagged_Member_ISINs": prices.loc[
            prices["_member_flag"], "isin"
        ].nunique(),
    })

daily = pd.DataFrame(daily_rows)
display(daily)

# ---------------------------------------------------------
# 2. Compare active symbol sets on Sep 30 and Oct 1
# ---------------------------------------------------------
print("\n" + "=" * 70)
print("2. SYMBOL SET DIFFERENCES: SEP 30 VS OCT 1")
print("=" * 70)

sep30 = pd.Timestamp("2026-09-30")
oct1 = pd.Timestamp("2026-10-01")

active_sep30 = active_intervals_on(sep30)
active_oct1 = active_intervals_on(oct1)

symbols_sep30 = set(active_sep30["symbol"].dropna().astype(str))
symbols_oct1 = set(active_oct1["symbol"].dropna().astype(str))

left_after_sep30 = sorted(symbols_sep30 - symbols_oct1)
entered_on_oct1 = sorted(symbols_oct1 - symbols_sep30)

print("Active symbols on Sep 30:", len(symbols_sep30))
print("Active symbols on Oct 1:", len(symbols_oct1))
print("Symbols active Sep 30 but not Oct 1:", len(left_after_sep30))
print(left_after_sep30)
print("\nSymbols active Oct 1 but not Sep 30:", len(entered_on_oct1))
print(entered_on_oct1)

# ---------------------------------------------------------
# 3. All interval start/end boundaries around this period
# ---------------------------------------------------------
print("\n" + "=" * 70)
print("3. INTERVAL BOUNDARIES: SEP 15 TO OCT 15")
print("=" * 70)

window_start = pd.Timestamp("2026-09-15")
window_end = pd.Timestamp("2026-10-15")

boundary_mask = (
    iv["start_date"].between(window_start, window_end)
    | iv["end_date"].between(window_start, window_end)
)

boundary_cols = [
    c for c in [
        "isin", "symbol", "company_name",
        "start_date", "start_confidence",
        "end_date", "end_confidence"
    ] if c in iv.columns
]

boundaries = iv.loc[boundary_mask, boundary_cols].copy()
boundaries["_start_sort"] = boundaries["start_date"].fillna(
    pd.Timestamp("1900-01-01")
)
boundaries = boundaries.sort_values(
    ["_start_sort", "symbol"]
).drop(columns="_start_sort")

print("Interval rows with start/end boundaries in this window:", len(boundaries))
display(boundaries)

# ---------------------------------------------------------
# 4. Show the records ending on Sep 30 and starting on Oct 1
# ---------------------------------------------------------
print("\n" + "=" * 70)
print("4. EXACT SEP 30 / OCT 1 BOUNDARIES")
print("=" * 70)

ending_sep30 = iv[iv["end_date"] == sep30]
starting_oct1 = iv[iv["start_date"] == oct1]

print("Intervals ending Sep 30:", len(ending_sep30))
display(ending_sep30[boundary_cols])

print("\nIntervals starting Oct 1:", len(starting_oct1))
display(starting_oct1[boundary_cols])

print("\nAUDIT COMPLETE — no source data or saved files modified.")

1. DAILY MEMBERSHIP AND PRICE COUNTS


,Date,Active_Interval_Rows,Active_Interval_ISINs,Active_Interval_Symbols,Price_Rows,Priced_ISINs,Flagged_Member_ISINs
0,2026-09-20,501,501,501,0,0,0
1,2026-09-21,501,501,501,506,506,499
2,2026-09-22,501,501,501,506,506,499
3,2026-09-23,501,501,501,506,506,499
4,2026-09-24,501,501,501,506,506,499
5,2026-09-25,501,501,501,506,506,499
6,2026-09-26,501,501,501,0,0,0
7,2026-09-27,501,501,501,0,0,0
8,2026-09-28,501,501,501,506,506,499
9,2026-09-29,501,501,501,507,507,499



2. SYMBOL SET DIFFERENCES: SEP 30 VS OCT 1
Active symbols on Sep 30: 528
Active symbols on Oct 1: 501
Symbols active Sep 30 but not Oct 1: 27
['3MINDIA', 'ABFRL', 'ABLBL', 'BAYERCROP', 'BBTC', 'BIKAJI', 'BLUEDART', 'BLUEJET', 'CHALET', 'DCMSHRIRAM', 'ERIS', 'GODIGIT', 'INDGN', 'JSWDULUX', 'JUBLPHARMA', 'LATENTVIEW', 'MAPMYINDIA', 'NEWGEN', 'NIVABUPA', 'PFIZER', 'SAPPHIRE', 'SAREGAMA', 'SBFC', 'SONATSOFTW', 'SPLPETRO', 'THELEELA', 'TRAVELFOOD']

Symbols active Oct 1 but not Sep 30: 0
[]

3. INTERVAL BOUNDARIES: SEP 15 TO OCT 15
Interval rows with start/end boundaries in this window: 54


,isin,symbol,company_name,start_date,start_confidence,end_date,end_confidence
778,INE470A01017,3MINDIA,3M India Ltd.,2010-01-02,left_censored,2026-09-30,interpolated_snapshot_adjacent
784,INE233B01017,BLUEDART,Blue Dart Express Ltd.,2010-01-02,left_censored,2026-09-30,interpolated_snapshot_adjacent
783,INE499A01024,DCMSHRIRAM,DCM Shriram Ltd.,2010-01-02,left_censored,2026-09-30,interpolated_snapshot_adjacent
789,INE700A01033,JUBLPHARMA,Jubilant Pharmova Ltd.,2010-01-02,left_censored,2026-09-30,interpolated_snapshot_adjacent
786,INE182A01018,PFIZER,Pfizer Ltd.,2010-01-02,left_censored,2026-09-30,interpolated_snapshot_adjacent
766,INE269A01021,SONATSOFTW,Sonata Software Ltd.,2010-01-02,left_censored,2026-09-30,interpolated_snapshot_adjacent
790,INE050A01025,BBTC,Bombay Burmah Trading Corporation Ltd.,2012-01-12,interpolated_long_gap,2026-09-30,interpolated_snapshot_adjacent
765,INE647O01011,ABFRL,Aditya Birla Fashion and Retail Ltd.,2016-02-22,press_release_exact,2026-09-30,interpolated_snapshot_adjacent
782,INE406M01024,ERIS,Eris Lifesciences Ltd.,2018-03-06,press_release_exact,2026-09-30,interpolated_snapshot_adjacent
788,INE462A01022,BAYERCROP,Bayer Cropscience Ltd.,2019-08-28,press_release_exact,2026-09-30,interpolated_snapshot_adjacent



4. EXACT SEP 30 / OCT 1 BOUNDARIES
Intervals ending Sep 30: 27


,isin,symbol,company_name,start_date,start_confidence,end_date,end_confidence
764,INE103V01028,TRAVELFOOD,Travel Food Services Ltd.,2026-03-05,interpolated_snapshot_adjacent,2026-09-30,interpolated_snapshot_adjacent
765,INE647O01011,ABFRL,Aditya Birla Fashion and Retail Ltd.,2016-02-22,press_release_exact,2026-09-30,interpolated_snapshot_adjacent
766,INE269A01021,SONATSOFTW,Sonata Software Ltd.,2010-01-02,left_censored,2026-09-30,interpolated_snapshot_adjacent
767,INE133A01011,JSWDULUX,JSW Dulux Ltd.,2025-08-22,press_release_exact,2026-09-30,interpolated_snapshot_adjacent
768,INE995S01015,NIVABUPA,Niva Bupa Health Insurance Company Ltd.,2025-02-21,press_release_exact,2026-09-30,interpolated_snapshot_adjacent
769,INE14LE01019,ABLBL,Aditya Birla Lifestyle Brands Ltd.,2025-08-22,press_release_exact,2026-09-30,interpolated_snapshot_adjacent
770,INE0KBH01020,BLUEJET,Blue Jet Healthcare Ltd.,2025-08-22,press_release_exact,2026-09-30,interpolated_snapshot_adjacent
771,INE0BV301023,MAPMYINDIA,C.E. Info Systems Ltd.,2022-05-02,interpolated_long_gap,2026-09-30,interpolated_snapshot_adjacent
772,INE619B01017,NEWGEN,Newgen Software Technologies Ltd.,2024-10-11,interpolated_long_gap,2026-09-30,interpolated_snapshot_adjacent
773,INE00E101023,BIKAJI,Bikaji Foods International Ltd.,2023-02-17,press_release_exact,2026-09-30,interpolated_snapshot_adjacent



Intervals starting Oct 1: 0


,isin,symbol,company_name,start_date,start_confidence,end_date,end_confidence



AUDIT COMPLETE — no source data or saved files modified.


In [37]:
import os
import pandas as pd

DATA_DIR = "/content/nifty500_pit_data"
EFFECTIVE_DATE = pd.Timestamp("2026-09-30")
PREVIOUS_DATE = EFFECTIVE_DATE - pd.Timedelta(days=1)

# Official Nifty 500 constituent changes effective 30 Sep 2026.
EXIT_SYMBOLS = {
    "3MINDIA", "ABFRL", "ABLBL", "BAYERCROP", "BIKAJI",
    "BLUEDART", "BLUEJET", "BBTC", "MAPMYINDIA", "CHALET",
    "DCMSHRIRAM", "ERIS", "GODIGIT", "INDGN", "JSWDULUX",
    "JUBLPHARMA", "LATENTVIEW", "THELEELA", "NEWGEN",
    "NIVABUPA", "PFIZER", "SAPPHIRE", "SAREGAMA", "SBFC",
    "SONATSOFTW", "SPLPETRO", "TRAVELFOOD"
}

ENTRY_SYMBOLS = {
    "AETHER", "AVANTIFEED", "AZAD", "BAGMANE", "BHARATCOAL",
    "BBOX", "BIRET", "CMPDI", "CLEANMAX", "CUPID", "EMBASSY",
    "FRACTAL", "INOXINDIA", "KIRLOSBROS", "KSB", "MTARTECH",
    "PRIVISCL", "RUBICON", "SANSERA", "SHRIPISTON", "STLTECH",
    "TDPOWERSYS", "THANGAMAYL", "VAML", "VEDPOWER", "VISL",
    "VOGL"
}

# Work on copies; preserve original objects until validation passes.
iv = intervals_df.copy()
p = final_df.copy()

iv["start_date"] = pd.to_datetime(
    iv["start_date"], errors="coerce"
).dt.normalize()
iv["end_date"] = pd.to_datetime(
    iv["end_date"], errors="coerce"
).dt.normalize()

p["date"] = pd.to_datetime(
    p["date"], errors="coerce"
).dt.normalize()

# Identify the outgoing and incoming interval records.
exit_mask = (
    iv["symbol"].isin(EXIT_SYMBOLS)
    & iv["end_date"].eq(EFFECTIVE_DATE)
)

entry_mask = (
    iv["symbol"].isin(ENTRY_SYMBOLS)
    & iv["start_date"].eq(EFFECTIVE_DATE)
)

found_exits = set(iv.loc[exit_mask, "symbol"])
found_entries = set(iv.loc[entry_mask, "symbol"])

print("=" * 65)
print("1. VERIFY OFFICIAL CHANGE LISTS")
print("=" * 65)
print("Expected exits:", len(EXIT_SYMBOLS))
print("Matched exit intervals:", int(exit_mask.sum()))
print("Missing exit symbols:", sorted(EXIT_SYMBOLS - found_exits))
print("Unexpected exit symbols:", sorted(found_exits - EXIT_SYMBOLS))

print("\nExpected entries:", len(ENTRY_SYMBOLS))
print("Matched entry intervals:", int(entry_mask.sum()))
print("Missing entry symbols:", sorted(ENTRY_SYMBOLS - found_entries))
print("Unexpected entry symbols:", sorted(found_entries - ENTRY_SYMBOLS))

# Stop if the source intervals do not match the expected official lists.
if (
    found_exits != EXIT_SYMBOLS
    or found_entries != ENTRY_SYMBOLS
    or int(exit_mask.sum()) != 27
    or int(entry_mask.sum()) != 27
):
    raise ValueError(
        "The interval lists did not match exactly. "
        "No changes have been applied; inspect the output above."
    )

# Save the outgoing ISINs before altering their end dates.
exit_isins = set(iv.loc[exit_mask, "isin"].dropna().astype(str))

# The changes took effect at the start of 30 September:
# outgoing constituents were last members on 29 September.
iv.loc[exit_mask, "end_date"] = PREVIOUS_DATE
iv.loc[exit_mask, "end_confidence"] = "press_release_exact"

# The official release confirms these incoming constituent starts.
iv.loc[entry_mask, "start_confidence"] = "press_release_exact"

# Correct membership flags only for the exiting ISINs on 30 September.
p["isin"] = p["isin"].astype(str)
boundary_rows = (
    p["date"].eq(EFFECTIVE_DATE)
    & p["isin"].isin(exit_isins)
)

flag_col = "is_nifty500_constituent"
before_flags = int(
    p.loc[boundary_rows, flag_col]
    .astype(str).str.strip().str.lower()
    .isin(["true", "1", "yes", "y"]).sum()
)

p.loc[boundary_rows, flag_col] = False

after_flags = int(
    p.loc[p["date"].eq(EFFECTIVE_DATE), flag_col]
    .astype(str).str.strip().str.lower()
    .isin(["true", "1", "yes", "y"]).sum()
)

# Count active intervals on the effective date after the correction.
active_now = iv[
    iv["start_date"].notna()
    & (iv["start_date"] <= EFFECTIVE_DATE)
    & (iv["end_date"].isna() | (iv["end_date"] >= EFFECTIVE_DATE))
]

print("\n" + "=" * 65)
print("2. CORRECTION RESULTS")
print("=" * 65)
print("Boundary price rows affected:", int(boundary_rows.sum()))
print("Flagged records among exiting ISINs before:", before_flags)
print("Flagged constituent ISINs on Sep 30 after:",
      p.loc[p["date"].eq(EFFECTIVE_DATE) & (
          p[flag_col].astype(str).str.strip().str.lower()
          .isin(["true", "1", "yes", "y"])
      ), "isin"].nunique())
print("Active interval rows on Sep 30:", len(active_now))
print("Active interval ISINs on Sep 30:", active_now["isin"].nunique())
print("Active interval symbols on Sep 30:", active_now["symbol"].nunique())

# Save versioned outputs, keeping previous files intact.
interval_path = os.path.join(
    DATA_DIR, "pit_intervals_corrected_v2.csv"
)
parquet_path = os.path.join(
    DATA_DIR, "nifty500_ohlcv_pit_membership_fixed_v2.parquet"
)

iv.to_csv(interval_path, index=False)
p.to_parquet(parquet_path, index=False)

# Update notebook variables so subsequent cells use the corrected versions.
intervals_df = iv
final_df = p

print("\nSaved corrected files:")
print(interval_path)
print(parquet_path)
print("\nBOUNDARY CORRECTION COMPLETE")

1. VERIFY OFFICIAL CHANGE LISTS
Expected exits: 27
Matched exit intervals: 27
Missing exit symbols: []
Unexpected exit symbols: []

Expected entries: 27
Matched entry intervals: 27
Missing entry symbols: []
Unexpected entry symbols: []

2. CORRECTION RESULTS
Boundary price rows affected: 27
Flagged records among exiting ISINs before: 27
Flagged constituent ISINs on Sep 30 after: 499
Active interval rows on Sep 30: 501
Active interval ISINs on Sep 30: 501
Active interval symbols on Sep 30: 501

Saved corrected files:
/content/nifty500_pit_data/pit_intervals_corrected_v2.csv
/content/nifty500_pit_data/nifty500_ohlcv_pit_membership_fixed_v2.parquet

BOUNDARY CORRECTION COMPLETE


In [38]:
import pandas as pd
import numpy as np

# Use the corrected objects already in memory.
p = final_df.copy()

tri = pd.read_csv(
    "/content/nifty500_pit_data/nifty500_tri.csv"
)
tri["Date"] = pd.to_datetime(
    tri["Date"].astype(str).str.strip(),
    format="%Y-%m-%d",
    errors="coerce"
).dt.normalize()

benchmark_dates = set(tri["Date"].dropna())

p["date"] = pd.to_datetime(p["date"], errors="coerce").dt.normalize()
p["isin"] = p["isin"].astype(str).str.strip()
p["symbol"] = p["symbol"].astype(str).str.strip()
p["_on_benchmark"] = p["date"].isin(benchmark_dates)

print("=" * 68)
print("1. SOURCE COVERAGE")
print("=" * 68)

source_summary = (
    p.groupby(["data_source", "adjustment"], dropna=False)
    .agg(
        Rows=("date", "size"),
        Unique_ISINs=("isin", "nunique"),
        First_Date=("date", "min"),
        Last_Date=("date", "max"),
        Rows_On_Benchmark=("_on_benchmark", "sum"),
    )
    .reset_index()
)

display(source_summary)

print("\n" + "=" * 68)
print("2. HOW MANY COMPANIES HAVE MULTIPLE SOURCES?")
print("=" * 68)

def describe_values(s):
    return " | ".join(sorted(s.dropna().astype(str).unique()))

per_isin = (
    p.groupby("isin")
    .agg(
        symbol=("symbol", "first"),
        Rows=("date", "size"),
        First_Date=("date", "min"),
        Last_Date=("date", "max"),
        Source_Count=("data_source", "nunique"),
        Sources=("data_source", describe_values),
        Adjustment_Count=("adjustment", "nunique"),
        Adjustments=("adjustment", describe_values),
    )
    .reset_index()
)

print("Companies/ISINs:", len(per_isin))
print("ISINs with multiple data sources:",
      int((per_isin["Source_Count"] > 1).sum()))
print("ISINs with multiple adjustment labels:",
      int((per_isin["Adjustment_Count"] > 1).sum()))

print("\nSource combinations across companies:")
display(
    per_isin.groupby(["Sources", "Adjustments"], dropna=False)
    .agg(
        Companies=("isin", "nunique"),
        Rows=("Rows", "sum")
    )
    .reset_index()
    .sort_values("Companies", ascending=False)
)

multi_source = per_isin[per_isin["Source_Count"] > 1]
if len(multi_source):
    print("\nSample ISINs with multiple sources:")
    display(multi_source.head(30))

print("\n" + "=" * 68)
print("3. RAW-FALLBACK COVERAGE")
print("=" * 68)

raw = p[p["data_source"] == "nse_bhavcopy_raw"].copy()
raw_on_calendar = raw[raw["_on_benchmark"]]

print("Raw fallback rows, all dates:", len(raw))
print("Raw fallback rows on benchmark dates:", len(raw_on_calendar))
print("Raw fallback companies on benchmark dates:",
      raw_on_calendar["isin"].nunique())

if len(raw_on_calendar):
    raw_by_company = (
        raw_on_calendar.groupby(["isin", "symbol"])
        .agg(
            Rows=("date", "size"),
            First_Date=("date", "min"),
            Last_Date=("date", "max"),
        )
        .reset_index()
        .sort_values("Rows", ascending=False)
    )

    print("\nRaw-price coverage by company (top 30):")
    display(raw_by_company.head(30))

print("\n" + "=" * 68)
print("4. SAME-DATE DUPLICATE AND STALE-PRICE CHECK")
print("=" * 68)

dup_mask = p.duplicated(["isin", "date"], keep=False)
print("Duplicate ISIN/date rows:", int(dup_mask.sum()))

if dup_mask.any():
    print("Examples:")
    display(
        p.loc[
            dup_mask,
            [
                c for c in [
                    "date", "isin", "symbol", "close",
                    "volume", "data_source", "adjustment"
                ] if c in p.columns
            ]
        ].sort_values(["isin", "date"]).head(30)
    )

ohlc_cols = ["open", "high", "low", "close"]
if all(c in p.columns for c in ohlc_cols) and "volume" in p.columns:
    q = p[p["_on_benchmark"]].copy()

    for c in ohlc_cols + ["volume"]:
        q[c] = pd.to_numeric(q[c], errors="coerce")

    q["_flat_ohlc"] = (
        q[ohlc_cols].notna().all(axis=1)
        & (q[ohlc_cols].max(axis=1) - q[ohlc_cols].min(axis=1) <= 1e-6)
    )
    q["_zero_volume"] = q["volume"].eq(0)

    print("\nOn-benchmark rows with zero volume:",
          int(q["_zero_volume"].sum()))
    print("On-benchmark rows with flat OHLC and zero volume:",
          int((q["_flat_ohlc"] & q["_zero_volume"]).sum()))

    print("\nBy source:")
    display(
        q.groupby("data_source")
        .agg(
            Rows=("date", "size"),
            Zero_Volume_Rows=("_zero_volume", "sum"),
            Flat_And_Zero=(
                "_flat_ohlc",
                lambda s: int(
                    (s & q.loc[s.index, "_zero_volume"]).sum()
                )
            )
        )
        .reset_index()
    )

print("\nAUDIT COMPLETE — source data unchanged.")

1. SOURCE COVERAGE


,data_source,adjustment,Rows,Unique_ISINs,First_Date,Last_Date,Rows_On_Benchmark
0,nse_bhavcopy_raw,raw_unadjusted,217132,163,2010-01-01,2026-10-22,83447
1,yfinance,split_adjusted_no_dividend,1907826,894,2010-01-04,2026-10-23,1413803



2. HOW MANY COMPANIES HAVE MULTIPLE SOURCES?
Companies/ISINs: 1057
ISINs with multiple data sources: 0
ISINs with multiple adjustment labels: 0

Source combinations across companies:


,Sources,Adjustments,Companies,Rows
1,yfinance,split_adjusted_no_dividend,894,1907826
0,nse_bhavcopy_raw,raw_unadjusted,163,217132



3. RAW-FALLBACK COVERAGE
Raw fallback rows, all dates: 217132
Raw fallback rows on benchmark dates: 83447
Raw fallback companies on benchmark dates: 97

Raw-price coverage by company (top 30):


,isin,symbol,Rows,First_Date,Last_Date
62,INE572A01036,JBCHEPHARM,2680,2015-01-01,2026-06-24
29,INE246F01010,GSPL,2649,2015-01-01,2026-05-08
16,INE140A01024,PEL,2502,2015-01-01,2025-09-12
91,INE886H01027,TV18BRDCST,2286,2015-01-01,2024-10-10
5,INE043D01016,IDFC,2284,2015-01-01,2024-10-08
40,INE312H01016,INOXLEISUR,1902,2015-01-01,2023-02-08
4,INE018I01017,MINDTREE,1795,2015-01-01,2022-08-29
0,IN9155A01020,TATAMTRDVR,1792,2016-12-27,2024-08-28
55,INE483S01020,INFIBEAM,1766,2016-08-11,2024-02-27
1,INE001A01036,HDFC,1715,2015-01-01,2022-04-29



4. SAME-DATE DUPLICATE AND STALE-PRICE CHECK
Duplicate ISIN/date rows: 0

On-benchmark rows with zero volume: 2567
On-benchmark rows with flat OHLC and zero volume: 2557

By source:


,data_source,Rows,Zero_Volume_Rows,Flat_And_Zero
0,nse_bhavcopy_raw,83447,0,0
1,yfinance,1413803,2567,2557



AUDIT COMPLETE — source data unchanged.


In [39]:
import pandas as pd
import numpy as np

p = final_df.copy()

tri = pd.read_csv(
    "/content/nifty500_pit_data/nifty500_tri.csv"
)
tri["Date"] = pd.to_datetime(
    tri["Date"].astype(str).str.strip(),
    format="%Y-%m-%d",
    errors="coerce"
).dt.normalize()

benchmark_dates = set(tri["Date"].dropna())

p["date"] = pd.to_datetime(p["date"], errors="coerce").dt.normalize()

ohlc = ["open", "high", "low", "close"]
for c in ohlc + ["volume"]:
    p[c] = pd.to_numeric(p[c], errors="coerce")

# Keep the full official TRI calendar, including special weekend sessions.
q = p[
    p["date"].isin(benchmark_dates)
    & (p["data_source"] == "yfinance")
].copy()

q["_flat_ohlc"] = (
    q[ohlc].notna().all(axis=1)
    & (
        q[ohlc].max(axis=1) - q[ohlc].min(axis=1)
        <= 1e-6
    )
)
q["_zero_volume"] = q["volume"].eq(0)
q["_suspect"] = q["_flat_ohlc"] & q["_zero_volume"]

print("=" * 65)
print("1. OVERALL COUNTS")
print("=" * 65)
print("Yahoo rows on benchmark dates:", len(q))
print("Flat OHLC + zero volume:", int(q["_suspect"].sum()))
print("Distinct affected ISINs:", q.loc[q["_suspect"], "isin"].nunique())
print("Distinct affected dates:", q.loc[q["_suspect"], "date"].nunique())

# ---------------------------------------------------------
# 2. Are anomalies clustered by date?
# ---------------------------------------------------------
print("\n" + "=" * 65)
print("2. DATES WITH MOST AFFECTED STOCKS")
print("=" * 65)

by_date = (
    q[q["_suspect"]]
    .groupby("date")
    .agg(
        Affected_Rows=("date", "size"),
        Affected_ISINs=("isin", "nunique"),
    )
    .sort_values("Affected_Rows", ascending=False)
)

display(by_date.head(25))

# ---------------------------------------------------------
# 3. Are anomalies concentrated in particular stocks?
# ---------------------------------------------------------
print("\n" + "=" * 65)
print("3. STOCKS WITH MOST AFFECTED OBSERVATIONS")
print("=" * 65)

by_stock = (
    q[q["_suspect"]]
    .groupby(["isin", "symbol"])
    .agg(
        Affected_Rows=("date", "size"),
        First_Affected_Date=("date", "min"),
        Last_Affected_Date=("date", "max"),
    )
    .reset_index()
    .sort_values("Affected_Rows", ascending=False)
)

display(by_stock.head(30))

# ---------------------------------------------------------
# 4. Inspect actual records for the largest clusters
# ---------------------------------------------------------
print("\n" + "=" * 65)
print("4. SAMPLE AFFECTED RECORDS")
print("=" * 65)

if len(by_date):
    top_dates = by_date.head(3).index
    sample = q[
        q["date"].isin(top_dates) & q["_suspect"]
    ].sort_values(["date", "symbol"])

    display(
        sample[
            [
                "date", "symbol", "isin",
                "open", "high", "low", "close",
                "volume", "is_nifty500_constituent"
            ]
        ].head(40)
    )

print("\nDIAGNOSTIC COMPLETE — source data unchanged.")

1. OVERALL COUNTS
Yahoo rows on benchmark dates: 1413803
Flat OHLC + zero volume: 2557
Distinct affected ISINs: 540
Distinct affected dates: 672

2. DATES WITH MOST AFFECTED STOCKS


,Affected_Rows,Affected_ISINs
date,,
2025-03-18,506,506
2024-01-15,91,91
2015-08-12,27,27
2021-11-04,18,18
2022-10-24,17,17
2018-11-07,14,14
2022-09-12,11,11
2015-09-15,10,10
2015-11-30,9,9



3. STOCKS WITH MOST AFFECTED OBSERVATIONS


,isin,symbol,Affected_Rows,First_Affected_Date,Last_Affected_Date
166,INE168A01041,J&KBANK,533,2015-01-01,2025-09-09
221,INE236A01020,HCL-INSYS,530,2015-01-01,2017-02-23
428,INE684F01012,FSL,528,2015-01-06,2025-03-18
191,INE200A01026,GVT&D,116,2016-09-14,2025-09-09
301,INE385W01011,DCAL,80,2017-05-30,2017-09-21
39,INE036A01016,RELINFRA,28,2015-08-12,2026-01-30
386,INE572E01012,PNBHOUSING,17,2021-10-20,2025-03-18
431,INE686F01025,UBL,11,2015-08-12,2025-03-18
381,INE548C01032,EMAMILTD,8,2015-08-12,2025-03-18
246,INE271C01023,DLF,7,2015-08-12,2025-03-18



4. SAMPLE AFFECTED RECORDS


,date,symbol,isin,open,high,low,close,volume,is_nifty500_constituent
1770317,2015-08-12,ADANIPORTS,INE742F01042,331.799988,331.799988,331.799988,331.799988,0,True
748774,2015-08-12,BRITANNIA,INE216A01030,1683.449951,1683.449951,1683.449951,1683.449951,0,True
1327880,2015-08-12,CENTENKA,INE485A01015,234.649994,234.649994,234.649994,234.649994,0,True
919482,2015-08-12,DLF,INE271C01023,124.349998,124.349998,124.349998,124.349998,0,True
771284,2015-08-12,DYNAMATECH,INE221B01012,3065.850098,3065.850098,3065.850098,3065.850098,0,True
1463561,2015-08-12,EMAMILTD,INE548C01032,634.250000,634.250000,634.250000,634.250000,0,True
1380562,2015-08-12,ENGINERSIN,INE510A01028,109.675003,109.675003,109.675003,109.675003,0,True
1671763,2015-08-12,FSL,INE684F01012,34.750000,34.750000,34.750000,34.750000,0,True
28828,2015-08-12,GEOJITFSL,INE007B01023,44.950001,44.950001,44.950001,44.950001,0,True
1828202,2015-08-12,GMRAIRPORT,INE776C01039,12.207581,12.207581,12.207581,12.207581,0,True



DIAGNOSTIC COMPLETE — source data unchanged.


In [40]:
import os
import pandas as pd
import numpy as np

DATA_DIR = "/content/nifty500_pit_data"

# Use the corrected in-memory dataset. Do not overwrite it.
p = final_df.copy()

# Load benchmark calendar using explicit ISO-date parsing.
tri = pd.read_csv(os.path.join(DATA_DIR, "nifty500_tri.csv"))
tri["Date"] = pd.to_datetime(
    tri["Date"].astype(str).str.strip(),
    format="%Y-%m-%d",
    errors="coerce"
).dt.normalize()

benchmark_dates = set(tri["Date"].dropna())
benchmark_start = tri["Date"].min()
benchmark_end = tri["Date"].max()

p["date"] = pd.to_datetime(p["date"], errors="coerce").dt.normalize()

for c in ["open", "high", "low", "close", "volume"]:
    p[c] = pd.to_numeric(p[c], errors="coerce")

# ---------------------------------------------------------
# 1. Detect suspicious flat OHLC / zero-volume records
# ---------------------------------------------------------
ohlc = ["open", "high", "low", "close"]

p["_flat_ohlc"] = (
    p[ohlc].notna().all(axis=1)
    & (
        p[ohlc].max(axis=1) - p[ohlc].min(axis=1)
        <= 1e-6
    )
)

is_yahoo = p["data_source"].eq("yfinance")
zero_volume = p["volume"].eq(0)

suspect = is_yahoo & p["_flat_ohlc"] & zero_volume

# This is a market-wide anomaly: quarantine all Yahoo records
# for that date, not merely the rows that passed the above test.
marketwide_date = pd.Timestamp("2025-03-18")
marketwide_rows = is_yahoo & p["date"].eq(marketwide_date)

p["price_quality_flag"] = "not_flagged"
p.loc[suspect, "price_quality_flag"] = "flat_ohlc_zero_volume"
p.loc[marketwide_rows, "price_quality_flag"] = (
    "marketwide_anomaly_2025_03_18"
)

# Preserve original close values, but provide a safe analytical column.
p["close_for_analysis"] = p["close"]
p.loc[suspect | marketwide_rows, "close_for_analysis"] = np.nan

# ---------------------------------------------------------
# 2. Record calendar status
# ---------------------------------------------------------
within_benchmark_window = p["date"].between(
    benchmark_start, benchmark_end
)
on_benchmark_calendar = p["date"].isin(benchmark_dates)

p["calendar_status"] = "outside_benchmark_coverage"
p.loc[
    within_benchmark_window & on_benchmark_calendar,
    "calendar_status"
] = "on_benchmark_calendar"

p.loc[
    within_benchmark_window & ~on_benchmark_calendar,
    "calendar_status"
] = "off_benchmark_calendar"

# ---------------------------------------------------------
# 3. Build a conservative candidate price flag
# ---------------------------------------------------------
# These rows have split-adjusted Yahoo prices, valid calendar
# dates, positive closes, and no detected zero-volume flat-OHLC issue.
# This is a candidate price panel, not yet a backtest-ready dataset.
p["price_candidate_eligible"] = (
    is_yahoo
    & p["adjustment"].eq("split_adjusted_no_dividend")
    & p["calendar_status"].eq("on_benchmark_calendar")
    & p["close_for_analysis"].notna()
    & p["close_for_analysis"].gt(0)
)

# ---------------------------------------------------------
# 4. Audit results
# ---------------------------------------------------------
print("=" * 65)
print("1. PRICE QUALITY SUMMARY")
print("=" * 65)
print("Total PIT rows:", len(p))
print("Flagged flat-OHLC / zero-volume rows:", int(suspect.sum()))
print("Rows on 2025-03-18 quarantined:", int(marketwide_rows.sum()))
print("Candidate Yahoo price rows:",
      int(p["price_candidate_eligible"].sum()))

print("\nPrice quality flags:")
display(
    p["price_quality_flag"].value_counts()
    .rename_axis("Quality_Flag")
    .to_frame("Rows")
)

print("\nCalendar status:")
display(
    p["calendar_status"].value_counts()
    .rename_axis("Calendar_Status")
    .to_frame("Rows")
)

print("\nCandidate rows by source and adjustment:")
display(
    p.groupby(
        ["data_source", "adjustment"], dropna=False
    )["price_candidate_eligible"]
    .agg(Rows="size", Candidate_Rows="sum")
    .reset_index()
)

# Check the quarantined date explicitly.
print("\n2025-03-18 audit:")
display(
    p.loc[marketwide_rows]
    .groupby(["price_quality_flag", "data_source"])
    .agg(
        Rows=("date", "size"),
        Unique_ISINs=("isin", "nunique")
    )
    .reset_index()
)

# Save a new version; original parquet and in-memory source remain untouched.
out_path = os.path.join(
    DATA_DIR, "nifty500_ohlcv_pit_quality_v1.parquet"
)
p.to_parquet(out_path, index=False)

print("\nSaved:", out_path)
print("Original source data unchanged.")

1. PRICE QUALITY SUMMARY
Total PIT rows: 2124958
Flagged flat-OHLC / zero-volume rows: 4781
Rows on 2025-03-18 quarantined: 510
Candidate Yahoo price rows: 1411242

Price quality flags:


,Rows
Quality_Flag,
not_flagged,2120173
flat_ohlc_zero_volume,4275
marketwide_anomaly_2025_03_18,510



Calendar status:


,Rows
Calendar_Status,
on_benchmark_calendar,1497250
outside_benchmark_coverage,622590
off_benchmark_calendar,5118



Candidate rows by source and adjustment:


,data_source,adjustment,Rows,Candidate_Rows
0,nse_bhavcopy_raw,raw_unadjusted,217132,0
1,yfinance,split_adjusted_no_dividend,1907826,1411242



2025-03-18 audit:


,price_quality_flag,data_source,Rows,Unique_ISINs
0,marketwide_anomaly_2025_03_18,yfinance,510,510



Saved: /content/nifty500_pit_data/nifty500_ohlcv_pit_quality_v1.parquet
Original source data unchanged.


In [41]:
import os
import pandas as pd
import numpy as np

DATA_DIR = "/content/nifty500_pit_data"

# Use the corrected objects already in memory.
iv = intervals_df.copy()

tri = pd.read_csv(os.path.join(DATA_DIR, "nifty500_tri.csv"))
tri["Date"] = pd.to_datetime(
    tri["Date"].astype(str).str.strip(),
    format="%Y-%m-%d",
    errors="coerce"
).dt.normalize()

benchmark_dates = pd.DatetimeIndex(
    tri["Date"].dropna().unique()
).sort_values()

iv["start_date"] = pd.to_datetime(
    iv["start_date"], errors="coerce"
).dt.normalize()
iv["end_date"] = pd.to_datetime(
    iv["end_date"], errors="coerce"
).dt.normalize()

# Missing ends mean the interval is still active through this audit's end.
audit_end = benchmark_dates.max()
iv["_effective_end"] = iv["end_date"].fillna(audit_end)

# ---------------------------------------------------------
# 1. Count active intervals and symbols on every benchmark date
# ---------------------------------------------------------
records = []

for d in benchmark_dates:
    active = iv[
        iv["start_date"].notna()
        & (iv["start_date"] <= d)
        & (iv["_effective_end"] >= d)
    ]

    starts = active["start_confidence"].fillna("").astype(str)
    ends = active["end_confidence"].fillna("").astype(str)

    records.append({
        "date": d,
        "active_interval_rows": len(active),
        "active_unique_isins": active["isin"].nunique(),
        "active_unique_symbols": active["symbol"].nunique(),
        "left_censored_symbols": active.loc[
            starts.eq("left_censored"), "symbol"
        ].nunique(),
        "long_gap_symbols": active.loc[
            starts.eq("interpolated_long_gap"), "symbol"
        ].nunique(),
        "snapshot_adjacent_symbols": active.loc[
            starts.eq("interpolated_snapshot_adjacent"), "symbol"
        ].nunique(),
        "press_release_start_symbols": active.loc[
            starts.eq("press_release_exact"), "symbol"
        ].nunique(),
        "isin_migration_start_symbols": active.loc[
            starts.str.startswith("isin_migration_"), "symbol"
        ].nunique(),
        "interpolated_end_symbols": active.loc[
            ends.isin([
                "interpolated_long_gap",
                "interpolated_snapshot_adjacent"
            ]), "symbol"
        ].nunique(),
    })

census = pd.DataFrame(records)
census["year"] = census["date"].dt.year

print("=" * 68)
print("1. MEMBERSHIP COUNTS BY YEAR")
print("=" * 68)

yearly = census.groupby("year").agg(
    Trading_Dates=("date", "size"),
    Min_Symbols=("active_unique_symbols", "min"),
    Median_Symbols=("active_unique_symbols", "median"),
    Max_Symbols=("active_unique_symbols", "max"),
    Dates_Under_490=(
        "active_unique_symbols", lambda x: int((x < 490).sum())
    ),
    Dates_490_to_515=(
        "active_unique_symbols",
        lambda x: int(((x >= 490) & (x <= 515)).sum())
    ),
    Dates_Over_515=(
        "active_unique_symbols", lambda x: int((x > 515).sum())
    ),
)

display(yearly)

# ---------------------------------------------------------
# 2. Dates with unusually low/high reconstructed membership
# ---------------------------------------------------------
print("\n" + "=" * 68)
print("2. DATES OUTSIDE THE 490-515 SYMBOL RANGE")
print("=" * 68)

unusual = census[
    (census["active_unique_symbols"] < 490)
    | (census["active_unique_symbols"] > 515)
].copy()

print("Total unusual benchmark dates:", len(unusual))
print("\nLowest reconstructed counts:")
display(
    unusual.sort_values("active_unique_symbols")
    .head(20)
)

print("\nHighest reconstructed counts:")
display(
    unusual.sort_values("active_unique_symbols", ascending=False)
    .head(20)
)

# ---------------------------------------------------------
# 3. Membership confidence on selected dates
# ---------------------------------------------------------
print("\n" + "=" * 68)
print("3. CONFIDENCE MIX ON SELECTED DATES")
print("=" * 68)

selected_dates = [
    "2015-01-23",
    "2016-12-28",
    "2022-05-02",
    "2025-02-21",
    "2026-09-29",
    "2026-09-30",
    "2026-10-01",
    "2026-10-19",
]

display(
    census[
        census["date"].isin(pd.to_datetime(selected_dates))
    ].drop(columns="year")
)

# ---------------------------------------------------------
# 4. Save an auditable report
# ---------------------------------------------------------
out_path = os.path.join(
    DATA_DIR, "pit_membership_interval_census_v1.csv"
)
census.to_csv(out_path, index=False)

print("\nSaved:", out_path)
print("Membership census complete. Source data unchanged.")

1. MEMBERSHIP COUNTS BY YEAR


,Trading_Dates,Min_Symbols,Median_Symbols,Max_Symbols,Dates_Under_490,Dates_490_to_515,Dates_Over_515
year,,,,,,,
2015,248,503,504.0,512,0,248,0
2016,247,507,512.0,569,0,153,94
2017,248,509,518.0,526,0,32,216
2018,246,502,514.0,523,0,208,38
2019,245,487,495.0,531,38,206,1
2020,252,495,499.0,505,0,252,0
2021,248,480,482.0,502,210,38,0
2022,248,488,495.0,549,80,167,1
2023,246,499,502.0,510,0,246,0



2. DATES OUTSIDE THE 490-515 SYMBOL RANGE
Total unusual benchmark dates: 716

Lowest reconstructed counts:


,date,active_interval_rows,active_unique_isins,active_unique_symbols,left_censored_symbols,long_gap_symbols,snapshot_adjacent_symbols,press_release_start_symbols,isin_migration_start_symbols,interpolated_end_symbols,year
2421,2024-10-10,502,502,479,135,121,4,134,105,27,2024
2420,2024-10-09,502,502,479,135,121,4,134,105,27,2024
2419,2024-10-08,502,502,479,135,121,4,134,105,27,2024
2418,2024-10-07,502,502,479,135,121,4,134,105,27,2024
2416,2024-10-03,502,502,479,135,121,4,134,105,27,2024
2417,2024-10-04,502,502,479,135,121,4,134,105,27,2024
2389,2024-08-26,502,502,479,135,121,4,134,105,27,2024
2390,2024-08-27,502,502,479,135,121,4,134,105,27,2024
2392,2024-08-29,502,502,479,135,121,4,134,105,27,2024
2393,2024-08-30,502,502,479,135,121,4,134,105,27,2024



Highest reconstructed counts:


,date,active_interval_rows,active_unique_isins,active_unique_symbols,left_censored_symbols,long_gap_symbols,snapshot_adjacent_symbols,press_release_start_symbols,isin_migration_start_symbols,interpolated_end_symbols,year
492,2016-12-28,623,623,569,281,140,1,83,113,140,2016
1814,2022-05-02,594,594,549,163,179,4,133,111,79,2022
1190,2019-10-29,575,575,531,187,138,4,127,114,109,2019
2515,2025-02-21,554,554,530,132,155,3,158,103,14,2025
283,2016-02-22,573,573,527,298,94,1,70,105,117,2016
656,2017-08-28,583,583,526,246,116,1,104,110,92,2017
527,2017-02-16,580,580,526,250,120,1,95,109,88,2017
748,2018-01-08,581,581,523,239,115,1,109,111,95,2018
400,2016-08-12,570,570,523,286,89,1,83,106,121,2016
778,2018-02-21,580,580,522,237,115,1,110,111,95,2018



3. CONFIDENCE MIX ON SELECTED DATES


,date,active_interval_rows,active_unique_isins,active_unique_symbols,left_censored_symbols,long_gap_symbols,snapshot_adjacent_symbols,press_release_start_symbols,isin_migration_start_symbols,interpolated_end_symbols
16,2015-01-23,560,560,504,317,97,1,38,102,111
492,2016-12-28,623,623,569,281,140,1,83,113,140
1814,2022-05-02,594,594,549,163,179,4,133,111,79
2515,2025-02-21,554,554,530,132,155,3,158,103,14
2909,2026-09-29,501,501,501,111,115,20,160,95,0
2910,2026-09-30,501,501,501,105,106,17,179,94,0
2911,2026-10-01,501,501,501,105,106,17,179,94,0
2922,2026-10-19,501,501,501,105,106,17,179,94,0



Saved: /content/nifty500_pit_data/pit_membership_interval_census_v1.csv
Membership census complete. Source data unchanged.


In [42]:
import os
import pandas as pd
import numpy as np

DATA_DIR = "/content/nifty500_pit_data"

# Load benchmark dates using strict ISO parsing
tri = pd.read_csv(os.path.join(DATA_DIR, "nifty500_tri.csv"))
tri["Date"] = pd.to_datetime(
    tri["Date"].astype(str).str.strip(),
    format="%Y-%m-%d",
    errors="coerce"
).dt.normalize()

benchmark_dates = pd.DatetimeIndex(
    tri["Date"].dropna().unique()
).sort_values()

# Use the corrected PIT data in memory
p = final_df.copy()
p["date"] = pd.to_datetime(p["date"], errors="coerce").dt.normalize()

p["_member"] = (
    p["is_nifty500_constituent"]
    .astype(str).str.strip().str.lower()
    .isin(["true", "1", "yes", "y"])
)

# Only rows whose dates appear in the benchmark calendar
p = p[p["date"].isin(set(benchmark_dates))].copy()

# Count actual flagged constituents by date
flagged = p[p["_member"]]

daily_flagged = (
    flagged.groupby("date")["isin"]
    .nunique()
    .reindex(benchmark_dates, fill_value=0)
    .rename("Flagged_Unique_ISINs")
)

daily_prices = (
    p.groupby("date")["isin"]
    .nunique()
    .reindex(benchmark_dates, fill_value=0)
    .rename("Priced_Unique_ISINs")
)

audit = pd.concat([daily_flagged, daily_prices], axis=1)
audit.index.name = "Date"
audit = audit.reset_index()
audit["Year"] = audit["Date"].dt.year

print("=" * 68)
print("1. ACTUAL MEMBERSHIP-FLAG COUNTS BY YEAR")
print("=" * 68)

yearly = audit.groupby("Year").agg(
    Trading_Dates=("Date", "size"),
    Min_Flagged=("Flagged_Unique_ISINs", "min"),
    Median_Flagged=("Flagged_Unique_ISINs", "median"),
    Max_Flagged=("Flagged_Unique_ISINs", "max"),
    Dates_Under_490=(
        "Flagged_Unique_ISINs",
        lambda x: int((x < 490).sum())
    ),
    Dates_490_to_507=(
        "Flagged_Unique_ISINs",
        lambda x: int(((x >= 490) & (x <= 507)).sum())
    ),
    Dates_508_to_515=(
        "Flagged_Unique_ISINs",
        lambda x: int(((x >= 508) & (x <= 515)).sum())
    ),
    Dates_Over_515=(
        "Flagged_Unique_ISINs",
        lambda x: int((x > 515).sum())
    ),
)

display(yearly)

print("\n" + "=" * 68)
print("2. DATES WITH EXTREME FLAG COUNTS")
print("=" * 68)

print("\nLowest counts:")
display(
    audit.sort_values("Flagged_Unique_ISINs")
    .head(20)
)

print("\nHighest counts:")
display(
    audit.sort_values("Flagged_Unique_ISINs", ascending=False)
    .head(20)
)

print("\n" + "=" * 68)
print("3. COMPARE ACTUAL FLAGS WITH ACTIVE INTERVALS")
print("=" * 68)

if "intervals_df" in globals():
    iv = intervals_df.copy()
    iv["start_date"] = pd.to_datetime(
        iv["start_date"], errors="coerce"
    ).dt.normalize()
    iv["end_date"] = pd.to_datetime(
        iv["end_date"], errors="coerce"
    ).dt.normalize()

    end_of_audit = benchmark_dates.max()
    iv["_end"] = iv["end_date"].fillna(end_of_audit)

    compare_dates = [
        "2015-01-23",
        "2016-12-28",
        "2017-08-28",
        "2019-10-29",
        "2022-05-02",
        "2025-02-21",
        "2026-09-29",
        "2026-09-30",
        "2026-10-19",
    ]

    comparison = []

    for date_str in compare_dates:
        d = pd.Timestamp(date_str)

        active = iv[
            iv["start_date"].notna()
            & (iv["start_date"] <= d)
            & (iv["_end"] >= d)
        ]

        actual = audit[audit["Date"].eq(d)]

        comparison.append({
            "Date": date_str,
            "Active_Interval_ISINs": active["isin"].nunique(),
            "Active_Interval_Symbols": active["symbol"].nunique(),
            "Actual_Flagged_ISINs": (
                int(actual["Flagged_Unique_ISINs"].iloc[0])
                if len(actual) else 0
            ),
            "Price_ISINs": (
                int(actual["Priced_Unique_ISINs"].iloc[0])
                if len(actual) else 0
            ),
        })

    display(pd.DataFrame(comparison))

print("\nAUDIT COMPLETE — no source data modified.")

1. ACTUAL MEMBERSHIP-FLAG COUNTS BY YEAR


,Trading_Dates,Min_Flagged,Median_Flagged,Max_Flagged,Dates_Under_490,Dates_490_to_507,Dates_508_to_515,Dates_Over_515
Year,,,,,,,,
2015,248,0,500.0,507,14,234,0,0
2016,247,0,506.0,526,14,145,86,2
2017,248,431,493.0,511,47,168,33,0
2018,246,457,507.0,512,14,139,93,0
2019,245,37,491.0,516,51,193,0,1
2020,252,0,496.0,502,13,239,0,0
2021,248,456,479.0,499,211,37,0,0
2022,248,464,490.0,522,86,161,0,1
2023,246,0,500.0,507,2,244,0,0



2. DATES WITH EXTREME FLAG COUNTS

Lowest counts:


,Date,Flagged_Unique_ISINs,Priced_Unique_ISINs,Year
40,2015-02-28,0,0,2015
451,2016-10-30,0,0,2016
2746,2026-02-01,0,0,2026
2195,2023-11-12,0,0,2023
214,2015-11-11,0,0,2015
2243,2024-01-20,0,0,2024
2272,2024-03-02,0,0,2024
1257,2020-02-01,0,0,2020
2322,2024-05-18,0,0,2024
1050,2019-03-29,37,38,2019



Highest counts:


,Date,Flagged_Unique_ISINs,Priced_Unique_ISINs,Year
492,2016-12-28,526,532,2016
2515,2025-02-21,525,535,2025
1814,2022-05-02,522,539,2022
2762,2026-02-23,519,525,2026
283,2016-02-22,519,523,2016
1190,2019-10-29,516,532,2019
2269,2024-02-28,514,534,2024
409,2016-08-26,513,519,2016
404,2016-08-19,513,519,2016
418,2016-09-09,512,518,2016



3. COMPARE ACTUAL FLAGS WITH ACTIVE INTERVALS


,Date,Active_Interval_ISINs,Active_Interval_Symbols,Actual_Flagged_ISINs,Price_ISINs
0,2015-01-23,560,504,420,425
1,2016-12-28,623,569,526,532
2,2017-08-28,583,526,500,517
3,2019-10-29,575,531,516,532
4,2022-05-02,594,549,522,539
5,2025-02-21,554,530,525,535
6,2026-09-29,501,501,499,507
7,2026-09-30,501,501,499,526
8,2026-10-19,501,501,499,499



AUDIT COMPLETE — no source data modified.


In [43]:
import os
import pandas as pd
import numpy as np

DATA_DIR = "/content/nifty500_pit_data"

# Benchmark calendar
tri = pd.read_csv(os.path.join(DATA_DIR, "nifty500_tri.csv"))
tri["Date"] = pd.to_datetime(
    tri["Date"].astype(str).str.strip(),
    format="%Y-%m-%d",
    errors="coerce"
).dt.normalize()

benchmark_dates = pd.DatetimeIndex(
    tri["Date"].dropna().unique()
).sort_values()

# Corrected PIT data
p = final_df.copy()
p["date"] = pd.to_datetime(p["date"], errors="coerce").dt.normalize()

p["_member"] = (
    p["is_nifty500_constituent"]
    .astype(str).str.strip().str.lower()
    .isin(["true", "1", "yes", "y"])
)

p = p[p["date"].isin(set(benchmark_dates))].copy()
members = p[p["_member"]].copy()

# ---------------------------------------------------------
# 1. Count both ISINs and symbols for every date
# ---------------------------------------------------------
daily_isin = members.groupby("date")["isin"].nunique()
daily_symbol = members.groupby("date")["symbol"].nunique()

# Count symbols with multiple flagged ISINs on a date.
isin_per_symbol = (
    members.groupby(["date", "symbol"])["isin"]
    .nunique()
)
multi_isin_symbols = (
    isin_per_symbol.gt(1)
    .groupby(level="date")
    .sum()
)

daily = pd.DataFrame(index=benchmark_dates)
daily.index.name = "date"

daily["Flagged_ISINs"] = daily_isin.reindex(
    benchmark_dates, fill_value=0
)
daily["Flagged_Symbols"] = daily_symbol.reindex(
    benchmark_dates, fill_value=0
)
daily["Symbols_With_Multiple_ISINs"] = multi_isin_symbols.reindex(
    benchmark_dates, fill_value=0
)

daily["Year"] = daily.index.year

print("=" * 68)
print("1. FLAGGED ISINS VS SYMBOLS BY YEAR")
print("=" * 68)

yearly = daily.groupby("Year").agg(
    Trading_Dates=("Flagged_Symbols", "size"),
    Median_Flagged_ISINs=("Flagged_ISINs", "median"),
    Median_Flagged_Symbols=("Flagged_Symbols", "median"),
    Min_Flagged_Symbols=("Flagged_Symbols", "min"),
    Max_Flagged_Symbols=("Flagged_Symbols", "max"),
    Dates_Over_515_Symbols=(
        "Flagged_Symbols", lambda s: int((s > 515).sum())
    ),
    Median_Multi_ISIN_Symbols=(
        "Symbols_With_Multiple_ISINs", "median"
    ),
)

display(yearly)

# ---------------------------------------------------------
# 2. Inspect dates with highest flagged-symbol counts
# ---------------------------------------------------------
print("\n" + "=" * 68)
print("2. HIGHEST FLAGGED-SYMBOL COUNTS")
print("=" * 68)

display(
    daily.sort_values(
        ["Flagged_Symbols", "Flagged_ISINs"],
        ascending=False
    ).head(20).reset_index()
)

# ---------------------------------------------------------
# 3. Inspect duplicate ISINs behind selected dates
# ---------------------------------------------------------
print("\n" + "=" * 68)
print("3. MULTIPLE FLAGGED ISINS PER SYMBOL")
print("=" * 68)

for date_str in [
    "2015-01-23",
    "2016-12-28",
    "2022-05-02",
    "2025-02-21",
    "2026-09-30",
]:
    d = pd.Timestamp(date_str)
    g = members[members["date"].eq(d)]

    per_symbol = (
        g.groupby("symbol")
        .agg(
            Unique_ISINs=("isin", "nunique"),
            Unique_Company_Names=(
                "company_name", "nunique"
            ) if "company_name" in g.columns else ("symbol", "size"),
        )
        .reset_index()
    )

    duplicates = per_symbol[per_symbol["Unique_ISINs"] > 1]

    print(f"\n{date_str}")
    print("Flagged ISINs:", g["isin"].nunique())
    print("Flagged symbols:", g["symbol"].nunique())
    print("Symbols with multiple flagged ISINs:", len(duplicates))

    if len(duplicates):
        display(
            duplicates.sort_values(
                "Unique_ISINs", ascending=False
            ).head(20)
        )

print("\nAUDIT COMPLETE — source data unchanged.")

1. FLAGGED ISINS VS SYMBOLS BY YEAR


,Trading_Dates,Median_Flagged_ISINs,Median_Flagged_Symbols,Min_Flagged_Symbols,Max_Flagged_Symbols,Dates_Over_515_Symbols,Median_Multi_ISIN_Symbols
Year,,,,,,,
2015,248,500.0,500.0,0,507,0,0.0
2016,247,506.0,506.0,0,526,2,0.0
2017,248,493.0,493.0,431,511,0,0.0
2018,246,507.0,507.0,457,512,0,0.0
2019,245,491.0,491.0,37,516,1,0.0
2020,252,496.0,496.0,0,502,0,0.0
2021,248,479.0,479.0,456,499,0,0.0
2022,248,490.0,490.0,464,522,1,0.0
2023,246,500.0,500.0,0,507,0,0.0



2. HIGHEST FLAGGED-SYMBOL COUNTS


,date,Flagged_ISINs,Flagged_Symbols,Symbols_With_Multiple_ISINs,Year
0,2016-12-28,526,526,0,2016
1,2025-02-21,525,525,0,2025
2,2022-05-02,522,522,0,2022
3,2016-02-22,519,519,0,2016
4,2026-02-23,519,519,0,2026
5,2019-10-29,516,516,0,2019
6,2024-02-28,514,514,0,2024
7,2016-08-19,513,513,0,2016
8,2016-08-26,513,513,0,2016
9,2016-08-22,512,512,0,2016



3. MULTIPLE FLAGGED ISINS PER SYMBOL

2015-01-23
Flagged ISINs: 420
Flagged symbols: 420
Symbols with multiple flagged ISINs: 0

2016-12-28
Flagged ISINs: 526
Flagged symbols: 526
Symbols with multiple flagged ISINs: 0

2022-05-02
Flagged ISINs: 522
Flagged symbols: 522
Symbols with multiple flagged ISINs: 0

2025-02-21
Flagged ISINs: 525
Flagged symbols: 525
Symbols with multiple flagged ISINs: 0

2026-09-30
Flagged ISINs: 499
Flagged symbols: 499
Symbols with multiple flagged ISINs: 0

AUDIT COMPLETE — source data unchanged.


In [44]:
import pandas as pd
import numpy as np

p = final_df.copy()
iv = intervals_df.copy()

p["date"] = pd.to_datetime(p["date"], errors="coerce").dt.normalize()
iv["start_date"] = pd.to_datetime(
    iv["start_date"], errors="coerce"
).dt.normalize()
iv["end_date"] = pd.to_datetime(
    iv["end_date"], errors="coerce"
).dt.normalize()

p["_member"] = (
    p["is_nifty500_constituent"]
    .astype(str).str.strip().str.lower()
    .isin(["true", "1", "yes", "y"])
)

tri = pd.read_csv("/content/nifty500_pit_data/nifty500_tri.csv")
tri["Date"] = pd.to_datetime(
    tri["Date"].astype(str).str.strip(),
    format="%Y-%m-%d",
    errors="coerce"
).dt.normalize()

calendar = pd.DatetimeIndex(
    tri["Date"].dropna().unique()
).sort_values()

calendar = calendar[(calendar >= pd.Timestamp("2015-01-01"))]
calendar_set = set(calendar)

def flagged_symbols(d):
    rows = p[(p["date"] == d) & p["_member"]]
    return set(rows["symbol"].dropna().astype(str).str.strip())

# High-count dates identified in the audit.
check_dates = [
    "2016-02-22",
    "2016-12-28",
    "2019-10-29",
    "2022-05-02",
    "2025-02-21",
    "2026-02-23",
]

iv["_effective_end"] = iv["end_date"].fillna(calendar.max())

print("=" * 68)
print("1. FLAGGED SYMBOL SET CHANGES AROUND SPIKE DATES")
print("=" * 68)

for date_str in check_dates:
    d = pd.Timestamp(date_str)

    if d not in calendar_set:
        print(f"\n{date_str}: not present in benchmark calendar; skipped")
        continue

    idx = calendar.get_loc(d)
    if idx == 0 or idx == len(calendar) - 1:
        continue

    prev_d = calendar[idx - 1]
    next_d = calendar[idx + 1]

    prev_set = flagged_symbols(prev_d)
    cur_set = flagged_symbols(d)
    next_set = flagged_symbols(next_d)

    added = sorted(cur_set - prev_set)
    removed = sorted(prev_set - cur_set)
    one_day_only = sorted(cur_set - (prev_set | next_set))

    print(f"\n--- Spike date: {date_str} ---")
    print(f"Previous session {prev_d.date()}: {len(prev_set)} symbols")
    print(f"Spike session    {d.date()}: {len(cur_set)} symbols")
    print(f"Next session     {next_d.date()}: {len(next_set)} symbols")
    print(f"Added vs previous: {len(added)}")
    print(f"Removed vs previous: {len(removed)}")
    print(f"Present only on spike date: {len(one_day_only)}")
    print("Added symbols:", added[:60])
    print("Removed symbols:", removed[:60])
    print("Spike-only symbols:", one_day_only[:60])

    # Show interval records for symbols that appear only on this date.
    if one_day_only:
        details = iv[iv["symbol"].astype(str).isin(one_day_only)].copy()
        details = details[
            details["start_date"].between(
                d - pd.Timedelta(days=45),
                d + pd.Timedelta(days=45)
            )
            | details["end_date"].between(
                d - pd.Timedelta(days=45),
                d + pd.Timedelta(days=45)
            )
        ]

        cols = [
            c for c in [
                "isin", "symbol", "start_date", "start_confidence",
                "end_date", "end_confidence"
            ] if c in details.columns
        ]

        print("\nSpike-only symbol interval details:")
        display(details[cols].sort_values(["symbol", "start_date"]))

print("\nSPIKE AUDIT COMPLETE — source data unchanged.")

1. FLAGGED SYMBOL SET CHANGES AROUND SPIKE DATES

--- Spike date: 2016-02-22 ---
Previous session 2016-02-19: 502 symbols
Spike session    2016-02-22: 519 symbols
Next session     2016-02-23: 506 symbols
Added vs previous: 20
Removed vs previous: 3
Present only on spike date: 0
Added symbols: ['ABFRL', 'ADANIENSOL', 'AEGISLOG', 'AVANTIFEED', 'BLISSGVS', 'CAPLIPOINT', 'CCL', 'CERA', 'DISHTV', 'GRANULES', 'GREENPLY', 'GRINDWELL', 'GUJENERGY', 'HEIDELBERG', 'ICIL', 'IFBIND', 'INDOCO', 'INTELLECT', 'JETAIRWAYS', 'SECURKLOUD']
Removed symbols: ['GAMMNINFRA', 'GEOMETRIC', 'GITANJALI']
Spike-only symbols: []

--- Spike date: 2016-12-28 ---
Previous session 2016-12-27: 512 symbols
Spike session    2016-12-28: 526 symbols
Next session     2016-12-29: 485 symbols
Added vs previous: 30
Removed vs previous: 16
Present only on spike date: 0
Added symbols: ['BOSCH-HCIL', 'CIEINDIA', 'DCAL', 'FCONSUMER', 'HSCL', 'JKCEMENT', 'KPRMILL', 'LUXIND', 'MANAPPURAM', 'NAVKARCORP', 'NESCO', 'PCJEWELLER', 'PHIL

In [45]:
====================================================================
1. FLAGGED SYMBOL SET CHANGES AROUND SPIKE DATES
====================================================================

--- Spike date: 2016-02-22 ---
Previous session 2016-02-19: 502 symbols
Spike session    2016-02-22: 519 symbols
Next session     2016-02-23: 506 symbols
Added vs previous: 20
Removed vs previous: 3
Present only on spike date: 0
Added symbols: ['ABFRL', 'ADANIENSOL', 'AEGISLOG', 'AVANTIFEED', 'BLISSGVS', 'CAPLIPOINT', 'CCL', 'CERA', 'DISHTV', 'GRANULES', 'GREENPLY', 'GRINDWELL', 'GUJENERGY', 'HEIDELBERG', 'ICIL', 'IFBIND', 'INDOCO', 'INTELLECT', 'JETAIRWAYS', 'SECURKLOUD']
Removed symbols: ['GAMMNINFRA', 'GEOMETRIC', 'GITANJALI']
Spike-only symbols: []

--- Spike date: 2016-12-28 ---
Previous session 2016-12-27: 512 symbols
Spike session    2016-12-28: 526 symbols
Next session     2016-12-29: 485 symbols
Added vs previous: 30
Removed vs previous: 16
Present only on spike date: 0
Added symbols: ['BOSCH-HCIL', 'CIEINDIA', 'DCAL', 'FCONSUMER', 'HSCL', 'JKCEMENT', 'KPRMILL', 'LUXIND', 'MANAPPURAM', 'NAVKARCORP', 'NESCO', 'PCJEWELLER', 'PHILIPCARB', 'RKFORGE', 'SADBHIN', 'SHILPAMED', 'SJVN', 'SOLARINDS', 'SUVEN', 'SYNGENE', 'TAKE', 'TATAMTRDVR', 'TEAMLEASE', 'TEXRAIL', 'TIMKEN', 'TVSSRICHAK', 'UNITDSPR', 'VGUARD', 'VINATIORGA', 'VMART']
Removed symbols: ['AMTEKINDIA', 'IVRCLINFRA', 'KSK', 'MANDHANA', 'MANGCHEFER', 'MERCATOR', 'MONNETISPA', 'OPTOCIRCUI', 'ORCHIDCHEM', 'RASOYPR', 'REIAGROLTD', 'SEINV', 'SHRENUJ', 'SINTEX', 'TECHNO', 'UTTAMSTL']
Spike-only symbols: []

--- Spike date: 2019-10-29 ---
Previous session 2019-10-27: 454 symbols
Spike session    2019-10-29: 516 symbols
Next session     2019-10-30: 491 symbols
Added vs previous: 62
Removed vs previous: 0
Present only on spike date: 0
Added symbols: ['AFFLE', 'ASAHIINDIA', 'DHFL', 'EQUITAS', 'GSPL', 'HDFC', 'HEXAWARE', 'IBVENTURES', 'ICRA', 'IDFC', 'IIFL', 'INDIAMART', 'INDOSTAR', 'INFIBEAM', 'INGERRAND', 'INOXLEISUR', 'IRCON', 'IRCTC', 'ISEC', 'JAMNAAUTO', 'JBCHEPHARM', 'JKPAPER', 'JSLHISAR', 'KEI', 'KPITTECH', 'KSB', 'LAOPALA', 'MAHSCOOTER', 'MAHSEAMLES', 'MCX', 'METROPOLIS', 'MIDHANI', 'MINDTREE', 'NBVENTURES', 'NESTLEIND', 'ORIENTELEC', 'PEL', 'PHILIPCARB', 'POLYCAB', 'POLYMED', 'RATNAMANI', 'RHIM', 'RVNL', 'SCHNEIDER', 'SHRIRAMCIT', 'SPANDANA', 'SPICEJET', 'SUPRAJIT', 'SWSOLAR', 'TASTYBITE', 'TATACOMM', 'TATAMTRDVR', 'TATASTLBSL', 'TCIEXP', 'TCNSBRANDS', 'TV18BRDCST', 'UJJIVAN', 'VAIBHAVGBL', 'VESUVIUS', 'VIYASH']
Removed symbols: []
Spike-only symbols: []

--- Spike date: 2022-05-02 ---
Previous session 2022-04-29: 485 symbols
Spike session    2022-05-02: 522 symbols
Next session     2022-05-04: 490 symbols
Added vs previous: 58
Removed vs previous: 21
Present only on spike date: 0
Added symbols: ['360ONE', 'AARTIDRUGS', 'ADANIENT', 'ANGELONE', 'APTUS', 'BCG', 'BORORENEW', 'CHALET', 'CHEMPLASTS', 'CLEAN', 'COHANCE', 'DEVYANI', 'EASEMYTRIP', 'EPIGRAL', 'ETERNAL', 'GAEL', 'GOCOLORS', 'GRINFRA', 'HLEGLAS', 'KIMS', 'LATENTVIEW', 'MAPMYINDIA', 'MAXHEALTH', 'MAZDOCK', 'MEDPLUS', 'METROBRAND', 'MTARTECH', 'NAZARA', 'NETWORK18', 'NUVOCO', 'NYKAA', 'PAYTM', 'PCBL', 'POLICYBZR', 'POLYPLEX', 'POONAWALLA', 'POWERINDIA', 'PRINCEPIPE', 'RBA', 'ROSSARI', 'ROUTE', 'RTNINDIA', 'SAFARI', 'SAPPHIRE', 'SAREGAMA', 'SBICARD', 'SHYAMMETL', 'SONACOMS', 'STARHEALTH', 'TANLA', 'TATACHEM', 'TRITURBINE', 'TRIVENI', 'TTML', 'USHAMART', 'UTIAMC', 'VIJAYA', 'YESBANK']
Removed symbols: ['EQUITAS', 'GSPL', 'HDFC', 'IBVENTURES', 'IDFC', 'IIFLWAM', 'INFIBEAM', 'INOXLEISUR', 'ISEC', 'JBCHEPHARM', 'JSLHISAR', 'MINDTREE', 'NBVENTURES', 'PEL', 'PHILIPCARB', 'SHRIRAMCIT', 'SPICEJET', 'TATAMTRDVR', 'TCNSBRANDS', 'TV18BRDCST', 'UJJIVAN']
Spike-only symbols: []

--- Spike date: 2025-02-21 ---
Previous session 2025-02-20: 497 symbols
Spike session    2025-02-21: 525 symbols
Next session     2025-02-24: 499 symbols
Added vs previous: 28
Removed vs previous: 0
Present only on spike date: 0
Added symbols: ['ACMESOLAR', 'AFCONS', 'AIIL', 'BAJAJHFL', 'FIRSTCRY', 'GRAVITA', 'HYUNDAI', 'IGIL', 'IKS', 'JSWHL', 'LTFOODS', 'NAVA', 'NEULANDLAB', 'NIVABUPA', 'NTPCGREEN', 'OLAELEC', 'PGEL', 'PREMIERENE', 'RPOWER', 'SAGILITY', 'SAILIFE', 'SARDAEN', 'SWIGGY', 'TARIL', 'VMM', 'WAAREEENER', 'WOCKPHARMA', 'ZENTEC']
Removed symbols: []
Spike-only symbols: []

--- Spike date: 2026-02-23 ---
Previous session 2026-02-20: 499 symbols
Spike session    2026-02-23: 519 symbols
Next session     2026-02-24: 501 symbols
Added vs previous: 20
Removed vs previous: 0
Present only on spike date: 0
Added symbols: ['ABDL', 'ACUTAAS', 'ANTHEM', 'ANURAS', 'BELRISE', 'CANHLIFE', 'CARTRADE', 'CEMPRO', 'CPPLUS', 'EMMVEE', 'GABRIEL', 'GALLANTT', 'GROWW', 'HDBFS', 'ICICIAMC', 'JAINREC', 'LENSKART', 'LGEINDIA', 'MEESHO', 'PARADEEP']
Removed symbols: []
Spike-only symbols: []

SPIKE AUDIT COMPLETE — source data unchanged.


SyntaxError: leading zeros in decimal integer literals are not permitted; use an 0o prefix for octal integers (558253302.py, line 5)

---

**Done.** `nifty500_pit_data/nifty500_ohlcv_pit.parquet` is your own reproduction of the dataset — filter on `is_nifty500_constituent` for any date to get the real historical NIFTY 500 universe, joined with clean OHLCV.

See the repo README for the full methodology writeup, known limitations, and adjustment-method reasoning.

In [46]:
import os
import zipfile
from pathlib import Path
from google.colab import files

DATA_DIR = Path("/content/nifty500_pit_data")
REPO_DIR = Path("/content/Nifty500_Pit_data")
BACKUP_PATH = Path("/content/Stock_Project_Resume_Backup_2026-10-24.zip")

# Core files needed to continue tomorrow
candidate_files = [
    DATA_DIR / "nifty500_ohlcv_pit_quality_v1.parquet",
    DATA_DIR / "nifty500_ohlcv_pit_membership_fixed_v2.parquet",
    DATA_DIR / "pit_intervals_corrected_v2.csv",
    DATA_DIR / "pit_intervals_corrected.csv",
    DATA_DIR / "pit_manifest_corrected.csv",
    DATA_DIR / "nifty500_tri.csv",
    DATA_DIR / "pit_membership_interval_census_v1.csv",
    DATA_DIR / "nse_equity_master.csv",
    DATA_DIR / "nse_fallback_failures.csv",
]

# Include repository notes when available.
candidate_files += [
    REPO_DIR / "README.md",
]

handoff_notes = """INDIAN STOCK INVESTMENT SYSTEM — SESSION HANDOFF
Date: 24 October 2026

CURRENT PROJECT
Building a point-in-time Nifty 500 stock research and backtesting
pipeline. No new backtest should be treated as reliable yet.

MAIN DATA FILE
nifty500_ohlcv_pit_quality_v1.parquet
Contains the PIT price records plus price-quality flags and a
conservative candidate-price eligibility flag.

KEY FILES
nifty500_tri.csv:
Benchmark TRI history, 2015-01-01 to 2026-10-19.

pit_intervals_corrected_v2.csv:
Corrected historical membership intervals, including the verified
27 exits and 27 entries effective 2026-09-30.

pit_membership_interval_census_v1.csv:
Historical interval membership count audit.

PRICE-DATA FINDINGS
- PIT dataset has about 2.12 million rows.
- Yahoo source: split-adjusted prices without dividend adjustment.
- NSE fallback: raw, unadjusted prices; do not mix into momentum
  calculations until corporate actions are addressed.
- Duplicate ISIN/date rows: zero in the last audit.
- Flat OHLC/zero-volume anomalies were flagged.
- 2025-03-18 was quarantined as a market-wide price-data anomaly.
- Some records fall outside the benchmark trading-date calendar.

MEMBERSHIP FINDINGS
- The 2026-09-30 boundary correction verified 27 outgoing and
  27 incoming symbols. The corrected active interval count is 501.
- Historical flags still require validation, especially around
  2016-2019 and other dates with unusually high counts.
- ISIN migrations and uncertain historical interval reconstruction
  must be accounted for; do not assume current or inferred historical
  membership is exact.

NEXT STEP
Run the interval-versus-price-versus-flag mismatch diagnostic from
the last conversation message. Review the mismatch counts before
changing any more membership flags.

THEN
Resolve membership uncertainty and raw-price corporate actions,
validate a conservative analytical price panel, and only then build
a realistic backtest with transaction costs and point-in-time
constituents.

IMPORTANT
Do not repeat the lengthy PIT download unless a required file is
actually missing. Keep the earlier Pre-PIT backup as a separate
recovery point.
"""

notes_path = Path("/content/SESSION_HANDOFF_2026-10-24.txt")
notes_path.write_text(handoff_notes, encoding="utf-8")

existing_files = [f for f in candidate_files if f.exists()]
missing_files = [str(f) for f in candidate_files if not f.exists()]

with zipfile.ZipFile(BACKUP_PATH, "w") as zf:
    for f in existing_files:
        # Parquet is already compressed, so avoid expensive recompression.
        compression = (
            zipfile.ZIP_STORED
            if f.suffix.lower() == ".parquet"
            else zipfile.ZIP_DEFLATED
        )
        zf.write(
            f,
            arcname=f"{f.parent.name}/{f.name}",
            compress_type=compression
        )

    zf.write(notes_path, arcname=notes_path.name)

print("Backup created:", BACKUP_PATH)
print("Files included:")
for f in existing_files:
    print("  OK:", f.name)

if missing_files:
    print("\nOptional or expected files not found:")
    for f in missing_files:
        print("  MISSING:", f)

print("\nZIP size (MB):", round(BACKUP_PATH.stat().st_size / (1024**2), 2))
print("Starting download to your Mac...")

files.download(str(BACKUP_PATH))

Backup created: /content/Stock_Project_Resume_Backup_2026-10-24.zip
Files included:
  OK: nifty500_ohlcv_pit_quality_v1.parquet
  OK: nifty500_ohlcv_pit_membership_fixed_v2.parquet
  OK: pit_intervals_corrected_v2.csv
  OK: pit_intervals_corrected.csv
  OK: pit_manifest_corrected.csv
  OK: nifty500_tri.csv
  OK: pit_membership_interval_census_v1.csv
  OK: nse_equity_master.csv
  OK: nse_fallback_failures.csv

Optional or expected files not found:
  MISSING: /content/Nifty500_Pit_data/README.md

ZIP size (MB): 106.77
Starting download to your Mac...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import zipfile

ROOT = Path("/content")
DATA_DIR = ROOT / "nifty500_pit_data"

# If yesterday's files aren't present, extract the saved backup if available.
def locate_file(filename):
    direct = DATA_DIR / filename
    if direct.exists():
        return direct
    matches = list(ROOT.rglob(filename))
    return matches[0] if matches else None

if locate_file("nifty500_ohlcv_pit_quality_v1.parquet") is None:
    backups = (
        list(ROOT.glob("Stock_Project_Resume_Backup*.zip"))
        + list(ROOT.glob("*PrePIT*Backup*.zip"))
    )
    if backups:
        backup_dir = ROOT / "stock_project_resume_backup"
        backup_dir.mkdir(exist_ok=True)
        with zipfile.ZipFile(backups[0]) as z:
            z.extractall(backup_dir)

# Load existing data; reuse in-memory objects when available.
if "final_df" in globals() and isinstance(final_df, pd.DataFrame):
    p = final_df.copy()
else:
    price_path = locate_file("nifty500_ohlcv_pit_quality_v1.parquet")
    if price_path is None:
        raise FileNotFoundError(
            "Price-quality parquet not found. Upload and extract "
            "Stock_Project_Resume_Backup_2026-10-24.zip, then rerun."
        )
    p = pd.read_parquet(price_path)

if "intervals_df" in globals() and isinstance(intervals_df, pd.DataFrame):
    iv = intervals_df.copy()
else:
    interval_path = locate_file("pit_intervals_corrected_v2.csv")
    if interval_path is None:
        raise FileNotFoundError(
            "Corrected interval CSV not found in /content."
        )
    iv = pd.read_csv(interval_path)

tri_path = locate_file("nifty500_tri.csv")
if tri_path is None:
    raise FileNotFoundError("nifty500_tri.csv not found.")

# Normalize dates and flags.
p["date"] = pd.to_datetime(p["date"], errors="coerce").dt.normalize()
p["isin"] = p["isin"].astype(str).str.strip()
p["symbol"] = p["symbol"].astype(str).str.strip()

p["_flag"] = (
    p["is_nifty500_constituent"].astype(str).str.strip().str.lower()
    .isin(["true", "1", "yes", "y"])
)

iv["start_date"] = pd.to_datetime(
    iv["start_date"], errors="coerce"
).dt.normalize()
iv["end_date"] = pd.to_datetime(
    iv["end_date"], errors="coerce"
).dt.normalize()
iv["isin"] = iv["isin"].astype(str).str.strip()
iv["symbol"] = iv["symbol"].astype(str).str.strip()

tri = pd.read_csv(tri_path)
tri["Date"] = pd.to_datetime(
    tri["Date"].astype(str).str.strip(),
    format="%Y-%m-%d",
    errors="coerce"
).dt.normalize()

calendar = pd.DatetimeIndex(
    tri["Date"].dropna().unique()
).sort_values()
calendar_set = set(calendar)
iv["_effective_end"] = iv["end_date"].fillna(calendar.max())

# Compare intervals, prices, and flags for representative dates.
check_dates = [
    "2015-01-23",
    "2016-12-28",
    "2017-08-28",
    "2019-10-29",
    "2022-05-02",
    "2025-02-21",
    "2026-09-29",
    "2026-09-30",
    "2026-10-19",
]

summary = []
examples = []

for date_str in check_dates:
    d = pd.Timestamp(date_str)
    if d not in calendar_set:
        print("Skipping date absent from benchmark calendar:", date_str)
        continue

    active = iv[
        iv["start_date"].notna()
        & (iv["start_date"] <= d)
        & (iv["_effective_end"] >= d)
    ]

    day = p[p["date"].eq(d)]
    expected = set(active["isin"])
    priced = set(day["isin"])
    flagged = set(day.loc[day["_flag"], "isin"])

    categories = {
        "Flagged_Not_In_Intervals": flagged - expected,
        "Expected_Priced_But_Unflagged": (expected & priced) - flagged,
        "Priced_Not_In_Intervals": priced - expected,
        "Expected_No_Price": expected - priced,
    }

    summary.append({
        "Date": date_str,
        "Active_Interval_ISINs": len(expected),
        "Price_ISINs": len(priced),
        "Flagged_ISINs": len(flagged),
        "Flagged_Not_In_Intervals": len(categories["Flagged_Not_In_Intervals"]),
        "Expected_Priced_But_Unflagged": len(categories["Expected_Priced_But_Unflagged"]),
        "Priced_Not_In_Intervals": len(categories["Priced_Not_In_Intervals"]),
        "Expected_No_Price": len(categories["Expected_No_Price"]),
    })

    symbol_map = dict(zip(active["isin"], active["symbol"]))
    symbol_map.update(dict(zip(day["isin"], day["symbol"])))

    for category, isins in categories.items():
        for isin in sorted(isins)[:10]:
            examples.append({
                "Date": date_str,
                "Mismatch": category,
                "ISIN": isin,
                "Symbol": symbol_map.get(isin, "Unknown"),
            })

summary_df = pd.DataFrame(summary)
examples_df = pd.DataFrame(examples)

print("=" * 68)
print("1. MEMBERSHIP RECONCILIATION SUMMARY")
print("=" * 68)
display(summary_df)

print("\n" + "=" * 68)
print("2. SAMPLE MISMATCHES")
print("=" * 68)
display(examples_df)

# Save separate audit reports, keeping source data unchanged.
out_dir = DATA_DIR if DATA_DIR.exists() else ROOT
summary_df.to_csv(out_dir / "membership_mismatch_audit_2026-10-25.csv", index=False)
examples_df.to_csv(out_dir / "membership_mismatch_examples_2026-10-25.csv", index=False)

print("\nAudit reports saved to:", out_dir)
print("Source data was not modified.")

FileNotFoundError: Price-quality parquet not found. Upload and extract Stock_Project_Resume_Backup_2026-10-24.zip, then rerun.

In [2]:
from google.colab import files
from pathlib import Path
import zipfile
import shutil

# 1. Upload the backup ZIP from your Mac.
print("Select Stock_Project_Resume_Backup_2026-10-24.zip from your Mac.")
uploaded = files.upload()

zip_names = [name for name in uploaded if name.lower().endswith(".zip")]

if not zip_names:
    raise ValueError("No ZIP file uploaded. Please upload the backup ZIP.")

# Prefer the newer session backup over the older Pre-PIT backup.
preferred = [
    name for name in zip_names
    if "Stock_Project_Resume_Backup" in name
]
zip_name = preferred[0] if preferred else zip_names[0]

print("\nUsing backup:", zip_name)

# 2. Extract to a separate restore folder.
restore_dir = Path("/content/stock_project_restore")
restore_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_name, "r") as z:
    z.extractall(restore_dir)

# 3. Locate the key files inside the extracted backup.
def find_restored_file(filename):
    matches = list(restore_dir.rglob(filename))
    return matches[0] if matches else None

required_files = [
    "nifty500_ohlcv_pit_quality_v1.parquet",
    "pit_intervals_corrected_v2.csv",
    "nifty500_tri.csv",
]

found = {name: find_restored_file(name) for name in required_files}

print("\nBackup file check:")
for name, path in found.items():
    print(("FOUND" if path else "MISSING"), name)

if any(path is None for path in found.values()):
    print("\nFiles available in the uploaded backup:")
    for f in restore_dir.rglob("*"):
        if f.is_file():
            print(" ", f.relative_to(restore_dir))

    raise FileNotFoundError(
        "This ZIP does not contain all the latest session files. "
        "Check that you uploaded Stock_Project_Resume_Backup_2026-10-24.zip, "
        "not the older Pre-PIT backup."
    )

# 4. Restore the files to the expected working directory.
DATA_DIR = Path("/content/nifty500_pit_data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

for name, source in found.items():
    destination = DATA_DIR / name
    shutil.copy2(source, destination)
    print("Restored:", destination)

# 5. Load the existing data into memory.
import pandas as pd

final_df = pd.read_parquet(
    DATA_DIR / "nifty500_ohlcv_pit_quality_v1.parquet"
)
intervals_df = pd.read_csv(
    DATA_DIR / "pit_intervals_corrected_v2.csv"
)

print("\nRESTORE SUCCESSFUL")
print("PIT rows:", len(final_df))
print("Membership interval rows:", len(intervals_df))
print("PIT date range:",
      pd.to_datetime(final_df["date"]).min(),
      "to",
      pd.to_datetime(final_df["date"]).max())
print("Benchmark exists:", (DATA_DIR / "nifty500_tri.csv").exists())

Select Stock_Project_Resume_Backup_2026-10-24.zip from your Mac.


Saving Stock_Project_Resume_Backup_2026-10-24.zip to Stock_Project_Resume_Backup_2026-10-24.zip

Using backup: Stock_Project_Resume_Backup_2026-10-24.zip

Backup file check:
FOUND nifty500_ohlcv_pit_quality_v1.parquet
FOUND pit_intervals_corrected_v2.csv
FOUND nifty500_tri.csv
Restored: /content/nifty500_pit_data/nifty500_ohlcv_pit_quality_v1.parquet
Restored: /content/nifty500_pit_data/pit_intervals_corrected_v2.csv
Restored: /content/nifty500_pit_data/nifty500_tri.csv

RESTORE SUCCESSFUL
PIT rows: 2124958
Membership interval rows: 1292
PIT date range: 2010-01-01 00:00:00 to 2026-10-23 00:00:00
Benchmark exists: True


In [5]:
import os
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("/content/nifty500_pit_data")

# Use restored data in memory.
p = final_df.copy()
iv = intervals_df.copy()

# Normalize identifiers and dates.
p["date"] = pd.to_datetime(p["date"], errors="coerce").dt.normalize()
p["isin"] = p["isin"].astype(str).str.strip()
p["symbol"] = p["symbol"].astype(str).str.strip()

iv["start_date"] = pd.to_datetime(
    iv["start_date"], errors="coerce"
).dt.normalize()
iv["end_date"] = pd.to_datetime(
    iv["end_date"], errors="coerce"
).dt.normalize()
iv["isin"] = iv["isin"].astype(str).str.strip()
iv["symbol"] = iv["symbol"].astype(str).str.strip()

p["_flag"] = (
    p["is_nifty500_constituent"].astype(str).str.strip().str.lower()
    .isin(["true", "1", "yes", "y"])
)

# Read the benchmark using strict ISO dates.
tri = pd.read_csv(DATA_DIR / "nifty500_tri.csv")
tri["Date"] = pd.to_datetime(
    tri["Date"].astype(str).str.strip(),
    format="%Y-%m-%d",
    errors="coerce"
).dt.normalize()

calendar = pd.DatetimeIndex(
    tri["Date"].dropna().unique()
).sort_values()
calendar_set = set(calendar)

iv["_effective_end"] = iv["end_date"].fillna(calendar.max())

# Candidate analytical prices were created in the quality dataset.
if "price_candidate_eligible" in p.columns:
    p["_candidate_price"] = (
        p["price_candidate_eligible"]
        .astype(str).str.strip().str.lower()
        .isin(["true", "1", "yes", "y"])
    )
else:
    p["_candidate_price"] = False

# Representative dates spanning the periods we previously investigated.
check_dates = [
    "2015-01-23",
    "2016-12-28",
    "2017-08-28",
    "2019-10-29",
    "2022-05-02",
    "2025-02-21",
    "2026-09-29",
    "2026-09-30",
    "2026-10-19",
]

summary = []
examples = []

for date_str in check_dates:
    d = pd.Timestamp(date_str)

    if d not in calendar_set:
        print("Skipping date not in TRI calendar:", date_str)
        continue

    active = iv[
        iv["start_date"].notna()
        & (iv["start_date"] <= d)
        & (iv["_effective_end"] >= d)
    ]

    day = p[p["date"].eq(d)].copy()

    # Sets of distinct ISINs for each definition.
    expected = set(active["isin"])
    recorded = set(day["isin"])
    flagged = set(day.loc[day["_flag"], "isin"])
    candidate = set(day.loc[day["_candidate_price"], "isin"])

    mismatch_sets = {
        "Flagged_Not_In_Intervals": flagged - expected,
        "Expected_Priced_But_Unflagged": (expected & recorded) - flagged,
        "Priced_Not_In_Intervals": recorded - expected,
        "Expected_No_Record": expected - recorded,
        "Candidate_Price_Not_In_Intervals": candidate - expected,
    }

    summary.append({
        "Date": date_str,
        "Active_Interval_ISINs": len(expected),
        "Active_Interval_Symbols": active["symbol"].nunique(),
        "Recorded_ISINs": len(recorded),
        "Candidate_Price_ISINs": len(candidate),
        "Flagged_ISINs": len(flagged),
        "Flagged_Not_In_Intervals": len(mismatch_sets["Flagged_Not_In_Intervals"]),
        "Expected_Priced_But_Unflagged": len(mismatch_sets["Expected_Priced_But_Unflagged"]),
        "Priced_Not_In_Intervals": len(mismatch_sets["Priced_Not_In_Intervals"]),
        "Expected_No_Record": len(mismatch_sets["Expected_No_Record"]),
        "Candidate_Price_Not_In_Intervals": len(mismatch_sets["Candidate_Price_Not_In_Intervals"]),
    })

    symbol_map = dict(zip(active["isin"], active["symbol"]))
    symbol_map.update(dict(zip(day["isin"], day["symbol"])))

    for category, isins in mismatch_sets.items():
        for isin in sorted(isins)[:12]:
            examples.append({
                "Date": date_str,
                "Mismatch": category,
                "ISIN": isin,
                "Symbol": symbol_map.get(isin, "Unknown"),
            })

summary_df = pd.DataFrame(summary)
examples_df = pd.DataFrame(examples)

print("=" * 70)
print("1. MEMBERSHIP RECONCILIATION SUMMARY")
print("=" * 70)
display(summary_df)

print("\n" + "=" * 70)
print("2. MISMATCH EXAMPLES")
print("=" * 70)
if len(examples_df):
    display(examples_df)
else:
    print("No mismatches found in the selected dates.")

# Save audit reports without modifying the source dataset.
summary_df.to_csv(
    DATA_DIR / "membership_mismatch_audit_2026-10-25.csv",
    index=False
)
examples_df.to_csv(
    DATA_DIR / "membership_mismatch_examples_2026-10-25.csv",
    index=False
)

print("\nAudit reports saved in:", DATA_DIR)
print("Source data remains unchanged.")

1. MEMBERSHIP RECONCILIATION SUMMARY


,Date,Active_Interval_ISINs,Active_Interval_Symbols,Recorded_ISINs,Candidate_Price_ISINs,Flagged_ISINs,Flagged_Not_In_Intervals,Expected_Priced_But_Unflagged,Priced_Not_In_Intervals,Expected_No_Record,Candidate_Price_Not_In_Intervals
0,2015-01-23,560,504,425,422,420,2,0,7,142,7
1,2016-12-28,623,569,532,472,526,2,0,8,99,8
2,2017-08-28,583,526,517,468,500,1,0,18,84,18
3,2019-10-29,575,531,532,509,516,0,0,16,59,16
4,2022-05-02,594,549,539,539,522,0,0,17,72,17
5,2025-02-21,554,530,535,532,525,0,0,10,29,10
6,2026-09-29,501,501,507,506,499,0,0,8,2,7
7,2026-09-30,501,501,526,525,499,0,0,27,2,27
8,2026-10-19,501,501,499,499,499,0,0,0,2,0



2. MISMATCH EXAMPLES


,Date,Mismatch,ISIN,Symbol
0,2015-01-23,Flagged_Not_In_Intervals,INE486A01021,CESC
1,2015-01-23,Flagged_Not_In_Intervals,INE884A01027,VAIBHAVGBL
2,2015-01-23,Priced_Not_In_Intervals,INE256C01024,TRIVENI
3,2015-01-23,Priced_Not_In_Intervals,INE310A01015,NILKAMAL
4,2015-01-23,Priced_Not_In_Intervals,INE486A01021,CESC
...,...,...,...,...
239,2026-09-30,Candidate_Price_Not_In_Intervals,INE133A01011,JSWDULUX
240,2026-09-30,Candidate_Price_Not_In_Intervals,INE14LE01019,ABLBL
241,2026-09-30,Candidate_Price_Not_In_Intervals,INE182A01018,PFIZER
242,2026-10-19,Expected_No_Record,DUM545A01024,DUMMYHEG



Audit reports saved in: /content/nifty500_pit_data
Source data remains unchanged.


In [4]:
# Correct the effective end date for open-ended membership intervals
iv["_effective_end"] = iv["end_date"].fillna(calendar.max())

print("Maximum benchmark date:", calendar.max())
print("Effective-end column type:", iv["_effective_end"].dtype)
print("Correction successful:", pd.api.types.is_datetime64_any_dtype(
    iv["_effective_end"]
))

Maximum benchmark date: 2026-10-19 00:00:00
Effective-end column type: datetime64[ns]
Correction successful: True


In [6]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("/content/nifty500_pit_data")

p = final_df.copy()
iv = intervals_df.copy()

# Normalize dates and identifiers.
p["date"] = pd.to_datetime(p["date"], errors="coerce").dt.normalize()
iv["start_date"] = pd.to_datetime(
    iv["start_date"], errors="coerce"
).dt.normalize()
iv["end_date"] = pd.to_datetime(
    iv["end_date"], errors="coerce"
).dt.normalize()

p["isin"] = p["isin"].astype(str).str.strip()
p["symbol"] = p["symbol"].astype(str).str.strip()
iv["isin"] = iv["isin"].astype(str).str.strip()
iv["symbol"] = iv["symbol"].astype(str).str.strip()

p["_flag"] = (
    p["is_nifty500_constituent"].astype(str).str.strip().str.lower()
    .isin(["true", "1", "yes", "y"])
)

tri = pd.read_csv(DATA_DIR / "nifty500_tri.csv")
tri["Date"] = pd.to_datetime(
    tri["Date"].astype(str).str.strip(),
    format="%Y-%m-%d",
    errors="coerce"
).dt.normalize()

calendar = pd.DatetimeIndex(
    tri["Date"].dropna().unique()
).sort_values()

# These are diagnostic event dates, not assumed membership change dates.
spike_dates = [
    "2016-02-22",
    "2016-12-28",
    "2019-10-29",
    "2022-05-02",
    "2025-02-21",
    "2026-02-23",
]

# Get previous, event, and next benchmark sessions.
sessions_to_check = set()

for date_str in spike_dates:
    d = pd.Timestamp(date_str)

    if d not in calendar:
        print("Date missing from TRI calendar:", date_str)
        continue

    i = calendar.get_loc(d)

    for j in [i - 1, i, i + 1]:
        if 0 <= j < len(calendar):
            sessions_to_check.add(calendar[j])

sessions_to_check = sorted(sessions_to_check)

# ---------------------------------------------------------
# 1. Price coverage by date and source
# ---------------------------------------------------------
summary = []

for d in sessions_to_check:
    day = p[p["date"].eq(d)]

    for source, group in day.groupby("data_source", dropna=False):
        summary.append({
            "Date": d.date(),
            "Source": source,
            "Rows": len(group),
            "Unique_ISINs": group["isin"].nunique(),
            "Unique_Symbols": group["symbol"].nunique(),
            "Flagged_ISINs": group.loc[
                group["_flag"], "isin"
            ].nunique(),
            "Flagged_Symbols": group.loc[
                group["_flag"], "symbol"
            ].nunique(),
        })

coverage = pd.DataFrame(summary)

print("=" * 70)
print("1. PRICE COVERAGE BY SOURCE AROUND SPIKE DATES")
print("=" * 70)
display(coverage)

# ---------------------------------------------------------
# 2. Actual membership-interval boundaries on these dates
# ---------------------------------------------------------
print("\n" + "=" * 70)
print("2. INTERVAL BOUNDARIES AROUND SPIKE DATES")
print("=" * 70)

boundary_dates = set(sessions_to_check)

boundary_rows = iv[
    iv["start_date"].isin(boundary_dates)
    | iv["end_date"].isin(boundary_dates)
].copy()

if len(boundary_rows):
    boundary_summary = (
        boundary_rows.assign(
            Boundary_Type=np.select(
                [
                    boundary_rows["start_date"].isin(boundary_dates)
                    & boundary_rows["end_date"].isin(boundary_dates),
                    boundary_rows["start_date"].isin(boundary_dates),
                    boundary_rows["end_date"].isin(boundary_dates),
                ],
                [
                    "Starts_and_Ends",
                    "Starts",
                    "Ends",
                ],
                default="Other",
            )
        )
        .groupby(["start_date", "end_date", "Boundary_Type"], dropna=False)
        .agg(
            Interval_Rows=("isin", "size"),
            Unique_ISINs=("isin", "nunique"),
            Unique_Symbols=("symbol", "nunique"),
        )
        .reset_index()
    )

    display(boundary_summary)

    print("\nDetailed interval changes:")
    detail_cols = [
        c for c in [
            "isin", "symbol", "start_date", "start_confidence",
            "end_date", "end_confidence"
        ] if c in boundary_rows.columns
    ]
    display(
        boundary_rows[detail_cols]
        .sort_values(["start_date", "end_date", "symbol"])
        .head(100)
    )
else:
    print("No interval starts or ends occurred on the selected sessions.")

# ---------------------------------------------------------
# 3. Detect bulk source gaps across consecutive sessions
# ---------------------------------------------------------
print("\n" + "=" * 70)
print("3. DAILY COVERAGE CHANGE")
print("=" * 70)

daily_total = (
    p[p["date"].isin(sessions_to_check)]
    .groupby("date")
    .agg(
        Total_Rows=("date", "size"),
        Unique_ISINs=("isin", "nunique"),
        Unique_Symbols=("symbol", "nunique"),
        Flagged_Symbols=(
            "symbol",
            lambda s: p.loc[s.index].loc[
                p.loc[s.index, "_flag"], "symbol"
            ].nunique()
        )
    )
    .reindex(sessions_to_check)
)

daily_total.index.name = "Date"
display(daily_total.reset_index())

print("\nAUDIT COMPLETE — source data unchanged.")

1. PRICE COVERAGE BY SOURCE AROUND SPIKE DATES


,Date,Source,Rows,Unique_ISINs,Unique_Symbols,Flagged_ISINs,Flagged_Symbols
0,2016-02-19,nse_bhavcopy_raw,76,76,76,76,76
1,2016-02-19,yfinance,431,431,431,426,426
2,2016-02-22,nse_bhavcopy_raw,73,73,73,73,73
3,2016-02-22,yfinance,450,450,450,446,446
4,2016-02-23,nse_bhavcopy_raw,73,73,73,73,73
5,2016-02-23,yfinance,440,440,440,433,433
6,2016-12-27,nse_bhavcopy_raw,72,72,72,70,70
7,2016-12-27,yfinance,448,448,448,442,442
8,2016-12-28,nse_bhavcopy_raw,56,56,56,56,56
9,2016-12-28,yfinance,476,476,476,470,470



2. INTERVAL BOUNDARIES AROUND SPIKE DATES


,start_date,end_date,Boundary_Type,Interval_Rows,Unique_ISINs,Unique_Symbols
0,2010-01-02,2016-02-22,Ends,12,12,12
1,2010-01-02,2016-12-28,Ends,33,33,33
2,2010-01-02,2019-10-29,Ends,16,16,16
3,2010-01-02,2022-05-02,Ends,10,10,10
4,2010-01-02,2025-02-21,Ends,6,6,6
...,...,...,...,...,...,...
100,2025-02-21,2025-08-22,Starts,1,1,1
101,2025-02-21,2026-09-29,Starts,1,1,1
102,2025-02-21,NaT,Starts,26,26,26
103,2025-08-22,2026-02-23,Ends,2,2,2



Detailed interval changes:


,isin,symbol,start_date,start_confidence,end_date,end_confidence
256,INE439A01020,ASAHIINDIA,2010-01-02,left_censored,2016-02-22,press_release_exact
367,INE209A01019,ATFL,2010-01-02,left_censored,2016-02-22,press_release_exact
290,INE449A01011,AUTOAXLES,2010-01-02,left_censored,2016-02-22,press_release_exact
248,INE485A01015,CENTENKA,2010-01-02,left_censored,2016-02-22,press_release_exact
375,INE265F01028,ENIL,2010-01-02,left_censored,2016-02-22,press_release_exact
...,...,...,...,...,...,...
340,INE952H01027,RAJTV,2012-01-12,isin_migration_from_INE952H01019,2016-12-28,interpolated_long_gap
365,INE904G01038,RASOYPR,2012-01-12,interpolated_long_gap,2016-12-28,interpolated_long_gap
343,INE420C01042,SEINV,2012-01-12,interpolated_long_gap,2016-12-28,interpolated_long_gap
255,INE550H01011,SUPREMEINF,2012-01-12,interpolated_long_gap,2016-12-28,interpolated_long_gap



3. DAILY COVERAGE CHANGE


,Date,Total_Rows,Unique_ISINs,Unique_Symbols,Flagged_Symbols
0,2016-02-19,507,507,507,502
1,2016-02-22,523,523,523,519
2,2016-02-23,513,513,513,506
3,2016-12-27,520,520,520,512
4,2016-12-28,532,532,532,526
5,2016-12-29,499,499,499,485
6,2019-10-27,483,483,483,454
7,2019-10-29,532,532,532,516
8,2019-10-30,514,514,514,491
9,2022-04-29,508,508,508,485



AUDIT COMPLETE — source data unchanged.


In [7]:
import pandas as pd
import numpy as np

p = final_df.copy()
iv = intervals_df.copy()

# Normalize dates and identifiers.
p["date"] = pd.to_datetime(p["date"], errors="coerce").dt.normalize()
iv["start_date"] = pd.to_datetime(
    iv["start_date"], errors="coerce"
).dt.normalize()
iv["end_date"] = pd.to_datetime(
    iv["end_date"], errors="coerce"
).dt.normalize()

for df in [p, iv]:
    df["isin"] = df["isin"].astype(str).str.strip()
    df["symbol"] = df["symbol"].astype(str).str.strip()

p["_flag"] = (
    p["is_nifty500_constituent"].astype(str).str.strip().str.lower()
    .isin(["true", "1", "yes", "y"])
)

tri = pd.read_csv("/content/nifty500_pit_data/nifty500_tri.csv")
tri["Date"] = pd.to_datetime(
    tri["Date"].astype(str).str.strip(),
    format="%Y-%m-%d",
    errors="coerce"
).dt.normalize()

calendar = pd.DatetimeIndex(
    tri["Date"].dropna().unique()
).sort_values()
calendar_set = set(calendar)

spike_dates = [
    "2016-02-22",
    "2016-12-28",
    "2019-10-29",
    "2022-05-02",
    "2025-02-21",
    "2026-02-23",
]

def day_records(d):
    return p[p["date"].eq(d)]

def flagged_isins(d):
    g = day_records(d)
    return set(g.loc[g["_flag"], "isin"])

def price_isins(d):
    return set(day_records(d)["isin"])

def count_flags_for_isins(d, isin_set):
    g = day_records(d)
    return g.loc[g["isin"].isin(isin_set) & g["_flag"], "isin"].nunique()

summary = []
details = []

for date_str in spike_dates:
    d = pd.Timestamp(date_str)
    if d not in calendar_set:
        continue

    i = calendar.get_loc(d)
    prev_d = calendar[i - 1] if i > 0 else None
    next_d = calendar[i + 1] if i < len(calendar) - 1 else None

    starts = iv[iv["start_date"].eq(d)].copy()
    ends = iv[iv["end_date"].eq(d)].copy()

    prev_flags = flagged_isins(prev_d) if prev_d is not None else set()
    event_flags = flagged_isins(d)
    next_flags = flagged_isins(next_d) if next_d is not None else set()

    event_added = event_flags - prev_flags
    event_removed = prev_flags - event_flags

    start_isins = set(starts["isin"])
    end_isins = set(ends["isin"])

    summary.append({
        "Date": date_str,
        "Previous_Session": prev_d.date() if prev_d is not None else None,
        "Next_Session": next_d.date() if next_d is not None else None,
        "Intervals_Starting": len(starts),
        "Intervals_Ending": len(ends),
        "Starts_Present_As_Flagged_On_Event": len(event_flags & start_isins),
        "Ends_Still_Flagged_On_Event": len(event_flags & end_isins),
        "Ends_Flagged_Next_Session": (
            len(next_flags & end_isins) if next_d is not None else np.nan
        ),
        "Flagged_Added_Vs_Previous": len(event_added),
        "Added_Matching_Interval_Start": len(event_added & start_isins),
        "Flagged_Removed_Vs_Previous": len(event_removed),
        "Removed_Matching_Interval_End": len(event_removed & end_isins),
        "Previous_Flag_Count": len(prev_flags),
        "Event_Flag_Count": len(event_flags),
        "Next_Flag_Count": len(next_flags),
        "Previous_Price_ISINs": len(price_isins(prev_d)) if prev_d is not None else np.nan,
        "Event_Price_ISINs": len(price_isins(d)),
        "Next_Price_ISINs": len(price_isins(next_d)) if next_d is not None else np.nan,
    })

    for boundary_type, group in [("STARTS", starts), ("ENDS", ends)]:
        for _, row in group.iterrows():
            isin = row["isin"]
            records = {
                "Date": date_str,
                "Boundary_Type": boundary_type,
                "ISIN": isin,
                "Symbol": row["symbol"],
                "Start_Confidence": row.get("start_confidence"),
                "End_Confidence": row.get("end_confidence"),
                "Previous_Price_Row": (
                    isin in price_isins(prev_d) if prev_d is not None else False
                ),
                "Previous_Flag": (
                    isin in prev_flags if prev_d is not None else False
                ),
                "Event_Price_Row": isin in price_isins(d),
                "Event_Flag": isin in event_flags,
                "Next_Price_Row": (
                    isin in price_isins(next_d) if next_d is not None else False
                ),
                "Next_Flag": isin in next_flags,
            }
            details.append(records)

summary_df = pd.DataFrame(summary)
details_df = pd.DataFrame(details)

print("=" * 70)
print("1. SPIKE-DATE SUMMARY")
print("=" * 70)
display(summary_df)

print("\n" + "=" * 70)
print("2. INTERVAL START/END RECORDS")
print("=" * 70)
display(details_df)

print("\nAUDIT COMPLETE — no files or source data modified.")

1. SPIKE-DATE SUMMARY


,Date,Previous_Session,Next_Session,Intervals_Starting,Intervals_Ending,Starts_Present_As_Flagged_On_Event,Ends_Still_Flagged_On_Event,Ends_Flagged_Next_Session,Flagged_Added_Vs_Previous,Added_Matching_Interval_Start,Flagged_Removed_Vs_Previous,Removed_Matching_Interval_End,Previous_Flag_Count,Event_Flag_Count,Next_Flag_Count,Previous_Price_ISINs,Event_Price_ISINs,Next_Price_ISINs
0,2016-02-22,2016-02-19,2016-02-23,21,16,20,13,0,20,20,3,3,502,519,506,507,523,513
1,2016-12-28,2016-12-27,2016-12-29,59,60,31,40,0,31,31,17,16,512,526,485,520,532,499
2,2019-10-29,2019-10-27,2019-10-30,49,36,42,25,0,62,42,0,0,454,516,491,483,532,514
3,2022-05-02,2022-04-29,2022-05-04,65,54,58,46,0,58,58,21,7,485,522,490,508,539,518
4,2025-02-21,2025-02-20,2025-02-24,28,26,28,26,0,28,28,0,0,497,525,499,510,535,513
5,2026-02-23,2026-02-20,2026-02-24,20,18,20,18,0,20,20,0,0,499,519,501,508,525,510



2. INTERVAL START/END RECORDS


,Date,Boundary_Type,ISIN,Symbol,Start_Confidence,End_Confidence,Previous_Price_Row,Previous_Flag,Event_Price_Row,Event_Flag,Next_Price_Row,Next_Flag
0,2016-02-22,STARTS,INE844O01022,GUJGASLTD,press_release_exact,isin_migration_to_INE844O01030,False,False,False,False,False,False
1,2016-02-22,STARTS,INE650K01021,8KMILES,press_release_exact,press_release_exact,False,False,True,True,True,True
2,2016-02-22,STARTS,INE483B01026,ICIL,press_release_exact,press_release_exact,False,False,True,True,True,True
3,2016-02-22,STARTS,INE802G01018,JETAIRWAYS,press_release_exact,interpolated_long_gap,True,False,True,True,True,True
4,2016-02-22,STARTS,INE461C01038,GREENPLY,press_release_exact,press_release_exact,False,False,True,True,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...
447,2026-02-23,ENDS,INE373A01013,BASF,interpolated_long_gap,press_release_exact,True,True,True,True,False,False
448,2026-02-23,ENDS,INE348B01021,CENTURYPLY,press_release_exact,press_release_exact,True,True,True,True,False,False
449,2026-02-23,ENDS,INE616N01034,INOXINDIA,interpolated_long_gap,press_release_exact,True,True,True,True,True,False
450,2026-02-23,ENDS,INE09XN01023,AKUMS,interpolated_long_gap,press_release_exact,True,True,True,True,False,False



AUDIT COMPLETE — no files or source data modified.


In [8]:
import pandas as pd
import numpy as np

p = final_df.copy()
iv = intervals_df.copy()

# Normalize dates and identifiers.
p["date"] = pd.to_datetime(p["date"], errors="coerce").dt.normalize()
iv["start_date"] = pd.to_datetime(
    iv["start_date"], errors="coerce"
).dt.normalize()
iv["end_date"] = pd.to_datetime(
    iv["end_date"], errors="coerce"
).dt.normalize()

for df in [p, iv]:
    df["isin"] = df["isin"].astype(str).str.strip()
    df["symbol"] = df["symbol"].astype(str).str.strip()

p["_flag"] = (
    p["is_nifty500_constituent"].astype(str).str.strip().str.lower()
    .isin(["true", "1", "yes", "y"])
)

tri = pd.read_csv("/content/nifty500_pit_data/nifty500_tri.csv")
tri["Date"] = pd.to_datetime(
    tri["Date"].astype(str).str.strip(),
    format="%Y-%m-%d",
    errors="coerce"
).dt.normalize()

calendar = pd.DatetimeIndex(
    tri["Date"].dropna().unique()
).sort_values()
calendar_set = set(calendar)

spike_dates = [
    "2016-02-22",
    "2016-12-28",
    "2019-10-29",
    "2022-05-02",
    "2025-02-21",
    "2026-02-23",
]

def day_records(d):
    return p[p["date"].eq(d)]

def flagged_isins(d):
    g = day_records(d)
    return set(g.loc[g["_flag"], "isin"])

def price_isins(d):
    return set(day_records(d)["isin"])

def count_flags_for_isins(d, isin_set):
    g = day_records(d)
    return g.loc[g["isin"].isin(isin_set) & g["_flag"], "isin"].nunique()

summary = []
details = []

for date_str in spike_dates:
    d = pd.Timestamp(date_str)
    if d not in calendar_set:
        continue

    i = calendar.get_loc(d)
    prev_d = calendar[i - 1] if i > 0 else None
    next_d = calendar[i + 1] if i < len(calendar) - 1 else None

    starts = iv[iv["start_date"].eq(d)].copy()
    ends = iv[iv["end_date"].eq(d)].copy()

    prev_flags = flagged_isins(prev_d) if prev_d is not None else set()
    event_flags = flagged_isins(d)
    next_flags = flagged_isins(next_d) if next_d is not None else set()

    event_added = event_flags - prev_flags
    event_removed = prev_flags - event_flags

    start_isins = set(starts["isin"])
    end_isins = set(ends["isin"])

    summary.append({
        "Date": date_str,
        "Previous_Session": prev_d.date() if prev_d is not None else None,
        "Next_Session": next_d.date() if next_d is not None else None,
        "Intervals_Starting": len(starts),
        "Intervals_Ending": len(ends),
        "Starts_Present_As_Flagged_On_Event": len(event_flags & start_isins),
        "Ends_Still_Flagged_On_Event": len(event_flags & end_isins),
        "Ends_Flagged_Next_Session": (
            len(next_flags & end_isins) if next_d is not None else np.nan
        ),
        "Flagged_Added_Vs_Previous": len(event_added),
        "Added_Matching_Interval_Start": len(event_added & start_isins),
        "Flagged_Removed_Vs_Previous": len(event_removed),
        "Removed_Matching_Interval_End": len(event_removed & end_isins),
        "Previous_Flag_Count": len(prev_flags),
        "Event_Flag_Count": len(event_flags),
        "Next_Flag_Count": len(next_flags),
        "Previous_Price_ISINs": len(price_isins(prev_d)) if prev_d is not None else np.nan,
        "Event_Price_ISINs": len(price_isins(d)),
        "Next_Price_ISINs": len(price_isins(next_d)) if next_d is not None else np.nan,
    })

    for boundary_type, group in [("STARTS", starts), ("ENDS", ends)]:
        for _, row in group.iterrows():
            isin = row["isin"]
            records = {
                "Date": date_str,
                "Boundary_Type": boundary_type,
                "ISIN": isin,
                "Symbol": row["symbol"],
                "Start_Confidence": row.get("start_confidence"),
                "End_Confidence": row.get("end_confidence"),
                "Previous_Price_Row": (
                    isin in price_isins(prev_d) if prev_d is not None else False
                ),
                "Previous_Flag": (
                    isin in prev_flags if prev_d is not None else False
                ),
                "Event_Price_Row": isin in price_isins(d),
                "Event_Flag": isin in event_flags,
                "Next_Price_Row": (
                    isin in price_isins(next_d) if next_d is not None else False
                ),
                "Next_Flag": isin in next_flags,
            }
            details.append(records)

summary_df = pd.DataFrame(summary)
details_df = pd.DataFrame(details)

print("=" * 70)
print("1. SPIKE-DATE SUMMARY")
print("=" * 70)
display(summary_df)

print("\n" + "=" * 70)
print("2. INTERVAL START/END RECORDS")
print("=" * 70)
display(details_df)

print("\nAUDIT COMPLETE — no files or source data modified.")

1. SPIKE-DATE SUMMARY


,Date,Previous_Session,Next_Session,Intervals_Starting,Intervals_Ending,Starts_Present_As_Flagged_On_Event,Ends_Still_Flagged_On_Event,Ends_Flagged_Next_Session,Flagged_Added_Vs_Previous,Added_Matching_Interval_Start,Flagged_Removed_Vs_Previous,Removed_Matching_Interval_End,Previous_Flag_Count,Event_Flag_Count,Next_Flag_Count,Previous_Price_ISINs,Event_Price_ISINs,Next_Price_ISINs
0,2016-02-22,2016-02-19,2016-02-23,21,16,20,13,0,20,20,3,3,502,519,506,507,523,513
1,2016-12-28,2016-12-27,2016-12-29,59,60,31,40,0,31,31,17,16,512,526,485,520,532,499
2,2019-10-29,2019-10-27,2019-10-30,49,36,42,25,0,62,42,0,0,454,516,491,483,532,514
3,2022-05-02,2022-04-29,2022-05-04,65,54,58,46,0,58,58,21,7,485,522,490,508,539,518
4,2025-02-21,2025-02-20,2025-02-24,28,26,28,26,0,28,28,0,0,497,525,499,510,535,513
5,2026-02-23,2026-02-20,2026-02-24,20,18,20,18,0,20,20,0,0,499,519,501,508,525,510



2. INTERVAL START/END RECORDS


,Date,Boundary_Type,ISIN,Symbol,Start_Confidence,End_Confidence,Previous_Price_Row,Previous_Flag,Event_Price_Row,Event_Flag,Next_Price_Row,Next_Flag
0,2016-02-22,STARTS,INE844O01022,GUJGASLTD,press_release_exact,isin_migration_to_INE844O01030,False,False,False,False,False,False
1,2016-02-22,STARTS,INE650K01021,8KMILES,press_release_exact,press_release_exact,False,False,True,True,True,True
2,2016-02-22,STARTS,INE483B01026,ICIL,press_release_exact,press_release_exact,False,False,True,True,True,True
3,2016-02-22,STARTS,INE802G01018,JETAIRWAYS,press_release_exact,interpolated_long_gap,True,False,True,True,True,True
4,2016-02-22,STARTS,INE461C01038,GREENPLY,press_release_exact,press_release_exact,False,False,True,True,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...
447,2026-02-23,ENDS,INE373A01013,BASF,interpolated_long_gap,press_release_exact,True,True,True,True,False,False
448,2026-02-23,ENDS,INE348B01021,CENTURYPLY,press_release_exact,press_release_exact,True,True,True,True,False,False
449,2026-02-23,ENDS,INE616N01034,INOXINDIA,interpolated_long_gap,press_release_exact,True,True,True,True,True,False
450,2026-02-23,ENDS,INE09XN01023,AKUMS,interpolated_long_gap,press_release_exact,True,True,True,True,False,False



AUDIT COMPLETE — no files or source data modified.


In [9]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("/content/nifty500_pit_data")
iv = intervals_df.copy()

iv["start_date"] = pd.to_datetime(
    iv["start_date"], errors="coerce"
).dt.normalize()
iv["end_date"] = pd.to_datetime(
    iv["end_date"], errors="coerce"
).dt.normalize()
iv["symbol"] = iv["symbol"].astype(str).str.strip()

ANNOUNCEMENT = pd.Timestamp("2026-02-23")
LAST_OLD_SESSION = pd.Timestamp("2026-03-27")
FIRST_NEW_SESSION = pd.Timestamp("2026-03-30")

# Official Nifty 500 exclusion list from the 23 Feb 2026 release.
EXITS = {
    "AKUMS", "APLLTD", "ALKYLAMINE", "ALOKINDS", "ASTRAZEN",
    "BASF", "CAMPUS", "CENTURYPLY", "CERA", "AGARWALEYE",
    "FINPIPE", "GODREJAGRO", "GUJGASLTD", "HAPPSTMNDS",
    "INOXINDIA", "JYOTHYLAB", "KIRLOSBROS", "KSB",
    "MAHSCOOTER", "MAHSEAMLES", "METROPOLIS", "PRAJIND",
    "PGHH", "RCF", "RELINFRA", "SUNDRMFAST", "TRIVENI",
    "DBREALTY", "MANYAVAR", "VENTIVE", "VGUARD"
}

# Official Nifty 500 inclusion list from the same release.
ENTRIES = {
    "ACUTAAS", "CPPLUS", "ABDL", "ANTHEM", "ANURAS",
    "BELRISE", "GROWW", "CANHLIFE", "CARTRADE", "CEMPRO",
    "EMMVEE", "GABRIEL", "GALLANTT", "HDBFS", "ICICIAMC",
    "JAINREC", "LENSKART", "LGEINDIA", "MEESHO", "PARADEEP",
    "PWL", "PINELABS", "PIRAMALFIN", "SPLPETRO", "TATACAP",
    "TMCV", "TEGA", "TENNIND", "TRAVELFOOD", "URBANCO",
    "ZYDUSWELL"
}

print("Official list sizes:", len(EXITS), "exits,", len(ENTRIES), "entries")

rows = []

for role, symbols in [("EXIT", EXITS), ("ENTRY", ENTRIES)]:
    for symbol in sorted(symbols):
        g = iv[iv["symbol"].eq(symbol)].copy()

        if g.empty:
            rows.append({
                "Role": role,
                "Symbol": symbol,
                "Found": False,
                "ISIN": "",
                "Start_Date": pd.NaT,
                "Start_Confidence": "",
                "End_Date": pd.NaT,
                "End_Confidence": "",
                "Active_Feb23": False,
                "Active_Mar27": False,
                "Active_Mar30": False,
                "Boundary_On_Announcement": False,
            })
            continue

        for _, r in g.iterrows():
            def active_on(d):
                return (
                    pd.notna(r["start_date"])
                    and r["start_date"] <= d
                    and (pd.isna(r["end_date"]) or r["end_date"] >= d)
                )

            rows.append({
                "Role": role,
                "Symbol": symbol,
                "Found": True,
                "ISIN": r.get("isin", ""),
                "Start_Date": r["start_date"],
                "Start_Confidence": r.get("start_confidence", ""),
                "End_Date": r["end_date"],
                "End_Confidence": r.get("end_confidence", ""),
                "Active_Feb23": active_on(ANNOUNCEMENT),
                "Active_Mar27": active_on(LAST_OLD_SESSION),
                "Active_Mar30": active_on(FIRST_NEW_SESSION),
                "Boundary_On_Announcement": (
                    r["start_date"] == ANNOUNCEMENT
                    or r["end_date"] == ANNOUNCEMENT
                ),
            })

report = pd.DataFrame(rows)

print("\n" + "=" * 70)
print("1. REVIEW SUMMARY")
print("=" * 70)

summary = report.groupby("Role").agg(
    Official_Symbols=("Symbol", "nunique"),
    Found_Symbols=("Found", "sum"),
    Records_With_Feb23_Boundary=("Boundary_On_Announcement", "sum"),
    Active_On_Mar27=("Active_Mar27", "sum"),
    Active_On_Mar30=("Active_Mar30", "sum"),
).reset_index()

display(summary)

print("\n" + "=" * 70)
print("2. COMPANY-LEVEL INTERVAL REVIEW")
print("=" * 70)

display(
    report.sort_values(["Role", "Symbol"])[[
        "Role", "Symbol", "Found", "ISIN",
        "Start_Date", "Start_Confidence",
        "End_Date", "End_Confidence",
        "Active_Feb23", "Active_Mar27", "Active_Mar30",
        "Boundary_On_Announcement"
    ]]
)

out_path = DATA_DIR / "official_review_crosscheck_2026-03-30.csv"
report.to_csv(out_path, index=False)

print("\nAudit saved to:", out_path)
print("No intervals or price records were modified.")

Official list sizes: 31 exits, 31 entries

1. REVIEW SUMMARY


,Role,Official_Symbols,Found_Symbols,Records_With_Feb23_Boundary,Active_On_Mar27,Active_On_Mar30
0,ENTRY,31,34,20,31,31
1,EXIT,31,51,18,0,0



2. COMPANY-LEVEL INTERVAL REVIEW


,Role,Symbol,Found,ISIN,Start_Date,Start_Confidence,End_Date,End_Confidence,Active_Feb23,Active_Mar27,Active_Mar30,Boundary_On_Announcement
51,ENTRY,ABDL,True,INE552Z01027,2026-02-23,press_release_exact,NaT,still_active,True,True,True,True
52,ENTRY,ACUTAAS,True,INE00FF01025,2026-02-23,press_release_exact,NaT,still_active,True,True,True,True
53,ENTRY,ANTHEM,True,INE0CZ201020,2026-02-23,press_release_exact,NaT,still_active,True,True,True,True
54,ENTRY,ANURAS,True,INE930P01018,2021-09-15,press_release_exact,2024-08-23,press_release_exact,False,False,False,False
55,ENTRY,ANURAS,True,INE930P01018,2026-02-23,press_release_exact,NaT,still_active,True,True,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...
46,EXIT,SUNDRMFAST,True,INE387A01021,2010-01-02,left_censored,2026-03-05,interpolated_snapshot_adjacent,True,False,False,False
47,EXIT,TRIVENI,True,INE256C01024,2010-01-02,left_censored,2012-01-12,interpolated_long_gap,False,False,False,False
48,EXIT,TRIVENI,True,INE256C01024,2022-05-02,interpolated_long_gap,2026-03-05,interpolated_snapshot_adjacent,True,False,False,False
49,EXIT,VENTIVE,True,INE781S01027,2025-10-29,interpolated_snapshot_adjacent,2026-03-05,interpolated_snapshot_adjacent,True,False,False,False



Audit saved to: /content/nifty500_pit_data/official_review_crosscheck_2026-03-30.csv
No intervals or price records were modified.


In [10]:
import os
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("/content/nifty500_pit_data")
ANNOUNCEMENT_DATE = pd.Timestamp("2026-02-23")
LAST_OLD_SESSION = pd.Timestamp("2026-03-27")
FIRST_NEW_SESSION = pd.Timestamp("2026-03-30")

# Official Nifty 500 exclusions in the NSE Indices release.
EXITS = {
    "AKUMS", "APLLTD", "ALKYLAMINE", "ALOKINDS", "ASTRAZEN",
    "BASF", "CAMPUS", "CENTURYPLY", "CERA", "AGARWALEYE",
    "FINPIPE", "GODREJAGRO", "GUJGASLTD", "HAPPSTMNDS",
    "INOXINDIA", "JYOTHYLAB", "KIRLOSBROS", "KSB",
    "MAHSCOOTER", "MAHSEAMLES", "METROPOLIS", "PRAJIND",
    "PGHH", "RCF", "RELINFRA", "SUNDRMFAST", "TRIVENI",
    "DBREALTY", "MANYAVAR", "VENTIVE", "VGUARD"
}

# Official Nifty 500 inclusions in the same release.
ENTRIES = {
    "ACUTAAS", "CPPLUS", "ABDL", "ANTHEM", "ANURAS",
    "BELRISE", "GROWW", "CANHLIFE", "CARTRADE", "CEMPRO",
    "EMMVEE", "GABRIEL", "GALLANTT", "HDBFS", "ICICIAMC",
    "JAINREC", "LENSKART", "LGEINDIA", "MEESHO", "PARADEEP",
    "PWL", "PINELABS", "PIRAMALFIN", "SPLPETRO", "TATACAP",
    "TMCV", "TEGA", "TENNIND", "TRAVELFOOD", "URBANCO",
    "ZYDUSWELL"
}

assert len(EXITS) == 31
assert len(ENTRIES) == 31

# Work on copies only. Original objects and files remain untouched.
iv = intervals_df.copy()
p = final_df.copy()

iv["start_date"] = pd.to_datetime(
    iv["start_date"], errors="coerce"
).dt.normalize()
iv["end_date"] = pd.to_datetime(
    iv["end_date"], errors="coerce"
).dt.normalize()

p["date"] = pd.to_datetime(
    p["date"], errors="coerce"
).dt.normalize()
p["isin"] = p["isin"].astype(str).str.strip()
p["symbol"] = p["symbol"].astype(str).str.strip()

iv["isin"] = iv["isin"].astype(str).str.strip()
iv["symbol"] = iv["symbol"].astype(str).str.strip()

# Preserve row identity for this audit.
iv["_row_id"] = np.arange(len(iv))

proposed = []
unresolved = []

def select_one_interval(symbol, role):
    """Select the relevant interval without guessing when ambiguous."""
    rows = iv[iv["symbol"].eq(symbol)].copy()

    if role == "EXIT":
        # The outgoing constituent must have been a member on the
        # announcement date. End dates are inclusive in this dataset.
        candidates = rows[
            rows["start_date"].notna()
            & (rows["start_date"] <= ANNOUNCEMENT_DATE)
            & (
                rows["end_date"].isna()
                | (rows["end_date"] >= ANNOUNCEMENT_DATE)
            )
        ]
    else:
        # The incoming constituent's current interval must span the
        # official effective date under the current reconstruction.
        candidates = rows[
            rows["start_date"].notna()
            & (rows["start_date"] <= FIRST_NEW_SESSION)
            & (
                rows["end_date"].isna()
                | (rows["end_date"] >= FIRST_NEW_SESSION)
            )
        ]

    if candidates.empty:
        unresolved.append({
            "Role": role,
            "Symbol": symbol,
            "Reason": "No interval matching expected event period"
        })
        return None

    # Historical episodes may repeat for a symbol. Select the most
    # recent eligible interval, and require a unique row at that start.
    latest_start = candidates["start_date"].max()
    latest = candidates[candidates["start_date"].eq(latest_start)]

    if len(latest) != 1:
        unresolved.append({
            "Role": role,
            "Symbol": symbol,
            "Reason": f"{len(latest)} intervals tied at latest start date"
        })
        return None

    row = latest.iloc[0]

    proposed.append({
        "Role": role,
        "Symbol": symbol,
        "ISIN": row["isin"],
        "Row_ID": int(row["_row_id"]),
        "Old_Start_Date": row["start_date"],
        "Old_End_Date": row["end_date"],
        "New_Start_Date": (
            row["start_date"] if role == "EXIT"
            else FIRST_NEW_SESSION
        ),
        "New_End_Date": (
            LAST_OLD_SESSION if role == "EXIT"
            else row["end_date"]
        ),
    })
    return int(row["_row_id"])

selected_ids = {}

for symbol in sorted(EXITS):
    row_id = select_one_interval(symbol, "EXIT")
    if row_id is not None:
        selected_ids[row_id] = "EXIT"

for symbol in sorted(ENTRIES):
    row_id = select_one_interval(symbol, "ENTRY")
    if row_id is not None:
        selected_ids[row_id] = "ENTRY"

print("=" * 68)
print("1. PREFLIGHT CHECK")
print("=" * 68)

proposal_df = pd.DataFrame(proposed)
unresolved_df = pd.DataFrame(
    unresolved, columns=["Role", "Symbol", "Reason"]
)

print("Selected exit intervals:",
      int((proposal_df["Role"] == "EXIT").sum()) if len(proposal_df) else 0)
print("Selected entry intervals:",
      int((proposal_df["Role"] == "ENTRY").sum()) if len(proposal_df) else 0)
print("Unresolved symbols:", len(unresolved_df))

if len(unresolved_df):
    display(unresolved_df)
    raise RuntimeError(
        "Correction stopped safely. No files were written. "
        "Review unresolved symbols before proceeding."
    )

# Ensure every official symbol has exactly one selected interval.
assert proposal_df.loc[
    proposal_df["Role"].eq("EXIT"), "Symbol"
].nunique() == 31

assert proposal_df.loc[
    proposal_df["Role"].eq("ENTRY"), "Symbol"
].nunique() == 31

print("\nProposed date changes:")
display(proposal_df.sort_values(["Role", "Symbol"]))

# ---------------------------------------------------------
# Apply the proposed interval corrections to the copy.
# ---------------------------------------------------------
iv_corrected = iv.drop(columns="_row_id").copy()

for _, change in proposal_df.iterrows():
    row_id = int(change["Row_ID"])
    role = change["Role"]

    iv_corrected.loc[row_id, "start_date"] = change["New_Start_Date"]
    iv_corrected.loc[row_id, "end_date"] = change["New_End_Date"]

    if role == "EXIT":
        iv_corrected.loc[row_id, "end_confidence"] = "press_release_exact"
    else:
        iv_corrected.loc[row_id, "start_confidence"] = "press_release_exact"

# Validate that no corrected interval has end before start.
invalid = iv_corrected[
    iv_corrected["start_date"].notna()
    & iv_corrected["end_date"].notna()
    & (iv_corrected["end_date"] < iv_corrected["start_date"])
]

if len(invalid):
    display(invalid)
    raise RuntimeError("Invalid intervals detected; no files were written.")

# ---------------------------------------------------------
# Recalculate membership flags for the affected ISINs only.
# This uses the corrected interval dates while preserving other
# companies' existing flags and all original price columns.
# ---------------------------------------------------------
affected_isins = set(proposal_df["ISIN"].astype(str))

p_corrected = p.copy()

for isin in affected_isins:
    row_mask = p_corrected["isin"].eq(isin)
    price_dates = p_corrected.loc[row_mask, "date"]

    active = pd.Series(False, index=price_dates.index)

    is_in = iv_corrected[iv_corrected["isin"].astype(str).eq(isin)]

    for _, interval in is_in.iterrows():
        start = interval["start_date"]
        end = interval["end_date"]

        if pd.isna(start):
            continue

        interval_active = price_dates.ge(start)

        if pd.notna(end):
            interval_active &= price_dates.le(end)

        active |= interval_active

    p_corrected.loc[row_mask, "is_nifty500_constituent"] = active.values

# Post-correction event checks.
print("\n" + "=" * 68)
print("2. POST-CORRECTION VALIDATION")
print("=" * 68)

check_rows = []

for role, symbols in [("EXIT", EXITS), ("ENTRY", ENTRIES)]:
    selected = proposal_df[proposal_df["Role"].eq(role)]

    for _, r in selected.iterrows():
        check_rows.append({
            "Role": role,
            "Symbol": r["Symbol"],
            "Corrected_Start": iv_corrected.loc[
                r["Row_ID"], "start_date"
            ],
            "Corrected_End": iv_corrected.loc[
                r["Row_ID"], "end_date"
            ],
        })

postcheck = pd.DataFrame(check_rows)
display(postcheck)

# Save new versioned artifacts. Do not overwrite any earlier files.
interval_out = DATA_DIR / "pit_intervals_official_2026-03-30_v1.csv"
price_out = DATA_DIR / "nifty500_ohlcv_pit_quality_official_2026-03-30_v2.parquet"
audit_out = DATA_DIR / "official_boundary_corrections_2026-03-30.csv"

iv_corrected.to_csv(interval_out, index=False)
p_corrected.to_parquet(price_out, index=False)
proposal_df.to_csv(audit_out, index=False)

# Update in-memory objects only after successful validation and saves.
intervals_df = iv_corrected
final_df = p_corrected

print("\nCorrection complete.")
print("Intervals:", interval_out)
print("Quality price dataset:", price_out)
print("Change audit:", audit_out)
print("Source price values were not altered.")

1. PREFLIGHT CHECK
Selected exit intervals: 31
Selected entry intervals: 31
Unresolved symbols: 0

Proposed date changes:


,Role,Symbol,ISIN,Row_ID,Old_Start_Date,Old_End_Date,New_Start_Date,New_End_Date
31,ENTRY,ABDL,INE552Z01027,1251,2026-02-23,NaT,2026-03-30,NaT
32,ENTRY,ACUTAAS,INE00FF01025,1243,2026-02-23,NaT,2026-03-30,NaT
33,ENTRY,ANTHEM,INE0CZ201020,1258,2026-02-23,NaT,2026-03-30,NaT
34,ENTRY,ANURAS,INE930P01018,1235,2026-02-23,NaT,2026-03-30,NaT
35,ENTRY,BELRISE,INE894V01022,1237,2026-02-23,NaT,2026-03-30,NaT
...,...,...,...,...,...,...,...,...
26,EXIT,RELINFRA,INE036A01016,729,2025-09-15,2026-03-05,2025-09-15,2026-03-27
27,EXIT,SUNDRMFAST,INE387A01021,732,2010-01-02,2026-03-05,2010-01-02,2026-03-27
28,EXIT,TRIVENI,INE256C01024,757,2022-05-02,2026-03-05,2022-05-02,2026-03-27
29,EXIT,VENTIVE,INE781S01027,753,2025-10-29,2026-03-05,2025-10-29,2026-03-27



2. POST-CORRECTION VALIDATION


,Role,Symbol,Corrected_Start,Corrected_End
0,EXIT,AGARWALEYE,2025-08-22,2026-03-27
1,EXIT,AKUMS,2024-10-11,2026-03-27
2,EXIT,ALKYLAMINE,2020-06-10,2026-03-27
3,EXIT,ALOKINDS,2023-08-17,2026-03-27
4,EXIT,APLLTD,2014-08-20,2026-03-27
...,...,...,...,...
57,ENTRY,TENNIND,2026-03-30,NaT
58,ENTRY,TMCV,2026-03-30,NaT
59,ENTRY,TRAVELFOOD,2026-03-30,2026-09-29
60,ENTRY,URBANCO,2026-03-30,NaT



Correction complete.
Intervals: /content/nifty500_pit_data/pit_intervals_official_2026-03-30_v1.csv
Quality price dataset: /content/nifty500_pit_data/nifty500_ohlcv_pit_quality_official_2026-03-30_v2.parquet
Change audit: /content/nifty500_pit_data/official_boundary_corrections_2026-03-30.csv
Source price values were not altered.


In [11]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("/content/nifty500_pit_data")

# Load the saved official correction audit.
audit_path = DATA_DIR / "official_boundary_corrections_2026-03-30.csv"
corrections = pd.read_csv(audit_path)

# Use the corrected data currently in memory.
p = final_df.copy()
iv = intervals_df.copy()

p["date"] = pd.to_datetime(p["date"], errors="coerce").dt.normalize()
iv["start_date"] = pd.to_datetime(
    iv["start_date"], errors="coerce"
).dt.normalize()
iv["end_date"] = pd.to_datetime(
    iv["end_date"], errors="coerce"
).dt.normalize()

for df in [p, iv, corrections]:
    if "isin" in df.columns:
        df["isin"] = df["isin"].astype(str).str.strip()
    if "symbol" in df.columns:
        df["symbol"] = df["symbol"].astype(str).str.strip()

p["_flag"] = (
    p["is_nifty500_constituent"].astype(str).str.strip().str.lower()
    .isin(["true", "1", "yes", "y"])
)

last_old_session = pd.Timestamp("2026-03-27")
first_new_session = pd.Timestamp("2026-03-30")

# Check the two critical dates.
dates_to_check = [
    pd.Timestamp("2026-02-23"),
    last_old_session,
    first_new_session,
    pd.Timestamp("2026-03-31"),
]

print("=" * 68)
print("1. DAILY MEMBERSHIP COUNTS")
print("=" * 68)

daily_rows = []

for d in dates_to_check:
    active = iv[
        iv["start_date"].notna()
        & (iv["start_date"] <= d)
        & (iv["end_date"].isna() | (iv["end_date"] >= d))
    ]

    day = p[p["date"].eq(d)]

    daily_rows.append({
        "Date": d.date(),
        "Active_Interval_ISINs": active["isin"].nunique(),
        "Active_Interval_Symbols": active["symbol"].nunique(),
        "Price_ISINs": day["isin"].nunique(),
        "Flagged_ISINs": day.loc[day["_flag"], "isin"].nunique(),
        "Flagged_Symbols": day.loc[day["_flag"], "symbol"].nunique(),
    })

display(pd.DataFrame(daily_rows))

# ---------------------------------------------------------
# 2. Check each official exit and entry at the boundary.
# ---------------------------------------------------------
print("\n" + "=" * 68)
print("2. OFFICIAL SYMBOL-LEVEL CHECK")
print("=" * 68)

checks = []

for _, rec in corrections.iterrows():
    symbol = rec["Symbol"]
    isin = rec["ISIN"]
    role = rec["Role"]

    for d in [last_old_session, first_new_session]:
        interval_rows = iv[iv["isin"].eq(isin)]

        active_interval = False
        for _, row in interval_rows.iterrows():
            start = row["start_date"]
            end = row["end_date"]

            if pd.isna(start):
                continue

            if start <= d and (pd.isna(end) or end >= d):
                active_interval = True
                break

        price_rows = p[
            p["date"].eq(d) & p["isin"].eq(isin)
        ]

        flag_value = (
            bool(price_rows["_flag"].iloc[0])
            if len(price_rows) else None
        )

        # Expected membership for this specific review.
        expected_active = (
            d <= last_old_session if role == "EXIT"
            else d >= first_new_session
        )

        checks.append({
            "Role": role,
            "Symbol": symbol,
            "Date": d.date(),
            "Price_Row_Available": len(price_rows) > 0,
            "Flag_Value": flag_value,
            "Interval_Active": active_interval,
            "Expected_Active": expected_active,
            "Flag_Matches_Interval": (
                flag_value == active_interval
                if flag_value is not None else None
            ),
            "Interval_Matches_Expected": (
                active_interval == expected_active
            ),
        })

checks_df = pd.DataFrame(checks)
display(checks_df)

print("\n" + "=" * 68)
print("3. CHECK SUMMARY")
print("=" * 68)

print("Official symbols checked:", corrections["Symbol"].nunique())
print("Checks performed:", len(checks_df))
print("Missing price rows:", int((~checks_df["Price_Row_Available"]).sum()))
print("Interval/expected mismatches:",
      int((~checks_df["Interval_Matches_Expected"]).sum()))
print("Flag/interval mismatches among available prices:",
      int((checks_df.loc[
          checks_df["Price_Row_Available"], "Flag_Matches_Interval"
      ] == False).sum()))

print("\nNo files modified by this verification.")

1. DAILY MEMBERSHIP COUNTS


,Date,Active_Interval_ISINs,Active_Interval_Symbols,Price_ISINs,Flagged_ISINs,Flagged_Symbols
0,2026-02-23,503,501,525,499,499
1,2026-03-27,506,504,508,471,471
2,2026-03-30,506,504,506,497,497
3,2026-03-31,506,504,2,2,2



2. OFFICIAL SYMBOL-LEVEL CHECK


,Role,Symbol,Date,Price_Row_Available,Flag_Value,Interval_Active,Expected_Active,Flag_Matches_Interval,Interval_Matches_Expected
0,EXIT,AGARWALEYE,2026-03-27,False,None,True,True,None,True
1,EXIT,AGARWALEYE,2026-03-30,False,None,False,False,None,True
2,EXIT,AKUMS,2026-03-27,False,None,True,True,None,True
3,EXIT,AKUMS,2026-03-30,False,None,False,False,None,True
4,EXIT,ALKYLAMINE,2026-03-27,False,None,True,True,None,True
...,...,...,...,...,...,...,...,...,...
119,ENTRY,TRAVELFOOD,2026-03-30,True,True,True,True,True,True
120,ENTRY,URBANCO,2026-03-27,True,False,False,False,True,True
121,ENTRY,URBANCO,2026-03-30,True,True,True,True,True,True
122,ENTRY,ZYDUSWELL,2026-03-27,True,False,False,False,True,True



3. CHECK SUMMARY
Official symbols checked: 62
Checks performed: 124
Missing price rows: 56
Interval/expected mismatches: 0
Flag/interval mismatches among available prices: 0

No files modified by this verification.


In [12]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("/content/nifty500_pit_data")

# Load TRI data
tri = pd.read_csv(DATA_DIR / "nifty500_tri.csv")
tri["Date"] = pd.to_datetime(
    tri["Date"].astype(str).str.strip(),
    format="%Y-%m-%d",
    errors="coerce"
).dt.normalize()

# Use the corrected PIT prices currently in memory
p = final_df.copy()
p["date"] = pd.to_datetime(
    p["date"], errors="coerce"
).dt.normalize()

start = pd.Timestamp("2026-03-25")
end = pd.Timestamp("2026-04-02")

print("=" * 68)
print("1. BENCHMARK VALUES AROUND THE HOLIDAY")
print("=" * 68)

tri_window = tri[tri["Date"].between(start, end)].copy()
tri_window["Weekday"] = tri_window["Date"].dt.day_name()
tri_window["Official_NSE_Holiday"] = tri_window["Date"].eq(
    pd.Timestamp("2026-03-31")
)

display(tri_window)

print("\n" + "=" * 68)
print("2. STOCK-PRICE COVERAGE BY DATE")
print("=" * 68)

daily = (
    p[p["date"].between(start, end)]
    .groupby(["date", "data_source"], dropna=False)
    .agg(
        Rows=("isin", "size"),
        Unique_ISINs=("isin", "nunique"),
        Flagged_Symbols=("symbol", lambda s: p.loc[s.index].loc[
            p.loc[s.index, "is_nifty500_constituent"].astype(str)
            .str.strip().str.lower().isin(["true", "1", "yes", "y"]),
            "symbol"
        ].nunique()),
    )
    .reset_index()
    .sort_values(["date", "data_source"])
)

display(daily)

print("\n" + "=" * 68)
print("3. RECORDS DATED 31 MARCH")
print("=" * 68)

holiday_rows = p[p["date"].eq(pd.Timestamp("2026-03-31"))]

print("Total PIT rows:", len(holiday_rows))

if len(holiday_rows):
    cols = [
        c for c in [
            "date", "symbol", "isin", "open", "high", "low",
            "close", "volume", "data_source", "adjustment",
            "is_nifty500_constituent"
        ] if c in holiday_rows.columns
    ]
    display(holiday_rows[cols])

print("\nAUDIT COMPLETE — no data modified.")

1. BENCHMARK VALUES AROUND THE HOLIDAY


,Date,NIFTY_500_TRI,Weekday,Official_NSE_Holiday
2783,2026-03-25,34327.17,Wednesday,False
2784,2026-03-27,33599.22,Friday,False
2785,2026-03-30,32813.14,Monday,False
2786,2026-04-01,33463.88,Wednesday,False
2787,2026-04-02,33469.45,Thursday,False



2. STOCK-PRICE COVERAGE BY DATE


,date,data_source,Rows,Unique_ISINs,Flagged_Symbols
0,2026-03-25,yfinance,506,506,469
1,2026-03-26,nse_bhavcopy_raw,2,2,2
2,2026-03-27,nse_bhavcopy_raw,2,2,2
3,2026-03-27,yfinance,506,506,469
4,2026-03-30,yfinance,506,506,497
5,2026-03-31,nse_bhavcopy_raw,2,2,2
6,2026-04-01,nse_bhavcopy_raw,2,2,2
7,2026-04-01,yfinance,506,506,497
8,2026-04-02,yfinance,506,506,497



3. RECORDS DATED 31 MARCH
Total PIT rows: 2


,date,symbol,isin,open,high,low,close,volume,data_source,adjustment,is_nifty500_constituent
842993,2026-03-31,GSPL,INE246F01010,239.0,240.46,231.78,234.38,740267,nse_bhavcopy_raw,raw_unadjusted,True
1495488,2026-03-31,JBCHEPHARM,INE572A01036,2071.9,2103.00,1990.60,2008.80,241774,nse_bhavcopy_raw,raw_unadjusted,True



AUDIT COMPLETE — no data modified.


In [13]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("/content/nifty500_pit_data")

# Use corrected data in memory, or load the saved corrected version.
if "final_df" in globals() and isinstance(final_df, pd.DataFrame):
    p = final_df.copy()
else:
    p = pd.read_parquet(
        DATA_DIR / "nifty500_ohlcv_pit_quality_official_2026-03-30_v2.parquet"
    )

corrections = pd.read_csv(
    DATA_DIR / "official_boundary_corrections_2026-03-30.csv"
)

p["date"] = pd.to_datetime(p["date"], errors="coerce").dt.normalize()
p["isin"] = p["isin"].astype(str).str.strip()

# Check only the 31 outgoing companies.
exits = corrections[corrections["Role"].eq("EXIT")].copy()
exits["ISIN"] = exits["ISIN"].astype(str).str.strip()

last_old_session = pd.Timestamp("2026-03-27")
first_new_session = pd.Timestamp("2026-03-30")
old_inferred_end = pd.Timestamp("2026-03-05")

rows = []

for _, rec in exits.iterrows():
    isin = rec["ISIN"]
    symbol = rec["Symbol"]
    g = p[p["isin"].eq(isin)]

    yf = g[g["data_source"].eq("yfinance")]
    raw = g[g["data_source"].eq("nse_bhavcopy_raw")]

    rows.append({
        "Symbol": symbol,
        "ISIN": isin,
        "Yahoo_Rows": len(yf),
        "First_Yahoo_Date": yf["date"].min(),
        "Last_Yahoo_Date": yf["date"].max(),
        "Yahoo_Rows_After_Old_End_Through_Mar27": int(
            (yf["date"].gt(old_inferred_end)
             & yf["date"].le(last_old_session)).sum()
        ),
        "Any_Price_On_Mar27": bool(
            g["date"].eq(last_old_session).any()
        ),
        "Any_Price_On_Mar30": bool(
            g["date"].eq(first_new_session).any()
        ),
        "Raw_Fallback_Rows": len(raw),
    })

coverage = pd.DataFrame(rows)

print("=" * 68)
print("1. EXIT PRICE-HISTORY COVERAGE")
print("=" * 68)
display(
    coverage.sort_values(
        ["Any_Price_On_Mar27", "Last_Yahoo_Date", "Symbol"],
        ascending=[True, True, True]
    )
)

print("\n" + "=" * 68)
print("2. COVERAGE SUMMARY")
print("=" * 68)
print("Outgoing companies:", len(coverage))
print(
    "Last Yahoo price before 27 March:",
    int((coverage["Last_Yahoo_Date"] < last_old_session).sum())
)
print(
    "Yahoo history ending exactly 5 March:",
    int((coverage["Last_Yahoo_Date"] == old_inferred_end).sum())
)
print("No price row on 27 March:", int((~coverage["Any_Price_On_Mar27"]).sum()))
print("No price row on 30 March:", int((~coverage["Any_Price_On_Mar30"]).sum()))

coverage.to_csv(
    DATA_DIR / "march_2026_exit_price_coverage_audit.csv",
    index=False
)
print("\nSaved report: march_2026_exit_price_coverage_audit.csv")
print("No price records or membership flags were modified.")

1. EXIT PRICE-HISTORY COVERAGE


,Symbol,ISIN,Yahoo_Rows,First_Yahoo_Date,Last_Yahoo_Date,Yahoo_Rows_After_Old_End_Through_Mar27,Any_Price_On_Mar27,Any_Price_On_Mar30,Raw_Fallback_Rows
0,AGARWALEYE,INE943P01029,125,2025-08-22,2026-02-23,0,False,False,0
1,AKUMS,INE09XN01023,339,2024-10-11,2026-02-23,0,False,False,0
2,ALKYLAMINE,INE150B01039,1417,2020-06-10,2026-02-23,0,False,False,0
3,ALOKINDS,INE270A01029,623,2023-08-17,2026-02-23,0,False,False,0
4,APLLTD,INE901L01018,2840,2014-08-20,2026-02-23,0,False,False,0
5,ASTRAZEN,INE203A01020,3985,2010-01-04,2026-02-23,0,False,False,0
6,BASF,INE373A01013,3986,2010-01-04,2026-02-23,0,False,False,0
7,CAMPUS,INE278Y01022,896,2022-07-11,2026-02-23,0,False,False,0
8,CENTURYPLY,INE348B01021,3985,2010-01-04,2026-02-23,0,False,False,0
9,CERA,INE739E01017,2471,2016-02-22,2026-02-23,0,False,False,0



2. COVERAGE SUMMARY
Outgoing companies: 31
Last Yahoo price before 27 March: 28
Yahoo history ending exactly 5 March: 13
No price row on 27 March: 28
No price row on 30 March: 28

Saved report: march_2026_exit_price_coverage_audit.csv
No price records or membership flags were modified.


In [14]:
import time
import numpy as np
import pandas as pd
import yfinance as yf
from pathlib import Path

DATA_DIR = Path("/content/nifty500_pit_data")
START_DATE = "2026-02-01"
END_DATE_EXCLUSIVE = "2026-04-03"  # Includes 2 April
LAST_OLD_SESSION = pd.Timestamp("2026-03-27")
FIRST_NEW_SESSION = pd.Timestamp("2026-03-30")

# ---------------------------------------------------------
# 1. Load existing data and correction audit
# ---------------------------------------------------------
p = final_df.copy()
iv = intervals_df.copy()

p["date"] = pd.to_datetime(p["date"], errors="coerce").dt.normalize()
p["isin"] = p["isin"].astype(str).str.strip()
p["symbol"] = p["symbol"].astype(str).str.strip()

iv["start_date"] = pd.to_datetime(
    iv["start_date"], errors="coerce"
).dt.normalize()
iv["end_date"] = pd.to_datetime(
    iv["end_date"], errors="coerce"
).dt.normalize()
iv["isin"] = iv["isin"].astype(str).str.strip()
iv["symbol"] = iv["symbol"].astype(str).str.strip()

corrections_path = DATA_DIR / "official_boundary_corrections_2026-03-30.csv"
if not corrections_path.exists():
    raise FileNotFoundError(
        "The official boundary correction audit is missing. "
        "Restore the latest session files before refreshing prices."
    )

corrections = pd.read_csv(corrections_path)
exits = corrections[
    corrections["Role"].astype(str).str.upper().eq("EXIT")
].copy()
exits["ISIN"] = exits["ISIN"].astype(str).str.strip()
exits["Symbol"] = exits["Symbol"].astype(str).str.strip()

# Target only exits with no price row on 27 March.
priced_on_exit_date = set(
    p.loc[p["date"].eq(LAST_OLD_SESSION), "isin"].astype(str)
)
targets = exits[~exits["ISIN"].isin(priced_on_exit_date)].copy()

print("Official outgoing constituents:", exits["ISIN"].nunique())
print("Missing price histories to refresh:", targets["ISIN"].nunique())

if len(targets) != 28:
    print("Note: expected 28 based on the previous audit; found", len(targets))

# ---------------------------------------------------------
# 2. Find the corrected manifest and map ISIN -> Yahoo ticker
# ---------------------------------------------------------
manifest_matches = list(Path("/content").rglob("pit_manifest_corrected.csv"))

if not manifest_matches:
    raise FileNotFoundError(
        "pit_manifest_corrected.csv was not found. "
        "Please extract the saved project backup before proceeding."
    )

manifest_path = manifest_matches[0]
manifest = pd.read_csv(manifest_path)

# Identify columns without assuming case.
column_lookup = {str(c).strip().lower(): c for c in manifest.columns}

isin_col = column_lookup.get("isin")
ticker_col = column_lookup.get("yfinance_ticker")
symbol_col = (
    column_lookup.get("current_symbol")
    or column_lookup.get("symbol")
)

if not isin_col or not ticker_col:
    print("Manifest columns:", manifest.columns.tolist())
    raise ValueError(
        "Could not identify the ISIN and yfinance_ticker columns safely."
    )

manifest["_isin_key"] = manifest[isin_col].astype(str).str.strip()
manifest["_yf_key"] = manifest[ticker_col].astype(str).str.strip()

ticker_map = {}
unresolved = []

for _, rec in targets.iterrows():
    isin = rec["ISIN"]
    symbol = rec["Symbol"]

    matches = manifest[
        manifest["_isin_key"].eq(isin)
        & manifest["_yf_key"].ne("")
        & manifest["_yf_key"].str.lower().ne("nan")
    ]

    tickers = sorted(matches["_yf_key"].unique())

    if len(tickers) != 1:
        unresolved.append({
            "ISIN": isin,
            "Symbol": symbol,
            "Matching_Yahoo_Tickers": " | ".join(tickers),
        })
        continue

    ticker_map[isin] = tickers[0]

if unresolved:
    print("Unresolved ticker mappings:")
    display(pd.DataFrame(unresolved))
    raise ValueError("Stopped before downloading or saving any changes.")

print("\nManifest:", manifest_path)
print("Ticker mappings verified:", len(ticker_map))
display(
    targets[["Symbol", "ISIN"]].assign(
        Yahoo_Ticker=targets["ISIN"].map(ticker_map)
    ).sort_values("Symbol")
)

# ---------------------------------------------------------
# 3. Download the targeted period, one ticker at a time
# ---------------------------------------------------------
downloaded = {}
download_log = []

for _, rec in targets.iterrows():
    isin = rec["ISIN"]
    symbol = rec["Symbol"]
    ticker = ticker_map[isin]

    try:
        d = yf.download(
            ticker,
            start=START_DATE,
            end=END_DATE_EXCLUSIVE,
            auto_adjust=False,
            actions=True,
            progress=False,
            threads=False,
        )

        if d is None or d.empty:
            download_log.append({
                "Symbol": symbol,
                "ISIN": isin,
                "Yahoo_Ticker": ticker,
                "Status": "EMPTY_DOWNLOAD",
                "Rows": 0,
                "Has_Mar27": False,
                "Has_Mar30": False,
            })
            continue

        # Recent yfinance versions may return MultiIndex columns,
        # even for one ticker.
        if isinstance(d.columns, pd.MultiIndex):
            d.columns = d.columns.get_level_values(0)

        d.columns = [str(c).strip() for c in d.columns]

        idx = pd.to_datetime(d.index)
        if getattr(idx, "tz", None) is not None:
            idx = idx.tz_localize(None)
        d.index = idx.normalize()
        d = d[~d.index.duplicated(keep="last")].sort_index()

        # The regular Close field is used, not dividend-adjusted Adj Close.
        if "Close" not in d.columns:
            download_log.append({
                "Symbol": symbol,
                "ISIN": isin,
                "Yahoo_Ticker": ticker,
                "Status": "NO_CLOSE_COLUMN",
                "Rows": len(d),
                "Has_Mar27": False,
                "Has_Mar30": False,
            })
            continue

        close = pd.to_numeric(d["Close"], errors="coerce")
        d = d.loc[close.notna() & close.gt(0)].copy()

        if d.empty:
            status = "NO_VALID_CLOSES"
        else:
            status = "OK"
            downloaded[isin] = {
                "symbol": symbol,
                "ticker": ticker,
                "data": d,
            }

        download_log.append({
            "Symbol": symbol,
            "ISIN": isin,
            "Yahoo_Ticker": ticker,
            "Status": status,
            "Rows": len(d),
            "First_Date": d.index.min() if len(d) else pd.NaT,
            "Last_Date": d.index.max() if len(d) else pd.NaT,
            "Has_Mar27": LAST_OLD_SESSION in d.index,
            "Has_Mar30": FIRST_NEW_SESSION in d.index,
        })

    except Exception as e:
        download_log.append({
            "Symbol": symbol,
            "ISIN": isin,
            "Yahoo_Ticker": ticker,
            "Status": f"ERROR: {str(e)[:160]}",
            "Rows": 0,
            "Has_Mar27": False,
            "Has_Mar30": False,
        })

    time.sleep(0.15)

download_log_df = pd.DataFrame(download_log)

print("\n" + "=" * 68)
print("1. TARGETED DOWNLOAD RESULTS")
print("=" * 68)
display(download_log_df.sort_values(["Status", "Symbol"]))

print("Successful ticker downloads:", len(downloaded), "/", len(targets))
print("Downloads containing 27 March:", int(download_log_df["Has_Mar27"].sum()))
print("Downloads containing 30 March:", int(download_log_df["Has_Mar30"].sum()))

if not downloaded:
    raise RuntimeError("No ticker downloads succeeded; no dataset was changed.")

# ---------------------------------------------------------
# 4. Build rows using the existing dataset schema
# ---------------------------------------------------------
tri = pd.read_csv(DATA_DIR / "nifty500_tri.csv")
tri["Date"] = pd.to_datetime(
    tri["Date"].astype(str).str.strip(),
    format="%Y-%m-%d",
    errors="coerce"
).dt.normalize()
benchmark_dates = set(tri["Date"].dropna())
benchmark_min = tri["Date"].min()
benchmark_max = tri["Date"].max()

# Prepare active intervals for membership flags.
def interval_active(isin, date):
    rows = iv[iv["isin"].eq(isin)]

    for _, r in rows.iterrows():
        start = r["start_date"]
        end = r["end_date"]

        if pd.isna(start) or start > date:
            continue
        if pd.isna(end) or date <= end:
            return True

    return False

def numeric_value(value):
    try:
        if pd.isna(value):
            return np.nan
        return float(value)
    except (TypeError, ValueError):
        return np.nan

new_rows = []

for isin, info in downloaded.items():
    symbol = info["symbol"]
    d = info["data"]

    for date, r in d.iterrows():
        date = pd.Timestamp(date).normalize()

        vals = {
            "open": numeric_value(r.get("Open", np.nan)),
            "high": numeric_value(r.get("High", np.nan)),
            "low": numeric_value(r.get("Low", np.nan)),
            "close": numeric_value(r.get("Close", np.nan)),
            "volume": numeric_value(r.get("Volume", np.nan)),
        }

        ohlc_values = [
            vals["open"], vals["high"], vals["low"], vals["close"]
        ]
        all_ohlc = all(np.isfinite(v) for v in ohlc_values)
        flat_ohlc = (
            all_ohlc and max(ohlc_values) - min(ohlc_values) <= 1e-6
        )
        zero_volume = vals["volume"] == 0
        suspect = flat_ohlc and zero_volume

        calendar_status = (
            "on_benchmark_calendar" if date in benchmark_dates
            else (
                "off_benchmark_calendar"
                if benchmark_min <= date <= benchmark_max
                else "outside_benchmark_coverage"
            )
        )

        base = {c: np.nan for c in p.columns}

        for col in p.columns:
            lc = str(col).strip().lower()

            if lc == "date":
                base[col] = date
            elif lc == "isin":
                base[col] = isin
            elif lc == "symbol":
                base[col] = symbol
            elif lc in vals:
                base[col] = vals[lc]
            elif lc in {"adj close", "adj_close", "adjusted_close"}:
                base[col] = numeric_value(r.get("Adj Close", np.nan))
            elif lc in {"dividends", "dividend"}:
                base[col] = numeric_value(r.get("Dividends", 0))
            elif lc in {"stock_splits", "stock split", "splits"}:
                base[col] = numeric_value(r.get("Stock Splits", 0))
            elif lc == "data_source":
                base[col] = "yfinance"
            elif lc == "adjustment":
                base[col] = "split_adjusted_no_dividend"
            elif lc == "is_nifty500_constituent":
                base[col] = interval_active(isin, date)
            elif lc == "price_quality_flag":
                base[col] = (
                    "flat_ohlc_zero_volume" if suspect else "not_flagged"
                )
            elif lc == "close_for_analysis":
                base[col] = np.nan if suspect else vals["close"]
            elif lc == "price_candidate_eligible":
                base[col] = (
                    date in benchmark_dates
                    and np.isfinite(vals["close"])
                    and vals["close"] > 0
                    and not suspect
                )
            elif lc == "calendar_status":
                base[col] = calendar_status
            elif lc == "_flat_ohlc":
                base[col] = flat_ohlc
            elif lc == "_zero_volume":
                base[col] = zero_volume

        new_rows.append(base)

refreshed_rows_df = pd.DataFrame(new_rows, columns=p.columns)

# ---------------------------------------------------------
# 5. Replace only the targeted Yahoo records in the requested
#    date window, preserving all other source rows untouched.
# ---------------------------------------------------------
successful_isins = set(downloaded)
window_start = pd.Timestamp(START_DATE)
window_end = pd.Timestamp("2026-04-02")

replace_mask = (
    p["isin"].isin(successful_isins)
    & p["data_source"].eq("yfinance")
    & p["date"].between(window_start, window_end)
)

p_refreshed = pd.concat(
    [p.loc[~replace_mask].copy(), refreshed_rows_df],
    ignore_index=True
)

p_refreshed = p_refreshed.sort_values(
    ["isin", "date", "data_source"]
).reset_index(drop=True)

# Verify there are no duplicate Yahoo rows for an ISIN/date.
yf_rows = p_refreshed[p_refreshed["data_source"].eq("yfinance")]
duplicate_yf = yf_rows.duplicated(["isin", "date"]).sum()

print("\n" + "=" * 68)
print("2. POST-REFRESH VALIDATION")
print("=" * 68)
print("New Yahoo rows added:", len(refreshed_rows_df))
print("Old Yahoo rows replaced in window:", int(replace_mask.sum()))
print("Duplicate Yahoo ISIN/date rows:", int(duplicate_yf))

# Coverage of refreshed target histories.
coverage_rows = []

for _, rec in targets.iterrows():
    isin = rec["ISIN"]
    symbol = rec["Symbol"]
    g = p_refreshed[
        p_refreshed["isin"].eq(isin)
        & p_refreshed["data_source"].eq("yfinance")
    ]

    coverage_rows.append({
        "Symbol": symbol,
        "ISIN": isin,
        "Last_Yahoo_Date": g["date"].max(),
        "Price_On_Mar27": bool(g["date"].eq(LAST_OLD_SESSION).any()),
        "Price_On_Mar30": bool(g["date"].eq(FIRST_NEW_SESSION).any()),
    })

post_coverage = pd.DataFrame(coverage_rows)
display(post_coverage.sort_values(["Price_On_Mar27", "Symbol"]))

# Save separate versioned artifacts; do not overwrite prior versions.
out_data = DATA_DIR / "nifty500_ohlcv_pit_quality_official_2026-03-30_v3.parquet"
out_log = DATA_DIR / "march_2026_targeted_yfinance_refresh_v1.csv"

p_refreshed.to_parquet(out_data, index=False)
download_log_df.to_csv(out_log, index=False)

final_df = p_refreshed

print("\nSaved refreshed dataset:", out_data)
print("Saved download log:", out_log)
print("Original price values outside the targeted window were preserved.")

Official outgoing constituents: 31
Missing price histories to refresh: 28

Manifest: /content/stock_project_restore/nifty500_pit_data/pit_manifest_corrected.csv
Ticker mappings verified: 28


,Symbol,ISIN,Yahoo_Ticker
0,AGARWALEYE,INE943P01029,AGARWALEYE.NS
1,AKUMS,INE09XN01023,AKUMS.NS
2,ALKYLAMINE,INE150B01039,ALKYLAMINE.NS
3,ALOKINDS,INE270A01029,ALOKINDS.NS
4,APLLTD,INE901L01018,APLLTD.NS
5,ASTRAZEN,INE203A01020,ASTRAZEN.NS
6,BASF,INE373A01013,BASF.NS
7,CAMPUS,INE278Y01022,CAMPUS.NS
8,CENTURYPLY,INE348B01021,CENTURYPLY.NS
9,CERA,INE739E01017,CERA.NS



1. TARGETED DOWNLOAD RESULTS


,Symbol,ISIN,Yahoo_Ticker,Status,Rows,First_Date,Last_Date,Has_Mar27,Has_Mar30
0,AGARWALEYE,INE943P01029,AGARWALEYE.NS,OK,41,2026-02-02,2026-04-02,True,True
1,AKUMS,INE09XN01023,AKUMS.NS,OK,41,2026-02-02,2026-04-02,True,True
2,ALKYLAMINE,INE150B01039,ALKYLAMINE.NS,OK,41,2026-02-02,2026-04-02,True,True
3,ALOKINDS,INE270A01029,ALOKINDS.NS,OK,41,2026-02-02,2026-04-02,True,True
4,APLLTD,INE901L01018,APLLTD.NS,OK,41,2026-02-02,2026-04-02,True,True
5,ASTRAZEN,INE203A01020,ASTRAZEN.NS,OK,41,2026-02-02,2026-04-02,True,True
6,BASF,INE373A01013,BASF.NS,OK,41,2026-02-02,2026-04-02,True,True
7,CAMPUS,INE278Y01022,CAMPUS.NS,OK,41,2026-02-02,2026-04-02,True,True
8,CENTURYPLY,INE348B01021,CENTURYPLY.NS,OK,41,2026-02-02,2026-04-02,True,True
9,CERA,INE739E01017,CERA.NS,OK,41,2026-02-02,2026-04-02,True,True


Successful ticker downloads: 28 / 28
Downloads containing 27 March: 28
Downloads containing 30 March: 28

2. POST-REFRESH VALIDATION
New Yahoo rows added: 1148
Old Yahoo rows replaced in window: 539
Duplicate Yahoo ISIN/date rows: 0


,Symbol,ISIN,Last_Yahoo_Date,Price_On_Mar27,Price_On_Mar30
0,AGARWALEYE,INE943P01029,2026-04-02,True,True
1,AKUMS,INE09XN01023,2026-04-02,True,True
2,ALKYLAMINE,INE150B01039,2026-04-02,True,True
3,ALOKINDS,INE270A01029,2026-04-02,True,True
4,APLLTD,INE901L01018,2026-04-02,True,True
5,ASTRAZEN,INE203A01020,2026-04-02,True,True
6,BASF,INE373A01013,2026-04-02,True,True
7,CAMPUS,INE278Y01022,2026-04-02,True,True
8,CENTURYPLY,INE348B01021,2026-04-02,True,True
9,CERA,INE739E01017,2026-04-02,True,True



Saved refreshed dataset: /content/nifty500_pit_data/nifty500_ohlcv_pit_quality_official_2026-03-30_v3.parquet
Saved download log: /content/nifty500_pit_data/march_2026_targeted_yfinance_refresh_v1.csv
Original price values outside the targeted window were preserved.


In [15]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("/content/nifty500_pit_data")

PRICE_PATH = DATA_DIR / (
    "nifty500_ohlcv_pit_quality_official_2026-03-30_v3.parquet"
)
INTERVAL_PATH = DATA_DIR / "pit_intervals_official_2026-03-30_v1.csv"
AUDIT_PATH = DATA_DIR / "official_boundary_corrections_2026-03-30.csv"

p = pd.read_parquet(PRICE_PATH)
iv = pd.read_csv(INTERVAL_PATH)
changes = pd.read_csv(AUDIT_PATH)

p["date"] = pd.to_datetime(p["date"], errors="coerce").dt.normalize()
iv["start_date"] = pd.to_datetime(
    iv["start_date"], errors="coerce"
).dt.normalize()
iv["end_date"] = pd.to_datetime(
    iv["end_date"], errors="coerce"
).dt.normalize()

for df in [p, iv, changes]:
    for col in ["isin", "ISIN", "symbol", "Symbol"]:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip()

for col in ["open", "high", "low", "close", "volume"]:
    p[col] = pd.to_numeric(p[col], errors="coerce")

def to_bool(s):
    if pd.api.types.is_bool_dtype(s):
        return s.fillna(False)
    return s.astype(str).str.strip().str.lower().isin(
        ["true", "1", "yes", "y"]
    )

p["_flag"] = to_bool(p["is_nifty500_constituent"])
p["_candidate"] = to_bool(p["price_candidate_eligible"])

# ---------------------------------------------------------
# 1. Check price coverage on the two important sessions
# ---------------------------------------------------------
print("=" * 68)
print("1. DAILY COVERAGE")
print("=" * 68)

for date_str in ["2026-03-27", "2026-03-30", "2026-03-31"]:
    d = pd.Timestamp(date_str)
    day = p[p["date"].eq(d)]

    print(f"\n{date_str}: {len(day)} total PIT rows")
    if len(day):
        display(
            day.groupby(["data_source", "adjustment"], dropna=False)
            .agg(
                Rows=("isin", "size"),
                Unique_ISINs=("isin", "nunique"),
                Flagged_ISINs=("_flag", "sum"),
                Candidate_Rows=("_candidate", "sum"),
            )
            .reset_index()
        )

# ---------------------------------------------------------
# 2. Check all official exits and entries on both dates
# ---------------------------------------------------------
print("\n" + "=" * 68)
print("2. OFFICIAL MEMBERSHIP CHECK")
print("=" * 68)

check_rows = []

for _, rec in changes.iterrows():
    role = str(rec["Role"]).upper()
    symbol = rec["Symbol"]
    isin = rec["ISIN"]

    for d in [pd.Timestamp("2026-03-27"),
              pd.Timestamp("2026-03-30")]:

        expected = (
            d == pd.Timestamp("2026-03-27")
            if role == "EXIT"
            else d == pd.Timestamp("2026-03-30")
        )

        g = p[
            p["date"].eq(d)
            & p["isin"].eq(isin)
            & p["data_source"].eq("yfinance")
        ]

        rows_available = len(g) > 0
        actual_flag = bool(g["_flag"].iloc[0]) if rows_available else None
        candidate = bool(g["_candidate"].iloc[0]) if rows_available else False

        interval_rows = iv[iv["isin"].eq(isin)]
        active_interval = False

        for _, r in interval_rows.iterrows():
            start = r["start_date"]
            end = r["end_date"]

            if pd.notna(start) and start <= d and (
                pd.isna(end) or d <= end
            ):
                active_interval = True
                break

        if rows_available:
            r = g.iloc[0]
            o, h, l, c, v = (
                r["open"], r["high"], r["low"],
                r["close"], r["volume"]
            )

            numeric_prices_valid = (
                np.isfinite([o, h, l, c]).all()
                and l <= h
                and l - 1e-6 <= o <= h + 1e-6
                and l - 1e-6 <= c <= h + 1e-6
                and c > 0
            )

            positive_volume = pd.notna(v) and v > 0

            quality_flag = str(r.get("price_quality_flag", ""))
        else:
            numeric_prices_valid = False
            positive_volume = False
            quality_flag = "NO_PRICE_ROW"

        check_rows.append({
            "Role": role,
            "Symbol": symbol,
            "Date": d.date(),
            "Price_Row_Available": rows_available,
            "Actual_Flag": actual_flag,
            "Expected_Flag": expected,
            "Flag_Matches_Expected": (
                actual_flag == expected if rows_available else False
            ),
            "Interval_Matches_Expected": active_interval == expected,
            "Candidate_Eligible": candidate,
            "OHLC_Valid": numeric_prices_valid,
            "Positive_Volume": positive_volume,
            "Quality_Flag": quality_flag,
        })

checks = pd.DataFrame(check_rows)
display(checks)

# ---------------------------------------------------------
# 3. Summary and duplicate check
# ---------------------------------------------------------
print("\n" + "=" * 68)
print("3. QA SUMMARY")
print("=" * 68)

print("Official symbol rows checked:", changes["Symbol"].nunique())
print("Total symbol/date checks:", len(checks))
print("Missing Yahoo price rows:", int((~checks["Price_Row_Available"]).sum()))
print("Flag mismatches:", int((~checks["Flag_Matches_Expected"]).sum()))
print("Interval mismatches:", int((~checks["Interval_Matches_Expected"]).sum()))
print("Invalid OHLC rows:", int((~checks["OHLC_Valid"]).sum()))
print("Non-positive-volume rows:", int((~checks["Positive_Volume"]).sum()))
print("Rows not candidate-eligible:", int((~checks["Candidate_Eligible"]).sum()))

yf = p[p["data_source"].eq("yfinance")]
print(
    "Duplicate Yahoo ISIN/date pairs:",
    int(yf.duplicated(["isin", "date"]).sum())
)

# Save a separate audit report.
out_path = DATA_DIR / "march_2026_refresh_QA_v1.csv"
checks.to_csv(out_path, index=False)
print("\nQA report saved:", out_path)
print("No source prices or membership intervals modified.")

1. DAILY COVERAGE

2026-03-27: 536 total PIT rows


,data_source,adjustment,Rows,Unique_ISINs,Flagged_ISINs,Candidate_Rows
0,nse_bhavcopy_raw,raw_unadjusted,2,2,2,0
1,yfinance,split_adjusted_no_dividend,534,534,497,534



2026-03-30: 534 total PIT rows


,data_source,adjustment,Rows,Unique_ISINs,Flagged_ISINs,Candidate_Rows
0,yfinance,split_adjusted_no_dividend,534,534,497,534



2026-03-31: 2 total PIT rows


,data_source,adjustment,Rows,Unique_ISINs,Flagged_ISINs,Candidate_Rows
0,nse_bhavcopy_raw,raw_unadjusted,2,2,2,0



2. OFFICIAL MEMBERSHIP CHECK


,Role,Symbol,Date,Price_Row_Available,Actual_Flag,Expected_Flag,Flag_Matches_Expected,Interval_Matches_Expected,Candidate_Eligible,OHLC_Valid,Positive_Volume,Quality_Flag
0,EXIT,AGARWALEYE,2026-03-27,True,True,True,True,True,True,True,True,not_flagged
1,EXIT,AGARWALEYE,2026-03-30,True,False,False,True,True,True,True,True,not_flagged
2,EXIT,AKUMS,2026-03-27,True,True,True,True,True,True,True,True,not_flagged
3,EXIT,AKUMS,2026-03-30,True,False,False,True,True,True,True,True,not_flagged
4,EXIT,ALKYLAMINE,2026-03-27,True,True,True,True,True,True,True,True,not_flagged
...,...,...,...,...,...,...,...,...,...,...,...,...
119,ENTRY,TRAVELFOOD,2026-03-30,True,True,True,True,True,True,True,True,not_flagged
120,ENTRY,URBANCO,2026-03-27,True,False,False,True,True,True,True,True,not_flagged
121,ENTRY,URBANCO,2026-03-30,True,True,True,True,True,True,True,True,not_flagged
122,ENTRY,ZYDUSWELL,2026-03-27,True,False,False,True,True,True,True,True,not_flagged



3. QA SUMMARY
Official symbol rows checked: 62
Total symbol/date checks: 124
Missing Yahoo price rows: 0
Flag mismatches: 0
Interval mismatches: 0
Invalid OHLC rows: 0
Non-positive-volume rows: 0
Rows not candidate-eligible: 0
Duplicate Yahoo ISIN/date pairs: 0

QA report saved: /content/nifty500_pit_data/march_2026_refresh_QA_v1.csv
No source prices or membership intervals modified.


In [16]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("/content/nifty500_pit_data")

ANNOUNCEMENT = pd.Timestamp("2016-02-22")
LAST_OLD_SESSION = pd.Timestamp("2016-03-31")
FIRST_NEW_SESSION = pd.Timestamp("2016-04-01")

# Official Nifty 500 exclusions in the 22 February 2016 release.
EXITS = {
    "ADLABS", "ATFL", "ASAHIINDIA", "AUTOAXLES", "CAMLINFINE",
    "CARBORUNIV", "CENTENKA", "DHANBANK", "ENIL", "ESABINDIA",
    "FMGOETZE", "FLEXITUFF", "GABRIEL", "GAMMNINFRA",
    "GEOJITBNPP", "GEOMETRIC", "GHCL", "GITANJALI", "GRAPHITE",
    "GTLINFRA", "GUJALKALI", "GIPCL", "GNFC", "GULFOILLUB",
    "HEG", "HERITGFOOD", "ILFSENGG", "IBVENTURES", "ITDCEM",
    "IVRCLINFRA", "JYOTISTRUC", "KCP", "KKCL", "KNRCON",
    "KSBPUMPS", "LGBBROSLTD", "LINDEINDIA", "MAHSCOOTER",
    "MAHSEAMLES", "MBLINFRA", "MERCATOR", "METALFORGE",
    "MTEDUCARE", "MUNJALSHOW", "NBVENTURES", "NAVNETEDUL",
    "NDTV", "NIITLTD", "NOCIL", "NOIDATOLL", "OPTOCIRCUI",
    "BINDALAGRO", "PATELENG", "PENINLAND", "PENIND", "PRAKASH",
    "RELIGARE", "RICOAUTO", "SEINV", "SHANTIGEAR", "SHRENUJ",
    "SONASTEER", "SRIPIPES", "SBT", "SUPREMEINF", "SUPPETRO",
    "SWARAJENG", "TATAINVEST", "TDPOWERSYS", "TVTODAY",
    "USHAMART", "UTTAMSTL", "VESUVIUS", "VIVIDHA", "WHEELS"
}

# Official Nifty 500 inclusions in the same release.
ENTRIES = {
    "8KMILES", "AARTIDRUGS", "ADANIENT", "ADANITRANS", "ABFRL",
    "AEGISCHEM", "AHLUCONT", "ASHIANA", "AVANTIFEED", "BGRENERGY",
    "BLISSGVS", "BRFL", "CAPLIPOINT", "CCL", "CERA", "DALMIABHA",
    "DHANUKA", "DISHTV", "GLOBOFFS", "GRANULES", "GREENPLY",
    "GRINDWELL", "GUJGASLTD", "HEIDELBERG", "HMVL", "HITACHIHOM",
    "HMT", "HOTELEELA", "IDFC", "IFBIND", "IGARASHI", "ICIL",
    "INDOCO", "INTELLECT", "IPAPPM", "JETAIRWAYS", "JINDALPOLY",
    "JMTAUTOLTD", "KESORAMIND", "KIRLOSENG", "KWALITY", "MTNL",
    "MAHINDCIE", "MANAPPURAM", "MANPASAND", "MINDACORP",
    "NAVKARCORP", "PCJEWELLER", "POLARIS", "RAMCOSYS", "RKFORGE",
    "RTNPOWER", "RELAXO", "SADBHIN", "SHARDACROP", "SHILPAMED",
    "SJVN", "SMLISUZU", "SNOWMAN", "SOLARINDS", "SOMANYCERA",
    "STCINDIA", "STYABS", "SYNGENE", "TAKE", "TATAMTRDVR",
    "TTML", "TEXRAIL", "TIDEWATER", "TIMKEN", "TCI", "TRITURBINE",
    "TVSSRICHAK", "VGUARD", "WONDERLA", "ZEELEARN"
}

assert len(EXITS) == 75, f"Unexpected exit count: {len(EXITS)}"
assert len(ENTRIES) == 76, f"Unexpected entry count: {len(ENTRIES)}"

# A small number of symbols may have changed names in the dataset.
ALIASES = {
    "ADANITRANS": {"ADANITRANS", "ADANIENSOL"},
    "AEGISCHEM": {"AEGISCHEM", "AEGISLOG"},
    "GUJGASLTD": {"GUJGASLTD", "GUJENERGY"},
    "ILFSENGG": {"ILFSENGG", "IL&FSENGG"},
}

def symbol_variants(symbol):
    return ALIASES.get(symbol, {symbol})

# Use existing data in memory.
p = final_df.copy()
iv = intervals_df.copy()

p["date"] = pd.to_datetime(p["date"], errors="coerce").dt.normalize()
iv["start_date"] = pd.to_datetime(
    iv["start_date"], errors="coerce"
).dt.normalize()
iv["end_date"] = pd.to_datetime(
    iv["end_date"], errors="coerce"
).dt.normalize()

for df in (p, iv):
    for col in ("isin", "symbol"):
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip()

p["_flag"] = (
    p["is_nifty500_constituent"].astype(str).str.strip().str.lower()
    .isin(["true", "1", "yes", "y"])
)

def matching_intervals(symbol):
    variants = symbol_variants(symbol)
    return iv[iv["symbol"].isin(variants)].copy()

def active_any(rows, date):
    if rows.empty:
        return False
    return bool((
        rows["start_date"].notna()
        & (rows["start_date"] <= date)
        & (rows["end_date"].isna() | (rows["end_date"] >= date))
    ).any())

def flags_on_date(symbol, isin_set, date):
    variants = symbol_variants(symbol)
    g = p[
        p["date"].eq(date)
        & (
            p["isin"].isin(isin_set)
            | p["symbol"].isin(variants)
        )
    ]
    return g["isin"].nunique(), g.loc[g["_flag"], "isin"].nunique()

report = []

for role, symbols in [("EXIT", EXITS), ("ENTRY", ENTRIES)]:
    for symbol in sorted(symbols):
        rows = matching_intervals(symbol)
        isins = set(rows["isin"])

        # Exits should remain members through 31 March.
        # Entries should not become members until 1 April.
        expected_march31 = role == "EXIT"
        expected_april1 = role == "ENTRY"

        price_march31, flag_march31 = flags_on_date(
            symbol, isins, LAST_OLD_SESSION
        )
        price_april1, flag_april1 = flags_on_date(
            symbol, isins, FIRST_NEW_SESSION
        )

        starts_on_announcement = int(
            rows["start_date"].eq(ANNOUNCEMENT).sum()
        )
        ends_on_announcement = int(
            rows["end_date"].eq(ANNOUNCEMENT).sum()
        )

        report.append({
            "Role": role,
            "Official_Symbol": symbol,
            "Matched_Interval_Rows": len(rows),
            "Matched_ISINs": len(isins),
            "Starts_On_Announcement": starts_on_announcement,
            "Ends_On_Announcement": ends_on_announcement,
            "Active_On_Feb22": active_any(rows, ANNOUNCEMENT),
            "Active_On_Mar31": active_any(rows, LAST_OLD_SESSION),
            "Active_On_Apr01": active_any(rows, FIRST_NEW_SESSION),
            "Expected_Active_Mar31": expected_march31,
            "Expected_Active_Apr01": expected_april1,
            "Price_ISINs_Mar31": price_march31,
            "Flagged_ISINs_Mar31": flag_march31,
            "Price_ISINs_Apr01": price_april1,
            "Flagged_ISINs_Apr01": flag_april1,
        })

audit = pd.DataFrame(report)

print("=" * 68)
print("1. PREFLIGHT SUMMARY")
print("=" * 68)

summary = audit.groupby("Role").agg(
    Official_Symbols=("Official_Symbol", "nunique"),
    Symbols_Found=("Matched_Interval_Rows", lambda x: int((x > 0).sum())),
    Symbols_With_Start_On_Feb22=(
        "Starts_On_Announcement", lambda x: int((x > 0).sum())
    ),
    Symbols_With_End_On_Feb22=(
        "Ends_On_Announcement", lambda x: int((x > 0).sum())
    ),
    Active_On_Mar31=("Active_On_Mar31", "sum"),
    Active_On_Apr01=("Active_On_Apr01", "sum"),
).reset_index()

display(summary)

print("\n" + "=" * 68)
print("2. ANNOUNCEMENT-DATE BOUNDARIES")
print("=" * 68)

boundary_rows = audit[
    (audit["Starts_On_Announcement"] > 0)
    | (audit["Ends_On_Announcement"] > 0)
].copy()

display(
    boundary_rows[
        [
            "Role", "Official_Symbol", "Matched_Interval_Rows",
            "Starts_On_Announcement", "Ends_On_Announcement",
            "Active_On_Mar31", "Active_On_Apr01",
            "Flagged_ISINs_Mar31", "Flagged_ISINs_Apr01"
        ]
    ].sort_values(["Role", "Official_Symbol"])
)

print("\n" + "=" * 68)
print("3. EXPECTED MEMBERSHIP MISMATCHES")
print("=" * 68)

mismatch = audit[
    (audit["Active_On_Mar31"] != audit["Expected_Active_Mar31"])
    | (audit["Active_On_Apr01"] != audit["Expected_Active_Apr01"])
]

display(
    mismatch[
        [
            "Role", "Official_Symbol", "Matched_Interval_Rows",
            "Active_On_Mar31", "Expected_Active_Mar31",
            "Active_On_Apr01", "Expected_Active_Apr01"
        ]
    ].sort_values(["Role", "Official_Symbol"])
)

out_path = DATA_DIR / "official_review_crosscheck_2016-04-01.csv"
audit.to_csv(out_path, index=False)

print("\nSaved read-only audit:", out_path)
print("No price rows, flags, or intervals were modified.")

1. PREFLIGHT SUMMARY


,Role,Official_Symbols,Symbols_Found,Symbols_With_Start_On_Feb22,Symbols_With_End_On_Feb22,Active_On_Mar31,Active_On_Apr01
0,ENTRY,76,57,20,0,23,23
1,EXIT,75,68,0,16,47,47



2. ANNOUNCEMENT-DATE BOUNDARIES


,Role,Official_Symbol,Matched_Interval_Rows,Starts_On_Announcement,Ends_On_Announcement,Active_On_Mar31,Active_On_Apr01,Flagged_ISINs_Mar31,Flagged_ISINs_Apr01
75,ENTRY,8KMILES,1,1,0,True,True,1,1
77,ENTRY,ABFRL,1,1,0,True,True,1,1
79,ENTRY,ADANITRANS,1,1,0,True,True,1,1
80,ENTRY,AEGISCHEM,1,1,0,True,True,1,1
83,ENTRY,AVANTIFEED,2,1,0,True,True,1,1
85,ENTRY,BLISSGVS,1,1,0,True,True,1,1
87,ENTRY,CAPLIPOINT,2,1,0,True,True,1,1
88,ENTRY,CCL,1,1,0,True,True,1,1
89,ENTRY,CERA,1,1,0,True,True,1,1
92,ENTRY,DISHTV,1,1,0,True,True,1,1



3. EXPECTED MEMBERSHIP MISMATCHES


,Role,Official_Symbol,Matched_Interval_Rows,Active_On_Mar31,Expected_Active_Mar31,Active_On_Apr01,Expected_Active_Apr01
75,ENTRY,8KMILES,1,True,False,True,True
76,ENTRY,AARTIDRUGS,1,False,False,False,True
77,ENTRY,ABFRL,1,True,False,True,True
78,ENTRY,ADANIENT,2,False,False,False,True
79,ENTRY,ADANITRANS,1,True,False,True,True
...,...,...,...,...,...,...,...
70,EXIT,USHAMART,2,True,True,True,False
71,EXIT,UTTAMSTL,1,True,True,True,False
72,EXIT,VESUVIUS,2,True,True,True,False
73,EXIT,VIVIDHA,0,False,True,False,False



Saved read-only audit: /content/nifty500_pit_data/official_review_crosscheck_2016-04-01.csv
No price rows, flags, or intervals were modified.


In [17]:
from pathlib import Path
import pandas as pd

AUDIT_PATH = Path(
    "/content/nifty500_pit_data/"
    "official_review_crosscheck_2016-04-01.csv"
)
OUT_PATH = Path(
    "/content/nifty500_pit_data/"
    "official_review_boundary_candidates_2016_v1.csv"
)

assert AUDIT_PATH.exists(), f"Audit report not found: {AUDIT_PATH}"
assert "intervals_df" in globals(), "intervals_df is not loaded"

audit = pd.read_csv(AUDIT_PATH)
required = {"Role", "Official_Symbol"}
assert required.issubset(audit.columns), (
    f"Unexpected audit columns: {audit.columns.tolist()}"
)

# Get the official symbols from the saved audit.
entry_symbols = set(
    audit.loc[
        audit["Role"].astype(str).str.upper().eq("ENTRY"),
        "Official_Symbol"
    ].dropna().astype(str).str.strip().str.upper()
)
exit_symbols = set(
    audit.loc[
        audit["Role"].astype(str).str.upper().eq("EXIT"),
        "Official_Symbol"
    ].dropna().astype(str).str.strip().str.upper()
)

# Known historical/current ticker aliases used in the audit.
aliases = {
    "ADANITRANS": {"ADANITRANS", "ADANIENSOL"},
    "AEGISCHEM": {"AEGISCHEM", "AEGISLOG"},
    "GUJGASLTD": {"GUJGASLTD", "GUJENERGY"},
    "ILFSENGG": {"ILFSENGG", "IL&FSENGG"},
}

def variants(symbol):
    return aliases.get(symbol, {symbol})

entry_aliases = {
    x for s in entry_symbols for x in variants(s)
}
exit_aliases = {
    x for s in exit_symbols for x in variants(s)
}

# Prepare a copy for inspection only.
iv = intervals_df.copy()
iv["_original_row_index"] = iv.index
iv["_symbol_norm"] = (
    iv["symbol"].astype(str).str.strip().str.upper()
)
iv["_start_dt"] = pd.to_datetime(iv["start_date"], errors="coerce")
iv["_end_dt"] = pd.to_datetime(iv["end_date"], errors="coerce")

announcement_date = pd.Timestamp("2016-02-22")

# Incoming candidates: interval starts on announcement date.
incoming = iv[
    iv["_start_dt"].eq(announcement_date)
    & iv["_symbol_norm"].isin(entry_aliases)
].copy()
incoming["Role"] = "ENTRY"
incoming["Official_Symbol"] = incoming["_symbol_norm"].map(
    {alias: official
     for official in entry_symbols
     for alias in variants(official)}
)

# Outgoing candidates: interval ends on announcement date.
outgoing = iv[
    iv["_end_dt"].eq(announcement_date)
    & iv["_symbol_norm"].isin(exit_aliases)
].copy()
outgoing["Role"] = "EXIT"
outgoing["Official_Symbol"] = outgoing["_symbol_norm"].map(
    {alias: official
     for official in exit_symbols
     for alias in variants(official)}
)

candidates = pd.concat([incoming, outgoing], ignore_index=True)

# Keep the original interval values visible for review.
display_cols = [
    "_original_row_index", "Role", "Official_Symbol",
    "isin", "symbol", "company_name",
    "start_date", "start_confidence",
    "end_date", "end_confidence"
]
display_cols = [c for c in display_cols if c in candidates.columns]

print("READ-ONLY PREVIEW — NO INTERVALS OR PRICES CHANGED")
print(f"Incoming boundary candidates: {len(incoming)}")
print(f"Outgoing boundary candidates: {len(outgoing)}")
print(
    "Unique incoming official symbols:",
    incoming["Official_Symbol"].nunique()
)
print(
    "Unique outgoing official symbols:",
    outgoing["Official_Symbol"].nunique()
)

print("\nINCOMING CANDIDATES")
display(
    candidates[candidates["Role"].eq("ENTRY")][display_cols]
    .sort_values(["Official_Symbol", "symbol"])
)

print("\nOUTGOING CANDIDATES")
display(
    candidates[candidates["Role"].eq("EXIT")][display_cols]
    .sort_values(["Official_Symbol", "symbol"])
)

# Show official names for which no exact boundary candidate was found.
entry_candidate_symbols = set(
    incoming["Official_Symbol"].dropna().astype(str).str.upper()
)
exit_candidate_symbols = set(
    outgoing["Official_Symbol"].dropna().astype(str).str.upper()
)

print("\nINCOMING OFFICIAL SYMBOLS WITHOUT A MATCHING FEB-22 START")
print(sorted(entry_symbols - entry_candidate_symbols))

print("\nOUTGOING OFFICIAL SYMBOLS WITHOUT A MATCHING FEB-22 END")
print(sorted(exit_symbols - exit_candidate_symbols))

candidates[display_cols].to_csv(OUT_PATH, index=False)
print(f"\nCandidate report saved: {OUT_PATH}")

READ-ONLY PREVIEW — NO INTERVALS OR PRICES CHANGED
Incoming boundary candidates: 21
Outgoing boundary candidates: 16
Unique incoming official symbols: 20
Unique outgoing official symbols: 16

INCOMING CANDIDATES


,_original_row_index,Role,Official_Symbol,isin,symbol,company_name,start_date,start_confidence,end_date,end_confidence
1,398,ENTRY,8KMILES,INE650K01021,8KMILES,8K Miles Soft Services Ltd.,2016-02-22,press_release_exact,2019-02-25,press_release_exact
15,765,ENTRY,ABFRL,INE647O01011,ABFRL,Aditya Birla Fashion and Retail Ltd.,2016-02-22,press_release_exact,2026-09-29,press_release_exact
19,1018,ENTRY,ADANITRANS,INE931S01010,ADANIENSOL,Adani Energy Solutions Ltd.,2016-02-22,press_release_exact,NaT,still_active
17,1007,ENTRY,AEGISCHEM,INE208C01025,AEGISLOG,Aegis Logistics Ltd.,2016-02-22,press_release_exact,NaT,still_active
12,660,ENTRY,AVANTIFEED,INE871C01038,AVANTIFEED,Avanti Feeds Ltd.,2016-02-22,press_release_exact,2025-02-21,press_release_exact
6,487,ENTRY,BLISSGVS,INE416D01022,BLISSGVS,Bliss GVS Pharma Ltd.,2016-02-22,press_release_exact,2021-09-15,press_release_exact
9,568,ENTRY,CAPLIPOINT,INE475E01026,CAPLIPOINT,Caplin Point Laboratories Ltd.,2016-02-22,press_release_exact,2022-05-02,interpolated_long_gap
20,1021,ENTRY,CCL,INE421D01022,CCL,CCL Products (I) Ltd.,2016-02-22,press_release_exact,NaT,still_active
13,739,ENTRY,CERA,INE739E01017,CERA,Cera Sanitaryware Ltd,2016-02-22,press_release_exact,2026-03-27,press_release_exact
8,543,ENTRY,DISHTV,INE836F01026,DISHTV,Dish TV India Ltd.,2016-02-22,press_release_exact,2021-09-15,press_release_exact



OUTGOING CANDIDATES


,_original_row_index,Role,Official_Symbol,isin,symbol,company_name,start_date,start_confidence,end_date,end_confidence
25,256,EXIT,ASAHIINDIA,INE439A01020,ASAHIINDIA,Asahi India Glass Ltd.,2010-01-02,left_censored,2016-02-22,press_release_exact
35,367,EXIT,ATFL,INE209A01019,ATFL,Agro Tech Foods Ltd.,2010-01-02,left_censored,2016-02-22,press_release_exact
28,290,EXIT,AUTOAXLES,INE449A01011,AUTOAXLES,Automotive Axles Ltd.,2010-01-02,left_censored,2016-02-22,press_release_exact
23,248,EXIT,CENTENKA,INE485A01015,CENTENKA,Century Enka Ltd.,2010-01-02,left_censored,2016-02-22,press_release_exact
34,350,EXIT,DHANBANK,INE680A01011,DHANBANK,Dhanlaxmi Bank Ltd.,2012-01-12,interpolated_long_gap,2016-02-22,press_release_exact
36,375,EXIT,ENIL,INE265F01028,ENIL,Entertainment Network India Ltd.,2010-01-02,left_censored,2016-02-22,press_release_exact
30,297,EXIT,ESABINDIA,INE284A01012,ESABINDIA,ESAB India Ltd.,2010-01-02,left_censored,2016-02-22,press_release_exact
32,319,EXIT,FLEXITUFF,INE060J01017,FLEXITUFF,Flexituff International Ltd.,2012-01-12,interpolated_long_gap,2016-02-22,press_release_exact
29,296,EXIT,FMGOETZE,INE529A01010,FMGOETZE,Federal-Mogul Goetze (India) Ltd.,2010-01-02,left_censored,2016-02-22,press_release_exact
24,253,EXIT,GABRIEL,INE524A01029,GABRIEL,Gabriel India Ltd.,2012-01-12,interpolated_long_gap,2016-02-22,press_release_exact



INCOMING OFFICIAL SYMBOLS WITHOUT A MATCHING FEB-22 START
['AARTIDRUGS', 'ADANIENT', 'AHLUCONT', 'ASHIANA', 'BGRENERGY', 'BRFL', 'DALMIABHA', 'DHANUKA', 'GLOBOFFS', 'HITACHIHOM', 'HMT', 'HMVL', 'HOTELEELA', 'IDFC', 'IGARASHI', 'IPAPPM', 'JINDALPOLY', 'JMTAUTOLTD', 'KESORAMIND', 'KIRLOSENG', 'KWALITY', 'MAHINDCIE', 'MANAPPURAM', 'MANPASAND', 'MINDACORP', 'MTNL', 'NAVKARCORP', 'PCJEWELLER', 'POLARIS', 'RAMCOSYS', 'RELAXO', 'RKFORGE', 'RTNPOWER', 'SADBHIN', 'SHARDACROP', 'SHILPAMED', 'SJVN', 'SMLISUZU', 'SNOWMAN', 'SOLARINDS', 'SOMANYCERA', 'STCINDIA', 'STYABS', 'SYNGENE', 'TAKE', 'TATAMTRDVR', 'TCI', 'TEXRAIL', 'TIDEWATER', 'TIMKEN', 'TRITURBINE', 'TTML', 'TVSSRICHAK', 'VGUARD', 'WONDERLA', 'ZEELEARN']

OUTGOING OFFICIAL SYMBOLS WITHOUT A MATCHING FEB-22 END
['ADLABS', 'BINDALAGRO', 'CAMLINFINE', 'CARBORUNIV', 'GHCL', 'GNFC', 'GRAPHITE', 'GUJALKALI', 'GULFOILLUB', 'HEG', 'HERITGFOOD', 'IBVENTURES', 'ILFSENGG', 'ITDCEM', 'IVRCLINFRA', 'JYOTISTRUC', 'KCP', 'KKCL', 'KNRCON', 'KSBPUMPS', 'L

In [18]:
from pathlib import Path
import pandas as pd

BASE = Path("/content/nifty500_pit_data")
AUDIT_PATH = BASE / "official_review_crosscheck_2016-04-01.csv"

assert AUDIT_PATH.exists(), f"Missing audit: {AUDIT_PATH}"
assert "intervals_df" in globals(), "intervals_df is not loaded"

audit = pd.read_csv(AUDIT_PATH)
iv = intervals_df.copy()

required_audit = {
    "Role", "Official_Symbol", "Matched_Interval_Rows",
    "Active_On_Mar31", "Expected_Active_Mar31",
    "Active_On_Apr01", "Expected_Active_Apr01"
}
assert required_audit.issubset(audit.columns), (
    f"Audit columns differ: {audit.columns.tolist()}"
)

iv["_row_index"] = iv.index
iv["_symbol_norm"] = iv["symbol"].astype(str).str.strip().str.upper()
iv["_start"] = pd.to_datetime(iv["start_date"], errors="coerce")
iv["_end"] = pd.to_datetime(iv["end_date"], errors="coerce")

aliases = {
    "ADANITRANS": {"ADANITRANS", "ADANIENSOL"},
    "AEGISCHEM": {"AEGISCHEM", "AEGISLOG"},
    "GUJGASLTD": {"GUJGASLTD", "GUJENERGY"},
    "ILFSENGG": {"ILFSENGG", "IL&FSENGG"},
}

mar31 = pd.Timestamp("2016-03-31")
apr01 = pd.Timestamp("2016-04-01")
announcement = pd.Timestamp("2016-02-22")

roster_rows = []
detail_rows = []

for _, a in audit.iterrows():
    role = str(a["Role"]).upper()
    official = str(a["Official_Symbol"]).strip().upper()
    symbols = aliases.get(official, {official})

    matches = iv[iv["_symbol_norm"].isin(symbols)].copy()

    active_mar31 = (
        (matches["_start"].le(mar31))
        & (matches["_end"].isna() | matches["_end"].ge(mar31))
    )
    active_apr01 = (
        (matches["_start"].le(apr01))
        & (matches["_end"].isna() | matches["_end"].ge(apr01))
    )

    # Company-level membership is true if at least one matching
    # interval is active on the date.
    actual_mar31 = bool(active_mar31.any())
    actual_apr01 = bool(active_apr01.any())

    expected_mar31 = role == "EXIT"
    expected_apr01 = role == "ENTRY"

    roster_rows.append({
        "Role": role,
        "Official_Symbol": official,
        "Matched_Interval_Rows": len(matches),
        "Actual_Active_Mar31": actual_mar31,
        "Expected_Active_Mar31": expected_mar31,
        "Mar31_Matches": actual_mar31 == expected_mar31,
        "Actual_Active_Apr01": actual_apr01,
        "Expected_Active_Apr01": expected_apr01,
        "Apr01_Matches": actual_apr01 == expected_apr01,
        "Needs_Review": (
            len(matches) == 0
            or actual_mar31 != expected_mar31
            or actual_apr01 != expected_apr01
        )
    })

    for _, r in matches.iterrows():
        is_active_mar31 = (
            pd.notna(r["_start"])
            and r["_start"] <= mar31
            and (pd.isna(r["_end"]) or r["_end"] >= mar31)
        )
        is_active_apr01 = (
            pd.notna(r["_start"])
            and r["_start"] <= apr01
            and (pd.isna(r["_end"]) or r["_end"] >= apr01)
        )

        detail_rows.append({
            "Role": role,
            "Official_Symbol": official,
            "Interval_Row_Index": r["_row_index"],
            "ISIN": r["isin"],
            "Interval_Symbol": r["symbol"],
            "Company": r["company_name"],
            "Start_Date": r["start_date"],
            "Start_Confidence": r["start_confidence"],
            "End_Date": r["end_date"],
            "End_Confidence": r["end_confidence"],
            "Active_Mar31": is_active_mar31,
            "Active_Apr01": is_active_apr01,
            "Starts_On_Announcement": r["_start"] == announcement,
            "Ends_On_Announcement": r["_end"] == announcement,
            "ISIN_Migration_Confidence": (
                "migration" in str(r["start_confidence"]).lower()
                or "migration" in str(r["end_confidence"]).lower()
            )
        })

roster = pd.DataFrame(roster_rows)
details = pd.DataFrame(detail_rows)

print("READ-ONLY RECONCILIATION — NO DATA MODIFIED")
print(f"Official symbols audited: {len(roster)}")

print("\nSUMMARY BY ROLE")
display(
    roster.groupby("Role").agg(
        Official_Symbols=("Official_Symbol", "nunique"),
        Missing_Interval_History=(
            "Matched_Interval_Rows", lambda s: int((s == 0).sum())
        ),
        Wrong_Mar31=("Mar31_Matches", lambda s: int((~s).sum())),
        Wrong_Apr01=("Apr01_Matches", lambda s: int((~s).sum())),
        Symbols_Needing_Review=("Needs_Review", "sum")
    ).reset_index()
)

print("\nOFFICIAL SYMBOLS NEEDING REVIEW")
display(
    roster[roster["Needs_Review"]].sort_values(
        ["Role", "Official_Symbol"]
    )
)

print("\nROWS WITH ANNOUNCEMENT-DATE BOUNDARIES OR ISIN MIGRATIONS")
if not details.empty:
    display(
        details[
            details["Starts_On_Announcement"]
            | details["Ends_On_Announcement"]
            | details["ISIN_Migration_Confidence"]
        ].sort_values(["Role", "Official_Symbol", "Interval_Row_Index"])
    )

roster_path = BASE / "official_review_roster_reconciliation_2016_v1.csv"
detail_path = BASE / "official_review_interval_details_2016_v1.csv"

roster.to_csv(roster_path, index=False)
details.to_csv(detail_path, index=False)

print(f"\nRoster report saved: {roster_path}")
print(f"Interval detail report saved: {detail_path}")
print("No prices, membership flags, or intervals were modified.")

READ-ONLY RECONCILIATION — NO DATA MODIFIED
Official symbols audited: 151

SUMMARY BY ROLE


,Role,Official_Symbols,Missing_Interval_History,Wrong_Mar31,Wrong_Apr01,Symbols_Needing_Review
0,ENTRY,76,19,23,53,76
1,EXIT,75,7,28,47,75



OFFICIAL SYMBOLS NEEDING REVIEW


,Role,Official_Symbol,Matched_Interval_Rows,Actual_Active_Mar31,Expected_Active_Mar31,Mar31_Matches,Actual_Active_Apr01,Expected_Active_Apr01,Apr01_Matches,Needs_Review
75,ENTRY,8KMILES,1,True,False,False,True,True,True,True
76,ENTRY,AARTIDRUGS,1,False,False,True,False,True,False,True
77,ENTRY,ABFRL,1,True,False,False,True,True,True,True
78,ENTRY,ADANIENT,2,False,False,True,False,True,False,True
79,ENTRY,ADANITRANS,1,True,False,False,True,True,True,True
...,...,...,...,...,...,...,...,...,...,...
70,EXIT,USHAMART,2,True,True,True,True,False,False,True
71,EXIT,UTTAMSTL,1,True,True,True,True,False,False,True
72,EXIT,VESUVIUS,2,True,True,True,True,False,False,True
73,EXIT,VIVIDHA,0,False,True,False,False,False,True,True



ROWS WITH ANNOUNCEMENT-DATE BOUNDARIES OR ISIN MIGRATIONS


,Role,Official_Symbol,Interval_Row_Index,ISIN,Interval_Symbol,Company,Start_Date,Start_Confidence,End_Date,End_Confidence,Active_Mar31,Active_Apr01,Starts_On_Announcement,Ends_On_Announcement,ISIN_Migration_Confidence
85,ENTRY,8KMILES,398,INE650K01021,8KMILES,8K Miles Soft Services Ltd.,2016-02-22,press_release_exact,2019-02-25,press_release_exact,True,True,True,False,False
87,ENTRY,ABFRL,765,INE647O01011,ABFRL,Aditya Birla Fashion and Retail Ltd.,2016-02-22,press_release_exact,2026-09-29,press_release_exact,True,True,True,False,False
90,ENTRY,ADANITRANS,1018,INE931S01010,ADANIENSOL,Adani Energy Solutions Ltd.,2016-02-22,press_release_exact,NaT,still_active,True,True,True,False,False
91,ENTRY,AEGISCHEM,1007,INE208C01025,AEGISLOG,Aegis Logistics Ltd.,2016-02-22,press_release_exact,NaT,still_active,True,True,True,False,False
92,ENTRY,AVANTIFEED,660,INE871C01038,AVANTIFEED,Avanti Feeds Ltd.,2016-02-22,press_release_exact,2025-02-21,press_release_exact,True,True,True,False,False
95,ENTRY,BLISSGVS,487,INE416D01022,BLISSGVS,Bliss GVS Pharma Ltd.,2016-02-22,press_release_exact,2021-09-15,press_release_exact,True,True,True,False,False
97,ENTRY,CAPLIPOINT,568,INE475E01026,CAPLIPOINT,Caplin Point Laboratories Ltd.,2016-02-22,press_release_exact,2022-05-02,interpolated_long_gap,True,True,True,False,False
99,ENTRY,CCL,1021,INE421D01022,CCL,CCL Products (I) Ltd.,2016-02-22,press_release_exact,NaT,still_active,True,True,True,False,False
100,ENTRY,CERA,739,INE739E01017,CERA,Cera Sanitaryware Ltd,2016-02-22,press_release_exact,2026-03-27,press_release_exact,True,True,True,False,False
102,ENTRY,DISHTV,543,INE836F01026,DISHTV,Dish TV India Ltd.,2016-02-22,press_release_exact,2021-09-15,press_release_exact,True,True,True,False,False



Roster report saved: /content/nifty500_pit_data/official_review_roster_reconciliation_2016_v1.csv
Interval detail report saved: /content/nifty500_pit_data/official_review_interval_details_2016_v1.csv
No prices, membership flags, or intervals were modified.


In [19]:
from pathlib import Path
import pandas as pd

BASE = Path("/content/nifty500_pit_data")
CANDIDATE_PATH = BASE / "official_review_boundary_candidates_2016_v1.csv"

STAGED_PATH = BASE / "pit_intervals_official_2016-04-01_v1_staged.csv"
LOG_PATH = BASE / "official_review_boundary_corrections_2016_v1.csv"

ANNOUNCEMENT_DATE = pd.Timestamp("2016-02-22")
EFFECTIVE_ENTRY_DATE = pd.Timestamp("2016-04-01")
LAST_MEMBER_DATE = pd.Timestamp("2016-03-31")

assert "intervals_df" in globals(), "intervals_df is not loaded."
assert CANDIDATE_PATH.exists(), f"Missing file: {CANDIDATE_PATH}"
assert intervals_df.index.is_unique, "Interval dataframe index is not unique."

candidates = pd.read_csv(CANDIDATE_PATH)
staged = intervals_df.copy(deep=True)

required = {
    "_original_row_index", "Role", "Official_Symbol",
    "isin", "symbol", "start_date", "start_confidence",
    "end_date", "end_confidence"
}
assert required.issubset(candidates.columns), (
    f"Candidate report has unexpected columns: {candidates.columns.tolist()}"
)

# Incoming: accept only explicit announcement-date starts with exact
# press-release confidence. GUJGASLTD is excluded pending ISIN review.
entries = candidates[
    candidates["Role"].eq("ENTRY")
    & candidates["Official_Symbol"].ne("GUJGASLTD")
    & candidates["start_date"].eq("2016-02-22")
    & candidates["start_confidence"].eq("press_release_exact")
].copy()

# Outgoing: accept only explicit announcement-date ends with exact
# press-release confidence.
exits = candidates[
    candidates["Role"].eq("EXIT")
    & candidates["end_date"].eq("2016-02-22")
    & candidates["end_confidence"].eq("press_release_exact")
].copy()

assert entries["Official_Symbol"].nunique() == 19, (
    f"Expected 19 clear entry symbols; found "
    f"{entries['Official_Symbol'].nunique()}"
)
assert exits["Official_Symbol"].nunique() == 16, (
    f"Expected 16 clear exit symbols; found "
    f"{exits['Official_Symbol'].nunique()}"
)
assert len(entries) == 19, f"Unexpected entry row count: {len(entries)}"
assert len(exits) == 16, f"Unexpected exit row count: {len(exits)}"

log = []

def validate_source_row(candidate, date_column, expected_date):
    row_id = int(candidate["_original_row_index"])
    assert row_id in staged.index, f"Interval row {row_id} is missing"

    row = staged.loc[row_id]

    assert str(row["isin"]) == str(candidate["isin"]), (
        f"ISIN mismatch on row {row_id}"
    )
    assert str(row["symbol"]) == str(candidate["symbol"]), (
        f"Symbol mismatch on row {row_id}"
    )
    assert pd.to_datetime(row[date_column]) == expected_date, (
        f"Unexpected {date_column} on row {row_id}: {row[date_column]}"
    )
    return row_id

for _, c in entries.iterrows():
    row_id = validate_source_row(c, "start_date", ANNOUNCEMENT_DATE)

    staged.loc[row_id, "start_date"] = EFFECTIVE_ENTRY_DATE.strftime("%Y-%m-%d")

    log.append({
        "Original_Row_Index": row_id,
        "Role": "ENTRY",
        "Official_Symbol": c["Official_Symbol"],
        "ISIN": c["isin"],
        "Interval_Symbol": c["symbol"],
        "Boundary": "start_date",
        "Old_Date": "2016-02-22",
        "New_Date": "2016-04-01",
        "Reason": "Official review effective date; exact boundary row"
    })

for _, c in exits.iterrows():
    row_id = validate_source_row(c, "end_date", ANNOUNCEMENT_DATE)

    staged.loc[row_id, "end_date"] = LAST_MEMBER_DATE.strftime("%Y-%m-%d")

    log.append({
        "Original_Row_Index": row_id,
        "Role": "EXIT",
        "Official_Symbol": c["Official_Symbol"],
        "ISIN": c["isin"],
        "Interval_Symbol": c["symbol"],
        "Boundary": "end_date",
        "Old_Date": "2016-02-22",
        "New_Date": "2016-03-31",
        "Reason": "Official review effective date; exact boundary row"
    })

log_df = pd.DataFrame(log)

# Validate all staged date changes.
assert len(log_df) == 35
assert (
    pd.to_datetime(
        staged.loc[log_df.loc[log_df["Role"].eq("ENTRY"),
                              "Original_Row_Index"].astype(int),
                   "start_date"]
    ).eq(EFFECTIVE_ENTRY_DATE).all()
)
assert (
    pd.to_datetime(
        staged.loc[log_df.loc[log_df["Role"].eq("EXIT"),
                              "Original_Row_Index"].astype(int),
                   "end_date"]
    ).eq(LAST_MEMBER_DATE).all()
)

# Write separate versioned files. The original dataframe remains untouched.
staged.to_csv(STAGED_PATH, index=False)
log_df.to_csv(LOG_PATH, index=False)

print("STAGED CORRECTIONS — ORIGINAL DATA LEFT UNCHANGED")
print(f"Incoming rows corrected: {len(entries)}")
print(f"Outgoing rows corrected: {len(exits)}")
print(f"Total boundary corrections: {len(log_df)}")
print(f"GUJGASLTD migration excluded: {True}")
print(f"Staged intervals: {STAGED_PATH}")
print(f"Correction log: {LOG_PATH}")

display(log_df.sort_values(["Role", "Official_Symbol"]))

STAGED CORRECTIONS — ORIGINAL DATA LEFT UNCHANGED
Incoming rows corrected: 19
Outgoing rows corrected: 16
Total boundary corrections: 35
GUJGASLTD migration excluded: True
Staged intervals: /content/nifty500_pit_data/pit_intervals_official_2016-04-01_v1_staged.csv
Correction log: /content/nifty500_pit_data/official_review_boundary_corrections_2016_v1.csv


,Original_Row_Index,Role,Official_Symbol,ISIN,Interval_Symbol,Boundary,Old_Date,New_Date,Reason
0,398,ENTRY,8KMILES,INE650K01021,8KMILES,start_date,2016-02-22,2016-04-01,Official review effective date; exact boundary...
13,765,ENTRY,ABFRL,INE647O01011,ABFRL,start_date,2016-02-22,2016-04-01,Official review effective date; exact boundary...
17,1018,ENTRY,ADANITRANS,INE931S01010,ADANIENSOL,start_date,2016-02-22,2016-04-01,Official review effective date; exact boundary...
15,1007,ENTRY,AEGISCHEM,INE208C01025,AEGISLOG,start_date,2016-02-22,2016-04-01,Official review effective date; exact boundary...
11,660,ENTRY,AVANTIFEED,INE871C01038,AVANTIFEED,start_date,2016-02-22,2016-04-01,Official review effective date; exact boundary...
5,487,ENTRY,BLISSGVS,INE416D01022,BLISSGVS,start_date,2016-02-22,2016-04-01,Official review effective date; exact boundary...
8,568,ENTRY,CAPLIPOINT,INE475E01026,CAPLIPOINT,start_date,2016-02-22,2016-04-01,Official review effective date; exact boundary...
18,1021,ENTRY,CCL,INE421D01022,CCL,start_date,2016-02-22,2016-04-01,Official review effective date; exact boundary...
12,739,ENTRY,CERA,INE739E01017,CERA,start_date,2016-02-22,2016-04-01,Official review effective date; exact boundary...
7,543,ENTRY,DISHTV,INE836F01026,DISHTV,start_date,2016-02-22,2016-04-01,Official review effective date; exact boundary...


In [20]:
import pandas as pd

assert "intervals_df" in globals(), "intervals_df is not loaded."
assert "final_df" in globals(), "final_df is not loaded."

target_isins = ["INE844O01022", "INE844O01030"]

# 1. Show the two membership intervals.
print("=" * 70)
print("1. GUJARAT GAS MEMBERSHIP INTERVALS")
print("=" * 70)

interval_cols = [
    c for c in [
        "isin", "symbol", "company_name",
        "start_date", "start_confidence",
        "end_date", "end_confidence"
    ]
    if c in intervals_df.columns
]

display(
    intervals_df[
        intervals_df["isin"].astype(str).isin(target_isins)
    ][interval_cols].sort_values(["isin", "start_date"])
)

# 2. Identify the date and ISIN columns in the price dataset.
print("\n" + "=" * 70)
print("2. PRICE HISTORY COVERAGE")
print("=" * 70)

df = final_df
date_col = next(
    (c for c in ["date", "trade_date", "datetime"]
     if c in df.columns),
    None
)
isin_col = next(
    (c for c in ["isin", "ISIN"]
     if c in df.columns),
    None
)

assert date_col is not None, f"Date column not found: {df.columns.tolist()}"
assert isin_col is not None, f"ISIN column not found: {df.columns.tolist()}"

prices = df[df[isin_col].astype(str).isin(target_isins)].copy()
prices["_date"] = pd.to_datetime(prices[date_col], errors="coerce")

summary = (
    prices.groupby(isin_col)
    .agg(
        Total_Rows=("_date", "size"),
        First_Price_Date=("_date", "min"),
        Last_Price_Date=("_date", "max"),
        Unique_Dates=("_date", "nunique")
    )
    .reset_index()
)

display(summary)

# 3. Check price availability around the 2016 index review and
# the apparent 2019 ISIN migration.
print("\n" + "=" * 70)
print("3. PRICE ROWS AROUND KEY DATES")
print("=" * 70)

windows = [
    ("2016 review", "2016-02-15", "2016-04-08"),
    ("2019 migration", "2019-01-25", "2019-02-12")
]

preferred_cols = [
    isin_col, date_col, "symbol",
    "open", "high", "low", "close", "volume",
    "data_source", "adjustment",
    "price_candidate_eligible", "close_for_analysis"
]
show_cols = [c for c in preferred_cols if c in prices.columns]

for label, start, end in windows:
    print(f"\n{label}: {start} to {end}")
    window = prices[
        prices["_date"].between(start, end)
    ].sort_values(["_date", isin_col])

    if window.empty:
        print("No price rows found.")
    else:
        display(window[show_cols])

print("\nRead-only inspection complete. No data was modified.")

1. GUJARAT GAS MEMBERSHIP INTERVALS


,isin,symbol,company_name,start_date,start_confidence,end_date,end_confidence
376,INE844O01022,GUJGASLTD,Gujarat Gas Ltd.,2016-02-22,press_release_exact,2019-02-01,isin_migration_to_INE844O01030
740,INE844O01030,GUJGASLTD,Gujarat Gas Ltd.,2016-02-22,isin_migration_from_INE844O01022,2026-03-27,press_release_exact



2. PRICE HISTORY COVERAGE


,isin,Total_Rows,First_Price_Date,Last_Price_Date,Unique_Dates
0,INE844O01030,2496,2016-02-22,2026-04-02,2496



3. PRICE ROWS AROUND KEY DATES

2016 review: 2016-02-15 to 2016-04-08


,isin,date,symbol,open,high,low,close,volume,data_source,adjustment,price_candidate_eligible,close_for_analysis
1926133,INE844O01030,2016-02-22,GUJENERGY,106.400002,107.980003,105.000000,105.620003,70075.0,yfinance,split_adjusted_no_dividend,True,105.620003
1926134,INE844O01030,2016-02-23,GUJENERGY,108.019997,108.339996,104.000000,105.980003,156360.0,yfinance,split_adjusted_no_dividend,True,105.980003
1926135,INE844O01030,2016-02-24,GUJENERGY,105.800003,106.430000,102.199997,103.699997,66890.0,yfinance,split_adjusted_no_dividend,True,103.699997
1926136,INE844O01030,2016-02-25,GUJENERGY,103.599998,105.389999,98.500000,100.019997,154775.0,yfinance,split_adjusted_no_dividend,True,100.019997
1926137,INE844O01030,2016-02-26,GUJENERGY,102.199997,102.199997,98.000000,98.360001,52080.0,yfinance,split_adjusted_no_dividend,True,98.360001
1926138,INE844O01030,2016-02-29,GUJENERGY,98.400002,99.639999,95.599998,96.230003,104465.0,yfinance,split_adjusted_no_dividend,True,96.230003
1926139,INE844O01030,2016-03-01,GUJENERGY,96.230003,101.000000,96.230003,98.620003,360590.0,yfinance,split_adjusted_no_dividend,True,98.620003
1926140,INE844O01030,2016-03-02,GUJENERGY,99.230003,101.470001,97.199997,97.889999,195560.0,yfinance,split_adjusted_no_dividend,True,97.889999
1926141,INE844O01030,2016-03-03,GUJENERGY,97.800003,101.400002,97.800003,99.639999,529060.0,yfinance,split_adjusted_no_dividend,True,99.639999
1926142,INE844O01030,2016-03-04,GUJENERGY,100.000000,101.790001,99.699997,100.129997,322650.0,yfinance,split_adjusted_no_dividend,True,100.129997



2019 migration: 2019-01-25 to 2019-02-12


,isin,date,symbol,open,high,low,close,volume,data_source,adjustment,price_candidate_eligible,close_for_analysis
1926856,INE844O01030,2019-01-25,GUJENERGY,126.500000,132.000000,122.750000,125.099998,244250.0,yfinance,split_adjusted_no_dividend,True,125.099998
1926857,INE844O01030,2019-01-28,GUJENERGY,127.949997,127.949997,125.099998,126.699997,123632.0,yfinance,split_adjusted_no_dividend,True,126.699997
1926858,INE844O01030,2019-01-29,GUJENERGY,126.699997,129.399994,123.199997,127.849998,80770.0,yfinance,split_adjusted_no_dividend,True,127.849998
1926859,INE844O01030,2019-01-30,GUJENERGY,127.199997,129.699997,125.199997,128.949997,119035.0,yfinance,split_adjusted_no_dividend,True,128.949997
1926860,INE844O01030,2019-01-31,GUJENERGY,129.000000,129.750000,126.000000,127.550003,57263.0,yfinance,split_adjusted_no_dividend,True,127.550003
1926861,INE844O01030,2019-02-01,GUJENERGY,129.050003,129.399994,126.699997,127.300003,96277.0,yfinance,split_adjusted_no_dividend,True,127.300003
1926862,INE844O01030,2019-02-04,GUJENERGY,127.050003,130.350006,120.000000,127.800003,199840.0,yfinance,split_adjusted_no_dividend,True,127.800003
1926863,INE844O01030,2019-02-05,GUJENERGY,128.800003,130.000000,126.099998,127.050003,72853.0,yfinance,split_adjusted_no_dividend,True,127.050003
1926864,INE844O01030,2019-02-06,GUJENERGY,128.000000,128.300003,125.000000,126.250000,480867.0,yfinance,split_adjusted_no_dividend,True,126.250000
1926865,INE844O01030,2019-02-07,GUJENERGY,126.800003,128.399994,125.550003,127.199997,201811.0,yfinance,split_adjusted_no_dividend,True,127.199997



Read-only inspection complete. No data was modified.


In [21]:
from pathlib import Path
import pandas as pd

BASE = Path("/content/nifty500_pit_data")

SOURCE_PATH = BASE / "pit_intervals_official_2016-04-01_v1_staged.csv"
OUTPUT_PATH = BASE / "pit_intervals_official_2016-04-01_v2_staged.csv"
QUARANTINE_PATH = BASE / "gujarat_gas_suspended_isin_quarantine_v1.csv"
LOG_PATH = BASE / "gujarat_gas_interval_correction_log_v1.csv"

assert SOURCE_PATH.exists(), f"Missing staged interval file: {SOURCE_PATH}"

iv = pd.read_csv(SOURCE_PATH)

required_cols = {
    "isin", "symbol", "start_date", "start_confidence",
    "end_date", "end_confidence"
}
assert required_cols.issubset(iv.columns), (
    f"Missing columns: {required_cols - set(iv.columns)}"
)

old_isin = "INE844O01022"
listed_isin = "INE844O01030"

old_mask = iv["isin"].astype(str).eq(old_isin)
listed_mask = iv["isin"].astype(str).eq(listed_isin)

# Stop instead of guessing if the expected interval structure differs.
assert int(old_mask.sum()) == 1, (
    f"Expected exactly one suspended-ISIN interval; found {old_mask.sum()}"
)
assert int(listed_mask.sum()) == 1, (
    f"Expected exactly one listed-ISIN interval; found {listed_mask.sum()}"
)

old_row = iv.loc[old_mask].copy()
new_row = iv.loc[listed_mask].copy()

assert old_row.iloc[0]["start_date"] == "2016-02-22"
assert old_row.iloc[0]["end_date"] == "2019-02-01"
assert new_row.iloc[0]["start_date"] == "2016-02-22"
assert new_row.iloc[0]["end_date"] == "2026-03-27"

# Preserve the excluded row and document why it is not part of
# the staged active membership interval table.
old_row["exclusion_reason"] = (
    "Company disclosure identifies this ISIN as suspended; "
    "no price history for this ISIN exists in the inspected dataset."
)
old_row["evidence_note"] = (
    "Company filing identifies INE844O01030 as the listed/traded ISIN "
    "from 2015-09-15. Verify corporate-action records before any "
    "future attempt to reinstate this interval."
)

# Remove the suspended-ISIN row only from the staged active table.
staged = iv.loc[~old_mask].copy()

# Set the active listed security's index-membership start to the
# official review effective date, while retaining its exit boundary.
listed_mask_staged = staged["isin"].astype(str).eq(listed_isin)
assert int(listed_mask_staged.sum()) == 1

staged.loc[listed_mask_staged, "start_date"] = "2016-04-01"
staged.loc[listed_mask_staged, "start_confidence"] = "press_release_exact"

# Final consistency checks.
assert not staged["isin"].astype(str).eq(old_isin).any()
corrected = staged.loc[listed_mask_staged].iloc[0]
assert corrected["start_date"] == "2016-04-01"
assert corrected["end_date"] == "2026-03-27"

log = pd.DataFrame([
    {
        "ISIN": old_isin,
        "Action": "Quarantine from active membership intervals",
        "Old_Start": old_row.iloc[0]["start_date"],
        "Old_End": old_row.iloc[0]["end_date"],
        "Reason": (
            "Suspended ISIN; preserved separately for audit, "
            "not silently deleted."
        )
    },
    {
        "ISIN": listed_isin,
        "Action": "Correct membership start date",
        "Old_Start": new_row.iloc[0]["start_date"],
        "New_Start": "2016-04-01",
        "End_Date_Preserved": corrected["end_date"],
        "Reason": (
            "Official Nifty 500 review effective date; "
            "listed/traded security interval."
        )
    }
])

# Save new versions only. Source file, intervals_df and final_df stay untouched.
staged.to_csv(OUTPUT_PATH, index=False)
old_row.to_csv(QUARANTINE_PATH, index=False)
log.to_csv(LOG_PATH, index=False)

print("GUJARAT GAS CORRECTION STAGED — NO ACTIVE DATA MODIFIED")
print(f"Source intervals: {len(iv)}")
print(f"Staged active intervals: {len(staged)}")
print(f"Rows quarantined: {len(old_row)}")
print(f"Listed ISIN membership: {corrected['start_date']} to {corrected['end_date']}")
print(f"Staged intervals: {OUTPUT_PATH}")
print(f"Quarantine record: {QUARANTINE_PATH}")
print(f"Correction log: {LOG_PATH}")

display(log)

GUJARAT GAS CORRECTION STAGED — NO ACTIVE DATA MODIFIED
Source intervals: 1292
Staged active intervals: 1291
Rows quarantined: 1
Listed ISIN membership: 2016-04-01 to 2026-03-27
Staged intervals: /content/nifty500_pit_data/pit_intervals_official_2016-04-01_v2_staged.csv
Quarantine record: /content/nifty500_pit_data/gujarat_gas_suspended_isin_quarantine_v1.csv
Correction log: /content/nifty500_pit_data/gujarat_gas_interval_correction_log_v1.csv


,ISIN,Action,Old_Start,Old_End,Reason,New_Start,End_Date_Preserved
0,INE844O01022,Quarantine from active membership intervals,2016-02-22,2019-02-01,Suspended ISIN; preserved separately for audit...,NaN,NaN
1,INE844O01030,Correct membership start date,2016-02-22,NaN,Official Nifty 500 review effective date; list...,2016-04-01,2026-03-27


In [22]:
from pathlib import Path
import pandas as pd

BASE = Path("/content/nifty500_pit_data")

STAGED_PATH = BASE / "pit_intervals_official_2016-04-01_v2_staged.csv"
BOUNDARY_LOG_PATH = BASE / "official_review_boundary_corrections_2016_v1.csv"
GUJ_LOG_PATH = BASE / "gujarat_gas_interval_correction_log_v1.csv"
QUARANTINE_PATH = BASE / "gujarat_gas_suspended_isin_quarantine_v1.csv"

QA_PATH = BASE / "official_review_staged_boundary_QA_2016_v1.csv"

for path in [
    STAGED_PATH, BOUNDARY_LOG_PATH, GUJ_LOG_PATH, QUARANTINE_PATH
]:
    assert path.exists(), f"Required file missing: {path}"

staged = pd.read_csv(STAGED_PATH, dtype={"isin": str})
boundary_log = pd.read_csv(BOUNDARY_LOG_PATH, dtype={"ISIN": str})
guj_log = pd.read_csv(GUJ_LOG_PATH, dtype={"ISIN": str})
quarantine = pd.read_csv(QUARANTINE_PATH, dtype={"isin": str})

assert {"isin", "symbol", "start_date", "end_date"}.issubset(staged.columns)
assert len(boundary_log) == 35, (
    f"Expected 35 boundary corrections; found {len(boundary_log)}"
)

staged["_start_dt"] = pd.to_datetime(staged["start_date"], errors="coerce")
staged["_end_dt"] = pd.to_datetime(staged["end_date"], errors="coerce")

# ------------------------------------------------------------
# 1. Verify all 35 staged boundary corrections
# ------------------------------------------------------------
checks = []

for _, item in boundary_log.iterrows():
    isin = str(item["ISIN"])
    symbol = str(item["Interval_Symbol"])
    boundary = str(item["Boundary"])
    expected = pd.Timestamp(item["New_Date"])

    col = "_start_dt" if boundary == "start_date" else "_end_dt"

    matches = staged[
        staged["isin"].eq(isin)
        & staged["symbol"].astype(str).eq(symbol)
    ]

    found = bool(matches[col].eq(expected).any())

    checks.append({
        "Role": item["Role"],
        "Official_Symbol": item["Official_Symbol"],
        "ISIN": isin,
        "Boundary": boundary,
        "Expected_Date": expected.date().isoformat(),
        "Matching_Interval_Rows": len(matches),
        "Expected_Date_Found": found,
        "QA_Status": "PASS" if found else "FAIL"
    })

checks_df = pd.DataFrame(checks)

print("=" * 70)
print("1. BOUNDARY CORRECTIONS")
print("=" * 70)
print(checks_df["QA_Status"].value_counts(dropna=False).to_dict())
print(
    "Verified correction rows:",
    int(checks_df["Expected_Date_Found"].sum()),
    "/",
    len(checks_df)
)

failures = checks_df[~checks_df["Expected_Date_Found"]]
if not failures.empty:
    print("\nBOUNDARY CORRECTIONS NEEDING INVESTIGATION")
    display(failures)

# ------------------------------------------------------------
# 2. Verify the Gujarat Gas interval and quarantine record
# ------------------------------------------------------------
old_isin = "INE844O01022"
listed_isin = "INE844O01030"

old_isin_in_staged = staged["isin"].eq(old_isin).any()
old_isin_in_quarantine = quarantine["isin"].eq(old_isin).sum()

listed_rows = staged[staged["isin"].eq(listed_isin)]

guj_checks = {
    "Suspended_ISIN_absent_from_staged": not old_isin_in_staged,
    "Suspended_ISIN_preserved_in_quarantine": old_isin_in_quarantine == 1,
    "Listed_ISIN_row_count_is_one": len(listed_rows) == 1,
    "Listed_ISIN_starts_2016_04_01": (
        len(listed_rows) == 1
        and listed_rows["_start_dt"].iloc[0] == pd.Timestamp("2016-04-01")
    ),
    "Listed_ISIN_ends_2026_03_27": (
        len(listed_rows) == 1
        and listed_rows["_end_dt"].iloc[0] == pd.Timestamp("2026-03-27")
    ),
}

print("\n" + "=" * 70)
print("2. GUJARAT GAS CHECK")
print("=" * 70)
for name, passed in guj_checks.items():
    print(f"{name}: {'PASS' if passed else 'FAIL'}")

print("\nGujarat Gas correction log:")
display(guj_log)

# ------------------------------------------------------------
# 3. General interval date validity
# ------------------------------------------------------------
invalid_ranges = staged[
    staged["_start_dt"].isna()
    | (
        staged["_end_dt"].notna()
        & staged["_start_dt"].gt(staged["_end_dt"])
    )
].copy()

print("\n" + "=" * 70)
print("3. DATE-RANGE VALIDITY")
print("=" * 70)
print("Staged interval rows:", len(staged))
print("Invalid or missing start dates / reversed ranges:", len(invalid_ranges))

if not invalid_ranges.empty:
    display(invalid_ranges[
        ["isin", "symbol", "start_date", "end_date"]
    ])

# Save a durable QA report.
checks_df.to_csv(QA_PATH, index=False)

print("\nQA report saved:", QA_PATH)
print("This cell did not modify the staged intervals, original data, or price flags.")

1. BOUNDARY CORRECTIONS
{'PASS': 35}
Verified correction rows: 35 / 35

2. GUJARAT GAS CHECK
Suspended_ISIN_absent_from_staged: PASS
Suspended_ISIN_preserved_in_quarantine: PASS
Listed_ISIN_row_count_is_one: PASS
Listed_ISIN_starts_2016_04_01: PASS
Listed_ISIN_ends_2026_03_27: PASS

Gujarat Gas correction log:


,ISIN,Action,Old_Start,Old_End,Reason,New_Start,End_Date_Preserved
0,INE844O01022,Quarantine from active membership intervals,2016-02-22,2019-02-01,Suspended ISIN; preserved separately for audit...,NaN,NaN
1,INE844O01030,Correct membership start date,2016-02-22,NaN,Official Nifty 500 review effective date; list...,2016-04-01,2026-03-27



3. DATE-RANGE VALIDITY
Staged interval rows: 1291
Invalid or missing start dates / reversed ranges: 0

QA report saved: /content/nifty500_pit_data/official_review_staged_boundary_QA_2016_v1.csv
This cell did not modify the staged intervals, original data, or price flags.


In [23]:
from pathlib import Path
import pandas as pd

BASE = Path("/content/nifty500_pit_data")

STAGED_PATH = BASE / "pit_intervals_official_2016-04-01_v2_staged.csv"
AUDIT_PATH = BASE / "official_review_crosscheck_2016-04-01.csv"

SUMMARY_PATH = BASE / "official_review_post_stage_2016_v1.csv"
DETAIL_PATH = BASE / "official_review_post_stage_interval_details_2016_v1.csv"

assert STAGED_PATH.exists(), f"Missing staged intervals: {STAGED_PATH}"
assert AUDIT_PATH.exists(), f"Missing official audit: {AUDIT_PATH}"

iv = pd.read_csv(STAGED_PATH, dtype={"isin": str})
audit = pd.read_csv(AUDIT_PATH)

required = {"isin", "symbol", "start_date", "end_date"}
assert required.issubset(iv.columns)

official = (
    audit[["Role", "Official_Symbol"]]
    .drop_duplicates()
    .copy()
)
official["Role"] = official["Role"].astype(str).str.upper()
official["Official_Symbol"] = (
    official["Official_Symbol"].astype(str).str.strip().str.upper()
)

assert len(official) == 151, f"Expected 151 symbols; found {len(official)}"

# Known historical ticker aliases used in the previous audit.
aliases = {
    "ADANITRANS": {"ADANITRANS", "ADANIENSOL"},
    "AEGISCHEM": {"AEGISCHEM", "AEGISLOG"},
    "GUJGASLTD": {"GUJGASLTD", "GUJENERGY"},
    "ILFSENGG": {"ILFSENGG", "IL&FSENGG"},
}

iv["_symbol_norm"] = iv["symbol"].astype(str).str.strip().str.upper()
iv["_start"] = pd.to_datetime(iv["start_date"], errors="coerce")
iv["_end"] = pd.to_datetime(iv["end_date"], errors="coerce")

mar31 = pd.Timestamp("2016-03-31")
apr01 = pd.Timestamp("2016-04-01")

summary_rows = []
detail_rows = []

for _, item in official.iterrows():
    role = item["Role"]
    symbol = item["Official_Symbol"]
    accepted_symbols = aliases.get(symbol, {symbol})

    matches = iv[iv["_symbol_norm"].isin(accepted_symbols)].copy()

    def active_on_date(row, date):
        return (
            pd.notna(row["_start"])
            and row["_start"] <= date
            and (pd.isna(row["_end"]) or row["_end"] >= date)
        )

    active_mar31 = any(
        active_on_date(row, mar31)
        for _, row in matches.iterrows()
    )
    active_apr01 = any(
        active_on_date(row, apr01)
        for _, row in matches.iterrows()
    )

    expected_mar31 = role == "EXIT"
    expected_apr01 = role == "ENTRY"

    mar31_match = active_mar31 == expected_mar31
    apr01_match = active_apr01 == expected_apr01

    if len(matches) == 0:
        category = "NO_INTERVAL_MATCH"
    elif mar31_match and apr01_match:
        category = "PASS"
    elif role == "ENTRY" and active_mar31:
        category = "ENTRY_ACTIVE_BEFORE_EFFECTIVE_DATE"
    elif role == "ENTRY" and not active_apr01:
        category = "ENTRY_NOT_ACTIVE_ON_EFFECTIVE_DATE"
    elif role == "EXIT" and not active_mar31:
        category = "EXIT_INACTIVE_BEFORE_EFFECTIVE_DATE"
    elif role == "EXIT" and active_apr01:
        category = "EXIT_STILL_ACTIVE_AFTER_EFFECTIVE_DATE"
    else:
        category = "OTHER_MISMATCH"

    range_descriptions = []

    for _, row in matches.iterrows():
        start = (
            row["_start"].date().isoformat()
            if pd.notna(row["_start"]) else "MISSING_START"
        )
        end = (
            row["_end"].date().isoformat()
            if pd.notna(row["_end"]) else "OPEN"
        )

        range_descriptions.append(
            f"{row['isin']} / {row['symbol']}: "
            f"{start} to {end}; "
            f"start_confidence={row.get('start_confidence', 'NA')}; "
            f"end_confidence={row.get('end_confidence', 'NA')}"
        )

        detail_rows.append({
            "Role": role,
            "Official_Symbol": symbol,
            "ISIN": row["isin"],
            "Interval_Symbol": row["symbol"],
            "Start_Date": row["start_date"],
            "Start_Confidence": row.get("start_confidence"),
            "End_Date": row["end_date"],
            "End_Confidence": row.get("end_confidence"),
            "Active_Mar31": active_on_date(row, mar31),
            "Active_Apr01": active_on_date(row, apr01),
            "Company": row.get("company_name")
        })

    summary_rows.append({
        "Role": role,
        "Official_Symbol": symbol,
        "Interval_Rows": len(matches),
        "Active_Mar31": active_mar31,
        "Expected_Mar31": expected_mar31,
        "Mar31_Match": mar31_match,
        "Active_Apr01": active_apr01,
        "Expected_Apr01": expected_apr01,
        "Apr01_Match": apr01_match,
        "Status": category,
        "Interval_Ranges": " | ".join(range_descriptions)
    })

summary = pd.DataFrame(summary_rows)
details = pd.DataFrame(detail_rows)

print("=" * 70)
print("POST-STAGING MEMBERSHIP QA — READ ONLY")
print("=" * 70)

print("\nSUMMARY BY OFFICIAL ROLE")
display(
    summary.groupby("Role").agg(
        Official_Symbols=("Official_Symbol", "nunique"),
        Symbols_Passing=("Status", lambda s: int((s == "PASS").sum())),
        Missing_Interval_Match=(
            "Status", lambda s: int((s == "NO_INTERVAL_MATCH").sum())
        ),
        Mar31_Mismatches=("Mar31_Match", lambda s: int((~s).sum())),
        Apr01_Mismatches=("Apr01_Match", lambda s: int((~s).sum()))
    ).reset_index()
)

print("\nISSUES BY CATEGORY")
display(
    summary[summary["Status"] != "PASS"]
    .groupby(["Role", "Status"])
    .size()
    .reset_index(name="Count")
    .sort_values(["Role", "Status"])
)

print("\nSYMBOLS NEEDING INVESTIGATION — INTERVAL DATE RANGES INCLUDED")
issues = summary[summary["Status"] != "PASS"].copy()
display(
    issues[
        [
            "Role", "Official_Symbol", "Interval_Rows",
            "Active_Mar31", "Mar31_Match",
            "Active_Apr01", "Apr01_Match",
            "Status", "Interval_Ranges"
        ]
    ].sort_values(["Role", "Status", "Official_Symbol"])
)

summary.to_csv(SUMMARY_PATH, index=False)
details.to_csv(DETAIL_PATH, index=False)

print("\nSaved summary:", SUMMARY_PATH)
print("Saved interval details:", DETAIL_PATH)
print("No interval or price records were modified.")

POST-STAGING MEMBERSHIP QA — READ ONLY

SUMMARY BY OFFICIAL ROLE


,Role,Official_Symbols,Symbols_Passing,Missing_Interval_Match,Mar31_Mismatches,Apr01_Mismatches
0,ENTRY,76,20,19,3,53
1,EXIT,75,16,7,12,47



ISSUES BY CATEGORY


,Role,Status,Count
0,ENTRY,ENTRY_ACTIVE_BEFORE_EFFECTIVE_DATE,3
1,ENTRY,ENTRY_NOT_ACTIVE_ON_EFFECTIVE_DATE,34
2,ENTRY,NO_INTERVAL_MATCH,19
3,EXIT,EXIT_INACTIVE_BEFORE_EFFECTIVE_DATE,5
4,EXIT,EXIT_STILL_ACTIVE_AFTER_EFFECTIVE_DATE,47
5,EXIT,NO_INTERVAL_MATCH,7



SYMBOLS NEEDING INVESTIGATION — INTERVAL DATE RANGES INCLUDED


,Role,Official_Symbol,Interval_Rows,Active_Mar31,Mar31_Match,Active_Apr01,Apr01_Match,Status,Interval_Ranges
104,ENTRY,IDFC,1,True,False,True,True,ENTRY_ACTIVE_BEFORE_EFFECTIVE_DATE,INE043D01016 / IDFC: 2010-01-02 to 2024-10-11;...
113,ENTRY,KESORAMIND,1,True,False,True,True,ENTRY_ACTIVE_BEFORE_EFFECTIVE_DATE,INE087A01019 / KESORAMIND: 2010-01-02 to 2018-...
127,ENTRY,RTNPOWER,1,True,False,True,True,ENTRY_ACTIVE_BEFORE_EFFECTIVE_DATE,INE399K01017 / RTNPOWER: 2012-01-12 to 2018-08...
76,ENTRY,AARTIDRUGS,1,False,True,False,False,ENTRY_NOT_ACTIVE_ON_EFFECTIVE_DATE,INE767A01016 / AARTIDRUGS: 2022-05-02 to 2024-...
78,ENTRY,ADANIENT,2,False,True,False,False,ENTRY_NOT_ACTIVE_ON_EFFECTIVE_DATE,INE423A01024 / ADANIENT: 2010-01-02 to 2015-04...
...,...,...,...,...,...,...,...,...,...
36,EXIT,LGBBROSLTD,0,False,False,False,True,NO_INTERVAL_MATCH,
42,EXIT,METALFORGE,0,False,False,False,True,NO_INTERVAL_MATCH,
53,EXIT,PENIND,0,False,False,False,True,NO_INTERVAL_MATCH,
63,EXIT,SRIPIPES,0,False,False,False,True,NO_INTERVAL_MATCH,



Saved summary: /content/nifty500_pit_data/official_review_post_stage_2016_v1.csv
Saved interval details: /content/nifty500_pit_data/official_review_post_stage_interval_details_2016_v1.csv
No interval or price records were modified.


In [24]:
from pathlib import Path
import pandas as pd

BASE = Path("/content/nifty500_pit_data")

SUMMARY_PATH = BASE / "official_review_post_stage_2016_v1.csv"
DETAIL_PATH = BASE / "official_review_post_stage_interval_details_2016_v1.csv"

OUT_PATH = BASE / "official_review_issue_patterns_2016_v1.csv"

assert SUMMARY_PATH.exists(), f"Missing: {SUMMARY_PATH}"
assert DETAIL_PATH.exists(), f"Missing: {DETAIL_PATH}"

summary = pd.read_csv(SUMMARY_PATH)
details = pd.read_csv(DETAIL_PATH)

mar31 = pd.Timestamp("2016-03-31")
apr01 = pd.Timestamp("2016-04-01")

details["_start"] = pd.to_datetime(details["Start_Date"], errors="coerce")
details["_end"] = pd.to_datetime(details["End_Date"], errors="coerce")

results = []

for _, s in summary.iterrows():
    role = str(s["Role"]).upper()
    symbol = str(s["Official_Symbol"]).strip().upper()

    m = details[
        details["Role"].astype(str).str.upper().eq(role)
        & details["Official_Symbol"].astype(str).str.upper().eq(symbol)
    ].copy()

    if m.empty:
        pattern = "NO_MATCHING_INTERVAL_OR_ALIAS"
        ranges = ""
        confidences = ""
    else:
        starts = m["_start"]
        ends = m["_end"]

        active_mar31 = (
            starts.notna() & starts.le(mar31)
            & (ends.isna() | ends.ge(mar31))
        )
        active_apr01 = (
            starts.notna() & starts.le(apr01)
            & (ends.isna() | ends.ge(apr01))
        )

        start_conf = (
            m["Start_Confidence"].fillna("missing").astype(str).str.lower()
        )
        end_conf = (
            m["End_Confidence"].fillna("missing").astype(str).str.lower()
        )

        ranges = " | ".join(
            f"{r['ISIN']} {r['Interval_Symbol']}: "
            f"{r['Start_Date']} to {r['End_Date']}; "
            f"start={r['Start_Confidence']}; end={r['End_Confidence']}"
            for _, r in m.iterrows()
        )

        confidences = " | ".join(sorted(set(
            start_conf.tolist() + end_conf.tolist()
        )))

        if role == "ENTRY":
            if bool(active_apr01.any()):
                pattern = "ALREADY_ACTIVE_ON_EFFECTIVE_DATE"
            elif bool(active_mar31.any()):
                if start_conf.str.contains("left_censored").any():
                    pattern = "ENTRY_BEFORE_DATE_LEFT_CENSORED_RANGE"
                else:
                    pattern = "ENTRY_BEFORE_DATE_EXISTING_RANGE"
            elif starts.notna().all() and starts.gt(apr01).all():
                pattern = "ENTRY_INTERVAL_STARTS_TOO_LATE"
            elif ends.notna().all() and ends.lt(apr01).all():
                pattern = "ENTRY_INTERVALS_END_BEFORE_EFFECTIVE_DATE"
            else:
                pattern = "ENTRY_HISTORY_GAP_OR_AMBIGUOUS"

        else:  # EXIT
            if bool(active_apr01.any()):
                if ends.isna().any() or ends.gt(mar31).any():
                    pattern = "EXIT_INTERVAL_CONTINUES_AFTER_REVIEW"
                else:
                    pattern = "EXIT_OVERLAPPING_INTERVALS"
            elif bool(active_mar31.any()):
                pattern = "ACTIVE_THROUGH_LAST_MEMBER_DATE"
            elif ends.notna().all() and ends.lt(mar31).all():
                pattern = "EXIT_INTERVALS_END_BEFORE_REVIEW"
            elif starts.notna().all() and starts.gt(mar31).all():
                pattern = "EXIT_INTERVALS_START_AFTER_REVIEW"
            else:
                pattern = "EXIT_HISTORY_GAP_OR_AMBIGUOUS"

    results.append({
        "Role": role,
        "Official_Symbol": symbol,
        "Current_QA_Status": s["Status"],
        "Interval_Rows": len(m),
        "Issue_Pattern": pattern,
        "Confidence_Labels": confidences,
        "Interval_Ranges": ranges
    })

report = pd.DataFrame(results)
issues = report[report["Current_QA_Status"] != "PASS"].copy()

print("=" * 70)
print("2016 MEMBERSHIP ISSUE PATTERNS — READ ONLY")
print("=" * 70)

print("\nCOUNTS BY ISSUE PATTERN")
pattern_counts = (
    issues.groupby(["Role", "Issue_Pattern"])
    .size()
    .reset_index(name="Symbols")
    .sort_values(["Role", "Issue_Pattern"])
)
display(pattern_counts)

print("\nCONFIDENCE LABELS AMONG OUTSTANDING CASES")
confidence_counts = (
    issues.assign(
        Confidence_Labels=issues["Confidence_Labels"].replace("", "(none)")
    )
    .groupby(["Role", "Confidence_Labels"])
    .size()
    .reset_index(name="Symbols")
    .sort_values(["Role", "Symbols"], ascending=[True, False])
)
display(confidence_counts)

# Show a few examples per pattern, including the original interval ranges.
print("\nEXAMPLES FOR REVIEW")
examples = (
    issues.sort_values(["Role", "Issue_Pattern", "Official_Symbol"])
    .groupby(["Role", "Issue_Pattern"], group_keys=False)
    .head(3)
)
display(examples[[
    "Role", "Official_Symbol", "Current_QA_Status",
    "Issue_Pattern", "Confidence_Labels", "Interval_Ranges"
]])

report.to_csv(OUT_PATH, index=False)
print("\nFull diagnostic saved:", OUT_PATH)
print("No interval, price, or membership flag was modified.")

2016 MEMBERSHIP ISSUE PATTERNS — READ ONLY

COUNTS BY ISSUE PATTERN


,Role,Issue_Pattern,Symbols
0,ENTRY,ALREADY_ACTIVE_ON_EFFECTIVE_DATE,3
1,ENTRY,ENTRY_HISTORY_GAP_OR_AMBIGUOUS,1
2,ENTRY,ENTRY_INTERVALS_END_BEFORE_EFFECTIVE_DATE,9
3,ENTRY,ENTRY_INTERVAL_STARTS_TOO_LATE,24
4,ENTRY,NO_MATCHING_INTERVAL_OR_ALIAS,19
5,EXIT,EXIT_INTERVALS_END_BEFORE_REVIEW,3
6,EXIT,EXIT_INTERVALS_START_AFTER_REVIEW,2
7,EXIT,EXIT_INTERVAL_CONTINUES_AFTER_REVIEW,47
8,EXIT,NO_MATCHING_INTERVAL_OR_ALIAS,7



CONFIDENCE LABELS AMONG OUTSTANDING CASES


,Role,Confidence_Labels,Symbols
0,ENTRY,(none),19
1,ENTRY,interpolated_long_gap,9
6,ENTRY,interpolated_long_gap | still_active,8
4,ENTRY,interpolated_long_gap | press_release_exact,7
7,ENTRY,left_censored | press_release_exact,6
2,ENTRY,interpolated_long_gap | left_censored,4
3,ENTRY,interpolated_long_gap | left_censored | press_...,1
5,ENTRY,interpolated_long_gap | press_release_exact | ...,1
8,ENTRY,press_release_exact | still_active,1
14,EXIT,interpolated_long_gap | left_censored,15



EXAMPLES FOR REVIEW


,Role,Official_Symbol,Current_QA_Status,Issue_Pattern,Confidence_Labels,Interval_Ranges
104,ENTRY,IDFC,ENTRY_ACTIVE_BEFORE_EFFECTIVE_DATE,ALREADY_ACTIVE_ON_EFFECTIVE_DATE,interpolated_long_gap | left_censored,INE043D01016 IDFC: 2010-01-02 to 2024-10-11; s...
113,ENTRY,KESORAMIND,ENTRY_ACTIVE_BEFORE_EFFECTIVE_DATE,ALREADY_ACTIVE_ON_EFFECTIVE_DATE,left_censored | press_release_exact,INE087A01019 KESORAMIND: 2010-01-02 to 2018-08...
127,ENTRY,RTNPOWER,ENTRY_ACTIVE_BEFORE_EFFECTIVE_DATE,ALREADY_ACTIVE_ON_EFFECTIVE_DATE,interpolated_long_gap | press_release_exact,INE399K01017 RTNPOWER: 2012-01-12 to 2018-08-2...
78,ENTRY,ADANIENT,ENTRY_NOT_ACTIVE_ON_EFFECTIVE_DATE,ENTRY_HISTORY_GAP_OR_AMBIGUOUS,interpolated_long_gap | left_censored | press_...,INE423A01024 ADANIENT: 2010-01-02 to 2015-04-2...
84,ENTRY,BGRENERGY,ENTRY_NOT_ACTIVE_ON_EFFECTIVE_DATE,ENTRY_INTERVALS_END_BEFORE_EFFECTIVE_DATE,interpolated_long_gap | press_release_exact,INE661I01014 BGRENERGY: 2012-01-12 to 2015-08-...
86,ENTRY,BRFL,ENTRY_NOT_ACTIVE_ON_EFFECTIVE_DATE,ENTRY_INTERVALS_END_BEFORE_EFFECTIVE_DATE,left_censored | press_release_exact,INE589G01011 BRFL: 2010-01-02 to 2014-02-27; s...
100,ENTRY,HMT,ENTRY_NOT_ACTIVE_ON_EFFECTIVE_DATE,ENTRY_INTERVALS_END_BEFORE_EFFECTIVE_DATE,interpolated_long_gap | left_censored,INE262A01018 HMT: 2010-01-02 to 2012-01-12; st...
76,ENTRY,AARTIDRUGS,ENTRY_NOT_ACTIVE_ON_EFFECTIVE_DATE,ENTRY_INTERVAL_STARTS_TOO_LATE,interpolated_long_gap | press_release_exact,INE767A01016 AARTIDRUGS: 2022-05-02 to 2024-02...
91,ENTRY,DHANUKA,ENTRY_NOT_ACTIVE_ON_EFFECTIVE_DATE,ENTRY_INTERVAL_STARTS_TOO_LATE,interpolated_long_gap | press_release_exact,INE435G01025 DHANUKA: 2020-06-10 to 2022-05-02...
114,ENTRY,KIRLOSENG,ENTRY_NOT_ACTIVE_ON_EFFECTIVE_DATE,ENTRY_INTERVAL_STARTS_TOO_LATE,interpolated_long_gap | still_active,INE146L01010 KIRLOSENG: 2024-10-11 to nan; sta...



Full diagnostic saved: /content/nifty500_pit_data/official_review_issue_patterns_2016_v1.csv
No interval, price, or membership flag was modified.


In [25]:
from pathlib import Path
from urllib.parse import urljoin
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup

BASE = Path("/content/nifty500_pit_data")
OUT = BASE / "nifty500_official_release_catalog_v1.csv"

URL = "https://www.niftyindices.com/press-release"

response = requests.get(
    URL,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=45
)
response.raise_for_status()

soup = BeautifulSoup(response.content, "html.parser")

date_pattern = re.compile(
    r"\b(?:Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|"
    r"May|Jun(?:e)?|Jul(?:y)?|Aug(?:ust)?|Sep(?:tember)?|"
    r"Oct(?:ober)?|Nov(?:ember)?|Dec(?:ember)?)\s+"
    r"\d{1,2},?\s+\d{4}\b",
    flags=re.I
)

terms = (
    "change in indices",
    "change in index",
    "change in nifty 500",
    "change in cnx 500",
    "nifty 500 &",
    "replacement in indices",
    "replacements in indices",
    "replacement in nifty 500",
    "changes in indices",
)

rows = []
seen_urls = set()

for a in soup.find_all("a", href=True):
    title = " ".join(a.get_text(" ", strip=True).split())
    lower = title.lower()

    if not title or not any(term in lower for term in terms):
        continue

    # Skip releases explicitly limited to fixed-income indices.
    if any(term in lower for term in (
        "fixed income", "g-sec", "gsec", "corporate bond"
    )):
        continue

    link = urljoin(URL, a["href"])

    if link in seen_urls:
        continue
    seen_urls.add(link)

    # Find the nearby publication date, excluding the link title
    # so a date in "effective from..." is not mistaken for publication.
    published_date = None
    for ancestor in list(a.parents)[:7]:
        nearby = " ".join(ancestor.get_text(" ", strip=True).split())

        if len(nearby) > 700:
            continue

        remainder = nearby.replace(title, " ", 1)
        matches = date_pattern.findall(remainder)

        if matches:
            published_date = matches[-1]
            break

    rows.append({
        "Title": title,
        "Published_Date": published_date,
        "URL": link
    })

catalog = pd.DataFrame(rows)

assert not catalog.empty, (
    "No candidate releases found. Check the archive response and HTML."
)

catalog["Parsed_Published_Date"] = pd.to_datetime(
    catalog["Published_Date"], errors="coerce"
)

# Keep releases in the project period. Preserve undated candidates
# for manual inspection rather than silently discarding them.
catalog = catalog[
    catalog["Parsed_Published_Date"].isna()
    | catalog["Parsed_Published_Date"].between(
        "2015-01-01", "2026-10-25"
    )
].copy()

catalog = catalog.sort_values(
    ["Parsed_Published_Date", "Title"],
    na_position="last"
)

catalog.drop(columns=["Parsed_Published_Date"]).to_csv(OUT, index=False)

print("OFFICIAL RELEASE CATALOG — READ ONLY")
print("Candidate releases:", len(catalog))
print("Releases with publication dates:", int(catalog["Published_Date"].notna().sum()))
print("Releases without parsed dates:", int(catalog["Published_Date"].isna().sum()))
print("Saved:", OUT)

display(catalog.drop(columns=["Parsed_Published_Date"]).tail(60))

OFFICIAL RELEASE CATALOG — READ ONLY
Candidate releases: 157
Releases with publication dates: 157
Releases without parsed dates: 0
Saved: /content/nifty500_pit_data/nifty500_official_release_catalog_v1.csv


,Title,Published_Date,URL
59,"Replacements in indices w.e.f. September 29, 2023","Sep 15, 2023",https://www.niftyindices.com/Press_Release/ind...
58,"Replacements in indices w.e.f. October 26, 2023","Oct 17, 2023",https://www.niftyindices.com/Press_Release/ind...
57,"Replacements in indices w.e.f. December 29, 2023","Dec 07, 2023",https://www.niftyindices.com/Press_Release/ind...
56,"Replacement in indices w.e.f. January 19, 2024","Jan 10, 2024",https://www.niftyindices.com/Press_Release/ind...
55,"Replacement in indices w.e.f. January 31, 2024","Jan 19, 2024",https://www.niftyindices.com/Press_Release/ind...
54,"Replacement in indices w.e.f. February 06, 2024","Jan 30, 2024",https://www.niftyindices.com/Press_Release/ind...
53,"Replacements in indices w.e.f. March 28, 2024","Feb 28, 2024",https://www.niftyindices.com/Press_Release/ind...
52,"Replacements in indices w.e.f. March 28, 2024","Mar 14, 2024",https://www.niftyindices.com/Press_Release/ind...
51,Replacements in indices w.e.f. March 28 2024,"Mar 19, 2024",https://www.niftyindices.com/Press_Release/ind...
49,Replacement in indices w.e.f. April 26 2024,"Apr 24, 2024",https://www.niftyindices.com/Press_Release/ind...


In [26]:
from pathlib import Path
import re
import sys
import subprocess
import time
import requests
import pandas as pd

BASE = Path("/content/nifty500_pit_data")
CATALOG_PATH = BASE / "nifty500_official_release_catalog_v1.csv"

CACHE_DIR = BASE / "official_release_pdf_cache"
TEXT_DIR = BASE / "official_release_text_cache"
SCAN_PATH = BASE / "nifty500_official_pdf_scan_v1.csv"

assert CATALOG_PATH.exists(), f"Missing catalog: {CATALOG_PATH}"

try:
    import fitz  # PyMuPDF
except ImportError:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "pymupdf"]
    )
    import fitz

CACHE_DIR.mkdir(parents=True, exist_ok=True)
TEXT_DIR.mkdir(parents=True, exist_ok=True)

catalog = pd.read_csv(CATALOG_PATH)
assert {"Title", "Published_Date", "URL"}.issubset(catalog.columns)

catalog["Published_Date_Parsed"] = pd.to_datetime(
    catalog["Published_Date"], errors="coerce"
)

# Persistent cache: on a rerun, use already-extracted text.
session = requests.Session()
session.headers.update({"User-Agent": "Mozilla/5.0 (compatible; research)"})

scan_rows = []
total = len(catalog)

for n, (_, item) in enumerate(catalog.iterrows(), start=1):
    title = str(item["Title"])
    published = item["Published_Date"]
    url = str(item["URL"]).strip()

    pdf_path = CACHE_DIR / f"release_{n:03d}.pdf"
    txt_path = TEXT_DIR / f"release_{n:03d}.txt"

    status = "cached_text"
    page_count = None
    byte_count = None
    full_text = ""
    error = ""

    try:
        if txt_path.exists() and txt_path.stat().st_size > 0:
            full_text = txt_path.read_text(
                encoding="utf-8", errors="replace"
            )
            if pdf_path.exists():
                byte_count = pdf_path.stat().st_size

        else:
            if not pdf_path.exists() or pdf_path.stat().st_size == 0:
                response = session.get(url, timeout=45)
                response.raise_for_status()

                content = response.content
                if not content.startswith(b"%PDF"):
                    raise ValueError(
                        "URL response does not begin with a PDF signature"
                    )

                pdf_path.write_bytes(content)

            byte_count = pdf_path.stat().st_size

            with fitz.open(pdf_path) as doc:
                page_count = len(doc)
                page_texts = [
                    f"\n--- PAGE {i + 1} ---\n{page.get_text('text')}"
                    for i, page in enumerate(doc)
                ]
                full_text = "\n".join(page_texts)

            txt_path.write_text(full_text, encoding="utf-8")
            status = "downloaded_and_extracted"

        # Match both modern NIFTY 500 and historical CNX 500 names.
        pattern = re.compile(
            r"\b(?:NIFTY|CNX)\s*[-_ ]?\s*500\b",
            re.IGNORECASE
        )

        matches = list(pattern.finditer(full_text))
        pages = sorted(set(
            int(m.group(1))
            for m in re.finditer(
                r"--- PAGE (\d+) ---", full_text
            )
            if False
        ))

        # Determine the page numbers that contain index mentions.
        current_page = 0
        hit_pages = set()
        mentions = 0

        for block in re.split(r"--- PAGE (\d+) ---", full_text):
            # Split markers are interleaved with the text.
            pass

        page_blocks = re.split(
            r"\n--- PAGE (\d+) ---\n", full_text
        )

        for i in range(1, len(page_blocks), 2):
            page_num = int(page_blocks[i])
            page_body = page_blocks[i + 1] if i + 1 < len(page_blocks) else ""
            page_hits = list(pattern.finditer(page_body))
            if page_hits:
                hit_pages.add(page_num)
                mentions += len(page_hits)

        # Keep a concise excerpt for initial inspection.
        excerpt = ""
        if matches:
            first = matches[0]
            excerpt = re.sub(
                r"\s+", " ",
                full_text[max(0, first.start() - 250):
                          min(len(full_text), first.end() + 500)]
            )

        scan_rows.append({
            "Title": title,
            "Published_Date": published,
            "URL": url,
            "PDF_Status": status,
            "Pages": page_count,
            "Bytes": byte_count,
            "Nifty500_Mention": bool(matches),
            "Nifty500_Mentions": mentions,
            "Hit_Pages": ", ".join(map(str, sorted(hit_pages))),
            "Excerpt": excerpt,
            "Text_Cache": str(txt_path),
            "Error": ""
        })

    except Exception as e:
        scan_rows.append({
            "Title": title,
            "Published_Date": published,
            "URL": url,
            "PDF_Status": "failed",
            "Pages": None,
            "Bytes": byte_count,
            "Nifty500_Mention": False,
            "Nifty500_Mentions": 0,
            "Hit_Pages": "",
            "Excerpt": "",
            "Text_Cache": "",
            "Error": f"{type(e).__name__}: {e}"
        })

    if n % 10 == 0 or n == total:
        print(f"Processed {n}/{total} catalog entries")

    time.sleep(0.05)

scan = pd.DataFrame(scan_rows)
scan.to_csv(SCAN_PATH, index=False)

hits = scan[scan["Nifty500_Mention"]].copy()
failures = scan[scan["PDF_Status"].eq("failed")].copy()

print("\n" + "=" * 70)
print("OFFICIAL PDF SCAN SUMMARY")
print("=" * 70)
print("Catalog entries:", len(scan))
print("PDFs mentioning Nifty 500 / CNX 500:", len(hits))
print("Downloads or extraction failures:", len(failures))
print("Extracted text files cached:", int(scan["Text_Cache"].ne("").sum()))
print("Full scan report:", SCAN_PATH)

print("\nRELEASES MENTIONING NIFTY 500")
display(
    hits[[
        "Published_Date", "Title", "Hit_Pages",
        "Nifty500_Mentions", "Excerpt", "URL"
    ]].sort_values("Published_Date")
)

if not failures.empty:
    print("\nFAILED DOCUMENTS TO RETRY OR INSPECT")
    display(failures[["Published_Date", "Title", "URL", "Error"]])

/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


Processed 10/157 catalog entries
Processed 20/157 catalog entries
Processed 30/157 catalog entries
Processed 40/157 catalog entries
Processed 50/157 catalog entries
Processed 60/157 catalog entries
Processed 70/157 catalog entries
Processed 80/157 catalog entries
Processed 90/157 catalog entries
Processed 100/157 catalog entries
Processed 110/157 catalog entries
Processed 120/157 catalog entries
Processed 130/157 catalog entries
Processed 140/157 catalog entries
Processed 150/157 catalog entries
Processed 157/157 catalog entries

OFFICIAL PDF SCAN SUMMARY
Catalog entries: 157
PDFs mentioning Nifty 500 / CNX 500: 134
Downloads or extraction failures: 0
Extracted text files cached: 157
Full scan report: /content/nifty500_pit_data/nifty500_official_pdf_scan_v1.csv

RELEASES MENTIONING NIFTY 500


,Published_Date,Title,Hit_Pages,Nifty500_Mentions,Excerpt,URL
127,"Apr 04, 2025","Replacement in indices w.e.f. April 11, 2025","1, 3",5,cided to make the following replacement in ind...,https://www.niftyindices.com/Press_Release/ind...
73,"Apr 05, 2022","Replacements in indices w.e.f. May 04, 2022",1,2,es Ltd. from various indices as listed hereund...,https://www.niftyindices.com/Press_Release/ind...
72,"Apr 05, 2022","Replacements in indices w.e.f. April 12, 2022","1, 2",3,ous indices as listed hereunder on account of ...,https://www.niftyindices.com/Press_Release/ind...
74,"Apr 06, 2022","Replacement in indices w.e.f. April 07, 2022","1, 2",3,Maintenance Sub-Committee - Equity (IMSC) of ...,https://www.niftyindices.com/Press_Release/ind...
42,"Apr 08, 2019","Change in Indices w.e.f. April 15, 2019",1,1,SE Indices Limited has decided to replace Exce...,https://www.niftyindices.com/Press_Release/ind...
...,...,...,...,...,...,...
46,"Sep 17, 2019",Revision in criteria of NIFTY Infrastructure I...,"1, 2",2,account of scheme of arrangement: On account ...,https://www.niftyindices.com/Press_Release/ind...
45,"Sep 17, 2019","Replacement in indices w.e.f. September 24, 2019",1,2,egory: NSE vide its circular no. NSE/CML/42146...,https://www.niftyindices.com/Press_Release/ind...
9,"Sep 18, 2015","Change in Indices w.e.f October 19, 2015",2,1,e following company is being excluded: Sr. No....,https://www.niftyindices.com/Press_Release/ind...
115,"Sep 23, 2024",Replacements in indices and revision in guidel...,5,1,following company is being excluded: Sr. No. C...,https://www.niftyindices.com/Press_Release/ind...


In [27]:
from pathlib import Path
import re
import pandas as pd

BASE = Path("/content/nifty500_pit_data")

SCAN_PATH = BASE / "nifty500_official_pdf_scan_v1.csv"
OCCURRENCES_PATH = BASE / "nifty500_event_page_occurrences_v1.csv"
CANDIDATES_PATH = BASE / "nifty500_event_candidates_v1.csv"

assert SCAN_PATH.exists(), f"Missing scan report: {SCAN_PATH}"

scan = pd.read_csv(SCAN_PATH)

nifty500_pattern = re.compile(
    r"\b(?:NIFTY|CNX)\s*[-_ ]?\s*500\b",
    re.IGNORECASE
)

action_pattern = re.compile(
    r"replacement|replaced|excluded|includ(?:ed|ing|e)|"
    r"addition|added|removed|delet(?:ed|ion)|"
    r"constituent|effective from|change in indices|"
    r"changes in indices|following companies",
    re.IGNORECASE
)

# Extract the effective date when explicitly stated in the release title.
effective_pattern = re.compile(
    r"\bw\.?\s*e\.?\s*f\.?\s+"
    r"(\d{1,2}\s+[A-Za-z]+\s+\d{4}|"
    r"[A-Za-z]+\s+\d{1,2},?\s+\d{4})",
    re.IGNORECASE
)

def extract_title_effective_date(title):
    match = effective_pattern.search(str(title))
    if not match:
        return None

    parsed = pd.to_datetime(match.group(1), errors="coerce")
    if pd.isna(parsed):
        return None

    return parsed.strftime("%Y-%m-%d")

occurrence_rows = []
candidate_rows = []

for _, release in scan.iterrows():
    text_path = str(release.get("Text_Cache", "")).strip()

    if not text_path or not Path(text_path).exists():
        continue

    full_text = Path(text_path).read_text(
        encoding="utf-8", errors="replace"
    )

    # Extract page-numbered blocks from the previously cached text.
    blocks = re.split(r"\n--- PAGE (\d+) ---\n", full_text)
    release_pages = []

    for i in range(1, len(blocks), 2):
        page_number = int(blocks[i])
        page_text = blocks[i + 1] if i + 1 < len(blocks) else ""

        matches = list(nifty500_pattern.finditer(page_text))
        if not matches:
            continue

        has_action_language = bool(action_pattern.search(page_text))

        snippets = []
        for match in matches:
            start = max(0, match.start() - 500)
            end = min(len(page_text), match.end() + 1400)

            snippet = re.sub(
                r"[ \t]+", " ",
                page_text[start:end]
            ).strip()

            snippets.append(snippet)

            occurrence_rows.append({
                "Published_Date": release.get("Published_Date"),
                "Title": release.get("Title"),
                "URL": release.get("URL"),
                "Effective_Date_From_Title":
                    extract_title_effective_date(release.get("Title")),
                "Page": page_number,
                "Action_Language_On_Page": has_action_language,
                "Occurrence_Context": snippet
            })

        release_pages.append({
            "page": page_number,
            "has_action_language": has_action_language,
            "snippet": "\n---\n".join(snippets[:3])
        })

    if release_pages:
        title = str(release.get("Title", ""))
        action_page_count = sum(
            page["has_action_language"] for page in release_pages
        )

        candidate_rows.append({
            "Published_Date": release.get("Published_Date"),
            "Title": title,
            "URL": release.get("URL"),
            "Effective_Date_From_Title":
                extract_title_effective_date(title),
            "Nifty500_Pages": ", ".join(
                str(page["page"]) for page in release_pages
            ),
            "Nifty500_Page_Count": len(release_pages),
            "Pages_With_Action_Language": action_page_count,
            "Likely_Change_Release": action_page_count > 0,
            "Context_Excerpt": "\n\n".join(
                page["snippet"] for page in release_pages[:2]
            )
        })

occurrences = pd.DataFrame(occurrence_rows)
candidates = pd.DataFrame(candidate_rows)

assert not candidates.empty, (
    "No Nifty 500 context found in cached text. "
    "Check the extracted text paths."
)

occurrences.to_csv(OCCURRENCES_PATH, index=False)
candidates.to_csv(CANDIDATES_PATH, index=False)

likely = candidates[
    candidates["Likely_Change_Release"]
].copy()

print("=" * 70)
print("NIFTY 500 EVENT DISCOVERY — READ ONLY")
print("=" * 70)
print("PDFs with Nifty 500 page context:", len(candidates))
print("PDF page occurrences recorded:", len(occurrences))
print("Likely change releases for review:", len(likely))
print(
    "Effective dates explicitly found in likely release titles:",
    int(likely["Effective_Date_From_Title"].notna().sum())
)
print("\nSaved page contexts:", OCCURRENCES_PATH)
print("Saved candidate releases:", CANDIDATES_PATH)

print("\nLIKELY CONSTITUENT-CHANGE RELEASES")
display(
    likely[
        [
            "Published_Date",
            "Effective_Date_From_Title",
            "Title",
            "Nifty500_Pages",
            "Pages_With_Action_Language",
            "URL"
        ]
    ].sort_values(
        ["Published_Date", "Title"],
        na_position="last"
    ).reset_index(drop=True)
)

print("\nRELEASES THAT MAY NEED MANUAL CLASSIFICATION")
display(
    candidates[
        ~candidates["Likely_Change_Release"]
    ][["Published_Date", "Title", "Nifty500_Pages", "URL"]]
    .head(20)
)

NIFTY 500 EVENT DISCOVERY — READ ONLY
PDFs with Nifty 500 page context: 134
PDF page occurrences recorded: 338
Likely change releases for review: 134
Effective dates explicitly found in likely release titles: 113

Saved page contexts: /content/nifty500_pit_data/nifty500_event_page_occurrences_v1.csv
Saved candidate releases: /content/nifty500_pit_data/nifty500_event_candidates_v1.csv

LIKELY CONSTITUENT-CHANGE RELEASES


,Published_Date,Effective_Date_From_Title,Title,Nifty500_Pages,Pages_With_Action_Language,URL
0,"Apr 04, 2025",2025-04-11,"Replacement in indices w.e.f. April 11, 2025","1, 3",2,https://www.niftyindices.com/Press_Release/ind...
1,"Apr 05, 2022",2022-04-12,"Replacements in indices w.e.f. April 12, 2022","1, 2",2,https://www.niftyindices.com/Press_Release/ind...
2,"Apr 05, 2022",2022-05-04,"Replacements in indices w.e.f. May 04, 2022",1,1,https://www.niftyindices.com/Press_Release/ind...
3,"Apr 06, 2022",2022-04-07,"Replacement in indices w.e.f. April 07, 2022","1, 2",2,https://www.niftyindices.com/Press_Release/ind...
4,"Apr 08, 2019",2019-04-15,"Change in Indices w.e.f. April 15, 2019",1,1,https://www.niftyindices.com/Press_Release/ind...
...,...,...,...,...,...,...
129,"Sep 17, 2019",2019-09-24,"Replacement in indices w.e.f. September 24, 2019",1,1,https://www.niftyindices.com/Press_Release/ind...
130,"Sep 17, 2019",2019-09-27,Revision in criteria of NIFTY Infrastructure I...,"1, 2",2,https://www.niftyindices.com/Press_Release/ind...
131,"Sep 18, 2015",2015-10-19,"Change in Indices w.e.f October 19, 2015",2,1,https://www.niftyindices.com/Press_Release/ind...
132,"Sep 23, 2024",None,Replacements in indices and revision in guidel...,5,1,https://www.niftyindices.com/Press_Release/ind...



RELEASES THAT MAY NEED MANUAL CLASSIFICATION


,Published_Date,Title,Nifty500_Pages,URL


In [28]:
from pathlib import Path
import re
import pandas as pd

BASE = Path("/content/nifty500_pit_data")
SCAN_PATH = BASE / "nifty500_official_pdf_scan_v1.csv"

OUT_OCCURRENCES = BASE / "nifty500_event_context_triage_v2.csv"
OUT_RELEASES = BASE / "nifty500_release_triage_v2.csv"

assert SCAN_PATH.exists(), f"Missing scan report: {SCAN_PATH}"

scan = pd.read_csv(SCAN_PATH)

index_pattern = re.compile(
    r"\b(?:NIFTY|CNX)\s*[-_ ]?\s*500\b",
    re.IGNORECASE
)

addition_pattern = re.compile(
    r"\b(addition|added|addition of|included|inclusion|"
    r"inducted|induction|entering|entry|entries|"
    r"new constituent|new constituents|"
    r"being included|being added|"
    r"will be included|will be added)\b",
    re.IGNORECASE
)

removal_pattern = re.compile(
    r"\b(exclusion|excluded|exclude|deletion|deleted|"
    r"removed|removal|being excluded|being deleted|"
    r"will be excluded|will be deleted|"
    r"outgoing constituent|outgoing constituents)\b",
    re.IGNORECASE
)

effective_pattern = re.compile(
    r"(?:w\.?\s*e\.?\s*f\.?|with effect from|"
    r"effective from|effective date|"
    r"after the close of|from the close of)"
    r".{0,100}",
    re.IGNORECASE
)

occurrences = []

for _, release in scan.iterrows():
    text_path = str(release.get("Text_Cache", "")).strip()

    if not text_path or not Path(text_path).exists():
        continue

    full_text = Path(text_path).read_text(
        encoding="utf-8", errors="replace"
    )

    # Split into page-numbered text blocks.
    blocks = re.split(r"\n--- PAGE (\d+) ---\n", full_text)

    for i in range(1, len(blocks), 2):
        page_num = int(blocks[i])
        page_text = blocks[i + 1] if i + 1 < len(blocks) else ""

        for match in index_pattern.finditer(page_text):
            # Keep a window around the exact Nifty 500 mention.
            start = max(0, match.start() - 900)
            end = min(len(page_text), match.end() + 1800)
            context = page_text[start:end]
            normalized = re.sub(r"\s+", " ", context).strip()

            add_hits = len(addition_pattern.findall(context))
            remove_hits = len(removal_pattern.findall(context))
            date_phrases = effective_pattern.findall(context)

            # Explicit mentions near the index name are stronger
            # than generic action language elsewhere on the page.
            immediate_start = max(0, match.start() - 300)
            immediate_end = min(len(page_text), match.end() + 600)
            immediate = page_text[immediate_start:immediate_end]

            immediate_add = bool(addition_pattern.search(immediate))
            immediate_remove = bool(removal_pattern.search(immediate))

            title = str(release.get("Title", ""))
            title_date = release.get("Published_Date", "")

            score = (
                int(add_hits > 0)
                + int(remove_hits > 0)
                + int(bool(date_phrases))
                + int(immediate_add or immediate_remove)
            )

            occurrences.append({
                "Published_Date": title_date,
                "Title": title,
                "URL": release.get("URL", ""),
                "Page": page_num,
                "Index_Mention": match.group(0),
                "Nearby_Addition_Terms": add_hits,
                "Nearby_Removal_Terms": remove_hits,
                "Nearby_Effective_Date_Phrases": len(date_phrases),
                "Immediate_Addition_Terms": immediate_add,
                "Immediate_Removal_Terms": immediate_remove,
                "Context_Score": score,
                "Context": normalized
            })

assert occurrences, "No Nifty 500 mention contexts were extracted."

occ = pd.DataFrame(occurrences)

# Roll the occurrence-level evidence up to release level.
release_rows = []

for (title, url), group in occ.groupby(["Title", "URL"], dropna=False):
    strong = group[
        group["Immediate_Addition_Terms"]
        | group["Immediate_Removal_Terms"]
    ]

    release_rows.append({
        "Published_Date": group["Published_Date"].iloc[0],
        "Title": title,
        "URL": url,
        "Nifty500_Mention_Count": len(group),
        "Pages_With_Mentions": ", ".join(
            map(str, sorted(group["Page"].unique()))
        ),
        "Pages_With_Immediate_Action_Terms":
            ", ".join(map(str, sorted(strong["Page"].unique()))),
        "Context_Score_Max": int(group["Context_Score"].max()),
        "Immediate_Action_Context_Count": len(strong),
        "Needs_Manual_Review": len(strong) == 0
    })

release_triage = pd.DataFrame(release_rows)

occ.to_csv(OUT_OCCURRENCES, index=False)
release_triage.to_csv(OUT_RELEASES, index=False)

print("=" * 70)
print("NIFTY 500 CONTEXT TRIAGE — READ ONLY")
print("=" * 70)
print("Nifty 500 context occurrences:", len(occ))
print("Distinct release titles:", len(release_triage))
print(
    "Releases with nearby explicit addition/removal language:",
    int((release_triage["Immediate_Action_Context_Count"] > 0).sum())
)
print(
    "Releases requiring manual review:",
    int(release_triage["Needs_Manual_Review"].sum())
)

print("\nRELEASES WITH STRONGER LOCAL EVIDENCE")
display(
    release_triage[
        release_triage["Immediate_Action_Context_Count"] > 0
    ].sort_values(["Published_Date", "Title"])
)

print("\nEXAMPLE CONTEXTS FOR INSPECTION")
display(
    occ.sort_values(
        ["Context_Score", "Published_Date"],
        ascending=[False, True]
    )[
        [
            "Published_Date", "Title", "Page",
            "Nearby_Addition_Terms", "Nearby_Removal_Terms",
            "Nearby_Effective_Date_Phrases", "Context"
        ]
    ].head(15)
)

print("\nSaved context report:", OUT_OCCURRENCES)
print("Saved release triage:", OUT_RELEASES)
print("No membership intervals, price rows or flags were modified.")

NIFTY 500 CONTEXT TRIAGE — READ ONLY
Nifty 500 context occurrences: 338
Distinct release titles: 134
Releases with nearby explicit addition/removal language: 133
Releases requiring manual review: 1

RELEASES WITH STRONGER LOCAL EVIDENCE


,Published_Date,Title,URL,Nifty500_Mention_Count,Pages_With_Mentions,Pages_With_Immediate_Action_Terms,Context_Score_Max,Immediate_Action_Context_Count,Needs_Manual_Review
42,"Apr 04, 2025","Replacement in indices w.e.f. April 11, 2025",https://www.niftyindices.com/Press_Release/ind...,5,"1, 3","1, 3",4,5,False
79,"Apr 05, 2022","Replacements in indices w.e.f. April 12, 2022",https://www.niftyindices.com/Press_Release/ind...,3,"1, 2","1, 2",4,3,False
109,"Apr 05, 2022","Replacements in indices w.e.f. May 04, 2022",https://www.niftyindices.com/Press_Release/ind...,2,1,1,4,2,False
41,"Apr 06, 2022","Replacement in indices w.e.f. April 07, 2022",https://www.niftyindices.com/Press_Release/ind...,3,"1, 2","1, 2",4,3,False
10,"Apr 08, 2019","Change in Indices w.e.f. April 15, 2019",https://www.niftyindices.com/Press_Release/ind...,1,1,1,4,1,False
...,...,...,...,...,...,...,...,...,...
61,"Sep 17, 2019","Replacement in indices w.e.f. September 24, 2019",https://www.niftyindices.com/Press_Release/ind...,2,1,1,4,2,False
132,"Sep 17, 2019",Revision in criteria of NIFTY Infrastructure I...,https://www.niftyindices.com/Press_Release/ind...,2,"1, 2","1, 2",4,2,False
6,"Sep 18, 2015","Change in Indices w.e.f October 19, 2015",https://www.niftyindices.com/Press_Release/ind...,1,2,2,3,1,False
76,"Sep 23, 2024",Replacements in indices and revision in guidel...,https://www.niftyindices.com/Press_Release/ind...,1,5,5,3,1,False



EXAMPLE CONTEXTS FOR INSPECTION


,Published_Date,Title,Page,Nearby_Addition_Terms,Nearby_Removal_Terms,Nearby_Effective_Date_Phrases,Context
243,"Apr 04, 2025","Replacement in indices w.e.f. April 11, 2025",1,2,2,1,"PRESS RELEASE Mumbai, April 04, 2025 Replaceme..."
244,"Apr 04, 2025","Replacement in indices w.e.f. April 11, 2025",1,2,2,1,"PRESS RELEASE Mumbai, April 04, 2025 Replaceme..."
245,"Apr 04, 2025","Replacement in indices w.e.f. April 11, 2025",1,2,2,1,"PRESS RELEASE Mumbai, April 04, 2025 Replaceme..."
246,"Apr 04, 2025","Replacement in indices w.e.f. April 11, 2025",1,2,2,1,"PRESS RELEASE Mumbai, April 04, 2025 Replaceme..."
113,"Apr 05, 2022","Replacements in indices w.e.f. April 12, 2022",1,2,2,1,"PRESS RELEASE Mumbai, April 05, 2022 Replaceme..."
114,"Apr 05, 2022","Replacements in indices w.e.f. April 12, 2022",1,2,2,1,"PRESS RELEASE Mumbai, April 05, 2022 Replaceme..."
116,"Apr 05, 2022","Replacements in indices w.e.f. May 04, 2022",1,2,2,1,"PRESS RELEASE Mumbai, April 05, 2022 Replaceme..."
117,"Apr 05, 2022","Replacements in indices w.e.f. May 04, 2022",1,2,2,1,"PRESS RELEASE Mumbai, April 05, 2022 Replaceme..."
118,"Apr 06, 2022","Replacement in indices w.e.f. April 07, 2022",1,2,2,2,"PRESS RELEASE Mumbai, April 06, 2022 Replaceme..."
119,"Apr 06, 2022","Replacement in indices w.e.f. April 07, 2022",1,2,2,2,"PRESS RELEASE Mumbai, April 06, 2022 Replaceme..."



Saved context report: /content/nifty500_pit_data/nifty500_event_context_triage_v2.csv
Saved release triage: /content/nifty500_pit_data/nifty500_release_triage_v2.csv
No membership intervals, price rows or flags were modified.


In [29]:
from pathlib import Path
import re
import pandas as pd

BASE = Path("/content/nifty500_pit_data")
SCAN_PATH = BASE / "nifty500_official_pdf_scan_v1.csv"

assert SCAN_PATH.exists(), f"Missing scan report: {SCAN_PATH}"

scan = pd.read_csv(SCAN_PATH)
scan["Parsed_Date"] = pd.to_datetime(
    scan["Published_Date"], errors="coerce"
)

docs_2016 = scan[
    scan["Parsed_Date"].dt.year.eq(2016)
    & scan["Nifty500_Mention"].astype(str).str.lower().isin(
        ["true", "1"]
    )
].copy()

docs_2016 = docs_2016.sort_values(
    ["Parsed_Date", "Title"]
)

print("2016 releases mentioning Nifty 500:", len(docs_2016))
display(
    docs_2016[
        ["Published_Date", "Title", "URL", "Text_Cache"]
    ].reset_index(drop=True)
)

def split_pages(text):
    parts = re.split(r"\n--- PAGE (\d+) ---\n", text)
    pages = []
    for i in range(1, len(parts), 2):
        page_num = int(parts[i])
        page_text = parts[i + 1] if i + 1 < len(parts) else ""
        pages.append((page_num, page_text))
    return pages

# Print the local line context around each Nifty 500 mention.
# No prices, flags, or membership intervals are changed.
for _, doc in docs_2016.iterrows():
    text_path = str(doc.get("Text_Cache", "")).strip()
    if not text_path or not Path(text_path).exists():
        continue

    text = Path(text_path).read_text(
        encoding="utf-8", errors="replace"
    )

    relevant_pages = []
    for page_num, page_text in split_pages(text):
        lines = page_text.splitlines()
        hit_indices = [
            i for i, line in enumerate(lines)
            if re.search(r"\b(?:NIFTY|CNX)\s*[-_ ]?\s*500\b",
                         line, re.IGNORECASE)
        ]
        if hit_indices:
            relevant_pages.append((page_num, lines, hit_indices))

    if not relevant_pages:
        continue

    print("\n" + "=" * 100)
    print(f"Published: {doc['Published_Date']} | {doc['Title']}")
    print(f"URL: {doc['URL']}")

    for page_num, lines, hit_indices in relevant_pages:
        print(f"\n--- PAGE {page_num} ---")

        # Merge nearby windows so duplicate mentions don't print
        # the same section repeatedly.
        windows = []
        for idx in hit_indices:
            start = max(0, idx - 12)
            end = min(len(lines), idx + 28)
            if windows and start <= windows[-1][1]:
                windows[-1] = (windows[-1][0], max(windows[-1][1], end))
            else:
                windows.append((start, end))

        for start, end in windows:
            for i in range(start, end):
                print(f"{i + 1:04d}: {lines[i]}")
            print("...")

print("\nInspection complete. No source data was modified.")

2016 releases mentioning Nifty 500: 5


,Published_Date,Title,URL,Text_Cache
0,"Jan 11, 2016","Change in Indices w.e.f January 18, 2016",https://www.niftyindices.com/Press_Release/ind...,/content/nifty500_pit_data/official_release_te...
1,"Feb 22, 2016","Change in Indices w.e.f. April 01, 2016.",https://www.niftyindices.com/Press_Release/ind...,/content/nifty500_pit_data/official_release_te...
2,"Apr 22, 2016","Change in Indices w.e.f. April 29, 2016.",https://www.niftyindices.com/Press_Release/ind...,/content/nifty500_pit_data/official_release_te...
3,"Aug 12, 2016","Change in Indices w.e.f. September 30, 2016.",https://www.niftyindices.com/Press_Release/ind...,/content/nifty500_pit_data/official_release_te...
4,"Oct 17, 2016","Change in Indices w.e.f. October 24, 2016 and ...",https://www.niftyindices.com/Press_Release/ind...,/content/nifty500_pit_data/official_release_te...



Published: Jan 11, 2016 | Change in Indices w.e.f January 18, 2016
URL: https://www.niftyindices.com/Press_Release/ind_prs11012016.pdf

--- PAGE 1 ---
0005:  
0006: Date: January 11, 2016 
0007:  
0008:  
0009: PRESS RELEASE 
0010:  
0011: The Index Maintenance Sub-Committee has decided to make the following 
0012: replacements in,  
0013:  
0014: A. Nifty Alpha 50, Nifty High Beta 50 and Nifty Low Volatility 50 indices as part of 
0015: the quarterly review, and  
0016:  
0017: B. Nifty 500, Nifty Smallcap 100 and Nifty High Beta 50 indices on account of 
0018: proposed scheme of arrangement for demerger of Power Products and 
0019: Transmission Grid Business from Sterlite Technologies Ltd. 
0020:  
0021: The changes shall become effective from January 18, 2016 (close of January 15, 2016). 
0022:  
0023:  
0024: 1) 
0025: Nifty Alpha 50 Index 
0026:  
0027: The following companies are being excluded: 
0028:  
0029: Sr. No. Company Name 
0030: Symbol 
0031: 1 
0032: Alstom T&D India L

In [30]:
from pathlib import Path
import re
import pandas as pd

BASE = Path("/content/nifty500_pit_data")
SCAN_PATH = BASE / "nifty500_official_pdf_scan_v1.csv"

EVENTS_PATH = BASE / "nifty500_official_event_candidates_v1.csv"
SUMMARY_PATH = BASE / "nifty500_event_extraction_summary_v1.csv"

assert SCAN_PATH.exists(), f"Missing scan report: {SCAN_PATH}"

scan = pd.read_csv(SCAN_PATH)

date_rx = re.compile(
    r"\b(?:Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|"
    r"May|Jun(?:e)?|Jul(?:y)?|Aug(?:ust)?|Sep(?:tember)?|"
    r"Oct(?:ober)?|Nov(?:ember)?|Dec(?:ember)?)\s+"
    r"\d{1,2},?\s+\d{4}\b"
    r"|\b\d{1,2}\s+"
    r"(?:Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|"
    r"May|Jun(?:e)?|Jul(?:y)?|Aug(?:ust)?|Sep(?:tember)?|"
    r"Oct(?:ober)?|Nov(?:ember)?|Dec(?:ember)?)\s+\d{4}\b",
    re.I
)

nifty500_heading_rx = re.compile(
    r"^(?:\d+\s*\)?\s*)?Nifty\s+500\s+Index$",
    re.I
)

other_index_heading_rx = re.compile(
    r"^(?:\d+\s*\)?\s*)?(?:Nifty|CNX)\s+.+\s+Index$",
    re.I
)

action_rx = re.compile(
    r"\bthe\s+following\s+"
    r"(?:companies|scrips|company|scrip)\s+"
    r"(?:are|is|will\s+be|shall\s+be)\s+"
    r"(?:(?:being|to\s+be)\s+)?"
    r"(excluded|included|removed|added)\b",
    re.I
)

symbol_rx = re.compile(r"^[A-Z0-9][A-Z0-9&.\-]{1,24}$")

header_symbols = {
    "SR", "SR.", "NO", "NO.", "SYMBOL", "SCRIP", "SCRIPS",
    "NAME", "COMPANY", "COMPANIES", "INDEX", "NIFTY", "CNX",
    "EXCLUDED", "INCLUDED", "REMOVED", "ADDED", "LTD", "LLP"
}

def parse_date(value):
    if not value:
        return None
    parsed = pd.to_datetime(value, errors="coerce")
    if pd.isna(parsed):
        return None
    return parsed.strftime("%Y-%m-%d")

def infer_effective_date(title, lines, section_start):
    """
    Prefer one unambiguous date from the title.
    When the title contains multiple dates, look for an explicit
    effective-date phrase after the last Nifty 500 mention in the
    introductory text before the section.
    """
    title_dates = []
    for phrase in date_rx.findall(str(title)):
        parsed = parse_date(phrase)
        if parsed and parsed not in title_dates:
            title_dates.append(parsed)

    if len(title_dates) == 1:
        return title_dates[0], "title_single_date"

    if section_start > 0:
        prefix = "\n".join(lines[:section_start])
        mentions = list(re.finditer(
            r"\b(?:Nifty|CNX)\s*[-_ ]?\s*500\b",
            prefix, re.I
        ))

        if mentions:
            last_mention = mentions[-1]
            local = prefix[last_mention.start():last_mention.start() + 1000]

            effective_phrase_rx = re.compile(
                r"(?:shall\s+become\s+effective\s+from|"
                r"become\s+effective\s+from|"
                r"effective\s+from|"
                r"with\s+effect\s+from|"
                r"w\.?\s*e\.?\s*f\.?)"
                r"\s*[:\-]?\s*"
                r"("
                r"(?:Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|"
                r"May|Jun(?:e)?|Jul(?:y)?|Aug(?:ust)?|Sep(?:tember)?|"
                r"Oct(?:ober)?|Nov(?:ember)?|Dec(?:ember)?)\s+"
                r"\d{1,2},?\s+\d{4}"
                r"|\d{1,2}\s+"
                r"(?:Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|"
                r"May|Jun(?:e)?|Jul(?:y)?|Aug(?:ust)?|Sep(?:tember)?|"
                r"Oct(?:ober)?|Nov(?:ember)?|Dec(?:ember)?)\s+\d{4}"
                r")",
                re.I
            )

            match = effective_phrase_rx.search(local)
            if match:
                parsed = parse_date(match.group(1))
                if parsed:
                    return parsed, "intro_clause_near_nifty500"

    return None, "unresolved_multiple_or_missing_dates"

def get_section_ranges(lines):
    starts = [
        i for i, line in enumerate(lines)
        if nifty500_heading_rx.fullmatch(line.strip())
    ]

    ranges = []
    for start in starts:
        end = len(lines)
        for j in range(start + 1, len(lines)):
            candidate = lines[j].strip()
            if (
                other_index_heading_rx.fullmatch(candidate)
                and not nifty500_heading_rx.fullmatch(candidate)
            ):
                end = j
                break
        ranges.append((start, end))

    return ranges

def extract_symbol_rows(block_text):
    """
    Text-based table assumption: a company name immediately precedes
    its uppercase ticker symbol. Every parsed row is retained for QA;
    this is a candidate extractor, not an authoritative parser.
    """
    raw_lines = [line.strip() for line in block_text.splitlines()]
    lines = [(i, line) for i, line in enumerate(raw_lines) if line]

    records = []
    seen = set()

    for pos, (raw_idx, line) in enumerate(lines):
        symbol = line.strip().upper()

        if not symbol_rx.fullmatch(symbol):
            continue
        if symbol in header_symbols:
            continue
        if not re.search(r"[A-Z]", symbol):
            continue
        if pos == 0:
            continue

        company = lines[pos - 1][1].strip()

        if not company or company.upper() in header_symbols:
            continue
        if re.fullmatch(r"[\d\s.)]+", company):
            continue
        if re.fullmatch(r"page\s+\d+", company, re.I):
            continue

        # Skip obvious sentence fragments; keep reviewable uncertain cases.
        if len(company) < 3:
            continue

        serial = None
        if pos >= 2:
            previous = lines[pos - 2][1].strip()
            if re.fullmatch(r"\d{1,3}", previous):
                serial = int(previous)

        key = (serial, company, symbol)
        if key in seen:
            continue
        seen.add(key)

        records.append({
            "Serial_No": serial,
            "Company_Name": company,
            "Symbol": symbol
        })

    return records

event_rows = []
summary_rows = []
release_failures = []

for _, release in scan.iterrows():
    title = str(release.get("Title", ""))
    published = release.get("Published_Date")
    url = str(release.get("URL", ""))
    text_path = str(release.get("Text_Cache", "")).strip()

    if not text_path or not Path(text_path).exists():
        release_failures.append({
            "Title": title,
            "URL": url,
            "Error": "Missing cached extracted text"
        })
        continue

    full_text = Path(text_path).read_text(
        encoding="utf-8", errors="replace"
    )
    full_text = re.sub(r"\n--- PAGE \d+ ---\n", "\n", full_text)
    lines = [line.strip() for line in full_text.splitlines()]

    ranges = get_section_ranges(lines)

    release_event_count = 0
    release_action_count = 0
    release_unresolved = False

    for section_number, (start, end) in enumerate(ranges, start=1):
        section_lines = lines[start + 1:end]
        section_text = "\n".join(section_lines)

        effective_date, date_source = infer_effective_date(
            title, lines, start
        )

        actions = list(action_rx.finditer(section_text))

        if not actions:
            summary_rows.append({
                "Published_Date": published,
                "Title": title,
                "URL": url,
                "Section_Number": section_number,
                "Effective_Date": effective_date,
                "Effective_Date_Source": date_source,
                "Exit_Rows_Parsed": 0,
                "Entry_Rows_Parsed": 0,
                "Action_Headings_Found": 0,
                "Section_Status": "NIFTY500_SECTION_NO_ACTION_HEADING"
            })
            release_unresolved |= effective_date is None
            continue

        section_exit_count = 0
        section_entry_count = 0

        for action_i, action in enumerate(actions):
            block_start = action.end()
            block_end = (
                actions[action_i + 1].start()
                if action_i + 1 < len(actions)
                else len(section_text)
            )
            block_text = section_text[block_start:block_end]

            action_word = action.group(1).lower()
            role = (
                "EXIT" if action_word in {"excluded", "removed"}
                else "ENTRY"
            )

            parsed_rows = extract_symbol_rows(block_text)
            release_action_count += 1

            if role == "EXIT":
                section_exit_count += len(parsed_rows)
            else:
                section_entry_count += len(parsed_rows)

            for parsed_row in parsed_rows:
                event_rows.append({
                    "Published_Date": published,
                    "Release_Title": title,
                    "Source_URL": url,
                    "Section_Number": section_number,
                    "Effective_Date": effective_date,
                    "Effective_Date_Source": date_source,
                    "Role": role,
                    "Company_Name": parsed_row["Company_Name"],
                    "Symbol": parsed_row["Symbol"],
                    "Serial_No": parsed_row["Serial_No"],
                    "Action_Heading": action.group(0),
                    "Event_Extraction_Status": (
                        "DATE_UNRESOLVED" if effective_date is None
                        else "CANDIDATE_REQUIRES_QA"
                    )
                })

        release_event_count += section_exit_count + section_entry_count
        release_unresolved |= effective_date is None

        summary_rows.append({
            "Published_Date": published,
            "Title": title,
            "URL": url,
            "Section_Number": section_number,
            "Effective_Date": effective_date,
            "Effective_Date_Source": date_source,
            "Exit_Rows_Parsed": section_exit_count,
            "Entry_Rows_Parsed": section_entry_count,
            "Action_Headings_Found": len(actions),
            "Section_Status": (
                "DATE_UNRESOLVED" if effective_date is None
                else "CANDIDATE_REQUIRES_QA"
            )
        })

events = pd.DataFrame(event_rows)
summary = pd.DataFrame(summary_rows)

events.to_csv(EVENTS_PATH, index=False)
summary.to_csv(SUMMARY_PATH, index=False)

print("=" * 75)
print("OFFICIAL NIFTY 500 EVENT EXTRACTION — CANDIDATES ONLY")
print("=" * 75)
print("Nifty 500 sections identified:", len(summary))
print("Action headings found:", int(summary["Action_Headings_Found"].sum()))
print("Candidate event rows parsed:", len(events))
print("Rows with unresolved effective dates:", (
    int(events["Effective_Date"].isna().sum()) if not events.empty else 0
))
print("Releases with missing text:", len(release_failures))

print("\n2016 SANITY CHECKS")
sanity = summary[
    summary["Published_Date"].astype(str).str.contains("2016", na=False)
    | summary["Title"].astype(str).str.contains("2016", na=False)
].copy()
display(
    sanity[[
        "Published_Date", "Title", "Effective_Date",
        "Effective_Date_Source", "Exit_Rows_Parsed",
        "Entry_Rows_Parsed", "Section_Status"
    ]].sort_values(["Published_Date", "Title"])
)

print("\nEVENT COUNTS BY ROLE")
if not events.empty:
    display(events.groupby("Role").size().reset_index(name="Rows"))

print("\nSaved candidate ledger:", EVENTS_PATH)
print("Saved extraction summary:", SUMMARY_PATH)
print("Nothing in the source membership or price data was modified.")

OFFICIAL NIFTY 500 EVENT EXTRACTION — CANDIDATES ONLY
Nifty 500 sections identified: 20
Action headings found: 225
Candidate event rows parsed: 1397
Rows with unresolved effective dates: 447
Releases with missing text: 0

2016 SANITY CHECKS


,Published_Date,Title,Effective_Date,Effective_Date_Source,Exit_Rows_Parsed,Entry_Rows_Parsed,Section_Status
3,"Aug 12, 2016","Change in Indices w.e.f. September 30, 2016.",2016-09-30,title_single_date,25,25,CANDIDATE_REQUIRES_QA
2,"Feb 22, 2016","Change in Indices w.e.f. April 01, 2016.",2016-04-01,title_single_date,76,76,CANDIDATE_REQUIRES_QA
1,"Jan 11, 2016","Change in Indices w.e.f January 18, 2016",2016-01-18,title_single_date,1,1,CANDIDATE_REQUIRES_QA
4,"Oct 17, 2016","Change in Indices w.e.f. October 24, 2016 and ...",2016-11-15,intro_clause_near_nifty500,1,1,CANDIDATE_REQUIRES_QA



EVENT COUNTS BY ROLE


,Role,Rows
0,ENTRY,701
1,EXIT,696



Saved candidate ledger: /content/nifty500_pit_data/nifty500_official_event_candidates_v1.csv
Saved extraction summary: /content/nifty500_pit_data/nifty500_event_extraction_summary_v1.csv
Nothing in the source membership or price data was modified.


In [31]:
from pathlib import Path
import pandas as pd

BASE = Path("/content/nifty500_pit_data")

EVENTS_PATH = BASE / "nifty500_official_event_candidates_v1.csv"
OFFICIAL_PATH = BASE / "official_review_crosscheck_2016-04-01.csv"
REPORT_PATH = BASE / "february_2016_event_extraction_reconciliation_v1.csv"

assert EVENTS_PATH.exists(), f"Missing: {EVENTS_PATH}"
assert OFFICIAL_PATH.exists(), f"Missing: {OFFICIAL_PATH}"

events = pd.read_csv(EVENTS_PATH, dtype={"Symbol": str})
official = pd.read_csv(OFFICIAL_PATH, dtype={"Official_Symbol": str})

required_events = {
    "Release_Title", "Role", "Symbol", "Company_Name",
    "Effective_Date", "Event_Extraction_Status"
}
required_official = {"Role", "Official_Symbol"}

assert required_events.issubset(events.columns), (
    f"Unexpected event columns: {events.columns.tolist()}"
)
assert required_official.issubset(official.columns), (
    f"Unexpected official roster columns: {official.columns.tolist()}"
)

# Isolate the official February review with the April 1 effective date.
mask = events["Release_Title"].astype(str).str.contains(
    r"April\s+01,\s*2016", case=False, regex=True, na=False
)
feb = events.loc[mask].copy()

feb["Role"] = feb["Role"].astype(str).str.upper().str.strip()
feb["Symbol_Norm"] = feb["Symbol"].astype(str).str.upper().str.strip()

official["Role"] = official["Role"].astype(str).str.upper().str.strip()
official["Symbol_Norm"] = (
    official["Official_Symbol"].astype(str).str.upper().str.strip()
)

assert not feb.empty, "No February 2016 event rows matched the release title."

print("=" * 72)
print("FEBRUARY 2016 OFFICIAL-LIST RECONCILIATION")
print("=" * 72)
print("Parsed rows:", len(feb))

print("\nPARSED ROW COUNTS")
display(
    feb.groupby("Role").agg(
        Parsed_Rows=("Symbol_Norm", "size"),
        Unique_Symbols=("Symbol_Norm", "nunique")
    ).reset_index()
)

print("\nOFFICIAL ROSTER COUNTS")
display(
    official.groupby("Role").agg(
        Official_Rows=("Symbol_Norm", "size"),
        Official_Unique_Symbols=("Symbol_Norm", "nunique")
    ).reset_index()
)

all_differences = []

for role in ["EXIT", "ENTRY"]:
    parsed_role = feb[feb["Role"].eq(role)].copy()
    official_role = official[official["Role"].eq(role)].copy()

    parsed_symbols = set(parsed_role["Symbol_Norm"].dropna())
    official_symbols = set(official_role["Symbol_Norm"].dropna())

    missing = sorted(official_symbols - parsed_symbols)
    unexpected = sorted(parsed_symbols - official_symbols)

    duplicate_counts = (
        parsed_role.groupby("Symbol_Norm")
        .size()
        .reset_index(name="Parsed_Row_Count")
    )
    duplicates = duplicate_counts[
        duplicate_counts["Parsed_Row_Count"] > 1
    ]

    print(f"\n{'-' * 60}")
    print(f"{role}: {len(parsed_role)} parsed rows")
    print(f"Unique parsed symbols: {len(parsed_symbols)}")
    print(f"Expected official symbols: {len(official_symbols)}")
    print(f"Missing official symbols: {len(missing)}")
    print(f"Unexpected parsed symbols: {len(unexpected)}")
    print(f"Symbols appearing multiple times: {len(duplicates)}")

    print("\nMissing official symbols:")
    print(missing)

    print("\nUnexpected parsed symbols:")
    print(unexpected)

    if not duplicates.empty:
        print("\nDuplicate parsed symbols and counts:")
        display(duplicates)

    if unexpected:
        print("\nRows behind unexpected symbols:")
        display(
            parsed_role[
                parsed_role["Symbol_Norm"].isin(unexpected)
            ][
                ["Role", "Company_Name", "Symbol", "Effective_Date",
                 "Event_Extraction_Status"]
            ].sort_values(["Symbol", "Company_Name"])
        )

    if not duplicates.empty:
        print("\nAll rows for duplicated symbols:")
        display(
            parsed_role[
                parsed_role["Symbol_Norm"].isin(
                    duplicates["Symbol_Norm"]
                )
            ][
                ["Role", "Company_Name", "Symbol", "Effective_Date",
                 "Event_Extraction_Status"]
            ].sort_values(["Symbol", "Company_Name"])
        )

    for symbol in missing:
        all_differences.append({
            "Role": role,
            "Issue": "MISSING_OFFICIAL_SYMBOL",
            "Symbol": symbol
        })
    for symbol in unexpected:
        all_differences.append({
            "Role": role,
            "Issue": "UNEXPECTED_PARSED_SYMBOL",
            "Symbol": symbol
        })
    for _, row in duplicates.iterrows():
        all_differences.append({
            "Role": role,
            "Issue": "DUPLICATE_PARSED_SYMBOL",
            "Symbol": row["Symbol_Norm"]
        })

pd.DataFrame(
    all_differences,
    columns=["Role", "Issue", "Symbol"]
).to_csv(REPORT_PATH, index=False)

print("\nReconciliation report saved:", REPORT_PATH)
print("No source membership intervals, prices, or flags were modified.")

FEBRUARY 2016 OFFICIAL-LIST RECONCILIATION
Parsed rows: 152

PARSED ROW COUNTS


,Role,Parsed_Rows,Unique_Symbols
0,ENTRY,76,76
1,EXIT,76,76



OFFICIAL ROSTER COUNTS


,Role,Official_Rows,Official_Unique_Symbols
0,ENTRY,76,76
1,EXIT,75,75



------------------------------------------------------------
EXIT: 76 parsed rows
Unique parsed symbols: 76
Expected official symbols: 75
Missing official symbols: 1
Unexpected parsed symbols: 2
Symbols appearing multiple times: 0

Missing official symbols:
['ILFSENGG']

Unexpected parsed symbols:
['IL&FSENGG', 'LTD.']

Rows behind unexpected symbols:


,Role,Company_Name,Symbol,Effective_Date,Event_Extraction_Status
33,EXIT,IL&FS Engineering and Construction Company Ltd.,IL&FSENGG,2016-04-01,CANDIDATE_REQUIRES_QA
28,EXIT,Gujarat Narmada Valley Fertilizers and Chemicals,LTD.,2016-04-01,CANDIDATE_REQUIRES_QA



------------------------------------------------------------
ENTRY: 76 parsed rows
Unique parsed symbols: 76
Expected official symbols: 76
Missing official symbols: 0
Unexpected parsed symbols: 0
Symbols appearing multiple times: 0

Missing official symbols:
[]

Unexpected parsed symbols:
[]

Reconciliation report saved: /content/nifty500_pit_data/february_2016_event_extraction_reconciliation_v1.csv
No source membership intervals, prices, or flags were modified.


In [32]:
from pathlib import Path
import re
import pandas as pd

BASE = Path("/content/nifty500_pit_data")

EVENTS_PATH = BASE / "nifty500_official_event_candidates_v1.csv"
OFFICIAL_PATH = BASE / "official_review_crosscheck_2016-04-01.csv"

assert EVENTS_PATH.exists()
assert OFFICIAL_PATH.exists()

events = pd.read_csv(EVENTS_PATH, dtype={"Symbol": str})
official = pd.read_csv(OFFICIAL_PATH, dtype={"Official_Symbol": str})

feb = events[
    events["Release_Title"].astype(str).str.contains(
        "April 01, 2016", case=False, na=False
    )
].copy()

def canonical_symbol(value):
    if pd.isna(value):
        return ""
    # Treat punctuation differences as equivalent, e.g. IL&FSENGG.
    return re.sub(r"[^A-Z0-9]", "", str(value).upper().strip())

feb["Role"] = feb["Role"].astype(str).str.upper().str.strip()
feb["Canonical_Symbol"] = feb["Symbol"].map(canonical_symbol)

official["Role"] = official["Role"].astype(str).str.upper().str.strip()
official["Canonical_Symbol"] = official["Official_Symbol"].map(
    canonical_symbol
)

# Words that can appear as fragments of company names but are not
# valid constituents in this particular official roster.
FALSE_TICKER_TOKENS = {"LTD", "LIMITED"}

results = []

for role in ["EXIT", "ENTRY"]:
    parsed = feb[feb["Role"].eq(role)].copy()
    expected = official[official["Role"].eq(role)].copy()

    parsed_clean = parsed[
        ~parsed["Canonical_Symbol"].isin(FALSE_TICKER_TOKENS)
    ].copy()

    parsed_set = set(parsed_clean["Canonical_Symbol"])
    expected_set = set(expected["Canonical_Symbol"])

    print("\n" + "=" * 60)
    print(role)
    print("=" * 60)
    print("Expected official symbols:", len(expected_set))
    print("Parsed unique symbols, before cleanup:",
          parsed["Canonical_Symbol"].nunique())
    print("Parsed rows removed as obvious company-name fragments:",
          int(parsed["Canonical_Symbol"].isin(FALSE_TICKER_TOKENS).sum()))
    print("Parsed unique symbols after cleanup:", len(parsed_set))
    print("Missing official symbols:", sorted(expected_set - parsed_set))
    print("Unexpected parsed symbols:", sorted(parsed_set - expected_set))

    # Check duplicate symbols after normalization.
    duplicate_rows = parsed_clean[
        parsed_clean["Canonical_Symbol"].duplicated(keep=False)
    ]

    print("Duplicate normalized-symbol rows:", len(duplicate_rows))

    if not duplicate_rows.empty:
        display(duplicate_rows[
            ["Role", "Company_Name", "Symbol", "Canonical_Symbol"]
        ])

    passed = parsed_set == expected_set and duplicate_rows.empty

    results.append({
        "Role": role,
        "Expected_Unique_Symbols": len(expected_set),
        "Parsed_Unique_Symbols_After_Cleanup": len(parsed_set),
        "Missing_Symbols": len(expected_set - parsed_set),
        "Unexpected_Symbols": len(parsed_set - expected_set),
        "Duplicate_Normalized_Rows": len(duplicate_rows),
        "Pass": passed
    })

qa = pd.DataFrame(results)

print("\nFINAL FEBRUARY 2016 CHECK")
display(qa)

assert qa["Pass"].all(), (
    "The February event list did not fully reconcile. "
    "Investigate the reported differences before proceeding."
)

print("PASS: February 2016 event symbols match the official roster.")
print("No source events, intervals, prices or flags were modified.")


EXIT
Expected official symbols: 75
Parsed unique symbols, before cleanup: 76
Parsed rows removed as obvious company-name fragments: 1
Parsed unique symbols after cleanup: 75
Missing official symbols: []
Unexpected parsed symbols: []
Duplicate normalized-symbol rows: 0

ENTRY
Expected official symbols: 76
Parsed unique symbols, before cleanup: 76
Parsed rows removed as obvious company-name fragments: 0
Parsed unique symbols after cleanup: 76
Missing official symbols: []
Unexpected parsed symbols: []
Duplicate normalized-symbol rows: 0

FINAL FEBRUARY 2016 CHECK


,Role,Expected_Unique_Symbols,Parsed_Unique_Symbols_After_Cleanup,Missing_Symbols,Unexpected_Symbols,Duplicate_Normalized_Rows,Pass
0,EXIT,75,75,0,0,0,True
1,ENTRY,76,76,0,0,0,True


PASS: February 2016 event symbols match the official roster.
No source events, intervals, prices or flags were modified.


In [33]:
from pathlib import Path
import re
import pandas as pd

BASE = Path("/content/nifty500_pit_data")
SCAN_PATH = BASE / "nifty500_official_pdf_scan_v1.csv"

EVENTS_PATH = BASE / "nifty500_official_event_candidates_v2.csv"
SUMMARY_PATH = BASE / "nifty500_event_extraction_summary_v2.csv"
REJECTED_PATH = BASE / "nifty500_rejected_symbol_candidates_v2.csv"

assert SCAN_PATH.exists(), f"Missing scan report: {SCAN_PATH}"

scan = pd.read_csv(SCAN_PATH)

MONTH = (
    r"(?:January|February|March|April|May|June|July|August|"
    r"September|October|November|December|Jan|Feb|Mar|Apr|"
    r"Jun|Jul|Aug|Sep|Oct|Nov|Dec)"
)
DATE_TEXT = (
    rf"(?:{MONTH}\s+\d{{1,2}},?\s+\d{{4}}|"
    rf"\d{{1,2}}\s+{MONTH}\s+\d{{4}})"
)
DATE_RE = re.compile(DATE_TEXT, re.I)

# Accept both "1. Nifty 500 Index" and "1) Nifty 500 Index".
def normalized_heading(line):
    return re.sub(
        r"^\s*\d+\s*[.)]\s*", "", str(line)
    ).strip()

def is_nifty500_heading(line):
    return bool(re.fullmatch(
        r"Nifty\s+500\s+Index",
        normalized_heading(line),
        flags=re.I
    ))

def is_any_index_heading(line):
    candidate = normalized_heading(line)
    return bool(re.fullmatch(
        r"(?:Nifty|CNX)\s+.+\s+Index",
        candidate,
        flags=re.I
    ))

ACTION_RE = re.compile(
    r"\bthe\s+following\s+"
    r"(?:companies|scrips|company|scrip)\s+"
    r"(?:(?:are|is)\s+(?:being\s+)?|will\s+be\s+|shall\s+be\s+)"
    r"(excluded|included|removed|added)\b",
    re.I
)

SYMBOL_RE = re.compile(r"^[A-Z0-9][A-Z0-9&.\-/]{1,24}$")

# These are company-name/table fragments, not ticker symbols.
STOP_SYMBOLS = {
    "LTD", "LIMITED", "PVT", "PRIVATE", "COMPANY",
    "COMPANIES", "CO", "SR", "NO", "SYMBOL", "SCRIP",
    "SCRIPS", "NAME", "INDEX", "EXCLUDED", "INCLUDED",
    "REMOVED", "ADDED", "PAGE"
}

def canonical_symbol(value):
    if pd.isna(value):
        return ""
    return re.sub(r"[^A-Z0-9]", "", str(value).strip().upper())

def parse_date(value):
    parsed = pd.to_datetime(value, errors="coerce")
    return None if pd.isna(parsed) else parsed.strftime("%Y-%m-%d")

def get_sections(lines):
    starts = [
        i for i, line in enumerate(lines)
        if is_nifty500_heading(line)
    ]

    sections = []
    for start in starts:
        end = len(lines)
        for j in range(start + 1, len(lines)):
            if is_any_index_heading(lines[j]):
                end = j
                break
        sections.append((start, end))

    return sections

def infer_effective_date(title, lines, section_start):
    title_dates = []
    for match in DATE_RE.finditer(str(title)):
        parsed = parse_date(match.group(0))
        if parsed and parsed not in title_dates:
            title_dates.append(parsed)

    # A single explicit effective date in the title is the clearest case.
    if len(title_dates) == 1:
        return title_dates[0], "title_single_date"

    # Multi-date releases require an explicit effective-date phrase
    # in the introduction near the reference to Nifty 500.
    prefix = "\n".join(lines[:section_start])
    mentions = list(re.finditer(
        r"\b(?:Nifty|CNX)\s*[-_ ]?\s*500\b", prefix, re.I
    ))

    if mentions:
        nearby = prefix[mentions[-1].start():
                        min(len(prefix), mentions[-1].start() + 1800)]

        effective_re = re.compile(
            rf"(?:shall\s+become\s+effective\s+from|"
            rf"become\s+effective\s+from|effective\s+from|"
            rf"with\s+effect\s+from|w\.?\s*e\.?\s*f\.?)"
            rf"\s*[:\-]?\s*({DATE_TEXT})",
            re.I
        )

        matches = list(effective_re.finditer(nearby))
        if matches:
            parsed = parse_date(matches[0].group(1))
            if parsed:
                return parsed, "intro_clause_near_nifty500"

    return None, "unresolved_multiple_or_missing_dates"

def extract_symbol_rows(block_text, release_meta):
    raw_lines = [line.strip() for line in block_text.splitlines()]
    lines = [(i, line) for i, line in enumerate(raw_lines) if line]

    records = []
    rejected = []
    seen = set()

    for pos, (raw_idx, line) in enumerate(lines):
        symbol = line.strip().upper()

        if not SYMBOL_RE.fullmatch(symbol):
            continue

        canonical = canonical_symbol(symbol)

        if (
            canonical in STOP_SYMBOLS
            or not re.search(r"[A-Z]", canonical)
            or pos == 0
        ):
            if canonical in STOP_SYMBOLS:
                rejected.append({
                    **release_meta,
                    "Raw_Symbol": symbol,
                    "Canonical_Symbol": canonical,
                    "Previous_Line": lines[pos - 1][1] if pos else "",
                    "Reason": "Known table/company-name fragment"
                })
            continue

        company = lines[pos - 1][1].strip()

        # Some PDF tables split the company suffix onto its own line:
        # "Gujarat Narmada Valley Fertilizers and Chemicals" / "LTD." / "GNFC"
        if canonical_symbol(company) in {"LTD", "LIMITED"} and pos >= 2:
            company = lines[pos - 2][1].strip() + " " + company

        if (
            not company
            or re.fullmatch(r"[\d\s.)]+", company)
            or canonical_symbol(company) in STOP_SYMBOLS
        ):
            continue

        serial = None
        if pos >= 2:
            previous = lines[pos - 2][1].strip()
            if re.fullmatch(r"\d{1,3}", previous):
                serial = int(previous)

        key = (serial, company, canonical)
        if key in seen:
            continue
        seen.add(key)

        records.append({
            "Serial_No": serial,
            "Company_Name": company,
            "Raw_Symbol": symbol,
            "Canonical_Symbol": canonical
        })

    return records, rejected

event_rows = []
summary_rows = []
rejected_rows = []

for _, release in scan.iterrows():
    title = str(release.get("Title", ""))
    published = release.get("Published_Date")
    url = str(release.get("URL", ""))
    text_path = str(release.get("Text_Cache", "")).strip()

    if not text_path or not Path(text_path).exists():
        continue

    full_text = Path(text_path).read_text(
        encoding="utf-8", errors="replace"
    )
    full_text = re.sub(r"\n--- PAGE \d+ ---\n", "\n", full_text)
    lines = [line.strip() for line in full_text.splitlines()]

    for section_no, (start, end) in enumerate(get_sections(lines), 1):
        section_text = "\n".join(lines[start + 1:end])
        effective_date, date_source = infer_effective_date(
            title, lines, start
        )
        actions = list(ACTION_RE.finditer(section_text))

        section_exits = 0
        section_entries = 0

        for action_i, action in enumerate(actions):
            block_start = action.end()
            block_end = (
                actions[action_i + 1].start()
                if action_i + 1 < len(actions)
                else len(section_text)
            )
            block_text = section_text[block_start:block_end]

            action_word = action.group(1).lower()
            role = (
                "EXIT" if action_word in {"excluded", "removed"}
                else "ENTRY"
            )

            meta = {
                "Published_Date": published,
                "Release_Title": title,
                "Source_URL": url,
                "Section_Number": section_no
            }

            parsed_rows, rejected = extract_symbol_rows(block_text, meta)
            rejected_rows.extend(rejected)

            if role == "EXIT":
                section_exits += len(parsed_rows)
            else:
                section_entries += len(parsed_rows)

            for row in parsed_rows:
                event_rows.append({
                    **meta,
                    "Effective_Date": effective_date,
                    "Effective_Date_Source": date_source,
                    "Role": role,
                    "Company_Name": row["Company_Name"],
                    "Raw_Symbol": row["Raw_Symbol"],
                    "Canonical_Symbol": row["Canonical_Symbol"],
                    "Serial_No": row["Serial_No"],
                    "Action_Heading": action.group(0),
                    "Event_Status": (
                        "DATE_UNRESOLVED" if effective_date is None
                        else "CANDIDATE_REQUIRES_QA"
                    )
                })

        summary_rows.append({
            "Published_Date": published,
            "Title": title,
            "URL": url,
            "Section_Number": section_no,
            "Effective_Date": effective_date,
            "Effective_Date_Source": date_source,
            "Action_Headings": len(actions),
            "Exit_Rows_Parsed": section_exits,
            "Entry_Rows_Parsed": section_entries,
            "Rows_Parsed": section_exits + section_entries,
            "Section_Status": (
                "NO_ACTION_HEADINGS" if not actions
                else "DATE_UNRESOLVED" if effective_date is None
                else "CANDIDATE_REQUIRES_QA"
            )
        })

events = pd.DataFrame(event_rows)
summary = pd.DataFrame(summary_rows)
rejected = pd.DataFrame(rejected_rows)

events.to_csv(EVENTS_PATH, index=False)
summary.to_csv(SUMMARY_PATH, index=False)
rejected.to_csv(REJECTED_PATH, index=False)

print("=" * 72)
print("NIFTY 500 EVENT EXTRACTOR V2 — CANDIDATES ONLY")
print("=" * 72)
print("Nifty 500 sections identified:", len(summary))
print("Action headings found:", int(summary["Action_Headings"].sum()))
print("Candidate event rows:", len(events))
print("Rows with unresolved effective dates:",
      int(events["Effective_Date"].isna().sum()) if not events.empty else 0)
print("Rejected ticker fragments:", len(rejected))

# Review all 2016 sections, including the previously missed April release.
summary_2016 = summary[
    pd.to_datetime(summary["Published_Date"], errors="coerce")
    .dt.year.eq(2016)
].copy()

print("\n2016 SANITY CHECK")
display(
    summary_2016[[
        "Published_Date", "Title", "Effective_Date",
        "Effective_Date_Source", "Exit_Rows_Parsed",
        "Entry_Rows_Parsed", "Section_Status"
    ]].sort_values(["Published_Date", "Title"])
)

# Exact February reconciliation.
feb = events[
    events["Release_Title"].astype(str).str.contains(
        "April 01, 2016", case=False, na=False
    )
].copy()

if not feb.empty:
    print("\nFEBRUARY REVIEW COUNTS")
    display(
        feb.groupby("Role").agg(
            Parsed_Rows=("Canonical_Symbol", "size"),
            Unique_Symbols=("Canonical_Symbol", "nunique")
        ).reset_index()
    )

print("\nSaved event candidates:", EVENTS_PATH)
print("Saved section summary:", SUMMARY_PATH)
print("Saved rejected fragments:", REJECTED_PATH)
print("No membership intervals, price rows, or flags were modified.")

NIFTY 500 EVENT EXTRACTOR V2 — CANDIDATES ONLY
Nifty 500 sections identified: 21
Action headings found: 227
Candidate event rows: 1401
Rows with unresolved effective dates: 447
Rejected ticker fragments: 530

2016 SANITY CHECK


,Published_Date,Title,Effective_Date,Effective_Date_Source,Exit_Rows_Parsed,Entry_Rows_Parsed,Section_Status
3,"Apr 22, 2016","Change in Indices w.e.f. April 29, 2016.",2016-04-29,title_single_date,3,3,CANDIDATE_REQUIRES_QA
4,"Aug 12, 2016","Change in Indices w.e.f. September 30, 2016.",2016-09-30,title_single_date,25,25,CANDIDATE_REQUIRES_QA
2,"Feb 22, 2016","Change in Indices w.e.f. April 01, 2016.",2016-04-01,title_single_date,75,76,CANDIDATE_REQUIRES_QA
1,"Jan 11, 2016","Change in Indices w.e.f January 18, 2016",2016-01-18,title_single_date,1,1,CANDIDATE_REQUIRES_QA
5,"Oct 17, 2016","Change in Indices w.e.f. October 24, 2016 and ...",2016-11-15,intro_clause_near_nifty500,1,1,CANDIDATE_REQUIRES_QA



FEBRUARY REVIEW COUNTS


,Role,Parsed_Rows,Unique_Symbols
0,ENTRY,76,76
1,EXIT,75,75



Saved event candidates: /content/nifty500_pit_data/nifty500_official_event_candidates_v2.csv
Saved section summary: /content/nifty500_pit_data/nifty500_event_extraction_summary_v2.csv
Saved rejected fragments: /content/nifty500_pit_data/nifty500_rejected_symbol_candidates_v2.csv
No membership intervals, price rows, or flags were modified.


In [34]:
from pathlib import Path
import re
import pandas as pd

BASE = Path("/content/nifty500_pit_data")

SUMMARY_PATH = BASE / "nifty500_event_extraction_summary_v2.csv"
EVENTS_PATH = BASE / "nifty500_official_event_candidates_v2.csv"
SCAN_PATH = BASE / "nifty500_official_pdf_scan_v1.csv"

OUT_PATH = BASE / "nifty500_unresolved_effective_dates_review_v1.csv"

for path in [SUMMARY_PATH, EVENTS_PATH, SCAN_PATH]:
    assert path.exists(), f"Missing file: {path}"

summary = pd.read_csv(SUMMARY_PATH)
events = pd.read_csv(EVENTS_PATH)
scan = pd.read_csv(SCAN_PATH)

summary["Effective_Date"] = pd.to_datetime(
    summary["Effective_Date"], errors="coerce"
)

unresolved = summary[
    summary["Effective_Date"].isna()
    & summary["Rows_Parsed"].fillna(0).gt(0)
].copy()

print("=" * 75)
print("UNRESOLVED EFFECTIVE-DATE AUDIT")
print("=" * 75)
print("Unresolved event rows from section summary:",
      int(unresolved["Rows_Parsed"].sum()))
print("Sections with unresolved dates:", len(unresolved))
print("Distinct releases affected:", unresolved["URL"].nunique())

if unresolved.empty:
    print("No unresolved sections with parsed event rows.")
else:
    rollup = (
        unresolved.groupby(
            ["Published_Date", "Title", "URL"],
            dropna=False
        )
        .agg(
            Unresolved_Sections=("Section_Number", "count"),
            Candidate_Event_Rows=("Rows_Parsed", "sum"),
            Exit_Rows=("Exit_Rows_Parsed", "sum"),
            Entry_Rows=("Entry_Rows_Parsed", "sum")
        )
        .reset_index()
        .sort_values(["Published_Date", "Title"])
    )

    print("\nAFFECTED RELEASES")
    display(rollup)

    # Map exact release URLs to their cached extracted text.
    text_map = (
        scan.dropna(subset=["URL"])
        .drop_duplicates("URL")
        .set_index("URL")["Text_Cache"]
        .to_dict()
    )

    heading_rx = re.compile(
        r"^\s*(?:\d+\s*[.)]\s*)?Nifty\s+500\s+Index\s*$",
        re.IGNORECASE
    )
    cue_rx = re.compile(
        r"effective|w\.?\s*e\.?\s*f\.?|close of|"
        r"shall become|will become|scheme of arrangement|"
        r"following changes|review|"
        r"\b20(?:1[5-9]|2[0-6])\b",
        re.IGNORECASE
    )
    date_rx = re.compile(
        r"\b(?:January|February|March|April|May|June|July|August|"
        r"September|October|November|December|Jan|Feb|Mar|Apr|"
        r"Jun|Jul|Aug|Sep|Oct|Nov|Dec)\s+"
        r"\d{1,2},?\s+\d{4}\b"
        r"|\b\d{1,2}\s+"
        r"(?:January|February|March|April|May|June|July|August|"
        r"September|October|November|December|Jan|Feb|Mar|Apr|"
        r"Jun|Jul|Aug|Sep|Oct|Nov|Dec)\s+\d{4}\b",
        re.IGNORECASE
    )

    context_rows = []

    for _, item in unresolved.iterrows():
        url = item["URL"]
        text_path = text_map.get(url)

        if not text_path or not Path(str(text_path)).exists():
            context_rows.append({
                "Published_Date": item["Published_Date"],
                "Title": item["Title"],
                "URL": url,
                "Section_Number": item["Section_Number"],
                "Candidate_Event_Rows": item["Rows_Parsed"],
                "Date_Evidence": "TEXT CACHE NOT FOUND"
            })
            continue

        text = Path(str(text_path)).read_text(
            encoding="utf-8", errors="replace"
        )
        text = re.sub(r"\n--- PAGE \d+ ---\n", "\n", text)
        lines = [line.strip() for line in text.splitlines()]

        heading_positions = [
            i for i, line in enumerate(lines)
            if heading_rx.fullmatch(line)
        ]

        section_number = int(item["Section_Number"])

        if len(heading_positions) < section_number:
            evidence = "NIFTY 500 HEADING NOT FOUND AT EXPECTED POSITION"
        else:
            heading_index = heading_positions[section_number - 1]

            # Inspect the previous 100 lines for a shared review-date statement.
            intro_start = max(0, heading_index - 100)
            intro_lines = lines[intro_start:heading_index]

            matched_lines = []
            for j, line in enumerate(intro_lines, start=intro_start):
                if cue_rx.search(line) or date_rx.search(line):
                    matched_lines.append(f"L{j + 1}: {line}")

            if matched_lines:
                evidence = "\n".join(matched_lines[-18:])
            else:
                evidence = "\n".join(
                    f"L{j + 1}: {line}"
                    for j, line in enumerate(
                        intro_lines[-15:], start=max(0, heading_index - 15)
                    )
                )

        context_rows.append({
            "Published_Date": item["Published_Date"],
            "Title": item["Title"],
            "URL": url,
            "Section_Number": section_number,
            "Candidate_Event_Rows": item["Rows_Parsed"],
            "Date_Evidence": evidence
        })

    context_df = pd.DataFrame(context_rows)
    context_df.to_csv(OUT_PATH, index=False)

    print("\nINTRODUCTION DATE EVIDENCE")
    display(
        context_df[
            [
                "Published_Date", "Title", "Section_Number",
                "Candidate_Event_Rows", "Date_Evidence"
            ]
        ]
    )

    print("\nSaved review file:", OUT_PATH)

print("\nRead-only audit complete. No membership intervals, prices, or flags changed.")

UNRESOLVED EFFECTIVE-DATE AUDIT
Unresolved event rows from section summary: 447
Sections with unresolved dates: 5
Distinct releases affected: 5

AFFECTED RELEASES


,Published_Date,Title,URL,Unresolved_Sections,Candidate_Event_Rows,Exit_Rows,Entry_Rows
0,"Apr 27, 2017",Replacements in indices and change in NIFTY100...,https://www.niftyindices.com/Press_Release/ind...,1,46,23,23
1,"Aug 01, 2018",Replacement in Indices and revision in treatme...,https://www.niftyindices.com/Press_Release/ind...,1,2,1,1
2,"Aug 20, 2020",Revision in criteria and replacements in Indices,https://www.niftyindices.com/Press_Release/ind...,1,349,174,175
3,"Aug 29, 2017",Replacement in Indices.,https://www.niftyindices.com/Press_Release/ind...,1,19,9,10
4,"Mar 07, 2017",Replacements in Indices.,https://www.niftyindices.com/Press_Release/ind...,1,31,15,16



INTRODUCTION DATE EVIDENCE


,Published_Date,Title,Section_Number,Candidate_Event_Rows,Date_Evidence
0,"Mar 07, 2017",Replacements in Indices.,1,31,"L7: Date: 7 March 2017\nL14: On February 16, 2..."
1,"Apr 27, 2017",Replacements in indices and change in NIFTY100...,1,46,L12: Date: 27 April 2017\nL22: replacement of ...
2,"Aug 29, 2017",Replacement in Indices.,1,19,"L11: Date: August 29, 2017\nL17: On August 28,..."
3,"Aug 01, 2018",Replacement in Indices and revision in treatme...,1,2,"L11: Date: August 01, 2018\nL18: Engineering C..."
4,"Aug 20, 2020",Revision in criteria and replacements in Indices,1,349,L245: remaining\nL246: eligible\nL247: replace...



Saved review file: /content/nifty500_pit_data/nifty500_unresolved_effective_dates_review_v1.csv

Read-only audit complete. No membership intervals, prices, or flags changed.


In [35]:
from pathlib import Path
import re
import pandas as pd

BASE = Path("/content/nifty500_pit_data")
SCAN_PATH = BASE / "nifty500_official_pdf_scan_v1.csv"

OUT_EVENTS = BASE / "nifty500_five_release_events_v3.csv"
OUT_SUMMARY = BASE / "nifty500_five_release_QA_v3.csv"

assert SCAN_PATH.exists(), f"Missing scan report: {SCAN_PATH}"

scan = pd.read_csv(SCAN_PATH)
scan["_pub_date"] = pd.to_datetime(
    scan["Published_Date"], errors="coerce"
).dt.strftime("%Y-%m-%d")

# Effective dates verified against the individual official releases.
# These are review-specific overrides, not general date inference rules.
overrides = {
    "2017-03-07": {
        "effective": "2017-03-16",
        "expected_exit": 3, "expected_entry": 3,
        "note": "Three replacements rescheduled from March 31."
    },
    "2017-04-27": {
        "effective": "2017-05-26",
        "expected_exit": 3, "expected_entry": 3,
        "note": "Scheme-of-arrangement replacements."
    },
    "2017-08-29": {
        "effective": "2017-09-05",
        "expected_exit": 1, "expected_entry": 1,
        "note": "Reliance Capital replacement rescheduled."
    },
    "2018-08-01": {
        "effective": "2018-08-08",
        "expected_exit": 1, "expected_entry": 1,
        "note": "Techno Electric replaced by Bharat Dynamics."
    },
    "2020-08-20": {
        "effective": "2020-09-25",
        "expected_exit": 17, "expected_entry": 17,
        "note": "Semi-annual review; excludes criteria revisions."
    }
}

targets = scan[scan["_pub_date"].isin(overrides)].copy()

assert len(targets) == 5, (
    f"Expected five target releases, found {len(targets)}. "
    "Inspect publication dates and titles before proceeding."
)

def strip_heading_prefix(line):
    return re.sub(
        r"^\s*(?:\d{1,2}|[A-Za-z])\s*[.)]\s*",
        "", str(line)
    ).strip()

def is_nifty500_heading(line):
    body = strip_heading_prefix(line)
    return bool(re.fullmatch(
        r"(?:NIFTY|CNX)\s+500(?:\s+INDEX)?",
        body,
        flags=re.IGNORECASE
    ))

def is_other_index_heading(line):
    # Important: many PDFs label subsequent sections as
    # "d) NIFTY 100" or "a) NIFTY 50", without the word "Index".
    return bool(re.match(
        r"^\s*(?:\d{1,2}|[A-Za-z])\s*[.)]\s*(?:NIFTY|CNX)\b",
        str(line),
        flags=re.IGNORECASE
    ))

ACTION_RE = re.compile(
    r"\bthe\s+following\s+"
    r"(?:companies|scrips|company|scrip)\s+"
    r"(?:(?:are|is)\s+(?:being\s+)?|will\s+be\s+|shall\s+be\s+)"
    r"(excluded|included|removed|added)\b",
    re.IGNORECASE
)

SYMBOL_RE = re.compile(r"^[A-Z0-9][A-Z0-9&.\-/]{1,24}$")

STOP_SYMBOLS = {
    "LTD", "LIMITED", "PVT", "PRIVATE", "COMPANY",
    "COMPANIES", "CO", "SR", "NO", "SYMBOL",
    "SCRIP", "SCRIPS", "NAME", "INDEX",
    "EXCLUDED", "INCLUDED", "REMOVED", "ADDED", "PAGE"
}

def canon(value):
    return re.sub(r"[^A-Z0-9]", "", str(value).strip().upper())

def extract_rows(block):
    raw = [line.strip() for line in block.splitlines()]
    lines = [(i, line) for i, line in enumerate(raw) if line]
    records = []
    seen = set()

    for pos, (line_no, line) in enumerate(lines):
        raw_symbol = line.strip().upper()
        symbol = canon(raw_symbol)

        if (
            not SYMBOL_RE.fullmatch(raw_symbol)
            or symbol in STOP_SYMBOLS
            or not re.search(r"[A-Z]", symbol)
            or pos == 0
        ):
            continue

        company = lines[pos - 1][1].strip()

        # Join a split company suffix such as "Company Name" + "LTD."
        if canon(company) in {"LTD", "LIMITED"} and pos >= 2:
            company = lines[pos - 2][1].strip() + " " + company

        if not company or re.fullmatch(r"[\d\s.)]+", company):
            continue

        serial = None
        if pos >= 2 and re.fullmatch(r"\d{1,3}", lines[pos - 2][1]):
            serial = int(lines[pos - 2][1])

        key = (serial, company, symbol)
        if key in seen:
            continue
        seen.add(key)

        records.append({
            "Serial_No": serial,
            "Company_Name": company,
            "Raw_Symbol": raw_symbol,
            "Canonical_Symbol": symbol
        })

    return records

def find_sections(lines):
    starts = [
        i for i, line in enumerate(lines)
        if is_nifty500_heading(line)
    ]

    sections = []
    for start in starts:
        end = len(lines)

        for j in range(start + 1, len(lines)):
            if is_other_index_heading(lines[j]):
                end = j
                break

        sections.append((start, end))

    return sections

event_rows = []
summary_rows = []

for _, doc in targets.iterrows():
    pub_date = doc["_pub_date"]
    title = str(doc["Title"])
    url = str(doc["URL"])
    text_path = str(doc.get("Text_Cache", "")).strip()
    cfg = overrides[pub_date]

    assert text_path and Path(text_path).exists(), (
        f"Cached text missing for {title}"
    )

    text = Path(text_path).read_text(
        encoding="utf-8", errors="replace"
    )
    text = re.sub(r"\n--- PAGE \d+ ---\n", "\n", text)
    lines = [line.strip() for line in text.splitlines()]

    sections = find_sections(lines)
    section_exit_count = 0
    section_entry_count = 0

    for section_no, (start, end) in enumerate(sections, start=1):
        block = "\n".join(lines[start + 1:end])
        actions = list(ACTION_RE.finditer(block))
        parsed_exits = 0
        parsed_entries = 0

        for action_i, action in enumerate(actions):
            block_start = action.end()
            block_end = (
                actions[action_i + 1].start()
                if action_i + 1 < len(actions)
                else len(block)
            )

            rows = extract_rows(block[block_start:block_end])
            word = action.group(1).lower()
            role = (
                "EXIT" if word in {"excluded", "removed"}
                else "ENTRY"
            )

            if role == "EXIT":
                parsed_exits += len(rows)
            else:
                parsed_entries += len(rows)

            for row in rows:
                event_rows.append({
                    "Published_Date": pub_date,
                    "Release_Title": title,
                    "Source_URL": url,
                    "Effective_Date": cfg["effective"],
                    "Effective_Date_Source":
                        "official_release_verified_manual_override",
                    "Review_Note": cfg["note"],
                    "Section_Number": section_no,
                    "Role": role,
                    **row,
                    "Status": "CANDIDATE_REQUIRES_QA"
                })

        section_exit_count += parsed_exits
        section_entry_count += parsed_entries

    passed = (
        section_exit_count == cfg["expected_exit"]
        and section_entry_count == cfg["expected_entry"]
    )

    summary_rows.append({
        "Published_Date": pub_date,
        "Title": title,
        "Effective_Date": cfg["effective"],
        "Nifty500_Sections_Found": len(sections),
        "Parsed_Exits": section_exit_count,
        "Expected_Exits": cfg["expected_exit"],
        "Parsed_Entries": section_entry_count,
        "Expected_Entries": cfg["expected_entry"],
        "Count_Check": "PASS" if passed else "REVIEW",
        "Source_URL": url
    })

events = pd.DataFrame(event_rows)
summary = pd.DataFrame(summary_rows)

events.to_csv(OUT_EVENTS, index=False)
summary.to_csv(OUT_SUMMARY, index=False)

print("=" * 72)
print("FIVE-RELEASE RE-EXTRACTION — CANDIDATES ONLY")
print("=" * 72)
display(
    summary[[
        "Published_Date", "Title", "Effective_Date",
        "Nifty500_Sections_Found",
        "Parsed_Exits", "Expected_Exits",
        "Parsed_Entries", "Expected_Entries",
        "Count_Check"
    ]]
)

print("\nTotal candidate rows extracted:", len(events))
print("Rows for the 2020 release:",
      int(events["Published_Date"].eq("2020-08-20").sum()))
print("Saved events:", OUT_EVENTS)
print("Saved QA summary:", OUT_SUMMARY)
print("No membership intervals, prices, or flags were modified.")

FIVE-RELEASE RE-EXTRACTION — CANDIDATES ONLY


,Published_Date,Title,Effective_Date,Nifty500_Sections_Found,Parsed_Exits,Expected_Exits,Parsed_Entries,Expected_Entries,Count_Check
0,2017-03-07,Replacements in Indices.,2017-03-16,1,15,3,16,3,REVIEW
1,2017-04-27,Replacements in indices and change in NIFTY100...,2017-05-26,1,32,3,23,3,REVIEW
2,2017-08-29,Replacement in Indices.,2017-09-05,1,9,1,10,1,REVIEW
3,2018-08-01,Replacement in Indices and revision in treatme...,2018-08-08,1,3,1,6,1,REVIEW
4,2020-08-20,Revision in criteria and replacements in Indices,2020-09-25,5,341,17,343,17,REVIEW



Total candidate rows extracted: 798
Rows for the 2020 release: 684
Saved events: /content/nifty500_pit_data/nifty500_five_release_events_v3.csv
Saved QA summary: /content/nifty500_pit_data/nifty500_five_release_QA_v3.csv
No membership intervals, prices, or flags were modified.


In [36]:
from pathlib import Path
import re
import pandas as pd

BASE = Path("/content/nifty500_pit_data")
SCAN_PATH = BASE / "nifty500_official_pdf_scan_v1.csv"

scan = pd.read_csv(SCAN_PATH)
match = scan[
    pd.to_datetime(scan["Published_Date"], errors="coerce")
      .dt.strftime("%Y-%m-%d").eq("2020-08-20")
    & scan["Title"].astype(str).str.contains(
        "Revision in criteria and replacements", case=False, na=False
    )
]

assert len(match) == 1, f"Expected one release row; found {len(match)}"

doc = match.iloc[0]
text_path = Path(str(doc["Text_Cache"]))
assert text_path.exists(), f"Missing cached text: {text_path}"

text = text_path.read_text(encoding="utf-8", errors="replace")
parts = re.split(r"\n--- PAGE (\d+) ---\n", text)

pages = []
for i in range(1, len(parts), 2):
    page_no = int(parts[i])
    page_text = parts[i + 1] if i + 1 < len(parts) else ""
    for line_no, line in enumerate(page_text.splitlines(), start=1):
        pages.append({
            "page": page_no,
            "line": line_no,
            "text": line.strip()
        })

def strip_prefix(line):
    return re.sub(
        r"^\s*(?:\d{1,2}|[A-Za-z])\s*[.)]\s*",
        "", line
    ).strip()

def looks_like_index_heading(line):
    s = strip_prefix(line)
    enumerated = bool(re.match(
        r"^\s*(?:\d{1,2}|[A-Za-z])\s*[.)]\s*(?:NIFTY|CNX)\b",
        line, re.I
    ))
    direct_heading = bool(re.fullmatch(
        r"(?:NIFTY|CNX)\b.{0,100}",
        s, re.I
    )) and len(s.split()) <= 9
    return enumerated or direct_heading

def is_nifty500_heading(line):
    return bool(re.fullmatch(
        r"(?:NIFTY|CNX)\s*[- ]?\s*500(?:\s+INDEX)?",
        strip_prefix(line), re.I
    ))

action_re = re.compile(
    r"\bthe\s+following\s+"
    r"(?:companies|scrips|company|scrip)\s+"
    r"(?:(?:are|is)\s+(?:being\s+)?|will\s+be\s+|shall\s+be\s+)"
    r"(excluded|included|removed|added)\b",
    re.I
)

print("=" * 80)
print("2020 OFFICIAL RELEASE: HEADING AND BOUNDARY DIAGNOSTIC")
print("=" * 80)
print("Title:", doc["Title"])
print("Cached text:", text_path)
print("Total extracted lines:", len(pages))

# 1. Print candidate index headings in document order.
headings = []
for i, row in enumerate(pages):
    if looks_like_index_heading(row["text"]):
        headings.append({
            "position": i,
            "page": row["page"],
            "line": row["line"],
            "text": row["text"],
            "is_nifty500": is_nifty500_heading(row["text"])
        })

print("\n1. CANDIDATE INDEX HEADINGS")
display(pd.DataFrame([
    {k: v for k, v in h.items() if k != "position"}
    for h in headings
]))

# 2. Show surrounding text for every Nifty 500 heading.
print("\n2. CONTEXT AROUND EACH NIFTY 500 HEADING")
all_500 = [
    i for i, row in enumerate(pages)
    if is_nifty500_heading(row["text"])
]
print("Exact Nifty 500 heading matches:", len(all_500))

for n, idx in enumerate(all_500, start=1):
    print(f"\n--- NIFTY 500 HEADING {n} ---")
    lo = max(0, idx - 4)
    hi = min(len(pages), idx + 13)
    for j in range(lo, hi):
        r = pages[j]
        print(f"P{r['page']:02d} L{r['line']:04d}: {r['text']}")

# 3. Map each action heading to the nearest prior candidate index heading.
print("\n3. ACTION HEADINGS AND THEIR NEAREST PRIOR INDEX HEADING")
current_heading = "(before first detected heading)"
action_rows = []

for row in pages:
    if looks_like_index_heading(row["text"]):
        current_heading = row["text"]

    m = action_re.search(row["text"])
    if m:
        action_rows.append({
            "Page": row["page"],
            "Line": row["line"],
            "Nearest_Prior_Index_Heading": current_heading,
            "Action": m.group(1).upper(),
            "Text": row["text"]
        })

action_df = pd.DataFrame(action_rows)
print("Action headings found:", len(action_df))
display(action_df)

# 4. For each Nifty 500 start, show the next heading that might terminate it.
print("\n4. BOUNDARY CANDIDATES")
boundary_rows = []

for idx in all_500:
    next_heading = next(
        (
            (j, pages[j]) for j in range(idx + 1, len(pages))
            if looks_like_index_heading(pages[j]["text"])
        ),
        None
    )
    start = pages[idx]

    if next_heading:
        j, end = next_heading
        # Count action headings before that candidate boundary.
        count = sum(
            bool(action_re.search(pages[k]["text"]))
            for k in range(idx + 1, j)
        )
        boundary_rows.append({
            "Start_Page": start["page"],
            "Start_Line": start["line"],
            "Start_Heading": start["text"],
            "Next_Heading_Page": end["page"],
            "Next_Heading_Line": end["line"],
            "Next_Heading_Text": end["text"],
            "Action_Headings_Between": count
        })
    else:
        boundary_rows.append({
            "Start_Page": start["page"],
            "Start_Line": start["line"],
            "Start_Heading": start["text"],
            "Next_Heading_Page": None,
            "Next_Heading_Line": None,
            "Next_Heading_Text": "(end of document)",
            "Action_Headings_Between": sum(
                bool(action_re.search(pages[k]["text"]))
                for k in range(idx + 1, len(pages))
            )
        })

display(pd.DataFrame(boundary_rows))

print("\nRead-only diagnostic complete. No files or datasets were changed.")

2020 OFFICIAL RELEASE: HEADING AND BOUNDARY DIAGNOSTIC
Title: Revision in criteria and replacements in Indices
Cached text: /content/nifty500_pit_data/official_release_text_cache/release_056.txt
Total extracted lines: 1920

1. CANDIDATE INDEX HEADINGS


,page,line,text,is_nifty500
0,1,16,A) NIFTY 500,True
1,1,52,B) NIFTY 100,False
2,2,40,NIFTY 500,True
3,2,47,NIFTY 500,True
4,3,76,C) NIFTY Midcap 150,False
5,3,82,NIFTY 500 index,True
6,4,24,NIFTY 50,False
7,4,53,NIFTY Next 50,False
8,5,28,NIFTY 500,True
9,6,42,NIFTY 100,False



2. CONTEXT AROUND EACH NIFTY 500 HEADING
Exact Nifty 500 heading matches: 5

--- NIFTY 500 HEADING 1 ---
P01 L0012: 
P01 L0013: 1. Revision of criteria for NIFTY 500, NIFTY 100 and NIFTY Midcap 150
P01 L0014: indices
P01 L0015: 
P01 L0016: A) NIFTY 500
P01 L0017: Parameter
P01 L0018: Existing Criteria
P01 L0019: Revised Criteria
P01 L0020: Eligible
P01 L0021: universe
P01 L0022: Companies should rank
P01 L0023: within top 800 based on
P01 L0024: both average daily turnover
P01 L0025: and average daily full
P01 L0026: market capitalization based
P01 L0027: on previous six months
P01 L0028: period data

--- NIFTY 500 HEADING 2 ---
P02 L0036: 
P02 L0037: • Constituents
P02 L0038: excluded
P02 L0039: from
P02 L0040: NIFTY 500
P02 L0041: 
P02 L0042: 
P02 L0043: 
P02 L0044: • Constituents
P02 L0045: excluded
P02 L0046: from
P02 L0047: NIFTY 500
P02 L0048: • Rank based on average full
P02 L0049: market capitalization falls below
P02 L0050: 110 (i.e. beyond 110)
P02 L0051: • In case the numbe

,Page,Line,Nearest_Prior_Index_Heading,Action,Text
0,4,26,NIFTY 50,EXCLUDED,The following companies are being excluded:
1,4,38,NIFTY 50,INCLUDED,The following companies are being included:
2,4,55,NIFTY Next 50,EXCLUDED,The following companies are being excluded:
3,5,6,NIFTY Next 50,INCLUDED,The following companies are being included:
4,5,30,NIFTY 500,EXCLUDED,The following companies are being excluded:
5,5,89,NIFTY 500,INCLUDED,The following companies are being included:
6,6,44,NIFTY 100,EXCLUDED,The following companies are being excluded:
7,6,62,NIFTY 100,INCLUDED,The following companies are being included:
8,6,86,NIFTY Midcap 150,EXCLUDED,The following companies are being excluded:
9,7,39,NIFTY Midcap 150,INCLUDED,The following companies are being included:



4. BOUNDARY CANDIDATES


,Start_Page,Start_Line,Start_Heading,Next_Heading_Page,Next_Heading_Line,Next_Heading_Text,Action_Headings_Between
0,1,16,A) NIFTY 500,1,52,B) NIFTY 100,0
1,2,40,NIFTY 500,2,47,NIFTY 500,0
2,2,47,NIFTY 500,3,76,C) NIFTY Midcap 150,0
3,3,82,NIFTY 500 index,4,24,NIFTY 50,0
4,5,28,NIFTY 500,6,42,NIFTY 100,2



Read-only diagnostic complete. No files or datasets were changed.


In [37]:
from pathlib import Path
import re
import pandas as pd

BASE = Path("/content/nifty500_pit_data")
SCAN_PATH = BASE / "nifty500_official_pdf_scan_v1.csv"

scan = pd.read_csv(SCAN_PATH)
scan["_pub"] = pd.to_datetime(
    scan["Published_Date"], errors="coerce"
).dt.strftime("%Y-%m-%d")

doc_rows = scan[
    scan["_pub"].eq("2020-08-20")
    & scan["Title"].str.contains(
        "Revision in criteria and replacements", case=False, na=False
    )
]
assert len(doc_rows) == 1, f"Expected one PDF, found {len(doc_rows)}"

doc = doc_rows.iloc[0]
text_path = Path(str(doc["Text_Cache"]))
assert text_path.exists()

text = text_path.read_text(encoding="utf-8", errors="replace")

# Preserve page and line metadata for the audit.
parts = re.split(r"\n--- PAGE (\d+) ---\n", text)
lines = []

for i in range(1, len(parts), 2):
    page_no = int(parts[i])
    page_text = parts[i + 1] if i + 1 < len(parts) else ""
    for line_no, raw in enumerate(page_text.splitlines(), 1):
        lines.append({
            "page": page_no,
            "line": line_no,
            "text": raw.strip()
        })

def strip_prefix(s):
    return re.sub(
        r"^\s*(?:\d{1,2}|[A-Za-z])\s*[.)]\s*", "", str(s)
    ).strip()

def is_nifty500_title(s):
    return bool(re.fullmatch(
        r"(?:NIFTY|CNX)\s*[- ]?\s*500(?:\s+INDEX)?",
        strip_prefix(s), re.I
    ))

# A start must be a clearly labelled section title, not a bare
# mention of NIFTY 500 inside a criteria table.
def is_numbered_nifty500_start(s):
    prefix = re.match(
        r"^\s*(?:\d{1,2}|[A-Za-z])\s*[.)]\s*",
        str(s)
    )
    return bool(prefix and is_nifty500_title(s))

def is_other_index_heading(s):
    if is_nifty500_title(s):
        return False

    # Explicitly labelled index heading.
    if re.match(
        r"^\s*(?:\d{1,2}|[A-Za-z])\s*[.)]\s*(?:NIFTY|CNX)\b",
        str(s), re.I
    ):
        return True

    # Some PDF headings are extracted without their preceding label.
    # Only accept a short, standalone index name.
    return bool(re.fullmatch(
        r"(?:NIFTY|CNX)\s+[A-Z0-9 &\-]{1,70}",
        str(s).strip(), re.I
    )) and len(str(s).split()) <= 8

ACTION_RE = re.compile(
    r"\bthe\s+following\s+"
    r"(?:companies|scrips|company|scrip)\s+"
    r"(?:(?:are|is)\s+(?:being\s+)?|will\s+be\s+|shall\s+be\s+)"
    r"(excluded|included|removed|added)\b",
    re.I
)

SYMBOL_RE = re.compile(r"^[A-Z0-9][A-Z0-9&.\-/]{1,24}$")
STOP = {
    "LTD", "LIMITED", "PVT", "PRIVATE", "COMPANY", "COMPANIES",
    "CO", "SR", "NO", "SYMBOL", "SCRIP", "SCRIPS", "NAME",
    "INDEX", "EXCLUDED", "INCLUDED", "REMOVED", "ADDED", "PAGE"
}

def canon(s):
    return re.sub(r"[^A-Z0-9]", "", str(s).strip().upper())

def extract_tickers(block):
    raw = [line.strip() for line in block.splitlines()]
    nonblank = [line for line in raw if line]
    found = []
    seen = set()

    for i, symbol_line in enumerate(nonblank):
        symbol = symbol_line.upper()
        normalized = canon(symbol)

        if (
            not SYMBOL_RE.fullmatch(symbol)
            or normalized in STOP
            or not re.search(r"[A-Z]", normalized)
            or i == 0
        ):
            continue

        company = nonblank[i - 1]

        # Company names sometimes have LTD. on a separate line.
        if canon(company) in {"LTD", "LIMITED"} and i >= 2:
            company = nonblank[i - 2] + " " + company

        if not company or re.fullmatch(r"[\d\s.)]+", company):
            continue

        if normalized not in seen:
            found.append({
                "Company_Name": company,
                "Raw_Symbol": symbol,
                "Canonical_Symbol": normalized
            })
            seen.add(normalized)

    return found

# Find only labelled Nifty 500 sections.
starts = [
    i for i, row in enumerate(lines)
    if is_numbered_nifty500_start(row["text"])
]

section_results = []
event_results = []

for section_no, start in enumerate(starts, 1):
    end = len(lines)

    for j in range(start + 1, len(lines)):
        if is_other_index_heading(lines[j]["text"]):
            end = j
            break

    section_lines = lines[start + 1:end]
    section_text = "\n".join(row["text"] for row in section_lines)
    action_matches = list(ACTION_RE.finditer(section_text))

    exits = []
    entries = []

    for i, action in enumerate(action_matches):
        block_start = action.end()
        block_end = (
            action_matches[i + 1].start()
            if i + 1 < len(action_matches)
            else len(section_text)
        )
        rows = extract_tickers(section_text[block_start:block_end])

        role = (
            "EXIT" if action.group(1).lower() in {"excluded", "removed"}
            else "ENTRY"
        )

        for row in rows:
            event_results.append({
                "Section_Number": section_no,
                "Role": role,
                **row
            })

        if role == "EXIT":
            exits.extend(rows)
        else:
            entries.extend(rows)

    start_row = lines[start]
    end_row = lines[end] if end < len(lines) else None

    section_results.append({
        "Section_Number": section_no,
        "Start_Page": start_row["page"],
        "Start_Line": start_row["line"],
        "Start_Heading": start_row["text"],
        "End_Page": end_row["page"] if end_row else None,
        "End_Line": end_row["line"] if end_row else None,
        "End_Heading": end_row["text"] if end_row else "(end of document)",
        "Action_Headings": len(action_matches),
        "Parsed_Exits": len(exits),
        "Parsed_Entries": len(entries),
        "Count_Check": (
            "PASS" if len(exits) == 17 and len(entries) == 17
            else "REVIEW"
        )
    })

print("=" * 72)
print("STRICT 2020 SECTION TEST — READ ONLY")
print("=" * 72)
print("Labelled Nifty 500 sections found:", len(starts))
display(pd.DataFrame(section_results))

events_df = pd.DataFrame(event_results)

for section_no in range(1, len(starts) + 1):
    section_events = events_df[
        events_df["Section_Number"].eq(section_no)
    ]
    if section_events.empty:
        continue

    print(f"\nSECTION {section_no}: EXTRACTED SYMBOLS")
    display(
        section_events[
            ["Role", "Company_Name", "Raw_Symbol", "Canonical_Symbol"]
        ].sort_values(["Role", "Canonical_Symbol"])
    )

print("\nNo source membership intervals, prices, or flags were modified.")

STRICT 2020 SECTION TEST — READ ONLY
Labelled Nifty 500 sections found: 1


,Section_Number,Start_Page,Start_Line,Start_Heading,End_Page,End_Line,End_Heading,Action_Headings,Parsed_Exits,Parsed_Entries,Count_Check
0,1,1,16,A) NIFTY 500,1,52,B) NIFTY 100,0,0,0,REVIEW


KeyError: 'Section_Number'

In [38]:
from pathlib import Path
import re
import pandas as pd

BASE = Path("/content/nifty500_pit_data")
SCAN_PATH = BASE / "nifty500_official_pdf_scan_v1.csv"

scan = pd.read_csv(SCAN_PATH)
scan["_pub"] = pd.to_datetime(
    scan["Published_Date"], errors="coerce"
).dt.strftime("%Y-%m-%d")

docs = scan[
    scan["_pub"].eq("2020-08-20")
    & scan["Title"].astype(str).str.contains(
        "Revision in criteria and replacements",
        case=False, na=False
    )
]
assert len(docs) == 1, f"Expected one release, found {len(docs)}"

doc = docs.iloc[0]
text_path = Path(str(doc["Text_Cache"]))
assert text_path.exists(), f"Missing cached text: {text_path}"

text = text_path.read_text(encoding="utf-8", errors="replace")
parts = re.split(r"\n--- PAGE (\d+) ---\n", text)

lines = []
for i in range(1, len(parts), 2):
    page_no = int(parts[i])
    page_text = parts[i + 1] if i + 1 < len(parts) else ""
    for line_no, raw in enumerate(page_text.splitlines(), 1):
        lines.append({
            "page": page_no,
            "line": line_no,
            "text": raw.strip()
        })

def is_bare_marker(s):
    return bool(re.fullmatch(r"\s*(?:[A-Za-z]|\d{1,2})\s*[.)]\s*", s))

def is_index_name(s):
    s = str(s).strip()
    if not re.match(r"^(?:NIFTY|CNX)\b", s, re.I):
        return False
    if len(s.split()) > 8:
        return False
    return bool(re.fullmatch(
        r"(?:NIFTY|CNX)\s+[A-Z0-9][A-Z0-9 &'./()%-]*",
        s, re.I
    ))

def find_index_sections(rows):
    sections = []
    i = 0

    while i < len(rows):
        line = rows[i]["text"]

        # Form 1: "A) NIFTY 500"
        same_line = re.match(
            r"^\s*(?:[A-Za-z]|\d{1,2})\s*[.)]\s*((?:NIFTY|CNX)\b.*)$",
            line, re.I
        )

        if same_line and is_index_name(same_line.group(1)):
            sections.append({
                "start_idx": i,
                "title_idx": i,
                "title": same_line.group(1).strip()
            })
            i += 1
            continue

        # Form 2: marker and title occupy separate lines:
        # "c)" followed by "NIFTY 500"
        if (
            is_bare_marker(line)
            and i + 1 < len(rows)
            and is_index_name(rows[i + 1]["text"])
        ):
            sections.append({
                "start_idx": i,
                "title_idx": i + 1,
                "title": rows[i + 1]["text"].strip()
            })
            i += 2
            continue

        i += 1

    return sections

ACTION_RE = re.compile(
    r"\bthe\s+following\s+"
    r"(?:companies|scrips|company|scrip)\s+"
    r"(?:(?:are|is)\s+(?:being\s+)?|will\s+be\s+|shall\s+be\s+)"
    r"(excluded|included|removed|added)\b",
    re.I
)

SYMBOL_RE = re.compile(r"^[A-Z0-9][A-Z0-9&.\-/]{1,24}$")
STOP = {
    "LTD", "LIMITED", "PVT", "PRIVATE", "COMPANY",
    "COMPANIES", "CO", "SR", "NO", "SYMBOL", "SCRIP",
    "SCRIPS", "NAME", "INDEX", "EXCLUDED", "INCLUDED",
    "REMOVED", "ADDED", "PAGE"
}

def canon(s):
    return re.sub(r"[^A-Z0-9]", "", str(s).strip().upper())

def extract_rows(block):
    nonblank = [
        line.strip()
        for line in block.splitlines()
        if line.strip()
    ]
    out, seen = [], set()

    for i, candidate in enumerate(nonblank):
        raw_symbol = candidate.upper()
        symbol = canon(raw_symbol)

        if (
            not SYMBOL_RE.fullmatch(raw_symbol)
            or symbol in STOP
            or not re.search(r"[A-Z]", symbol)
            or i == 0
        ):
            continue

        company = nonblank[i - 1]

        if canon(company) in {"LTD", "LIMITED"} and i >= 2:
            company = nonblank[i - 2] + " " + company

        if not company or re.fullmatch(r"[\d\s.)]+", company):
            continue

        serial = None
        if i >= 2 and re.fullmatch(r"\d{1,3}", nonblank[i - 2]):
            serial = int(nonblank[i - 2])

        key = (serial, company, symbol)
        if key not in seen:
            out.append({
                "Serial_No": serial,
                "Company_Name": company,
                "Symbol": raw_symbol,
                "Canonical_Symbol": symbol
            })
            seen.add(key)

    return out

sections = find_index_sections(lines)
event_rows, report_rows = [], []

for section_no, section in enumerate(sections, start=1):
    start = section["start_idx"]
    title_idx = section["title_idx"]
    end = len(lines)

    # Stop at the next labelled index section.
    if section_no < len(sections):
        end = sections[section_no]["start_idx"]

    # Also stop if an unlabelled index heading follows its own
    # separate marker line.
    for j in range(title_idx + 1, end - 1):
        if (
            is_bare_marker(lines[j]["text"])
            and is_index_name(lines[j + 1]["text"])
        ):
            end = j
            break

    is_target = bool(re.fullmatch(
        r"(?:NIFTY|CNX)\s*[- ]?\s*500(?:\s+INDEX)?",
        section["title"], re.I
    ))

    body_start = title_idx + 1
    body = "\n".join(row["text"] for row in lines[body_start:end])
    actions = list(ACTION_RE.finditer(body))

    exits, entries = [], []

    for k, action in enumerate(actions):
        block_start = action.end()
        block_end = (
            actions[k + 1].start()
            if k + 1 < len(actions) else len(body)
        )

        parsed = extract_rows(body[block_start:block_end])
        role = (
            "EXIT" if action.group(1).lower() in {"excluded", "removed"}
            else "ENTRY"
        )

        if role == "EXIT":
            exits.extend(parsed)
        else:
            entries.extend(parsed)

        if is_target:
            for item in parsed:
                event_rows.append({
                    "Section_Number": section_no,
                    "Section_Title": section["title"],
                    "Role": role,
                    **item
                })

    report_rows.append({
        "Section_Number": section_no,
        "Start_Page": lines[start]["page"],
        "Start_Line": lines[start]["line"],
        "Section_Title": section["title"],
        "Is_Nifty500": is_target,
        "Action_Headings": len(actions),
        "Parsed_Exits": len(exits),
        "Parsed_Entries": len(entries),
        "Expected_Count_Check": (
            "PASS" if is_target and len(exits) == 17
            and len(entries) == 17
            else "NO_ACTIONS" if not actions
            else "REVIEW"
        )
    })

events_df = pd.DataFrame(event_rows, columns=[
    "Section_Number", "Section_Title", "Role",
    "Serial_No", "Company_Name", "Symbol", "Canonical_Symbol"
])
report_df = pd.DataFrame(report_rows)

print("=" * 72)
print("CORRECTED 2020 SECTION TEST — READ ONLY")
print("=" * 72)
display(report_df)

print("\nNIFTY 500 CONSTITUENT CANDIDATES")
if events_df.empty:
    print("No events extracted. Inspect section boundaries before proceeding.")
else:
    display(events_df.sort_values(["Role", "Canonical_Symbol"]))
    print("\nCounts by role:")
    display(events_df.groupby("Role").size().reset_index(name="Rows"))

print("\nNo membership intervals, prices, or flags were modified.")

CORRECTED 2020 SECTION TEST — READ ONLY


,Section_Number,Start_Page,Start_Line,Section_Title,Is_Nifty500,Action_Headings,Parsed_Exits,Parsed_Entries,Expected_Count_Check
0,1,1,16,NIFTY 500,True,0,0,0,NO_ACTIONS
1,2,1,52,NIFTY 100,False,0,0,0,NO_ACTIONS
2,3,3,76,NIFTY Midcap 150,False,0,0,0,NO_ACTIONS
3,4,4,23,NIFTY 50,False,2,2,2,REVIEW
4,5,4,52,NIFTY Next 50,False,2,5,5,REVIEW
5,6,5,27,NIFTY 500,True,2,17,17,PASS
6,7,6,41,NIFTY 100,False,2,4,4,REVIEW
7,8,6,83,NIFTY Midcap 150,False,2,15,15,REVIEW
8,9,7,91,NIFTY Smallcap 250,False,2,22,22,REVIEW
9,10,9,7,NIFTY Midcap 50,False,2,5,5,REVIEW



NIFTY 500 CONSTITUENT CANDIDATES


,Section_Number,Section_Title,Role,Serial_No,Company_Name,Symbol,Canonical_Symbol
17,6,NIFTY 500,ENTRY,1,Aarti Drugs Ltd.,AARTIDRUGS,AARTIDRUGS
19,6,NIFTY 500,ENTRY,3,Adani Enterprises Ltd.,ADANIENT,ADANIENT
20,6,NIFTY 500,ENTRY,4,Alembic Ltd.,ALEMBICLTD,ALEMBICLTD
21,6,NIFTY 500,ENTRY,5,Alok Industries Ltd.,ALOKINDS,ALOKINDS
22,6,NIFTY 500,ENTRY,6,Chalet Hotels Ltd.,CHALET,CHALET
23,6,NIFTY 500,ENTRY,7,Gujarat Ambuja Exports Ltd.,GAEL,GAEL
25,6,NIFTY 500,ENTRY,9,IOL Chem and Pharma Ltd.,IOLCP,IOLCP
26,6,NIFTY 500,ENTRY,10,Navneet Education Ltd.,NAVNETEDUL,NAVNETEDUL
24,6,NIFTY 500,ENTRY,8,Huhtamaki PPL Ltd.,PAPERPROD,PAPERPROD
27,6,NIFTY 500,ENTRY,11,Polyplex Corporation Ltd.,POLYPLEX,POLYPLEX



Counts by role:


,Role,Rows
0,ENTRY,17
1,EXIT,17



No membership intervals, prices, or flags were modified.


In [39]:
from pathlib import Path
import re
import pandas as pd

BASE = Path("/content/nifty500_pit_data")
SCAN_PATH = BASE / "nifty500_official_pdf_scan_v1.csv"

EVENTS_PATH = BASE / "nifty500_official_event_candidates_v3.csv"
SECTION_PATH = BASE / "nifty500_event_sections_v3.csv"
RELEASE_PATH = BASE / "nifty500_event_release_summary_v3.csv"
CHECK_PATH = BASE / "nifty500_event_known_list_checks_v3.csv"

assert SCAN_PATH.exists(), f"Missing scan report: {SCAN_PATH}"
scan = pd.read_csv(SCAN_PATH)

# Publication date + title substring -> effective date verified from the
# specific official release. These overrides are narrow and explicit.
DATE_OVERRIDES = [
    ("2017-03-07", "Replacements in Indices", "2017-03-16"),
    ("2017-04-27", "Replacements in indices and change in NIFTY100", "2017-05-26"),
    ("2017-08-29", "Replacement in Indices", "2017-09-05"),
    ("2018-08-01", "Replacement in Indices", "2018-08-08"),
    ("2020-08-20", "Revision in criteria and replacements in Indices", "2020-09-25"),
]

# Known list counts for QA. They are checks, not rules used by the parser.
KNOWN_LIST_CHECKS = [
    ("2016-01-11", "January 18, 2016", 1, 1),
    ("2016-02-22", "April 01, 2016", 75, 76),
    ("2016-04-22", "April 29, 2016", 3, 3),
    ("2016-08-12", "September 30, 2016", 25, 25),
    ("2016-10-17", "October 24, 2016", 1, 1),
    ("2020-08-20", "Revision in criteria", 17, 17),
]

MONTH = (
    r"(?:January|February|March|April|May|June|July|August|"
    r"September|October|November|December|Jan|Feb|Mar|Apr|"
    r"Jun|Jul|Aug|Sep|Oct|Nov|Dec)"
)
DATE_TEXT = (
    rf"(?:{MONTH}\s+\d{{1,2}},?\s+\d{{4}}|"
    rf"\d{{1,2}}\s+{MONTH}\s+\d{{4}})"
)
DATE_RE = re.compile(rf"\b{DATE_TEXT}\b", re.I)

ACTION_RE = re.compile(
    r"\bthe\s+following\s+"
    r"(?:companies|scrips|company|scrip)\s+"
    r"(?:(?:are|is)\s+(?:being\s+)?|will\s+be\s+|shall\s+be\s+)"
    r"(excluded|included|removed|added)\b",
    re.I
)

SYMBOL_RE = re.compile(r"^[A-Z0-9][A-Z0-9&.\-/]{1,24}$")
STOP_SYMBOLS = {
    "LTD", "LIMITED", "PVT", "PRIVATE", "COMPANY", "COMPANIES",
    "CO", "SR", "NO", "SYMBOL", "SCRIP", "SCRIPS", "NAME",
    "INDEX", "EXCLUDED", "INCLUDED", "REMOVED", "ADDED", "PAGE"
}

def canonical_symbol(value):
    return re.sub(r"[^A-Z0-9]", "", str(value).strip().upper())

def parse_date(value):
    result = pd.to_datetime(value, errors="coerce")
    return None if pd.isna(result) else result.strftime("%Y-%m-%d")

def strip_heading_prefix(value):
    return re.sub(
        r"^\s*(?:\d{1,2}|[A-Za-z])\s*[.)]\s*",
        "", str(value)
    ).strip()

def is_bare_marker(value):
    return bool(re.fullmatch(
        r"\s*(?:[A-Za-z]|\d{1,2})\s*[.)]\s*",
        str(value)
    ))

def is_index_name(value):
    s = strip_heading_prefix(value)
    if not re.match(r"^(?:NIFTY|CNX)\b", s, re.I):
        return False
    if len(s.split()) > 8:
        return False
    return bool(re.fullmatch(
        r"(?:NIFTY|CNX)\s+[A-Z0-9][A-Z0-9 &'./()%-]*",
        s, re.I
    ))

def is_nifty500_name(value):
    return bool(re.fullmatch(
        r"(?:NIFTY|CNX)\s*[- ]?\s*500(?:\s+INDEX)?",
        strip_heading_prefix(value), re.I
    ))

def find_index_sections(lines):
    """Find standalone, numbered, lettered and split-line index headings."""
    found = []
    i = 0

    while i < len(lines):
        line = lines[i]

        # Split-line pattern: "c)" on one line, "NIFTY 500" on next.
        if (
            is_bare_marker(line)
            and i + 1 < len(lines)
            and is_index_name(lines[i + 1])
        ):
            found.append({
                "start_idx": i,
                "title_idx": i + 1,
                "title": strip_heading_prefix(lines[i + 1])
            })
            i += 2
            continue

        # Same-line pattern: "4) Nifty 500 Index", "A) NIFTY 100".
        same = re.match(
            r"^\s*(?:\d{1,2}|[A-Za-z])\s*[.)]\s*((?:NIFTY|CNX)\b.*)$",
            line, re.I
        )
        if same and is_index_name(same.group(1)):
            found.append({
                "start_idx": i,
                "title_idx": i,
                "title": strip_heading_prefix(line)
            })
            i += 1
            continue

        # Unnumbered headings are retained as candidates; those without
        # an action list will not become membership events.
        if is_index_name(line):
            found.append({
                "start_idx": i,
                "title_idx": i,
                "title": line.strip()
            })

        i += 1

    # Avoid duplicate start positions.
    unique = {}
    for item in found:
        unique[item["start_idx"]] = item
    return [unique[k] for k in sorted(unique)]

def get_effective_date(pub_date, title, lines, section_start):
    for pub, substring, effective in DATE_OVERRIDES:
        if pub == pub_date and substring.lower() in title.lower():
            return effective, "verified_manual_override"

    title_dates = []
    for match in DATE_RE.finditer(title):
        parsed = parse_date(match.group(0))
        if parsed and parsed not in title_dates:
            title_dates.append(parsed)

    if len(title_dates) == 1:
        return title_dates[0], "title_single_date"

    # For multi-date or undated titles, look for an explicit effective
    # date in the introductory text close to a Nifty 500 reference.
    prefix = "\n".join(lines[:section_start])
    refs = list(re.finditer(
        r"\b(?:NIFTY|CNX)\s*[- ]?\s*500\b", prefix, re.I
    ))

    if refs:
        local = prefix[
            refs[-1].start():min(len(prefix), refs[-1].start() + 1800)
        ]
        phrase = re.compile(
            rf"(?:shall\s+become\s+effective\s+from|"
            rf"become\s+effective\s+from|effective\s+from|"
            rf"with\s+effect\s+from|w\.?\s*e\.?\s*f\.?)"
            rf"\s*[:\-]?\s*({DATE_TEXT})",
            re.I
        )
        match = phrase.search(local)
        if match:
            parsed = parse_date(match.group(1))
            if parsed:
                return parsed, "intro_clause_near_nifty500"

    return None, "unresolved_multiple_or_missing_dates"

def extract_symbol_rows(block):
    raw = [
        line.strip()
        for line in block.splitlines()
        if line.strip()
    ]
    output = []
    seen = set()

    for i, candidate in enumerate(raw):
        raw_symbol = candidate.upper()
        symbol = canonical_symbol(raw_symbol)

        if (
            not SYMBOL_RE.fullmatch(raw_symbol)
            or symbol in STOP_SYMBOLS
            or not re.search(r"[A-Z]", symbol)
            or i == 0
        ):
            continue

        company = raw[i - 1]

        # Join suffixes split across lines in extracted PDF tables.
        if canonical_symbol(company) in {"LTD", "LIMITED"} and i >= 2:
            company = raw[i - 2] + " " + company

        if not company or re.fullmatch(r"[\d\s.)]+", company):
            continue
        if canonical_symbol(company) in STOP_SYMBOLS:
            continue

        serial = None
        if i >= 2 and re.fullmatch(r"\d{1,3}", raw[i - 2]):
            serial = int(raw[i - 2])

        key = (serial, company, symbol)
        if key not in seen:
            output.append({
                "Serial_No": serial,
                "Company_Name": company,
                "Raw_Symbol": raw_symbol,
                "Canonical_Symbol": symbol
            })
            seen.add(key)

    return output

event_rows = []
section_rows = []
missing_text = []

for _, item in scan.iterrows():
    pub_parsed = pd.to_datetime(item["Published_Date"], errors="coerce")
    pub_date = (
        pub_parsed.strftime("%Y-%m-%d")
        if pd.notna(pub_parsed) else ""
    )
    title = str(item.get("Title", ""))
    url = str(item.get("URL", ""))
    text_path = str(item.get("Text_Cache", "")).strip()

    if not text_path or not Path(text_path).exists():
        missing_text.append({"Published_Date": pub_date, "Title": title, "URL": url})
        continue

    full_text = Path(text_path).read_text(
        encoding="utf-8", errors="replace"
    )
    full_text = re.sub(r"\n--- PAGE \d+ ---\n", "\n", full_text)
    lines = [line.strip() for line in full_text.splitlines()]
    headings = find_index_sections(lines)

    for section_idx, heading in enumerate(headings):
        start = heading["start_idx"]
        title_idx = heading["title_idx"]
        end = (
            headings[section_idx + 1]["start_idx"]
            if section_idx + 1 < len(headings)
            else len(lines)
        )

        # Only exact Nifty 500/CNX 500 titles are eligible membership
        # sections; criteria mentions without actions remain non-events.
        if not is_nifty500_name(heading["title"]):
            continue

        body = "\n".join(lines[title_idx + 1:end])
        actions = list(ACTION_RE.finditer(body))
        effective, date_source = get_effective_date(
            pub_date, title, lines, start
        )

        section_events = []
        for action_idx, action in enumerate(actions):
            block_start = action.end()
            block_end = (
                actions[action_idx + 1].start()
                if action_idx + 1 < len(actions)
                else len(body)
            )
            parsed_rows = extract_symbol_rows(
                body[block_start:block_end]
            )

            role = (
                "EXIT"
                if action.group(1).lower() in {"excluded", "removed"}
                else "ENTRY"
            )

            for parsed in parsed_rows:
                row = {
                    "Published_Date": pub_date,
                    "Release_Title": title,
                    "Source_URL": url,
                    "Effective_Date": effective,
                    "Effective_Date_Source": date_source,
                    "Section_Title": heading["title"],
                    "Role": role,
                    "Action_Heading": action.group(0),
                    **parsed,
                    "Event_Status": (
                        "DATE_UNRESOLVED" if effective is None
                        else "CANDIDATE_REQUIRES_QA"
                    )
                }
                event_rows.append(row)
                section_events.append(row)

        exits = sum(r["Role"] == "EXIT" for r in section_events)
        entries = sum(r["Role"] == "ENTRY" for r in section_events)

        section_rows.append({
            "Published_Date": pub_date,
            "Title": title,
            "URL": url,
            "Section_Number": section_idx + 1,
            "Section_Title": heading["title"],
            "Effective_Date": effective,
            "Effective_Date_Source": date_source,
            "Action_Headings": len(actions),
            "Parsed_Exits": exits,
            "Parsed_Entries": entries,
            "Parsed_Rows": len(section_events),
            "Section_Status": (
                "NO_ACTIONS" if not actions
                else "DATE_UNRESOLVED" if effective is None
                else "CANDIDATE_REQUIRES_QA"
            )
        })

events = pd.DataFrame(event_rows)
sections = pd.DataFrame(section_rows)

events.to_csv(EVENTS_PATH, index=False)
sections.to_csv(SECTION_PATH, index=False)

# Summarize only Nifty 500 sections containing actual action headings.
action_sections = sections[sections["Action_Headings"].gt(0)].copy()

if action_sections.empty:
    releases = pd.DataFrame(columns=[
        "Published_Date", "Title", "URL", "Action_Sections",
        "Parsed_Exits", "Parsed_Entries", "Candidate_Rows",
        "Effective_Dates", "Unresolved_Date_Sections"
    ])
else:
    releases = (
        action_sections.groupby(
            ["Published_Date", "Title", "URL"], dropna=False
        )
        .agg(
            Action_Sections=("Action_Headings", "sum"),
            Parsed_Exits=("Parsed_Exits", "sum"),
            Parsed_Entries=("Parsed_Entries", "sum"),
            Candidate_Rows=("Parsed_Rows", "sum"),
            Effective_Dates=(
                "Effective_Date",
                lambda s: ", ".join(sorted(set(
                    str(v) for v in s.dropna()
                )))
            ),
            Unresolved_Date_Sections=(
                "Effective_Date", lambda s: int(s.isna().sum())
            )
        )
        .reset_index()
    )

    releases.to_csv(RELEASE_PATH, index=False)

# Check six independently known list counts.
checks = []
for pub_date, title_fragment, expected_exits, expected_entries in KNOWN_LIST_CHECKS:
    matches = releases[
        releases["Published_Date"].eq(pub_date)
        & releases["Title"].str.contains(
            title_fragment, case=False, regex=False, na=False
        )
    ]

    if len(matches) != 1:
        checks.append({
            "Published_Date": pub_date,
            "Title_Fragment": title_fragment,
            "Expected_Exits": expected_exits,
            "Expected_Entries": expected_entries,
            "Parsed_Exits": None,
            "Parsed_Entries": None,
            "Result": f"REVIEW: matched {len(matches)} release rows"
        })
        continue

    found = matches.iloc[0]
    passed = (
        int(found["Parsed_Exits"]) == expected_exits
        and int(found["Parsed_Entries"]) == expected_entries
    )

    checks.append({
        "Published_Date": pub_date,
        "Title_Fragment": title_fragment,
        "Expected_Exits": expected_exits,
        "Expected_Entries": expected_entries,
        "Parsed_Exits": int(found["Parsed_Exits"]),
        "Parsed_Entries": int(found["Parsed_Entries"]),
        "Result": "PASS" if passed else "REVIEW"
    })

check_df = pd.DataFrame(checks)
check_df.to_csv(CHECK_PATH, index=False)

print("=" * 75)
print("NIFTY 500 EVENT EXTRACTOR V3 — CANDIDATES ONLY")
print("=" * 75)
print("Cached releases scanned:", len(scan))
print("Nifty 500 candidate sections:", len(sections))
print("Sections with action headings:", len(action_sections))
print("Candidate event rows:", len(events))
print("Rows with unresolved effective dates:",
      int(events["Effective_Date"].isna().sum()) if not events.empty else 0)
print("Releases with missing cached text:", len(missing_text))

print("\nKNOWN-LIST VALIDATION")
display(check_df)

print("\nACTIONABLE RELEASE SUMMARY — 2015–2018")
display(
    releases[
        pd.to_datetime(releases["Published_Date"], errors="coerce")
        .dt.year.between(2015, 2018)
    ][[
        "Published_Date", "Title", "Effective_Dates",
        "Parsed_Exits", "Parsed_Entries", "Candidate_Rows",
        "Unresolved_Date_Sections"
    ]].sort_values("Published_Date")
)

print("\nRELEASES WITH UNRESOLVED EFFECTIVE DATES")
display(
    releases[releases["Unresolved_Date_Sections"].gt(0)][[
        "Published_Date", "Title", "Parsed_Exits",
        "Parsed_Entries", "Candidate_Rows",
        "Unresolved_Date_Sections", "URL"
    ]].sort_values("Published_Date")
)

print("\nSaved candidate events:", EVENTS_PATH)
print("Saved section QA:", SECTION_PATH)
print("Saved release summary:", RELEASE_PATH)
print("Saved known-list checks:", CHECK_PATH)
print("No source membership intervals, prices or flags were modified.")

NIFTY 500 EVENT EXTRACTOR V3 — CANDIDATES ONLY
Cached releases scanned: 157
Nifty 500 candidate sections: 92
Sections with action headings: 84
Candidate event rows: 1484
Rows with unresolved effective dates: 160
Releases with missing cached text: 0

KNOWN-LIST VALIDATION


,Published_Date,Title_Fragment,Expected_Exits,Expected_Entries,Parsed_Exits,Parsed_Entries,Result
0,2016-01-11,"January 18, 2016",1,1,1,1,PASS
1,2016-02-22,"April 01, 2016",75,76,75,76,PASS
2,2016-04-22,"April 29, 2016",3,3,3,3,PASS
3,2016-08-12,"September 30, 2016",25,25,25,25,PASS
4,2016-10-17,"October 24, 2016",1,1,1,1,PASS
5,2020-08-20,Revision in criteria,17,17,17,17,PASS



ACTIONABLE RELEASE SUMMARY — 2015–2018


,Published_Date,Title,Effective_Dates,Parsed_Exits,Parsed_Entries,Candidate_Rows,Unresolved_Date_Sections
0,2015-02-20,"Change in indices w.e.f March 27, 2015.",2015-03-27,16,16,32,0
1,2015-03-18,"Change in Indices w.e.f March 24, 2015 and Mar...",,3,3,6,1
2,2015-08-12,Change in Indices and Methodology w.e.f Septem...,2015-09-28,20,19,39,0
3,2015-08-24,"Change in Indices w.e.f September 28, 2015",2015-09-28,3,3,6,0
4,2015-09-18,"Change in Indices w.e.f October 19, 2015",2015-10-19,1,1,2,0
5,2015-12-07,Change in Nifty 500 & Nifty Smallcap 100 Indic...,2015-12-14,2,2,4,0
6,2016-01-11,"Change in Indices w.e.f January 18, 2016",2016-01-18,1,1,2,0
7,2016-02-22,"Change in Indices w.e.f. April 01, 2016.",2016-04-01,75,76,151,0
8,2016-04-22,"Change in Indices w.e.f. April 29, 2016.",2016-04-29,3,3,6,0
9,2016-08-12,"Change in Indices w.e.f. September 30, 2016.",2016-09-30,25,25,50,0



RELEASES WITH UNRESOLVED EFFECTIVE DATES


,Published_Date,Title,Parsed_Exits,Parsed_Entries,Candidate_Rows,Unresolved_Date_Sections,URL
1,2015-03-18,"Change in Indices w.e.f March 24, 2015 and Mar...",3,3,6,1,https://www.niftyindices.com/Press_Release/ind...
23,2018-03-06,Replacements in Indices,1,1,2,1,https://www.niftyindices.com/Press_Release/ind...
43,2020-07-02,Replacement in Indices,1,1,2,1,https://www.niftyindices.com/Press_Release/ind...
45,2020-09-07,Replacements in Indices,5,5,10,1,https://www.niftyindices.com/Press_Release/ind...
51,2021-09-15,Replacements in indices and revision in criteria,22,22,44,1,https://www.niftyindices.com/Press_Release/ind...
52,2021-10-22,Revision in index maintenance guidelines and r...,1,1,2,1,https://www.niftyindices.com/Press_Release/ind...
53,2022-02-24,Revision in criteria and replacements in indices,32,32,64,1,https://www.niftyindices.com/Press_Release/ind...
67,2023-08-17,Replacements in indices and revision in criteria,18,18,36,1,https://www.niftyindices.com/Press_Release/ind...
74,2025-03-17,Replacement in indices,1,1,2,1,https://www.niftyindices.com/Press_Release/ind...



Saved candidate events: /content/nifty500_pit_data/nifty500_official_event_candidates_v3.csv
Saved section QA: /content/nifty500_pit_data/nifty500_event_sections_v3.csv
Saved release summary: /content/nifty500_pit_data/nifty500_event_release_summary_v3.csv
Saved known-list checks: /content/nifty500_pit_data/nifty500_event_known_list_checks_v3.csv
No source membership intervals, prices or flags were modified.


In [40]:
from pathlib import Path
import re
import pandas as pd

BASE = Path("/content/nifty500_pit_data")

SUMMARY_PATH = BASE / "nifty500_event_release_summary_v3.csv"
SCAN_PATH = BASE / "nifty500_official_pdf_scan_v1.csv"

OUT_PATH = BASE / "nifty500_unresolved_date_evidence_v2.csv"

summary = pd.read_csv(SUMMARY_PATH)
scan = pd.read_csv(SCAN_PATH)

summary["Unresolved_Date_Sections"] = pd.to_numeric(
    summary["Unresolved_Date_Sections"], errors="coerce"
).fillna(0)

unresolved = summary[
    summary["Unresolved_Date_Sections"].gt(0)
].copy()

assert not unresolved.empty, "No unresolved release dates remain."

scan_by_url = (
    scan.dropna(subset=["URL"])
    .drop_duplicates("URL")
    .set_index("URL")
)

date_cue = re.compile(
    r"effective|w\.?\s*e\.?\s*f\.?|close of|"
    r"shall become|will become|changes shall|"
    r"effective from|with effect from|date:",
    re.IGNORECASE
)

date_rx = re.compile(
    r"\b(?:Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|"
    r"May|Jun(?:e)?|Jul(?:y)?|Aug(?:ust)?|Sep(?:tember)?|"
    r"Oct(?:ober)?|Nov(?:ember)?|Dec(?:ember)?)\s+"
    r"\d{1,2},?\s+\d{4}\b"
    r"|\b\d{1,2}\s+"
    r"(?:Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|"
    r"May|Jun(?:e)?|Jul(?:y)?|Aug(?:ust)?|Sep(?:tember)?|"
    r"Oct(?:ober)?|Nov(?:ember)?|Dec(?:ember)?)\s+\d{4}\b",
    re.IGNORECASE
)

evidence_rows = []

print("=" * 78)
print("UNRESOLVED DATE EVIDENCE — CACHED OFFICIAL RELEASE TEXT")
print("=" * 78)
print("Affected releases:", unresolved["URL"].nunique())
print("Candidate rows in affected releases:",
      int(unresolved["Candidate_Rows"].sum()))

for _, rel in unresolved.sort_values(
    ["Published_Date", "Title"]
).iterrows():

    url = str(rel["URL"])
    if url not in scan_by_url.index:
        print("\nSCAN ENTRY NOT FOUND:", rel["Title"])
        continue

    doc = scan_by_url.loc[url]
    text_path = Path(str(doc["Text_Cache"]))

    print("\n" + "-" * 78)
    print("Published:", rel["Published_Date"])
    print("Title:", rel["Title"])
    print("Candidate exits / entries:",
          rel["Parsed_Exits"], "/", rel["Parsed_Entries"])
    print("URL:", url)

    if not text_path.exists():
        print("Cached text missing:", text_path)
        continue

    text = text_path.read_text(
        encoding="utf-8", errors="replace"
    )

    # Preserve page labels to make the evidence auditable.
    parts = re.split(r"\n--- PAGE (\d+) ---\n", text)
    doc_lines = []

    for i in range(1, len(parts), 2):
        page_no = int(parts[i])
        page_text = parts[i + 1] if i + 1 < len(parts) else ""

        for line_no, line in enumerate(page_text.splitlines(), 1):
            cleaned = re.sub(r"\s+", " ", line).strip()
            if cleaned:
                doc_lines.append({
                    "Page": page_no,
                    "Line": line_no,
                    "Text": cleaned
                })

    # Locate date-cue lines and include adjacent lines in case the PDF
    # extraction split the date onto the next line.
    selected = set()

    for i, row in enumerate(doc_lines):
        if date_cue.search(row["Text"]) or date_rx.search(row["Text"]):
            for j in range(max(0, i - 1), min(len(doc_lines), i + 3)):
                selected.add(j)

    # Also capture the first few lines around text mentioning Nifty 500.
    for i, row in enumerate(doc_lines):
        if re.search(r"\b(?:NIFTY|CNX)\s*[- ]?\s*500\b",
                     row["Text"], re.IGNORECASE):
            for j in range(max(0, i - 2), min(len(doc_lines), i + 4)):
                selected.add(j)

    selected = sorted(selected)

    # Keep the inspection compact; the full selected evidence is saved.
    display_indices = selected[:65]

    for i in display_indices:
        row = doc_lines[i]
        print(f"P{row['Page']:02d} L{row['Line']:04d}: {row['Text']}")

        evidence_rows.append({
            "Published_Date": rel["Published_Date"],
            "Title": rel["Title"],
            "URL": url,
            "Candidate_Rows": rel["Candidate_Rows"],
            "Page": row["Page"],
            "Line": row["Line"],
            "Evidence_Text": row["Text"]
        })

    if len(selected) > len(display_indices):
        print(f"... {len(selected) - len(display_indices)} further lines "
              "saved to the evidence CSV")

pd.DataFrame(evidence_rows).to_csv(OUT_PATH, index=False)

print("\n" + "=" * 78)
print("Evidence report saved:", OUT_PATH)
print("Read-only inspection complete. No dates or data were modified.")

UNRESOLVED DATE EVIDENCE — CACHED OFFICIAL RELEASE TEXT
Affected releases: 9
Candidate rows in affected releases: 168

------------------------------------------------------------------------------
Published: 2015-03-18
Title: Change in Indices w.e.f March 24, 2015 and March 27, 2015
Candidate exits / entries: 3 / 3
URL: https://www.niftyindices.com/Press_Release/ind_prs18032015_2.pdf
P01 L0008: amalgamations and demergers.
P01 L0010: A. Replacements effective March 24, 2015 (close of March 23, 2015):
P01 L0012: (1)
P01 L0013: CNX 500 Index:
P01 L0015: The following company is being excluded:
P01 L0017: Sr. No. Company Name
P01 L0018: Symbol
P01 L0048: MANAPPURAM
P01 L0051: B. Replacements effective March 27, 2015 (close of March 26, 2015):
P01 L0053: (1)
P01 L0054: CNX 200 Index
P01 L0070: COX&KINGS
P02 L0001: (2)
P02 L0002: CNX 500 Index
P02 L0004: The following companies are being excluded:
P02 L0006: Sr. No. Company Name
P02 L0007: Symbol
P02 L0071: Place: Mumbai
P02 L0073: Date: M

In [41]:
from pathlib import Path
import re
import pandas as pd

BASE = Path("/content/nifty500_pit_data")

SCAN_PATH = BASE / "nifty500_official_pdf_scan_v1.csv"
EVENTS_PATH = BASE / "nifty500_official_event_candidates_v3.csv"
SECTIONS_PATH = BASE / "nifty500_event_sections_v3.csv"

assert SCAN_PATH.exists()
assert EVENTS_PATH.exists()
assert SECTIONS_PATH.exists()

scan = pd.read_csv(SCAN_PATH)
events = pd.read_csv(EVENTS_PATH)
sections = pd.read_csv(SECTIONS_PATH)

targets = {
    "2015-03-18": "Change in Indices",
    "2020-09-07": "Replacements in Indices"
}

scan["_pub"] = pd.to_datetime(
    scan["Published_Date"], errors="coerce"
).dt.strftime("%Y-%m-%d")

for pub_date, title_fragment in targets.items():
    docs = scan[
        scan["_pub"].eq(pub_date)
        & scan["Title"].astype(str).str.contains(
            title_fragment, case=False, na=False
        )
    ]

    print("\n" + "=" * 85)
    print(f"PUBLICATION DATE: {pub_date}")
    print("=" * 85)

    if len(docs) != 1:
        print(f"Expected one release, found {len(docs)}")
        display(docs[["Published_Date", "Title", "URL"]])
        continue

    doc = docs.iloc[0]
    url = str(doc["URL"])
    text_path = Path(str(doc["Text_Cache"]))

    # Display the currently extracted event rows for this release.
    release_events = events[
        events["Published_Date"].astype(str).eq(pub_date)
    ].copy()

    print("\nCURRENT CANDIDATE EVENT ROWS")
    print("Rows:", len(release_events))

    event_cols = [
        c for c in [
            "Role", "Company_Name", "Raw_Symbol",
            "Canonical_Symbol", "Effective_Date",
            "Effective_Date_Source", "Event_Status"
        ]
        if c in release_events.columns
    ]

    if not release_events.empty:
        display(
            release_events[event_cols]
            .sort_values(["Role", "Canonical_Symbol"])
        )

    # Display section summaries for this publication.
    section_cols = [
        c for c in [
            "Published_Date", "Section_Number", "Section_Title",
            "Effective_Date", "Effective_Date_Source",
            "Action_Headings", "Parsed_Exits", "Parsed_Entries",
            "Parsed_Rows", "Section_Status"
        ]
        if c in sections.columns
    ]

    release_sections = sections[
        sections["Published_Date"].astype(str).eq(pub_date)
    ].copy()

    print("\nEXISTING SECTION SUMMARY")
    display(release_sections[section_cols])

    # Read cached official PDF text and preserve page labels.
    assert text_path.exists(), f"Missing cached text: {text_path}"
    full_text = text_path.read_text(
        encoding="utf-8", errors="replace"
    )

    parts = re.split(r"\n--- PAGE (\d+) ---\n", full_text)
    source_lines = []

    for i in range(1, len(parts), 2):
        page_no = int(parts[i])
        page_text = parts[i + 1] if i + 1 < len(parts) else ""

        for line_no, line in enumerate(page_text.splitlines(), 1):
            cleaned = re.sub(r"\s+", " ", line).strip()
            if cleaned:
                source_lines.append({
                    "Page": page_no,
                    "Line": line_no,
                    "Text": cleaned
                })

    cues = re.compile(
        r"effective|w\.?\s*e\.?\s*f\.?|close of|"
        r"replacement|NIFTY\s*500|CNX\s*500|"
        r"being excluded|being included|"
        r"being removed|being added",
        re.IGNORECASE
    )

    matching_indices = [
        i for i, row in enumerate(source_lines)
        if cues.search(row["Text"])
    ]

    # Print selected context around key source lines, with a little
    # adjacent text so split-line dates and table headings are visible.
    selected = set()
    for i in matching_indices:
        for j in range(max(0, i - 1), min(len(source_lines), i + 2)):
            selected.add(j)

    print("\nOFFICIAL SOURCE TEXT: DATE AND SECTION EVIDENCE")
    for i in sorted(selected):
        row = source_lines[i]
        print(f"P{row['Page']:02d} L{row['Line']:04d}: {row['Text']}")

print("\nRead-only diagnostic complete. No event dates or source data modified.")


PUBLICATION DATE: 2015-03-18

CURRENT CANDIDATE EVENT ROWS
Rows: 6


,Role,Company_Name,Raw_Symbol,Canonical_Symbol,Effective_Date,Effective_Date_Source,Event_Status
35,ENTRY,Mayur Uniquoters Ltd.,MAYURUNIQ,MAYURUNIQ,NaN,unresolved_multiple_or_missing_dates,DATE_UNRESOLVED
36,ENTRY,Munjal Showa Ltd.,MUNJALSHOW,MUNJALSHOW,NaN,unresolved_multiple_or_missing_dates,DATE_UNRESOLVED
37,ENTRY,Suven Life Sciences Ltd.,SUVEN,SUVEN,NaN,unresolved_multiple_or_missing_dates,DATE_UNRESOLVED
32,EXIT,CMC Ltd.,CMC,CMC,NaN,unresolved_multiple_or_missing_dates,DATE_UNRESOLVED
33,EXIT,Mastek Ltd.,MASTEK,MASTEK,NaN,unresolved_multiple_or_missing_dates,DATE_UNRESOLVED
34,EXIT,Shasun Pharmaceuticals Ltd.,SHASUNPHAR,SHASUNPHAR,NaN,unresolved_multiple_or_missing_dates,DATE_UNRESOLVED



EXISTING SECTION SUMMARY


,Published_Date,Section_Number,Section_Title,Effective_Date,Effective_Date_Source,Action_Headings,Parsed_Exits,Parsed_Entries,Parsed_Rows,Section_Status
1,2015-03-18,3,CNX 500 Index,NaN,unresolved_multiple_or_missing_dates,2,3,3,6,DATE_UNRESOLVED



OFFICIAL SOURCE TEXT: DATE AND SECTION EVIDENCE
P01 L0008: amalgamations and demergers.
P01 L0010: A. Replacements effective March 24, 2015 (close of March 23, 2015):
P01 L0012: (1)
P01 L0013: CNX 500 Index:
P01 L0015: The following company is being excluded:
P01 L0017: Sr. No. Company Name
P01 L0021: BALKRISIND
P01 L0023: The following company is being included:
P01 L0025: Sr. No. Company Name
P01 L0032: CNX Alpha Index
P01 L0034: The following company is being excluded:
P01 L0036: Sr. No. Company Name
P01 L0040: BALKRISIND
P01 L0042: The following company is being included:
P01 L0044: Sr. No. Company Name
P01 L0048: MANAPPURAM
P01 L0051: B. Replacements effective March 27, 2015 (close of March 26, 2015):
P01 L0053: (1)
P01 L0054: CNX 200 Index
P01 L0056: The following company is being excluded:
P01 L0058: Sr. No. Company Name
P01 L0062: CMC
P01 L0064: The following company is being included:
P01 L0066: Sr. No. Company Name
P02 L0001: (2)
P02 L0002: CNX 500 Index
P02 L0004: The follo

,Role,Company_Name,Raw_Symbol,Canonical_Symbol,Effective_Date,Effective_Date_Source,Event_Status
811,ENTRY,JTEKT India Ltd.,JTEKTINDIA,JTEKTINDIA,NaN,unresolved_multiple_or_missing_dates,DATE_UNRESOLVED
816,ENTRY,NOCIL Ltd.,NOCIL,NOCIL,2020-09-14,intro_clause_near_nifty500,CANDIDATE_REQUIRES_QA
817,ENTRY,PSP Projects Ltd.,PSPPROJECT,PSPPROJECT,2020-09-14,intro_clause_near_nifty500,CANDIDATE_REQUIRES_QA
818,ENTRY,Swaraj Engines Ltd.,SWARAJENG,SWARAJENG,2020-09-14,intro_clause_near_nifty500,CANDIDATE_REQUIRES_QA
819,ENTRY,Tata Coffee Ltd.,TATACOFFEE,TATACOFFEE,2020-09-14,intro_clause_near_nifty500,CANDIDATE_REQUIRES_QA
812,EXIT,Arvind Fashions Ltd.,ARVINDFASN,ARVINDFASN,2020-09-14,intro_clause_near_nifty500,CANDIDATE_REQUIRES_QA
810,EXIT,Asahi India Glass Ltd.,ASAHIINDIA,ASAHIINDIA,NaN,unresolved_multiple_or_missing_dates,DATE_UNRESOLVED
813,EXIT,KPIT Technologies Ltd.,KPITTECH,KPITTECH,2020-09-14,intro_clause_near_nifty500,CANDIDATE_REQUIRES_QA
814,EXIT,Repco Home Finance Ltd.,REPCOHOME,REPCOHOME,2020-09-14,intro_clause_near_nifty500,CANDIDATE_REQUIRES_QA
815,EXIT,Spandana Sphoorty Financial Ltd.,SPANDANA,SPANDANA,2020-09-14,intro_clause_near_nifty500,CANDIDATE_REQUIRES_QA



EXISTING SECTION SUMMARY


,Published_Date,Section_Number,Section_Title,Effective_Date,Effective_Date_Source,Action_Headings,Parsed_Exits,Parsed_Entries,Parsed_Rows,Section_Status
50,2020-09-07,1,NIFTY 500,NaN,unresolved_multiple_or_missing_dates,2,1,1,2,DATE_UNRESOLVED
51,2020-09-07,7,NIFTY 500,2020-09-14,intro_clause_near_nifty500,2,4,4,8,CANDIDATE_REQUIRES_QA



OFFICIAL SOURCE TEXT: DATE AND SECTION EVIDENCE
P01 L0007: September 07, 2020
P01 L0009: Replacements in Indices
P01 L0011: The Index Maintenance Sub-Committee (IMSC) of NSE Indices Limited has decided to make the
P01 L0012: replacements in various indices as given hereunder:
P01 L0014: A. Replacement of stock on account of shifting to Z category
P01 L0016: NSE vide its circular NSE/CML/42939 dated September 04, 2020 has announced shifting of
P01 L0017: securities to Z category effective from September 14, 2020.
P01 L0019: In view of the above, the Index Maintenance Sub-Committee (IMSC) of NSE Indices Limited has
P01 L0020: decided to make replacements in various indices as given below.
P01 L0022: 1)
P01 L0023: NIFTY 500
P01 L0025: The following company is being excluded:
P01 L0027: Sr. No.
P01 L0032: ASAHIINDIA
P01 L0034: The following company is being included:
P01 L0036: Sr. No.
P01 L0044: NIFTY Smallcap 250
P01 L0046: The following company is being excluded:
P01 L0048: Sr. No.
P01

In [42]:
from pathlib import Path
import pandas as pd

BASE = Path("/content/nifty500_pit_data")
SOURCE = BASE / "nifty500_official_event_candidates_v3.csv"

OUTPUT = BASE / "nifty500_official_event_candidates_v4.csv"
CORRECTION_LOG = BASE / "nifty500_event_date_corrections_v4.csv"
QA_OUTPUT = BASE / "nifty500_event_date_QA_v4.csv"

assert SOURCE.exists(), f"Missing source ledger: {SOURCE}"

events = pd.read_csv(SOURCE, dtype={"Raw_Symbol": str})
required = {
    "Published_Date", "Release_Title", "Source_URL",
    "Effective_Date", "Effective_Date_Source",
    "Role", "Company_Name", "Raw_Symbol",
    "Canonical_Symbol", "Event_Status"
}
assert required.issubset(events.columns), (
    f"Unexpected columns. Missing: {required - set(events.columns)}"
)

events["Published_Date"] = pd.to_datetime(
    events["Published_Date"], errors="coerce"
).dt.strftime("%Y-%m-%d")
events["Role"] = events["Role"].astype(str).str.upper().str.strip()
events["Canonical_Symbol"] = (
    events["Canonical_Symbol"].astype(str).str.upper().str.strip()
)

if "Date_Override_Reason" not in events.columns:
    events["Date_Override_Reason"] = ""

corrections = []

def patch_rows(mask, new_date, reason, source_url):
    idxs = events.index[mask].tolist()
    assert idxs, f"No rows found for correction: {reason}"

    for idx in idxs:
        old_date = events.at[idx, "Effective_Date"]
        corrections.append({
            "Action": "DATE_CORRECTION",
            "Published_Date": events.at[idx, "Published_Date"],
            "Role": events.at[idx, "Role"],
            "Company_Name": events.at[idx, "Company_Name"],
            "Symbol": events.at[idx, "Canonical_Symbol"],
            "Old_Effective_Date": old_date,
            "New_Effective_Date": new_date,
            "Reason": reason,
            "Source_URL": source_url
        })

    events.loc[idxs, "Effective_Date"] = new_date
    events.loc[idxs, "Effective_Date_Source"] = (
        "verified_official_pdf_section"
    )
    events.loc[idxs, "Date_Override_Reason"] = reason
    events.loc[idxs, "Event_Status"] = (
        "OFFICIAL_LIST_VERIFIED_PENDING_INTERVAL_QA"
    )

# ------------------------------------------------------------
# A. 18 March 2015: separate the two CNX 500 event groups
# ------------------------------------------------------------
url_2015 = (
    "https://www.niftyindices.com/Press_Release/"
    "ind_prs18032015_2.pdf"
)
mask_2015 = (
    events["Published_Date"].eq("2015-03-18")
    & events["Release_Title"].astype(str).str.contains(
        "March 24, 2015 and March 27, 2015",
        case=False, regex=False, na=False
    )
)

rows_2015 = events.loc[mask_2015].copy()
assert len(rows_2015) == 6, (
    f"Expected the six March 27 rows; found {len(rows_2015)}"
)

expected_2015_exits = {"CMC", "MASTEK", "SHASUNPHAR"}
expected_2015_entries = {"MAYURUNIQ", "MUNJALSHOW", "SUVEN"}

actual_2015_exits = set(
    rows_2015.loc[rows_2015["Role"].eq("EXIT"), "Canonical_Symbol"]
)
actual_2015_entries = set(
    rows_2015.loc[rows_2015["Role"].eq("ENTRY"), "Canonical_Symbol"]
)

assert actual_2015_exits == expected_2015_exits, (
    f"Unexpected March 27 exits: {actual_2015_exits}"
)
assert actual_2015_entries == expected_2015_entries, (
    f"Unexpected March 27 entries: {actual_2015_entries}"
)

patch_rows(
    mask_2015,
    "2015-03-27",
    "Official release section B specifies CNX 500 changes effective March 27, 2015.",
    url_2015
)

# Add the separately listed March 24 CNX 500 change.
existing_2015_symbols = set(
    events.loc[mask_2015, "Canonical_Symbol"]
)
assert not {"BALKRISIND", "ITDCEM"}.intersection(existing_2015_symbols)

manual_events = [
    {
        "Published_Date": "2015-03-18",
        "Release_Title": (
            "Change in Indices w.e.f March 24, 2015 and March 27, 2015"
        ),
        "Source_URL": url_2015,
        "Effective_Date": "2015-03-24",
        "Effective_Date_Source": "verified_official_pdf_section",
        "Section_Title": "CNX 500 Index",
        "Role": "EXIT",
        "Action_Heading": (
            "A. Replacements effective March 24, 2015 — company excluded"
        ),
        "Serial_No": 1,
        "Company_Name": "Balkrishna Industries Ltd.",
        "Raw_Symbol": "BALKRISIND",
        "Canonical_Symbol": "BALKRISIND",
        "Event_Status": "OFFICIAL_LIST_VERIFIED_PENDING_INTERVAL_QA",
        "Date_Override_Reason": (
            "Official PDF section A; page 1 lists BALKRISIND as excluded."
        )
    },
    {
        "Published_Date": "2015-03-18",
        "Release_Title": (
            "Change in Indices w.e.f March 24, 2015 and March 27, 2015"
        ),
        "Source_URL": url_2015,
        "Effective_Date": "2015-03-24",
        "Effective_Date_Source": "verified_official_pdf_section",
        "Section_Title": "CNX 500 Index",
        "Role": "ENTRY",
        "Action_Heading": (
            "A. Replacements effective March 24, 2015 — company included"
        ),
        "Serial_No": 1,
        "Company_Name": "ITD Cementation India Ltd.",
        "Raw_Symbol": "ITDCEM",
        "Canonical_Symbol": "ITDCEM",
        "Event_Status": "OFFICIAL_LIST_VERIFIED_PENDING_INTERVAL_QA",
        "Date_Override_Reason": (
            "Official PDF section A; page 1 lists ITDCEM as included."
        )
    }
]

manual_df = pd.DataFrame(manual_events)

# Match the schema and retain any other columns introduced in v3.
for col in events.columns:
    if col not in manual_df.columns:
        manual_df[col] = pd.NA
for col in manual_df.columns:
    if col not in events.columns:
        events[col] = pd.NA

events = pd.concat(
    [events, manual_df[events.columns]],
    ignore_index=True
)

for _, row in manual_df.iterrows():
    corrections.append({
        "Action": "EVENT_ADDED_FROM_OFFICIAL_PDF",
        "Published_Date": row["Published_Date"],
        "Role": row["Role"],
        "Company_Name": row["Company_Name"],
        "Symbol": row["Canonical_Symbol"],
        "Old_Effective_Date": None,
        "New_Effective_Date": row["Effective_Date"],
        "Reason": row["Date_Override_Reason"],
        "Source_URL": row["Source_URL"]
    })

# ------------------------------------------------------------
# B. 7 September 2020: assign the date by the actual PDF section
# ------------------------------------------------------------
url_2020 = (
    "https://www.niftyindices.com/Press_Release/"
    "ind_prs07092020.pdf"
)
mask_2020 = events["Published_Date"].eq("2020-09-07")

rows_2020 = events.loc[mask_2020].copy()
assert len(rows_2020) == 10, (
    f"Expected 10 Nifty 500 events; found {len(rows_2020)}"
)

expected_2020_exits = {
    "ASAHIINDIA", "ARVINDFASN", "KPITTECH",
    "REPCOHOME", "SPANDANA"
}
expected_2020_entries = {
    "JTEKTINDIA", "NOCIL", "PSPPROJECT",
    "SWARAJENG", "TATACOFFEE"
}
actual_2020_exits = set(
    rows_2020.loc[rows_2020["Role"].eq("EXIT"), "Canonical_Symbol"]
)
actual_2020_entries = set(
    rows_2020.loc[rows_2020["Role"].eq("ENTRY"), "Canonical_Symbol"]
)

assert actual_2020_exits == expected_2020_exits, (
    f"Unexpected September 2020 exits: {actual_2020_exits}"
)
assert actual_2020_entries == expected_2020_entries, (
    f"Unexpected September 2020 entries: {actual_2020_entries}"
)

first_group = {"ASAHIINDIA", "JTEKTINDIA"}
mask_2020_first = mask_2020 & events["Canonical_Symbol"].isin(first_group)
mask_2020_second = mask_2020 & ~events["Canonical_Symbol"].isin(first_group)

assert int(mask_2020_first.sum()) == 2
assert int(mask_2020_second.sum()) == 8

patch_rows(
    mask_2020_first,
    "2020-09-14",
    "Official PDF section A: Z-category replacement effective September 14, 2020.",
    url_2020
)
patch_rows(
    mask_2020_second,
    "2020-09-25",
    "Official PDF section D: portfolio-concentration replacement effective September 25, 2020.",
    url_2020
)

# ------------------------------------------------------------
# C. Validate the four targeted effective-date groups
# ------------------------------------------------------------
expected_groups = [
    ("2015-03-18", "2015-03-24", 1, 1),
    ("2015-03-18", "2015-03-27", 3, 3),
    ("2020-09-07", "2020-09-14", 1, 1),
    ("2020-09-07", "2020-09-25", 4, 4),
]

qa_rows = []

for pub_date, effective_date, expected_exits, expected_entries in expected_groups:
    grp = events[
        events["Published_Date"].eq(pub_date)
        & events["Effective_Date"].astype(str).eq(effective_date)
    ]

    exits = grp[grp["Role"].eq("EXIT")]
    entries = grp[grp["Role"].eq("ENTRY")]

    qa_rows.append({
        "Published_Date": pub_date,
        "Effective_Date": effective_date,
        "Parsed_Exits": len(exits),
        "Expected_Exits": expected_exits,
        "Parsed_Entries": len(entries),
        "Expected_Entries": expected_entries,
        "Check": (
            "PASS"
            if len(exits) == expected_exits
            and len(entries) == expected_entries
            else "REVIEW"
        )
    })

qa = pd.DataFrame(qa_rows)
assert qa["Check"].eq("PASS").all(), (
    "Targeted event QA failed; review the table before using v4."
)

# Check targeted symbols for duplicate rows within each release/date/role.
target_pub = {"2015-03-18", "2020-09-07"}
target = events[events["Published_Date"].isin(target_pub)]
duplicates = target.duplicated(
    ["Published_Date", "Effective_Date", "Role", "Canonical_Symbol"],
    keep=False
)
assert not duplicates.any(), (
    "Duplicate event keys found in targeted releases:\n"
    + target.loc[duplicates, [
        "Published_Date", "Effective_Date", "Role", "Canonical_Symbol"
    ]].to_string(index=False)
)

events.to_csv(OUTPUT, index=False)
pd.DataFrame(corrections).to_csv(CORRECTION_LOG, index=False)
qa.to_csv(QA_OUTPUT, index=False)

print("=" * 72)
print("EVENT LEDGER V4 — TARGETED CORRECTIONS")
print("=" * 72)
display(qa)

print("\nManual event/date changes logged:", len(corrections))
print("Candidate event rows in v4:", len(events))
print("Unresolved dates remaining:",
      int(events["Effective_Date"].isna().sum()))
print("\nSaved corrected ledger:", OUTPUT)
print("Saved correction log:", CORRECTION_LOG)
print("Saved targeted QA:", QA_OUTPUT)
print("\nNo membership intervals, prices, or membership flags were modified.")

EVENT LEDGER V4 — TARGETED CORRECTIONS


,Published_Date,Effective_Date,Parsed_Exits,Expected_Exits,Parsed_Entries,Expected_Entries,Check
0,2015-03-18,2015-03-24,1,1,1,1,PASS
1,2015-03-18,2015-03-27,3,3,3,3,PASS
2,2020-09-07,2020-09-14,1,1,1,1,PASS
3,2020-09-07,2020-09-25,4,4,4,4,PASS



Manual event/date changes logged: 18
Candidate event rows in v4: 1486
Unresolved dates remaining: 152

Saved corrected ledger: /content/nifty500_pit_data/nifty500_official_event_candidates_v4.csv
Saved correction log: /content/nifty500_pit_data/nifty500_event_date_corrections_v4.csv
Saved targeted QA: /content/nifty500_pit_data/nifty500_event_date_QA_v4.csv

No membership intervals, prices, or membership flags were modified.


In [43]:
from pathlib import Path
import pandas as pd

BASE = Path("/content/nifty500_pit_data")
EVENTS_PATH = BASE / "nifty500_official_event_candidates_v4.csv"

SUMMARY_OUT = BASE / "nifty500_remaining_event_audit_v1.csv"
DETAIL_OUT = BASE / "nifty500_remaining_event_rows_v1.csv"

assert EVENTS_PATH.exists(), f"Missing ledger: {EVENTS_PATH}"

events = pd.read_csv(EVENTS_PATH, dtype={"Raw_Symbol": str})
events["Published_Date"] = pd.to_datetime(
    events["Published_Date"], errors="coerce"
).dt.strftime("%Y-%m-%d")
events["Role"] = events["Role"].astype(str).str.upper().str.strip()
events["Canonical_Symbol"] = (
    events["Canonical_Symbol"].astype(str).str.upper().str.strip()
)
events["Effective_Date"] = events["Effective_Date"].astype("string")

unresolved = events[
    events["Effective_Date"].isna()
    | events["Effective_Date"].str.strip().eq("")
].copy()

# Proposed dates are based on the effective-date wording seen in the
# cached official release text. We are auditing the lists, not applying
# these dates yet.
proposed_dates = {
    "2018-03-06": "2018-03-13",
    "2020-07-02": "2020-07-31",
    "2021-09-15": "2021-09-30",
    "2021-10-22": "2021-10-29",
    "2022-02-24": "2022-03-31",
    "2023-08-17": "2023-09-29",
    "2025-03-17": "2025-03-21",
}

summary_rows = []

for pub_date, group in unresolved.groupby("Published_Date", dropna=False):
    pub_date = str(pub_date)
    exits = group[group["Role"].eq("EXIT")]
    entries = group[group["Role"].eq("ENTRY")]

    exit_symbols = sorted(exits["Canonical_Symbol"].dropna().unique())
    entry_symbols = sorted(entries["Canonical_Symbol"].dropna().unique())

    duplicate_mask = group.duplicated(
        ["Role", "Canonical_Symbol"], keep=False
    )
    duplicate_rows = group.loc[duplicate_mask]

    summary_rows.append({
        "Published_Date": pub_date,
        "Release_Title": " | ".join(
            group["Release_Title"].dropna().astype(str).unique()
        ),
        "Proposed_Effective_Date": proposed_dates.get(pub_date, ""),
        "Unresolved_Rows": len(group),
        "Exit_Rows": len(exits),
        "Unique_Exit_Symbols": len(exit_symbols),
        "Exit_Symbols": ", ".join(exit_symbols),
        "Entry_Rows": len(entries),
        "Unique_Entry_Symbols": len(entry_symbols),
        "Entry_Symbols": ", ".join(entry_symbols),
        "Duplicate_Role_Symbol_Rows": len(duplicate_rows),
        "Source_URL": " | ".join(
            group["Source_URL"].dropna().astype(str).unique()
        )
    })

summary = pd.DataFrame(summary_rows).sort_values("Published_Date")

print("=" * 75)
print("REMAINING EVENT AUDIT — READ ONLY")
print("=" * 75)
print("Unresolved event rows:", len(unresolved))
print("Distinct unresolved releases:", unresolved["Published_Date"].nunique())

print("\nRELEASE-BY-RELEASE SYMBOL LISTS")
display(summary)

print("\nDUPLICATE ROLE/SYMBOL ROWS")
duplicate_mask = unresolved.duplicated(
    ["Published_Date", "Role", "Canonical_Symbol"], keep=False
)
duplicates = unresolved.loc[duplicate_mask].sort_values(
    ["Published_Date", "Role", "Canonical_Symbol"]
)

if duplicates.empty:
    print("No duplicate event rows found within release and role.")
else:
    display(duplicates[[
        "Published_Date", "Release_Title", "Role",
        "Company_Name", "Raw_Symbol", "Canonical_Symbol",
        "Source_URL"
    ]])

summary.to_csv(SUMMARY_OUT, index=False)
unresolved.to_csv(DETAIL_OUT, index=False)

print("\nSaved summary:", SUMMARY_OUT)
print("Saved unresolved event rows:", DETAIL_OUT)
print("No effective dates, membership intervals, prices, or flags changed.")

REMAINING EVENT AUDIT — READ ONLY
Unresolved event rows: 152
Distinct unresolved releases: 7

RELEASE-BY-RELEASE SYMBOL LISTS


,Published_Date,Release_Title,Proposed_Effective_Date,Unresolved_Rows,Exit_Rows,Unique_Exit_Symbols,Exit_Symbols,Entry_Rows,Unique_Entry_Symbols,Entry_Symbols,Duplicate_Role_Symbol_Rows,Source_URL
0,2018-03-06,Replacements in Indices,2018-03-13,2,1,1,VIDEOIND,1,1,ERIS,0,https://www.niftyindices.com/Press_Release/ind...
1,2020-07-02,Replacement in Indices,2020-07-31,2,1,1,VEDL,1,1,SBICARD,0,https://www.niftyindices.com/Press_Release/ind...
2,2021-09-15,Replacements in indices and revision in criteria,2021-09-30,44,22,22,"AKZOINDIA, BALMLAWRIE, BLISSGVS, DCAL, DISHTV,...",22,22,"ANURAS, CGPOWER, EQUITASBNK, FACT, HATHWAY, HG...",0,https://www.niftyindices.com/Press_Release/ind...
3,2021-10-22,Revision in index maintenance guidelines and r...,2021-10-29,2,1,1,GMRINFRA,1,1,SONACOMS,0,https://www.niftyindices.com/Press_Release/ind...
4,2022-02-24,Revision in criteria and replacements in indices,2022-03-31,64,32,32,"ALEMBICLTD, ASHOKA, BAJAJCON, BHARATRAS, CGCL,...",32,32,"ABSLAMC, ALLCARGO, APTUS, BCG, BORORENEW, CHEM...",0,https://www.niftyindices.com/Press_Release/ind...
5,2023-08-17,Replacements in indices and revision in criteria,2023-09-29,36,18,18,"BASF, GARFIBRES, GODREJAGRO, GREENPANEL, HGS, ...",18,18,"ALLCARGO, ALOKINDS, GILLETTE, GLS, GPIL, IRCON...",0,https://www.niftyindices.com/Press_Release/ind...
6,2025-03-17,Replacement in indices,2025-03-21,2,1,1,ISEC,1,1,RAYMONDLSL,0,https://www.niftyindices.com/Press_Release/ind...



DUPLICATE ROLE/SYMBOL ROWS
No duplicate event rows found within release and role.

Saved summary: /content/nifty500_pit_data/nifty500_remaining_event_audit_v1.csv
Saved unresolved event rows: /content/nifty500_pit_data/nifty500_remaining_event_rows_v1.csv
No effective dates, membership intervals, prices, or flags changed.


In [44]:
from pathlib import Path
import pandas as pd

BASE = Path("/content/nifty500_pit_data")

SOURCE = BASE / "nifty500_official_event_candidates_v4.csv"
OUTPUT = BASE / "nifty500_official_event_candidates_v5.csv"
LOG_PATH = BASE / "nifty500_event_date_corrections_v5.csv"
QA_PATH = BASE / "nifty500_event_date_QA_v5.csv"

assert SOURCE.exists(), f"Missing source ledger: {SOURCE}"

events = pd.read_csv(SOURCE, dtype={"Raw_Symbol": str},
                     keep_default_na=False)

required = {
    "Published_Date", "Release_Title", "Source_URL",
    "Effective_Date", "Effective_Date_Source",
    "Role", "Company_Name", "Canonical_Symbol", "Event_Status"
}
assert required.issubset(events.columns), (
    f"Missing required columns: {required - set(events.columns)}"
)

events["Published_Date"] = pd.to_datetime(
    events["Published_Date"], errors="coerce"
).dt.strftime("%Y-%m-%d")
events["Role"] = events["Role"].astype(str).str.upper().str.strip()
events["Canonical_Symbol"] = (
    events["Canonical_Symbol"].astype(str).str.upper().str.strip()
)
events["Effective_Date"] = events["Effective_Date"].astype(str).str.strip()

if "Date_Override_Reason" not in events.columns:
    events["Date_Override_Reason"] = ""

# Publication date, source filename, effective date, expected exits,
# expected entries, and explanation from the inspected official release.
SPECS = [
    {
        "published": "2018-03-06",
        "source_file": "ind_prs06032018.pdf",
        "effective": "2018-03-13",
        "exits": 1,
        "entries": 1,
        "reason": (
            "Official release states that the Videocon Industries "
            "replacement is effective March 13, 2018."
        )
    },
    {
        "published": "2020-07-02",
        "source_file": "ind_prs02072020_1.pdf",
        "effective": "2020-07-31",
        "exits": 1,
        "entries": 1,
        "reason": (
            "Official release states that the Vedanta replacement "
            "is effective July 31, 2020."
        )
    },
    {
        "published": "2021-09-15",
        "source_file": "ind_prs15092021.pdf",
        "effective": "2021-09-30",
        "exits": 22,
        "entries": 22,
        "reason": (
            "Official release states that the revised replacement "
            "list is effective September 30, 2021; the earlier list "
            "is superseded."
        )
    },
    {
        "published": "2021-10-22",
        "source_file": "ind_prs22102021.pdf",
        "effective": "2021-10-29",
        "exits": 1,
        "entries": 1,
        "reason": (
            "Official release states that the GMR Infrastructure "
            "replacement is effective October 29, 2021."
        )
    },
    {
        "published": "2022-02-24",
        "source_file": "ind_prs24022022_1.pdf",
        "effective": "2022-03-31",
        "exits": 32,
        "entries": 32,
        "reason": (
            "Official release states that the periodic-review "
            "changes are effective March 31, 2022."
        )
    },
    {
        "published": "2023-08-17",
        "source_file": "ind_prs17082023.pdf",
        "effective": "2023-09-29",
        "exits": 18,
        "entries": 18,
        "reason": (
            "Official release states that the semi-annual review "
            "changes are effective September 29, 2023."
        )
    },
    {
        "published": "2025-03-17",
        "source_file": "ind_prs17032025.pdf",
        "effective": "2025-03-21",
        "exits": 1,
        "entries": 1,
        "reason": (
            "Official release states that the ICICI Securities "
            "replacement is effective March 21, 2025."
        )
    }
]

# Keep source ledger as-is until every precondition has been checked.
all_target_indices = []
qa_rows = []

for spec in SPECS:
    release_mask = events["Published_Date"].eq(spec["published"])
    release = events[release_mask].copy()

    assert not release.empty, (
        f"No events found for publication date {spec['published']}"
    )

    # Ensure the rows are from the expected official PDF.
    assert release["Source_URL"].astype(str).str.contains(
        spec["source_file"], regex=False, na=False
    ).all(), (
        f"Unexpected source URL for release {spec['published']}"
    )

    unresolved_mask = (
        release["Effective_Date"].eq("")
        | release["Effective_Date"].isna()
    )
    target = release[unresolved_mask].copy()

    expected_rows = spec["exits"] + spec["entries"]
    assert len(target) == expected_rows, (
        f"{spec['published']}: expected {expected_rows} unresolved "
        f"rows; found {len(target)}"
    )

    exit_rows = target[target["Role"].eq("EXIT")]
    entry_rows = target[target["Role"].eq("ENTRY")]

    assert len(exit_rows) == spec["exits"], (
        f"{spec['published']}: exit count mismatch"
    )
    assert len(entry_rows) == spec["entries"], (
        f"{spec['published']}: entry count mismatch"
    )

    for role, group in [("EXIT", exit_rows), ("ENTRY", entry_rows)]:
        assert group["Canonical_Symbol"].ne("").all(), (
            f"{spec['published']}: blank ticker in {role} list"
        )
        assert group["Canonical_Symbol"].is_unique, (
            f"{spec['published']}: duplicate ticker in {role} list"
        )

    all_target_indices.extend(target.index.tolist())

    qa_rows.append({
        "Published_Date": spec["published"],
        "Effective_Date": spec["effective"],
        "Expected_Exits": spec["exits"],
        "Verified_Exits": len(exit_rows),
        "Expected_Entries": spec["entries"],
        "Verified_Entries": len(entry_rows),
        "Source_File_Check": "PASS",
        "Unique_Symbol_Check": "PASS",
        "Preflight": "PASS"
    })

# Apply only after all seven releases have passed preflight.
correction_log = []

for spec in SPECS:
    release_mask = events["Published_Date"].eq(spec["published"])
    unresolved_mask = (
        events["Effective_Date"].eq("")
        | events["Effective_Date"].isna()
    )
    target_mask = release_mask & unresolved_mask

    for idx in events.index[target_mask]:
        row = events.loc[idx]

        correction_log.append({
            "Original_Row_Index": idx,
            "Published_Date": spec["published"],
            "Release_Title": row["Release_Title"],
            "Role": row["Role"],
            "Company_Name": row["Company_Name"],
            "Symbol": row["Canonical_Symbol"],
            "Old_Effective_Date": "",
            "New_Effective_Date": spec["effective"],
            "Reason": spec["reason"],
            "Source_URL": row["Source_URL"]
        })

    events.loc[target_mask, "Effective_Date"] = spec["effective"]
    events.loc[target_mask, "Effective_Date_Source"] = (
        "verified_official_release_wording"
    )
    events.loc[target_mask, "Date_Override_Reason"] = spec["reason"]
    events.loc[target_mask, "Event_Status"] = (
        "DATE_VERIFIED_PENDING_MEMBERSHIP_QA"
    )

# Verify all intended date assignments, including the final role counts.
for spec in SPECS:
    mask = (
        events["Published_Date"].eq(spec["published"])
        & events["Effective_Date"].eq(spec["effective"])
        & events["Event_Status"].eq("DATE_VERIFIED_PENDING_MEMBERSHIP_QA")
    )
    corrected = events[mask]

    assert len(corrected) == spec["exits"] + spec["entries"]
    assert int(corrected["Role"].eq("EXIT").sum()) == spec["exits"]
    assert int(corrected["Role"].eq("ENTRY").sum()) == spec["entries"]

# No unresolved candidate dates should remain after these seven overrides.
remaining_unresolved = events[
    events["Effective_Date"].eq("") | events["Effective_Date"].isna()
]
assert remaining_unresolved.empty, (
    f"{len(remaining_unresolved)} unresolved event dates remain; "
    "review before promoting this ledger."
)

qa = pd.DataFrame(qa_rows)
qa["Postflight"] = "PASS"

assert qa["Preflight"].eq("PASS").all()

# Save new versioned outputs; never overwrite v4.
events.to_csv(OUTPUT, index=False)
pd.DataFrame(correction_log).to_csv(LOG_PATH, index=False)
qa.to_csv(QA_PATH, index=False)

print("=" * 72)
print("EVENT LEDGER V5 — DATE RESOLUTION")
print("=" * 72)
display(qa)

print("\nRows corrected:", len(correction_log))
print("Total event rows:", len(events))
print("Unresolved effective dates remaining:", len(remaining_unresolved))
print("\nSaved ledger:", OUTPUT)
print("Saved correction log:", LOG_PATH)
print("Saved QA:", QA_PATH)
print("\nNo membership intervals, prices, or membership flags were modified.")

EVENT LEDGER V5 — DATE RESOLUTION


,Published_Date,Effective_Date,Expected_Exits,Verified_Exits,Expected_Entries,Verified_Entries,Source_File_Check,Unique_Symbol_Check,Preflight,Postflight
0,2018-03-06,2018-03-13,1,1,1,1,PASS,PASS,PASS,PASS
1,2020-07-02,2020-07-31,1,1,1,1,PASS,PASS,PASS,PASS
2,2021-09-15,2021-09-30,22,22,22,22,PASS,PASS,PASS,PASS
3,2021-10-22,2021-10-29,1,1,1,1,PASS,PASS,PASS,PASS
4,2022-02-24,2022-03-31,32,32,32,32,PASS,PASS,PASS,PASS
5,2023-08-17,2023-09-29,18,18,18,18,PASS,PASS,PASS,PASS
6,2025-03-17,2025-03-21,1,1,1,1,PASS,PASS,PASS,PASS



Rows corrected: 152
Total event rows: 1486
Unresolved effective dates remaining: 0

Saved ledger: /content/nifty500_pit_data/nifty500_official_event_candidates_v5.csv
Saved correction log: /content/nifty500_pit_data/nifty500_event_date_corrections_v5.csv
Saved QA: /content/nifty500_pit_data/nifty500_event_date_QA_v5.csv

No membership intervals, prices, or membership flags were modified.


In [45]:
from pathlib import Path
import pandas as pd

BASE = Path("/content/nifty500_pit_data")
INPUT = BASE / "nifty500_official_event_candidates_v5.csv"

assert INPUT.exists(), f"Missing ledger: {INPUT}"

events = pd.read_csv(INPUT, dtype=str, keep_default_na=False)

required = {
    "Published_Date", "Effective_Date", "Source_URL",
    "Release_Title", "Role", "Company_Name", "Canonical_Symbol"
}
assert required.issubset(events.columns), (
    f"Missing columns: {required - set(events.columns)}"
)

events["Published_dt"] = pd.to_datetime(
    events["Published_Date"], errors="coerce"
)
events["Effective_dt"] = pd.to_datetime(
    events["Effective_Date"], errors="coerce"
)
events["Role_norm"] = events["Role"].str.upper().str.strip()
events["Symbol_norm"] = (
    events["Canonical_Symbol"].str.upper().str.strip()
)

# Use only rows with enough key information for chronology analysis.
valid_key = (
    events["Published_dt"].notna()
    & events["Effective_dt"].notna()
    & events["Symbol_norm"].ne("")
    & events["Role_norm"].isin(["ENTRY", "EXIT"])
)

print("=" * 72)
print("EVENT LEDGER V5 — CONSISTENCY AUDIT")
print("=" * 72)
print("Total rows:", len(events))
print("Valid event keys:", int(valid_key.sum()))
print("Invalid/missing key rows:", int((~valid_key).sum()))
print("Rows with unresolved effective dates:",
      int(events["Effective_dt"].isna().sum()))
print("Rows effective before publication:",
      int((events["Effective_dt"] < events["Published_dt"]).sum()))

# 1. Duplicate same-symbol, same-role events on the same effective date.
duplicate_key = ["Effective_dt", "Role_norm", "Symbol_norm"]
dup_counts = (
    events.loc[valid_key]
    .groupby(duplicate_key, dropna=False)
    .agg(
        Row_Count=("Symbol_norm", "size"),
        Distinct_Releases=("Source_URL", "nunique"),
        Release_Titles=("Release_Title",
                        lambda s: " | ".join(sorted(set(s)))),
        Companies=("Company_Name",
                   lambda s: " | ".join(sorted(set(s))))
    )
    .reset_index()
)

duplicate_groups = dup_counts[dup_counts["Row_Count"] > 1].copy()

dup_rows = events.merge(
    duplicate_groups[duplicate_key],
    on=duplicate_key,
    how="inner"
).sort_values(duplicate_key)

# 2. Same symbol both entering and exiting on the same effective date.
role_conflict_groups = (
    events.loc[valid_key]
    .groupby(["Effective_dt", "Symbol_norm"], dropna=False)
    .agg(
        Roles=("Role_norm", lambda s: ", ".join(sorted(set(s)))),
        Role_Count=("Role_norm", "nunique"),
        Row_Count=("Role_norm", "size"),
        Release_Titles=("Release_Title",
                        lambda s: " | ".join(sorted(set(s))))
    )
    .reset_index()
)

role_conflicts = role_conflict_groups[
    role_conflict_groups["Role_Count"] > 1
].copy()

# 3. Events whose effective date precedes the publication date.
# Usually suspicious, but kept as an audit exception rather than assumed wrong.
before_publication = events[
    valid_key & (events["Effective_dt"] < events["Published_dt"])
].copy()

# 4. Consecutive identical membership actions for a symbol on different dates.
# Example: ENTRY, then ENTRY again with no EXIT between. This can reveal a
# missing event or a superseded/revoked release. It's a review flag, not proof.
unique_events = (
    events.loc[valid_key, [
        "Effective_dt", "Role_norm", "Symbol_norm",
        "Published_Date", "Effective_Date", "Release_Title",
        "Source_URL", "Company_Name"
    ]]
    .drop_duplicates(["Effective_dt", "Role_norm", "Symbol_norm"])
    .sort_values(["Symbol_norm", "Effective_dt", "Role_norm"])
)

repeat_rows = []

for symbol, group in unique_events.groupby("Symbol_norm", sort=False):
    group = group.sort_values("Effective_dt").reset_index(drop=True)

    for i in range(1, len(group)):
        prev = group.iloc[i - 1]
        cur = group.iloc[i]

        if (
            prev["Role_norm"] == cur["Role_norm"]
            and prev["Effective_dt"] < cur["Effective_dt"]
        ):
            repeat_rows.append({
                "Symbol": symbol,
                "Previous_Date": prev["Effective_Date"],
                "Previous_Role": prev["Role_norm"],
                "Previous_Release": prev["Release_Title"],
                "Current_Date": cur["Effective_Date"],
                "Current_Role": cur["Role_norm"],
                "Current_Release": cur["Release_Title"],
                "Issue": "CONSECUTIVE_SAME_ROLE_WITHOUT_OPPOSITE_EVENT"
            })

repeat_actions = pd.DataFrame(repeat_rows)

# 5. Release-level summary, to check event counts and missing dates.
release_summary = (
    events.groupby(
        ["Published_Date", "Effective_Date", "Release_Title", "Source_URL"],
        dropna=False
    )
    .agg(
        Total_Rows=("Symbol_norm", "size"),
        Exit_Rows=("Role_norm", lambda s: int((s == "EXIT").sum())),
        Entry_Rows=("Role_norm", lambda s: int((s == "ENTRY").sum())),
        Unique_Symbols=("Symbol_norm", "nunique"),
        Blank_Symbols=("Symbol_norm", lambda s: int((s == "").sum()))
    )
    .reset_index()
    .sort_values(["Effective_Date", "Published_Date"])
)

# 6. Event counts by effective date, including any exit/entry imbalance.
daily_summary = (
    events.loc[valid_key]
    .pivot_table(
        index="Effective_Date",
        columns="Role_norm",
        values="Symbol_norm",
        aggfunc="size",
        fill_value=0
    )
    .reset_index()
)

for col in ["ENTRY", "EXIT"]:
    if col not in daily_summary.columns:
        daily_summary[col] = 0

daily_summary["Net_Entries_Minus_Exits"] = (
    daily_summary["ENTRY"] - daily_summary["EXIT"]
)

# Persist reports separately; do not overwrite v5.
paths = {
    "duplicates": BASE / "nifty500_event_duplicate_groups_v5.csv",
    "duplicate_rows": BASE / "nifty500_event_duplicate_rows_v5.csv",
    "role_conflicts": BASE / "nifty500_event_same_date_role_conflicts_v5.csv",
    "before_publication": BASE / "nifty500_event_before_publication_v5.csv",
    "repeated_actions": BASE / "nifty500_event_consecutive_actions_v5.csv",
    "release_summary": BASE / "nifty500_event_release_integrity_v5.csv",
    "daily_summary": BASE / "nifty500_event_daily_counts_v5.csv",
}

duplicate_groups.to_csv(paths["duplicates"], index=False)
dup_rows.to_csv(paths["duplicate_rows"], index=False)
role_conflicts.to_csv(paths["role_conflicts"], index=False)
before_publication.to_csv(paths["before_publication"], index=False)
repeat_actions.to_csv(paths["repeated_actions"], index=False)
release_summary.to_csv(paths["release_summary"], index=False)
daily_summary.to_csv(paths["daily_summary"], index=False)

print("\n" + "=" * 72)
print("EXCEPTION COUNTS")
print("=" * 72)
print("Duplicate (date, role, symbol) groups:", len(duplicate_groups))
print("Rows in duplicate groups:", len(dup_rows))
print("Same-symbol same-date role conflicts:", len(role_conflicts))
print("Events effective before publication:", len(before_publication))
print("Consecutive same-role symbol transitions:", len(repeat_actions))

print("\nDUPLICATE EVENT GROUPS — FIRST 30")
display(duplicate_groups.head(30))

print("\nSAME-DATE ENTRY/EXIT CONFLICTS")
display(role_conflicts.head(30))

print("\nCONSECUTIVE SAME-ROLE TRANSITIONS — FIRST 30")
display(repeat_actions.head(30))

print("\nEVENT COUNTS BY EFFECTIVE DATE — FIRST 30")
display(daily_summary.head(30))

print("\nRELEASE SUMMARY — LATEST 25")
display(release_summary.tail(25))

print("\nReports saved:")
for name, path in paths.items():
    print(f"{name}: {path}")

print("\nRead-only audit complete. No membership intervals, prices, or flags changed.")

EVENT LEDGER V5 — CONSISTENCY AUDIT
Total rows: 1486
Valid event keys: 1486
Invalid/missing key rows: 0
Rows with unresolved effective dates: 0
Rows effective before publication: 0

EXCEPTION COUNTS
Duplicate (date, role, symbol) groups: 1
Rows in duplicate groups: 2
Same-symbol same-date role conflicts: 2
Events effective before publication: 0
Consecutive same-role symbol transitions: 56

DUPLICATE EVENT GROUPS — FIRST 30


,Effective_dt,Role_norm,Symbol_norm,Row_Count,Distinct_Releases,Release_Titles,Companies
65,2015-09-28,EXIT,CROMPGREAV,2,1,"Change in Indices w.e.f September 28, 2015",Crompton Greaves Ltd.



SAME-DATE ENTRY/EXIT CONFLICTS


,Effective_dt,Symbol_norm,Roles,Role_Count,Row_Count,Release_Titles
3,2015-03-27,ADANIPORTS,"ENTRY, EXIT",2,2,"Change in indices w.e.f March 27, 2015."
714,2020-03-27,SHILPAMED,"ENTRY, EXIT",2,2,"Replacement in indices w.e.f. March 27, 2020 |..."



CONSECUTIVE SAME-ROLE TRANSITIONS — FIRST 30


...


EVENT COUNTS BY EFFECTIVE DATE — FIRST 30


Role_norm,Effective_Date,ENTRY,EXIT,Net_Entries_Minus_Exits
0,2015-03-24,1,1,0
1,2015-03-27,19,19,0
2,2015-09-28,22,23,-1
3,2015-10-19,1,1,0
4,2015-12-14,2,2,0
5,2016-01-18,1,1,0
6,2016-04-01,76,75,1
7,2016-04-29,3,3,0
8,2016-09-30,25,25,0
9,2016-11-15,1,1,0



RELEASE SUMMARY — LATEST 25


,Published_Date,Effective_Date,Release_Title,Source_URL,Total_Rows,Exit_Rows,Entry_Rows,Unique_Symbols,Blank_Symbols
60,2022-07-11,2022-08-08,"Replacements in indices w.e.f. August 08, 2022",https://www.niftyindices.com/Press_Release/ind...,6,3,3,6,0
61,2022-09-01,2022-09-30,"Replacements in indices w.e.f. September 30, 2022",https://www.niftyindices.com/Press_Release/ind...,36,18,18,36,0
62,2022-09-16,2022-09-30,"Replacements in indices w.e.f. September 30, 2022",https://www.niftyindices.com/Press_Release/ind...,2,1,1,2,0
63,2022-12-06,2022-12-30,"Replacements in indices w.e.f. December 30, 2022",https://www.niftyindices.com/Press_Release/ind...,6,3,3,6,0
64,2022-12-22,2022-12-30,"Replacements in indices w.e.f December 30, 2022",https://www.niftyindices.com/Press_Release/ind...,2,1,1,2,0
65,2023-02-09,2023-02-17,"Replacements in indices w.e.f. February 17, 2023",https://www.niftyindices.com/Press_Release/ind...,2,1,1,2,0
66,2023-02-17,2023-03-31,"Replacements in indices w.e.f March 31, 2023",https://www.niftyindices.com/Press_Release/ind...,40,20,20,40,0
67,2023-04-19,2023-04-28,"Replacements in indices w.e.f. April 28, 2023",https://www.niftyindices.com/Press_Release/ind...,4,2,2,4,0
68,2023-07-04,2023-07-13,"Replacement in indices w.e.f. July 13, 2023",https://www.niftyindices.com/Press_Release/ind...,2,1,1,2,0
69,2023-08-17,2023-09-29,Replacements in indices and revision in criteria,https://www.niftyindices.com/Press_Release/ind...,36,18,18,36,0



Reports saved:
duplicates: /content/nifty500_pit_data/nifty500_event_duplicate_groups_v5.csv
duplicate_rows: /content/nifty500_pit_data/nifty500_event_duplicate_rows_v5.csv
role_conflicts: /content/nifty500_pit_data/nifty500_event_same_date_role_conflicts_v5.csv
before_publication: /content/nifty500_pit_data/nifty500_event_before_publication_v5.csv
repeated_actions: /content/nifty500_pit_data/nifty500_event_consecutive_actions_v5.csv
release_summary: /content/nifty500_pit_data/nifty500_event_release_integrity_v5.csv
daily_summary: /content/nifty500_pit_data/nifty500_event_daily_counts_v5.csv

Read-only audit complete. No membership intervals, prices, or flags changed.


In [46]:
from pathlib import Path
import re
import pandas as pd

BASE = Path("/content/nifty500_pit_data")
EVENTS_PATH = BASE / "nifty500_official_event_candidates_v5.csv"
SCAN_PATH = BASE / "nifty500_official_pdf_scan_v1.csv"
REPEAT_PATH = BASE / "nifty500_event_consecutive_actions_v5.csv"

for p in [EVENTS_PATH, SCAN_PATH, REPEAT_PATH]:
    assert p.exists(), f"Missing required file: {p}"

events = pd.read_csv(EVENTS_PATH, dtype=str, keep_default_na=False)
scan = pd.read_csv(SCAN_PATH, dtype=str, keep_default_na=False)
repeats = pd.read_csv(REPEAT_PATH, dtype=str, keep_default_na=False)

events["Role"] = events["Role"].str.upper().str.strip()
events["Effective_Date"] = events["Effective_Date"].str.strip()
events["Canonical_Symbol"] = (
    events["Canonical_Symbol"].str.upper().str.strip()
)

# Three known anomaly groups from the consistency audit.
cases = [
    ("DUPLICATE", "CROMPGREAV", "2015-09-28"),
    ("SAME_DATE_ROLE_CONFLICT", "ADANIPORTS", "2015-03-27"),
    ("SAME_DATE_ROLE_CONFLICT", "SHILPAMED", "2020-03-27"),
]

source_map = (
    scan.drop_duplicates("URL")
    .set_index("URL")["Text_Cache"]
    .to_dict()
)

print("=" * 78)
print("1. FULL EVENT ROWS FOR THE THREE ANOMALY GROUPS")
print("=" * 78)

event_cols = [
    c for c in [
        "Published_Date", "Effective_Date", "Role", "Company_Name",
        "Raw_Symbol", "Canonical_Symbol", "Release_Title",
        "Section_Title", "Action_Heading", "Effective_Date_Source",
        "Date_Override_Reason", "Source_URL"
    ]
    if c in events.columns
]

case_urls = []

for issue, symbol, date in cases:
    group = events[
        events["Canonical_Symbol"].eq(symbol)
        & events["Effective_Date"].eq(date)
    ].copy()

    print(f"\n--- {issue}: {symbol} on {date} ---")

    if group.empty:
        print("No matching event rows in v5.")
        continue

    display(group[event_cols])

    for url in group["Source_URL"].unique():
        case_urls.append((issue, symbol, date, url))

print("\n" + "=" * 78)
print("2. SOURCE PDF TEXT AROUND EACH ANOMALOUS TICKER")
print("=" * 78)

seen = set()

for issue, symbol, date, url in case_urls:
    key = (symbol, url)
    if key in seen:
        continue
    seen.add(key)

    text_path = source_map.get(url)
    print(f"\n--- {symbol} | {date} | {issue} ---")
    print("Source URL:", url)

    if not text_path or not Path(text_path).exists():
        print("Cached source text not found for this URL.")
        continue

    text = Path(text_path).read_text(
        encoding="utf-8", errors="replace"
    )
    parts = re.split(r"\n--- PAGE (\d+) ---\n", text)
    hits = []

    for i in range(1, len(parts), 2):
        page_no = int(parts[i])
        page_text = parts[i + 1] if i + 1 < len(parts) else ""
        lines = page_text.splitlines()

        for line_idx, line in enumerate(lines):
            if symbol.casefold() in line.casefold():
                start = max(0, line_idx - 5)
                end = min(len(lines), line_idx + 6)
                hits.append((page_no, line_idx + 1, lines[start:end]))

    if not hits:
        print("Ticker text not found in cached PDF extraction.")
        continue

    for page_no, line_no, context in hits[:8]:
        print(f"\nPage {page_no}, line {line_no}:")
        for line in context:
            print("  ", line)

    if len(hits) > 8:
        print(f"... {len(hits) - 8} additional occurrences omitted.")

print("\n" + "=" * 78)
print("3. REPEATED-ACTION FLAG SUMMARY")
print("=" * 78)

if repeats.empty:
    print("No consecutive same-role transitions recorded.")
else:
    repeats["Current_Year"] = pd.to_datetime(
        repeats["Current_Date"], errors="coerce"
    ).dt.year

    print("Repeated-action flags:", len(repeats))

    print("\nBy current event year and role:")
    display(
        repeats.groupby(
            ["Current_Year", "Current_Role"], dropna=False
        ).size().reset_index(name="Count")
        .sort_values(["Current_Year", "Current_Role"])
    )

    print("\nAll repeated-action flags (56 or fewer):")
    show_cols = [
        c for c in [
            "Symbol", "Previous_Date", "Previous_Role",
            "Previous_Release", "Current_Date", "Current_Role",
            "Current_Release", "Issue"
        ]
        if c in repeats.columns
    ]
    display(repeats[show_cols].sort_values(
        ["Symbol", "Previous_Date", "Current_Date"]
    ))

print("\nRead-only trace complete.")
print("No event ledger, interval, price row, or membership flag was changed.")

1. FULL EVENT ROWS FOR THE THREE ANOMALY GROUPS

--- DUPLICATE: CROMPGREAV on 2015-09-28 ---


,Published_Date,Effective_Date,Role,Company_Name,Raw_Symbol,Canonical_Symbol,Release_Title,Section_Title,Action_Heading,Effective_Date_Source,Date_Override_Reason,Source_URL
77,2015-08-24,2015-09-28,EXIT,Crompton Greaves Ltd.,CROMPGREAV,CROMPGREAV,"Change in Indices w.e.f September 28, 2015",CNX 500 Index,The following companies are being excluded,title_single_date,,https://www.niftyindices.com/Press_Release/ind...
81,2015-08-24,2015-09-28,EXIT,Crompton Greaves Ltd.,CROMPGREAV,CROMPGREAV,"Change in Indices w.e.f September 28, 2015",CNX 500 Index,The following company is being excluded,title_single_date,,https://www.niftyindices.com/Press_Release/ind...



--- SAME_DATE_ROLE_CONFLICT: ADANIPORTS on 2015-03-27 ---


,Published_Date,Effective_Date,Role,Company_Name,Raw_Symbol,Canonical_Symbol,Release_Title,Section_Title,Action_Heading,Effective_Date_Source,Date_Override_Reason,Source_URL
26,2015-02-20,2015-03-27,ENTRY,Adani Ports and Special Economic Zone Ltd.,ADANIPORTS,ADANIPORTS,"Change in indices w.e.f March 27, 2015.",CNX 500 Index,The following company is being included,title_single_date,,https://www.niftyindices.com/Press_Release/ind...
29,2015-02-20,2015-03-27,EXIT,Adani Ports and Special Economic Zone Ltd.,ADANIPORTS,ADANIPORTS,"Change in indices w.e.f March 27, 2015.",CNX 500 Index,The following company is being excluded,title_single_date,,https://www.niftyindices.com/Press_Release/ind...



--- SAME_DATE_ROLE_CONFLICT: SHILPAMED on 2020-03-27 ---


...


2. SOURCE PDF TEXT AROUND EACH ANOMALOUS TICKER

--- CROMPGREAV | 2015-09-28 | DUPLICATE ---
Source URL: https://www.niftyindices.com/Press_Release/ind_prs24082015.pdf

Page 1, line 27:
    
   Sr. No. Company Name 
   Symbol 
   1 
   Crompton Greaves Ltd. 
   CROMPGREAV 
    
   The following company is being included: 
    
   Sr. No. Company Name 
   Symbol 

Page 1, line 47:
    
   Sr. No. Company Name 
   Symbol 
   1 
   Crompton Greaves Ltd. 
   CROMPGREAV 
    
   The following company is being included: 
    
   Sr. No. Company Name 
   Symbol 

Page 1, line 69:
    
   Sr. No. Company Name 
   Symbol 
     1 
   Crompton Greaves Ltd. 
   CROMPGREAV 
   2 
   Max India Ltd. 
   MAX 
    
   

Page 2, line 27:
    
   Sr. No. Company Name 
   Symbol 
   1 
   Crompton Greaves Ltd. 
   CROMPGREAV 
   2 
   Max India Ltd. 
   MAX 
    
   The following companies are being included: 

Page 2, line 53:
    
   Sr. No. Company Name 
   Symbol 
   1 
   Crompton Greaves Ltd. 
   C

,Current_Year,Current_Role,Count
0,2017,ENTRY,4
1,2017,EXIT,4
2,2018,ENTRY,1
3,2018,EXIT,1
4,2020,ENTRY,15
5,2020,EXIT,21
6,2021,ENTRY,2
7,2021,EXIT,1
8,2022,ENTRY,2
9,2022,EXIT,3



All repeated-action flags (56 or fewer):


,Symbol,Previous_Date,Previous_Role,Previous_Release,Current_Date,Current_Role,Current_Release,Issue
0,AARTIIND,2019-03-29,EXIT,"Change in Indices w.e.f. March 29, 2019",2022-05-04,EXIT,"Replacements in indices w.e.f. May 04, 2022",CONSECUTIVE_SAME_ROLE_WITHOUT_OPPOSITE_EVENT
1,ASAHIINDIA,2018-09-28,EXIT,"Change in Indices w.e.f. September 28, 2018",2020-09-14,EXIT,Replacements in Indices,CONSECUTIVE_SAME_ROLE_WITHOUT_OPPOSITE_EVENT
2,BHARATRAS,2020-03-27,ENTRY,Revision in criteria and replacements in Indices,2020-06-26,ENTRY,"Change in indices w.e.f. June 26, 2020",CONSECUTIVE_SAME_ROLE_WITHOUT_OPPOSITE_EVENT
3,BLISSGVS,2020-03-27,EXIT,Revision in criteria and replacements in Indices,2021-09-30,EXIT,Replacements in indices and revision in criteria,CONSECUTIVE_SAME_ROLE_WITHOUT_OPPOSITE_EVENT
4,CENTURYTEX,2020-03-27,ENTRY,Revision in criteria and replacements in Indices,2020-06-26,ENTRY,"Change in indices w.e.f. June 26, 2020",CONSECUTIVE_SAME_ROLE_WITHOUT_OPPOSITE_EVENT
5,DHFL,2020-03-27,EXIT,Revision in criteria and replacements in Indices,2020-06-26,EXIT,"Change in indices w.e.f. June 26, 2020",CONSECUTIVE_SAME_ROLE_WITHOUT_OPPOSITE_EVENT
6,ERIS,2018-03-13,ENTRY,Replacements in Indices,2018-04-02,ENTRY,"Change in Indices w.e.f. April 02, 2018",CONSECUTIVE_SAME_ROLE_WITHOUT_OPPOSITE_EVENT
7,FLUOROCHEM,2020-03-27,ENTRY,Revision in criteria and replacements in Indices,2020-06-26,ENTRY,"Change in indices w.e.f. June 26, 2020",CONSECUTIVE_SAME_ROLE_WITHOUT_OPPOSITE_EVENT
8,FRETAIL,2017-03-16,ENTRY,Replacements in Indices.,2017-03-31,ENTRY,"Change in Indices w.e.f. March 31, 2017.",CONSECUTIVE_SAME_ROLE_WITHOUT_OPPOSITE_EVENT
9,GAYAPROJ,2020-03-27,EXIT,"Replacement in indices w.e.f. March 27, 2020",2020-06-26,EXIT,"Change in indices w.e.f. June 26, 2020",CONSECUTIVE_SAME_ROLE_WITHOUT_OPPOSITE_EVENT



Read-only trace complete.
No event ledger, interval, price row, or membership flag was changed.


In [47]:
from pathlib import Path
import re
import pandas as pd

BASE = Path("/content/nifty500_pit_data")
EVENTS_PATH = BASE / "nifty500_official_event_candidates_v5.csv"
SCAN_PATH = BASE / "nifty500_official_pdf_scan_v1.csv"
REPEAT_PATH = BASE / "nifty500_event_consecutive_actions_v5.csv"

for p in [EVENTS_PATH, SCAN_PATH, REPEAT_PATH]:
    assert p.exists(), f"Missing required file: {p}"

events = pd.read_csv(EVENTS_PATH, dtype=str, keep_default_na=False)
scan = pd.read_csv(SCAN_PATH, dtype=str, keep_default_na=False)
repeats = pd.read_csv(REPEAT_PATH, dtype=str, keep_default_na=False)

events["Role"] = events["Role"].str.upper().str.strip()
events["Effective_Date"] = events["Effective_Date"].str.strip()
events["Canonical_Symbol"] = (
    events["Canonical_Symbol"].str.upper().str.strip()
)

# Three known anomaly groups from the consistency audit.
cases = [
    ("DUPLICATE", "CROMPGREAV", "2015-09-28"),
    ("SAME_DATE_ROLE_CONFLICT", "ADANIPORTS", "2015-03-27"),
    ("SAME_DATE_ROLE_CONFLICT", "SHILPAMED", "2020-03-27"),
]

source_map = (
    scan.drop_duplicates("URL")
    .set_index("URL")["Text_Cache"]
    .to_dict()
)

print("=" * 78)
print("1. FULL EVENT ROWS FOR THE THREE ANOMALY GROUPS")
print("=" * 78)

event_cols = [
    c for c in [
        "Published_Date", "Effective_Date", "Role", "Company_Name",
        "Raw_Symbol", "Canonical_Symbol", "Release_Title",
        "Section_Title", "Action_Heading", "Effective_Date_Source",
        "Date_Override_Reason", "Source_URL"
    ]
    if c in events.columns
]

case_urls = []

for issue, symbol, date in cases:
    group = events[
        events["Canonical_Symbol"].eq(symbol)
        & events["Effective_Date"].eq(date)
    ].copy()

    print(f"\n--- {issue}: {symbol} on {date} ---")

    if group.empty:
        print("No matching event rows in v5.")
        continue

    display(group[event_cols])

    for url in group["Source_URL"].unique():
        case_urls.append((issue, symbol, date, url))

print("\n" + "=" * 78)
print("2. SOURCE PDF TEXT AROUND EACH ANOMALOUS TICKER")
print("=" * 78)

seen = set()

for issue, symbol, date, url in case_urls:
    key = (symbol, url)
    if key in seen:
        continue
    seen.add(key)

    text_path = source_map.get(url)
    print(f"\n--- {symbol} | {date} | {issue} ---")
    print("Source URL:", url)

    if not text_path or not Path(text_path).exists():
        print("Cached source text not found for this URL.")
        continue

    text = Path(text_path).read_text(
        encoding="utf-8", errors="replace"
    )
    parts = re.split(r"\n--- PAGE (\d+) ---\n", text)
    hits = []

    for i in range(1, len(parts), 2):
        page_no = int(parts[i])
        page_text = parts[i + 1] if i + 1 < len(parts) else ""
        lines = page_text.splitlines()

        for line_idx, line in enumerate(lines):
            if symbol.casefold() in line.casefold():
                start = max(0, line_idx - 5)
                end = min(len(lines), line_idx + 6)
                hits.append((page_no, line_idx + 1, lines[start:end]))

    if not hits:
        print("Ticker text not found in cached PDF extraction.")
        continue

    for page_no, line_no, context in hits[:8]:
        print(f"\nPage {page_no}, line {line_no}:")
        for line in context:
            print("  ", line)

    if len(hits) > 8:
        print(f"... {len(hits) - 8} additional occurrences omitted.")

print("\n" + "=" * 78)
print("3. REPEATED-ACTION FLAG SUMMARY")
print("=" * 78)

if repeats.empty:
    print("No consecutive same-role transitions recorded.")
else:
    repeats["Current_Year"] = pd.to_datetime(
        repeats["Current_Date"], errors="coerce"
    ).dt.year

    print("Repeated-action flags:", len(repeats))

    print("\nBy current event year and role:")
    display(
        repeats.groupby(
            ["Current_Year", "Current_Role"], dropna=False
        ).size().reset_index(name="Count")
        .sort_values(["Current_Year", "Current_Role"])
    )

    print("\nAll repeated-action flags (56 or fewer):")
    show_cols = [
        c for c in [
            "Symbol", "Previous_Date", "Previous_Role",
            "Previous_Release", "Current_Date", "Current_Role",
            "Current_Release", "Issue"
        ]
        if c in repeats.columns
    ]
    display(repeats[show_cols].sort_values(
        ["Symbol", "Previous_Date", "Current_Date"]
    ))

print("\nRead-only trace complete.")
print("No event ledger, interval, price row, or membership flag was changed.")

1. FULL EVENT ROWS FOR THE THREE ANOMALY GROUPS

--- DUPLICATE: CROMPGREAV on 2015-09-28 ---


,Published_Date,Effective_Date,Role,Company_Name,Raw_Symbol,Canonical_Symbol,Release_Title,Section_Title,Action_Heading,Effective_Date_Source,Date_Override_Reason,Source_URL
77,2015-08-24,2015-09-28,EXIT,Crompton Greaves Ltd.,CROMPGREAV,CROMPGREAV,"Change in Indices w.e.f September 28, 2015",CNX 500 Index,The following companies are being excluded,title_single_date,,https://www.niftyindices.com/Press_Release/ind...
81,2015-08-24,2015-09-28,EXIT,Crompton Greaves Ltd.,CROMPGREAV,CROMPGREAV,"Change in Indices w.e.f September 28, 2015",CNX 500 Index,The following company is being excluded,title_single_date,,https://www.niftyindices.com/Press_Release/ind...



--- SAME_DATE_ROLE_CONFLICT: ADANIPORTS on 2015-03-27 ---


,Published_Date,Effective_Date,Role,Company_Name,Raw_Symbol,Canonical_Symbol,Release_Title,Section_Title,Action_Heading,Effective_Date_Source,Date_Override_Reason,Source_URL
26,2015-02-20,2015-03-27,ENTRY,Adani Ports and Special Economic Zone Ltd.,ADANIPORTS,ADANIPORTS,"Change in indices w.e.f March 27, 2015.",CNX 500 Index,The following company is being included,title_single_date,,https://www.niftyindices.com/Press_Release/ind...
29,2015-02-20,2015-03-27,EXIT,Adani Ports and Special Economic Zone Ltd.,ADANIPORTS,ADANIPORTS,"Change in indices w.e.f March 27, 2015.",CNX 500 Index,The following company is being excluded,title_single_date,,https://www.niftyindices.com/Press_Release/ind...



--- SAME_DATE_ROLE_CONFLICT: SHILPAMED on 2020-03-27 ---


...


2. SOURCE PDF TEXT AROUND EACH ANOMALOUS TICKER

--- CROMPGREAV | 2015-09-28 | DUPLICATE ---
Source URL: https://www.niftyindices.com/Press_Release/ind_prs24082015.pdf

Page 1, line 27:
    
   Sr. No. Company Name 
   Symbol 
   1 
   Crompton Greaves Ltd. 
   CROMPGREAV 
    
   The following company is being included: 
    
   Sr. No. Company Name 
   Symbol 

Page 1, line 47:
    
   Sr. No. Company Name 
   Symbol 
   1 
   Crompton Greaves Ltd. 
   CROMPGREAV 
    
   The following company is being included: 
    
   Sr. No. Company Name 
   Symbol 

Page 1, line 69:
    
   Sr. No. Company Name 
   Symbol 
     1 
   Crompton Greaves Ltd. 
   CROMPGREAV 
   2 
   Max India Ltd. 
   MAX 
    
   

Page 2, line 27:
    
   Sr. No. Company Name 
   Symbol 
   1 
   Crompton Greaves Ltd. 
   CROMPGREAV 
   2 
   Max India Ltd. 
   MAX 
    
   The following companies are being included: 

Page 2, line 53:
    
   Sr. No. Company Name 
   Symbol 
   1 
   Crompton Greaves Ltd. 
   C

,Current_Year,Current_Role,Count
0,2017,ENTRY,4
1,2017,EXIT,4
2,2018,ENTRY,1
3,2018,EXIT,1
4,2020,ENTRY,15
5,2020,EXIT,21
6,2021,ENTRY,2
7,2021,EXIT,1
8,2022,ENTRY,2
9,2022,EXIT,3



All repeated-action flags (56 or fewer):


,Symbol,Previous_Date,Previous_Role,Previous_Release,Current_Date,Current_Role,Current_Release,Issue
0,AARTIIND,2019-03-29,EXIT,"Change in Indices w.e.f. March 29, 2019",2022-05-04,EXIT,"Replacements in indices w.e.f. May 04, 2022",CONSECUTIVE_SAME_ROLE_WITHOUT_OPPOSITE_EVENT
1,ASAHIINDIA,2018-09-28,EXIT,"Change in Indices w.e.f. September 28, 2018",2020-09-14,EXIT,Replacements in Indices,CONSECUTIVE_SAME_ROLE_WITHOUT_OPPOSITE_EVENT
2,BHARATRAS,2020-03-27,ENTRY,Revision in criteria and replacements in Indices,2020-06-26,ENTRY,"Change in indices w.e.f. June 26, 2020",CONSECUTIVE_SAME_ROLE_WITHOUT_OPPOSITE_EVENT
3,BLISSGVS,2020-03-27,EXIT,Revision in criteria and replacements in Indices,2021-09-30,EXIT,Replacements in indices and revision in criteria,CONSECUTIVE_SAME_ROLE_WITHOUT_OPPOSITE_EVENT
4,CENTURYTEX,2020-03-27,ENTRY,Revision in criteria and replacements in Indices,2020-06-26,ENTRY,"Change in indices w.e.f. June 26, 2020",CONSECUTIVE_SAME_ROLE_WITHOUT_OPPOSITE_EVENT
5,DHFL,2020-03-27,EXIT,Revision in criteria and replacements in Indices,2020-06-26,EXIT,"Change in indices w.e.f. June 26, 2020",CONSECUTIVE_SAME_ROLE_WITHOUT_OPPOSITE_EVENT
6,ERIS,2018-03-13,ENTRY,Replacements in Indices,2018-04-02,ENTRY,"Change in Indices w.e.f. April 02, 2018",CONSECUTIVE_SAME_ROLE_WITHOUT_OPPOSITE_EVENT
7,FLUOROCHEM,2020-03-27,ENTRY,Revision in criteria and replacements in Indices,2020-06-26,ENTRY,"Change in indices w.e.f. June 26, 2020",CONSECUTIVE_SAME_ROLE_WITHOUT_OPPOSITE_EVENT
8,FRETAIL,2017-03-16,ENTRY,Replacements in Indices.,2017-03-31,ENTRY,"Change in Indices w.e.f. March 31, 2017.",CONSECUTIVE_SAME_ROLE_WITHOUT_OPPOSITE_EVENT
9,GAYAPROJ,2020-03-27,EXIT,"Replacement in indices w.e.f. March 27, 2020",2020-06-26,EXIT,"Change in indices w.e.f. June 26, 2020",CONSECUTIVE_SAME_ROLE_WITHOUT_OPPOSITE_EVENT



Read-only trace complete.
No event ledger, interval, price row, or membership flag was changed.


In [48]:
from pathlib import Path
import pandas as pd

BASE = Path("/content/nifty500_pit_data")
SOURCE = BASE / "nifty500_official_event_candidates_v5.csv"

if not SOURCE.exists():
    raise FileNotFoundError(f"Expected event ledger not found: {SOURCE}")

events = pd.read_csv(SOURCE, dtype=str, keep_default_na=False)
events["_Ledger_Row_Index"] = events.index

# Normalized fields used only for matching and QA.
effective = pd.to_datetime(
    events["Effective_Date"], errors="coerce"
).dt.strftime("%Y-%m-%d")

published = pd.to_datetime(
    events["Published_Date"], errors="coerce"
).dt.strftime("%Y-%m-%d")

symbol = events["Canonical_Symbol"].str.strip().str.upper()
role = events["Role"].str.strip().str.upper()
url = events["Source_URL"].str.lower()
heading = events["Action_Heading"].str.lower()

decisions = []

def record_rows(mask, decision, issue, reason):
    for idx, row in events.loc[mask].iterrows():
        item = row.drop(labels=["_Ledger_Row_Index"]).to_dict()
        item["Ledger_Row_Index"] = idx
        item["Audit_Issue"] = issue
        item["Proposed_Decision"] = decision
        item["Reason"] = reason
        decisions.append(item)

# -------------------------------------------------------------------
# 1. CROMPGREAV: preserve one verified NIFTY 500 exit; flag duplicate.
# The official NIFTY 500 section has a plural exclusion heading and
# lists both CROMPGREAV and MAX.
# -------------------------------------------------------------------
crom_mask = (
    effective.eq("2015-09-28")
    & symbol.eq("CROMPGREAV")
    & role.eq("EXIT")
    & url.str.contains("ind_prs24082015", regex=False)
)
crom = events.loc[crom_mask]

if len(crom):
    valid_heading = crom.index[
        heading.loc[crom.index].str.contains(
            "companies are being excluded", regex=False
        )
    ]
    keep_idx = valid_heading[0] if len(valid_heading) else crom.index[0]

    for idx, row in crom.iterrows():
        item = row.drop(labels=["_Ledger_Row_Index"]).to_dict()
        item["Ledger_Row_Index"] = idx
        item["Audit_Issue"] = "CROMPGREAV_2015_DUPLICATE"
        if idx == keep_idx:
            item["Proposed_Decision"] = "KEEP_ONE_VERIFIED_EVENT"
            item["Reason"] = (
                "Official CNX 500 exclusion list includes CROMPGREAV once."
            )
        else:
            item["Proposed_Decision"] = "EXCLUDE_DUPLICATE_EXTRACTION"
            item["Reason"] = (
                "Duplicate extraction from the same release and effective date."
            )
        decisions.append(item)

# -------------------------------------------------------------------
# 2. ADANIPORTS: both rows are incorrectly attributed to NIFTY 500.
# The source lists ADANIPORTS under LIX15 and LIX15 Midcap instead.
# -------------------------------------------------------------------
adani_mask = (
    effective.eq("2015-03-27")
    & symbol.eq("ADANIPORTS")
    & url.str.contains("ind_prs20022015", regex=False)
)

record_rows(
    adani_mask,
    "EXCLUDE_WRONG_INDEX_SECTION",
    "ADANIPORTS_2015_SECTION_SCOPE",
    "ADANIPORTS appears in LIX15/LIX15 Midcap sections, not the CNX 500 section."
)

# -------------------------------------------------------------------
# 3. SHILPAMED: the Feb 18 exit was explicitly reversed on Mar 12.
# Treat these as a membership-transition conflict to resolve, not as
# a normal exit followed by a new entry.
# -------------------------------------------------------------------
shilpa_feb_mask = (
    effective.eq("2020-03-27")
    & symbol.eq("SHILPAMED")
    & role.eq("EXIT")
    & url.str.contains("ind_prs18022020", regex=False)
)

record_rows(
    shilpa_feb_mask,
    "DO_NOT_APPLY_REVERSED_EXIT",
    "SHILPAMED_2020_REVERSAL",
    "The March 12 release explicitly reverses the February 18 exclusion announcement."
)

shilpa_mar_mask = (
    effective.eq("2020-03-27")
    & symbol.eq("SHILPAMED")
    & role.eq("ENTRY")
    & url.str.contains("ind_prs12032020", regex=False)
)

record_rows(
    shilpa_mar_mask,
    "REVIEW_AS_NO_NET_MEMBERSHIP_TRANSITION",
    "SHILPAMED_2020_REVERSAL",
    "The March 12 inclusion accompanies the reversal; do not infer a fresh membership start without reconciling the full review lifecycle."
)

# -------------------------------------------------------------------
# 4. Release-level audit of March 27, 2020 events.
# These are audit counts only; no broad event cancellation is applied.
# -------------------------------------------------------------------
review_release_mask = (
    effective.eq("2020-03-27")
    & url.str.contains(
        r"ind_prs18022020|ind_prs12032020|ind_prs19032020",
        regex=True
    )
)

review_rows = events.loc[review_release_mask].copy()

if len(review_rows):
    review_rows["Audit_Published_Date"] = published.loc[review_rows.index]
    review_rows["Audit_Role"] = role.loc[review_rows.index]

    release_audit = (
        review_rows.groupby(
            [
                "Audit_Published_Date",
                "Release_Title",
                "Source_URL",
                "Section_Title",
                "Audit_Role"
            ],
            dropna=False
        )
        .size()
        .rename("Event_Rows")
        .reset_index()
        .sort_values(
            ["Audit_Published_Date", "Source_URL", "Section_Title", "Audit_Role"]
        )
    )
else:
    release_audit = pd.DataFrame(
        columns=[
            "Audit_Published_Date", "Release_Title", "Source_URL",
            "Section_Title", "Audit_Role", "Event_Rows"
        ]
    )

# Save to the next available version, never overwriting a previous audit.
def next_version_path(stem):
    version = 1
    while True:
        path = BASE / f"{stem}_v{version}.csv"
        if not path.exists():
            return path
        version += 1

decision_path = next_version_path("nifty500_event_anomaly_decisions")
release_path = next_version_path("nifty500_event_20200327_release_audit")

decision_df = pd.DataFrame(decisions)
decision_df.to_csv(decision_path, index=False)
release_audit.to_csv(release_path, index=False)

print("=" * 78)
print("STAGED AUDIT COMPLETE — ORIGINAL FILES NOT MODIFIED")
print("=" * 78)

print(f"\nOriginal event rows: {len(events):,}")
print(f"CROMPGREAV matching rows: {len(crom)}")
print(f"ADANIPORTS matching rows: {int(adani_mask.sum())}")
print(f"SHILPAMED Feb 18 exit rows: {int(shilpa_feb_mask.sum())}")
print(f"SHILPAMED Mar 12 entry rows: {int(shilpa_mar_mask.sum())}")
print(f"March 2020 review rows audited: {len(review_rows)}")

print("\nProposed anomaly decisions:")
if len(decision_df):
    cols = [
        "Ledger_Row_Index", "Published_Date", "Effective_Date",
        "Role", "Canonical_Symbol", "Release_Title",
        "Audit_Issue", "Proposed_Decision"
    ]
    display(decision_df[cols].sort_values(
        ["Effective_Date", "Canonical_Symbol", "Ledger_Row_Index"]
    ))
else:
    print("No matching rows found; inspect the file path and event columns.")

print("\nMarch 27, 2020 release-level counts:")
if len(release_audit):
    display(release_audit)
else:
    print("No March 27, 2020 rows matched the three release URLs.")

print("\nFiles created:")
print(decision_path)
print(release_path)

STAGED AUDIT COMPLETE — ORIGINAL FILES NOT MODIFIED

Original event rows: 1,486
CROMPGREAV matching rows: 2
ADANIPORTS matching rows: 2
SHILPAMED Feb 18 exit rows: 1
SHILPAMED Mar 12 entry rows: 1
March 2020 review rows audited: 42

Proposed anomaly decisions:


,Ledger_Row_Index,Published_Date,Effective_Date,Role,Canonical_Symbol,Release_Title,Audit_Issue,Proposed_Decision
2,26,2015-02-20,2015-03-27,ENTRY,ADANIPORTS,"Change in indices w.e.f March 27, 2015.",ADANIPORTS_2015_SECTION_SCOPE,EXCLUDE_WRONG_INDEX_SECTION
3,29,2015-02-20,2015-03-27,EXIT,ADANIPORTS,"Change in indices w.e.f March 27, 2015.",ADANIPORTS_2015_SECTION_SCOPE,EXCLUDE_WRONG_INDEX_SECTION
0,77,2015-08-24,2015-09-28,EXIT,CROMPGREAV,"Change in Indices w.e.f September 28, 2015",CROMPGREAV_2015_DUPLICATE,KEEP_ONE_VERIFIED_EVENT
1,81,2015-08-24,2015-09-28,EXIT,CROMPGREAV,"Change in Indices w.e.f September 28, 2015",CROMPGREAV_2015_DUPLICATE,EXCLUDE_DUPLICATE_EXTRACTION
4,698,2020-02-18,2020-03-27,EXIT,SHILPAMED,Revision in criteria and replacements in Indices,SHILPAMED_2020_REVERSAL,DO_NOT_APPLY_REVERSED_EXIT
5,723,2020-03-12,2020-03-27,ENTRY,SHILPAMED,"Replacement in indices w.e.f. March 27, 2020",SHILPAMED_2020_REVERSAL,REVIEW_AS_NO_NET_MEMBERSHIP_TRANSITION



March 27, 2020 release-level counts:


,Audit_Published_Date,Release_Title,Source_URL,Section_Title,Audit_Role,Event_Rows
0,2020-02-18,Revision in criteria and replacements in Indices,https://www.niftyindices.com/Press_Release/ind...,NIFTY 500,ENTRY,20
1,2020-02-18,Revision in criteria and replacements in Indices,https://www.niftyindices.com/Press_Release/ind...,NIFTY 500,EXIT,20
2,2020-03-12,"Replacement in indices w.e.f. March 27, 2020",https://www.niftyindices.com/Press_Release/ind...,NIFTY 500,ENTRY,1
3,2020-03-12,"Replacement in indices w.e.f. March 27, 2020",https://www.niftyindices.com/Press_Release/ind...,NIFTY 500,EXIT,1



Files created:
/content/nifty500_pit_data/nifty500_event_anomaly_decisions_v1.csv
/content/nifty500_pit_data/nifty500_event_20200327_release_audit_v1.csv


In [49]:
from pathlib import Path
import pandas as pd

BASE = Path("/content/nifty500_pit_data")
EVENT_FILE = BASE / "nifty500_official_event_candidates_v5.csv"

if not EVENT_FILE.exists():
    raise FileNotFoundError(f"Missing event ledger: {EVENT_FILE}")

events = pd.read_csv(EVENT_FILE, dtype=str, keep_default_na=False)

required = {
    "Published_Date", "Effective_Date", "Role", "Canonical_Symbol",
    "Release_Title", "Section_Title", "Source_URL"
}
missing = required - set(events.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

pub = pd.to_datetime(events["Published_Date"], errors="coerce")
eff = pd.to_datetime(events["Effective_Date"], errors="coerce")

pub_day = pub.dt.strftime("%Y-%m-%d")
eff_day = eff.dt.strftime("%Y-%m-%d")
url = events["Source_URL"].str.lower()
section = events["Section_Title"].str.upper()
role = events["Role"].str.upper()

# Match the official release PDF names within the source URLs.
r_feb18 = url.str.contains("ind_prs18022020", regex=False)
r_mar12 = url.str.contains("ind_prs12032020", regex=False)
r_mar19 = url.str.contains("ind_prs19032020", regex=False)
r_mar23 = url.str.contains("ind_prs23032020", regex=False)
r_jun10 = url.str.contains("ind_prs10062020", regex=False)

# Events announced for March 27 by the releases the May 13 notice
# subsequently declared null and void.
void_mask = (
    eff_day.eq("2020-03-27")
    & (r_feb18 | r_mar12 | r_mar19)
)

# The March 23 notice described the special concentration-norm changes;
# the March 25 notice then deferred those changes.
deferred_mask = (
    eff_day.eq("2020-03-27")
    & r_mar23
)

# The June 10 list is the replacement list for the fresh review.
active_mask = (
    eff_day.eq("2020-06-26")
    & r_jun10
)

masks = [
    (void_mask, "VOID_BY_2020-05-13",
     "May 13 official notice: March 27 replacements from the February 18, March 12 and March 19 releases stand null and void."),
    (deferred_mask, "DEFERRED_NOT_EFFECTIVE_2020-03-27",
     "March 25 official notice deferred the March 27 concentration-norm changes."),
    (active_mask, "ACTIVE_REVIEW_EFFECTIVE_2020-06-26",
     "June 10 official replacement list: effective June 26, 2020, after the June 25 close.")
]

staged_parts = []

for mask, disposition, reason in masks:
    part = events.loc[mask].copy()
    if not part.empty:
        part["Audit_Disposition"] = disposition
        part["Audit_Basis"] = reason
        part["Audit_Source_Stem"] = (
            part["Source_URL"].str.extract(r"(ind_prs\d{8}(?:_\d+)?)", expand=False)
            .fillna("source_stem_unmatched")
        )
        staged_parts.append(part)

if staged_parts:
    staged = pd.concat(staged_parts, ignore_index=True)
else:
    staged = events.iloc[0:0].copy()
    staged["Audit_Disposition"] = pd.Series(dtype=str)
    staged["Audit_Basis"] = pd.Series(dtype=str)
    staged["Audit_Source_Stem"] = pd.Series(dtype=str)

# Save a versioned sidecar. Never overwrite the event ledger.
def next_path(stem):
    version = 1
    while True:
        path = BASE / f"{stem}_v{version}.csv"
        if not path.exists():
            return path
        version += 1

OUT = next_path("nifty500_event_lifecycle_dispositions")
staged.to_csv(OUT, index=False)

# ---------------------------------------------------------------
# Diagnostic summary: all NIFTY 500 / CNX 500 candidate rows on the
# affected dates, including rows that did not match a known release.
# ---------------------------------------------------------------
nifty500_section = section.str.contains(
    r"NIFTY\s*500|CNX\s*500", regex=True, na=False
)

date_audit_mask = (
    eff_day.isin(["2020-03-27", "2020-06-26"])
    & nifty500_section
)

date_audit = events.loc[date_audit_mask].copy()
date_audit["Audit_Published_Date"] = pub_day.loc[date_audit.index]
date_audit["Audit_Effective_Date"] = eff_day.loc[date_audit.index]
date_audit["Audit_Role"] = role.loc[date_audit.index]

print("=" * 78)
print("MARCH–JUNE 2020 LIFECYCLE AUDIT")
print("=" * 78)
print(f"Original event rows: {len(events):,}")
print(f"Rows staged with a disposition: {len(staged):,}")
print(f"March 27 NIFTY 500/CNX 500 candidate rows: {int((date_audit['Audit_Effective_Date'] == '2020-03-27').sum())}")
print(f"June 26 NIFTY 500/CNX 500 candidate rows: {int((date_audit['Audit_Effective_Date'] == '2020-06-26').sum())}")

print("\nDisposition counts:")
if not staged.empty:
    display(
        staged.groupby(
            ["Audit_Disposition", "Audit_Source_Stem", "Role"],
            dropna=False
        ).size().rename("Rows").reset_index()
    )
else:
    print("No rows matched the release/date rules. Check the source URLs and date fields.")

print("\nAffected-date source counts:")
if not date_audit.empty:
    display(
        date_audit.groupby(
            [
                "Audit_Published_Date",
                "Audit_Effective_Date",
                "Release_Title",
                "Source_URL",
                "Section_Title",
                "Audit_Role"
            ],
            dropna=False
        ).size().rename("Rows").reset_index().sort_values(
            ["Audit_Effective_Date", "Audit_Published_Date", "Audit_Role"]
        )
    )
else:
    print("No NIFTY 500/CNX 500 rows found on the two affected dates.")

print("\nSHILPAMED lifecycle check:")
if not staged.empty:
    sh = staged[
        staged["Canonical_Symbol"].str.upper().eq("SHILPAMED")
    ]
    if sh.empty:
        print("SHILPAMED did not match a staged disposition; inspect its source URL/date.")
    else:
        display(sh[[
            "Published_Date", "Effective_Date", "Role",
            "Release_Title", "Audit_Disposition", "Audit_Basis"
        ]])

print("\nStaged file saved to:")
print(OUT)
print("\nNo original event, interval, price, or membership file was modified.")

MARCH–JUNE 2020 LIFECYCLE AUDIT
Original event rows: 1,486
Rows staged with a disposition: 92
March 27 NIFTY 500/CNX 500 candidate rows: 42
June 26 NIFTY 500/CNX 500 candidate rows: 50

Disposition counts:


,Audit_Disposition,Audit_Source_Stem,Role,Rows
0,ACTIVE_REVIEW_EFFECTIVE_2020-06-26,ind_prs10062020,ENTRY,25
1,ACTIVE_REVIEW_EFFECTIVE_2020-06-26,ind_prs10062020,EXIT,25
2,VOID_BY_2020-05-13,ind_prs12032020,ENTRY,1
3,VOID_BY_2020-05-13,ind_prs12032020,EXIT,1
4,VOID_BY_2020-05-13,ind_prs18022020,ENTRY,20
5,VOID_BY_2020-05-13,ind_prs18022020,EXIT,20



Affected-date source counts:


,Audit_Published_Date,Audit_Effective_Date,Release_Title,Source_URL,Section_Title,Audit_Role,Rows
0,2020-02-18,2020-03-27,Revision in criteria and replacements in Indices,https://www.niftyindices.com/Press_Release/ind...,NIFTY 500,ENTRY,20
1,2020-02-18,2020-03-27,Revision in criteria and replacements in Indices,https://www.niftyindices.com/Press_Release/ind...,NIFTY 500,EXIT,20
2,2020-03-12,2020-03-27,"Replacement in indices w.e.f. March 27, 2020",https://www.niftyindices.com/Press_Release/ind...,NIFTY 500,ENTRY,1
3,2020-03-12,2020-03-27,"Replacement in indices w.e.f. March 27, 2020",https://www.niftyindices.com/Press_Release/ind...,NIFTY 500,EXIT,1
4,2020-06-10,2020-06-26,"Change in indices w.e.f. June 26, 2020",https://www.niftyindices.com/Press_Release/ind...,NIFTY 500,ENTRY,25
5,2020-06-10,2020-06-26,"Change in indices w.e.f. June 26, 2020",https://www.niftyindices.com/Press_Release/ind...,NIFTY 500,EXIT,25



SHILPAMED lifecycle check:


,Published_Date,Effective_Date,Role,Release_Title,Audit_Disposition,Audit_Basis
16,2020-02-18,2020-03-27,EXIT,Revision in criteria and replacements in Indices,VOID_BY_2020-05-13,May 13 official notice: March 27 replacements ...
41,2020-03-12,2020-03-27,ENTRY,"Replacement in indices w.e.f. March 27, 2020",VOID_BY_2020-05-13,May 13 official notice: March 27 replacements ...
62,2020-06-10,2020-06-26,EXIT,"Change in indices w.e.f. June 26, 2020",ACTIVE_REVIEW_EFFECTIVE_2020-06-26,June 10 official replacement list: effective J...



Staged file saved to:
/content/nifty500_pit_data/nifty500_event_lifecycle_dispositions_v1.csv

No original event, interval, price, or membership file was modified.


In [50]:
from pathlib import Path
import pandas as pd
import re

BASE = Path("/content/nifty500_pit_data")
EVENT_FILE = BASE / "nifty500_official_event_candidates_v5.csv"
CATALOG_FILE = BASE / "nifty500_official_release_catalog_v1.csv"
CACHE_DIR = BASE / "official_release_text_cache"

events = pd.read_csv(EVENT_FILE, dtype=str, keep_default_na=False)

print("=" * 78)
print("MARCH–JUNE 2020 RELEASE COVERAGE DIAGNOSTIC — READ ONLY")
print("=" * 78)

release_patterns = {
    "2020-03-19 release": r"ind_prs19032020",
    "2020-03-23 deferral": r"ind_prs23032020",
    "2020-03-25 deferral": r"ind_prs25032020",
    "2020-05-13 lifecycle notice": r"ind_prs13052020",
    "2020-06-10 replacement list": r"ind_prs10062020",
}

url = events["Source_URL"].astype(str).str.lower()
pub = pd.to_datetime(events["Published_Date"], errors="coerce")
eff = pd.to_datetime(events["Effective_Date"], errors="coerce")

# ------------------------------------------------------------------
# 1. Check whether each official release is represented in the ledger.
# ------------------------------------------------------------------
print("\n1. RELEASE URL MATCHES IN EVENT LEDGER")

for label, pattern in release_patterns.items():
    mask = url.str.contains(pattern, regex=True, na=False)
    matched = events.loc[mask]

    print(f"\n{label}: {len(matched)} event rows")

    if not matched.empty:
        cols = [
            c for c in [
                "Published_Date", "Effective_Date", "Role",
                "Canonical_Symbol", "Release_Title",
                "Section_Title", "Action_Heading", "Source_URL"
            ]
            if c in matched.columns
        ]

        summary = (
            matched.groupby(
                [c for c in [
                    "Published_Date", "Effective_Date", "Release_Title",
                    "Section_Title", "Role", "Source_URL"
                ] if c in matched.columns],
                dropna=False
            )
            .size()
            .rename("Rows")
            .reset_index()
        )
        print(summary.to_string(index=False, max_colwidth=100))

# ------------------------------------------------------------------
# 2. Search the official release catalog by filename, title, or URL.
# ------------------------------------------------------------------
print("\n\n2. MATCHES IN OFFICIAL RELEASE CATALOG")

if CATALOG_FILE.exists():
    catalog = pd.read_csv(CATALOG_FILE, dtype=str, keep_default_na=False)
    print("Catalog columns:", list(catalog.columns))

    all_text = catalog.astype(str).agg(" | ".join, axis=1).str.lower()

    search_terms = [
        "19032020", "23032020", "25032020",
        "13052020", "10062020",
        "march 19, 2020", "march 23, 2020",
        "march 25, 2020", "may 13, 2020",
        "june 10, 2020", "deferment of index rebalancing",
        "index reconstitution and revision in eligibility criteria",
    ]

    catalog_mask = pd.Series(False, index=catalog.index)
    for term in search_terms:
        catalog_mask |= all_text.str.contains(
            re.escape(term.lower()), regex=True, na=False
        )

    found = catalog.loc[catalog_mask]
    print(f"Matching catalog rows: {len(found)}")

    if not found.empty:
        display(found)
else:
    print("Catalog file not found:", CATALOG_FILE)

# ------------------------------------------------------------------
# 3. Search the local extracted-text cache filenames.
# ------------------------------------------------------------------
print("\n\n3. LOCAL TEXT-CACHE FILES")

if CACHE_DIR.exists():
    cache_files = sorted(
        p.name for p in CACHE_DIR.rglob("*")
        if p.is_file()
    )

    terms = [
        "19032020", "23032020", "25032020",
        "13052020", "10062020",
        "march19", "march23", "march25",
        "may13", "june10",
    ]

    cache_matches = [
        name for name in cache_files
        if any(term in name.lower() for term in terms)
    ]

    print(f"Files in cache: {len(cache_files)}")
    print(f"Filename matches: {len(cache_matches)}")
    for name in cache_matches:
        print(name)
else:
    print("Text-cache directory not found:", CACHE_DIR)

# ------------------------------------------------------------------
# 4. Summarize all NIFTY 500/CNX 500 event rows around the review.
# ------------------------------------------------------------------
print("\n\n4. NIFTY 500/CNX 500 EVENTS PUBLISHED MAR 18–JUN 12, 2020")

section = events["Section_Title"].astype(str).str.upper()
is_broad_index = section.str.contains(
    r"NIFTY\s*500|CNX\s*500", regex=True, na=False
)

date_window = pub.between(
    pd.Timestamp("2020-03-18"),
    pd.Timestamp("2020-06-12")
)

window_rows = events.loc[date_window & is_broad_index].copy()
window_rows["Audit_Published_Date"] = pub.loc[window_rows.index].dt.strftime("%Y-%m-%d")
window_rows["Audit_Effective_Date"] = eff.loc[window_rows.index].dt.strftime("%Y-%m-%d")

if not window_rows.empty:
    group_cols = [
        "Audit_Published_Date", "Audit_Effective_Date",
        "Release_Title", "Source_URL", "Section_Title", "Role"
    ]
    group_cols = [c for c in group_cols if c in window_rows.columns]

    display(
        window_rows.groupby(group_cols, dropna=False)
        .size()
        .rename("Rows")
        .reset_index()
        .sort_values(["Audit_Published_Date", "Audit_Effective_Date"])
    )
else:
    print("No matching rows in the date window.")

# ------------------------------------------------------------------
# 5. Targeted tracing for known symbols in the concentration-norm list
# and the later full review.
# ------------------------------------------------------------------
print("\n\n5. TARGET SYMBOL TRACE")

target_symbols = {"GAYAPROJ", "FLUOROCHEM", "SHILPAMED"}
target_mask = (
    events["Canonical_Symbol"].astype(str).str.upper().isin(target_symbols)
    & date_window
)

target_rows = events.loc[target_mask].copy()

if not target_rows.empty:
    cols = [
        c for c in [
            "Published_Date", "Effective_Date", "Role",
            "Canonical_Symbol", "Release_Title",
            "Section_Title", "Action_Heading", "Source_URL"
        ]
        if c in target_rows.columns
    ]
    display(
        target_rows[cols].sort_values(
            ["Published_Date", "Canonical_Symbol", "Role"]
        )
    )
else:
    print("No target-symbol rows found in the publication date window.")

print("\nDiagnostic complete. No files or event rows were modified.")

MARCH–JUNE 2020 RELEASE COVERAGE DIAGNOSTIC — READ ONLY

1. RELEASE URL MATCHES IN EVENT LEDGER

2020-03-19 release: 0 event rows

2020-03-23 deferral: 0 event rows

2020-03-25 deferral: 0 event rows

2020-05-13 lifecycle notice: 0 event rows

2020-06-10 replacement list: 50 event rows
Published_Date Effective_Date                          Release_Title Section_Title  Role                                                     Source_URL  Rows
    2020-06-10     2020-06-26 Change in indices w.e.f. June 26, 2020     NIFTY 500 ENTRY https://www.niftyindices.com/Press_Release/ind_prs10062020.pdf    25
    2020-06-10     2020-06-26 Change in indices w.e.f. June 26, 2020     NIFTY 500  EXIT https://www.niftyindices.com/Press_Release/ind_prs10062020.pdf    25


2. MATCHES IN OFFICIAL RELEASE CATALOG
Catalog columns: ['Title', 'Published_Date', 'URL']
Matching catalog rows: 2


,Title,Published_Date,URL
52,"Change in indices w.e.f. March 27, 2020","Mar 19, 2020",https://www.niftyindices.com/Press_Release/ind...
53,"Change in indices w.e.f. June 26, 2020","Jun 10, 2020",https://www.niftyindices.com/Press_Release/ind...




3. LOCAL TEXT-CACHE FILES
Files in cache: 157
Filename matches: 0


4. NIFTY 500/CNX 500 EVENTS PUBLISHED MAR 18–JUN 12, 2020


,Audit_Published_Date,Audit_Effective_Date,Release_Title,Source_URL,Section_Title,Role,Rows
0,2020-06-10,2020-06-26,"Change in indices w.e.f. June 26, 2020",https://www.niftyindices.com/Press_Release/ind...,NIFTY 500,ENTRY,25
1,2020-06-10,2020-06-26,"Change in indices w.e.f. June 26, 2020",https://www.niftyindices.com/Press_Release/ind...,NIFTY 500,EXIT,25




5. TARGET SYMBOL TRACE


,Published_Date,Effective_Date,Role,Canonical_Symbol,Release_Title,Section_Title,Action_Heading,Source_URL
758,2020-06-10,2020-06-26,ENTRY,FLUOROCHEM,"Change in indices w.e.f. June 26, 2020",NIFTY 500,The following companies are being included,https://www.niftyindices.com/Press_Release/ind...
728,2020-06-10,2020-06-26,EXIT,GAYAPROJ,"Change in indices w.e.f. June 26, 2020",NIFTY 500,The following companies are being excluded,https://www.niftyindices.com/Press_Release/ind...
744,2020-06-10,2020-06-26,EXIT,SHILPAMED,"Change in indices w.e.f. June 26, 2020",NIFTY 500,The following companies are being excluded,https://www.niftyindices.com/Press_Release/ind...



Diagnostic complete. No files or event rows were modified.


In [51]:
from pathlib import Path
import pandas as pd
import re

BASE = Path("/content/nifty500_pit_data")
CACHE = BASE / "official_release_text_cache"
CATALOG_FILE = BASE / "nifty500_official_release_catalog_v1.csv"

if not CACHE.exists():
    raise FileNotFoundError(f"Text cache not found: {CACHE}")
if not CATALOG_FILE.exists():
    raise FileNotFoundError(f"Release catalog not found: {CATALOG_FILE}")

catalog = pd.read_csv(CATALOG_FILE, dtype=str, keep_default_na=False)
catalog["Parsed_Date"] = pd.to_datetime(
    catalog["Published_Date"], errors="coerce"
)

print("=" * 78)
print("OFFICIAL RELEASE TEXT SEARCH — READ ONLY")
print("=" * 78)

# 1. Show the release catalog in the relevant period.
catalog_window = catalog[
    catalog["Parsed_Date"].between("2020-03-01", "2020-06-30")
].copy()

print(f"\n1. Catalog entries published Mar–Jun 2020: {len(catalog_window)}")
if not catalog_window.empty:
    display(
        catalog_window[
            ["Published_Date", "Title", "URL"]
        ].sort_values("Published_Date")
    )

# 2. Search the contents of every cached text file.
patterns = {
    "STERLING_WILSON": r"sterling\s+and\s+wilson",
    "ALKYL_AMINES": r"alkyl\s+amines|alkylamine",
    "NULL_AND_VOID": r"null\s+and\s+void",
    "MAY_13_LIFECYCLE": r"index reconstitution and revision in eligibility criteria",
    "MARCH_23_DEFERRAL": r"deferment of index rebalancing",
    "GAYATRI_PROJECTS": r"gayatri projects|gayaproj",
    "GUJARAT_FLUOROCHEMICALS": r"gujarat fluorochemicals|fluorochem",
    "MARCH_19_CONTEXT": r"march 19, 2020",
    "MARCH_25_CONTEXT": r"march 25, 2020",
}

text_files = sorted(
    p for p in CACHE.rglob("*")
    if p.is_file() and p.suffix.lower() in {".txt", ".text"}
)

print(f"\n2. Cached text files searched: {len(text_files)}")

hits = []
for path in text_files:
    try:
        text = path.read_text(encoding="utf-8", errors="replace")
    except Exception as exc:
        print(f"Could not read {path.name}: {exc}")
        continue

    found = [
        label for label, pattern in patterns.items()
        if re.search(pattern, text, flags=re.IGNORECASE)
    ]

    if found:
        hits.append((path, text, found))

print(f"Files with at least one target phrase: {len(hits)}")

# Print each matching document's opening text and target snippets.
for path, text, found in hits:
    lines = [line.strip() for line in text.splitlines() if line.strip()]

    print("\n" + "-" * 78)
    print("CACHE FILE:", path.name)
    print("MATCHED TOPICS:", ", ".join(found))
    print("OPENING TEXT:")
    for line in lines[:8]:
        print(" ", line[:240])

    printed = set()
    shown = 0
    for i, line in enumerate(lines):
        if any(
            re.search(pattern, line, flags=re.IGNORECASE)
            for pattern in patterns.values()
        ):
            snippet = " | ".join(lines[max(0, i-1):min(len(lines), i+2)])
            if snippet not in printed:
                print(" MATCH:", snippet[:500])
                printed.add(snippet)
                shown += 1
            if shown >= 5:
                break

# 3. Check the March 19 release row and identify its PDF filename.
print("\n\n3. MARCH 19 CATALOG RECORD")
march19 = catalog[
    catalog["Parsed_Date"].dt.strftime("%Y-%m-%d").eq("2020-03-19")
]
if not march19.empty:
    display(march19[["Published_Date", "Title", "URL"]])
else:
    print("No March 19, 2020 release row found in the catalog.")

# 4. Focused index changes in the existing ledger.
EVENT_FILE = BASE / "nifty500_official_event_candidates_v5.csv"
events = pd.read_csv(EVENT_FILE, dtype=str, keep_default_na=False)

print("\n\n4. EXISTING LEDGER ROWS FOR KEY SYMBOLS")
symbols = {"STERLINGWIL", "SWSOLAR", "ALKYLAMINE", "YESBANK",
           "GAYAPROJ", "FLUOROCHEM"}

symbol_col = events["Canonical_Symbol"].str.upper()
focused = events[symbol_col.isin(symbols)].copy()

if focused.empty:
    print("No rows found for the requested symbols.")
else:
    display(
        focused[
            [
                "Published_Date", "Effective_Date", "Role",
                "Canonical_Symbol", "Release_Title",
                "Section_Title", "Source_URL"
            ]
        ].sort_values(["Published_Date", "Canonical_Symbol"])
    )

print("\nDiagnostic complete. No files or data were modified.")

OFFICIAL RELEASE TEXT SEARCH — READ ONLY

1. Catalog entries published Mar–Jun 2020: 3


,Published_Date,Title,URL
53,"Jun 10, 2020","Change in indices w.e.f. June 26, 2020",https://www.niftyindices.com/Press_Release/ind...
51,"Mar 12, 2020","Replacement in indices w.e.f. March 27, 2020",https://www.niftyindices.com/Press_Release/ind...
52,"Mar 19, 2020","Change in indices w.e.f. March 27, 2020",https://www.niftyindices.com/Press_Release/ind...



2. Cached text files searched: 157
Files with at least one target phrase: 36

------------------------------------------------------------------------------
CACHE FILE: release_011.txt
MATCHED TOPICS: GAYATRI_PROJECTS
OPENING TEXT:
  --- PAGE 1 ---
  December 07, 2015
  PRESS RELEASE
  The Index Maintenance Sub-Committee has decided to make the following
  replacements in Nifty 500 and Nifty Smallcap 100 indices on account of proposed
  schemes of arrangement for transfer by way of a demerger of the Retail Business
  Undertakings of Future Retail Ltd. and Mandhana Industries Ltd. The changes shall
  become effective from December 14, 2015 (close of December 11, 2015).
 MATCH: 2 | Gayatri Projects Ltd. | GAYAPROJ
 MATCH: Gayatri Projects Ltd. | GAYAPROJ | 2)

------------------------------------------------------------------------------
CACHE FILE: release_013.txt
MATCHED TOPICS: GUJARAT_FLUOROCHEMICALS
OPENING TEXT:
  --- PAGE 1 ---
  1
  Date: February 22, 2016
  PRESS RELEASE
  Semi

,Published_Date,Title,URL
52,"Mar 19, 2020","Change in indices w.e.f. March 27, 2020",https://www.niftyindices.com/Press_Release/ind...




4. EXISTING LEDGER ROWS FOR KEY SYMBOLS


,Published_Date,Effective_Date,Role,Canonical_Symbol,Release_Title,Section_Title,Source_URL
88,2015-12-07,2015-12-14,ENTRY,GAYAPROJ,Change in Nifty 500 & Nifty Smallcap 100 Indic...,Nifty 500 Index,https://www.niftyindices.com/Press_Release/ind...
430,2017-11-10,2017-11-20,EXIT,GAYAPROJ,"Change in Indices w.e.f. November 20, 2017",NIFTY 500 Index,https://www.niftyindices.com/Press_Release/ind...
532,2018-08-28,2018-09-28,ENTRY,GAYAPROJ,"Change in Indices w.e.f. September 28, 2018",NIFTY 500,https://www.niftyindices.com/Press_Release/ind...
707,2020-02-18,2020-03-27,ENTRY,FLUOROCHEM,Revision in criteria and replacements in Indices,NIFTY 500,https://www.niftyindices.com/Press_Release/ind...
717,2020-02-18,2020-03-27,ENTRY,SWSOLAR,Revision in criteria and replacements in Indices,NIFTY 500,https://www.niftyindices.com/Press_Release/ind...
722,2020-03-12,2020-03-27,EXIT,GAYAPROJ,"Replacement in indices w.e.f. March 27, 2020",NIFTY 500,https://www.niftyindices.com/Press_Release/ind...
750,2020-06-10,2020-06-26,ENTRY,ALKYLAMINE,"Change in indices w.e.f. June 26, 2020",NIFTY 500,https://www.niftyindices.com/Press_Release/ind...
758,2020-06-10,2020-06-26,ENTRY,FLUOROCHEM,"Change in indices w.e.f. June 26, 2020",NIFTY 500,https://www.niftyindices.com/Press_Release/ind...
728,2020-06-10,2020-06-26,EXIT,GAYAPROJ,"Change in indices w.e.f. June 26, 2020",NIFTY 500,https://www.niftyindices.com/Press_Release/ind...
809,2020-08-20,2020-09-25,ENTRY,YESBANK,Revision in criteria and replacements in Indices,NIFTY 500,https://www.niftyindices.com/Press_Release/ind...



Diagnostic complete. No files or data were modified.


In [52]:
from pathlib import Path
import pandas as pd

BASE = Path("/content/nifty500_pit_data")
EVENT_FILE = BASE / "nifty500_official_event_candidates_v5.csv"

events = pd.read_csv(EVENT_FILE, dtype=str, keep_default_na=False)
events["_Ledger_Row_Index"] = events.index

pub = pd.to_datetime(events["Published_Date"], errors="coerce").dt.strftime("%Y-%m-%d")
eff = pd.to_datetime(events["Effective_Date"], errors="coerce").dt.strftime("%Y-%m-%d")
symbol = events["Canonical_Symbol"].str.upper().str.strip()
role = events["Role"].str.upper().str.strip()
url = events["Source_URL"].str.lower()

# Find the original SWSOLAR entry that was planned for March 27.
old_swsolar_mask = (
    pub.eq("2020-02-18")
    & eff.eq("2020-03-27")
    & symbol.eq("SWSOLAR")
    & role.eq("ENTRY")
)

old_swsolar = events.loc[old_swsolar_mask]

if len(old_swsolar) != 1:
    raise ValueError(
        "Expected exactly one February 18 SWSOLAR entry planned for March 27; "
        f"found {len(old_swsolar)}. Stop and inspect before staging."
    )

# Helper to locate the confirmed June 26 entries/exits already in v5.
def followup_rows(ticker, action):
    mask = (
        pub.eq("2020-06-10")
        & eff.eq("2020-06-26")
        & symbol.eq(ticker)
        & role.eq(action)
        & url.str.contains("ind_prs10062020", regex=False)
    )
    return events.loc[mask, "_Ledger_Row_Index"].astype(str).tolist()

decisions = []

def add_record(
    record_type, published_date, effective_date, action, ticker,
    company, source_url, disposition, active, reason,
    target_row="", corrected_date="", control_url="",
    related_rows=""
):
    decisions.append({
        "Record_Type": record_type,
        "Published_Date": published_date,
        "Effective_Date": effective_date,
        "Corrected_Effective_Date": corrected_date,
        "Role": action,
        "Canonical_Symbol": ticker,
        "Company_Name": company,
        "Source_URL": source_url,
        "Control_Source_URL": control_url,
        "Target_Ledger_Row_Index": target_row,
        "Related_Active_Ledger_Row_Indices": related_rows,
        "Proposed_Disposition": disposition,
        "Active_Event": active,
        "Reason": reason,
    })

# 1. Supersede the original SWSOLAR date with the officially advanced date.
old = old_swsolar.iloc[0]
add_record(
    record_type="SUPERSEDE_EXISTING_EVENT",
    published_date="2020-02-18",
    effective_date="2020-03-27",
    corrected_date="2020-03-19",
    action="ENTRY",
    ticker="SWSOLAR",
    company="Sterling And Wilson Solar Ltd.",
    source_url="https://www.niftyindices.com/Press_Release/ind_prs18022020.pdf",
    control_url="https://www.niftyindices.com/Press_Release/ind_prs19032020.pdf",
    target_row=str(old["_Ledger_Row_Index"]),
    disposition="SUPERSEDED_BY_ADVANCED_EFFECTIVE_DATE",
    active="NO_AS_WRITTEN",
    reason=(
        "March 19 release explicitly advances this NIFTY 500 inclusion "
        "from March 27 to March 19."
    )
)

# 2. Add the two missing, active March 19 membership events.
add_record(
    record_type="ADD_MANUAL_EVENT",
    published_date="2020-03-16",
    effective_date="2020-03-19",
    action="EXIT",
    ticker="YESBANK",
    company="Yes Bank Ltd.",
    source_url="https://www.niftyindices.com/Press_Release/ind_prs16032020.pdf",
    disposition="ACTIVE_CONFIRMED_EVENT",
    active="YES",
    reason=(
        "Official March 16 notice accelerates Yes Bank's removal from all "
        "NIFTY equity indices effective March 19."
    )
)

add_record(
    record_type="ADD_MANUAL_EVENT",
    published_date="2020-03-19",
    effective_date="2020-03-19",
    action="ENTRY",
    ticker="SWSOLAR",
    company="Sterling And Wilson Solar Ltd.",
    source_url="https://www.niftyindices.com/Press_Release/ind_prs19032020.pdf",
    disposition="ACTIVE_CONFIRMED_EVENT",
    active="YES",
    reason=(
        "Official March 19 release confirms SWSOLAR inclusion in NIFTY 500 "
        "effective March 19 instead of March 27."
    )
)

# 3. Record ALKYLAMINE's old March 27 proposal as void, not active.
add_record(
    record_type="RECORD_VOIDED_PROPOSAL",
    published_date="2020-03-19",
    effective_date="2020-03-27",
    action="ENTRY",
    ticker="ALKYLAMINE",
    company="Alkyl Amines Chemicals Ltd.",
    source_url="https://www.niftyindices.com/Press_Release/ind_prs19032020.pdf",
    control_url="https://www.niftyindices.com/Press_Release/ind_prs13052020.pdf",
    related_rows=",".join(followup_rows("ALKYLAMINE", "ENTRY")),
    disposition="VOID_BY_2020-05-13",
    active="NO",
    reason=(
        "The March 19 release proposes this inclusion for March 27. "
        "The May 13 notice nullifies the covered March 27 replacements; "
        "the fresh June 26 entry is already present in v5."
    )
)

# 4. Record the March 23 concentration-norm changes as deferred proposals.
add_record(
    record_type="RECORD_DEFERRED_PROPOSAL",
    published_date="2020-03-23",
    effective_date="2020-03-27",
    action="EXIT",
    ticker="GAYAPROJ",
    company="Gayatri Projects Ltd.",
    source_url="https://www.niftyindices.com/Press_Release/ind_prs23032020.pdf",
    control_url="https://www.niftyindices.com/Press_Release/ind_prs25032020.pdf",
    related_rows=",".join(followup_rows("GAYAPROJ", "EXIT")),
    disposition="DEFERRED_BY_2020-03-25",
    active="NO",
    reason=(
        "The March 23 concentration-norm adjustment was explicitly deferred "
        "on March 25. The June 26 GAYAPROJ exit is already present in v5."
    )
)

add_record(
    record_type="RECORD_DEFERRED_PROPOSAL",
    published_date="2020-03-23",
    effective_date="2020-03-27",
    action="ENTRY",
    ticker="FLUOROCHEM",
    company="Gujarat Fluorochemicals Ltd.",
    source_url="https://www.niftyindices.com/Press_Release/ind_prs23032020.pdf",
    control_url="https://www.niftyindices.com/Press_Release/ind_prs25032020.pdf",
    related_rows=",".join(followup_rows("FLUOROCHEM", "ENTRY")),
    disposition="DEFERRED_BY_2020-03-25",
    active="NO",
    reason=(
        "The March 23 concentration-norm adjustment was explicitly deferred "
        "on March 25. The June 26 FLUOROCHEM entry is already present in v5."
    )
)

# Save a versioned sidecar; do not alter the active event ledger.
def next_version_path(stem):
    n = 1
    while True:
        path = BASE / f"{stem}_v{n}.csv"
        if not path.exists():
            return path
        n += 1

out = next_version_path("nifty500_event_march2020_manual_corrections")
decision_df = pd.DataFrame(decisions)
decision_df.to_csv(out, index=False)

print("=" * 78)
print("MARCH 2020 MANUAL CORRECTIONS — STAGED ONLY")
print("=" * 78)
print(f"Original event-ledger rows: {len(events):,}")
print(f"Correction/disposition records staged: {len(decision_df)}")

print("\nStaged decisions:")
display(decision_df[[
    "Record_Type", "Published_Date", "Effective_Date",
    "Corrected_Effective_Date", "Role", "Canonical_Symbol",
    "Target_Ledger_Row_Index", "Proposed_Disposition",
    "Active_Event", "Related_Active_Ledger_Row_Indices"
]])

print("\nJune 26 follow-up checks:")
for ticker, action in [
    ("ALKYLAMINE", "ENTRY"),
    ("GAYAPROJ", "EXIT"),
    ("FLUOROCHEM", "ENTRY"),
]:
    matches = followup_rows(ticker, action)
    print(f"{ticker:12s} {action:5s}: {matches or 'NOT FOUND — needs investigation'}")

print("\nSaved:", out)
print("No original event, interval, price, or membership file was modified.")

MARCH 2020 MANUAL CORRECTIONS — STAGED ONLY
Original event-ledger rows: 1,486
Correction/disposition records staged: 6

Staged decisions:


,Record_Type,Published_Date,Effective_Date,Corrected_Effective_Date,Role,Canonical_Symbol,Target_Ledger_Row_Index,Proposed_Disposition,Active_Event,Related_Active_Ledger_Row_Indices
0,SUPERSEDE_EXISTING_EVENT,2020-02-18,2020-03-27,2020-03-19,ENTRY,SWSOLAR,717,SUPERSEDED_BY_ADVANCED_EFFECTIVE_DATE,NO_AS_WRITTEN,
1,ADD_MANUAL_EVENT,2020-03-16,2020-03-19,,EXIT,YESBANK,,ACTIVE_CONFIRMED_EVENT,YES,
2,ADD_MANUAL_EVENT,2020-03-19,2020-03-19,,ENTRY,SWSOLAR,,ACTIVE_CONFIRMED_EVENT,YES,
3,RECORD_VOIDED_PROPOSAL,2020-03-19,2020-03-27,,ENTRY,ALKYLAMINE,,VOID_BY_2020-05-13,NO,750
4,RECORD_DEFERRED_PROPOSAL,2020-03-23,2020-03-27,,EXIT,GAYAPROJ,,DEFERRED_BY_2020-03-25,NO,728
5,RECORD_DEFERRED_PROPOSAL,2020-03-23,2020-03-27,,ENTRY,FLUOROCHEM,,DEFERRED_BY_2020-03-25,NO,758



June 26 follow-up checks:
ALKYLAMINE   ENTRY: ['750']
GAYAPROJ     EXIT : ['728']
FLUOROCHEM   ENTRY: ['758']

Saved: /content/nifty500_pit_data/nifty500_event_march2020_manual_corrections_v1.csv
No original event, interval, price, or membership file was modified.


In [53]:
from pathlib import Path
import pandas as pd

BASE = Path("/content/nifty500_pit_data")
EVENT_FILE = BASE / "nifty500_official_event_candidates_v5.csv"

if not EVENT_FILE.exists():
    raise FileNotFoundError(f"Missing source ledger: {EVENT_FILE}")

original = pd.read_csv(EVENT_FILE, dtype=str, keep_default_na=False)

required = {
    "Published_Date", "Effective_Date", "Role", "Company_Name",
    "Raw_Symbol", "Canonical_Symbol", "Release_Title",
    "Section_Title", "Action_Heading", "Effective_Date_Source",
    "Date_Override_Reason", "Source_URL"
}
missing = required - set(original.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

pub = pd.to_datetime(original["Published_Date"], errors="coerce").dt.strftime("%Y-%m-%d")
eff = pd.to_datetime(original["Effective_Date"], errors="coerce").dt.strftime("%Y-%m-%d")
role = original["Role"].str.upper().str.strip()
symbol = original["Canonical_Symbol"].str.upper().str.strip()
url = original["Source_URL"].str.lower()
heading = original["Action_Heading"].str.lower().str.strip()

section_norm = (
    original["Section_Title"].str.upper()
    .str.replace(r"[^A-Z0-9]+", " ", regex=True)
    .str.strip()
)
is_nifty500_section = section_norm.isin({
    "NIFTY 500", "NIFTY 500 INDEX",
    "CNX 500", "CNX 500 INDEX"
})

# --------------------------------------------------------------
# 1. Preflight: only proceed when each known issue matches exactly.
# --------------------------------------------------------------
feb18_mask = (
    pub.eq("2020-02-18")
    & eff.eq("2020-03-27")
    & url.str.contains("ind_prs18022020", regex=False)
    & is_nifty500_section
)

mar12_mask = (
    pub.eq("2020-03-12")
    & eff.eq("2020-03-27")
    & url.str.contains("ind_prs12032020", regex=False)
    & is_nifty500_section
)

swsolar_old_mask = (
    feb18_mask & symbol.eq("SWSOLAR") & role.eq("ENTRY")
)

adani_mask = (
    pub.eq("2015-02-20")
    & eff.eq("2015-03-27")
    & url.str.contains("ind_prs20022015", regex=False)
    & symbol.eq("ADANIPORTS")
)

crom_mask = (
    pub.eq("2015-08-24")
    & eff.eq("2015-09-28")
    & url.str.contains("ind_prs24082015", regex=False)
    & symbol.eq("CROMPGREAV")
    & role.eq("EXIT")
)

checks = {
    "February 18 NIFTY 500 rows for March 27": (int(feb18_mask.sum()), 40),
    "March 12 NIFTY 500 rows for March 27": (int(mar12_mask.sum()), 2),
    "Old SWSOLAR entry to supersede": (int(swsolar_old_mask.sum()), 1),
    "ADANIPORTS wrong-section rows": (int(adani_mask.sum()), 2),
    "CROMPGREAV duplicate-group rows": (int(crom_mask.sum()), 2),
}

print("=" * 78)
print("EVENT LEDGER V6 — PREFLIGHT")
print("=" * 78)
for label, (actual, expected) in checks.items():
    print(f"{label}: {actual} (expected {expected})")

if any(actual != expected for actual, expected in checks.values()):
    raise ValueError(
        "Preflight mismatch. Nothing was saved. Inspect the listed rows before proceeding."
    )

feb_counts = original.loc[feb18_mask, "Role"].str.upper().value_counts().to_dict()
mar12_counts = original.loc[mar12_mask, "Role"].str.upper().value_counts().to_dict()

if feb_counts != {"ENTRY": 20, "EXIT": 20}:
    raise ValueError(f"Unexpected Feb 18 role counts: {feb_counts}")
if mar12_counts != {"ENTRY": 1, "EXIT": 1}:
    raise ValueError(f"Unexpected Mar 12 role counts: {mar12_counts}")

# Keep the CROMPGREAV row with the plural NIFTY 500 exclusion heading.
crom_plural_indices = original.index[
    crom_mask & heading.str.contains("companies are being excluded", regex=False)
].tolist()

if len(crom_plural_indices) != 1:
    raise ValueError(
        "Could not uniquely identify the verified CROMPGREAV row. Nothing was saved."
    )

crom_keep_idx = crom_plural_indices[0]

# --------------------------------------------------------------
# 2. Prepare explicit dispositions for rows removed from v5.
# --------------------------------------------------------------
drop_reasons = {}

for idx in original.index[adani_mask]:
    drop_reasons[idx] = (
        "EXCLUDE_WRONG_INDEX_SECTION: ADANIPORTS rows came from "
        "non-CNX-500 sections in the February 20, 2015 release."
    )

for idx in original.index[crom_mask]:
    if idx != crom_keep_idx:
        drop_reasons[idx] = (
            "EXCLUDE_DUPLICATE_EXTRACTION: same CROMPGREAV exit was "
            "extracted twice from the August 24, 2015 release."
        )

for idx in original.index[feb18_mask]:
    if bool(swsolar_old_mask.loc[idx]):
        drop_reasons[idx] = (
            "SUPERSEDED: February 18 SWSOLAR entry planned for March 27 "
            "was advanced to March 19 by the official March 16 notice."
        )
    else:
        drop_reasons[idx] = (
            "VOID_BY_2020-05-13: March 27 replacement proposal from "
            "the February 18 review was declared null and void."
        )

for idx in original.index[mar12_mask]:
    drop_reasons[idx] = (
        "VOID_BY_2020-05-13: March 27 replacement proposal from "
        "the March 12 release was declared null and void."
    )

if len(drop_reasons) != 45:
    raise ValueError(
        f"Expected 45 unique rows to remove; found {len(drop_reasons)}. Nothing was saved."
    )

# --------------------------------------------------------------
# 3. Add the two confirmed NIFTY 500 changes effective March 19.
# Both are directly listed in the official March 16 release.
# --------------------------------------------------------------
march16_url = (
    "https://www.niftyindices.com/Press_Release/ind_prs16032020.pdf"
)

def make_event(role_value, company, ticker, action_heading):
    row = {col: "" for col in original.columns}
    row.update({
        "Published_Date": "2020-03-16",
        "Effective_Date": "2020-03-19",
        "Role": role_value,
        "Company_Name": company,
        "Raw_Symbol": ticker,
        "Canonical_Symbol": ticker,
        "Release_Title": "Replacement in indices",
        "Section_Title": "NIFTY 500",
        "Action_Heading": action_heading,
        "Effective_Date_Source": "explicit_date_text",
        "Date_Override_Reason": (
            "Reconstructed from the official March 16, 2020 release; "
            "the source explicitly states the changes are effective March 19, 2020."
        ),
        "Source_URL": march16_url,
    })
    return row

added_events = pd.DataFrame([
    make_event(
        "EXIT",
        "Yes Bank Ltd.",
        "YESBANK",
        "The following company is being excluded"
    ),
    make_event(
        "ENTRY",
        "Sterling And Wilson Solar Ltd.",
        "SWSOLAR",
        "The following company is being included"
    ),
], columns=original.columns)

# The original source row for SWSOLAR is removed above, so the entry
# will occur exactly once in the new ledger, with the corrected date.

# --------------------------------------------------------------
# 4. Construct a separate staged ledger; original v5 stays unchanged.
# --------------------------------------------------------------
removed_indices = sorted(drop_reasons)
remaining = original.drop(index=removed_indices).copy()

staged = pd.concat(
    [remaining, added_events],
    ignore_index=True
)

# --------------------------------------------------------------
# 5. Quality checks before saving.
# --------------------------------------------------------------
spub = pd.to_datetime(staged["Published_Date"], errors="coerce").dt.strftime("%Y-%m-%d")
seff = pd.to_datetime(staged["Effective_Date"], errors="coerce").dt.strftime("%Y-%m-%d")
srole = staged["Role"].str.upper().str.strip()
ssymbol = staged["Canonical_Symbol"].str.upper().str.strip()
surl = staged["Source_URL"].str.lower()
ssection = (
    staged["Section_Title"].str.upper()
    .str.replace(r"[^A-Z0-9]+", " ", regex=True)
    .str.strip()
)
snifty500 = ssection.isin({
    "NIFTY 500", "NIFTY 500 INDEX",
    "CNX 500", "CNX 500 INDEX"
})

# No superseded or voided March 27 rows from the two releases remain.
assert not (
    seff.eq("2020-03-27")
    & snifty500
    & (
        surl.str.contains("ind_prs18022020", regex=False)
        | surl.str.contains("ind_prs12032020", regex=False)
    )
).any(), "Voided/superseded March 27 rows remain."

# The false positive ADANIPORTS rows are removed.
assert not (
    seff.eq("2015-03-27")
    & ssymbol.eq("ADANIPORTS")
    & surl.str.contains("ind_prs20022015", regex=False)
).any(), "ADANIPORTS wrong-section rows remain."

# Exactly one verified CROMPGREAV exit remains in that release.
assert int((
    seff.eq("2015-09-28")
    & ssymbol.eq("CROMPGREAV")
    & srole.eq("EXIT")
    & surl.str.contains("ind_prs24082015", regex=False)
).sum()) == 1, "CROMPGREAV duplicate check failed."

# The active March 19 NIFTY 500 events are exactly one YESBANK exit
# and one SWSOLAR entry from the March 16 official release.
for ticker, action in [("YESBANK", "EXIT"), ("SWSOLAR", "ENTRY")]:
    n = int((
        seff.eq("2020-03-19")
        & ssymbol.eq(ticker)
        & srole.eq(action)
        & snifty500
        & surl.str.contains("ind_prs16032020", regex=False)
    ).sum())
    assert n == 1, f"Expected one March 19 {ticker} {action}; found {n}."

# June 26 active list remains unchanged: 25 entries and 25 exits.
june_mask = (
    seff.eq("2020-06-26")
    & surl.str.contains("ind_prs10062020", regex=False)
    & snifty500
)
june_counts = srole[june_mask].value_counts().to_dict()
assert june_counts == {"ENTRY": 25, "EXIT": 25}, (
    f"June 26 review count changed unexpectedly: {june_counts}"
)

expected_n = len(original) - len(drop_reasons) + len(added_events)
assert len(staged) == expected_n, (
    f"Unexpected output row count: {len(staged)} vs expected {expected_n}"
)

# --------------------------------------------------------------
# 6. Write the new ledger and an auditable change log using new names.
# --------------------------------------------------------------
def next_version_path(stem):
    version = 1
    while True:
        path = BASE / f"{stem}_v{version}.csv"
        if not path.exists():
            return path
        version += 1

ledger_out = next_version_path("nifty500_official_event_candidates")
change_out = next_version_path("nifty500_event_ledger_change_log")

change_records = []

for idx, reason in drop_reasons.items():
    rec = original.loc[idx].to_dict()
    rec["Original_Ledger_Row_Index"] = idx
    rec["Change_Type"] = "REMOVE_FROM_ACTIVE_STAGED_LEDGER"
    rec["Change_Reason"] = reason
    change_records.append(rec)

for _, row in added_events.iterrows():
    rec = row.to_dict()
    rec["Original_Ledger_Row_Index"] = ""
    rec["Change_Type"] = "ADD_CONFIRMED_MARCH_19_EVENT"
    rec["Change_Reason"] = (
        "Official March 16, 2020 release confirms the NIFTY 500 change "
        "effective March 19, 2020."
    )
    change_records.append(rec)

change_log = pd.DataFrame(change_records)

# Save only after all assertions have passed.
staged.to_csv(ledger_out, index=False)
change_log.to_csv(change_out, index=False)

print("\n" + "=" * 78)
print("STAGED LEDGER CREATED — QA PASSED")
print("=" * 78)
print(f"Input ledger rows: {len(original):,}")
print(f"Rows removed from active staged ledger: {len(drop_reasons)}")
print(f"Confirmed March 19 events added: {len(added_events)}")
print(f"Output ledger rows: {len(staged):,}")

print("\nRemoved-row disposition counts:")
display(
    change_log[change_log["Change_Type"].eq("REMOVE_FROM_ACTIVE_STAGED_LEDGER")]
    ["Change_Reason"].value_counts()
    .rename_axis("Disposition")
    .reset_index(name="Rows")
)

print("\nMarch 19 events in staged ledger:")
display(staged[
    seff.eq("2020-03-19")
    & snifty500
    & ssymbol.isin(["YESBANK", "SWSOLAR"])
][[
    "Published_Date", "Effective_Date", "Role",
    "Canonical_Symbol", "Release_Title", "Source_URL"
]])

print("\nJune 26 verification:")
print("Entries:", june_counts.get("ENTRY", 0))
print("Exits:  ", june_counts.get("EXIT", 0))

print("\nFiles created:")
print(ledger_out)
print(change_out)
print("\nOriginal v5 ledger and all original price/interval files remain unchanged.")

EVENT LEDGER V6 — PREFLIGHT
February 18 NIFTY 500 rows for March 27: 40 (expected 40)
March 12 NIFTY 500 rows for March 27: 2 (expected 2)
Old SWSOLAR entry to supersede: 1 (expected 1)
ADANIPORTS wrong-section rows: 2 (expected 2)
CROMPGREAV duplicate-group rows: 2 (expected 2)

STAGED LEDGER CREATED — QA PASSED
Input ledger rows: 1,486
Rows removed from active staged ledger: 45
Confirmed March 19 events added: 2
Output ledger rows: 1,443

Removed-row disposition counts:


,Disposition,Rows
0,VOID_BY_2020-05-13: March 27 replacement propo...,39
1,EXCLUDE_WRONG_INDEX_SECTION: ADANIPORTS rows c...,2
2,VOID_BY_2020-05-13: March 27 replacement propo...,2
3,EXCLUDE_DUPLICATE_EXTRACTION: same CROMPGREAV ...,1
4,SUPERSEDED: February 18 SWSOLAR entry planned ...,1



March 19 events in staged ledger:


,Published_Date,Effective_Date,Role,Canonical_Symbol,Release_Title,Source_URL
1441,2020-03-16,2020-03-19,EXIT,YESBANK,Replacement in indices,https://www.niftyindices.com/Press_Release/ind...
1442,2020-03-16,2020-03-19,ENTRY,SWSOLAR,Replacement in indices,https://www.niftyindices.com/Press_Release/ind...



June 26 verification:
Entries: 25
Exits:   25

Files created:
/content/nifty500_pit_data/nifty500_official_event_candidates_v6.csv
/content/nifty500_pit_data/nifty500_event_ledger_change_log_v1.csv

Original v5 ledger and all original price/interval files remain unchanged.


In [54]:
from pathlib import Path
import pandas as pd

BASE = Path("/content/nifty500_pit_data")
LEDGER = BASE / "nifty500_official_event_candidates_v6.csv"

if not LEDGER.exists():
    raise FileNotFoundError(f"Staged v6 ledger not found: {LEDGER}")

df = pd.read_csv(LEDGER, dtype=str, keep_default_na=False)
df["_Row"] = df.index

# Normalize for audit calculations without changing source columns.
df["_Pub"] = pd.to_datetime(df["Published_Date"], errors="coerce")
df["_Eff"] = pd.to_datetime(df["Effective_Date"], errors="coerce")
df["_Symbol"] = df["Canonical_Symbol"].str.upper().str.strip()
df["_Role"] = df["Role"].str.upper().str.strip()

print("=" * 78)
print("V6 EVENT LEDGER CONSISTENCY AUDIT — READ ONLY")
print("=" * 78)
print(f"Ledger: {LEDGER}")
print(f"Event rows: {len(df):,}")

# Version-safe report names.
def next_path(stem):
    n = 1
    while True:
        p = BASE / f"{stem}_v{n}.csv"
        if not p.exists():
            return p
        n += 1

# 1. Basic validity checks.
invalid_dates = df[df["_Pub"].isna() | df["_Eff"].isna()].copy()
invalid_roles = df[~df["_Role"].isin(["ENTRY", "EXIT"])].copy()
invalid_symbols = df[df["_Symbol"].eq("")].copy()

print("\n1. BASIC VALIDITY")
print("Invalid/missing publication or effective dates:", len(invalid_dates))
print("Invalid/missing roles:", len(invalid_roles))
print("Missing canonical symbols:", len(invalid_symbols))

# 2. Duplicate event keys: same effective date, role and symbol.
valid = df[
    df["_Eff"].notna()
    & df["_Symbol"].ne("")
    & df["_Role"].isin(["ENTRY", "EXIT"])
].copy()

duplicate_counts = (
    valid.groupby(["Effective_Date", "_Role", "_Symbol"], dropna=False)
    .size()
    .rename("Rows")
    .reset_index()
)
duplicate_groups = duplicate_counts[duplicate_counts["Rows"] > 1].copy()

if len(duplicate_groups):
    duplicate_rows = valid.merge(
        duplicate_groups[["Effective_Date", "_Role", "_Symbol", "Rows"]],
        on=["Effective_Date", "_Role", "_Symbol"],
        how="inner"
    )
else:
    duplicate_rows = valid.iloc[0:0].copy()
    duplicate_rows["Rows"] = pd.Series(dtype="int64")

print("\n2. DUPLICATE EVENT KEYS")
print("Duplicate groups:", len(duplicate_groups))
print("Rows in duplicate groups:", len(duplicate_rows))

# 3. Same effective date and symbol with both ENTRY and EXIT roles.
role_counts = (
    valid.groupby(["Effective_Date", "_Symbol"])["_Role"]
    .nunique()
    .rename("Distinct_Roles")
    .reset_index()
)
conflict_groups = role_counts[role_counts["Distinct_Roles"] > 1].copy()

if len(conflict_groups):
    conflict_rows = valid.merge(
        conflict_groups[["Effective_Date", "_Symbol"]],
        on=["Effective_Date", "_Symbol"],
        how="inner"
    ).sort_values(["Effective_Date", "_Symbol", "_Role"])
else:
    conflict_rows = valid.iloc[0:0].copy()

print("\n3. SAME-DATE ROLE CONFLICTS")
print("Symbol/date groups with both roles:", len(conflict_groups))
print("Rows in conflict groups:", len(conflict_rows))

# 4. Effective dates earlier than publication dates.
before_pub = df[
    df["_Pub"].notna()
    & df["_Eff"].notna()
    & (df["_Eff"] < df["_Pub"])
].copy()

print("\n4. EVENTS EFFECTIVE BEFORE PUBLICATION")
print("Rows:", len(before_pub))

# 5. Repeated same-role actions on consecutive distinct event dates.
# These are REVIEW FLAGS, not automatic errors.
sequence_rows = valid[
    valid["_Eff"].notna()
].copy()

sequence_rows = sequence_rows.sort_values(
    ["_Symbol", "_Eff", "_Role", "_Row"]
)

# Build one role set per symbol/date, so multiple actions on the same
# date do not incorrectly count as a previous date in the sequence.
date_role_sets = (
    sequence_rows.groupby(["_Symbol", "_Eff"])["_Role"]
    .agg(lambda s: sorted(set(s)))
    .reset_index(name="Roles")
    .sort_values(["_Symbol", "_Eff"])
)

repeat_flags = []
for ticker, group in date_role_sets.groupby("_Symbol", sort=True):
    group = group.sort_values("_Eff").reset_index(drop=True)
    for i in range(1, len(group)):
        prev = group.iloc[i - 1]
        curr = group.iloc[i]

        shared_roles = sorted(set(prev["Roles"]) & set(curr["Roles"]))
        if shared_roles:
            repeat_flags.append({
                "Symbol": ticker,
                "Previous_Date": prev["_Eff"].strftime("%Y-%m-%d"),
                "Previous_Roles": ",".join(prev["Roles"]),
                "Current_Date": curr["_Eff"].strftime("%Y-%m-%d"),
                "Current_Roles": ",".join(curr["Roles"]),
                "Repeated_Role": ",".join(shared_roles),
                "Issue": "REVIEW_CONSECUTIVE_SAME_ROLE",
            })

repeat_df = pd.DataFrame(repeat_flags)
print("\n5. REPEATED-ACTION REVIEW FLAGS")
print("Consecutive same-role transitions:", len(repeat_df))

# 6. Targeted March 2020 / June 2020 checks.
print("\n6. TARGETED MARCH–JUNE 2020 CHECKS")
target = valid[
    valid["_Eff"].dt.strftime("%Y-%m-%d").isin(
        ["2020-03-19", "2020-03-27", "2020-06-26"]
    )
].copy()

target["Audit_Effective_Date"] = target["_Eff"].dt.strftime("%Y-%m-%d")
target_summary = (
    target.groupby(
        ["Audit_Effective_Date", "Published_Date",
         "Release_Title", "Source_URL", "_Role"],
        dropna=False
    ).size().rename("Rows").reset_index()
)

display(target_summary.sort_values(
    ["Audit_Effective_Date", "Published_Date", "_Role"]
))

print("\nMarch 19 events:")
display(target[
    (target["Audit_Effective_Date"] == "2020-03-19")
    & target["_Symbol"].isin(["YESBANK", "SWSOLAR"])
][[
    "Published_Date", "Effective_Date", "Role",
    "Canonical_Symbol", "Source_URL"
]])

# 7. Save detailed reports without modifying the event ledger.
reports = {
    "nifty500_v6_duplicate_event_groups": duplicate_groups,
    "nifty500_v6_duplicate_event_rows": duplicate_rows,
    "nifty500_v6_same_date_role_conflicts": conflict_rows,
    "nifty500_v6_effective_before_publication": before_pub,
    "nifty500_v6_consecutive_same_role_flags": repeat_df,
    "nifty500_v6_basic_invalid_rows": pd.concat(
        [
            invalid_dates.assign(Audit_Issue="INVALID_DATE"),
            invalid_roles.assign(Audit_Issue="INVALID_ROLE"),
            invalid_symbols.assign(Audit_Issue="MISSING_SYMBOL"),
        ],
        ignore_index=True
    ),
}

print("\n7. QA REPORT FILES")
for stem, report in reports.items():
    path = next_path(stem)
    report.to_csv(path, index=False)
    print(f"{stem}: {len(report)} rows -> {path}")

print("\nAudit complete. v6 and all original data files remain unchanged.")

V6 EVENT LEDGER CONSISTENCY AUDIT — READ ONLY
Ledger: /content/nifty500_pit_data/nifty500_official_event_candidates_v6.csv
Event rows: 1,443

1. BASIC VALIDITY
Invalid/missing publication or effective dates: 0
Invalid/missing roles: 0
Missing canonical symbols: 0

2. DUPLICATE EVENT KEYS
Duplicate groups: 0
Rows in duplicate groups: 0

3. SAME-DATE ROLE CONFLICTS
Symbol/date groups with both roles: 0
Rows in conflict groups: 0

4. EVENTS EFFECTIVE BEFORE PUBLICATION
Rows: 0

5. REPEATED-ACTION REVIEW FLAGS
Consecutive same-role transitions: 16

6. TARGETED MARCH–JUNE 2020 CHECKS


,Audit_Effective_Date,Published_Date,Release_Title,Source_URL,_Role,Rows
0,2020-03-19,2020-03-16,Replacement in indices,https://www.niftyindices.com/Press_Release/ind...,ENTRY,1
1,2020-03-19,2020-03-16,Replacement in indices,https://www.niftyindices.com/Press_Release/ind...,EXIT,1
2,2020-06-26,2020-06-10,"Change in indices w.e.f. June 26, 2020",https://www.niftyindices.com/Press_Release/ind...,ENTRY,25
3,2020-06-26,2020-06-10,"Change in indices w.e.f. June 26, 2020",https://www.niftyindices.com/Press_Release/ind...,EXIT,25



March 19 events:


,Published_Date,Effective_Date,Role,Canonical_Symbol,Source_URL
1441,2020-03-16,2020-03-19,EXIT,YESBANK,https://www.niftyindices.com/Press_Release/ind...
1442,2020-03-16,2020-03-19,ENTRY,SWSOLAR,https://www.niftyindices.com/Press_Release/ind...



7. QA REPORT FILES
nifty500_v6_duplicate_event_groups: 0 rows -> /content/nifty500_pit_data/nifty500_v6_duplicate_event_groups_v1.csv
nifty500_v6_duplicate_event_rows: 0 rows -> /content/nifty500_pit_data/nifty500_v6_duplicate_event_rows_v1.csv
nifty500_v6_same_date_role_conflicts: 0 rows -> /content/nifty500_pit_data/nifty500_v6_same_date_role_conflicts_v1.csv
nifty500_v6_effective_before_publication: 0 rows -> /content/nifty500_pit_data/nifty500_v6_effective_before_publication_v1.csv
nifty500_v6_consecutive_same_role_flags: 16 rows -> /content/nifty500_pit_data/nifty500_v6_consecutive_same_role_flags_v1.csv
nifty500_v6_basic_invalid_rows: 0 rows -> /content/nifty500_pit_data/nifty500_v6_basic_invalid_rows_v1.csv

Audit complete. v6 and all original data files remain unchanged.


In [55]:
from pathlib import Path
import pandas as pd
import re

BASE = Path("/content/nifty500_pit_data")
LEDGER_FILE = BASE / "nifty500_official_event_candidates_v6.csv"
FLAGS_FILE = BASE / "nifty500_v6_consecutive_same_role_flags_v1.csv"
CATALOG_FILE = BASE / "nifty500_official_release_catalog_v1.csv"
CACHE_DIR = BASE / "official_release_text_cache"

for p in [LEDGER_FILE, FLAGS_FILE, CATALOG_FILE]:
    if not p.exists():
        raise FileNotFoundError(f"Required file not found: {p}")

events = pd.read_csv(LEDGER_FILE, dtype=str, keep_default_na=False)
flags = pd.read_csv(FLAGS_FILE, dtype=str, keep_default_na=False)
catalog = pd.read_csv(CATALOG_FILE, dtype=str, keep_default_na=False)

events["_symbol"] = events["Canonical_Symbol"].str.upper().str.strip()
events["_role"] = events["Role"].str.upper().str.strip()
events["_effective"] = pd.to_datetime(
    events["Effective_Date"], errors="coerce"
).dt.strftime("%Y-%m-%d")

catalog["_url_key"] = catalog["URL"].str.strip().str.lower()
catalog["_catalog_position"] = range(len(catalog))

# Reconstruct the cache index from catalog order, but verify the
# cached text's publication date before trusting a match.
cache_by_url = {}
for _, row in catalog.iterrows():
    position = int(row["_catalog_position"])
    candidate = CACHE_DIR / f"release_{position + 1:03d}.txt"
    if candidate.exists():
        cache_by_url[row["_url_key"]] = candidate

def compact(text, limit=420):
    return re.sub(r"\s+", " ", str(text)).strip()[:limit]

def source_snippets(event_row, ticker, max_snippets=2):
    url_key = event_row["Source_URL"].strip().lower()
    cache_path = cache_by_url.get(url_key)

    if cache_path is None or not cache_path.exists():
        return [f"[No cached text mapped for source URL: {event_row['Source_URL']}]"]

    text = cache_path.read_text(encoding="utf-8", errors="replace")
    lines = [x.strip() for x in text.splitlines() if x.strip()]

    # Validate the expected publication date is present in the file.
    published = pd.to_datetime(
        event_row["Published_Date"], errors="coerce"
    )
    if pd.notna(published):
        date_formats = [
            published.strftime("%B %d, %Y"),
            published.strftime("%B %d %Y"),
            published.strftime("%d %B %Y"),
            published.strftime("%B %-d, %Y"),
        ]
        if not any(
            fmt.lower() in text.lower()
            for fmt in date_formats
        ):
            # Avoid silently relying on a potentially wrong cache mapping.
            return [
                f"[Cache mapping not date-verified: {cache_path.name}; "
                f"expected publication date {published.date()}]"
            ]

    ticker_pattern = re.compile(
        rf"(?<![A-Z0-9]){re.escape(ticker)}(?![A-Z0-9])",
        flags=re.IGNORECASE
    )

    found_indices = [
        i for i, line in enumerate(lines)
        if ticker_pattern.search(line)
    ]

    if not found_indices:
        return [f"[Ticker {ticker} not found in {cache_path.name}]"]

    snippets = []
    for idx in found_indices[:max_snippets]:
        start = max(0, idx - 3)
        end = min(len(lines), idx + 4)
        snippets.append(
            f"{cache_path.name}: " +
            compact(" | ".join(lines[start:end]), 650)
        )

    # Flag when this source has many ticker occurrences to review.
    if len(found_indices) > max_snippets:
        snippets.append(
            f"[{len(found_indices)} total ticker occurrences; "
            f"showing first {max_snippets}]"
        )

    return snippets

# Join every previous/current flag to the exact ledger event rows.
matched_records = []
pair_summaries = []

for i, flag in flags.iterrows():
    ticker = flag["Symbol"].upper().strip()
    prev_date = flag["Previous_Date"]
    curr_date = flag["Current_Date"]

    prev_events = events[
        events["_symbol"].eq(ticker)
        & events["_effective"].eq(prev_date)
    ]
    curr_events = events[
        events["_symbol"].eq(ticker)
        & events["_effective"].eq(curr_date)
    ]

    pair_summaries.append({
        "Flag_Number": i + 1,
        "Symbol": ticker,
        "Previous_Date": prev_date,
        "Previous_Roles_Flagged": flag.get("Previous_Roles", ""),
        "Current_Date": curr_date,
        "Current_Roles_Flagged": flag.get("Current_Roles", ""),
        "Previous_Ledger_Rows": len(prev_events),
        "Current_Ledger_Rows": len(curr_events),
    })

    for period, matched in [
        ("PREVIOUS", prev_events),
        ("CURRENT", curr_events)
    ]:
        for _, row in matched.iterrows():
            matched_records.append({
                "Flag_Number": i + 1,
                "Period": period,
                "Symbol": ticker,
                "Published_Date": row["Published_Date"],
                "Effective_Date": row["Effective_Date"],
                "Role": row["Role"],
                "Company_Name": row["Company_Name"],
                "Release_Title": row["Release_Title"],
                "Section_Title": row["Section_Title"],
                "Action_Heading": row["Action_Heading"],
                "Source_URL": row["Source_URL"],
                "Cache_Snippet": " || ".join(
                    source_snippets(row, ticker)
                )
            })

pairs_df = pd.DataFrame(pair_summaries)
records_df = pd.DataFrame(matched_records)

print("=" * 78)
print("V6 REPEATED-ACTION SOURCE TRACE — READ ONLY")
print("=" * 78)
print("Repeated-action flags:", len(flags))
print("Flags with no prior event match:",
      int((pairs_df["Previous_Ledger_Rows"] == 0).sum()))
print("Flags with no current event match:",
      int((pairs_df["Current_Ledger_Rows"] == 0).sum()))

print("\n1. ALL 16 FLAGGED TRANSITIONS")
display(pairs_df)

print("\n2. EVENT ROWS AND SOURCE SNIPPETS")
if not records_df.empty:
    display(
        records_df.sort_values(
            ["Flag_Number", "Period", "Published_Date", "Role"]
        )
    )
else:
    print("No ledger rows matched the flags.")

print("\n3. FLAGGED SYMBOLS WITH MULTIPLE EVENTS ON A BOUNDARY DATE")
if not records_df.empty:
    per_date = (
        records_df.groupby(
            ["Flag_Number", "Period", "Symbol", "Effective_Date"]
        ).size().rename("Rows_For_Symbol_Date").reset_index()
    )
    display(per_date[per_date["Rows_For_Symbol_Date"] > 1])
else:
    print("No matched rows.")

print("\nDiagnostic complete: no source, ledger, price, or interval files modified.")

V6 REPEATED-ACTION SOURCE TRACE — READ ONLY
Repeated-action flags: 16
Flags with no prior event match: 0
Flags with no current event match: 0

1. ALL 16 FLAGGED TRANSITIONS


,Flag_Number,Symbol,Previous_Date,Previous_Roles_Flagged,Current_Date,Current_Roles_Flagged,Previous_Ledger_Rows,Current_Ledger_Rows
0,1,AARTIIND,2019-03-29,EXIT,2022-05-04,EXIT,1,1
1,2,ASAHIINDIA,2018-09-28,EXIT,2020-09-14,EXIT,1,1
2,3,ERIS,2018-03-13,ENTRY,2018-04-02,ENTRY,1,1
3,4,FRETAIL,2017-03-16,ENTRY,2017-03-31,ENTRY,1,1
4,5,IREDA,2024-03-28,ENTRY,2024-09-30,ENTRY,1,1
5,6,MAXINDIA,2017-03-16,ENTRY,2017-03-31,ENTRY,1,1
6,7,MFSL,2017-09-05,ENTRY,2017-09-29,ENTRY,1,1
7,8,MGL,2017-03-16,ENTRY,2017-03-31,ENTRY,1,1
8,9,MYSOREBANK,2017-03-16,EXIT,2017-03-31,EXIT,1,1
9,10,RELCAPITAL,2017-09-05,EXIT,2017-09-29,EXIT,1,1



2. EVENT ROWS AND SOURCE SNIPPETS


,Flag_Number,Period,Symbol,Published_Date,Effective_Date,Role,Company_Name,Release_Title,Section_Title,Action_Heading,Source_URL,Cache_Snippet
1,1,CURRENT,AARTIIND,2022-04-05,2022-05-04,EXIT,Aarti Industries Ltd.,"Replacements in indices w.e.f. May 04, 2022",NIFTY 500,The following company is being excluded,https://www.niftyindices.com/Press_Release/ind...,release_074.txt: Symbol | 1 | Aarti Industries...
0,1,PREVIOUS,AARTIIND,2019-02-25,2019-03-29,EXIT,Aarti Industries Ltd.,"Change in Indices w.e.f. March 29, 2019",NIFTY 500,The following companies are being excluded,https://www.niftyindices.com/Press_Release/ind...,release_042.txt: 8KMILES | 2 | Aarti Industrie...
3,2,CURRENT,ASAHIINDIA,2020-09-07,2020-09-14,EXIT,Asahi India Glass Ltd.,Replacements in Indices,NIFTY 500,The following company is being excluded,https://www.niftyindices.com/Press_Release/ind...,release_057.txt: Symbol | 1 | Asahi India Glas...
2,2,PREVIOUS,ASAHIINDIA,2018-08-28,2018-09-28,EXIT,Asahi India Glass Ltd.,"Change in Indices w.e.f. September 28, 2018",NIFTY 500,The following companies are being excluded,https://www.niftyindices.com/Press_Release/ind...,release_038.txt: ABAN | 2 | Asahi India Glass ...
5,3,CURRENT,ERIS,2018-02-21,2018-04-02,ENTRY,Eris Lifesciences Ltd.,"Change in Indices w.e.f. April 02, 2018",NIFTY 500,The following companies are being included,https://www.niftyindices.com/Press_Release/ind...,release_032.txt: Dixon Technologies (India) Lt...
4,3,PREVIOUS,ERIS,2018-03-06,2018-03-13,ENTRY,Eris Lifesciences Ltd.,Replacements in Indices,NIFTY 500,The following company is being included,https://www.niftyindices.com/Press_Release/ind...,release_033.txt: Company Name | Symbol | 1 | E...
7,4,CURRENT,FRETAIL,2017-02-16,2017-03-31,ENTRY,Future Retail Ltd.,"Change in Indices w.e.f. March 31, 2017.",NIFTY 500 Index,The following companies are being included,https://www.niftyindices.com/Press_Release/ind...,release_020.txt: FMGOETZE | 4 | Future Retail ...
6,4,PREVIOUS,FRETAIL,2017-03-07,2017-03-16,ENTRY,Future Retail Ltd.,Replacements in Indices.,NIFTY 500 Index,The following companies are being included,https://www.niftyindices.com/Press_Release/ind...,[Cache mapping not date-verified: release_021....
9,5,CURRENT,IREDA,2024-08-23,2024-09-30,ENTRY,Indian Renewable Energy Development Agency Ltd.,Replacements in indices and revision in criter...,Nifty 500,The following companies are being included,https://www.niftyindices.com/Press_Release/ind...,release_113.txt: INDGN | 12 | Indian Renewable...
8,5,PREVIOUS,IREDA,2024-02-28,2024-03-28,ENTRY,Indian Renewable Energy Development Agency Ltd.,"Replacements in indices w.e.f. March 28, 2024",Nifty 500,The following companies are being included,https://www.niftyindices.com/Press_Release/ind...,release_104.txt: HONASA | 15 | Indian Renewabl...



3. FLAGGED SYMBOLS WITH MULTIPLE EVENTS ON A BOUNDARY DATE


,Flag_Number,Period,Symbol,Effective_Date,Rows_For_Symbol_Date



Diagnostic complete: no source, ledger, price, or interval files modified.


In [56]:
from pathlib import Path
import pandas as pd

BASE = Path("/content/nifty500_pit_data")
INPUT = BASE / "nifty500_official_event_candidates_v6.csv"

if not INPUT.exists():
    raise FileNotFoundError(f"Missing input ledger: {INPUT}")

df = pd.read_csv(INPUT, dtype=str, keep_default_na=False)
pub = pd.to_datetime(df["Published_Date"], errors="coerce").dt.strftime("%Y-%m-%d")
eff = pd.to_datetime(df["Effective_Date"], errors="coerce").dt.strftime("%Y-%m-%d")
sym = df["Canonical_Symbol"].str.upper().str.strip()
role = df["Role"].str.upper().str.strip()
url = df["Source_URL"].str.lower()

section = (
    df["Section_Title"].str.upper()
    .str.replace(r"[^A-Z0-9]+", " ", regex=True)
    .str.strip()
)
is_nifty500 = section.isin({
    "NIFTY 500", "NIFTY 500 INDEX", "CNX 500", "CNX 500 INDEX"
})

# Rows in the original schedules that were moved to earlier effective dates.
rescheduled_groups = [
    {
        "mask": (
            pub.eq("2017-02-16")
            & eff.eq("2017-03-31")
            & url.str.contains("ind_prs16022017", regex=False)
            & is_nifty500
            & sym.isin(["FRETAIL", "MAXINDIA", "MGL", "MYSOREBANK", "SBBJ", "SBT"])
        ),
        "expected": 6,
        "group": "2017_MARCH_RESCHEDULE",
        "superseding_url": (
            "https://www.niftyindices.com/docs/default-source/"
            "press-release-upload/ind_prs07032017.pdf"
        ),
        "reason": (
            "March 7, 2017 official notice rescheduled these NIFTY 500 "
            "changes from March 31 to March 16, 2017."
        ),
    },
    {
        "mask": (
            pub.eq("2018-02-21")
            & eff.eq("2018-04-02")
            & url.str.contains("ind_prs21022018", regex=False)
            & is_nifty500
            & sym.isin(["ERIS", "VIDEOIND"])
        ),
        "expected": 2,
        "group": "2018_ERIS_VIDEOIND_RESCHEDULE",
        "superseding_url": (
            "https://www.niftyindices.com/Press_Release/ind_prs06032018.pdf"
        ),
        "reason": (
            "March 6, 2018 official notice advanced the Videocon exit and "
            "Eris entry from April 2 to March 13, 2018."
        ),
    },
    {
        "mask": (
            pub.eq("2017-08-28")
            & eff.eq("2017-09-29")
            & url.str.contains("ind_prs28082017", regex=False)
            & is_nifty500
            & sym.isin(["MFSL", "RELCAPITAL"])
        ),
        "expected": 2,
        "group": "2017_SEPTEMBER_RESCHEDULE",
        "superseding_url": (
            "https://www.niftyindices.com/Press_Release/ind_prs29082017.pdf"
        ),
        "reason": (
            "August 29, 2017 official notice advanced the Reliance Capital "
            "exit and Max Financial entry from September 29 to September 5, 2017."
        ),
    },
]

remove_reasons = {}
for item in rescheduled_groups:
    matches = df.index[item["mask"]].tolist()
    print(f"{item['group']}: matched {len(matches)} rows; expected {item['expected']}")
    if len(matches) != item["expected"]:
        raise ValueError(
            f"Preflight failed for {item['group']}. Nothing saved."
        )
    for idx in matches:
        remove_reasons[idx] = {
            "group": item["group"],
            "reason": item["reason"],
            "superseding_url": item["superseding_url"],
        }

if len(remove_reasons) != 10:
    raise ValueError(
        f"Expected 10 unique superseded rows, found {len(remove_reasons)}. Nothing saved."
    )

# Verify that the earlier effective events are already present before
# removing these superseded later-dated rows.
expected_active_events = [
    ("2017-03-16", "FRETAIL", "ENTRY", "ind_prs07032017"),
    ("2017-03-16", "MAXINDIA", "ENTRY", "ind_prs07032017"),
    ("2017-03-16", "MGL", "ENTRY", "ind_prs07032017"),
    ("2017-03-16", "MYSOREBANK", "EXIT", "ind_prs07032017"),
    ("2017-03-16", "SBBJ", "EXIT", "ind_prs07032017"),
    ("2017-03-16", "SBT", "EXIT", "ind_prs07032017"),
    ("2018-03-13", "ERIS", "ENTRY", "ind_prs06032018"),
    ("2018-03-13", "VIDEOIND", "EXIT", "ind_prs06032018"),
    ("2017-09-05", "MFSL", "ENTRY", "ind_prs29082017"),
    ("2017-09-05", "RELCAPITAL", "EXIT", "ind_prs29082017"),
]

for date, ticker, action, source_stem in expected_active_events:
    count = int((
        eff.eq(date)
        & sym.eq(ticker)
        & role.eq(action)
        & is_nifty500
        & url.str.contains(source_stem, regex=False)
    ).sum())
    print(f"Earlier event check: {date} {ticker} {action}: {count}")
    if count != 1:
        raise ValueError(
            f"Expected one confirmed earlier event for {ticker}; found {count}. "
            "Nothing saved."
        )

# Prepare a change log from the exact source rows being superseded.
change_records = []
for idx, info in remove_reasons.items():
    record = df.loc[idx].to_dict()
    record["Original_Ledger_Row_Index"] = idx
    record["Change_Type"] = "REMOVE_SUPERSEDED_SCHEDULED_EVENT"
    record["Change_Group"] = info["group"]
    record["Change_Reason"] = info["reason"]
    record["Superseding_Release_URL"] = info["superseding_url"]
    change_records.append(record)

change_log = pd.DataFrame(change_records)
staged = df.drop(index=list(remove_reasons)).copy()

# Final checks on the staged ledger.
assert len(staged) == len(df) - 10

# Verify that none of the 10 superseded release/date/symbol rows remains.
for idx in remove_reasons:
    row = df.loc[idx]
    mask = (
        staged["Published_Date"].eq(row["Published_Date"])
        & staged["Effective_Date"].eq(row["Effective_Date"])
        & staged["Canonical_Symbol"].eq(row["Canonical_Symbol"])
        & staged["Role"].eq(row["Role"])
        & staged["Source_URL"].eq(row["Source_URL"])
    )
    assert not mask.any(), (
        f"Superseded row still present: {row['Canonical_Symbol']}"
    )

# Ensure the already-staged June 2020 review remains balanced.
spub = pd.to_datetime(staged["Effective_Date"], errors="coerce").dt.strftime("%Y-%m-%d")
surl = staged["Source_URL"].str.lower()
srole = staged["Role"].str.upper().str.strip()

june_mask = (
    spub.eq("2020-06-26")
    & surl.str.contains("ind_prs10062020", regex=False)
    & is_nifty500.reindex(staged.index, fill_value=False)
)
june_counts = srole[june_mask].value_counts().to_dict()
if june_counts != {"ENTRY": 25, "EXIT": 25}:
    raise ValueError(f"June 26 review balance changed: {june_counts}. Nothing saved.")

def next_version_path(stem):
    version = 1
    while True:
        path = BASE / f"{stem}_v{version}.csv"
        if not path.exists():
            return path
        version += 1

ledger_out = next_version_path("nifty500_official_event_candidates")
change_out = next_version_path("nifty500_event_ledger_change_log")

# Save only after all checks have passed.
staged.to_csv(ledger_out, index=False)
change_log.to_csv(change_out, index=False)

print("\n" + "=" * 78)
print("STAGED LEDGER V7 — RESCHEDULING FIXES QA PASSED")
print("=" * 78)
print(f"Input v6 rows: {len(df):,}")
print(f"Superseded rows removed: {len(change_log)}")
print(f"Output rows: {len(staged):,}")
print("\nRemoval counts by group:")
display(change_log.groupby("Change_Group").size().rename("Rows").reset_index())

print("\nJune 26 review still has:")
print("Entries:", june_counts.get("ENTRY", 0))
print("Exits:  ", june_counts.get("EXIT", 0))

print("\nFiles created:")
print(ledger_out)
print(change_out)
print("\nV6 and all price/interval files remain unchanged.")

2017_MARCH_RESCHEDULE: matched 6 rows; expected 6
2018_ERIS_VIDEOIND_RESCHEDULE: matched 2 rows; expected 2
2017_SEPTEMBER_RESCHEDULE: matched 2 rows; expected 2
Earlier event check: 2017-03-16 FRETAIL ENTRY: 1
Earlier event check: 2017-03-16 MAXINDIA ENTRY: 1
Earlier event check: 2017-03-16 MGL ENTRY: 1
Earlier event check: 2017-03-16 MYSOREBANK EXIT: 1
Earlier event check: 2017-03-16 SBBJ EXIT: 1
Earlier event check: 2017-03-16 SBT EXIT: 1
Earlier event check: 2018-03-13 ERIS ENTRY: 1
Earlier event check: 2018-03-13 VIDEOIND EXIT: 1
Earlier event check: 2017-09-05 MFSL ENTRY: 1
Earlier event check: 2017-09-05 RELCAPITAL EXIT: 1

STAGED LEDGER V7 — RESCHEDULING FIXES QA PASSED
Input v6 rows: 1,443
Superseded rows removed: 10
Output rows: 1,433

Removal counts by group:


,Change_Group,Rows
0,2017_MARCH_RESCHEDULE,6
1,2017_SEPTEMBER_RESCHEDULE,2
2,2018_ERIS_VIDEOIND_RESCHEDULE,2



June 26 review still has:
Entries: 25
Exits:   25

Files created:
/content/nifty500_pit_data/nifty500_official_event_candidates_v7.csv
/content/nifty500_pit_data/nifty500_event_ledger_change_log_v2.csv

V6 and all price/interval files remain unchanged.


In [57]:
from pathlib import Path
import pandas as pd
import re

BASE = Path("/content/nifty500_pit_data")
LEDGER_FILE = BASE / "nifty500_official_event_candidates_v7.csv"
CACHE_DIR = BASE / "official_release_text_cache"

if not LEDGER_FILE.exists():
    raise FileNotFoundError(f"Missing staged ledger: {LEDGER_FILE}")
if not CACHE_DIR.exists():
    raise FileNotFoundError(f"Missing source text cache: {CACHE_DIR}")

events = pd.read_csv(LEDGER_FILE, dtype=str, keep_default_na=False)
events["_row"] = events.index
events["_symbol"] = events["Canonical_Symbol"].str.upper().str.strip()
events["_role"] = events["Role"].str.upper().str.strip()
events["_eff_dt"] = pd.to_datetime(events["Effective_Date"], errors="coerce")
events["_pub_dt"] = pd.to_datetime(events["Published_Date"], errors="coerce")

targets = [
    "AARTIIND", "ASAHIINDIA", "IREDA",
    "TRITURBINE", "VEDL", "VGUARD"
]

text_files = sorted(CACHE_DIR.rglob("*.txt"))

def date_forms(dt):
    if pd.isna(dt):
        return []
    # Handles common date styles used in the release headers.
    return [
        dt.strftime("%B %d, %Y").lower(),
        dt.strftime("%B %-d, %Y").lower(),
        dt.strftime("%B %d %Y").lower(),
        dt.strftime("%d %B %Y").lower(),
    ]

def ticker_pattern(ticker):
    return re.compile(
        rf"(?<![A-Z0-9]){re.escape(ticker)}(?![A-Z0-9])",
        flags=re.IGNORECASE
    )

def snippets_for_source(ticker, pub_date):
    pattern = ticker_pattern(ticker)
    date_variants = date_forms(pub_date)
    hits = []

    for path in text_files:
        try:
            raw = path.read_text(encoding="utf-8", errors="replace")
        except Exception:
            continue

        low = raw.lower()
        date_hit = any(d in low for d in date_variants)

        # Do not trust filename position: require both date and ticker
        # to appear in the actual cached PDF text.
        if date_hit and pattern.search(raw):
            lines = [s.strip() for s in raw.splitlines() if s.strip()]
            indices = [i for i, line in enumerate(lines) if pattern.search(line)]

            context = []
            for i in indices[:3]:
                start = max(0, i - 3)
                end = min(len(lines), i + 4)
                context.append(
                    " | ".join(lines[start:end])[:650]
                )

            hits.append({
                "Cache_File": path.name,
                "Date_Match": True,
                "Ticker_Occurrences": len(indices),
                "Context": " || ".join(context)
            })

    return hits

print("=" * 78)
print("REMAINING REPEATED-ACTION INVESTIGATION — READ ONLY")
print("=" * 78)

# Section 1: full ledger chronology for each flagged ticker.
for ticker in targets:
    part = events[events["_symbol"].eq(ticker)].sort_values(
        ["_eff_dt", "_pub_dt", "_row"]
    )
    print(f"\n{'=' * 30} {ticker} {'=' * 30}")
    print(f"Total ledger rows for symbol: {len(part)}")

    if part.empty:
        print("No event rows found.")
        continue

    cols = [
        "_row", "Published_Date", "Effective_Date", "Role",
        "Company_Name", "Release_Title", "Section_Title",
        "Action_Heading", "Effective_Date_Source", "Source_URL"
    ]
    display(part[cols])

# Section 2: search actual cached text around each flagged event.
print("\n" + "=" * 78)
print("SOURCE PDF TEXT MATCHES — DATE AND TICKER BOTH VERIFIED")
print("=" * 78)

for ticker in targets:
    part = events[events["_symbol"].eq(ticker)].sort_values(
        ["_eff_dt", "_pub_dt", "_row"]
    )

    for _, row in part.iterrows():
        print(
            f"\n--- {ticker} | {row['Role']} | "
            f"published {row['Published_Date']} | "
            f"effective {row['Effective_Date']} ---"
        )
        print("Ledger row:", row["_row"])
        print("Section:", row["Section_Title"])
        print("Source URL:", row["Source_URL"])

        hits = snippets_for_source(ticker, row["_pub_dt"])

        if not hits:
            print(
                "No cached text file matched both this publication date "
                "and ticker. This may indicate a cache-coverage gap."
            )
        else:
            for hit in hits:
                print(
                    f"\nCache: {hit['Cache_File']} | "
                    f"occurrences: {hit['Ticker_Occurrences']}"
                )
                print(hit["Context"])

print("\nDiagnostic complete. No ledger, interval, price, or membership file modified.")

REMAINING REPEATED-ACTION INVESTIGATION — READ ONLY

============================== AARTIIND ==============================
Total ledger rows for symbol: 3


,_row,Published_Date,Effective_Date,Role,Company_Name,Release_Title,Section_Title,Action_Heading,Effective_Date_Source,Source_URL
556,556,2019-02-25,2019-03-29,EXIT,Aarti Industries Ltd.,"Change in Indices w.e.f. March 29, 2019",NIFTY 500,The following companies are being excluded,title_single_date,https://www.niftyindices.com/Press_Release/ind...
935,935,2022-04-05,2022-05-04,EXIT,Aarti Industries Ltd.,"Replacements in indices w.e.f. May 04, 2022",NIFTY 500,The following company is being excluded,title_single_date,https://www.niftyindices.com/Press_Release/ind...
1015,1015,2023-02-17,2023-03-31,ENTRY,Aarti Industries Ltd.,"Replacements in indices w.e.f March 31, 2023",Nifty 500,The following companies are being included,title_single_date,https://www.niftyindices.com/Press_Release/ind...



============================== ASAHIINDIA ==============================
Total ledger rows for symbol: 7


,_row,Published_Date,Effective_Date,Role,Company_Name,Release_Title,Section_Title,Action_Heading,Effective_Date_Source,Source_URL
90,90,2016-02-22,2016-04-01,EXIT,Asahi India Glass Ltd.,"Change in Indices w.e.f. April 01, 2016.",Nifty 500 Index,The following scrips are being excluded,title_single_date,https://www.niftyindices.com/Press_Release/ind...
302,302,2017-01-16,2017-01-23,ENTRY,Asahi India Glass Ltd.,"Change in Indices w.e.f. January 23, 2017.",NIFTY 500 Index,The following companies are being included,title_single_date,https://www.niftyindices.com/Press_Release/ind...
365,365,2017-08-28,2017-09-29,EXIT,Asahi India Glass Ltd.,"Change in Indices w.e.f. September 29, 2017",NIFTY 500 Index,The following companies are being excluded,title_single_date,https://www.niftyindices.com/Press_Release/ind...
454,454,2018-02-21,2018-04-02,ENTRY,Asahi India Glass Ltd.,"Change in Indices w.e.f. April 02, 2018",NIFTY 500,The following companies are being included,title_single_date,https://www.niftyindices.com/Press_Release/ind...
490,490,2018-08-28,2018-09-28,EXIT,Asahi India Glass Ltd.,"Change in Indices w.e.f. September 28, 2018",NIFTY 500,The following companies are being excluded,title_single_date,https://www.niftyindices.com/Press_Release/ind...
755,755,2020-09-07,2020-09-14,EXIT,Asahi India Glass Ltd.,Replacements in Indices,NIFTY 500,The following company is being excluded,verified_official_pdf_section,https://www.niftyindices.com/Press_Release/ind...
794,794,2021-02-23,2021-03-31,ENTRY,Asahi India Glass Ltd.,"Replacements in Indices w.e.f. March 31, 2021",NIFTY 500,The following companies are being included,title_single_date,https://www.niftyindices.com/Press_Release/ind...



============================== IREDA ==============================
Total ledger rows for symbol: 2


...


============================== VEDL ==============================
Total ledger rows for symbol: 3


,_row,Published_Date,Effective_Date,Role,Company_Name,Release_Title,Section_Title,Action_Heading,Effective_Date_Source,Source_URL
65,65,2015-08-12,2015-09-28,EXIT,Vedanta Ltd.,Change in Indices and Methodology w.e.f Septem...,CNX 500 Index,The following companies are being excluded,title_single_date,https://www.niftyindices.com/Press_Release/ind...
719,719,2020-07-02,2020-07-31,EXIT,Vedanta Ltd.,Replacement in Indices,NIFTY 500,The following company is being excluded,verified_official_release_wording,https://www.niftyindices.com/Press_Release/ind...
816,816,2021-02-23,2021-03-31,ENTRY,Vedanta Ltd.,"Replacements in Indices w.e.f. March 31, 2021",NIFTY 500,The following companies are being included,title_single_date,https://www.niftyindices.com/Press_Release/ind...



============================== VGUARD ==============================
Total ledger rows for symbol: 3


,_row,Published_Date,Effective_Date,Role,Company_Name,Release_Title,Section_Title,Action_Heading,Effective_Date_Source,Source_URL
236,236,2016-02-22,2016-04-01,ENTRY,V-Guard Industries Ltd.,"Change in Indices w.e.f. April 01, 2016.",Nifty 500 Index,The following scrips are being included,title_single_date,https://www.niftyindices.com/Press_Release/ind...
1109,1109,2024-02-28,2024-03-28,EXIT,V-Guard Industries Ltd.,"Replacements in indices w.e.f. March 28, 2024",Nifty 500,The following companies are being excluded,title_single_date,https://www.niftyindices.com/Press_Release/ind...
1339,1339,2026-02-23,2026-03-30,EXIT,V-Guard Industries Ltd.,"Replacements in indices w.e.f. March 30, 2026",Nifty 500,The following companies are being excluded,title_single_date,https://www.niftyindices.com/Press_Release/ind...



SOURCE PDF TEXT MATCHES — DATE AND TICKER BOTH VERIFIED

--- AARTIIND | EXIT | published 2019-02-25 | effective 2019-03-29 ---
Ledger row: 556
Section: NIFTY 500
Source URL: https://www.niftyindices.com/Press_Release/ind_prs25022019.pdf

Cache: release_042.txt | occurrences: 4
8KMILES | 2 | Aarti Industries Ltd. | AARTIIND | 3 | Den Networks Ltd. | DEN || Symbol | 1 | Aarti Industries Ltd. | AARTIIND | 2 | Avanti Feeds Ltd. | AVANTIFEED || Symbol | 1 | Aarti Industries Ltd. | AARTIIND | 2 | Avanti Feeds Ltd. | AVANTIFEED

--- AARTIIND | EXIT | published 2022-04-05 | effective 2022-05-04 ---
Ledger row: 935
Section: NIFTY 500
Source URL: https://www.niftyindices.com/Press_Release/ind_prs05042022.pdf

Cache: release_074.txt | occurrences: 11
Symbol | 1 | Aarti Industries Ltd. | AARTIIND | The following company is being included: | Sr. No. | Company Name || Symbol | 1 | Aarti Industries Ltd. | AARTIIND | The following company is being included: | Sr. No. | Company Name || Symbol | 1 | Aa

In [58]:
from pathlib import Path
import pandas as pd

BASE = Path("/content/nifty500_pit_data")
INPUT = BASE / "nifty500_official_event_candidates_v7.csv"

if not INPUT.exists():
    raise FileNotFoundError(f"Missing staged ledger: {INPUT}")

df = pd.read_csv(INPUT, dtype=str, keep_default_na=False)

pub = pd.to_datetime(df["Published_Date"], errors="coerce").dt.strftime("%Y-%m-%d")
eff = pd.to_datetime(df["Effective_Date"], errors="coerce").dt.strftime("%Y-%m-%d")
sym = df["Canonical_Symbol"].str.upper().str.strip()
role = df["Role"].str.upper().str.strip()
url = df["Source_URL"].str.lower()
section = (
    df["Section_Title"].str.upper()
      .str.replace(r"[^A-Z0-9]+", " ", regex=True)
      .str.strip()
)
nifty500 = section.isin({
    "NIFTY 500", "NIFTY 500 INDEX", "CNX 500", "CNX 500 INDEX"
})

# Match only the two February 28, 2024 changes that were revoked.
ireda_mask = (
    pub.eq("2024-02-28")
    & eff.eq("2024-03-28")
    & sym.eq("IREDA")
    & role.eq("ENTRY")
    & nifty500
    & url.str.contains("ind_prs28022024", regex=False)
)

vguard_mask = (
    pub.eq("2024-02-28")
    & eff.eq("2024-03-28")
    & sym.eq("VGUARD")
    & role.eq("EXIT")
    & nifty500
    & url.str.contains("ind_prs28022024", regex=False)
)

print("=" * 76)
print("V8 REVOCATION CORRECTION — PREFLIGHT")
print("=" * 76)
print("IREDA planned entry rows:", int(ireda_mask.sum()), "(expected 1)")
print("VGUARD planned exit rows:", int(vguard_mask.sum()), "(expected 1)")

if int(ireda_mask.sum()) != 1 or int(vguard_mask.sum()) != 1:
    raise ValueError("Preflight failed. Nothing saved.")

# Preserve these exact rows in the change log as revoked events.
revocation_url = (
    "https://www.niftyindices.com/Press_Release/ind_prs19032024.pdf"
)

change_rows = []
for mask, ticker, explanation in [
    (
        ireda_mask, "IREDA",
        "NSE Indices revoked the February 28, 2024 NIFTY 500 inclusion."
    ),
    (
        vguard_mask, "VGUARD",
        "NSE Indices revoked the February 28, 2024 NIFTY 500 exclusion; "
        "V-Guard remained in the index."
    ),
]:
    for idx in df.index[mask]:
        rec = df.loc[idx].to_dict()
        rec["Original_Ledger_Row_Index"] = idx
        rec["Change_Type"] = "REMOVE_REVOKED_EVENT"
        rec["Change_Reason"] = explanation
        rec["Superseding_Release_URL"] = revocation_url
        change_rows.append(rec)

change_log = pd.DataFrame(change_rows)
staged = df.drop(index=df.index[ireda_mask | vguard_mask]).copy()

# ---------------------------------------------------------------
# QA 1: ensure the revocations removed only the intended events.
# ---------------------------------------------------------------
assert len(staged) == len(df) - 2

s_pub = pd.to_datetime(staged["Published_Date"], errors="coerce").dt.strftime("%Y-%m-%d")
s_eff = pd.to_datetime(staged["Effective_Date"], errors="coerce").dt.strftime("%Y-%m-%d")
s_sym = staged["Canonical_Symbol"].str.upper().str.strip()
s_role = staged["Role"].str.upper().str.strip()
s_url = staged["Source_URL"].str.lower()

s_section = (
    staged["Section_Title"].str.upper()
      .str.replace(r"[^A-Z0-9]+", " ", regex=True)
      .str.strip()
)
s_nifty500 = s_section.isin({
    "NIFTY 500", "NIFTY 500 INDEX", "CNX 500", "CNX 500 INDEX"
})

assert not (
    s_pub.eq("2024-02-28")
    & s_eff.eq("2024-03-28")
    & s_sym.eq("IREDA")
    & s_role.eq("ENTRY")
    & s_nifty500
    & s_url.str.contains("ind_prs28022024", regex=False)
).any()

assert not (
    s_pub.eq("2024-02-28")
    & s_eff.eq("2024-03-28")
    & s_sym.eq("VGUARD")
    & s_role.eq("EXIT")
    & s_nifty500
    & s_url.str.contains("ind_prs28022024", regex=False)
).any()

# The later real events must remain.
assert int((
    s_eff.eq("2024-09-30") & s_sym.eq("IREDA")
    & s_role.eq("ENTRY") & s_nifty500
).sum()) >= 1

assert int((
    s_eff.eq("2026-03-30") & s_sym.eq("VGUARD")
    & s_role.eq("EXIT") & s_nifty500
).sum()) >= 1

# The previously verified June 26, 2020 review remains balanced.
june = (
    s_eff.eq("2020-06-26")
    & s_url.str.contains("ind_prs10062020", regex=False)
    & s_nifty500
)
june_counts = s_role[june].value_counts().to_dict()
assert june_counts == {"ENTRY": 25, "EXIT": 25}, june_counts

# ---------------------------------------------------------------
# QA 2: recompute repeated same-role transitions on staged v8.
# These are review flags, not automatic errors.
# ---------------------------------------------------------------
seq = staged.copy()
seq["_sym"] = s_sym
seq["_role"] = s_role
seq["_eff"] = pd.to_datetime(staged["Effective_Date"], errors="coerce")
seq = seq[
    seq["_sym"].ne("")
    & seq["_eff"].notna()
    & seq["_role"].isin(["ENTRY", "EXIT"])
].copy()

by_date = (
    seq.groupby(["_sym", "_eff"])["_role"]
       .agg(lambda x: sorted(set(x)))
       .reset_index(name="Roles")
       .sort_values(["_sym", "_eff"])
)

flags = []
for ticker, group in by_date.groupby("_sym", sort=True):
    group = group.sort_values("_eff").reset_index(drop=True)
    for i in range(1, len(group)):
        prev = group.iloc[i - 1]
        curr = group.iloc[i]
        common = sorted(set(prev["Roles"]) & set(curr["Roles"]))
        if common:
            flags.append({
                "Symbol": ticker,
                "Previous_Date": prev["_eff"].strftime("%Y-%m-%d"),
                "Previous_Roles": ",".join(prev["Roles"]),
                "Current_Date": curr["_eff"].strftime("%Y-%m-%d"),
                "Current_Roles": ",".join(curr["Roles"]),
                "Repeated_Role": ",".join(common),
                "Review_Status": "UNRESOLVED_REVIEW_FLAG"
            })

flags_df = pd.DataFrame(flags)

# Basic structural QA.
valid = seq.copy()
duplicate_groups = (
    valid.groupby(["_eff", "_role", "_sym"]).size()
         .reset_index(name="Rows")
)
duplicate_groups = duplicate_groups[duplicate_groups["Rows"] > 1]

role_conflicts = (
    valid.groupby(["_eff", "_sym"])["_role"].nunique()
         .reset_index(name="Role_Count")
)
role_conflicts = role_conflicts[role_conflicts["Role_Count"] > 1]

assert len(duplicate_groups) == 0, "Duplicate event keys found."
assert len(role_conflicts) == 0, "Same-date role conflicts found."

# ---------------------------------------------------------------
# Write a new ledger and changelog only after all QA passes.
# ---------------------------------------------------------------
def next_version_path(stem):
    n = 1
    while True:
        path = BASE / f"{stem}_v{n}.csv"
        if not path.exists():
            return path
        n += 1

ledger_out = next_version_path("nifty500_official_event_candidates")
change_out = next_version_path("nifty500_event_ledger_change_log")
flags_out = next_version_path("nifty500_v8_repeated_action_flags")

staged.to_csv(ledger_out, index=False)
change_log.to_csv(change_out, index=False)
flags_df.to_csv(flags_out, index=False)

print("\n" + "=" * 76)
print("V8 STAGED LEDGER — QA PASSED")
print("=" * 76)
print(f"Input v7 rows: {len(df):,}")
print(f"Revoked events removed: {len(change_log)}")
print(f"Output v8 rows: {len(staged):,}")
print("Duplicate event-key groups:", len(duplicate_groups))
print("Same-date role-conflict groups:", len(role_conflicts))
print("Remaining consecutive same-role review flags:", len(flags_df))

print("\nRevoked events recorded in change log:")
display(change_log[[
    "Published_Date", "Effective_Date", "Role",
    "Canonical_Symbol", "Change_Type", "Change_Reason",
    "Superseding_Release_URL"
]])

print("\nRemaining repeated-action flags:")
if flags_df.empty:
    print("None")
else:
    display(flags_df)

print("\nFiles created:")
print(ledger_out)
print(change_out)
print(flags_out)
print("\nV7 and all price/interval files remain unchanged.")

V8 REVOCATION CORRECTION — PREFLIGHT
IREDA planned entry rows: 1 (expected 1)
VGUARD planned exit rows: 1 (expected 1)

V8 STAGED LEDGER — QA PASSED
Input v7 rows: 1,433
Revoked events removed: 2
Output v8 rows: 1,431
Duplicate event-key groups: 0
Same-date role-conflict groups: 0
Remaining consecutive same-role review flags: 4

Revoked events recorded in change log:


,Published_Date,Effective_Date,Role,Canonical_Symbol,Change_Type,Change_Reason,Superseding_Release_URL
0,2024-02-28,2024-03-28,ENTRY,IREDA,REMOVE_REVOKED_EVENT,"NSE Indices revoked the February 28, 2024 NIFT...",https://www.niftyindices.com/Press_Release/ind...
1,2024-02-28,2024-03-28,EXIT,VGUARD,REMOVE_REVOKED_EVENT,"NSE Indices revoked the February 28, 2024 NIFT...",https://www.niftyindices.com/Press_Release/ind...



Remaining repeated-action flags:


,Symbol,Previous_Date,Previous_Roles,Current_Date,Current_Roles,Repeated_Role,Review_Status
0,AARTIIND,2019-03-29,EXIT,2022-05-04,EXIT,EXIT,UNRESOLVED_REVIEW_FLAG
1,ASAHIINDIA,2018-09-28,EXIT,2020-09-14,EXIT,EXIT,UNRESOLVED_REVIEW_FLAG
2,TRITURBINE,2019-09-27,EXIT,2020-06-26,EXIT,EXIT,UNRESOLVED_REVIEW_FLAG
3,VEDL,2015-09-28,EXIT,2020-07-31,EXIT,EXIT,UNRESOLVED_REVIEW_FLAG



Files created:
/content/nifty500_pit_data/nifty500_official_event_candidates_v8.csv
/content/nifty500_pit_data/nifty500_event_ledger_change_log_v3.csv
/content/nifty500_pit_data/nifty500_v8_repeated_action_flags_v1.csv

V7 and all price/interval files remain unchanged.


In [59]:
from pathlib import Path
import pandas as pd
import re

BASE = Path("/content/nifty500_pit_data")
LEDGER_FILE = BASE / "nifty500_official_event_candidates_v8.csv"
CATALOG_FILE = BASE / "nifty500_official_release_catalog_v1.csv"
CACHE_DIR = BASE / "official_release_text_cache"

for p in [LEDGER_FILE, CATALOG_FILE]:
    if not p.exists():
        raise FileNotFoundError(f"Required file missing: {p}")

events = pd.read_csv(LEDGER_FILE, dtype=str, keep_default_na=False)
catalog = pd.read_csv(CATALOG_FILE, dtype=str, keep_default_na=False)

targets = ["AARTIIND", "ASAHIINDIA", "TRITURBINE", "VEDL"]

# Display the current event chronology for each ticker.
events["_symbol"] = events["Canonical_Symbol"].str.upper().str.strip()
events["_effective_dt"] = pd.to_datetime(
    events["Effective_Date"], errors="coerce"
)

print("=" * 78)
print("V8 EVENT HISTORY — FOUR UNRESOLVED SYMBOLS")
print("=" * 78)

for ticker in targets:
    part = events[events["_symbol"].eq(ticker)].sort_values("_effective_dt")
    print(f"\n{ticker}: {len(part)} ledger events")
    display(part[[
        "Published_Date", "Effective_Date", "Role",
        "Release_Title", "Section_Title", "Source_URL"
    ]])

# Map release_NNN.txt to the corresponding row in the official catalog.
# Every match is checked against the actual text content, not filename alone.
text_files = sorted(
    CACHE_DIR.glob("release_*.txt"),
    key=lambda p: int(re.search(r"release_(\d+)", p.name).group(1))
)

records = []

def ticker_re(ticker):
    return re.compile(
        rf"(?<![A-Z0-9]){re.escape(ticker)}(?![A-Z0-9])",
        flags=re.IGNORECASE
    )

for path in text_files:
    m = re.search(r"release_(\d+)", path.name)
    if not m:
        continue

    catalog_idx = int(m.group(1)) - 1
    if catalog_idx < 0 or catalog_idx >= len(catalog):
        continue

    cat = catalog.iloc[catalog_idx]

    try:
        raw = path.read_text(encoding="utf-8", errors="replace")
    except Exception as exc:
        print(f"Could not read {path.name}: {exc}")
        continue

    lines = [line.strip() for line in raw.splitlines() if line.strip()]
    has_500 = bool(re.search(r"\bNIFTY\s*500\b|\bCNX\s*500\b", raw, re.I))

    for ticker in targets:
        pattern = ticker_re(ticker)
        positions = [
            i for i, line in enumerate(lines)
            if pattern.search(line)
        ]

        if not positions:
            continue

        # Capture a few distinct contexts for each ticker in this release.
        contexts = []
        seen = set()

        for i in positions:
            start = max(0, i - 4)
            end = min(len(lines), i + 5)
            context = " | ".join(lines[start:end])
            context = re.sub(r"\s+", " ", context).strip()

            if context not in seen:
                contexts.append(context[:700])
                seen.add(context)

            if len(contexts) >= 3:
                break

        # Find nearby broad-market section labels; this is a clue, not
        # conclusive proof that the ticker itself belongs to NIFTY 500.
        section_labels = []
        for i in positions:
            page_start = i
            while page_start > 0 and not lines[page_start].startswith("--- PAGE"):
                page_start -= 1

            preceding = lines[page_start:i + 1]
            nearby = [
                line for line in preceding[-35:]
                if re.search(r"\bNIFTY\s*500\b|\bCNX\s*500\b", line, re.I)
            ]
            if nearby:
                section_labels.extend(nearby[-2:])

        records.append({
            "Symbol": ticker,
            "Cache_File": path.name,
            "Catalog_Published_Date": cat.get("Published_Date", ""),
            "Catalog_Title": cat.get("Title", ""),
            "Catalog_URL": cat.get("URL", ""),
            "Mentions_NIFTY500_Anywhere": has_500,
            "Ticker_Occurrences": len(positions),
            "Nearby_500_Labels": " | ".join(dict.fromkeys(section_labels)),
            "Context_1": contexts[0] if len(contexts) > 0 else "",
            "Context_2": contexts[1] if len(contexts) > 1 else "",
            "Context_3": contexts[2] if len(contexts) > 2 else "",
        })

report = pd.DataFrame(records)

if not report.empty:
    report["_sort_date"] = pd.to_datetime(
        report["Catalog_Published_Date"], errors="coerce"
    )
    report = report.sort_values(
        ["Symbol", "_sort_date", "Cache_File"]
    ).drop(columns="_sort_date")

    print("\n" + "=" * 78)
    print("ALL CACHED RELEASES MENTIONING THE FOUR SYMBOLS")
    print("=" * 78)

    print("\nMatched release documents per ticker:")
    display(
        report.groupby("Symbol")["Cache_File"]
        .nunique()
        .rename("Distinct_Release_Documents")
        .reset_index()
    )

    for ticker in targets:
        sub = report[report["Symbol"].eq(ticker)].copy()
        print(f"\n{'=' * 25} {ticker}: SOURCE DOCUMENTS {'=' * 25}")
        if sub.empty:
            print("No cached PDF mentions this ticker.")
            continue

        display(sub[[
            "Catalog_Published_Date", "Catalog_Title",
            "Cache_File", "Mentions_NIFTY500_Anywhere",
            "Nearby_500_Labels", "Context_1", "Context_2", "Context_3",
            "Catalog_URL"
        ]])
else:
    print("No ticker matches found in cached official release texts.")

# Save the full audit output under a fresh versioned name.
n = 1
while True:
    out = BASE / f"nifty500_v8_missing_transition_source_search_v{n}.csv"
    if not out.exists():
        break
    n += 1

report.to_csv(out, index=False)

print("\nSource search report saved to:", out)
print("Read-only diagnostic complete. No event or interval data changed.")

V8 EVENT HISTORY — FOUR UNRESOLVED SYMBOLS

AARTIIND: 3 ledger events


,Published_Date,Effective_Date,Role,Release_Title,Section_Title,Source_URL
556,2019-02-25,2019-03-29,EXIT,"Change in Indices w.e.f. March 29, 2019",NIFTY 500,https://www.niftyindices.com/Press_Release/ind...
935,2022-04-05,2022-05-04,EXIT,"Replacements in indices w.e.f. May 04, 2022",NIFTY 500,https://www.niftyindices.com/Press_Release/ind...
1015,2023-02-17,2023-03-31,ENTRY,"Replacements in indices w.e.f March 31, 2023",Nifty 500,https://www.niftyindices.com/Press_Release/ind...



ASAHIINDIA: 7 ledger events


,Published_Date,Effective_Date,Role,Release_Title,Section_Title,Source_URL
90,2016-02-22,2016-04-01,EXIT,"Change in Indices w.e.f. April 01, 2016.",Nifty 500 Index,https://www.niftyindices.com/Press_Release/ind...
302,2017-01-16,2017-01-23,ENTRY,"Change in Indices w.e.f. January 23, 2017.",NIFTY 500 Index,https://www.niftyindices.com/Press_Release/ind...
365,2017-08-28,2017-09-29,EXIT,"Change in Indices w.e.f. September 29, 2017",NIFTY 500 Index,https://www.niftyindices.com/Press_Release/ind...
454,2018-02-21,2018-04-02,ENTRY,"Change in Indices w.e.f. April 02, 2018",NIFTY 500,https://www.niftyindices.com/Press_Release/ind...
490,2018-08-28,2018-09-28,EXIT,"Change in Indices w.e.f. September 28, 2018",NIFTY 500,https://www.niftyindices.com/Press_Release/ind...
755,2020-09-07,2020-09-14,EXIT,Replacements in Indices,NIFTY 500,https://www.niftyindices.com/Press_Release/ind...
794,2021-02-23,2021-03-31,ENTRY,"Replacements in Indices w.e.f. March 31, 2021",NIFTY 500,https://www.niftyindices.com/Press_Release/ind...



TRITURBINE: 6 ledger events


...


========================= TRITURBINE: SOURCE DOCUMENTS =========================


,Catalog_Published_Date,Catalog_Title,Cache_File,Mentions_NIFTY500_Anywhere,Nearby_500_Labels,Context_1,Context_2,Context_3,Catalog_URL
2,"Feb 22, 2016","Change in Indices w.e.f. April 01, 2016.",release_013.txt,True,,Transport Corporation of India Ltd. | TCI | 72...,,,https://www.niftyindices.com/Press_Release/ind...
9,"Aug 28, 2017","Change in Indices w.e.f. September 29, 2017",release_025.txt,True,,Tide Water Oil (India) Ltd. | TIDEWATER | 22 |...,Tide Water Oil (India) Ltd. | TIDEWATER | 23 |...,Tide Water Oil (India) Ltd. | TIDEWATER | 23 |...,https://www.niftyindices.com/Press_Release/ind...
16,"Feb 25, 2019","Change in Indices w.e.f. March 29, 2019",release_042.txt,True,,TCNS Clothing Co. Ltd. | TCNSBRANDS | 24 | Tri...,TCNS Clothing Co. Ltd. | TCNSBRANDS | 26 | Tri...,TCNS Clothing Co. Ltd. | TCNSBRANDS | 28 | Tri...,https://www.niftyindices.com/Press_Release/ind...
17,"Aug 28, 2019","Change in Indices w.e.f. September 27, 2019",release_045.txt,True,,Tata Coffee Ltd. | TATACOFFEE | 21 | Triveni T...,Tata Coffee Ltd. | TATACOFFEE | 24 | Triveni T...,Tata Coffee Ltd. | TATACOFFEE | 23 | Triveni T...,https://www.niftyindices.com/Press_Release/ind...
20,"Jun 10, 2020","Change in indices w.e.f. June 26, 2020",release_054.txt,True,,Techno Electric & Engineering Company Ltd. | T...,Techno Electric & Engineering Company Ltd. | T...,Torrent Pharmaceuticals Ltd. | TORNTPHARM | 29...,https://www.niftyindices.com/Press_Release/ind...
26,"Feb 23, 2021","Replacements in Indices w.e.f. March 31, 2021",release_062.txt,True,,Tanla Platforms Ltd. | TANLA | 21 | Triveni Tu...,Tanla Platforms Ltd. | TANLA | 30 | Triveni Tu...,Tanla Platforms Ltd. | TANLA | 27 | Triveni Tu...,https://www.niftyindices.com/Press_Release/ind...
36,"Sep 16, 2022","Replacements in indices w.e.f. September 30, 2022",release_082.txt,True,,Strides Pharma Science Ltd. | STAR | 13 | Triv...,,,https://www.niftyindices.com/Press_Release/ind...
39,"Aug 17, 2023",Replacements in indices and revision in criteria,release_096.txt,True,,Tata Teleservices (Maharashtra) Ltd. | TTML | ...,,,https://www.niftyindices.com/Press_Release/ind...
40,"Dec 07, 2023","Replacements in indices w.e.f. December 29, 2023",release_100.txt,True,,Triveni Engineering & Industries Ltd. | TRIVEN...,,,https://www.niftyindices.com/Press_Release/ind...
45,"Dec 11, 2024",Replacements in indices and Revision in criter...,release_122.txt,True,,Trent Ltd. | TRENT | 24 | Triveni Turbine Ltd....,Torrent Power Ltd. | TORNTPOWER | 30 | Triveni...,,https://www.niftyindices.com/Press_Release/ind...



========================= VEDL: SOURCE DOCUMENTS =========================


,Catalog_Published_Date,Catalog_Title,Cache_File,Mentions_NIFTY500_Anywhere,Nearby_500_Labels,Context_1,Context_2,Context_3,Catalog_URL
0,"Aug 12, 2015",Change in Indices and Methodology w.e.f Septem...,release_008.txt,True,,Bharat Heavy Electricals Ltd. | BHEL | 2 | Ved...,,,https://www.niftyindices.com/Press_Release/ind...
3,"Feb 22, 2016","Change in Indices w.e.f. April 01, 2016.",release_013.txt,True,,Punjab National Bank | PNB | 3 | Vedanta Ltd. ...,Torrent Pharmaceuticals Ltd. | TORNTPHARM | 16...,,https://www.niftyindices.com/Press_Release/ind...
4,"Oct 17, 2016","Change in Indices w.e.f. October 24, 2016 and ...",release_017.txt,True,,TVS Motor Company Ltd. | TVSMOTOR | 18 | Vedan...,,,https://www.niftyindices.com/Press_Release/ind...
6,"Feb 16, 2017","Change in Indices w.e.f. March 31, 2017.",release_020.txt,True,,Indian Oil Corporation Ltd. | IOC | 2 | Vedant...,,,https://www.niftyindices.com/Press_Release/ind...
7,"Apr 27, 2017",Replacements in indices and change in NIFTY100...,release_022.txt,True,,Company Name | Symbol | 1 | Vedanta Ltd. | VED...,Company Name | Symbol | 1 | Vedanta Ltd. | VED...,,https://www.niftyindices.com/Press_Release/ind...
10,"Dec 18, 2017","Change in Indices w.e.f. December 29, 2017",release_030.txt,False,,Siemens Ltd. | SIEMENS | 14 | Vedanta Ltd. | V...,UPL Ltd. | UPL | 13 | Vedanta Ltd. | VEDL | 14...,,https://www.niftyindices.com/Press_Release/ind...
14,"Dec 14, 2018","Change in Indices w.e.f. December 28, 2018",release_040.txt,True,,Power Grid Corporation of India Ltd. | POWERGR...,,,https://www.niftyindices.com/Press_Release/ind...
18,"Feb 18, 2020",Revision in criteria and replacements in Indices,release_051.txt,True,,South Indian Bank Ltd. | SOUTHBANK | 16 | Veda...,,,https://www.niftyindices.com/Press_Release/ind...
21,"Jun 10, 2020","Change in indices w.e.f. June 26, 2020",release_054.txt,True,,Indiabulls Housing Finance Ltd. | IBULHSGFIN |...,South Indian Bank Ltd. | SOUTHBANK | 17 | Veda...,,https://www.niftyindices.com/Press_Release/ind...
22,"Jul 02, 2020",Replacement in Indices,release_055.txt,True,NIFTY 500 | NIFTY500 Value 50,Replacement in Indices | The Index Maintenance...,Company Name | Symbol | 1 | Vedanta Ltd. | VED...,,https://www.niftyindices.com/Press_Release/ind...



Source search report saved to: /content/nifty500_pit_data/nifty500_v8_missing_transition_source_search_v1.csv
Read-only diagnostic complete. No event or interval data changed.


In [60]:
from pathlib import Path
import pandas as pd

BASE = Path("/content/nifty500_pit_data")
LEDGER_FILE = BASE / "nifty500_official_event_candidates_v8.csv"

events = pd.read_csv(LEDGER_FILE, dtype=str, keep_default_na=False)

pub = pd.to_datetime(events["Published_Date"], errors="coerce").dt.strftime("%Y-%m-%d")
eff = pd.to_datetime(events["Effective_Date"], errors="coerce").dt.strftime("%Y-%m-%d")
sym = events["Canonical_Symbol"].str.upper().str.strip()
role = events["Role"].str.upper().str.strip()
url = events["Source_URL"].str.lower()

source_match = url.str.contains("ind_prs16122019", regex=False)

expected = [
    ("ALBK", "EXIT"),
    ("ANDHRABANK", "EXIT"),
    ("CORPBANK", "EXIT"),
    ("MAXINDIA", "EXIT"),
    ("ORIENTBANK", "EXIT"),
    ("SYNDIBANK", "EXIT"),
    ("TATACHEM", "EXIT"),
    ("AARTIIND", "ENTRY"),
    ("ASAHIINDIA", "ENTRY"),
    ("INDIAMART", "ENTRY"),
    ("SPANDANA", "ENTRY"),
    ("SPICEJET", "ENTRY"),
    ("TRITURBINE", "ENTRY"),
    ("WESTLIFE", "ENTRY"),
]

print("=" * 76)
print("DECEMBER 2019 NIFTY 500 RELEASE — COVERAGE CHECK")
print("=" * 76)
print("Official source: https://www.niftyindices.com/Press_Release/ind_prs16122019.pdf")
print("Expected NIFTY 500 action rows: 14 (7 exits and 7 entries)")
print("Existing v8 rows tied to this source:", int(source_match.sum()))

if source_match.any():
    display(events.loc[source_match, [
        "Published_Date", "Effective_Date", "Role",
        "Canonical_Symbol", "Section_Title", "Source_URL"
    ]])

rows = []
for ticker, action in expected:
    exact_match = (
        pub.eq("2019-12-16")
        & eff.eq("2019-12-27")
        & sym.eq(ticker)
        & role.eq(action)
    )
    source_exact = exact_match & source_match

    rows.append({
        "Expected_Symbol": ticker,
        "Expected_Role": action,
        "Correct_Published_Date_Count": int(pub[exact_match].notna().sum()),
        "Rows_From_Official_Release_URL": int(source_exact.sum()),
        "Status": (
            "PRESENT" if int(source_exact.sum()) == 1
            else "MISSING" if int(source_exact.sum()) == 0
            else "DUPLICATE"
        )
    })

coverage = pd.DataFrame(rows)

print("\nCoverage of all 14 official actions:")
display(coverage)

print("\nCoverage summary:")
display(coverage.groupby("Status").size().rename("Rows").reset_index())

# Show any event rows with the same date that might have been extracted
# under a different source URL.
date_window = pub.eq("2019-12-16") | eff.eq("2019-12-27")
print("\nAll v8 rows published Dec 16 or effective Dec 27, 2019:")
display(events.loc[date_window, [
    "Published_Date", "Effective_Date", "Role",
    "Canonical_Symbol", "Release_Title", "Section_Title", "Source_URL"
]].sort_values(["Effective_Date", "Role", "Canonical_Symbol"]))

print("\nRead-only check complete. No files were modified.")

DECEMBER 2019 NIFTY 500 RELEASE — COVERAGE CHECK
Official source: https://www.niftyindices.com/Press_Release/ind_prs16122019.pdf
Expected NIFTY 500 action rows: 14 (7 exits and 7 entries)
Existing v8 rows tied to this source: 0

Coverage of all 14 official actions:


,Expected_Symbol,Expected_Role,Correct_Published_Date_Count,Rows_From_Official_Release_URL,Status
0,ALBK,EXIT,0,0,MISSING
1,ANDHRABANK,EXIT,0,0,MISSING
2,CORPBANK,EXIT,0,0,MISSING
3,MAXINDIA,EXIT,0,0,MISSING
4,ORIENTBANK,EXIT,0,0,MISSING
5,SYNDIBANK,EXIT,0,0,MISSING
6,TATACHEM,EXIT,0,0,MISSING
7,AARTIIND,ENTRY,0,0,MISSING
8,ASAHIINDIA,ENTRY,0,0,MISSING
9,INDIAMART,ENTRY,0,0,MISSING



Coverage summary:


,Status,Rows
0,MISSING,14



All v8 rows published Dec 16 or effective Dec 27, 2019:


,Published_Date,Effective_Date,Role,Canonical_Symbol,Release_Title,Section_Title,Source_URL
665,2019-12-19,2019-12-27,ENTRY,KENNAMET,"Replacement in indices w.e.f. December 27, 2019",NIFTY 500,https://www.niftyindices.com/Press_Release/ind...
666,2019-12-19,2019-12-27,ENTRY,TASTYBITE,"Replacement in indices w.e.f. December 27, 2019",NIFTY 500,https://www.niftyindices.com/Press_Release/ind...
663,2019-12-19,2019-12-27,EXIT,DEN,"Replacement in indices w.e.f. December 27, 2019",NIFTY 500,https://www.niftyindices.com/Press_Release/ind...
664,2019-12-19,2019-12-27,EXIT,HATHWAY,"Replacement in indices w.e.f. December 27, 2019",NIFTY 500,https://www.niftyindices.com/Press_Release/ind...



Read-only check complete. No files were modified.


In [61]:
from pathlib import Path
import pandas as pd

BASE = Path("/content/nifty500_pit_data")
INPUT = BASE / "nifty500_official_event_candidates_v8.csv"

SOURCE_URL = (
    "https://www.niftyindices.com/Press_Release/"
    "ind_prs16122019.pdf"
)

if not INPUT.exists():
    raise FileNotFoundError(f"Missing v8 ledger: {INPUT}")

df = pd.read_csv(INPUT, dtype=str, keep_default_na=False)

required = {
    "Published_Date", "Effective_Date", "Role", "Company_Name",
    "Raw_Symbol", "Canonical_Symbol", "Release_Title",
    "Section_Title", "Action_Heading", "Effective_Date_Source",
    "Date_Override_Reason", "Source_URL"
}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing columns: {sorted(missing)}")

# Official NIFTY 500 section of the December 16, 2019 release.
exits = [
    ("Allahabad Bank", "ALBK"),
    ("Andhra Bank", "ANDHRABANK"),
    ("Corporation Bank", "CORPBANK"),
    ("Max India Ltd.", "MAXINDIA"),
    ("Oriental Bank of Commerce", "ORIENTBANK"),
    ("Syndicate Bank", "SYNDIBANK"),
    ("Tata Chemicals Ltd.", "TATACHEM"),
]

entries = [
    ("Aarti Industries Ltd.", "AARTIIND"),
    ("Asahi India Glass Ltd.", "ASAHIINDIA"),
    ("Indiamart Intermesh Ltd.", "INDIAMART"),
    ("Spandana Sphoorty Financial Ltd.", "SPANDANA"),
    ("Spicejet Ltd.", "SPICEJET"),
    ("Triveni Turbine Ltd.", "TRITURBINE"),
    ("Westlife Development Ltd.", "WESTLIFE"),
]

published_date = "2019-12-16"
effective_date = "2019-12-27"
release_title = "Replacements and revision in Index criteria"

def next_version_path(stem):
    n = 1
    while True:
        path = BASE / f"{stem}_v{n}.csv"
        if not path.exists():
            return path
        n += 1

# Normalize existing ledger fields for preflight checks.
pub = pd.to_datetime(df["Published_Date"], errors="coerce").dt.strftime("%Y-%m-%d")
eff = pd.to_datetime(df["Effective_Date"], errors="coerce").dt.strftime("%Y-%m-%d")
sym = df["Canonical_Symbol"].str.upper().str.strip()
role = df["Role"].str.upper().str.strip()
url = df["Source_URL"].str.lower()

section = (
    df["Section_Title"].str.upper()
      .str.replace(r"[^A-Z0-9]+", " ", regex=True)
      .str.strip()
)
nifty500 = section.isin({
    "NIFTY 500", "NIFTY 500 INDEX", "CNX 500", "CNX 500 INDEX"
})

# Preflight: v8 should have no events attributed to this release.
source_rows = int(url.str.contains("ind_prs16122019", regex=False).sum())
print("=" * 78)
print("DECEMBER 2019 RELEASE — V9 PREFLIGHT")
print("=" * 78)
print("Existing rows attributed to official source:", source_rows)
print("Expected entries:", len(entries))
print("Expected exits:", len(exits))

if source_rows != 0:
    raise ValueError(
        "The source URL already has event rows in v8. Inspect them before adding."
    )

# Ensure none of the 14 exact effective-date/symbol/role events exists already,
# even if a row was attributed to a different release.
expected_pairs = [
    (ticker.upper(), "EXIT") for _, ticker in exits
] + [
    (ticker.upper(), "ENTRY") for _, ticker in entries
]

for ticker, action in expected_pairs:
    existing_count = int((
        eff.eq(effective_date)
        & sym.eq(ticker)
        & role.eq(action)
        & nifty500
    ).sum())
    if existing_count:
        raise ValueError(
            f"{ticker} {action} effective {effective_date} already exists "
            f"{existing_count} time(s). Nothing saved."
        )

if len(expected_pairs) != 14 or len(set(expected_pairs)) != 14:
    raise ValueError("Expected 14 unique symbol/role actions. Nothing saved.")

def make_event(role_value, company, ticker, action_heading):
    row = {col: "" for col in df.columns}
    row.update({
        "Published_Date": published_date,
        "Effective_Date": effective_date,
        "Role": role_value,
        "Company_Name": company,
        "Raw_Symbol": ticker,
        "Canonical_Symbol": ticker,
        "Release_Title": release_title,
        "Section_Title": "NIFTY 500",
        "Action_Heading": action_heading,
        "Effective_Date_Source": "verified_official_pdf_section",
        "Date_Override_Reason": (
            "Official December 16, 2019 release explicitly states that "
            "the changes become effective December 27, 2019, close of "
            "December 26, 2019."
        ),
        "Source_URL": SOURCE_URL,
    })
    return row

new_rows = []

for company, ticker in exits:
    new_rows.append(make_event(
        "EXIT", company, ticker,
        "The following companies are being excluded"
    ))

for company, ticker in entries:
    new_rows.append(make_event(
        "ENTRY", company, ticker,
        "The following companies are being included"
    ))

added = pd.DataFrame(new_rows, columns=df.columns)
staged = pd.concat([df, added], ignore_index=True)

# ------------------------------------------------------------
# QA checks before saving.
# ------------------------------------------------------------
spub = pd.to_datetime(staged["Published_Date"], errors="coerce").dt.strftime("%Y-%m-%d")
seff = pd.to_datetime(staged["Effective_Date"], errors="coerce").dt.strftime("%Y-%m-%d")
ssym = staged["Canonical_Symbol"].str.upper().str.strip()
srole = staged["Role"].str.upper().str.strip()
surl = staged["Source_URL"].str.lower()

ssection = (
    staged["Section_Title"].str.upper()
      .str.replace(r"[^A-Z0-9]+", " ", regex=True)
      .str.strip()
)
snifty500 = ssection.isin({
    "NIFTY 500", "NIFTY 500 INDEX", "CNX 500", "CNX 500 INDEX"
})

new_source = (
    spub.eq(published_date)
    & seff.eq(effective_date)
    & surl.str.contains("ind_prs16122019", regex=False)
    & snifty500
)

new_events = staged.loc[new_source].copy()
new_counts = new_events["Role"].str.upper().value_counts().to_dict()

assert len(new_events) == 14, f"Expected 14 rows; found {len(new_events)}"
assert new_counts == {"EXIT": 7, "ENTRY": 7}, new_counts

# All event-key duplicates should remain absent.
valid = staged[
    seff.notna()
    & ssym.ne("")
    & srole.isin(["ENTRY", "EXIT"])
].copy()
valid["_eff"] = pd.to_datetime(valid["Effective_Date"], errors="coerce")
valid["_sym"] = ssym.loc[valid.index]
valid["_role"] = srole.loc[valid.index]

dups = valid.groupby(["_eff", "_role", "_sym"]).size()
dups = dups[dups > 1]
assert len(dups) == 0, f"Duplicate event keys found: {dups.to_dict()}"

# No same-date entry/exit conflict for the same symbol.
role_counts = valid.groupby(["_eff", "_sym"])["_role"].nunique()
conflicts = role_counts[role_counts > 1]
assert len(conflicts) == 0, f"Same-date role conflicts found: {conflicts.to_dict()}"

# Check the separate December 19 review remains intact.
dec19 = (
    seff.eq("2019-12-27")
    & surl.str.contains("ind_prs19122019", regex=False)
    & snifty500
)
dec19_counts = srole[dec19].value_counts().to_dict()
assert dec19_counts == {"ENTRY": 2, "EXIT": 2}, (
    f"December 19 rows changed unexpectedly: {dec19_counts}"
)

# June 2020 review must remain balanced.
june = (
    seff.eq("2020-06-26")
    & surl.str.contains("ind_prs10062020", regex=False)
    & snifty500
)
june_counts = srole[june].value_counts().to_dict()
assert june_counts == {"ENTRY": 25, "EXIT": 25}, (
    f"June 26 review changed unexpectedly: {june_counts}"
)

assert len(staged) == len(df) + 14

# Recompute same-role transition flags for staged v9.
by_date = (
    valid.groupby(["_sym", "_eff"])["_role"]
      .agg(lambda x: sorted(set(x)))
      .reset_index(name="Roles")
      .sort_values(["_sym", "_eff"])
)

flags = []
for ticker, group in by_date.groupby("_sym", sort=True):
    group = group.sort_values("_eff").reset_index(drop=True)
    for i in range(1, len(group)):
        prev = group.iloc[i - 1]
        curr = group.iloc[i]
        shared = sorted(set(prev["Roles"]) & set(curr["Roles"]))
        if shared:
            flags.append({
                "Symbol": ticker,
                "Previous_Date": prev["_eff"].strftime("%Y-%m-%d"),
                "Previous_Roles": ",".join(prev["Roles"]),
                "Current_Date": curr["_eff"].strftime("%Y-%m-%d"),
                "Current_Roles": ",".join(curr["Roles"]),
                "Repeated_Role": ",".join(shared),
                "Review_Status": "UNRESOLVED_REVIEW_FLAG"
            })

flags_df = pd.DataFrame(flags)

# Build a provenance log for all 14 inserted records.
change_log = added.copy()
change_log.insert(0, "Change_Type", "ADD_MISSING_OFFICIAL_NIFTY500_EVENT")
change_log["Change_Reason"] = (
    "NIFTY 500 section of the official December 16, 2019 release "
    "was absent from v8."
)

ledger_out = next_version_path("nifty500_official_event_candidates")
change_out = next_version_path("nifty500_event_ledger_change_log")
flags_out = next_version_path("nifty500_v9_repeated_action_flags")

# Save only after every preflight and QA assertion passes.
staged.to_csv(ledger_out, index=False)
change_log.to_csv(change_out, index=False)
flags_df.to_csv(flags_out, index=False)

print("\n" + "=" * 78)
print("STAGED LEDGER V9 — QA PASSED")
print("=" * 78)
print("Input v8 rows:", len(df))
print("Events added:", len(added))
print("Output v9 rows:", len(staged))
print("Duplicate event-key groups:", len(dups))
print("Same-date role-conflict groups:", len(conflicts))
print("December 16 entries/exits:", new_counts.get("ENTRY", 0), "/", new_counts.get("EXIT", 0))
print("December 19 entries/exits preserved:", dec19_counts.get("ENTRY", 0), "/", dec19_counts.get("EXIT", 0))
print("June 26, 2020 entries/exits preserved:", june_counts.get("ENTRY", 0), "/", june_counts.get("EXIT", 0))
print("Repeated-action flags remaining:", len(flags_df))

print("\nAdded official events:")
display(new_events[[
    "Published_Date", "Effective_Date", "Role",
    "Canonical_Symbol", "Company_Name", "Source_URL"
]].sort_values(["Role", "Canonical_Symbol"]))

print("\nRemaining repeated-action flags:")
if flags_df.empty:
    print("None")
else:
    display(flags_df)

print("\nFiles created:")
print(ledger_out)
print(change_out)
print(flags_out)
print("\nV8 and all price/interval files remain unchanged.")

DECEMBER 2019 RELEASE — V9 PREFLIGHT
Existing rows attributed to official source: 0
Expected entries: 7
Expected exits: 7

STAGED LEDGER V9 — QA PASSED
Input v8 rows: 1431
Events added: 14
Output v9 rows: 1445
Duplicate event-key groups: 0
Same-date role-conflict groups: 0
December 16 entries/exits: 7 / 7
December 19 entries/exits preserved: 2 / 2
June 26, 2020 entries/exits preserved: 25 / 25
Repeated-action flags remaining: 1

Added official events:


,Published_Date,Effective_Date,Role,Canonical_Symbol,Company_Name,Source_URL
1438,2019-12-16,2019-12-27,ENTRY,AARTIIND,Aarti Industries Ltd.,https://www.niftyindices.com/Press_Release/ind...
1439,2019-12-16,2019-12-27,ENTRY,ASAHIINDIA,Asahi India Glass Ltd.,https://www.niftyindices.com/Press_Release/ind...
1440,2019-12-16,2019-12-27,ENTRY,INDIAMART,Indiamart Intermesh Ltd.,https://www.niftyindices.com/Press_Release/ind...
1441,2019-12-16,2019-12-27,ENTRY,SPANDANA,Spandana Sphoorty Financial Ltd.,https://www.niftyindices.com/Press_Release/ind...
1442,2019-12-16,2019-12-27,ENTRY,SPICEJET,Spicejet Ltd.,https://www.niftyindices.com/Press_Release/ind...
1443,2019-12-16,2019-12-27,ENTRY,TRITURBINE,Triveni Turbine Ltd.,https://www.niftyindices.com/Press_Release/ind...
1444,2019-12-16,2019-12-27,ENTRY,WESTLIFE,Westlife Development Ltd.,https://www.niftyindices.com/Press_Release/ind...
1431,2019-12-16,2019-12-27,EXIT,ALBK,Allahabad Bank,https://www.niftyindices.com/Press_Release/ind...
1432,2019-12-16,2019-12-27,EXIT,ANDHRABANK,Andhra Bank,https://www.niftyindices.com/Press_Release/ind...
1433,2019-12-16,2019-12-27,EXIT,CORPBANK,Corporation Bank,https://www.niftyindices.com/Press_Release/ind...



Remaining repeated-action flags:


,Symbol,Previous_Date,Previous_Roles,Current_Date,Current_Roles,Repeated_Role,Review_Status
0,VEDL,2015-09-28,EXIT,2020-07-31,EXIT,EXIT,UNRESOLVED_REVIEW_FLAG



Files created:
/content/nifty500_pit_data/nifty500_official_event_candidates_v9.csv
/content/nifty500_pit_data/nifty500_event_ledger_change_log_v4.csv
/content/nifty500_pit_data/nifty500_v9_repeated_action_flags_v1.csv

V8 and all price/interval files remain unchanged.


In [62]:
from pathlib import Path
import pandas as pd

BASE = Path("/content/nifty500_pit_data")
INPUT = BASE / "nifty500_official_event_candidates_v9.csv"

if not INPUT.exists():
    raise FileNotFoundError(f"Missing staged v9 ledger: {INPUT}")

df = pd.read_csv(INPUT, dtype=str, keep_default_na=False)

pub = pd.to_datetime(df["Published_Date"], errors="coerce").dt.strftime("%Y-%m-%d")
eff = pd.to_datetime(df["Effective_Date"], errors="coerce").dt.strftime("%Y-%m-%d")
sym = df["Canonical_Symbol"].str.upper().str.strip()
role = df["Role"].str.upper().str.strip()
url = df["Source_URL"].str.lower()
section = (
    df["Section_Title"].str.upper()
      .str.replace(r"[^A-Z0-9]+", " ", regex=True)
      .str.strip()
)

# Precisely identify VEDL's false-positive NIFTY 500 event.
wrong_vedl_mask = (
    pub.eq("2015-08-12")
    & eff.eq("2015-09-28")
    & sym.eq("VEDL")
    & role.eq("EXIT")
    & url.str.contains("ind_prs12082015", regex=False)
    & section.eq("CNX 500 INDEX")
)

print("=" * 76)
print("VEDL WRONG-SECTION CORRECTION — PREFLIGHT")
print("=" * 76)
print("Incorrect VEDL/CNX 500 rows found:", int(wrong_vedl_mask.sum()))
print("Expected: 1")

if int(wrong_vedl_mask.sum()) != 1:
    raise ValueError("Preflight failed. Nothing has been saved.")

print("\nRow being removed from the NIFTY 500 candidate ledger:")
display(df.loc[wrong_vedl_mask, [
    "Published_Date", "Effective_Date", "Role",
    "Canonical_Symbol", "Section_Title", "Source_URL"
]])

# Log the correction separately, retaining the original record.
change_log = df.loc[wrong_vedl_mask].copy()
change_log.insert(
    0, "Change_Type", "EXCLUDE_WRONG_INDEX_SECTION"
)
change_log["Change_Reason"] = (
    "The official August 12, 2015 release lists VEDL under LIX15 exclusions, "
    "not CNX 500 exclusions. This event was incorrectly attributed to NIFTY 500."
)
change_log["Verified_Source_URL"] = (
    "https://www.niftyindices.com/Press_Release/ind_prs12082015.pdf"
)

staged = df.loc[~wrong_vedl_mask].copy()

# ---------------------------------------------------------------
# QA: ensure only the false-positive row is removed.
# ---------------------------------------------------------------
spub = pd.to_datetime(staged["Published_Date"], errors="coerce").dt.strftime("%Y-%m-%d")
seff = pd.to_datetime(staged["Effective_Date"], errors="coerce").dt.strftime("%Y-%m-%d")
ssym = staged["Canonical_Symbol"].str.upper().str.strip()
srole = staged["Role"].str.upper().str.strip()
surl = staged["Source_URL"].str.lower()

ssection = (
    staged["Section_Title"].str.upper()
      .str.replace(r"[^A-Z0-9]+", " ", regex=True)
      .str.strip()
)
snifty500 = ssection.isin({
    "NIFTY 500", "NIFTY 500 INDEX", "CNX 500", "CNX 500 INDEX"
})

assert len(staged) == len(df) - 1

# The false-positive event must no longer be present.
assert not (
    spub.eq("2015-08-12")
    & seff.eq("2015-09-28")
    & ssym.eq("VEDL")
    & srole.eq("EXIT")
    & surl.str.contains("ind_prs12082015", regex=False)
    & snifty500
).any()

# The two later, legitimate VEDL events must remain intact.
valid_vedl_exit = (
    seff.eq("2020-07-31")
    & ssym.eq("VEDL")
    & srole.eq("EXIT")
    & surl.str.contains("ind_prs02072020_1", regex=False)
    & snifty500
)
valid_vedl_entry = (
    seff.eq("2021-03-31")
    & ssym.eq("VEDL")
    & srole.eq("ENTRY")
    & surl.str.contains("ind_prs23022021", regex=False)
    & snifty500
)

assert int(valid_vedl_exit.sum()) == 1, "Valid 2020 VEDL exit not preserved."
assert int(valid_vedl_entry.sum()) == 1, "Valid 2021 VEDL entry not preserved."

# Basic data validity.
assert pd.to_datetime(staged["Published_Date"], errors="coerce").notna().all()
assert pd.to_datetime(staged["Effective_Date"], errors="coerce").notna().all()
assert srole.isin(["ENTRY", "EXIT"]).all()
assert ssym.ne("").all()

# No duplicate effective-date/role/symbol keys.
valid = staged.copy()
valid["_eff"] = pd.to_datetime(valid["Effective_Date"], errors="coerce")
valid["_sym"] = ssym
valid["_role"] = srole

dups = valid.groupby(["_eff", "_role", "_sym"]).size()
dups = dups[dups > 1]
assert len(dups) == 0, f"Duplicate event keys found: {dups.to_dict()}"

# No same-day ENTRY/EXIT conflicts.
conflicts = valid.groupby(["_eff", "_sym"])["_role"].nunique()
conflicts = conflicts[conflicts > 1]
assert len(conflicts) == 0, f"Same-day role conflicts found: {conflicts.to_dict()}"

# Recalculate repeated same-role review flags.
by_date = (
    valid.groupby(["_sym", "_eff"])["_role"]
      .agg(lambda x: sorted(set(x)))
      .reset_index(name="Roles")
      .sort_values(["_sym", "_eff"])
)

flags = []
for ticker, group in by_date.groupby("_sym", sort=True):
    group = group.sort_values("_eff").reset_index(drop=True)
    for i in range(1, len(group)):
        prev = group.iloc[i - 1]
        curr = group.iloc[i]
        repeated = sorted(set(prev["Roles"]) & set(curr["Roles"]))
        if repeated:
            flags.append({
                "Symbol": ticker,
                "Previous_Date": prev["_eff"].strftime("%Y-%m-%d"),
                "Previous_Roles": ",".join(prev["Roles"]),
                "Current_Date": curr["_eff"].strftime("%Y-%m-%d"),
                "Current_Roles": ",".join(curr["Roles"]),
                "Repeated_Role": ",".join(repeated),
                "Review_Status": "UNRESOLVED_REVIEW_FLAG"
            })

flags_df = pd.DataFrame(flags)

def next_version_path(stem):
    n = 1
    while True:
        path = BASE / f"{stem}_v{n}.csv"
        if not path.exists():
            return path
        n += 1

ledger_out = next_version_path("nifty500_official_event_candidates")
change_out = next_version_path("nifty500_event_ledger_change_log")
flags_out = next_version_path("nifty500_v10_repeated_action_flags")

# Save only after every assertion passed.
staged.to_csv(ledger_out, index=False)
change_log.to_csv(change_out, index=False)
flags_df.to_csv(flags_out, index=False)

print("\n" + "=" * 76)
print("STAGED LEDGER V10 — QA PASSED")
print("=" * 76)
print("Input v9 rows:", len(df))
print("False-positive rows removed:", len(change_log))
print("Output v10 rows:", len(staged))
print("Invalid rows: 0")
print("Duplicate event-key groups:", len(dups))
print("Same-date role-conflict groups:", len(conflicts))
print("Repeated-action flags remaining:", len(flags_df))

print("\nPreserved VEDL events:")
display(staged[
    ssym.isin(["VEDL"])
][[
    "Published_Date", "Effective_Date", "Role",
    "Canonical_Symbol", "Section_Title", "Source_URL"
]].sort_values("Effective_Date"))

print("\nRemaining repeated-action flags:")
if flags_df.empty:
    print("None")
else:
    display(flags_df)

print("\nFiles created:")
print(ledger_out)
print(change_out)
print(flags_out)
print("\nV9 and all original price/interval files remain unchanged.")

VEDL WRONG-SECTION CORRECTION — PREFLIGHT
Incorrect VEDL/CNX 500 rows found: 1
Expected: 1

Row being removed from the NIFTY 500 candidate ledger:


,Published_Date,Effective_Date,Role,Canonical_Symbol,Section_Title,Source_URL
65,2015-08-12,2015-09-28,EXIT,VEDL,CNX 500 Index,https://www.niftyindices.com/Press_Release/ind...



STAGED LEDGER V10 — QA PASSED
Input v9 rows: 1445
False-positive rows removed: 1
Output v10 rows: 1444
Invalid rows: 0
Duplicate event-key groups: 0
Same-date role-conflict groups: 0
Repeated-action flags remaining: 0

Preserved VEDL events:


,Published_Date,Effective_Date,Role,Canonical_Symbol,Section_Title,Source_URL
719,2020-07-02,2020-07-31,EXIT,VEDL,NIFTY 500,https://www.niftyindices.com/Press_Release/ind...
816,2021-02-23,2021-03-31,ENTRY,VEDL,NIFTY 500,https://www.niftyindices.com/Press_Release/ind...



Remaining repeated-action flags:
None

Files created:
/content/nifty500_pit_data/nifty500_official_event_candidates_v10.csv
/content/nifty500_pit_data/nifty500_event_ledger_change_log_v5.csv
/content/nifty500_pit_data/nifty500_v10_repeated_action_flags_v1.csv

V9 and all original price/interval files remain unchanged.


In [63]:
from pathlib import Path
import pandas as pd

BASE = Path("/content/nifty500_pit_data")

if not BASE.exists():
    raise FileNotFoundError(f"Project directory not found: {BASE}")

print("=" * 78)
print("MEMBERSHIP INTERVAL FILE INVENTORY — READ ONLY")
print("=" * 78)

# Find likely membership/interval artifacts, excluding ordinary price files.
keywords = ("interval", "membership", "constituent", "pit_member", "pit_intervals")

candidates = sorted([
    p for p in BASE.iterdir()
    if p.is_file()
    and any(k in p.name.lower() for k in keywords)
    and p.suffix.lower() in {".csv", ".parquet", ".pq"}
])

if not candidates:
    print("No matching interval/membership files found.")
else:
    rows = []

    for path in candidates:
        info = {
            "File": path.name,
            "Size_MB": round(path.stat().st_size / (1024 * 1024), 3),
            "Format": path.suffix.lower(),
            "Rows": None,
            "Columns": "",
            "Start_Date_Column": "",
            "End_Date_Column": "",
            "Ticker_Column": "",
        }

        try:
            if path.suffix.lower() == ".csv":
                sample = pd.read_csv(path, nrows=5, dtype=str)
                info["Columns"] = " | ".join(sample.columns.astype(str))

                # Count data rows without loading the full CSV into memory.
                with path.open("r", encoding="utf-8-sig", errors="replace") as f:
                    info["Rows"] = max(sum(1 for _ in f) - 1, 0)

                columns = list(sample.columns)
            else:
                import pyarrow.parquet as pq
                pf = pq.ParquetFile(path)
                info["Rows"] = pf.metadata.num_rows
                columns = pf.schema.names
                info["Columns"] = " | ".join(columns)

            lower_map = {c.lower(): c for c in columns}

            start_names = [
                "start_date", "valid_from", "effective_from",
                "interval_start", "membership_start"
            ]
            end_names = [
                "end_date", "valid_to", "effective_to",
                "interval_end", "membership_end"
            ]
            ticker_names = [
                "canonical_symbol", "symbol", "ticker",
                "tradingsymbol", "isin"
            ]

            info["Start_Date_Column"] = next(
                (lower_map[n] for n in start_names if n in lower_map), ""
            )
            info["End_Date_Column"] = next(
                (lower_map[n] for n in end_names if n in lower_map), ""
            )
            info["Ticker_Column"] = next(
                (lower_map[n] for n in ticker_names if n in lower_map), ""
            )

        except Exception as exc:
            info["Columns"] = f"Inspection error: {exc}"

        rows.append(info)

    inventory = pd.DataFrame(rows)
    display(inventory[[
        "File", "Size_MB", "Format", "Rows",
        "Start_Date_Column", "End_Date_Column",
        "Ticker_Column", "Columns"
    ]])

# Print the relevant in-memory DataFrame names if the runtime still has them.
print("\nRelevant in-memory DataFrames:")
for name in ["intervals_df", "final_df"]:
    obj = globals().get(name)
    if isinstance(obj, pd.DataFrame):
        print(f"{name}: {len(obj):,} rows; columns={list(obj.columns)}")
    else:
        print(f"{name}: not available in this runtime")

print("\nInventory complete. No files were modified.")

MEMBERSHIP INTERVAL FILE INVENTORY — READ ONLY


,File,Size_MB,Format,Rows,Start_Date_Column,End_Date_Column,Ticker_Column,Columns
0,gujarat_gas_interval_correction_log_v1.csv,0.000,.csv,2,,,ISIN,ISIN | Action | Old_Start | Old_End | Reason |...
1,membership_mismatch_audit_2026-10-25.csv,0.001,.csv,9,,,,Date | Active_Interval_ISINs | Active_Interval...
2,membership_mismatch_examples_2026-10-25.csv,0.013,.csv,244,,,Symbol,Date | Mismatch | ISIN | Symbol
3,official_review_interval_details_2016_v1.csv,0.022,.csv,148,Start_Date,End_Date,ISIN,Role | Official_Symbol | Interval_Row_Index | ...
4,official_review_post_stage_interval_details_20...,0.018,.csv,147,Start_Date,End_Date,ISIN,Role | Official_Symbol | ISIN | Interval_Symbo...
5,pit_intervals_corrected_v2.csv,0.126,.csv,1292,start_date,end_date,symbol,isin | symbol | company_name | start_date | st...
6,pit_intervals_official_2016-04-01_v1_staged.csv,0.126,.csv,1292,start_date,end_date,symbol,isin | symbol | company_name | start_date | st...
7,pit_intervals_official_2016-04-01_v2_staged.csv,0.126,.csv,1291,start_date,end_date,symbol,isin | symbol | company_name | start_date | st...
8,pit_intervals_official_2026-03-30_v1.csv,0.126,.csv,1292,start_date,end_date,symbol,isin | symbol | company_name | start_date | st...



Relevant in-memory DataFrames:
intervals_df: 1,292 rows; columns=['isin', 'symbol', 'company_name', 'start_date', 'start_confidence', 'end_date', 'end_confidence']
final_df: 2,125,567 rows; columns=['isin', 'symbol', 'date', 'open', 'high', 'low', 'close', 'volume', 'dividends', 'stock_splits', 'is_nifty500_constituent', 'data_source', 'adjustment', '_flat_ohlc', 'price_quality_flag', 'close_for_analysis', 'calendar_status', 'price_candidate_eligible']

Inventory complete. No files were modified.


In [64]:
from pathlib import Path
import pandas as pd
import numpy as np

BASE = Path("/content/nifty500_pit_data")
EVENT_FILE = BASE / "nifty500_official_event_candidates_v10.csv"
TRI_FILE = BASE / "nifty500_tri.csv"

CANDIDATE_FILES = [
    "pit_intervals_corrected_v2.csv",
    "pit_intervals_official_2016-04-01_v1_staged.csv",
    "pit_intervals_official_2016-04-01_v2_staged.csv",
    "pit_intervals_official_2026-03-30_v1.csv",
]

for p in [EVENT_FILE, TRI_FILE]:
    if not p.exists():
        raise FileNotFoundError(f"Required file not found: {p}")

events = pd.read_csv(EVENT_FILE, dtype=str, keep_default_na=False)
tri = pd.read_csv(TRI_FILE)

print("=" * 78)
print("V10 EVENT-TO-INTERVAL RECONCILIATION — READ ONLY")
print("=" * 78)
print("Event ledger:", EVENT_FILE.name)
print("Event rows:", f"{len(events):,}")

# ---------------------------------------------------------------
# 1. Load the benchmark calendar and derive the prior trading date.
# ---------------------------------------------------------------
date_candidates = [
    c for c in tri.columns
    if c.strip().lower() in {
        "date", "index_date", "trading_date", "timestamp", "datetime"
    }
]
if not date_candidates:
    raise ValueError(
        "Could not identify the date column in nifty500_tri.csv. "
        f"Columns found: {list(tri.columns)}"
    )

tri_date_col = date_candidates[0]
benchmark_dates = pd.DatetimeIndex(
    pd.to_datetime(tri[tri_date_col], errors="coerce")
    .dropna()
    .dt.normalize()
    .unique()
).sort_values()

if len(benchmark_dates) == 0:
    raise ValueError("No valid benchmark dates found.")

def previous_benchmark_date(dt):
    if pd.isna(dt):
        return pd.NaT
    pos = benchmark_dates.searchsorted(dt.normalize(), side="left") - 1
    if pos < 0:
        return pd.NaT
    return benchmark_dates[pos]

print("Benchmark calendar column:", tri_date_col)
print("Benchmark dates:", len(benchmark_dates))
print("Benchmark range:",
      benchmark_dates.min().date(), "to", benchmark_dates.max().date())

# Normalize event fields.
events["_event_row"] = events.index
events["_symbol"] = events["Canonical_Symbol"].str.upper().str.strip()
events["_role"] = events["Role"].str.upper().str.strip()
events["_effective_dt"] = pd.to_datetime(
    events["Effective_Date"], errors="coerce"
)

events["_expected_boundary"] = events.apply(
    lambda r: (
        r["_effective_dt"]
        if r["_role"] == "ENTRY"
        else previous_benchmark_date(r["_effective_dt"])
        if r["_role"] == "EXIT"
        else pd.NaT
    ),
    axis=1
)

valid_events = events[
    events["_symbol"].ne("")
    & events["_effective_dt"].notna()
    & events["_role"].isin(["ENTRY", "EXIT"])
].copy()

print("Events eligible for reconciliation:", f"{len(valid_events):,}")
print("Events excluded for invalid date/role/symbol:",
      len(events) - len(valid_events))

# ---------------------------------------------------------------
# 2. Candidate column detection.
# ---------------------------------------------------------------
def find_column(df, options, label, filename):
    lookup = {c.strip().lower(): c for c in df.columns}
    for name in options:
        if name in lookup:
            return lookup[name]
    raise ValueError(
        f"Could not identify {label} in {filename}. "
        f"Columns: {list(df.columns)}"
    )

# ---------------------------------------------------------------
# 3. Compare every event with each candidate interval file.
# ---------------------------------------------------------------
summary_rows = []
detail_reports = {}

for filename in CANDIDATE_FILES:
    path = BASE / filename
    if not path.exists():
        print(f"\nSkipping missing candidate: {filename}")
        continue

    intervals = pd.read_csv(path, dtype=str, keep_default_na=False)

    symbol_col = find_column(
        intervals,
        ["symbol", "canonical_symbol", "ticker", "tradingsymbol"],
        "symbol column", filename
    )
    start_col = find_column(
        intervals,
        ["start_date", "valid_from", "effective_from",
         "interval_start", "membership_start"],
        "start-date column", filename
    )
    end_col = find_column(
        intervals,
        ["end_date", "valid_to", "effective_to",
         "interval_end", "membership_end"],
        "end-date column", filename
    )

    intervals["_symbol"] = intervals[symbol_col].str.upper().str.strip()
    intervals["_start_dt"] = pd.to_datetime(
        intervals[start_col], errors="coerce"
    ).dt.normalize()
    intervals["_end_dt"] = pd.to_datetime(
        intervals[end_col], errors="coerce"
    ).dt.normalize()

    # Basic interval integrity counts.
    invalid_ranges = int((
        intervals["_start_dt"].notna()
        & intervals["_end_dt"].notna()
        & (intervals["_start_dt"] > intervals["_end_dt"])
    ).sum())

    detail_rows = []

    # Match using canonical symbol and exact expected boundary.
    for _, event in valid_events.iterrows():
        ticker = event["_symbol"]
        role = event["_role"]
        expected = event["_expected_boundary"]

        same_symbol = intervals[intervals["_symbol"].eq(ticker)]
        boundary_col = "_start_dt" if role == "ENTRY" else "_end_dt"

        exact = same_symbol[
            same_symbol[boundary_col].eq(expected)
        ] if pd.notna(expected) else same_symbol.iloc[0:0]

        nearest = None
        valid_boundaries = same_symbol[
            same_symbol[boundary_col].notna()
        ]

        if pd.notna(expected) and not valid_boundaries.empty:
            distances = (
                valid_boundaries[boundary_col] - expected
            ).abs()
            nearest_idx = distances.idxmin()
            nearest = valid_boundaries.loc[nearest_idx]

        if len(exact) > 0:
            status = "EXACT_BOUNDARY_MATCH"
        elif same_symbol.empty:
            status = "NO_INTERVAL_FOR_SYMBOL"
        elif pd.isna(expected):
            status = "NO_EXPECTED_BOUNDARY_DATE"
        else:
            status = "BOUNDARY_MISMATCH"

        exact_details = []
        for _, int_row in exact.iterrows():
            exact_details.append(
                f"ISIN={int_row.get('isin', '')};"
                f"start={int_row['_start_dt']};"
                f"end={int_row['_end_dt']}"
            )

        detail_rows.append({
            "Event_Row": int(event["_event_row"]),
            "Published_Date": event["Published_Date"],
            "Effective_Date": event["Effective_Date"],
            "Role": role,
            "Symbol": ticker,
            "Release_Title": event["Release_Title"],
            "Expected_Boundary_Date": (
                expected.strftime("%Y-%m-%d")
                if pd.notna(expected) else ""
            ),
            "Boundary_Field_Checked": boundary_col,
            "Same_Symbol_Interval_Count": len(same_symbol),
            "Exact_Match_Count": len(exact),
            "Status": status,
            "Exact_Match_Intervals": " || ".join(exact_details),
            "Nearest_Boundary_Date": (
                nearest[boundary_col].strftime("%Y-%m-%d")
                if nearest is not None
                and pd.notna(nearest[boundary_col]) else ""
            ),
            "Nearest_ISIN": (
                str(nearest.get("isin", ""))
                if nearest is not None else ""
            ),
            "Nearest_Interval_Start": (
                nearest["_start_dt"].strftime("%Y-%m-%d")
                if nearest is not None
                and pd.notna(nearest["_start_dt"]) else ""
            ),
            "Nearest_Interval_End": (
                nearest["_end_dt"].strftime("%Y-%m-%d")
                if nearest is not None
                and pd.notna(nearest["_end_dt"]) else ""
            ),
            "Interval_File": filename,
        })

    detail = pd.DataFrame(detail_rows)
    detail_reports[filename] = detail

    n_events = len(detail)
    entry_detail = detail[detail["Role"].eq("ENTRY")]
    exit_detail = detail[detail["Role"].eq("EXIT")]
    exact_count = int(detail["Status"].eq("EXACT_BOUNDARY_MATCH").sum())

    summary_rows.append({
        "Interval_File": filename,
        "Interval_Rows": len(intervals),
        "Unique_Nonblank_Symbols": intervals.loc[
            intervals["_symbol"].ne(""), "_symbol"
        ].nunique(),
        "Invalid_Start_End_Ranges": invalid_ranges,
        "Events_Checked": n_events,
        "Exact_Boundary_Matches": exact_count,
        "Exact_Match_Rate_Pct": round(
            100 * exact_count / n_events, 2
        ) if n_events else np.nan,
        "Entry_Events": len(entry_detail),
        "Entry_Exact_Matches": int(
            entry_detail["Status"].eq("EXACT_BOUNDARY_MATCH").sum()
        ),
        "Exit_Events": len(exit_detail),
        "Exit_Exact_Matches": int(
            exit_detail["Status"].eq("EXACT_BOUNDARY_MATCH").sum()
        ),
        "Events_No_Symbol_Interval": int(
            detail["Status"].eq("NO_INTERVAL_FOR_SYMBOL").sum()
        ),
        "Boundary_Mismatches": int(
            detail["Status"].eq("BOUNDARY_MISMATCH").sum()
        ),
    })

    # Save per-candidate event detail to a new versioned report.
    out = BASE / f"event_interval_reconciliation_{path.stem}_v1.csv"
    if out.exists():
        n = 2
        while (BASE / f"event_interval_reconciliation_{path.stem}_v{n}.csv").exists():
            n += 1
        out = BASE / f"event_interval_reconciliation_{path.stem}_v{n}.csv"
    detail.to_csv(out, index=False)

# ---------------------------------------------------------------
# 4. Summaries and representative mismatches.
# ---------------------------------------------------------------
summary = pd.DataFrame(summary_rows)

if summary.empty:
    raise RuntimeError("No interval candidate files were available.")

print("\n" + "=" * 78)
print("CANDIDATE COMPARISON SUMMARY")
print("=" * 78)
display(summary.sort_values(
    ["Exact_Match_Rate_Pct", "Invalid_Start_End_Ranges"],
    ascending=[False, True]
))

# Show event-level results for the two most relevant staged candidates.
focus_files = [
    "pit_intervals_official_2016-04-01_v2_staged.csv",
    "pit_intervals_corrected_v2.csv",
]

for filename in focus_files:
    if filename not in detail_reports:
        continue

    detail = detail_reports[filename]
    mismatches = detail[
        detail["Status"].ne("EXACT_BOUNDARY_MATCH")
    ].copy()

    print("\n" + "=" * 78)
    print(f"MISMATCH EXAMPLES — {filename}")
    print(f"Total non-exact events: {len(mismatches)}")
    print("=" * 78)

    if mismatches.empty:
        print("No mismatches in this candidate.")
    else:
        display(mismatches[[
            "Event_Row", "Published_Date", "Effective_Date",
            "Role", "Symbol", "Expected_Boundary_Date",
            "Status", "Same_Symbol_Interval_Count",
            "Nearest_Boundary_Date", "Nearest_ISIN",
            "Nearest_Interval_Start", "Nearest_Interval_End"
        ]].head(25))

print("\nReports saved:")
for filename in detail_reports:
    stem = Path(filename).stem
    print(stem, "-> event_interval_reconciliation_" + stem + "_v1.csv")

print("\nReconciliation complete. No event, interval, price, or membership file was modified.")

V10 EVENT-TO-INTERVAL RECONCILIATION — READ ONLY
Event ledger: nifty500_official_event_candidates_v10.csv
Event rows: 1,444
Benchmark calendar column: Date
Benchmark dates: 2923
Benchmark range: 2015-01-01 to 2026-10-19
Events eligible for reconciliation: 1,444
Events excluded for invalid date/role/symbol: 0

CANDIDATE COMPARISON SUMMARY


,Interval_File,Interval_Rows,Unique_Nonblank_Symbols,Invalid_Start_End_Ranges,Events_Checked,Exact_Boundary_Matches,Exact_Match_Rate_Pct,Entry_Events,Entry_Exact_Matches,Exit_Events,Exit_Exact_Matches,Events_No_Symbol_Interval,Boundary_Mismatches
2,pit_intervals_official_2016-04-01_v2_staged.csv,1291,1074,0,1444,150,10.39,723,76,721,74,162,1132
1,pit_intervals_official_2016-04-01_v1_staged.csv,1292,1074,0,1444,149,10.32,723,75,721,74,162,1133
3,pit_intervals_official_2026-03-30_v1.csv,1292,1074,0,1444,116,8.03,723,58,721,58,162,1166
0,pit_intervals_corrected_v2.csv,1292,1074,0,1444,54,3.74,723,27,721,27,162,1228



MISMATCH EXAMPLES — pit_intervals_official_2016-04-01_v2_staged.csv
Total non-exact events: 1294


,Event_Row,Published_Date,Effective_Date,Role,Symbol,Expected_Boundary_Date,Status,Same_Symbol_Interval_Count,Nearest_Boundary_Date,Nearest_ISIN,Nearest_Interval_Start,Nearest_Interval_End
0,0,2015-02-20,2015-03-27,EXIT,DCW,2015-03-26,BOUNDARY_MISMATCH,1,2016-12-28,INE500A01029,2010-01-02,2016-12-28
1,1,2015-02-20,2015-03-27,EXIT,EDUCOMP,2015-03-26,BOUNDARY_MISMATCH,1,2016-12-28,INE216H01027,2010-01-02,2016-12-28
2,2,2015-02-20,2015-03-27,EXIT,ELDERPHARM,2015-03-26,BOUNDARY_MISMATCH,1,2016-12-28,INE975A01015,2010-01-02,2016-12-28
3,3,2015-02-20,2015-03-27,EXIT,HUBTOWN,2015-03-26,BOUNDARY_MISMATCH,1,2016-12-28,INE703H01016,2010-01-02,2016-12-28
4,4,2015-02-20,2015-03-27,EXIT,ILFSENGG,2015-03-26,NO_INTERVAL_FOR_SYMBOL,0,,,,
5,5,2015-02-20,2015-03-27,EXIT,IMFA,2015-03-26,BOUNDARY_MISMATCH,1,2016-12-28,INE919H01018,2012-01-12,2016-12-28
6,6,2015-02-20,2015-03-27,EXIT,LOVABLE,2015-03-26,BOUNDARY_MISMATCH,1,2016-12-28,INE597L01014,2012-01-12,2016-12-28
7,7,2015-02-20,2015-03-27,EXIT,PFOCUS,2015-03-26,BOUNDARY_MISMATCH,2,2016-12-28,INE367G01038,2012-01-12,2016-12-28
8,8,2015-02-20,2015-03-27,EXIT,RAJTV,2015-03-26,BOUNDARY_MISMATCH,2,2014-07-09,INE952H01019,2012-01-12,2014-07-09
9,9,2015-02-20,2015-03-27,EXIT,REIAGROLTD,2015-03-26,BOUNDARY_MISMATCH,1,2016-12-28,INE385B01031,2010-01-02,2016-12-28



MISMATCH EXAMPLES — pit_intervals_corrected_v2.csv
Total non-exact events: 1390


,Event_Row,Published_Date,Effective_Date,Role,Symbol,Expected_Boundary_Date,Status,Same_Symbol_Interval_Count,Nearest_Boundary_Date,Nearest_ISIN,Nearest_Interval_Start,Nearest_Interval_End
0,0,2015-02-20,2015-03-27,EXIT,DCW,2015-03-26,BOUNDARY_MISMATCH,1,2016-12-28,INE500A01029,2010-01-02,2016-12-28
1,1,2015-02-20,2015-03-27,EXIT,EDUCOMP,2015-03-26,BOUNDARY_MISMATCH,1,2016-12-28,INE216H01027,2010-01-02,2016-12-28
2,2,2015-02-20,2015-03-27,EXIT,ELDERPHARM,2015-03-26,BOUNDARY_MISMATCH,1,2016-12-28,INE975A01015,2010-01-02,2016-12-28
3,3,2015-02-20,2015-03-27,EXIT,HUBTOWN,2015-03-26,BOUNDARY_MISMATCH,1,2016-12-28,INE703H01016,2010-01-02,2016-12-28
4,4,2015-02-20,2015-03-27,EXIT,ILFSENGG,2015-03-26,NO_INTERVAL_FOR_SYMBOL,0,,,,
5,5,2015-02-20,2015-03-27,EXIT,IMFA,2015-03-26,BOUNDARY_MISMATCH,1,2016-12-28,INE919H01018,2012-01-12,2016-12-28
6,6,2015-02-20,2015-03-27,EXIT,LOVABLE,2015-03-26,BOUNDARY_MISMATCH,1,2016-12-28,INE597L01014,2012-01-12,2016-12-28
7,7,2015-02-20,2015-03-27,EXIT,PFOCUS,2015-03-26,BOUNDARY_MISMATCH,2,2016-12-28,INE367G01038,2012-01-12,2016-12-28
8,8,2015-02-20,2015-03-27,EXIT,RAJTV,2015-03-26,BOUNDARY_MISMATCH,2,2014-07-09,INE952H01019,2012-01-12,2014-07-09
9,9,2015-02-20,2015-03-27,EXIT,REIAGROLTD,2015-03-26,BOUNDARY_MISMATCH,1,2016-12-28,INE385B01031,2010-01-02,2016-12-28



Reports saved:
pit_intervals_corrected_v2 -> event_interval_reconciliation_pit_intervals_corrected_v2_v1.csv
pit_intervals_official_2016-04-01_v1_staged -> event_interval_reconciliation_pit_intervals_official_2016-04-01_v1_staged_v1.csv
pit_intervals_official_2016-04-01_v2_staged -> event_interval_reconciliation_pit_intervals_official_2016-04-01_v2_staged_v1.csv
pit_intervals_official_2026-03-30_v1 -> event_interval_reconciliation_pit_intervals_official_2026-03-30_v1_v1.csv

Reconciliation complete. No event, interval, price, or membership file was modified.


In [65]:
from pathlib import Path
import pandas as pd
import numpy as np

BASE = Path("/content/nifty500_pit_data")
EVENT_FILE = BASE / "nifty500_official_event_candidates_v10.csv"
INTERVAL_FILE = BASE / "pit_intervals_official_2016-04-01_v2_staged.csv"
TRI_FILE = BASE / "nifty500_tri.csv"
MISMATCH_AUDIT = BASE / "membership_mismatch_audit_2026-10-25.csv"
MISMATCH_EXAMPLES = BASE / "membership_mismatch_examples_2026-10-25.csv"

for p in [EVENT_FILE, INTERVAL_FILE, TRI_FILE]:
    if not p.exists():
        raise FileNotFoundError(f"Required file not found: {p}")

events = pd.read_csv(EVENT_FILE, dtype=str, keep_default_na=False)
intervals = pd.read_csv(INTERVAL_FILE, dtype=str, keep_default_na=False)
tri = pd.read_csv(TRI_FILE)

print("=" * 78)
print("INTERVAL QUALITY AND EVENT BOUNDARY DIAGNOSTICS — READ ONLY")
print("=" * 78)

# ---------------------------------------------------------------
# 1. Benchmark calendar and expected event boundaries.
# ---------------------------------------------------------------
date_col = next(
    (c for c in tri.columns if c.strip().lower() in
     {"date", "index_date", "trading_date", "timestamp", "datetime"}),
    None
)
if date_col is None:
    raise ValueError(f"No date column found in TRI file: {list(tri.columns)}")

calendar = pd.DatetimeIndex(
    pd.to_datetime(tri[date_col], errors="coerce")
      .dropna().dt.normalize().unique()
).sort_values()

def prior_trading_day(dt):
    if pd.isna(dt):
        return pd.NaT
    pos = calendar.searchsorted(dt.normalize(), side="left") - 1
    return calendar[pos] if pos >= 0 else pd.NaT

events["_symbol"] = events["Canonical_Symbol"].str.upper().str.strip()
events["_role"] = events["Role"].str.upper().str.strip()
events["_effective_dt"] = pd.to_datetime(
    events["Effective_Date"], errors="coerce"
).dt.normalize()

events["_expected_boundary"] = events.apply(
    lambda r: (
        r["_effective_dt"]
        if r["_role"] == "ENTRY"
        else prior_trading_day(r["_effective_dt"])
        if r["_role"] == "EXIT"
        else pd.NaT
    ),
    axis=1
)

# ---------------------------------------------------------------
# 2. Inspect the interval file's date and confidence quality.
# ---------------------------------------------------------------
intervals["_symbol"] = intervals["symbol"].str.upper().str.strip()
intervals["_start_dt"] = pd.to_datetime(
    intervals["start_date"], errors="coerce"
).dt.normalize()
intervals["_end_dt"] = pd.to_datetime(
    intervals["end_date"], errors="coerce"
).dt.normalize()

invalid_range = (
    intervals["_start_dt"].notna()
    & intervals["_end_dt"].notna()
    & (intervals["_start_dt"] > intervals["_end_dt"])
)

print("\n1. INTERVAL FILE PROFILE")
print("Rows:", len(intervals))
print("Distinct nonblank symbols:", intervals.loc[
    intervals["_symbol"].ne(""), "_symbol"
].nunique())
print("Blank starts:", int(intervals["_start_dt"].isna().sum()))
print("Blank ends:", int(intervals["_end_dt"].isna().sum()))
print("Invalid start > end ranges:", int(invalid_range.sum()))
print("Start-date range:",
      intervals["_start_dt"].min(), "to", intervals["_start_dt"].max())
print("End-date range:",
      intervals["_end_dt"].min(), "to", intervals["_end_dt"].max())

print("\nStart confidence counts:")
if "start_confidence" in intervals.columns:
    display(
        intervals["start_confidence"].replace("", "(blank)")
        .value_counts(dropna=False).rename_axis("start_confidence")
        .reset_index(name="Rows")
    )

print("\nEnd confidence counts:")
if "end_confidence" in intervals.columns:
    display(
        intervals["end_confidence"].replace("", "(blank)")
        .value_counts(dropna=False).rename_axis("end_confidence")
        .reset_index(name="Rows")
    )

print("\nIntervals per symbol:")
per_symbol = intervals.groupby("_symbol", dropna=False).size()
print("Symbols with 1 interval:", int((per_symbol == 1).sum()))
print("Symbols with 2 intervals:", int((per_symbol == 2).sum()))
print("Symbols with 3+ intervals:", int((per_symbol >= 3).sum()))
print("Max intervals for one symbol:",
      int(per_symbol.max()) if len(per_symbol) else 0)

# Check overlap within ticker histories.
overlap_pairs = []
for ticker, group in intervals[
    intervals["_symbol"].ne("") & intervals["_start_dt"].notna()
].sort_values(["_symbol", "_start_dt"]).groupby("_symbol"):
    group = group.sort_values("_start_dt")
    rows = list(group.iterrows())
    for i in range(1, len(rows)):
        prev_idx, prev = rows[i - 1]
        curr_idx, curr = rows[i]
        if (
            pd.notna(prev["_end_dt"])
            and pd.notna(curr["_start_dt"])
            and curr["_start_dt"] <= prev["_end_dt"]
        ):
            overlap_pairs.append({
                "Symbol": ticker,
                "Previous_Row": prev_idx,
                "Previous_ISIN": prev.get("isin", ""),
                "Previous_Start": prev["_start_dt"],
                "Previous_End": prev["_end_dt"],
                "Current_Row": curr_idx,
                "Current_ISIN": curr.get("isin", ""),
                "Current_Start": curr["_start_dt"],
                "Current_End": curr["_end_dt"],
            })

overlap_df = pd.DataFrame(overlap_pairs)
print("\nOverlapping interval pairs:", len(overlap_df))
if not overlap_df.empty:
    display(overlap_df.head(20))

# ---------------------------------------------------------------
# 3. Compute nearest-boundary distance for each event.
# ---------------------------------------------------------------
detail_rows = []

usable_events = events[
    events["_symbol"].ne("")
    & events["_effective_dt"].notna()
    & events["_expected_boundary"].notna()
    & events["_role"].isin(["ENTRY", "EXIT"])
]

for _, event in usable_events.iterrows():
    ticker = event["_symbol"]
    role = event["_role"]
    expected = event["_expected_boundary"]

    ticker_intervals = intervals[intervals["_symbol"].eq(ticker)]
    boundary_col = "_start_dt" if role == "ENTRY" else "_end_dt"
    candidate = ticker_intervals[
        ticker_intervals[boundary_col].notna()
    ].copy()

    if candidate.empty:
        detail_rows.append({
            "Event_Row": event.name,
            "Symbol": ticker,
            "Role": role,
            "Published_Date": event["Published_Date"],
            "Effective_Date": event["Effective_Date"],
            "Expected_Boundary": expected,
            "Candidate_Interval_Count": len(ticker_intervals),
            "Exact_Match": False,
            "Nearest_Boundary": pd.NaT,
            "Signed_Delta_Days": np.nan,
            "Absolute_Delta_Days": np.nan,
            "Nearest_ISIN": "",
            "Nearest_Start_Confidence": "",
            "Nearest_End_Confidence": "",
            "Release_Title": event["Release_Title"],
        })
        continue

    candidate["_delta_days"] = (
        candidate[boundary_col] - expected
    ).dt.days
    candidate["_abs_delta"] = candidate["_delta_days"].abs()

    nearest = candidate.sort_values(
        ["_abs_delta", boundary_col]
    ).iloc[0]

    detail_rows.append({
        "Event_Row": event.name,
        "Symbol": ticker,
        "Role": role,
        "Published_Date": event["Published_Date"],
        "Effective_Date": event["Effective_Date"],
        "Expected_Boundary": expected,
        "Candidate_Interval_Count": len(ticker_intervals),
        "Exact_Match": bool((candidate["_abs_delta"] == 0).any()),
        "Nearest_Boundary": nearest[boundary_col],
        "Signed_Delta_Days": int(nearest["_delta_days"]),
        "Absolute_Delta_Days": int(nearest["_abs_delta"]),
        "Nearest_ISIN": nearest.get("isin", ""),
        "Nearest_Start_Confidence": nearest.get("start_confidence", ""),
        "Nearest_End_Confidence": nearest.get("end_confidence", ""),
        "Release_Title": event["Release_Title"],
    })

detail = pd.DataFrame(detail_rows)
detail["Expected_Boundary"] = pd.to_datetime(
    detail["Expected_Boundary"], errors="coerce"
)
detail["Nearest_Boundary"] = pd.to_datetime(
    detail["Nearest_Boundary"], errors="coerce"
)

print("\n" + "=" * 78)
print("2. BOUNDARY MISMATCH DISTANCE")
print("=" * 78)
print("Events measured:", len(detail))
print("Exact boundary matches:", int(detail["Exact_Match"].sum()))
print("Events without any interval boundary:", int(
    detail["Absolute_Delta_Days"].isna().sum()
))

def distance_band(days):
    if pd.isna(days):
        return "No interval boundary"
    days = int(days)
    if days == 0:
        return "Exact"
    if days <= 5:
        return "1–5 days"
    if days <= 30:
        return "6–30 days"
    if days <= 90:
        return "31–90 days"
    if days <= 365:
        return "91–365 days"
    return "Over 365 days"

detail["Distance_Band"] = detail["Absolute_Delta_Days"].apply(distance_band)

print("\nDistance bands overall:")
display(
    detail.groupby("Distance_Band", dropna=False)
    .size().rename("Events").reset_index()
    .sort_values("Events", ascending=False)
)

print("\nDistance bands by role:")
display(
    detail.groupby(["Role", "Distance_Band"], dropna=False)
    .size().rename("Events").reset_index()
    .sort_values(["Role", "Events"], ascending=[True, False])
)

print("\nNearest-boundary delta summary by role (days):")
display(
    detail.groupby("Role")["Signed_Delta_Days"]
    .agg(["count", "median", "min", "max"])
    .reset_index()
)

print("\nLargest 25 absolute mismatches:")
display(
    detail[detail["Absolute_Delta_Days"].notna()]
    .sort_values("Absolute_Delta_Days", ascending=False)
    [[
        "Event_Row", "Symbol", "Role", "Published_Date",
        "Effective_Date", "Expected_Boundary", "Nearest_Boundary",
        "Signed_Delta_Days", "Absolute_Delta_Days",
        "Candidate_Interval_Count", "Nearest_ISIN",
        "Nearest_Start_Confidence", "Nearest_End_Confidence",
        "Release_Title"
    ]].head(25)
)

print("\nNearest matches within 5 days but not exact:")
display(
    detail[
        detail["Absolute_Delta_Days"].between(1, 5)
    ][[
        "Event_Row", "Symbol", "Role", "Effective_Date",
        "Expected_Boundary", "Nearest_Boundary",
        "Signed_Delta_Days", "Nearest_ISIN",
        "Nearest_Start_Confidence", "Nearest_End_Confidence"
    ]].head(30)
)

# ---------------------------------------------------------------
# 4. Existing membership mismatch report summaries.
# ---------------------------------------------------------------
print("\n" + "=" * 78)
print("3. EXISTING MEMBERSHIP MISMATCH AUDIT")
print("=" * 78)

if MISMATCH_AUDIT.exists():
    audit = pd.read_csv(MISMATCH_AUDIT, dtype=str, keep_default_na=False)
    print("Audit rows:", len(audit))
    display(audit.head(20))
else:
    print("Mismatch audit file not found.")

if MISMATCH_EXAMPLES.exists():
    examples = pd.read_csv(
        MISMATCH_EXAMPLES, dtype=str, keep_default_na=False
    )
    print("\nMismatch example rows:", len(examples))
    if "Mismatch" in examples.columns:
        display(
            examples["Mismatch"].value_counts(dropna=False)
            .rename_axis("Mismatch").reset_index(name="Rows")
        )
    display(examples.head(15))
else:
    print("Mismatch examples file not found.")

# Save the measured event-by-boundary differences as a separate report.
n = 1
while True:
    out = BASE / f"interval_boundary_distance_diagnostic_v{n}.csv"
    if not out.exists():
        break
    n += 1

detail.to_csv(out, index=False)
print("\nBoundary-distance report saved to:", out)
print("Read-only diagnostics complete. No underlying data changed.")

INTERVAL QUALITY AND EVENT BOUNDARY DIAGNOSTICS — READ ONLY

1. INTERVAL FILE PROFILE
Rows: 1291
Distinct nonblank symbols: 1074
Blank starts: 0
Blank ends: 501
Invalid start > end ranges: 0
Start-date range: 2010-01-02 00:00:00 to 2026-09-30 00:00:00
End-date range: 2011-01-18 00:00:00 to 2026-09-29 00:00:00

Start confidence counts:


,start_confidence,Rows
0,left_censored,500
1,press_release_exact,327
2,interpolated_long_gap,306
3,interpolated_snapshot_adjacent,20
4,isin_migration_from_INE561H01018,1
...,...,...
137,isin_migration_from_INE814H01011,1
138,isin_migration_from_INE258A01016,1
139,isin_migration_from_INE531F01015,1
140,isin_migration_from_INE237A01028,1



End confidence counts:


,end_confidence,Rows
0,still_active,501
1,press_release_exact,365
2,interpolated_long_gap,275
3,interpolated_snapshot_adjacent,12
4,isin_migration_to_INE389H01022,1
...,...,...
137,isin_migration_to_INE745G01043,1
138,isin_migration_to_INE258A01024,1
139,isin_migration_to_INE531F01023,1
140,isin_migration_to_INE237A01036,1



Intervals per symbol:
Symbols with 1 interval: 874
Symbols with 2 intervals: 183
Symbols with 3+ intervals: 17
Max intervals for one symbol: 3

Overlapping interval pairs: 130


...

,Date,Active_Interval_ISINs,Active_Interval_Symbols,Recorded_ISINs,Candidate_Price_ISINs,Flagged_ISINs,Flagged_Not_In_Intervals,Expected_Priced_But_Unflagged,Priced_Not_In_Intervals,Expected_No_Record,Candidate_Price_Not_In_Intervals
0,2015-01-23,560,504,425,422,420,2,0,7,142,7
1,2016-12-28,623,569,532,472,526,2,0,8,99,8
2,2017-08-28,583,526,517,468,500,1,0,18,84,18
3,2019-10-29,575,531,532,509,516,0,0,16,59,16
4,2022-05-02,594,549,539,539,522,0,0,17,72,17
5,2025-02-21,554,530,535,532,525,0,0,10,29,10
6,2026-09-29,501,501,507,506,499,0,0,8,2,7
7,2026-09-30,501,501,526,525,499,0,0,27,2,27
8,2026-10-19,501,501,499,499,499,0,0,0,2,0



Mismatch example rows: 244


,Mismatch,Rows
0,Priced_Not_In_Intervals,81
1,Candidate_Price_Not_In_Intervals,80
2,Expected_No_Record,78
3,Flagged_Not_In_Intervals,5


,Date,Mismatch,ISIN,Symbol
0,2015-01-23,Flagged_Not_In_Intervals,INE486A01021,CESC
1,2015-01-23,Flagged_Not_In_Intervals,INE884A01027,VAIBHAVGBL
2,2015-01-23,Priced_Not_In_Intervals,INE256C01024,TRIVENI
3,2015-01-23,Priced_Not_In_Intervals,INE310A01015,NILKAMAL
4,2015-01-23,Priced_Not_In_Intervals,INE486A01021,CESC
5,2015-01-23,Priced_Not_In_Intervals,INE802G01018,JETAIRWAYS
6,2015-01-23,Priced_Not_In_Intervals,INE814H01029,ADANIPOWER
7,2015-01-23,Priced_Not_In_Intervals,INE870D01012,NFL
8,2015-01-23,Priced_Not_In_Intervals,INE884A01027,VAIBHAVGBL
9,2015-01-23,Expected_No_Record,INE001A01036,HDFC



Boundary-distance report saved to: /content/nifty500_pit_data/interval_boundary_distance_diagnostic_v1.csv
Read-only diagnostics complete. No underlying data changed.


In [66]:
from pathlib import Path
import pandas as pd

BASE = Path("/content/nifty500_pit_data")
EVENT_FILE = BASE / "nifty500_official_event_candidates_v10.csv"
INTERVAL_FILE = BASE / "pit_intervals_official_2016-04-01_v2_staged.csv"

for p in [EVENT_FILE, INTERVAL_FILE]:
    if not p.exists():
        raise FileNotFoundError(f"Required file not found: {p}")

ANCHOR = pd.Timestamp("2026-09-30")

events = pd.read_csv(EVENT_FILE, dtype=str, keep_default_na=False)
intervals = pd.read_csv(INTERVAL_FILE, dtype=str, keep_default_na=False)

# ------------------------------------------------------------------
# 1. Build the starting roster from intervals active on the anchor date.
# ------------------------------------------------------------------
intervals["_symbol"] = intervals["symbol"].str.upper().str.strip()
intervals["_start"] = pd.to_datetime(
    intervals["start_date"], errors="coerce"
).dt.normalize()
intervals["_end"] = pd.to_datetime(
    intervals["end_date"].replace("", pd.NA), errors="coerce"
).dt.normalize()

anchor_rows = intervals[
    intervals["_start"].notna()
    & (intervals["_start"] <= ANCHOR)
    & (intervals["_end"].isna() | (intervals["_end"] >= ANCHOR))
].copy()

anchor_symbols = set(anchor_rows["_symbol"]) - {""}

print("=" * 78)
print("BACKWARD MEMBERSHIP RECONSTRUCTION — DRY RUN")
print("=" * 78)
print("Anchor date:", ANCHOR.date())
print("Active interval rows:", len(anchor_rows))
print("Distinct active symbols:", len(anchor_symbols))
print("Duplicate symbols in anchor rows:",
      int(anchor_rows["_symbol"].duplicated(keep=False).sum()))

if not anchor_symbols:
    raise ValueError("The anchor roster is empty. Stop before reconstruction.")

# ------------------------------------------------------------------
# 2. Prepare the official event ledger.
# ------------------------------------------------------------------
events["_symbol"] = events["Canonical_Symbol"].str.upper().str.strip()
events["_role"] = events["Role"].str.upper().str.strip()
events["_date"] = pd.to_datetime(
    events["Effective_Date"], errors="coerce"
).dt.normalize()
events["_row"] = events.index

bad = events[
    events["_symbol"].eq("")
    | events["_date"].isna()
    | ~events["_role"].isin(["ENTRY", "EXIT"])
].copy()

if not bad.empty:
    raise ValueError(
        f"{len(bad)} invalid event rows found. Stop before reconstruction."
    )

future_events = events[events["_date"] > ANCHOR].copy()
historical_events = events[events["_date"] <= ANCHOR].copy()

print("Valid event rows:", len(events))
print("Events after anchor date:", len(future_events))
print("Events on/before anchor:", len(historical_events))

if not future_events.empty:
    print("Future-effective events require a later anchor:")
    display(future_events[[
        "Published_Date", "Effective_Date", "Role",
        "Canonical_Symbol", "Source_URL"
    ]].head(20))
    raise ValueError(
        "The anchor roster cannot be reconciled while future-effective events exist."
    )

# ------------------------------------------------------------------
# 3. Walk backward one effective date at a time.
#
# At the end of an effective date:
# - ENTRY symbols should be in the post-change roster.
# - EXIT symbols should be absent from the post-change roster.
#
# To reverse the event date:
# - remove ENTRY symbols;
# - restore EXIT symbols.
# ------------------------------------------------------------------
state_after = set(anchor_symbols)

count_path = []
state_mismatches = []

for date, group in historical_events.groupby("_date", sort=False):
    # Reverse latest event date first.
    pass

event_dates = sorted(
    historical_events["_date"].dropna().unique(),
    reverse=True
)

event_groups = {
    date: group.copy()
    for date, group in historical_events.groupby("_date")
}

for date_value in event_dates:
    date = pd.Timestamp(date_value)
    group = event_groups[date_value]

    entries = set(group.loc[group["_role"].eq("ENTRY"), "_symbol"]) - {""}
    exits = set(group.loc[group["_role"].eq("EXIT"), "_symbol"]) - {""}

    # Check whether the post-event state is consistent with each event.
    entry_absent = entries - state_after
    exit_present = exits & state_after

    for ticker in sorted(entry_absent):
        rows = group[
            group["_symbol"].eq(ticker) & group["_role"].eq("ENTRY")
        ]
        for _, row in rows.iterrows():
            state_mismatches.append({
                "Effective_Date": date.strftime("%Y-%m-%d"),
                "Symbol": ticker,
                "Role": "ENTRY",
                "Issue": "ENTRY_SYMBOL_ABSENT_AFTER_EVENT",
                "Ledger_Row": int(row["_row"]),
                "Release_Title": row["Release_Title"],
                "Source_URL": row["Source_URL"],
            })

    for ticker in sorted(exit_present):
        rows = group[
            group["_symbol"].eq(ticker) & group["_role"].eq("EXIT")
        ]
        for _, row in rows.iterrows():
            state_mismatches.append({
                "Effective_Date": date.strftime("%Y-%m-%d"),
                "Symbol": ticker,
                "Role": "EXIT",
                "Issue": "EXIT_SYMBOL_STILL_PRESENT_AFTER_EVENT",
                "Ledger_Row": int(row["_row"]),
                "Release_Title": row["Release_Title"],
                "Source_URL": row["Source_URL"],
            })

    count_after = len(state_after)

    # Reverse this day's actions to reconstruct the pre-event state.
    state_before = (state_after - entries) | exits

    count_before = len(state_before)
    expected_before = count_after - len(entries) + len(exits)

    count_path.append({
        "Effective_Date": date.strftime("%Y-%m-%d"),
        "Entries": len(entries),
        "Exits": len(exits),
        "Net_Change_Forward": len(entries) - len(exits),
        "Count_After": count_after,
        "Count_Before": count_before,
        "Expected_Count_Before": expected_before,
        "Count_Arithmetic_OK": count_before == expected_before,
        "Entry_Symbols_Absent_After": len(entry_absent),
        "Exit_Symbols_Present_After": len(exit_present),
        "State_Check_OK": len(entry_absent) == 0 and len(exit_present) == 0,
    })

    state_after = state_before

count_df = pd.DataFrame(count_path).sort_values("Effective_Date")
mismatch_df = pd.DataFrame(state_mismatches)

# ------------------------------------------------------------------
# 4. Summary and saved reports.
# ------------------------------------------------------------------
print("\n" + "=" * 78)
print("BACKWARD WALK SUMMARY")
print("=" * 78)

print("Effective dates processed:", len(count_df))
print("State/event inconsistencies:", len(mismatch_df))

if not count_df.empty:
    print("Minimum reconstructed symbol count:",
          int(min(count_df["Count_Before"].min(), count_df["Count_After"].min())))
    print("Maximum reconstructed symbol count:",
          int(max(count_df["Count_Before"].max(), count_df["Count_After"].max())))
    print("Dates with event-state mismatches:",
          int((~count_df["State_Check_OK"]).sum()))

    print("\nCounts at selected historical dates:")
    selected = count_df[
        count_df["Effective_Date"].isin([
            "2015-03-24",
            "2015-03-27",
            "2016-04-01",
            "2019-12-27",
            "2020-03-19",
            "2020-06-26",
            "2024-03-28",
            "2024-09-30",
            "2026-03-30",
            "2026-09-30",
        ])
    ]
    display(selected)

    print("\nLargest state/event inconsistencies by date:")
    issues = count_df[
        (count_df["Entry_Symbols_Absent_After"] > 0)
        | (count_df["Exit_Symbols_Present_After"] > 0)
    ].copy()
    display(issues.sort_values(
        ["Entry_Symbols_Absent_After", "Exit_Symbols_Present_After"],
        ascending=False
    ).head(25))

print("\nFirst 30 individual event-state mismatches:")
if mismatch_df.empty:
    print("None found.")
else:
    display(mismatch_df.head(30))

def next_version_path(stem):
    n = 1
    while True:
        p = BASE / f"{stem}_v{n}.csv"
        if not p.exists():
            return p
        n += 1

count_out = next_version_path("membership_backward_walk_count_path")
mismatch_out = next_version_path("membership_backward_walk_state_mismatches")

count_df.to_csv(count_out, index=False)
mismatch_df.to_csv(mismatch_out, index=False)

print("\nReports saved:")
print(count_out)
print(mismatch_out)
print("\nDry run complete. No event or interval files were modified.")

BACKWARD MEMBERSHIP RECONSTRUCTION — DRY RUN
Anchor date: 2026-09-30
Active interval rows: 501
Distinct active symbols: 501
Duplicate symbols in anchor rows: 0
Valid event rows: 1444
Events after anchor date: 0
Events on/before anchor: 1444

BACKWARD WALK SUMMARY
Effective dates processed: 76
State/event inconsistencies: 43
Minimum reconstructed symbol count: 501
Maximum reconstructed symbol count: 532
Dates with event-state mismatches: 23

Counts at selected historical dates:


,Effective_Date,Entries,Exits,Net_Change_Forward,Count_After,Count_Before,Expected_Count_Before,Count_Arithmetic_OK,Entry_Symbols_Absent_After,Exit_Symbols_Present_After,State_Check_OK
75,2015-03-24,1,1,0,528,528,528,True,0,0,True
74,2015-03-27,18,18,0,527,528,527,False,2,1,False
69,2016-04-01,76,75,1,528,532,527,False,5,0,False
41,2019-12-27,9,9,0,515,515,515,True,0,0,True
39,2020-03-19,1,1,0,515,515,515,True,0,0,True
38,2020-06-26,25,25,0,513,515,513,False,2,0,False
13,2024-03-28,33,33,0,502,504,502,False,2,0,False
12,2024-09-30,27,27,0,502,502,502,True,1,1,False
3,2026-03-30,31,31,0,501,501,501,True,0,0,True
0,2026-09-30,27,27,0,501,501,501,True,0,0,True



Largest state/event inconsistencies by date:


,Effective_Date,Entries,Exits,Net_Change_Forward,Count_After,Count_Before,Expected_Count_Before,Count_Arithmetic_OK,Entry_Symbols_Absent_After,Exit_Symbols_Present_After,State_Check_OK
69,2016-04-01,76,75,1,528,532,527,False,5,0,False
63,2017-03-31,22,22,0,523,526,523,False,3,0,False
74,2015-03-27,18,18,0,527,528,527,False,2,1,False
67,2016-09-30,25,25,0,526,528,526,False,2,0,False
50,2018-09-28,26,26,0,518,520,518,False,2,0,False
43,2019-09-27,22,22,0,515,517,515,False,2,0,False
38,2020-06-26,25,25,0,513,515,513,False,2,0,False
35,2020-09-25,21,21,0,511,513,511,False,2,0,False
13,2024-03-28,33,33,0,502,504,502,False,2,0,False
73,2015-09-28,22,21,1,532,527,531,False,1,5,False



First 30 individual event-state mismatches:


,Effective_Date,Symbol,Role,Issue,Ledger_Row,Release_Title,Source_URL
0,2025-09-30,AKZOINDIA,ENTRY,ENTRY_SYMBOL_ABSENT_AFTER_EVENT,1286,"Replacements in indices w.e.f. September 30, 2025",https://www.niftyindices.com/Press_Release/ind...
1,2024-09-30,GETD,ENTRY,ENTRY_SYMBOL_ABSENT_AFTER_EVENT,1177,Replacements in indices and revision in criter...,https://www.niftyindices.com/Press_Release/ind...
2,2024-09-30,IDEA,EXIT,EXIT_SYMBOL_STILL_PRESENT_AFTER_EVENT,1170,Replacements in indices and revision in criter...,https://www.niftyindices.com/Press_Release/ind...
3,2024-03-28,HBLPOWER,ENTRY,ENTRY_SYMBOL_ABSENT_AFTER_EVENT,1122,"Replacements in indices w.e.f. March 28, 2024",https://www.niftyindices.com/Press_Release/ind...
4,2024-03-28,JKBANK,ENTRY,ENTRY_SYMBOL_ABSENT_AFTER_EVENT,1127,"Replacements in indices w.e.f. March 28, 2024",https://www.niftyindices.com/Press_Release/ind...
5,2023-03-31,MFL,ENTRY,ENTRY_SYMBOL_ABSENT_AFTER_EVENT,1030,"Replacements in indices w.e.f March 31, 2023",https://www.niftyindices.com/Press_Release/ind...
6,2022-09-30,SWANENERGY,ENTRY,ENTRY_SYMBOL_ABSENT_AFTER_EVENT,978,"Replacements in indices w.e.f. September 30, 2022",https://www.niftyindices.com/Press_Release/ind...
7,2022-05-04,GMRINFRA,ENTRY,ENTRY_SYMBOL_ABSENT_AFTER_EVENT,935,"Replacements in indices w.e.f. May 04, 2022",https://www.niftyindices.com/Press_Release/ind...
8,2022-03-31,ZOMATO,ENTRY,ENTRY_SYMBOL_ABSENT_AFTER_EVENT,931,Revision in criteria and replacements in indices,https://www.niftyindices.com/Press_Release/ind...
9,2021-04-29,BURGERKING,ENTRY,ENTRY_SYMBOL_ABSENT_AFTER_EVENT,818,"Replacements in Indices w.e.f. April 29, 2021",https://www.niftyindices.com/Press_Release/ind...



Reports saved:
/content/nifty500_pit_data/membership_backward_walk_count_path_v1.csv
/content/nifty500_pit_data/membership_backward_walk_state_mismatches_v1.csv

Dry run complete. No event or interval files were modified.


In [68]:
from pathlib import Path
import pandas as pd

BASE = Path("/content/nifty500_pit_data")
EVENT_FILE = BASE / "nifty500_official_event_candidates_v10.csv"
INTERVAL_FILE = BASE / "pit_intervals_official_2016-04-01_v2_staged.csv"
MISMATCH_FILE = BASE / "membership_backward_walk_state_mismatches_v1.csv"

for p in [EVENT_FILE, INTERVAL_FILE, MISMATCH_FILE]:
    if not p.exists():
        raise FileNotFoundError(f"Required file not found: {p}")

events = pd.read_csv(EVENT_FILE, dtype=str, keep_default_na=False)
intervals = pd.read_csv(INTERVAL_FILE, dtype=str, keep_default_na=False)
mismatches = pd.read_csv(MISMATCH_FILE, dtype=str, keep_default_na=False)

ANCHOR = pd.Timestamp("2026-09-30")

# ---------------------------------------------------------------
# 1. Attach exact v10 event records to the mismatch rows.
# ---------------------------------------------------------------
events["_Ledger_Row"] = events.index
events["_Symbol"] = events["Canonical_Symbol"].str.upper().str.strip()
events["_Role"] = events["Role"].str.upper().str.strip()
events["_Effective"] = pd.to_datetime(
    events["Effective_Date"], errors="coerce"
).dt.normalize()
events["_Published"] = pd.to_datetime(
    events["Published_Date"], errors="coerce"
).dt.normalize()

mismatches["_Ledger_Row"] = pd.to_numeric(
    mismatches["Ledger_Row"], errors="coerce"
).astype("Int64")

trace = mismatches.merge(
    events,
    on="_Ledger_Row",
    how="left",
    suffixes=("_Mismatch", "_Event"),
    validate="many_to_one"
)

if trace["Canonical_Symbol"].isna().any():
    raise ValueError("At least one mismatch row could not be joined to v10.")

trace["_Symbol"] = trace["Canonical_Symbol"].str.upper().str.strip()
trace["_Effective"] = pd.to_datetime(
    trace["Effective_Date"], errors="coerce"
).dt.normalize()

# ---------------------------------------------------------------
# 2. Determine active intervals as recorded at the anchor date.
# ---------------------------------------------------------------
intervals["_Symbol"] = intervals["symbol"].str.upper().str.strip()
intervals["_Start"] = pd.to_datetime(
    intervals["start_date"], errors="coerce"
).dt.normalize()
intervals["_End"] = pd.to_datetime(
    intervals["end_date"].replace("", pd.NA), errors="coerce"
).dt.normalize()

intervals["_Active_At_Anchor"] = (
    intervals["_Start"].notna()
    & (intervals["_Start"] <= ANCHOR)
    & (intervals["_End"].isna() | (intervals["_End"] >= ANCHOR))
)

intervals["_ISIN_Text"] = intervals["isin"].replace("", "(blank)")

def date_text(value):
    return value.strftime("%Y-%m-%d") if pd.notna(value) else "OPEN/UNKNOWN"

interval_details = []
for ticker, group in intervals.groupby("_Symbol", sort=True):
    descriptions = []
    for _, r in group.sort_values("_Start").iterrows():
        start = date_text(r["_Start"])
        end = date_text(r["_End"])
        sc = r.get("start_confidence", "")
        ec = r.get("end_confidence", "")
        descriptions.append(
            f"{r['_ISIN_Text']} [{start} to {end}; "
            f"start_conf={sc}; end_conf={ec}]"
        )

    active_rows = group[group["_Active_At_Anchor"]]
    interval_details.append({
        "_Symbol": ticker,
        "Interval_Row_Count": len(group),
        "Active_At_Anchor": not active_rows.empty,
        "Active_Anchor_ISINs": " | ".join(
            active_rows["_ISIN_Text"].tolist()
        ),
        "All_Intervals": " || ".join(descriptions),
    })

interval_summary = pd.DataFrame(interval_details)

trace = trace.merge(
    interval_summary,
    on="_Symbol",
    how="left",
    validate="many_to_one"
)

# ---------------------------------------------------------------
# 3. Summarize complete event chronology for each flagged symbol.
# ---------------------------------------------------------------
symbol_chronology = []

for ticker in sorted(trace["_Symbol"].dropna().unique()):
    history = events[events["_Symbol"].eq(ticker)].sort_values(
        ["_Effective", "_Published", "_Ledger_Row"]
    )

    actions = []
    for _, r in history.iterrows():
        if pd.isna(r["_Effective"]):
            continue
        actions.append(
            f"{r['_Effective'].strftime('%Y-%m-%d')} "
            f"{r['_Role']} (pub "
            f"{r['_Published'].strftime('%Y-%m-%d') if pd.notna(r['_Published']) else 'unknown'})"
        )

    symbol_chronology.append({
        "_Symbol": ticker,
        "Total_Recorded_Events": len(history),
        "Event_Chronology": " -> ".join(actions),
    })

chronology = pd.DataFrame(symbol_chronology)

trace = trace.merge(
    chronology,
    on="_Symbol",
    how="left",
    validate="many_to_one"
)

# ---------------------------------------------------------------
# 4. Print the diagnostics.
# ---------------------------------------------------------------
print("=" * 78)
print("BACKWARD-WALK CONFLICT DIAGNOSIS — READ ONLY")
print("=" * 78)
print("Anchor date:", ANCHOR.date())
print("Conflicting event rows:", len(trace))
print("Distinct affected symbols:", trace["_Symbol"].nunique())

print("\n1. CONFLICT COUNTS BY ISSUE")
display(
    trace.groupby("Issue", dropna=False)
    .size().rename("Rows").reset_index()
)

print("\n2. CONFLICTS BY EFFECTIVE DATE")
date_summary = (
    trace.groupby(["Effective_Date", "Issue"], dropna=False)
    .agg(
        Conflicting_Rows=("_Ledger_Row", "count"),
        Symbols=("_Symbol", lambda x: ", ".join(sorted(set(x))))
    )
    .reset_index()
    .sort_values("Effective_Date")
)

display(date_summary)

print("\n3. EVERY CONFLICTING EVENT WITH INTERVAL CONTEXT")
display(trace[[
    "_Ledger_Row",
    "Effective_Date",
    "Symbol",
    "Role",
    "Issue",
    "Active_At_Anchor",
    "Interval_Row_Count",
    "Active_Anchor_ISINs",
    "All_Intervals",
    "Total_Recorded_Events",
    "Event_Chronology",
    "Release_Title",
    "Source_URL"
]].sort_values(["Effective_Date", "Symbol"]))

print("\n4. DISTINCT AFFECTED SYMBOLS AND THEIR INTERVAL HISTORIES")
display(
    interval_summary[
        interval_summary["_Symbol"].isin(trace["_Symbol"].unique())
    ].sort_values("_Symbol")
)

print("\n5. EVENT ROWS FOR INSPECTION")
print("The following mismatch classes need special care:")
print("- ENTRY_SYMBOL_ABSENT_AFTER_EVENT: entry may be wrong-scope, superseded, or followed by a missing exit.")
print("- EXIT_SYMBOL_STILL_PRESENT_AFTER_EVENT: exit may be wrong-scope, revoked, or followed by a missing entry.")
print("These are hypotheses only; no automatic correction is made.")

print("\nRead-only diagnostic complete. No source data changed.")

KeyError: 'Effective_Date'

In [69]:
# ================================================================
# DIAGNOSTIC: EVENT MISMATCHES + OFFICIAL LEDGER + INTERVAL CONTEXT
# Corrected version: prevents duplicate column-name collisions
# ================================================================

from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

# ------------------------------------------------
# 1. FILE PATHS
# ------------------------------------------------

BASE_DIR = Path("/content/nifty500_pit_data")

EVENT_FILE = (
    BASE_DIR / "nifty500_official_event_candidates_v10.csv"
)

INTERVAL_FILE = (
    BASE_DIR / "pit_intervals_official_2016-04-01_v2_staged.csv"
)

# Leave as None to auto-detect the reconstruction mismatch CSV.
# If auto-detection selects the wrong file, enter its exact path here.
MISMATCH_FILE_OVERRIDE = None


# ------------------------------------------------
# 2. VERIFY THE MAIN INPUT FILES
# ------------------------------------------------

for file_path in [EVENT_FILE, INTERVAL_FILE]:
    if not file_path.exists():
        raise FileNotFoundError(
            f"Required file not found: {file_path}\n"
            "Check that the files exist in /content/nifty500_pit_data/."
        )

print("MAIN INPUT FILES")
print(f"Event ledger:     {EVENT_FILE}")
print(f"Membership data:  {INTERVAL_FILE}")


# ------------------------------------------------
# 3. FIND THE BACKWARD-RECONSTRUCTION MISMATCH FILE
# ------------------------------------------------

if MISMATCH_FILE_OVERRIDE is not None:
    MISMATCH_FILE = Path(MISMATCH_FILE_OVERRIDE)

    if not MISMATCH_FILE.exists():
        raise FileNotFoundError(
            f"Mismatch file not found: {MISMATCH_FILE}"
        )

else:
    required_mismatch_columns = {
        "Ledger_Row",
        "Issue",
        "Effective_Date",
        "Symbol",
    }

    candidates = []

    for path in BASE_DIR.rglob("*.csv"):
        # These are inputs, not the mismatch report.
        if path.resolve() in {
            EVENT_FILE.resolve(),
            INTERVAL_FILE.resolve(),
        }:
            continue

        try:
            columns = set(pd.read_csv(path, nrows=0).columns)
        except Exception:
            continue

        if required_mismatch_columns.issubset(columns):
            name = path.name.lower()

            # Prefer the backward-reconstruction diagnostic report.
            score = 0
            if "backward" in name:
                score += 8
            if "reconstruction" in name:
                score += 6
            if "mismatch" in name:
                score += 4
            if "event" in name:
                score += 1

            candidates.append(
                (score, path.stat().st_mtime, path)
            )

    if not candidates:
        raise FileNotFoundError(
            "Could not automatically locate the mismatch CSV.\n"
            "It must contain these columns:\n"
            "Ledger_Row, Issue, Effective_Date, Symbol\n\n"
            "Set MISMATCH_FILE_OVERRIDE near the top of this cell "
            "to the exact path of your mismatch report."
        )

    # Highest filename score first; most recently modified breaks ties.
    candidates.sort(key=lambda item: (item[0], item[1]), reverse=True)
    MISMATCH_FILE = candidates[0][2]

print(f"Mismatch report:  {MISMATCH_FILE}")


# ------------------------------------------------
# 4. LOAD THE FILES
# ------------------------------------------------

events = pd.read_csv(
    EVENT_FILE,
    dtype=str,
    keep_default_na=False,
)

intervals = pd.read_csv(
    INTERVAL_FILE,
    dtype=str,
    keep_default_na=False,
)

mismatches = pd.read_csv(
    MISMATCH_FILE,
    dtype=str,
    keep_default_na=False,
)

print("\nINPUT COUNTS")
print(f"Ledger event rows:       {len(events):,}")
print(f"Membership interval rows:{len(intervals):,}")
print(f"Mismatch report rows:    {len(mismatches):,}")


# ------------------------------------------------
# 5. VALIDATE REQUIRED COLUMNS
# ------------------------------------------------

required_event_columns = {"Effective_Date", "Role"}

if not required_event_columns.issubset(events.columns):
    raise ValueError(
        "Event ledger is missing required columns: "
        f"{sorted(required_event_columns - set(events.columns))}"
    )

if "Canonical_Symbol" not in events.columns and "Symbol" not in events.columns:
    raise ValueError(
        "Event ledger has neither Canonical_Symbol nor Symbol."
    )

required_mismatch_columns = {
    "Ledger_Row",
    "Issue",
    "Effective_Date",
    "Symbol",
}

if not required_mismatch_columns.issubset(mismatches.columns):
    raise ValueError(
        "Mismatch report is missing required columns: "
        f"{sorted(required_mismatch_columns - set(mismatches.columns))}"
    )

if "symbol" not in intervals.columns:
    raise ValueError(
        "Membership interval file does not contain a 'symbol' column."
    )


# ------------------------------------------------
# 6. PREPARE THE EVENT LEDGER JOIN KEY
# ------------------------------------------------

# The mismatch report's Ledger_Row refers to the row index used
# by the backward-reconstruction diagnostic.
events = events.copy()
events["_Ledger_Row"] = np.arange(len(events), dtype=np.int64)

mismatches = mismatches.copy()

mismatches["_Ledger_Row"] = pd.to_numeric(
    mismatches["Ledger_Row"],
    errors="coerce",
).astype("Int64")

# Rename columns from the mismatch report BEFORE merging.
# This avoids Effective_Date_x / Effective_Date_y and similar names.
rename_mismatch_columns = {
    "Effective_Date": "Mismatch_Effective_Date",
    "Symbol": "Mismatch_Symbol",
    "Role": "Mismatch_Role",
    "Issue": "Mismatch_Issue",
    "Release_Title": "Mismatch_Release_Title",
    "Source_URL": "Mismatch_Source_URL",
}

rename_mismatch_columns = {
    old: new
    for old, new in rename_mismatch_columns.items()
    if old in mismatches.columns
}

mismatches = mismatches.rename(
    columns=rename_mismatch_columns
)


# ------------------------------------------------
# 7. MERGE MISMATCHES WITH THE EVENT LEDGER
# ------------------------------------------------

trace = mismatches.merge(
    events,
    on="_Ledger_Row",
    how="left",
    validate="many_to_one",
    indicator="_Event_Merge_Status",
)

trace["Event_Matched"] = trace["_Event_Merge_Status"].eq("both")

unmatched_count = int((~trace["Event_Matched"]).sum())

print("\nLEDGER JOIN CHECK")
print(f"Mismatch rows joined to ledger: {trace['Event_Matched'].sum():,}")
print(f"Mismatch rows not joined:       {unmatched_count:,}")

if unmatched_count:
    print("\nWARNING: Some mismatch rows did not match an event-ledger row.")
    display(
        trace.loc[
            ~trace["Event_Matched"],
            [
                "Ledger_Row",
                "_Ledger_Row",
                "Mismatch_Effective_Date",
                "Mismatch_Symbol",
                "Mismatch_Issue",
            ],
        ].head(30)
    )


# ------------------------------------------------
# 8. NORMALIZE SYMBOLS, DATES, AND ISSUE LABELS
# ------------------------------------------------

# Use the canonical ledger symbol first, with fallback fields
# for rows where the ledger symbol is blank.
if "Canonical_Symbol" in trace.columns:
    canonical_symbol = (
        trace["Canonical_Symbol"]
        .replace("", np.nan)
    )
else:
    canonical_symbol = pd.Series(
        np.nan, index=trace.index, dtype="object"
    )

if "Symbol" in trace.columns:
    ledger_symbol = trace["Symbol"].replace("", np.nan)
else:
    ledger_symbol = pd.Series(
        np.nan, index=trace.index, dtype="object"
    )

mismatch_symbol = trace["Mismatch_Symbol"].replace("", np.nan)

trace["_Symbol"] = (
    canonical_symbol
    .fillna(ledger_symbol)
    .fillna(mismatch_symbol)
    .astype("string")
    .str.upper()
    .str.strip()
)

# The event ledger's date is the authoritative date for this join.
trace["_Effective"] = pd.to_datetime(
    trace["Effective_Date"],
    errors="coerce",
).dt.normalize()

trace["Mismatch_Effective_Date_Parsed"] = pd.to_datetime(
    trace["Mismatch_Effective_Date"],
    errors="coerce",
).dt.normalize()

trace["Issue"] = (
    trace["Mismatch_Issue"]
    .fillna("")
    .astype(str)
    .str.strip()
)

# Keep a readable date column for output.
trace["Effective_Date_Display"] = trace["_Effective"].dt.strftime("%Y-%m-%d")


# ------------------------------------------------
# 9. SECTION 2: CONFLICTS / ISSUES BY EFFECTIVE DATE
# ------------------------------------------------

date_summary = (
    trace.groupby(
        ["_Effective", "Issue"],
        dropna=False,
    )
    .agg(
        Conflicting_Rows=("_Ledger_Row", "nunique"),
        Symbols=(
            "_Symbol",
            lambda values: ", ".join(
                sorted(
                    set(
                        values.dropna()
                        .astype(str)
                        .loc[lambda x: x.ne("")]
                    )
                )
            ),
        ),
    )
    .reset_index()
    .rename(columns={"_Effective": "Effective_Date"})
    .sort_values(
        ["Effective_Date", "Issue"],
        na_position="last",
    )
)

print("\n" + "=" * 90)
print("SECTION 2 — CONFLICTS / ISSUES BY EFFECTIVE DATE")
print("=" * 90)

with pd.option_context(
    "display.max_rows", 200,
    "display.max_columns", 30,
    "display.max_colwidth", 180,
    "display.width", 240,
):
    display(date_summary)


# ------------------------------------------------
# 10. PREPARE MEMBERSHIP INTERVALS FOR CONTEXT
# ------------------------------------------------

intervals = intervals.copy()

intervals["_Symbol"] = (
    intervals["symbol"]
    .astype("string")
    .str.upper()
    .str.strip()
)

# Parse interval boundary dates.
for col in ["start_date", "end_date"]:
    if col not in intervals.columns:
        intervals[col] = ""

intervals["_Interval_Start"] = pd.to_datetime(
    intervals["start_date"],
    errors="coerce",
).dt.normalize()

intervals["_Interval_End"] = pd.to_datetime(
    intervals["end_date"].replace("", np.nan),
    errors="coerce",
).dt.normalize()

# Add optional fields if this staged file does not contain them.
optional_interval_columns = [
    "isin",
    "company_name",
    "start_confidence",
    "end_confidence",
]

for col in optional_interval_columns:
    if col not in intervals.columns:
        intervals[col] = ""

interval_context = intervals[
    [
        "_Symbol",
        "isin",
        "symbol",
        "company_name",
        "_Interval_Start",
        "start_confidence",
        "_Interval_End",
        "end_confidence",
    ]
].rename(
    columns={
        "isin": "Interval_ISIN",
        "symbol": "Interval_Symbol",
        "company_name": "Interval_Company",
        "_Interval_Start": "Interval_Start",
        "start_confidence": "Start_Confidence",
        "_Interval_End": "Interval_End",
        "end_confidence": "End_Confidence",
    }
)


# ------------------------------------------------
# 11. SECTION 3: MISMATCH EVENTS WITH INTERVAL CONTEXT
# ------------------------------------------------

# One symbol may have multiple interval rows, including ISIN migrations.
# This intentionally shows all intervals associated with each symbol.
context = trace.merge(
    interval_context,
    on="_Symbol",
    how="left",
)

# Calculate distances from the event date to each interval boundary.
context["Start_Delta_Days"] = (
    context["Interval_Start"] - context["_Effective"]
).dt.days

context["End_Delta_Days"] = (
    context["Interval_End"] - context["_Effective"]
).dt.days

# Does this interval cover the event's effective date?
context["Interval_Covers_Event_Date"] = (
    context["_Effective"].notna()
    & context["Interval_Start"].notna()
    & (context["Interval_Start"] <= context["_Effective"])
    & (
        context["Interval_End"].isna()
        | (context["Interval_End"] >= context["_Effective"])
    )
)

# Useful context:
# ENTRY -> compare the interval start with the event's effective date.
# EXIT  -> compare the interval end with the event's effective date.
role_normalized = (
    context["Role"]
    .fillna("")
    .astype(str)
    .str.upper()
    .str.strip()
)

context["Relevant_Boundary_Delta_Days"] = np.select(
    [
        role_normalized.eq("ENTRY"),
        role_normalized.eq("EXIT"),
    ],
    [
        context["Start_Delta_Days"].fillna(np.nan),
        context["End_Delta_Days"].fillna(np.nan),
    ],
    default=np.nan,
)

# Put the most useful diagnostic columns first.
desired_context_columns = [
    "Effective_Date_Display",
    "Mismatch_Effective_Date",
    "Role",
    "Canonical_Symbol",
    "Mismatch_Symbol",
    "Issue",
    "Ledger_Row",
    "Event_Matched",
    "Release_Title",
    "Source_URL",
    "Interval_ISIN",
    "Interval_Symbol",
    "Interval_Company",
    "Interval_Start",
    "Start_Confidence",
    "Interval_End",
    "End_Confidence",
    "Interval_Covers_Event_Date",
    "Start_Delta_Days",
    "End_Delta_Days",
    "Relevant_Boundary_Delta_Days",
]

# Include only columns that are available.
available_context_columns = [
    col for col in desired_context_columns
    if col in context.columns
]

context_table = context[available_context_columns].copy()

# Sort chronologically, then by issue and symbol.
sort_columns = [
    col for col in
    ["Effective_Date_Display", "Issue", "Canonical_Symbol"]
    if col in context_table.columns
]

if sort_columns:
    context_table = context_table.sort_values(
        sort_columns,
        na_position="last",
    )

print("\n" + "=" * 90)
print("SECTION 3 — MISMATCH EVENTS WITH MEMBERSHIP INTERVAL CONTEXT")
print("=" * 90)

with pd.option_context(
    "display.max_rows", 250,
    "display.max_columns", 40,
    "display.max_colwidth", 160,
    "display.width", 280,
):
    display(context_table.reset_index(drop=True))


# ------------------------------------------------
# 12. FINAL DIAGNOSTIC COUNTS
# ------------------------------------------------

print("\n" + "=" * 90)
print("FINAL DIAGNOSTIC COUNTS")
print("=" * 90)

print(f"Events in v10 ledger:                    {len(events):,}")
print(f"Rows in mismatch report:                 {len(mismatches):,}")
print(f"Mismatch rows joined to ledger:          {int(trace['Event_Matched'].sum()):,}")
print(f"Mismatch rows with no matching interval: {int(context['Interval_ISIN'].isna().sum()):,}")
print(f"Distinct effective dates in report:      {trace['_Effective'].nunique(dropna=True):,}")
print(f"Distinct symbols in report:              {trace['_Symbol'].nunique(dropna=True):,}")

print("\nNo input files were written or modified by this diagnostic cell.")

MAIN INPUT FILES
Event ledger:     /content/nifty500_pit_data/nifty500_official_event_candidates_v10.csv
Membership data:  /content/nifty500_pit_data/pit_intervals_official_2016-04-01_v2_staged.csv
Mismatch report:  /content/nifty500_pit_data/membership_backward_walk_state_mismatches_v1.csv

INPUT COUNTS
Ledger event rows:       1,444
Membership interval rows:1,291
Mismatch report rows:    43

LEDGER JOIN CHECK
Mismatch rows joined to ledger: 43
Mismatch rows not joined:       0

SECTION 2 — CONFLICTS / ISSUES BY EFFECTIVE DATE


,Effective_Date,Issue,Conflicting_Rows,Symbols
0,2015-03-27,ENTRY_SYMBOL_ABSENT_AFTER_EVENT,2,"FCEL, MCDOWELLN"
1,2015-03-27,EXIT_SYMBOL_STILL_PRESENT_AFTER_EVENT,1,HCLTECH
2,2015-09-28,ENTRY_SYMBOL_ABSENT_AFTER_EVENT,1,SITICABLE
3,2015-09-28,EXIT_SYMBOL_STILL_PRESENT_AFTER_EVENT,5,"BHEL, IOC, LICHSGFIN, MRF, SAIL"
4,2016-04-01,ENTRY_SYMBOL_ABSENT_AFTER_EVENT,5,"ADANITRANS, AEGISCHEM, HITACHIHOM, STYABS, TATAMTRDVR"
5,2016-09-30,ENTRY_SYMBOL_ABSENT_AFTER_EVENT,2,"CROMPGREAV, IDFCBANK"
6,2017-03-31,ENTRY_SYMBOL_ABSENT_AFTER_EVENT,3,"LTI, MINDAIND, STRTECH"
7,2017-09-29,ENTRY_SYMBOL_ABSENT_AFTER_EVENT,1,IBVENTURES
8,2018-02-05,ENTRY_SYMBOL_ABSENT_AFTER_EVENT,1,TIFIN
9,2018-04-02,ENTRY_SYMBOL_ABSENT_AFTER_EVENT,1,HEG



SECTION 3 — MISMATCH EVENTS WITH MEMBERSHIP INTERVAL CONTEXT


,Effective_Date_Display,Mismatch_Effective_Date,Role,Canonical_Symbol,Mismatch_Symbol,Issue,Ledger_Row,Event_Matched,Release_Title,Source_URL,Interval_ISIN,Interval_Symbol,Interval_Company,Interval_Start,Start_Confidence,Interval_End,End_Confidence,Interval_Covers_Event_Date,Start_Delta_Days,End_Delta_Days,Relevant_Boundary_Delta_Days
0,2015-03-27,2015-03-27,ENTRY,FCEL,FCEL,ENTRY_SYMBOL_ABSENT_AFTER_EVENT,12,True,"Change in indices w.e.f March 27, 2015.",https://www.niftyindices.com/Press_Release/ind_prs20022015.pdf,NaN,NaN,NaN,NaT,NaN,NaT,NaN,False,NaN,NaN,NaN
1,2015-03-27,2015-03-27,ENTRY,MCDOWELLN,MCDOWELLN,ENTRY_SYMBOL_ABSENT_AFTER_EVENT,22,True,"Change in indices w.e.f March 27, 2015.",https://www.niftyindices.com/Press_Release/ind_prs20022015.pdf,NaN,NaN,NaN,NaT,NaN,NaT,NaN,False,NaN,NaN,NaN
2,2015-03-27,2015-03-27,EXIT,HCLTECH,HCLTECH,EXIT_SYMBOL_STILL_PRESENT_AFTER_EVENT,24,True,"Change in indices w.e.f March 27, 2015.",https://www.niftyindices.com/Press_Release/ind_prs20022015.pdf,INE860A01027,HCLTECH,HCL Technologies Ltd.,2010-01-02,left_censored,NaT,still_active,True,-1910.0,NaN,NaN
3,2015-09-28,2015-09-28,ENTRY,SITICABLE,SITICABLE,ENTRY_SYMBOL_ABSENT_AFTER_EVENT,60,True,"Change in Indices and Methodology w.e.f September 28, 2015",https://www.niftyindices.com/Press_Release/ind_prs12082015.pdf,NaN,NaN,NaN,NaT,NaN,NaT,NaN,False,NaN,NaN,NaN
4,2015-09-28,2015-09-28,EXIT,BHEL,BHEL,EXIT_SYMBOL_STILL_PRESENT_AFTER_EVENT,64,True,"Change in Indices and Methodology w.e.f September 28, 2015",https://www.niftyindices.com/Press_Release/ind_prs12082015.pdf,INE257A01018,BHEL,Bharat Heavy Electricals Ltd.,2010-01-02,left_censored,2014-01-22,isin_migration_to_INE257A01026,False,-2095.0,-614.0,-614.0
5,2015-09-28,2015-09-28,EXIT,BHEL,BHEL,EXIT_SYMBOL_STILL_PRESENT_AFTER_EVENT,64,True,"Change in Indices and Methodology w.e.f September 28, 2015",https://www.niftyindices.com/Press_Release/ind_prs12082015.pdf,INE257A01026,BHEL,Bharat Heavy Electricals Ltd.,2010-01-02,isin_migration_from_INE257A01018,NaT,still_active,True,-2095.0,NaN,NaN
6,2015-09-28,2015-09-28,EXIT,IOC,IOC,EXIT_SYMBOL_STILL_PRESENT_AFTER_EVENT,67,True,"Change in Indices and Methodology w.e.f September 28, 2015",https://www.niftyindices.com/Press_Release/ind_prs12082015.pdf,INE242A01010,IOC,Indian Oil Corporation Ltd.,2010-01-02,left_censored,NaT,still_active,True,-2095.0,NaN,NaN
7,2015-09-28,2015-09-28,EXIT,LICHSGFIN,LICHSGFIN,EXIT_SYMBOL_STILL_PRESENT_AFTER_EVENT,68,True,"Change in Indices and Methodology w.e.f September 28, 2015",https://www.niftyindices.com/Press_Release/ind_prs12082015.pdf,INE115A01018,LICHSGFIN,LIC Housing Finance Ltd.,2010-01-02,left_censored,2014-01-22,isin_migration_to_INE115A01026,False,-2095.0,-614.0,-614.0
8,2015-09-28,2015-09-28,EXIT,LICHSGFIN,LICHSGFIN,EXIT_SYMBOL_STILL_PRESENT_AFTER_EVENT,68,True,"Change in Indices and Methodology w.e.f September 28, 2015",https://www.niftyindices.com/Press_Release/ind_prs12082015.pdf,INE115A01026,LICHSGFIN,LIC Housing Finance Ltd.,2010-01-02,isin_migration_from_INE115A01018,NaT,still_active,True,-2095.0,NaN,NaN
9,2015-09-28,2015-09-28,EXIT,MRF,MRF,EXIT_SYMBOL_STILL_PRESENT_AFTER_EVENT,69,True,"Change in Indices and Methodology w.e.f September 28, 2015",https://www.niftyindices.com/Press_Release/ind_prs12082015.pdf,INE883A01011,MRF,MRF Ltd.,2010-01-02,left_censored,NaT,still_active,True,-2095.0,NaN,NaN



FINAL DIAGNOSTIC COUNTS
Events in v10 ledger:                    1,444
Rows in mismatch report:                 43
Mismatch rows joined to ledger:          43
Mismatch rows with no matching interval: 25
Distinct effective dates in report:      23
Distinct symbols in report:              43

No input files were written or modified by this diagnostic cell.


In [70]:
# ================================================================
# AUDIT 2015 OFFICIAL CNX 500 EVENT LISTS AGAINST LEDGER V10
# READ-ONLY: does not modify any existing files
# ================================================================

from pathlib import Path
import pandas as pd
import re
from IPython.display import display

BASE_DIR = Path("/content/nifty500_pit_data")
EVENT_FILE = BASE_DIR / "nifty500_official_event_candidates_v10.csv"

if not EVENT_FILE.exists():
    raise FileNotFoundError(f"Cannot find: {EVENT_FILE}")

events = pd.read_csv(EVENT_FILE, dtype=str, keep_default_na=False)

required = {"Effective_Date", "Role", "Source_URL"}
missing_columns = required - set(events.columns)
if missing_columns:
    raise ValueError(f"Missing required ledger columns: {sorted(missing_columns)}")

symbol_col = (
    "Canonical_Symbol"
    if "Canonical_Symbol" in events.columns
    else "Symbol"
    if "Symbol" in events.columns
    else None
)
if symbol_col is None:
    raise ValueError("No Canonical_Symbol or Symbol column found.")

events["_Ledger_Row"] = range(len(events))
events["_Date"] = pd.to_datetime(
    events["Effective_Date"], errors="coerce"
).dt.strftime("%Y-%m-%d")

def normalize_symbol(value):
    """Normalize symbol punctuation for comparison only."""
    return re.sub(r"[^A-Z0-9]", "", str(value).upper().strip())

events["_Symbol_Normalized"] = events[symbol_col].map(normalize_symbol)
events["_Role_Normalized"] = events["Role"].str.upper().str.strip()
events["_URL_Normalized"] = events["Source_URL"].str.lower()

# Lists transcribed from the official CNX 500 sections of the releases.
official_lists = {
    "2015-03-27": {
        "filename": "ind_prs20022015.pdf",
        "exits": [
            "DCW", "EDUCOMP", "ELDERPHARM", "HUBTOWN",
            "ILFSENGG", "IMFA", "LOVABLE", "PFOCUS",
            "RAJTV", "REIAGROLTD", "TI", "VENKEYS",
        ],
        "entries": [
            "FCEL", "GULFOILLUB", "IFBIND", "JKCEMENT",
            "KPRMILL", "KNRCON", "LGBBROSLTD", "MARKSANS",
            "NDTV", "PENIND", "MCDOWELLN", "VINATIORGA",
        ],
    },
    "2015-09-28": {
        "filename": "ind_prs12082015.pdf",
        "exits": [
            "ANSALAPI", "BGRENERGY", "GAMMONIND", "GUJNRECOKE",
            "HINDOILEXP", "HOTELEELA", "IFBIND", "KESORAMIND",
            "MANGCHEFER", "MONNETISPA", "ORCHIDCHEM", "RAMCOIND",
            "RASOYPR", "RTNPOWER",
        ],
        "entries": [
            "ADANIPOWER", "BALKRISIND", "CAMLINFINE", "FLFL",
            "ILFSENGG", "INOXWIND", "KRBL", "METALFORGE",
            "ORIENTCEM", "RICOAUTO", "SITICABLE", "SRIPIPES",
            "SYMPHONY", "VRLLOG",
        ],
    },
}

all_results = []

for effective_date, spec in official_lists.items():
    filename = spec["filename"]

    # Limit this comparison to rows attributed to the relevant release.
    source_rows = events[
        events["_URL_Normalized"].str.contains(
            re.escape(filename.lower()), regex=True, na=False
        )
    ].copy()

    target_rows = source_rows[
        source_rows["_Date"].eq(effective_date)
    ].copy()

    print("\n" + "=" * 100)
    print(f"OFFICIAL REVIEW: {effective_date} | {filename}")
    print("=" * 100)

    print(f"All ledger rows tied to this release: {len(source_rows)}")
    print(f"Rows assigned the expected effective date: {len(target_rows)}")

    expected_by_role = {
        "EXIT": {normalize_symbol(s) for s in spec["exits"]},
        "ENTRY": {normalize_symbol(s) for s in spec["entries"]},
    }

    actual_by_role = {
        role: set(
            target_rows.loc[
                target_rows["_Role_Normalized"].eq(role),
                "_Symbol_Normalized",
            ]
        )
        for role in ["EXIT", "ENTRY"]
    }

    summary_rows = []

    for role in ["EXIT", "ENTRY"]:
        expected = expected_by_role[role]
        actual = actual_by_role[role]

        summary_rows.append({
            "Effective_Date": effective_date,
            "Role": role,
            "Official_Symbol_Count": len(expected),
            "Ledger_Unique_Symbol_Count": len(actual),
            "Missing_From_Ledger": ", ".join(sorted(expected - actual)),
            "Unexpected_In_Ledger": ", ".join(sorted(actual - expected)),
            "Counts_Match": expected == actual,
        })

    summary = pd.DataFrame(summary_rows)
    display(summary)

    # Show each actual event, including ones attributed to the wrong index.
    target_rows["_Expected_Status"] = target_rows.apply(
        lambda row: (
            "EXPECTED"
            if row["_Symbol_Normalized"]
            in expected_by_role.get(row["_Role_Normalized"], set())
            else "UNEXPECTED — CHECK SCOPE"
        ),
        axis=1,
    )

    output_columns = [
        "_Ledger_Row",
        "Effective_Date",
        "Role",
        symbol_col,
        "_Expected_Status",
        "Release_Title",
        "Source_URL",
    ]
    output_columns = [
        col for col in output_columns if col in target_rows.columns
    ]

    print("\nALL LEDGER EVENTS FOR THIS RELEASE AND EFFECTIVE DATE")
    display(
        target_rows[output_columns]
        .sort_values(["Role", symbol_col])
        .reset_index(drop=True)
    )

    # Check whether any rows from this release were assigned another date.
    other_date_rows = source_rows[
        ~source_rows["_Date"].eq(effective_date)
    ]

    if not other_date_rows.empty:
        print("\nRows from this release assigned a different effective date")
        other_cols = [
            col for col in [
                "_Ledger_Row",
                "Effective_Date",
                "Role",
                symbol_col,
                "Release_Title",
                "Source_URL",
            ]
            if col in other_date_rows.columns
        ]
        display(other_date_rows[other_cols].reset_index(drop=True))

    all_results.append(summary)

print("\n" + "=" * 100)
print("FINAL TWO-RELEASE COMPARISON")
print("=" * 100)
display(pd.concat(all_results, ignore_index=True))

print("\nREAD-ONLY AUDIT COMPLETE — no files were modified.")


OFFICIAL REVIEW: 2015-03-27 | ind_prs20022015.pdf
All ledger rows tied to this release: 30
Rows assigned the expected effective date: 30


,Effective_Date,Role,Official_Symbol_Count,Ledger_Unique_Symbol_Count,Missing_From_Ledger,Unexpected_In_Ledger,Counts_Match
0,2015-03-27,EXIT,12,15,,"ADANIENT, HCLTECH, TECHM",False
1,2015-03-27,ENTRY,12,15,,"BPCL, IOC, MRF",False



ALL LEDGER EVENTS FOR THIS RELEASE AND EFFECTIVE DATE


,_Ledger_Row,Effective_Date,Role,Canonical_Symbol,_Expected_Status,Release_Title,Source_URL
0,26,2015-03-27,ENTRY,BPCL,UNEXPECTED — CHECK SCOPE,"Change in indices w.e.f March 27, 2015.",https://www.niftyindices.com/Press_Release/ind...
1,12,2015-03-27,ENTRY,FCEL,EXPECTED,"Change in indices w.e.f March 27, 2015.",https://www.niftyindices.com/Press_Release/ind...
2,13,2015-03-27,ENTRY,GULFOILLUB,EXPECTED,"Change in indices w.e.f March 27, 2015.",https://www.niftyindices.com/Press_Release/ind...
3,14,2015-03-27,ENTRY,IFBIND,EXPECTED,"Change in indices w.e.f March 27, 2015.",https://www.niftyindices.com/Press_Release/ind...
4,28,2015-03-27,ENTRY,IOC,UNEXPECTED — CHECK SCOPE,"Change in indices w.e.f March 27, 2015.",https://www.niftyindices.com/Press_Release/ind...
5,15,2015-03-27,ENTRY,JKCEMENT,EXPECTED,"Change in indices w.e.f March 27, 2015.",https://www.niftyindices.com/Press_Release/ind...
6,17,2015-03-27,ENTRY,KNRCON,EXPECTED,"Change in indices w.e.f March 27, 2015.",https://www.niftyindices.com/Press_Release/ind...
7,16,2015-03-27,ENTRY,KPRMILL,EXPECTED,"Change in indices w.e.f March 27, 2015.",https://www.niftyindices.com/Press_Release/ind...
8,18,2015-03-27,ENTRY,LGBBROSLTD,EXPECTED,"Change in indices w.e.f March 27, 2015.",https://www.niftyindices.com/Press_Release/ind...
9,19,2015-03-27,ENTRY,MARKSANS,EXPECTED,"Change in indices w.e.f March 27, 2015.",https://www.niftyindices.com/Press_Release/ind...



OFFICIAL REVIEW: 2015-09-28 | ind_prs12082015.pdf
All ledger rows tied to this release: 38
Rows assigned the expected effective date: 38


...


ALL LEDGER EVENTS FOR THIS RELEASE AND EFFECTIVE DATE


,_Ledger_Row,Effective_Date,Role,Canonical_Symbol,_Expected_Status,Release_Title,Source_URL
0,50,2015-09-28,ENTRY,ADANIPOWER,EXPECTED,Change in Indices and Methodology w.e.f Septem...,https://www.niftyindices.com/Press_Release/ind...
1,51,2015-09-28,ENTRY,BALKRISIND,EXPECTED,Change in Indices and Methodology w.e.f Septem...,https://www.niftyindices.com/Press_Release/ind...
2,71,2015-09-28,ENTRY,BEL,UNEXPECTED — CHECK SCOPE,Change in Indices and Methodology w.e.f Septem...,https://www.niftyindices.com/Press_Release/ind...
3,52,2015-09-28,ENTRY,CAMLINFINE,EXPECTED,Change in Indices and Methodology w.e.f Septem...,https://www.niftyindices.com/Press_Release/ind...
4,72,2015-09-28,ENTRY,EXIDEIND,UNEXPECTED — CHECK SCOPE,Change in Indices and Methodology w.e.f Septem...,https://www.niftyindices.com/Press_Release/ind...
5,53,2015-09-28,ENTRY,FLFL,EXPECTED,Change in Indices and Methodology w.e.f Septem...,https://www.niftyindices.com/Press_Release/ind...
6,54,2015-09-28,ENTRY,ILFSENGG,EXPECTED,Change in Indices and Methodology w.e.f Septem...,https://www.niftyindices.com/Press_Release/ind...
7,55,2015-09-28,ENTRY,INOXWIND,EXPECTED,Change in Indices and Methodology w.e.f Septem...,https://www.niftyindices.com/Press_Release/ind...
8,56,2015-09-28,ENTRY,KRBL,EXPECTED,Change in Indices and Methodology w.e.f Septem...,https://www.niftyindices.com/Press_Release/ind...
9,65,2015-09-28,ENTRY,LUPIN,UNEXPECTED — CHECK SCOPE,Change in Indices and Methodology w.e.f Septem...,https://www.niftyindices.com/Press_Release/ind...



FINAL TWO-RELEASE COMPARISON


,Effective_Date,Role,Official_Symbol_Count,Ledger_Unique_Symbol_Count,Missing_From_Ledger,Unexpected_In_Ledger,Counts_Match
0,2015-03-27,EXIT,12,15,,"ADANIENT, HCLTECH, TECHM",False
1,2015-03-27,ENTRY,12,15,,"BPCL, IOC, MRF",False
2,2015-09-28,EXIT,14,19,,"BHEL, IOC, LICHSGFIN, MRF, SAIL",False
3,2015-09-28,ENTRY,14,19,,"BEL, EXIDEIND, LUPIN, TECHM, VOLTAS",False



READ-ONLY AUDIT COMPLETE — no files were modified.


In [71]:
# ================================================================
# NIFTY 500 OFFICIAL EVENT LEDGER V11
# Remove 16 verified wrong-index-scope events from v10.
#
# Original v10 remains unchanged.
# Writes:
#   1. nifty500_official_event_candidates_v11.csv
#   2. nifty500_event_ledger_change_log_v6.csv
#   3. nifty500_2015_scope_correction_QA_v1.csv
# ================================================================

from pathlib import Path
import re
import pandas as pd
from IPython.display import display

# ------------------------------------------------
# 1. PATHS
# ------------------------------------------------

BASE_DIR = Path("/content/nifty500_pit_data")

INPUT_FILE = (
    BASE_DIR / "nifty500_official_event_candidates_v10.csv"
)

OUTPUT_FILE = (
    BASE_DIR / "nifty500_official_event_candidates_v11.csv"
)

CHANGELOG_FILE = (
    BASE_DIR / "nifty500_event_ledger_change_log_v6.csv"
)

QA_FILE = (
    BASE_DIR / "nifty500_2015_scope_correction_QA_v1.csv"
)

if not INPUT_FILE.exists():
    raise FileNotFoundError(f"Input ledger not found: {INPUT_FILE}")


# ------------------------------------------------
# 2. LOAD V10 WITHOUT MODIFYING IT
# ------------------------------------------------

events = pd.read_csv(
    INPUT_FILE,
    dtype=str,
    keep_default_na=False,
)

if "Source_URL" not in events.columns:
    raise ValueError("Source_URL column is missing.")

if not {"Effective_Date", "Role"}.issubset(events.columns):
    raise ValueError("Effective_Date or Role column is missing.")

symbol_col = (
    "Canonical_Symbol"
    if "Canonical_Symbol" in events.columns
    else "Symbol"
    if "Symbol" in events.columns
    else None
)

if symbol_col is None:
    raise ValueError(
        "Neither Canonical_Symbol nor Symbol exists in the ledger."
    )

original_count = len(events)


# ------------------------------------------------
# 3. NORMALIZE FIELDS FOR MATCHING
# ------------------------------------------------

def normalize_symbol(value):
    """Normalize punctuation/case for comparison, not for output."""
    return re.sub(
        r"[^A-Z0-9]",
        "",
        str(value).upper().strip(),
    )

date_key = pd.to_datetime(
    events["Effective_Date"],
    errors="coerce",
).dt.strftime("%Y-%m-%d")

role_key = events["Role"].str.upper().str.strip()
symbol_key = events[symbol_col].map(normalize_symbol)
url_key = events["Source_URL"].str.lower()


# ------------------------------------------------
# 4. EXACT LIST OF CONFIRMED WRONG-SCOPE EVENTS
# ------------------------------------------------

# Each record specifies:
# Source PDF, effective date, event role, symbol, reason.

removal_rules = [
    # February 20, 2015 release:
    # These six rows belong to LIX15 / LIX15 Midcap, not CNX 500.
    ("ind_prs20022015.pdf", "2015-03-27", "EXIT",  "ADANIENT"),
    ("ind_prs20022015.pdf", "2015-03-27", "EXIT",  "HCLTECH"),
    ("ind_prs20022015.pdf", "2015-03-27", "EXIT",  "TECHM"),
    ("ind_prs20022015.pdf", "2015-03-27", "ENTRY", "BPCL"),
    ("ind_prs20022015.pdf", "2015-03-27", "ENTRY", "IOC"),
    ("ind_prs20022015.pdf", "2015-03-27", "ENTRY", "MRF"),

    # August 12, 2015 release:
    # These ten rows belong to LIX15 / LIX15 Midcap, not CNX 500.
    ("ind_prs12082015.pdf", "2015-09-28", "EXIT",  "BHEL"),
    ("ind_prs12082015.pdf", "2015-09-28", "EXIT",  "IOC"),
    ("ind_prs12082015.pdf", "2015-09-28", "EXIT",  "LICHSGFIN"),
    ("ind_prs12082015.pdf", "2015-09-28", "EXIT",  "MRF"),
    ("ind_prs12082015.pdf", "2015-09-28", "EXIT",  "SAIL"),
    ("ind_prs12082015.pdf", "2015-09-28", "ENTRY", "BEL"),
    ("ind_prs12082015.pdf", "2015-09-28", "ENTRY", "EXIDEIND"),
    ("ind_prs12082015.pdf", "2015-09-28", "ENTRY", "LUPIN"),
    ("ind_prs12082015.pdf", "2015-09-28", "ENTRY", "TECHM"),
    ("ind_prs12082015.pdf", "2015-09-28", "ENTRY", "VOLTAS"),
]

assert len(removal_rules) == 16


# ------------------------------------------------
# 5. PRE-FLIGHT: REQUIRE EXACTLY ONE MATCH PER RULE
# ------------------------------------------------

rows_to_remove = []
change_log_records = []

for filename, effective_date, role, symbol in removal_rules:

    mask = (
        url_key.str.contains(filename, regex=False, na=False)
        & date_key.eq(effective_date)
        & role_key.eq(role)
        & symbol_key.eq(normalize_symbol(symbol))
    )

    matching_indices = events.index[mask].tolist()

    if len(matching_indices) != 1:
        matching_rows = events.loc[mask]

        print(
            f"\nERROR: Expected exactly one match but found "
            f"{len(matching_indices)}."
        )
        print(
            f"File={filename}, date={effective_date}, "
            f"role={role}, symbol={symbol}"
        )

        display(matching_rows)

        raise ValueError(
            "Stopping without writing output. "
            "Review the mismatched removal rule."
        )

    idx = matching_indices[0]
    rows_to_remove.append(idx)

    source_row = events.loc[idx].to_dict()

    source_row["Original_V10_Row"] = idx
    source_row["Change_Action"] = "REMOVE"
    source_row["Correction_Reason"] = (
        "Wrong index scope: event belongs to LIX15 or "
        "LIX15 Midcap, not CNX 500."
    )
    source_row["Correction_Evidence"] = (
        f"Official release {filename}; "
        f"effective date {effective_date}; "
        f"role {role}; symbol {symbol}."
    )

    change_log_records.append(source_row)

if len(set(rows_to_remove)) != 16:
    raise ValueError(
        "The removal rules did not resolve to 16 distinct ledger rows."
    )

print("PRE-FLIGHT CHECK PASSED")
print(f"V10 rows:                  {original_count:,}")
print(f"Confirmed unique removals: {len(set(rows_to_remove))}")


# ------------------------------------------------
# 6. CREATE V11 AS A NEW DATAFRAME
# ------------------------------------------------

# Do not overwrite the source file.
v11 = (
    events.drop(index=rows_to_remove)
    .reset_index(drop=True)
    .copy()
)

if len(v11) != original_count - 16:
    raise AssertionError(
        f"Expected {original_count - 16} rows; got {len(v11)}."
    )

change_log = pd.DataFrame(change_log_records)


# ------------------------------------------------
# 7. VERIFY THE OFFICIAL CNX 500 LISTS FOR BOTH RELEASES
# ------------------------------------------------

official_lists = {
    ("ind_prs20022015.pdf", "2015-03-27"): {
        "EXIT": [
            "DCW", "EDUCOMP", "ELDERPHARM", "HUBTOWN",
            "ILFSENGG", "IMFA", "LOVABLE", "PFOCUS",
            "RAJTV", "REIAGROLTD", "TI", "VENKEYS",
        ],
        "ENTRY": [
            "FCEL", "GULFOILLUB", "IFBIND", "JKCEMENT",
            "KPRMILL", "KNRCON", "LGBBROSLTD", "MARKSANS",
            "NDTV", "PENIND", "MCDOWELLN", "VINATIORGA",
        ],
    },
    ("ind_prs12082015.pdf", "2015-09-28"): {
        "EXIT": [
            "ANSALAPI", "BGRENERGY", "GAMMONIND", "GUJNRECOKE",
            "HINDOILEXP", "HOTELEELA", "IFBIND", "KESORAMIND",
            "MANGCHEFER", "MONNETISPA", "ORCHIDCHEM", "RAMCOIND",
            "RASOYPR", "RTNPOWER",
        ],
        "ENTRY": [
            "ADANIPOWER", "BALKRISIND", "CAMLINFINE", "FLFL",
            "ILFSENGG", "INOXWIND", "KRBL", "METALFORGE",
            "ORIENTCEM", "RICOAUTO", "SITICABLE", "SRIPIPES",
            "SYMPHONY", "VRLLOG",
        ],
    },
}

v11_date = pd.to_datetime(
    v11["Effective_Date"], errors="coerce"
).dt.strftime("%Y-%m-%d")

v11_role = v11["Role"].str.upper().str.strip()
v11_symbol = v11[symbol_col].map(normalize_symbol)
v11_url = v11["Source_URL"].str.lower()

qa_records = []

for (filename, effective_date), expected_roles in official_lists.items():

    scope_mask = (
        v11_url.str.contains(filename, regex=False, na=False)
        & v11_date.eq(effective_date)
    )

    scoped = v11.loc[scope_mask]

    for role in ["EXIT", "ENTRY"]:
        expected = {
            normalize_symbol(s)
            for s in expected_roles[role]
        }

        actual = set(
            v11_symbol.loc[
                scope_mask & v11_role.eq(role)
            ]
        )

        missing = sorted(expected - actual)
        unexpected = sorted(actual - expected)

        qa_records.append({
            "Source_PDF": filename,
            "Effective_Date": effective_date,
            "Role": role,
            "Official_Unique_Count": len(expected),
            "Ledger_Unique_Count": len(actual),
            "Missing_From_Ledger": ", ".join(missing),
            "Unexpected_In_Ledger": ", ".join(unexpected),
            "Pass": (
                expected == actual
                and len(actual) == len(scoped.loc[
                    v11_role.loc[scope_mask].eq(role)
                ])
            ),
        })

qa = pd.DataFrame(qa_records)

if not qa["Pass"].all():
    print("\nQA FAILED — do not use v11 yet.")
    display(qa)
    raise AssertionError(
        "The corrected ledger does not exactly match "
        "one or more official CNX 500 event lists."
    )


# ------------------------------------------------
# 8. GLOBAL EVENT KEY AND ROLE-CONFLICT QA
# ------------------------------------------------

global_keys = pd.DataFrame({
    "Effective_Date": v11_date,
    "Role": v11_role,
    "Symbol": v11_symbol,
})

duplicate_event_rows = global_keys.duplicated(
    ["Effective_Date", "Role", "Symbol"],
    keep=False,
)

role_conflicts = (
    global_keys.groupby(
        ["Effective_Date", "Symbol"],
        dropna=False,
    )["Role"]
    .nunique()
    .reset_index(name="Distinct_Roles")
)

role_conflicts = role_conflicts[
    role_conflicts["Distinct_Roles"] > 1
]

print("\n" + "=" * 80)
print("OFFICIAL EVENT LEDGER V11 — QA")
print("=" * 80)

print(f"V10 events:                      {original_count:,}")
print(f"Removed wrong-scope rows:        {len(rows_to_remove)}")
print(f"V11 events:                      {len(v11):,}")
print(f"Duplicate event-key rows:        {int(duplicate_event_rows.sum())}")
print(f"Date/symbol cross-role conflicts:{len(role_conflicts)}")

print("\nOFFICIAL LIST MATCHES")
display(qa)

print("\nREMOVED EVENTS")
removed_display_columns = [
    "Original_V10_Row",
    "Effective_Date",
    "Role",
    symbol_col,
    "Source_URL",
    "Correction_Reason",
]
removed_display_columns = [
    c for c in removed_display_columns
    if c in change_log.columns
]
display(
    change_log[removed_display_columns]
    .sort_values(["Effective_Date", "Role", symbol_col])
    .reset_index(drop=True)
)


# ------------------------------------------------
# 9. WRITE NEW VERSIONED FILES
# ------------------------------------------------

v11.to_csv(OUTPUT_FILE, index=False)
change_log.to_csv(CHANGELOG_FILE, index=False)
qa.to_csv(QA_FILE, index=False)

print("\nFILES WRITTEN")
print(f"Corrected ledger: {OUTPUT_FILE}")
print(f"Change log:       {CHANGELOG_FILE}")
print(f"Official-list QA: {QA_FILE}")

print("\nV10 remains unchanged.")
print(
    "\nIMPORTANT: V11 is a corrected candidate ledger, not yet "
    "a fully validated historical membership timeline."
)

PRE-FLIGHT CHECK PASSED
V10 rows:                  1,444
Confirmed unique removals: 16

OFFICIAL EVENT LEDGER V11 — QA
V10 events:                      1,444
Removed wrong-scope rows:        16
V11 events:                      1,428
Duplicate event-key rows:        0
Date/symbol cross-role conflicts:0

OFFICIAL LIST MATCHES


,Source_PDF,Effective_Date,Role,Official_Unique_Count,Ledger_Unique_Count,Missing_From_Ledger,Unexpected_In_Ledger,Pass
0,ind_prs20022015.pdf,2015-03-27,EXIT,12,12,,,True
1,ind_prs20022015.pdf,2015-03-27,ENTRY,12,12,,,True
2,ind_prs12082015.pdf,2015-09-28,EXIT,14,14,,,True
3,ind_prs12082015.pdf,2015-09-28,ENTRY,14,14,,,True



REMOVED EVENTS


,Original_V10_Row,Effective_Date,Role,Canonical_Symbol,Source_URL,Correction_Reason
0,26,2015-03-27,ENTRY,BPCL,https://www.niftyindices.com/Press_Release/ind...,Wrong index scope: event belongs to LIX15 or L...
1,28,2015-03-27,ENTRY,IOC,https://www.niftyindices.com/Press_Release/ind...,Wrong index scope: event belongs to LIX15 or L...
2,29,2015-03-27,ENTRY,MRF,https://www.niftyindices.com/Press_Release/ind...,Wrong index scope: event belongs to LIX15 or L...
3,27,2015-03-27,EXIT,ADANIENT,https://www.niftyindices.com/Press_Release/ind...,Wrong index scope: event belongs to LIX15 or L...
4,24,2015-03-27,EXIT,HCLTECH,https://www.niftyindices.com/Press_Release/ind...,Wrong index scope: event belongs to LIX15 or L...
5,25,2015-03-27,EXIT,TECHM,https://www.niftyindices.com/Press_Release/ind...,Wrong index scope: event belongs to LIX15 or L...
6,71,2015-09-28,ENTRY,BEL,https://www.niftyindices.com/Press_Release/ind...,Wrong index scope: event belongs to LIX15 or L...
7,72,2015-09-28,ENTRY,EXIDEIND,https://www.niftyindices.com/Press_Release/ind...,Wrong index scope: event belongs to LIX15 or L...
8,65,2015-09-28,ENTRY,LUPIN,https://www.niftyindices.com/Press_Release/ind...,Wrong index scope: event belongs to LIX15 or L...
9,66,2015-09-28,ENTRY,TECHM,https://www.niftyindices.com/Press_Release/ind...,Wrong index scope: event belongs to LIX15 or L...



FILES WRITTEN
Corrected ledger: /content/nifty500_pit_data/nifty500_official_event_candidates_v11.csv
Change log:       /content/nifty500_pit_data/nifty500_event_ledger_change_log_v6.csv
Official-list QA: /content/nifty500_pit_data/nifty500_2015_scope_correction_QA_v1.csv

V10 remains unchanged.

IMPORTANT: V11 is a corrected candidate ledger, not yet a fully validated historical membership timeline.


In [72]:
# ================================================================
# BACKWARD MEMBERSHIP RECONSTRUCTION — V11
#
# Anchor: 2026-09-30 staged membership roster
# Event ledger: official_event_candidates_v11.csv
#
# Creates:
#   membership_backward_walk_state_mismatches_v2.csv
#   membership_backward_walk_date_summary_v2.csv
#
# Does NOT modify the input ledger or membership intervals.
# ================================================================

from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

# ------------------------------------------------
# 1. INPUTS AND OUTPUTS
# ------------------------------------------------

BASE_DIR = Path("/content/nifty500_pit_data")

EVENT_FILE = (
    BASE_DIR / "nifty500_official_event_candidates_v11.csv"
)

INTERVAL_FILE = (
    BASE_DIR / "pit_intervals_official_2016-04-01_v2_staged.csv"
)

MISMATCH_OUT = (
    BASE_DIR / "membership_backward_walk_state_mismatches_v2.csv"
)

SUMMARY_OUT = (
    BASE_DIR / "membership_backward_walk_date_summary_v2.csv"
)

ANCHOR_DATE = pd.Timestamp("2026-09-30")

for path in [EVENT_FILE, INTERVAL_FILE]:
    if not path.exists():
        raise FileNotFoundError(f"Required input file missing: {path}")

# ------------------------------------------------
# 2. LOAD INPUTS
# ------------------------------------------------

events = pd.read_csv(EVENT_FILE, dtype=str, keep_default_na=False)
intervals = pd.read_csv(INTERVAL_FILE, dtype=str, keep_default_na=False)

symbol_col = (
    "Canonical_Symbol"
    if "Canonical_Symbol" in events.columns
    else "Symbol"
    if "Symbol" in events.columns
    else None
)

required_event_cols = {"Effective_Date", "Role", "Source_URL"}
if symbol_col is None or not required_event_cols.issubset(events.columns):
    raise ValueError(
        "Event ledger is missing a required symbol/date/role/source column."
    )

required_interval_cols = {"symbol", "start_date", "end_date"}
if not required_interval_cols.issubset(intervals.columns):
    raise ValueError(
        "Interval file needs symbol, start_date, and end_date columns."
    )

if "Release_Title" not in events.columns:
    events["Release_Title"] = ""

# Retain the row number of the v11 ledger for diagnostic traceability.
events = events.copy()
events["_Ledger_Row"] = np.arange(len(events), dtype=int)

events["_Effective"] = pd.to_datetime(
    events["Effective_Date"],
    errors="coerce"
).dt.normalize()

events["_Role"] = events["Role"].str.upper().str.strip()

events["_Symbol"] = (
    events[symbol_col].astype(str).str.upper().str.strip()
)

if events["_Effective"].isna().any():
    bad = events.loc[
        events["_Effective"].isna(),
        ["_Ledger_Row", "Effective_Date", "_Role", "_Symbol"]
    ]
    display(bad.head(30))
    raise ValueError("Some ledger events have an invalid effective date.")

if events["_Symbol"].eq("").any():
    raise ValueError("Some ledger events have a blank symbol.")

invalid_roles = sorted(set(events["_Role"]) - {"ENTRY", "EXIT"})
if invalid_roles:
    raise ValueError(f"Unexpected event roles found: {invalid_roles}")

# Check duplicate event keys.
dupes = events.duplicated(
    subset=["_Effective", "_Role", "_Symbol"],
    keep=False
)

if dupes.any():
    print("Duplicate event keys detected:")
    display(
        events.loc[
            dupes,
            ["_Ledger_Row", "Effective_Date", "_Role", "_Symbol",
             "Release_Title", "Source_URL"]
        ].sort_values(["Effective_Date", "_Role", "_Symbol"])
    )
    raise ValueError("Resolve duplicate event keys before reconstruction.")

# ------------------------------------------------
# 3. PREPARE INTERVAL BOUNDARIES
# ------------------------------------------------

intervals = intervals.copy()

intervals["_Symbol"] = (
    intervals["symbol"].astype(str).str.upper().str.strip()
)

intervals["_Start"] = pd.to_datetime(
    intervals["start_date"],
    errors="coerce"
).dt.normalize()

end_raw = intervals["end_date"].astype(str).str.strip()

intervals["_End"] = pd.to_datetime(
    end_raw.replace("", np.nan),
    errors="coerce"
).dt.normalize()

invalid_start = intervals["start_date"].astype(str).str.strip().eq("") | (
    intervals["_Start"].isna()
)

invalid_end = end_raw.ne("") & intervals["_End"].isna()

if invalid_start.any() or invalid_end.any():
    print("Invalid interval boundary dates detected.")
    display(
        intervals.loc[
            invalid_start | invalid_end,
            ["symbol", "start_date", "end_date", "_Start", "_End"]
        ].head(50)
    )
    raise ValueError("Fix invalid interval boundaries before reconstructing.")

invalid_ranges = (
    intervals["_End"].notna()
    & (intervals["_Start"] > intervals["_End"])
)

if invalid_ranges.any():
    print("Intervals with end_date earlier than start_date:")
    display(
        intervals.loc[
            invalid_ranges,
            ["symbol", "start_date", "end_date"]
        ].head(50)
    )
    raise ValueError("Invalid date ranges found.")

# ------------------------------------------------
# 4. BUILD THE ANCHOR ROSTER
# ------------------------------------------------

# The interval's end date is inclusive.
# Blank end_date means the interval has not yet ended.
active_at_anchor = intervals.loc[
    (intervals["_Start"] <= ANCHOR_DATE)
    & (
        intervals["_End"].isna()
        | (intervals["_End"] >= ANCHOR_DATE)
    )
].copy()

anchor_symbols = set(
    active_at_anchor.loc[
        active_at_anchor["_Symbol"].ne(""),
        "_Symbol"
    ].unique()
)

if not anchor_symbols:
    raise ValueError("The anchor roster is empty.")

print("=" * 85)
print("V11 BACKWARD MEMBERSHIP RECONSTRUCTION")
print("=" * 85)
print(f"Anchor date:                         {ANCHOR_DATE.date()}")
print(f"Active interval rows at anchor:      {len(active_at_anchor):,}")
print(f"Unique symbols in anchor roster:     {len(anchor_symbols):,}")
print(f"V11 event ledger rows:               {len(events):,}")

print(
    "\nNote: active interval rows can exceed unique symbols when "
    "multiple ISINs represent the same company."
)

# ------------------------------------------------
# 5. SELECT EVENTS ON OR BEFORE THE ANCHOR
# ------------------------------------------------

future_events = events.loc[
    events["_Effective"] > ANCHOR_DATE
].copy()

eligible_events = events.loc[
    events["_Effective"] <= ANCHOR_DATE
].copy()

print(f"Events after anchor (not processed): {len(future_events):,}")
print(f"Events on/before anchor:              {len(eligible_events):,}")
print(
    f"Distinct effective dates processed:  "
    f"{eligible_events['_Effective'].nunique():,}"
)

if len(future_events):
    print("\nFuture-dated events found:")
    display(
        future_events[
            ["_Ledger_Row", "Effective_Date", "_Role", "_Symbol", "Source_URL"]
        ].head(30)
    )

# Every event date must be checked against the state after that change.
event_dates = sorted(
    eligible_events["_Effective"].unique(),
    reverse=True
)

# ------------------------------------------------
# 6. WALK BACKWARD THROUGH ALL EVENT DATES
# ------------------------------------------------

current_symbols = set(anchor_symbols)

mismatch_records = []
daily_records = []

for event_date in event_dates:

    date = pd.Timestamp(event_date)

    day_events = eligible_events.loc[
        eligible_events["_Effective"].eq(date)
    ].copy()

    entry_events = day_events.loc[day_events["_Role"].eq("ENTRY")]
    exit_events = day_events.loc[day_events["_Role"].eq("EXIT")]

    entry_symbols = set(entry_events["_Symbol"].unique())
    exit_symbols = set(exit_events["_Symbol"].unique())

    if entry_symbols & exit_symbols:
        raise ValueError(
            f"Same symbol has ENTRY and EXIT on {date.date()}: "
            f"{sorted(entry_symbols & exit_symbols)}"
        )

    # This set represents the membership immediately AFTER the effective
    # changes on this date.
    state_after = set(current_symbols)
    count_after = len(state_after)

    date_mismatch_records = []

    # Check all entrant events against the post-change roster.
    for _, row in entry_events.iterrows():
        symbol = row["_Symbol"]

        if symbol not in state_after:
            issue = "ENTRY_SYMBOL_ABSENT_AFTER_EVENT"

            rec = {
                "Effective_Date": date.strftime("%Y-%m-%d"),
                "Symbol": symbol,
                "Role": "ENTRY",
                "Issue": issue,
                "Ledger_Row": int(row["_Ledger_Row"]),
                "Release_Title": row["Release_Title"],
                "Source_URL": row["Source_URL"],
                "Count_After": count_after,
            }

            mismatch_records.append(rec)
            date_mismatch_records.append(rec)

    # Check all exit events against the post-change roster.
    for _, row in exit_events.iterrows():
        symbol = row["_Symbol"]

        if symbol in state_after:
            issue = "EXIT_SYMBOL_STILL_PRESENT_AFTER_EVENT"

            rec = {
                "Effective_Date": date.strftime("%Y-%m-%d"),
                "Symbol": symbol,
                "Role": "EXIT",
                "Issue": issue,
                "Ledger_Row": int(row["_Ledger_Row"]),
                "Release_Title": row["Release_Title"],
                "Source_URL": row["Source_URL"],
                "Count_After": count_after,
            }

            mismatch_records.append(rec)
            date_mismatch_records.append(rec)

    # Reverse the event to reconstruct the roster immediately BEFORE it:
    # - Remove entrants.
    # - Restore exits.
    state_before = (state_after - entry_symbols) | exit_symbols

    # Apply the same event forward again as a set-level consistency check.
    replayed_after = (state_before - exit_symbols) | entry_symbols

    count_before = len(state_before)

    daily_records.append({
        "Effective_Date": date.strftime("%Y-%m-%d"),
        "Entry_Count": len(entry_symbols),
        "Exit_Count": len(exit_symbols),
        "Count_After": count_after,
        "Count_Before": count_before,
        "Mismatch_Count": len(date_mismatch_records),
        "Mismatch_Symbols": ", ".join(
            sorted({r["Symbol"] for r in date_mismatch_records})
        ),
        "State_Replay_Consistent": replayed_after == state_after,
        "State_Checks_Pass": len(date_mismatch_records) == 0,
    })

    # Continue backward from the inferred pre-change roster.
    current_symbols = state_before

# ------------------------------------------------
# 7. BUILD REPORTS
# ------------------------------------------------

mismatch_columns = [
    "Effective_Date",
    "Symbol",
    "Role",
    "Issue",
    "Ledger_Row",
    "Release_Title",
    "Source_URL",
    "Count_After",
]

mismatches = pd.DataFrame(
    mismatch_records,
    columns=mismatch_columns
)

daily_summary = pd.DataFrame(daily_records)

if not mismatches.empty:
    mismatches = mismatches.sort_values(
        ["Effective_Date", "Issue", "Symbol"]
    ).reset_index(drop=True)

if not daily_summary.empty:
    daily_summary = daily_summary.sort_values(
        "Effective_Date"
    ).reset_index(drop=True)

# ------------------------------------------------
# 8. OVERALL RESULTS
# ------------------------------------------------

print("\n" + "=" * 85)
print("OVERALL BACKWARD-WALK RESULTS")
print("=" * 85)

print(f"Effective dates processed:          {len(daily_summary):,}")
print(f"Event-state mismatches:              {len(mismatches):,}")
print(
    f"Dates with at least one mismatch:    "
    f"{int((daily_summary['Mismatch_Count'] > 0).sum()):,}"
)
print(
    f"Dates with all event checks passing: "
    f"{int(daily_summary['State_Checks_Pass'].sum()):,}"
)
print(
    f"Minimum reconstructed roster size:   "
    f"{int(daily_summary['Count_Before'].min()):,}"
)
print(
    f"Maximum reconstructed roster size:   "
    f"{int(daily_summary['Count_Before'].max()):,}"
)
print(f"Old v1 mismatch report reused:        No")

# ------------------------------------------------
# 9. DISPLAY SUMMARY AND MISMATCHES
# ------------------------------------------------

print("\n" + "=" * 85)
print("DATE SUMMARY — ALL DATES WITH MISMATCHES")
print("=" * 85)

dates_with_mismatches = daily_summary.loc[
    daily_summary["Mismatch_Count"] > 0
]

with pd.option_context(
    "display.max_rows", 200,
    "display.max_columns", 20,
    "display.max_colwidth", 140,
    "display.width", 220,
):
    display(dates_with_mismatches)

print("\n" + "=" * 85)
print("MISMATCHES — FIRST 100 ROWS")
print("=" * 85)

with pd.option_context(
    "display.max_rows", 100,
    "display.max_columns", 20,
    "display.max_colwidth", 140,
    "display.width", 220,
):
    display(mismatches.head(100))

print("\n" + "=" * 85)
print("SELECTED HISTORICAL EVENT DATES")
print("=" * 85)

selected_dates = [
    "2015-03-27",
    "2015-09-28",
    "2016-04-01",
    "2017-03-16",
    "2018-03-13",
    "2019-12-27",
    "2020-03-19",
    "2020-06-26",
    "2024-03-28",
    "2024-09-30",
    "2026-03-30",
    "2026-09-30",
]

display(
    daily_summary.loc[
        daily_summary["Effective_Date"].isin(selected_dates)
    ].reset_index(drop=True)
)

# ------------------------------------------------
# 10. SAVE NEW REPORTS
# ------------------------------------------------

mismatches.to_csv(MISMATCH_OUT, index=False)
daily_summary.to_csv(SUMMARY_OUT, index=False)

print("\n" + "=" * 85)
print("NEW REPORTS SAVED")
print("=" * 85)
print(f"Mismatch report: {MISMATCH_OUT}")
print(f"Daily summary:   {SUMMARY_OUT}")
print("\nInput event ledger and membership intervals were not modified.")

V11 BACKWARD MEMBERSHIP RECONSTRUCTION
Anchor date:                         2026-09-30
Active interval rows at anchor:      501
Unique symbols in anchor roster:     501
V11 event ledger rows:               1,428

Note: active interval rows can exceed unique symbols when multiple ISINs represent the same company.
Events after anchor (not processed): 0
Events on/before anchor:              1,428
Distinct effective dates processed:  76

OVERALL BACKWARD-WALK RESULTS
Effective dates processed:          76
Event-state mismatches:              37
Dates with at least one mismatch:    23
Dates with all event checks passing: 53
Minimum reconstructed roster size:   501
Maximum reconstructed roster size:   534
Old v1 mismatch report reused:        No

DATE SUMMARY — ALL DATES WITH MISMATCHES


,Effective_Date,Entry_Count,Exit_Count,Count_After,Count_Before,Mismatch_Count,Mismatch_Symbols,State_Replay_Consistent,State_Checks_Pass
1,2015-03-27,15,15,532,534,2,"FCEL, MCDOWELLN",False,False
2,2015-09-28,17,16,532,532,1,SITICABLE,False,False
6,2016-04-01,76,75,528,532,5,"ADANITRANS, AEGISCHEM, HITACHIHOM, STYABS, TATAMTRDVR",False,False
8,2016-09-30,25,25,526,528,2,"CROMPGREAV, IDFCBANK",False,False
12,2017-03-31,22,22,523,526,3,"LTI, MINDAIND, STRTECH",False,False
16,2017-09-29,24,24,522,523,1,IBVENTURES,False,False
20,2018-02-05,4,4,521,522,1,TIFIN,False,False
22,2018-04-02,24,24,520,521,1,HEG,False,False
25,2018-09-28,26,26,518,520,2,"PHILIPCARB, RNAM",False,False
28,2019-03-29,25,25,517,518,1,ESSELPACK,False,False



MISMATCHES — FIRST 100 ROWS


,Effective_Date,Symbol,Role,Issue,Ledger_Row,Release_Title,Source_URL,Count_After
0,2015-03-27,FCEL,ENTRY,ENTRY_SYMBOL_ABSENT_AFTER_EVENT,12,"Change in indices w.e.f March 27, 2015.",https://www.niftyindices.com/Press_Release/ind_prs20022015.pdf,532
1,2015-03-27,MCDOWELLN,ENTRY,ENTRY_SYMBOL_ABSENT_AFTER_EVENT,22,"Change in indices w.e.f March 27, 2015.",https://www.niftyindices.com/Press_Release/ind_prs20022015.pdf,532
2,2015-09-28,SITICABLE,ENTRY,ENTRY_SYMBOL_ABSENT_AFTER_EVENT,54,"Change in Indices and Methodology w.e.f September 28, 2015",https://www.niftyindices.com/Press_Release/ind_prs12082015.pdf,532
3,2016-04-01,ADANITRANS,ENTRY,ENTRY_SYMBOL_ABSENT_AFTER_EVENT,149,"Change in Indices w.e.f. April 01, 2016.",https://www.niftyindices.com/Press_Release/ind_prs22022016_2.pdf,528
4,2016-04-01,AEGISCHEM,ENTRY,ENTRY_SYMBOL_ABSENT_AFTER_EVENT,151,"Change in Indices w.e.f. April 01, 2016.",https://www.niftyindices.com/Press_Release/ind_prs22022016_2.pdf,528
5,2016-04-01,HITACHIHOM,ENTRY,ENTRY_SYMBOL_ABSENT_AFTER_EVENT,171,"Change in Indices w.e.f. April 01, 2016.",https://www.niftyindices.com/Press_Release/ind_prs22022016_2.pdf,528
6,2016-04-01,STYABS,ENTRY,ENTRY_SYMBOL_ABSENT_AFTER_EVENT,208,"Change in Indices w.e.f. April 01, 2016.",https://www.niftyindices.com/Press_Release/ind_prs22022016_2.pdf,528
7,2016-04-01,TATAMTRDVR,ENTRY,ENTRY_SYMBOL_ABSENT_AFTER_EVENT,211,"Change in Indices w.e.f. April 01, 2016.",https://www.niftyindices.com/Press_Release/ind_prs22022016_2.pdf,528
8,2016-09-30,CROMPGREAV,ENTRY,ENTRY_SYMBOL_ABSENT_AFTER_EVENT,258,"Change in Indices w.e.f. September 30, 2016.",https://www.niftyindices.com/Press_Release/ind_prs12082016.pdf,526
9,2016-09-30,IDFCBANK,ENTRY,ENTRY_SYMBOL_ABSENT_AFTER_EVENT,261,"Change in Indices w.e.f. September 30, 2016.",https://www.niftyindices.com/Press_Release/ind_prs12082016.pdf,526



SELECTED HISTORICAL EVENT DATES


,Effective_Date,Entry_Count,Exit_Count,Count_After,Count_Before,Mismatch_Count,Mismatch_Symbols,State_Replay_Consistent,State_Checks_Pass
0,2015-03-27,15,15,532,534,2,"FCEL, MCDOWELLN",False,False
1,2015-09-28,17,16,532,532,1,SITICABLE,False,False
2,2016-04-01,76,75,528,532,5,"ADANITRANS, AEGISCHEM, HITACHIHOM, STYABS, TAT...",False,False
3,2017-03-16,3,3,526,526,0,,True,True
4,2018-03-13,1,1,521,521,0,,True,True
5,2019-12-27,9,9,515,515,0,,True,True
6,2020-03-19,1,1,515,515,0,,True,True
7,2020-06-26,25,25,513,515,2,"CENTURYTEX, IIFLWAM",False,False
8,2024-03-28,33,33,502,504,2,"HBLPOWER, JKBANK",False,False
9,2024-09-30,27,27,502,502,2,"GETD, IDEA",False,False



NEW REPORTS SAVED
Mismatch report: /content/nifty500_pit_data/membership_backward_walk_state_mismatches_v2.csv
Daily summary:   /content/nifty500_pit_data/membership_backward_walk_date_summary_v2.csv

Input event ledger and membership intervals were not modified.


In [73]:
# ================================================================
# V11 MISMATCH ROOT-CAUSE AUDIT
#
# 1. Trace each mismatch through its full event history.
# 2. Check for later exits and entries.
# 3. Find all intervals for the same normalized symbol.
# 4. Show all source-release groups for the two 2015 dates.
#
# READ-ONLY INPUTS; creates new diagnostic CSVs only.
# ================================================================

from pathlib import Path
import re
import numpy as np
import pandas as pd
from IPython.display import display

BASE_DIR = Path("/content/nifty500_pit_data")

EVENT_FILE = BASE_DIR / "nifty500_official_event_candidates_v11.csv"
MISMATCH_FILE = BASE_DIR / "membership_backward_walk_state_mismatches_v2.csv"
INTERVAL_FILE = BASE_DIR / "pit_intervals_official_2016-04-01_v2_staged.csv"

SUMMARY_OUT = BASE_DIR / "v11_mismatch_root_cause_summary_v1.csv"
TIMELINE_OUT = BASE_DIR / "v11_mismatch_symbol_event_timeline_v1.csv"
INTERVAL_OUT = BASE_DIR / "v11_mismatch_symbol_intervals_v1.csv"
RELEASE_OUT = BASE_DIR / "v11_2015_release_date_composition_v1.csv"

for path in [EVENT_FILE, MISMATCH_FILE, INTERVAL_FILE]:
    if not path.exists():
        raise FileNotFoundError(f"Required input not found: {path}")

# ------------------------------------------------
# 1. LOAD AND NORMALIZE
# ------------------------------------------------

events = pd.read_csv(EVENT_FILE, dtype=str, keep_default_na=False)
mismatches = pd.read_csv(MISMATCH_FILE, dtype=str, keep_default_na=False)
intervals = pd.read_csv(INTERVAL_FILE, dtype=str, keep_default_na=False)

symbol_col = (
    "Canonical_Symbol" if "Canonical_Symbol" in events.columns
    else "Symbol" if "Symbol" in events.columns
    else None
)

if symbol_col is None:
    raise ValueError("Ledger has no Canonical_Symbol or Symbol column.")

def norm_symbol(value):
    # Used for matching only; original symbols remain in the outputs.
    return re.sub(r"[^A-Z0-9]", "", str(value).upper().strip())

events = events.copy()
events["_Ledger_Row"] = np.arange(len(events), dtype=int)
events["_Event_Date"] = pd.to_datetime(
    events["Effective_Date"], errors="coerce"
).dt.normalize()
events["_Symbol_Raw"] = events[symbol_col].astype(str).str.strip()
events["_Symbol_Key"] = events["_Symbol_Raw"].map(norm_symbol)
events["_Role"] = events["Role"].astype(str).str.upper().str.strip()

mismatches = mismatches.copy()
mismatches["_Ledger_Row"] = pd.to_numeric(
    mismatches["Ledger_Row"], errors="coerce"
).astype("Int64")
mismatches["_Mismatch_Date"] = pd.to_datetime(
    mismatches["Effective_Date"], errors="coerce"
).dt.normalize()
mismatches["_Mismatch_Symbol_Key"] = mismatches["Symbol"].map(norm_symbol)

intervals = intervals.copy()
intervals["_Symbol_Raw"] = intervals["symbol"].astype(str).str.strip()
intervals["_Symbol_Key"] = intervals["_Symbol_Raw"].map(norm_symbol)
intervals["_Start"] = pd.to_datetime(
    intervals["start_date"], errors="coerce"
).dt.normalize()
intervals["_End"] = pd.to_datetime(
    intervals["end_date"].replace("", np.nan), errors="coerce"
).dt.normalize()

# ------------------------------------------------
# 2. CONFIRM THAT MISMATCH ROW NUMBERS STILL MATCH V11
# ------------------------------------------------

ledger_lookup = events[
    [
        "_Ledger_Row", "_Event_Date", "_Role", "_Symbol_Raw",
        "_Symbol_Key", "Release_Title", "Source_URL"
    ]
].rename(columns={
    "_Event_Date": "Ledger_Event_Date",
    "_Role": "Ledger_Role",
    "_Symbol_Raw": "Ledger_Symbol",
    "_Symbol_Key": "Ledger_Symbol_Key",
    "Release_Title": "Ledger_Release_Title",
    "Source_URL": "Ledger_Source_URL",
})

joined = mismatches.merge(
    ledger_lookup,
    on="_Ledger_Row",
    how="left",
    validate="many_to_one",
    indicator=True,
)

unmatched = joined["_merge"].ne("both")

print("=" * 90)
print("V11 ROOT-CAUSE AUDIT")
print("=" * 90)
print(f"Ledger events: {len(events):,}")
print(f"Mismatch records: {len(mismatches):,}")
print(f"Mismatch records joined by v11 row number: {(~unmatched).sum():,}")
print(f"Rows without a matching v11 event: {unmatched.sum():,}")

if unmatched.any():
    display(
        joined.loc[
            unmatched,
            ["Ledger_Row", "Effective_Date", "Symbol", "Issue"]
        ]
    )
    raise ValueError(
        "Mismatch row numbers do not align with v11. "
        "Stop and check the report's source ledger."
    )

# ------------------------------------------------
# 3. BUILD A SYMBOL-BY-SYMBOL EVENT AND INTERVAL SUMMARY
# ------------------------------------------------

summary_rows = []
timeline_parts = []
interval_parts = []

# Process each mismatched event independently so the report retains
# its original mismatch date and issue.
for _, mismatch in joined.iterrows():
    sym_key = mismatch["Ledger_Symbol_Key"]
    event_date = mismatch["Ledger_Event_Date"]

    if pd.isna(event_date) or not sym_key:
        continue

    sym_events = events.loc[
        events["_Symbol_Key"].eq(sym_key)
    ].copy().sort_values("_Event_Date")

    sym_intervals = intervals.loc[
        intervals["_Symbol_Key"].eq(sym_key)
    ].copy().sort_values(["_Start", "_End"], na_position="last")

    later_events = sym_events.loc[
        sym_events["_Event_Date"] > event_date
    ]

    later_exits = later_events.loc[
        later_events["_Role"].eq("EXIT")
    ]

    later_entries = later_events.loc[
        later_events["_Role"].eq("ENTRY")
    ]

    # Summarize intervals without confusing separate ISIN rows.
    interval_descriptions = []
    for _, iv in sym_intervals.iterrows():
        start = (
            iv["_Start"].strftime("%Y-%m-%d")
            if pd.notna(iv["_Start"]) else "INVALID/BLANK"
        )
        end = (
            iv["_End"].strftime("%Y-%m-%d")
            if pd.notna(iv["_End"]) else "OPEN"
        )
        isin = str(iv.get("isin", "")).strip()
        raw_sym = str(iv.get("symbol", "")).strip()
        start_conf = str(iv.get("start_confidence", "")).strip()
        end_conf = str(iv.get("end_confidence", "")).strip()

        interval_descriptions.append(
            f"{raw_sym} | ISIN={isin} | {start} to {end} "
            f"| start_conf={start_conf} | end_conf={end_conf}"
        )

    later_exit_description = "; ".join(
        f"{r['_Event_Date'].strftime('%Y-%m-%d')} "
        f"{r['_Role']} ({r['Source_URL']})"
        for _, r in later_exits.iterrows()
    )

    later_entry_description = "; ".join(
        f"{r['_Event_Date'].strftime('%Y-%m-%d')} "
        f"{r['_Role']} ({r['Source_URL']})"
        for _, r in later_entries.iterrows()
    )

    summary_rows.append({
        "Mismatch_Date": event_date.strftime("%Y-%m-%d"),
        "Mismatch_Symbol": mismatch["Symbol"],
        "Mismatch_Role": mismatch["Role"],
        "Mismatch_Issue": mismatch["Issue"],
        "V11_Ledger_Row": int(mismatch["_Ledger_Row"]),
        "Ledger_Symbol": mismatch["Ledger_Symbol"],
        "Ledger_Release_Title": mismatch["Ledger_Release_Title"],
        "Ledger_Source_URL": mismatch["Ledger_Source_URL"],
        "Total_Events_For_Normalized_Symbol": len(sym_events),
        "Later_Event_Count": len(later_events),
        "Later_Exit_Count": len(later_exits),
        "Later_Exit_Dates": ", ".join(
            later_exits["_Event_Date"].dt.strftime("%Y-%m-%d").tolist()
        ),
        "Later_Exit_Details": later_exit_description,
        "Later_Entry_Count": len(later_entries),
        "Later_Entry_Dates": ", ".join(
            later_entries["_Event_Date"].dt.strftime("%Y-%m-%d").tolist()
        ),
        "Later_Entry_Details": later_entry_description,
        "Interval_Row_Count": len(sym_intervals),
        "Interval_ISINs": ", ".join(
            sym_intervals.get(
                "isin", pd.Series(dtype=str)
            ).astype(str).tolist()
        ),
        "Interval_History": " || ".join(interval_descriptions),
    })

    # Keep the full event history, including prior events.
    for _, ev in sym_events.iterrows():
        if ev["_Event_Date"] < event_date:
            relation = "BEFORE_MISMATCH"
        elif ev["_Event_Date"] == event_date:
            relation = "ON_MISMATCH_DATE"
        else:
            relation = "AFTER_MISMATCH"

        timeline_parts.append({
            "Mismatch_Date": event_date.strftime("%Y-%m-%d"),
            "Mismatch_Symbol": mismatch["Symbol"],
            "Mismatch_Issue": mismatch["Issue"],
            "V11_Ledger_Row": int(mismatch["_Ledger_Row"]),
            "Event_Date": (
                ev["_Event_Date"].strftime("%Y-%m-%d")
                if pd.notna(ev["_Event_Date"]) else ""
            ),
            "Relation_To_Mismatch": relation,
            "Event_Role": ev["_Role"],
            "Ledger_Symbol": ev["_Symbol_Raw"],
            "Release_Title": ev.get("Release_Title", ""),
            "Source_URL": ev.get("Source_URL", ""),
        })

    # Keep all intervals, even when several ISINs share a symbol.
    if len(sym_intervals):
        for _, iv in sym_intervals.iterrows():
            interval_parts.append({
                "Mismatch_Date": event_date.strftime("%Y-%m-%d"),
                "Mismatch_Symbol": mismatch["Symbol"],
                "Mismatch_Issue": mismatch["Issue"],
                "V11_Ledger_Row": int(mismatch["_Ledger_Row"]),
                "Interval_Symbol": iv.get("symbol", ""),
                "ISIN": iv.get("isin", ""),
                "Company_Name": iv.get("company_name", ""),
                "Start_Date": iv["_Start"].strftime("%Y-%m-%d")
                    if pd.notna(iv["_Start"]) else "",
                "Start_Confidence": iv.get("start_confidence", ""),
                "End_Date": iv["_End"].strftime("%Y-%m-%d")
                    if pd.notna(iv["_End"]) else "",
                "End_Confidence": iv.get("end_confidence", ""),
                "Mismatch_Date_Inside_Interval": bool(
                    pd.notna(iv["_Start"])
                    and iv["_Start"] <= event_date
                    and (
                        pd.isna(iv["_End"])
                        or iv["_End"] >= event_date
                    )
                ),
            })

summary_df = pd.DataFrame(summary_rows)
timeline_df = pd.DataFrame(timeline_parts)
interval_df = pd.DataFrame(interval_parts)

# ------------------------------------------------
# 4. AUDIT ALL SOURCES CONTRIBUTING EVENTS ON 2015 DATES
# ------------------------------------------------

target_dates = {"2015-03-27", "2015-09-28"}

date_events = events.loc[
    events["_Event_Date"].dt.strftime("%Y-%m-%d").isin(target_dates)
].copy()

release_group_rows = []

for (date, url, title, role), group in date_events.groupby(
    [
        date_events["_Event_Date"].dt.strftime("%Y-%m-%d"),
        "Source_URL",
        "Release_Title",
        "_Role",
    ],
    dropna=False,
):
    release_group_rows.append({
        "Effective_Date": date,
        "Release_Title": title,
        "Role": role,
        "Unique_Symbol_Count": group["_Symbol_Key"].nunique(),
        "Symbols": ", ".join(
            sorted(group["_Symbol_Raw"].dropna().unique().tolist())
        ),
        "Source_URL": url,
    })

release_df = pd.DataFrame(release_group_rows).sort_values(
    ["Effective_Date", "Release_Title", "Role"]
)

# ------------------------------------------------
# 5. WRITE VERSIONED DIAGNOSTICS
# ------------------------------------------------

summary_df.to_csv(SUMMARY_OUT, index=False)
timeline_df.to_csv(TIMELINE_OUT, index=False)
interval_df.to_csv(INTERVAL_OUT, index=False)
release_df.to_csv(RELEASE_OUT, index=False)

# ------------------------------------------------
# 6. SHOW THE MOST USEFUL RESULTS
# ------------------------------------------------

print("\n" + "=" * 90)
print("SECTION A — MISMATCH SUMMARY WITH LATER TRANSITIONS")
print("=" * 90)

show_cols = [
    "Mismatch_Date",
    "Mismatch_Symbol",
    "Mismatch_Role",
    "Mismatch_Issue",
    "Later_Event_Count",
    "Later_Exit_Count",
    "Later_Exit_Dates",
    "Interval_Row_Count",
    "Interval_ISINs",
    "Interval_History",
]

with pd.option_context(
    "display.max_rows", 100,
    "display.max_columns", 30,
    "display.max_colwidth", 220,
    "display.width", 280,
):
    display(summary_df[show_cols])

print("\n" + "=" * 90)
print("SECTION B — OFFICIAL RELEASE COMPOSITION FOR 2015-03-27 AND 2015-09-28")
print("=" * 90)

with pd.option_context(
    "display.max_rows", 100,
    "display.max_colwidth", 200,
    "display.width", 260,
):
    display(release_df)

print("\n" + "=" * 90)
print("SECTION C — COMPLETE EVENT HISTORY FOR MISMATCHED SYMBOLS")
print("=" * 90)

with pd.option_context(
    "display.max_rows", 250,
    "display.max_columns", 20,
    "display.max_colwidth", 180,
    "display.width", 260,
):
    display(
        timeline_df.sort_values(
            ["Mismatch_Date", "Mismatch_Symbol", "Event_Date"]
        ).reset_index(drop=True)
    )

print("\n" + "=" * 90)
print("OUTPUT FILES")
print("=" * 90)
print(f"Summary:              {SUMMARY_OUT}")
print(f"Full event timelines: {TIMELINE_OUT}")
print(f"Matching intervals:   {INTERVAL_OUT}")
print(f"2015 release groups:  {RELEASE_OUT}")
print("\nNo source ledger or interval file was modified.")


V11 ROOT-CAUSE AUDIT
Ledger events: 1,428
Mismatch records: 37
Mismatch records joined by v11 row number: 37
Rows without a matching v11 event: 0

SECTION A — MISMATCH SUMMARY WITH LATER TRANSITIONS


,Mismatch_Date,Mismatch_Symbol,Mismatch_Role,Mismatch_Issue,Later_Event_Count,Later_Exit_Count,Later_Exit_Dates,Interval_Row_Count,Interval_ISINs,Interval_History
0,2015-03-27,FCEL,ENTRY,ENTRY_SYMBOL_ABSENT_AFTER_EVENT,0,0,,0,,
1,2015-03-27,MCDOWELLN,ENTRY,ENTRY_SYMBOL_ABSENT_AFTER_EVENT,0,0,,1,INE854D01016,MCDOWELL-N | ISIN=INE854D01016 | 2010-01-02 to 2014-08-20 | start_conf=left_censored | end_conf=press_release_exact
2,2015-09-28,SITICABLE,ENTRY,ENTRY_SYMBOL_ABSENT_AFTER_EVENT,0,0,,0,,
3,2016-04-01,ADANITRANS,ENTRY,ENTRY_SYMBOL_ABSENT_AFTER_EVENT,0,0,,0,,
4,2016-04-01,AEGISCHEM,ENTRY,ENTRY_SYMBOL_ABSENT_AFTER_EVENT,0,0,,0,,
5,2016-04-01,HITACHIHOM,ENTRY,ENTRY_SYMBOL_ABSENT_AFTER_EVENT,0,0,,0,,
6,2016-04-01,STYABS,ENTRY,ENTRY_SYMBOL_ABSENT_AFTER_EVENT,0,0,,0,,
7,2016-04-01,TATAMTRDVR,ENTRY,ENTRY_SYMBOL_ABSENT_AFTER_EVENT,0,0,,1,IN9155A01020,TATAMTRDVR | ISIN=IN9155A01020 | 2016-12-28 to 2024-10-11 | start_conf=interpolated_long_gap | end_conf=interpolated_long_gap
8,2016-09-30,CROMPGREAV,ENTRY,ENTRY_SYMBOL_ABSENT_AFTER_EVENT,0,0,,0,,
9,2016-09-30,IDFCBANK,ENTRY,ENTRY_SYMBOL_ABSENT_AFTER_EVENT,0,0,,0,,



SECTION B — OFFICIAL RELEASE COMPOSITION FOR 2015-03-27 AND 2015-09-28


,Effective_Date,Release_Title,Role,Unique_Symbol_Count,Symbols,Source_URL
0,2015-03-27,"Change in Indices w.e.f March 24, 2015 and March 27, 2015",ENTRY,3,"MAYURUNIQ, MUNJALSHOW, SUVEN",https://www.niftyindices.com/Press_Release/ind_prs18032015_2.pdf
1,2015-03-27,"Change in Indices w.e.f March 24, 2015 and March 27, 2015",EXIT,3,"CMC, MASTEK, SHASUNPHAR",https://www.niftyindices.com/Press_Release/ind_prs18032015_2.pdf
2,2015-03-27,"Change in indices w.e.f March 27, 2015.",ENTRY,12,"FCEL, GULFOILLUB, IFBIND, JKCEMENT, KNRCON, KPRMILL, LGBBROSLTD, MARKSANS, MCDOWELLN, NDTV, PENIND, VINATIORGA",https://www.niftyindices.com/Press_Release/ind_prs20022015.pdf
3,2015-03-27,"Change in indices w.e.f March 27, 2015.",EXIT,12,"DCW, EDUCOMP, ELDERPHARM, HUBTOWN, ILFSENGG, IMFA, LOVABLE, PFOCUS, RAJTV, REIAGROLTD, TI, VENKEYS",https://www.niftyindices.com/Press_Release/ind_prs20022015.pdf
4,2015-09-28,"Change in Indices and Methodology w.e.f September 28, 2015",ENTRY,14,"ADANIPOWER, BALKRISIND, CAMLINFINE, FLFL, ILFSENGG, INOXWIND, KRBL, METALFORGE, ORIENTCEM, RICOAUTO, SITICABLE, SRIPIPES, SYMPHONY, VRLLOG",https://www.niftyindices.com/Press_Release/ind_prs12082015.pdf
5,2015-09-28,"Change in Indices and Methodology w.e.f September 28, 2015",EXIT,14,"ANSALAPI, BGRENERGY, GAMMONIND, GUJNRECOKE, HINDOILEXP, HOTELEELA, IFBIND, KESORAMIND, MANGCHEFER, MONNETISPA, ORCHIDCHEM, RAMCOIND, RASOYPR, RTNPOWER",https://www.niftyindices.com/Press_Release/ind_prs12082015.pdf
6,2015-09-28,"Change in Indices w.e.f September 28, 2015",ENTRY,3,"ADLABS, EVEREADY, HAVELLS",https://www.niftyindices.com/Press_Release/ind_prs24082015.pdf
7,2015-09-28,"Change in Indices w.e.f September 28, 2015",EXIT,2,"CROMPGREAV, MAX",https://www.niftyindices.com/Press_Release/ind_prs24082015.pdf



SECTION C — COMPLETE EVENT HISTORY FOR MISMATCHED SYMBOLS


,Mismatch_Date,Mismatch_Symbol,Mismatch_Issue,V11_Ledger_Row,Event_Date,Relation_To_Mismatch,Event_Role,Ledger_Symbol,Release_Title,Source_URL
0,2015-03-27,FCEL,ENTRY_SYMBOL_ABSENT_AFTER_EVENT,12,2015-03-27,ON_MISMATCH_DATE,ENTRY,FCEL,"Change in indices w.e.f March 27, 2015.",https://www.niftyindices.com/Press_Release/ind_prs20022015.pdf
1,2015-03-27,MCDOWELLN,ENTRY_SYMBOL_ABSENT_AFTER_EVENT,22,2015-03-27,ON_MISMATCH_DATE,ENTRY,MCDOWELLN,"Change in indices w.e.f March 27, 2015.",https://www.niftyindices.com/Press_Release/ind_prs20022015.pdf
2,2015-09-28,SITICABLE,ENTRY_SYMBOL_ABSENT_AFTER_EVENT,54,2015-09-28,ON_MISMATCH_DATE,ENTRY,SITICABLE,"Change in Indices and Methodology w.e.f September 28, 2015",https://www.niftyindices.com/Press_Release/ind_prs12082015.pdf
3,2016-04-01,ADANITRANS,ENTRY_SYMBOL_ABSENT_AFTER_EVENT,149,2016-04-01,ON_MISMATCH_DATE,ENTRY,ADANITRANS,"Change in Indices w.e.f. April 01, 2016.",https://www.niftyindices.com/Press_Release/ind_prs22022016_2.pdf
4,2016-04-01,AEGISCHEM,ENTRY_SYMBOL_ABSENT_AFTER_EVENT,151,2016-04-01,ON_MISMATCH_DATE,ENTRY,AEGISCHEM,"Change in Indices w.e.f. April 01, 2016.",https://www.niftyindices.com/Press_Release/ind_prs22022016_2.pdf
5,2016-04-01,HITACHIHOM,ENTRY_SYMBOL_ABSENT_AFTER_EVENT,171,2016-04-01,ON_MISMATCH_DATE,ENTRY,HITACHIHOM,"Change in Indices w.e.f. April 01, 2016.",https://www.niftyindices.com/Press_Release/ind_prs22022016_2.pdf
6,2016-04-01,STYABS,ENTRY_SYMBOL_ABSENT_AFTER_EVENT,208,2016-04-01,ON_MISMATCH_DATE,ENTRY,STYABS,"Change in Indices w.e.f. April 01, 2016.",https://www.niftyindices.com/Press_Release/ind_prs22022016_2.pdf
7,2016-04-01,TATAMTRDVR,ENTRY_SYMBOL_ABSENT_AFTER_EVENT,211,2016-04-01,ON_MISMATCH_DATE,ENTRY,TATAMTRDVR,"Change in Indices w.e.f. April 01, 2016.",https://www.niftyindices.com/Press_Release/ind_prs22022016_2.pdf
8,2016-09-30,CROMPGREAV,ENTRY_SYMBOL_ABSENT_AFTER_EVENT,258,2015-09-28,BEFORE_MISMATCH,EXIT,CROMPGREAV,"Change in Indices w.e.f September 28, 2015",https://www.niftyindices.com/Press_Release/ind_prs24082015.pdf
9,2016-09-30,CROMPGREAV,ENTRY_SYMBOL_ABSENT_AFTER_EVENT,258,2016-09-30,ON_MISMATCH_DATE,ENTRY,CROMPGREAV,"Change in Indices w.e.f. September 30, 2016.",https://www.niftyindices.com/Press_Release/ind_prs12082016.pdf



OUTPUT FILES
Summary:              /content/nifty500_pit_data/v11_mismatch_root_cause_summary_v1.csv
Full event timelines: /content/nifty500_pit_data/v11_mismatch_symbol_event_timeline_v1.csv
Matching intervals:   /content/nifty500_pit_data/v11_mismatch_symbol_intervals_v1.csv
2015 release groups:  /content/nifty500_pit_data/v11_2015_release_date_composition_v1.csv

No source ledger or interval file was modified.


In [74]:
# ================================================================
# NIFTY 500 EVENT LEDGER V12
# Remove one confirmed wrong-index-scope event: HAVELLS
#
# V11 remains unchanged.
# ================================================================

from pathlib import Path
import re
import pandas as pd
from IPython.display import display

BASE_DIR = Path("/content/nifty500_pit_data")

INPUT_FILE = BASE_DIR / "nifty500_official_event_candidates_v11.csv"
OUTPUT_FILE = BASE_DIR / "nifty500_official_event_candidates_v12.csv"
CHANGELOG_FILE = BASE_DIR / "nifty500_event_ledger_change_log_v7.csv"
QA_FILE = BASE_DIR / "nifty500_2015_sep28_scope_QA_v2.csv"

if not INPUT_FILE.exists():
    raise FileNotFoundError(f"Missing input: {INPUT_FILE}")

df = pd.read_csv(INPUT_FILE, dtype=str, keep_default_na=False)

symbol_col = (
    "Canonical_Symbol" if "Canonical_Symbol" in df.columns
    else "Symbol" if "Symbol" in df.columns
    else None
)

if symbol_col is None:
    raise ValueError("Could not find the event symbol column.")

def norm_symbol(s):
    return re.sub(r"[^A-Z0-9]", "", str(s).upper().strip())

date_key = pd.to_datetime(
    df["Effective_Date"], errors="coerce"
).dt.strftime("%Y-%m-%d")

role_key = df["Role"].str.upper().str.strip()
symbol_key = df[symbol_col].map(norm_symbol)
url_key = df["Source_URL"].str.lower()

# The specific HAVELLS event attributed to the 24 Aug 2015 release.
mask = (
    url_key.str.contains("ind_prs24082015.pdf", regex=False, na=False)
    & date_key.eq("2015-09-28")
    & role_key.eq("ENTRY")
    & symbol_key.eq("HAVELLS")
)

matches = df.loc[mask].copy()

print("PRE-FLIGHT")
print(f"V11 rows: {len(df):,}")
print(f"Exact HAVELLS candidate rows: {len(matches)}")

if len(matches) != 1:
    display(matches)
    raise ValueError(
        "Expected exactly one HAVELLS entry from this release. "
        "No output files have been written."
    )

removed_row = matches.iloc[0].to_dict()
removed_row["Change_Action"] = "REMOVE"
removed_row["Correction_Reason"] = (
    "Wrong index scope: HAVELLS is listed in the LIX15 Midcap "
    "section of the 2015-08-24 release, not the CNX 500 section."
)
removed_row["Correction_Evidence"] = (
    "Official PDF ind_prs24082015.pdf, page 2: "
    "CNX 500 entries are ADLABS and EVEREADY; "
    "HAVELLS appears under LIX15 Midcap."
)

# Create a separate ledger; do not alter v11.
v12 = df.loc[~mask].copy().reset_index(drop=True)

if len(v12) != len(df) - 1:
    raise AssertionError("Unexpected v12 row count.")

# ------------------------------------------------
# QA 1: expected source-specific CNX 500 changes
# ------------------------------------------------

checks = [
    {
        "Source_PDF": "ind_prs12082015.pdf",
        "Role": "ENTRY",
        "Expected_Count": 14,
        "Expected_Symbols": {
            "ADANIPOWER", "BALKRISIND", "CAMLINFINE", "FLFL",
            "ILFSENGG", "INOXWIND", "KRBL", "METALFORGE",
            "ORIENTCEM", "RICOAUTO", "SITICABLE", "SRIPIPES",
            "SYMPHONY", "VRLLOG",
        },
    },
    {
        "Source_PDF": "ind_prs12082015.pdf",
        "Role": "EXIT",
        "Expected_Count": 14,
        "Expected_Symbols": {
            "ANSALAPI", "BGRENERGY", "GAMMONIND", "GUJNRECOKE",
            "HINDOILEXP", "HOTELEELA", "IFBIND", "KESORAMIND",
            "MANGCHEFER", "MONNETISPA", "ORCHIDCHEM", "RAMCOIND",
            "RASOYPR", "RTNPOWER",
        },
    },
    {
        "Source_PDF": "ind_prs24082015.pdf",
        "Role": "ENTRY",
        "Expected_Count": 2,
        "Expected_Symbols": {"ADLABS", "EVEREADY"},
    },
    {
        "Source_PDF": "ind_prs24082015.pdf",
        "Role": "EXIT",
        "Expected_Count": 2,
        "Expected_Symbols": {"CROMPGREAV", "MAX"},
    },
]

v12_date = pd.to_datetime(
    v12["Effective_Date"], errors="coerce"
).dt.strftime("%Y-%m-%d")

v12_role = v12["Role"].str.upper().str.strip()
v12_symbol = v12[symbol_col].map(norm_symbol)
v12_url = v12["Source_URL"].str.lower()

qa_rows = []

for check in checks:
    source = check["Source_PDF"]
    role = check["Role"]

    scope = (
        v12_url.str.contains(source, regex=False, na=False)
        & v12_date.eq("2015-09-28")
        & v12_role.eq(role)
    )

    actual_symbols = set(v12_symbol.loc[scope])
    expected_symbols = check["Expected_Symbols"]

    qa_rows.append({
        "Source_PDF": source,
        "Effective_Date": "2015-09-28",
        "Role": role,
        "Expected_Count": check["Expected_Count"],
        "Actual_Count": len(actual_symbols),
        "Missing_Symbols": ", ".join(
            sorted(expected_symbols - actual_symbols)
        ),
        "Unexpected_Symbols": ", ".join(
            sorted(actual_symbols - expected_symbols)
        ),
        "Pass": (
            actual_symbols == expected_symbols
            and len(actual_symbols) == check["Expected_Count"]
        ),
    })

qa = pd.DataFrame(qa_rows)

# ------------------------------------------------
# QA 2: total combined event counts for 28 Sep 2015
# ------------------------------------------------

date_mask = v12_date.eq("2015-09-28")

combined_counts = (
    pd.DataFrame({
        "Role": v12_role.loc[date_mask],
        "Symbol": v12_symbol.loc[date_mask],
    })
    .groupby("Role")
    .agg(
        Event_Rows=("Symbol", "size"),
        Unique_Symbols=("Symbol", "nunique"),
    )
    .reset_index()
)

print("\nOFFICIAL SOURCE-SPECIFIC QA")
display(qa)

print("\nCOMBINED EVENTS EFFECTIVE 2015-09-28")
display(combined_counts)

if not qa["Pass"].all():
    raise AssertionError(
        "Source-specific QA failed. Review the output before using v12."
    )

expected_combined = {"ENTRY": 16, "EXIT": 16}
actual_combined = dict(
    zip(combined_counts["Role"], combined_counts["Unique_Symbols"])
)

if actual_combined != expected_combined:
    raise AssertionError(
        f"Expected 16 entries and 16 exits; got {actual_combined}"
    )

# ------------------------------------------------
# Save versioned files only after QA passes
# ------------------------------------------------

pd.DataFrame([removed_row]).to_csv(CHANGELOG_FILE, index=False)
v12.to_csv(OUTPUT_FILE, index=False)
qa.to_csv(QA_FILE, index=False)

print("\nV12 CREATED SUCCESSFULLY")
print(f"V11 rows: {len(df):,}")
print(f"Removed rows: 1")
print(f"V12 rows: {len(v12):,}")
print(f"Ledger: {OUTPUT_FILE}")
print(f"Change log: {CHANGELOG_FILE}")
print(f"QA report: {QA_FILE}")
print("\nV11 remains unchanged.")

PRE-FLIGHT
V11 rows: 1,428
Exact HAVELLS candidate rows: 1

OFFICIAL SOURCE-SPECIFIC QA


,Source_PDF,Effective_Date,Role,Expected_Count,Actual_Count,Missing_Symbols,Unexpected_Symbols,Pass
0,ind_prs12082015.pdf,2015-09-28,ENTRY,14,14,,,True
1,ind_prs12082015.pdf,2015-09-28,EXIT,14,14,,,True
2,ind_prs24082015.pdf,2015-09-28,ENTRY,2,2,,,True
3,ind_prs24082015.pdf,2015-09-28,EXIT,2,2,,,True



COMBINED EVENTS EFFECTIVE 2015-09-28


,Role,Event_Rows,Unique_Symbols
0,ENTRY,16,16
1,EXIT,16,16



V12 CREATED SUCCESSFULLY
V11 rows: 1,428
Removed rows: 1
V12 rows: 1,427
Ledger: /content/nifty500_pit_data/nifty500_official_event_candidates_v12.csv
Change log: /content/nifty500_pit_data/nifty500_event_ledger_change_log_v7.csv
QA report: /content/nifty500_pit_data/nifty500_2015_sep28_scope_QA_v2.csv

V11 remains unchanged.


In [75]:
# ================================================================
# NIFTY 500 POINT-IN-TIME MEMBERSHIP RECONSTRUCTION — V12
#
# Anchor roster: 2026-09-30
# Event ledger:  nifty500_official_event_candidates_v12.csv
#
# Creates fresh reports:
#   membership_backward_walk_state_mismatches_v3.csv
#   membership_backward_walk_date_summary_v3.csv
#
# Does not modify v12 or any membership interval source file.
# ================================================================

from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

# ------------------------------------------------
# 1. FILE PATHS
# ------------------------------------------------

BASE_DIR = Path("/content/nifty500_pit_data")

EVENT_FILE = (
    BASE_DIR / "nifty500_official_event_candidates_v12.csv"
)

INTERVAL_FILE = (
    BASE_DIR / "pit_intervals_official_2016-04-01_v2_staged.csv"
)

MISMATCH_OUT = (
    BASE_DIR / "membership_backward_walk_state_mismatches_v3.csv"
)

SUMMARY_OUT = (
    BASE_DIR / "membership_backward_walk_date_summary_v3.csv"
)

ANCHOR_DATE = pd.Timestamp("2026-09-30")

for path in [EVENT_FILE, INTERVAL_FILE]:
    if not path.exists():
        raise FileNotFoundError(f"Required file not found: {path}")

# ------------------------------------------------
# 2. LOAD INPUTS
# ------------------------------------------------

events = pd.read_csv(
    EVENT_FILE, dtype=str, keep_default_na=False
)

intervals = pd.read_csv(
    INTERVAL_FILE, dtype=str, keep_default_na=False
)

symbol_col = (
    "Canonical_Symbol"
    if "Canonical_Symbol" in events.columns
    else "Symbol"
    if "Symbol" in events.columns
    else None
)

if symbol_col is None:
    raise ValueError(
        "The v12 ledger has no Canonical_Symbol or Symbol column."
    )

required_event_columns = {
    "Effective_Date", "Role", "Source_URL"
}
required_interval_columns = {
    "symbol", "start_date", "end_date"
}

if not required_event_columns.issubset(events.columns):
    raise ValueError(
        "Missing event columns: "
        f"{sorted(required_event_columns - set(events.columns))}"
    )

if not required_interval_columns.issubset(intervals.columns):
    raise ValueError(
        "Missing interval columns: "
        f"{sorted(required_interval_columns - set(intervals.columns))}"
    )

if "Release_Title" not in events.columns:
    events["Release_Title"] = ""

# Keep v12 ledger row numbers stable for this reconstruction.
events = events.copy()
events["_Ledger_Row"] = np.arange(len(events), dtype=int)

events["_Effective"] = pd.to_datetime(
    events["Effective_Date"], errors="coerce"
).dt.normalize()

events["_Role"] = (
    events["Role"].astype(str).str.upper().str.strip()
)

events["_Symbol"] = (
    events[symbol_col].astype(str).str.upper().str.strip()
)

# ------------------------------------------------
# 3. INPUT VALIDATION
# ------------------------------------------------

if events["_Effective"].isna().any():
    print("Events with invalid dates:")
    display(
        events.loc[
            events["_Effective"].isna(),
            ["_Ledger_Row", "Effective_Date", "_Role", "_Symbol"]
        ]
    )
    raise ValueError("Invalid event dates found.")

if events["_Symbol"].eq("").any():
    raise ValueError("Blank symbols found in the event ledger.")

invalid_roles = sorted(
    set(events["_Role"]) - {"ENTRY", "EXIT"}
)

if invalid_roles:
    raise ValueError(f"Unexpected event roles: {invalid_roles}")

duplicate_keys = events.duplicated(
    subset=["_Effective", "_Role", "_Symbol"],
    keep=False
)

if duplicate_keys.any():
    print("Duplicate event keys found:")
    display(
        events.loc[
            duplicate_keys,
            [
                "_Ledger_Row", "Effective_Date",
                "_Role", "_Symbol", "Source_URL"
            ]
        ].sort_values(
            ["Effective_Date", "_Role", "_Symbol"]
        )
    )
    raise ValueError(
        "Resolve duplicate event keys before reconstruction."
    )

# ------------------------------------------------
# 4. PREPARE INTERVALS
# ------------------------------------------------

intervals = intervals.copy()

intervals["_Symbol"] = (
    intervals["symbol"].astype(str).str.upper().str.strip()
)

intervals["_Start"] = pd.to_datetime(
    intervals["start_date"], errors="coerce"
).dt.normalize()

end_raw = intervals["end_date"].astype(str).str.strip()

intervals["_End"] = pd.to_datetime(
    end_raw.replace("", np.nan), errors="coerce"
).dt.normalize()

if intervals["_Start"].isna().any():
    print("Intervals with missing or invalid start dates:")
    display(
        intervals.loc[
            intervals["_Start"].isna(),
            ["symbol", "start_date", "end_date"]
        ].head(50)
    )
    raise ValueError("Invalid interval start dates found.")

invalid_end = end_raw.ne("") & intervals["_End"].isna()

if invalid_end.any():
    print("Intervals with invalid nonblank end dates:")
    display(
        intervals.loc[
            invalid_end,
            ["symbol", "start_date", "end_date"]
        ].head(50)
    )
    raise ValueError("Invalid interval end dates found.")

invalid_ranges = (
    intervals["_End"].notna()
    & (intervals["_Start"] > intervals["_End"])
)

if invalid_ranges.any():
    print("Intervals where start_date is after end_date:")
    display(
        intervals.loc[
            invalid_ranges,
            ["symbol", "start_date", "end_date"]
        ].head(50)
    )
    raise ValueError("Invalid interval ranges found.")

# ------------------------------------------------
# 5. CREATE THE SEPTEMBER 30, 2026 ANCHOR ROSTER
# ------------------------------------------------

# Interval boundaries are treated as inclusive.
# A blank end_date means the interval remains open.
active_at_anchor = intervals.loc[
    (intervals["_Start"] <= ANCHOR_DATE)
    & (
        intervals["_End"].isna()
        | (intervals["_End"] >= ANCHOR_DATE)
    )
].copy()

anchor_symbols = set(
    active_at_anchor.loc[
        active_at_anchor["_Symbol"].ne(""),
        "_Symbol"
    ].unique()
)

if len(anchor_symbols) == 0:
    raise ValueError("Anchor roster is empty.")

print("=" * 85)
print("V12 BACKWARD MEMBERSHIP RECONSTRUCTION")
print("=" * 85)
print(f"Anchor date:                         {ANCHOR_DATE.date()}")
print(f"Active interval rows at anchor:      {len(active_at_anchor):,}")
print(f"Unique symbols in anchor roster:     {len(anchor_symbols):,}")
print(f"V12 event rows:                      {len(events):,}")

# ------------------------------------------------
# 6. SELECT EVENTS ON OR BEFORE THE ANCHOR
# ------------------------------------------------

future_events = events.loc[
    events["_Effective"] > ANCHOR_DATE
].copy()

eligible_events = events.loc[
    events["_Effective"] <= ANCHOR_DATE
].copy()

event_dates = sorted(
    eligible_events["_Effective"].dropna().unique(),
    reverse=True
)

print(f"Events after anchor:                 {len(future_events):,}")
print(f"Events on/before anchor:             {len(eligible_events):,}")
print(f"Distinct effective dates:            {len(event_dates):,}")

if len(future_events):
    print("\nEvents dated after the anchor:")
    display(
        future_events[
            [
                "_Ledger_Row", "Effective_Date",
                "_Role", "_Symbol", "Source_URL"
            ]
        ].head(50)
    )

# ------------------------------------------------
# 7. WALK BACKWARD THROUGH EVENT DATES
# ------------------------------------------------

current_symbols = set(anchor_symbols)

mismatch_records = []
daily_records = []

for raw_date in event_dates:

    date = pd.Timestamp(raw_date)

    day_events = eligible_events.loc[
        eligible_events["_Effective"].eq(date)
    ].copy()

    entry_events = day_events.loc[
        day_events["_Role"].eq("ENTRY")
    ]

    exit_events = day_events.loc[
        day_events["_Role"].eq("EXIT")
    ]

    entry_symbols = set(entry_events["_Symbol"].unique())
    exit_symbols = set(exit_events["_Symbol"].unique())

    same_day_conflicts = entry_symbols & exit_symbols

    if same_day_conflicts:
        raise ValueError(
            f"ENTRY/EXIT conflict on {date.date()}: "
            f"{sorted(same_day_conflicts)}"
        )

    # Current state represents membership immediately AFTER this date's
    # events, because we began at the post-change anchor and walk backward.
    state_after = set(current_symbols)
    count_after = len(state_after)

    date_issues = []

    # Every entry should be present in the roster after its effective date.
    for _, row in entry_events.iterrows():
        symbol = row["_Symbol"]

        if symbol not in state_after:
            record = {
                "Effective_Date": date.strftime("%Y-%m-%d"),
                "Symbol": symbol,
                "Role": "ENTRY",
                "Issue": "ENTRY_SYMBOL_ABSENT_AFTER_EVENT",
                "Ledger_Row": int(row["_Ledger_Row"]),
                "Release_Title": row["Release_Title"],
                "Source_URL": row["Source_URL"],
                "Count_After": count_after,
            }
            mismatch_records.append(record)
            date_issues.append(record)

    # Every exit should be absent from the roster after its effective date.
    for _, row in exit_events.iterrows():
        symbol = row["_Symbol"]

        if symbol in state_after:
            record = {
                "Effective_Date": date.strftime("%Y-%m-%d"),
                "Symbol": symbol,
                "Role": "EXIT",
                "Issue": "EXIT_SYMBOL_STILL_PRESENT_AFTER_EVENT",
                "Ledger_Row": int(row["_Ledger_Row"]),
                "Release_Title": row["Release_Title"],
                "Source_URL": row["Source_URL"],
                "Count_After": count_after,
            }
            mismatch_records.append(record)
            date_issues.append(record)

    # Reverse the events:
    # entries are removed, exits are restored.
    state_before = (
        state_after - entry_symbols
    ) | exit_symbols

    # Replay forward, checking that the transformation is reversible.
    replayed_after = (
        state_before - exit_symbols
    ) | entry_symbols

    count_before = len(state_before)

    daily_records.append({
        "Effective_Date": date.strftime("%Y-%m-%d"),
        "Entry_Count": len(entry_symbols),
        "Exit_Count": len(exit_symbols),
        "Count_After": count_after,
        "Count_Before": count_before,
        "Mismatch_Count": len(date_issues),
        "Mismatch_Symbols": ", ".join(
            sorted({r["Symbol"] for r in date_issues})
        ),
        "State_Replay_Consistent": replayed_after == state_after,
        "State_Checks_Pass": len(date_issues) == 0,
    })

    current_symbols = state_before

# ------------------------------------------------
# 8. CREATE REPORT DATAFRAMES
# ------------------------------------------------

mismatch_columns = [
    "Effective_Date",
    "Symbol",
    "Role",
    "Issue",
    "Ledger_Row",
    "Release_Title",
    "Source_URL",
    "Count_After",
]

mismatches = pd.DataFrame(
    mismatch_records,
    columns=mismatch_columns
)

daily_summary = pd.DataFrame(daily_records)

if not mismatches.empty:
    mismatches = mismatches.sort_values(
        ["Effective_Date", "Issue", "Symbol"]
    ).reset_index(drop=True)

if not daily_summary.empty:
    daily_summary = daily_summary.sort_values(
        "Effective_Date"
    ).reset_index(drop=True)

# ------------------------------------------------
# 9. OVERALL RESULTS
# ------------------------------------------------

dates_with_issues = (
    daily_summary["Mismatch_Count"] > 0
)

print("\n" + "=" * 85)
print("V12 OVERALL RESULTS")
print("=" * 85)
print(f"Effective dates processed:           {len(daily_summary):,}")
print(f"Event-state mismatch rows:            {len(mismatches):,}")
print(f"Dates with mismatches:                {int(dates_with_issues.sum()):,}")
print(
    f"Dates with all event checks passing:  "
    f"{int(daily_summary['State_Checks_Pass'].sum()):,}"
)
print(
    f"Minimum reconstructed roster size:   "
    f"{int(daily_summary['Count_Before'].min()):,}"
)
print(
    f"Maximum reconstructed roster size:   "
    f"{int(daily_summary['Count_Before'].max()):,}"
)

# ------------------------------------------------
# 10. DISPLAY DATES WITH MISMATCHES
# ------------------------------------------------

print("\n" + "=" * 85)
print("DATE SUMMARY — DATES WITH MISMATCHES")
print("=" * 85)

with pd.option_context(
    "display.max_rows", 150,
    "display.max_columns", 20,
    "display.max_colwidth", 140,
    "display.width", 220,
):
    display(
        daily_summary.loc[dates_with_issues].reset_index(drop=True)
    )

# ------------------------------------------------
# 11. DISPLAY ALL MISMATCHES
# ------------------------------------------------

print("\n" + "=" * 85)
print("MISMATCH DETAILS")
print("=" * 85)

with pd.option_context(
    "display.max_rows", 100,
    "display.max_columns", 20,
    "display.max_colwidth", 140,
    "display.width", 220,
):
    display(mismatches)

# ------------------------------------------------
# 12. IMPORTANT DATE CHECKS
# ------------------------------------------------

selected_dates = [
    "2015-03-24",
    "2015-03-27",
    "2015-09-28",
    "2016-04-01",
    "2016-09-30",
    "2017-03-16",
    "2017-09-05",
    "2018-03-13",
    "2019-12-27",
    "2020-03-19",
    "2020-06-26",
    "2020-09-25",
    "2024-03-28",
    "2024-09-30",
    "2026-03-30",
    "2026-09-30",
]

print("\n" + "=" * 85)
print("SELECTED HISTORICAL DATES")
print("=" * 85)

display(
    daily_summary.loc[
        daily_summary["Effective_Date"].isin(selected_dates)
    ].reset_index(drop=True)
)

# ------------------------------------------------
# 13. SAVE FRESH V3 REPORTS
# ------------------------------------------------

mismatches.to_csv(MISMATCH_OUT, index=False)
daily_summary.to_csv(SUMMARY_OUT, index=False)

print("\n" + "=" * 85)
print("FRESH V12 REPORTS SAVED")
print("=" * 85)
print(f"Mismatch report: {MISMATCH_OUT}")
print(f"Date summary:    {SUMMARY_OUT}")
print("\nV12 ledger and staged intervals remain unchanged.")
print("These are diagnostic reports, not final membership flags.")

V12 BACKWARD MEMBERSHIP RECONSTRUCTION
Anchor date:                         2026-09-30
Active interval rows at anchor:      501
Unique symbols in anchor roster:     501
V12 event rows:                      1,427
Events after anchor:                 0
Events on/before anchor:             1,427
Distinct effective dates:            76

V12 OVERALL RESULTS
Effective dates processed:           76
Event-state mismatch rows:            37
Dates with mismatches:                23
Dates with all event checks passing:  53
Minimum reconstructed roster size:   501
Maximum reconstructed roster size:   535

DATE SUMMARY — DATES WITH MISMATCHES


,Effective_Date,Entry_Count,Exit_Count,Count_After,Count_Before,Mismatch_Count,Mismatch_Symbols,State_Replay_Consistent,State_Checks_Pass
0,2015-03-27,15,15,533,535,2,"FCEL, MCDOWELLN",False,False
1,2015-09-28,16,16,532,533,1,SITICABLE,False,False
2,2016-04-01,76,75,528,532,5,"ADANITRANS, AEGISCHEM, HITACHIHOM, STYABS, TATAMTRDVR",False,False
3,2016-09-30,25,25,526,528,2,"CROMPGREAV, IDFCBANK",False,False
4,2017-03-31,22,22,523,526,3,"LTI, MINDAIND, STRTECH",False,False
5,2017-09-29,24,24,522,523,1,IBVENTURES,False,False
6,2018-02-05,4,4,521,522,1,TIFIN,False,False
7,2018-04-02,24,24,520,521,1,HEG,False,False
8,2018-09-28,26,26,518,520,2,"PHILIPCARB, RNAM",False,False
9,2019-03-29,25,25,517,518,1,ESSELPACK,False,False



MISMATCH DETAILS


,Effective_Date,Symbol,Role,Issue,Ledger_Row,Release_Title,Source_URL,Count_After
0,2015-03-27,FCEL,ENTRY,ENTRY_SYMBOL_ABSENT_AFTER_EVENT,12,"Change in indices w.e.f March 27, 2015.",https://www.niftyindices.com/Press_Release/ind_prs20022015.pdf,533
1,2015-03-27,MCDOWELLN,ENTRY,ENTRY_SYMBOL_ABSENT_AFTER_EVENT,22,"Change in indices w.e.f March 27, 2015.",https://www.niftyindices.com/Press_Release/ind_prs20022015.pdf,533
2,2015-09-28,SITICABLE,ENTRY,ENTRY_SYMBOL_ABSENT_AFTER_EVENT,54,"Change in Indices and Methodology w.e.f September 28, 2015",https://www.niftyindices.com/Press_Release/ind_prs12082015.pdf,532
3,2016-04-01,ADANITRANS,ENTRY,ENTRY_SYMBOL_ABSENT_AFTER_EVENT,148,"Change in Indices w.e.f. April 01, 2016.",https://www.niftyindices.com/Press_Release/ind_prs22022016_2.pdf,528
4,2016-04-01,AEGISCHEM,ENTRY,ENTRY_SYMBOL_ABSENT_AFTER_EVENT,150,"Change in Indices w.e.f. April 01, 2016.",https://www.niftyindices.com/Press_Release/ind_prs22022016_2.pdf,528
5,2016-04-01,HITACHIHOM,ENTRY,ENTRY_SYMBOL_ABSENT_AFTER_EVENT,170,"Change in Indices w.e.f. April 01, 2016.",https://www.niftyindices.com/Press_Release/ind_prs22022016_2.pdf,528
6,2016-04-01,STYABS,ENTRY,ENTRY_SYMBOL_ABSENT_AFTER_EVENT,207,"Change in Indices w.e.f. April 01, 2016.",https://www.niftyindices.com/Press_Release/ind_prs22022016_2.pdf,528
7,2016-04-01,TATAMTRDVR,ENTRY,ENTRY_SYMBOL_ABSENT_AFTER_EVENT,210,"Change in Indices w.e.f. April 01, 2016.",https://www.niftyindices.com/Press_Release/ind_prs22022016_2.pdf,528
8,2016-09-30,CROMPGREAV,ENTRY,ENTRY_SYMBOL_ABSENT_AFTER_EVENT,257,"Change in Indices w.e.f. September 30, 2016.",https://www.niftyindices.com/Press_Release/ind_prs12082016.pdf,526
9,2016-09-30,IDFCBANK,ENTRY,ENTRY_SYMBOL_ABSENT_AFTER_EVENT,260,"Change in Indices w.e.f. September 30, 2016.",https://www.niftyindices.com/Press_Release/ind_prs12082016.pdf,526



SELECTED HISTORICAL DATES


,Effective_Date,Entry_Count,Exit_Count,Count_After,Count_Before,Mismatch_Count,Mismatch_Symbols,State_Replay_Consistent,State_Checks_Pass
0,2015-03-24,1,1,535,535,0,,True,True
1,2015-03-27,15,15,533,535,2,"FCEL, MCDOWELLN",False,False
2,2015-09-28,16,16,532,533,1,SITICABLE,False,False
3,2016-04-01,76,75,528,532,5,"ADANITRANS, AEGISCHEM, HITACHIHOM, STYABS, TAT...",False,False
4,2016-09-30,25,25,526,528,2,"CROMPGREAV, IDFCBANK",False,False
5,2017-03-16,3,3,526,526,0,,True,True
6,2017-09-05,1,1,523,523,0,,True,True
7,2018-03-13,1,1,521,521,0,,True,True
8,2019-12-27,9,9,515,515,0,,True,True
9,2020-03-19,1,1,515,515,0,,True,True



FRESH V12 REPORTS SAVED
Mismatch report: /content/nifty500_pit_data/membership_backward_walk_state_mismatches_v3.csv
Date summary:    /content/nifty500_pit_data/membership_backward_walk_date_summary_v3.csv

V12 ledger and staged intervals remain unchanged.
These are diagnostic reports, not final membership flags.


In [76]:
# ================================================================
# NIFTY 500 POINT-IN-TIME MEMBERSHIP RECONSTRUCTION — V12
#
# Anchor roster: 2026-09-30
# Event ledger:  nifty500_official_event_candidates_v12.csv
#
# Creates fresh reports:
#   membership_backward_walk_state_mismatches_v3.csv
#   membership_backward_walk_date_summary_v3.csv
#
# Does not modify v12 or any membership interval source file.
# ================================================================

from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

# ------------------------------------------------
# 1. FILE PATHS
# ------------------------------------------------

BASE_DIR = Path("/content/nifty500_pit_data")

EVENT_FILE = (
    BASE_DIR / "nifty500_official_event_candidates_v12.csv"
)

INTERVAL_FILE = (
    BASE_DIR / "pit_intervals_official_2016-04-01_v2_staged.csv"
)

MISMATCH_OUT = (
    BASE_DIR / "membership_backward_walk_state_mismatches_v3.csv"
)

SUMMARY_OUT = (
    BASE_DIR / "membership_backward_walk_date_summary_v3.csv"
)

ANCHOR_DATE = pd.Timestamp("2026-09-30")

for path in [EVENT_FILE, INTERVAL_FILE]:
    if not path.exists():
        raise FileNotFoundError(f"Required file not found: {path}")

# ------------------------------------------------
# 2. LOAD INPUTS
# ------------------------------------------------

events = pd.read_csv(
    EVENT_FILE, dtype=str, keep_default_na=False
)

intervals = pd.read_csv(
    INTERVAL_FILE, dtype=str, keep_default_na=False
)

symbol_col = (
    "Canonical_Symbol"
    if "Canonical_Symbol" in events.columns
    else "Symbol"
    if "Symbol" in events.columns
    else None
)

if symbol_col is None:
    raise ValueError(
        "The v12 ledger has no Canonical_Symbol or Symbol column."
    )

required_event_columns = {
    "Effective_Date", "Role", "Source_URL"
}
required_interval_columns = {
    "symbol", "start_date", "end_date"
}

if not required_event_columns.issubset(events.columns):
    raise ValueError(
        "Missing event columns: "
        f"{sorted(required_event_columns - set(events.columns))}"
    )

if not required_interval_columns.issubset(intervals.columns):
    raise ValueError(
        "Missing interval columns: "
        f"{sorted(required_interval_columns - set(intervals.columns))}"
    )

if "Release_Title" not in events.columns:
    events["Release_Title"] = ""

# Keep v12 ledger row numbers stable for this reconstruction.
events = events.copy()
events["_Ledger_Row"] = np.arange(len(events), dtype=int)

events["_Effective"] = pd.to_datetime(
    events["Effective_Date"], errors="coerce"
).dt.normalize()

events["_Role"] = (
    events["Role"].astype(str).str.upper().str.strip()
)

events["_Symbol"] = (
    events[symbol_col].astype(str).str.upper().str.strip()
)

# ------------------------------------------------
# 3. INPUT VALIDATION
# ------------------------------------------------

if events["_Effective"].isna().any():
    print("Events with invalid dates:")
    display(
        events.loc[
            events["_Effective"].isna(),
            ["_Ledger_Row", "Effective_Date", "_Role", "_Symbol"]
        ]
    )
    raise ValueError("Invalid event dates found.")

if events["_Symbol"].eq("").any():
    raise ValueError("Blank symbols found in the event ledger.")

invalid_roles = sorted(
    set(events["_Role"]) - {"ENTRY", "EXIT"}
)

if invalid_roles:
    raise ValueError(f"Unexpected event roles: {invalid_roles}")

duplicate_keys = events.duplicated(
    subset=["_Effective", "_Role", "_Symbol"],
    keep=False
)

if duplicate_keys.any():
    print("Duplicate event keys found:")
    display(
        events.loc[
            duplicate_keys,
            [
                "_Ledger_Row", "Effective_Date",
                "_Role", "_Symbol", "Source_URL"
            ]
        ].sort_values(
            ["Effective_Date", "_Role", "_Symbol"]
        )
    )
    raise ValueError(
        "Resolve duplicate event keys before reconstruction."
    )

# ------------------------------------------------
# 4. PREPARE INTERVALS
# ------------------------------------------------

intervals = intervals.copy()

intervals["_Symbol"] = (
    intervals["symbol"].astype(str).str.upper().str.strip()
)

intervals["_Start"] = pd.to_datetime(
    intervals["start_date"], errors="coerce"
).dt.normalize()

end_raw = intervals["end_date"].astype(str).str.strip()

intervals["_End"] = pd.to_datetime(
    end_raw.replace("", np.nan), errors="coerce"
).dt.normalize()

if intervals["_Start"].isna().any():
    print("Intervals with missing or invalid start dates:")
    display(
        intervals.loc[
            intervals["_Start"].isna(),
            ["symbol", "start_date", "end_date"]
        ].head(50)
    )
    raise ValueError("Invalid interval start dates found.")

invalid_end = end_raw.ne("") & intervals["_End"].isna()

if invalid_end.any():
    print("Intervals with invalid nonblank end dates:")
    display(
        intervals.loc[
            invalid_end,
            ["symbol", "start_date", "end_date"]
        ].head(50)
    )
    raise ValueError("Invalid interval end dates found.")

invalid_ranges = (
    intervals["_End"].notna()
    & (intervals["_Start"] > intervals["_End"])
)

if invalid_ranges.any():
    print("Intervals where start_date is after end_date:")
    display(
        intervals.loc[
            invalid_ranges,
            ["symbol", "start_date", "end_date"]
        ].head(50)
    )
    raise ValueError("Invalid interval ranges found.")

# ------------------------------------------------
# 5. CREATE THE SEPTEMBER 30, 2026 ANCHOR ROSTER
# ------------------------------------------------

# Interval boundaries are treated as inclusive.
# A blank end_date means the interval remains open.
active_at_anchor = intervals.loc[
    (intervals["_Start"] <= ANCHOR_DATE)
    & (
        intervals["_End"].isna()
        | (intervals["_End"] >= ANCHOR_DATE)
    )
].copy()

anchor_symbols = set(
    active_at_anchor.loc[
        active_at_anchor["_Symbol"].ne(""),
        "_Symbol"
    ].unique()
)

if len(anchor_symbols) == 0:
    raise ValueError("Anchor roster is empty.")

print("=" * 85)
print("V12 BACKWARD MEMBERSHIP RECONSTRUCTION")
print("=" * 85)
print(f"Anchor date:                         {ANCHOR_DATE.date()}")
print(f"Active interval rows at anchor:      {len(active_at_anchor):,}")
print(f"Unique symbols in anchor roster:     {len(anchor_symbols):,}")
print(f"V12 event rows:                      {len(events):,}")

# ------------------------------------------------
# 6. SELECT EVENTS ON OR BEFORE THE ANCHOR
# ------------------------------------------------

future_events = events.loc[
    events["_Effective"] > ANCHOR_DATE
].copy()

eligible_events = events.loc[
    events["_Effective"] <= ANCHOR_DATE
].copy()

event_dates = sorted(
    eligible_events["_Effective"].dropna().unique(),
    reverse=True
)

print(f"Events after anchor:                 {len(future_events):,}")
print(f"Events on/before anchor:             {len(eligible_events):,}")
print(f"Distinct effective dates:            {len(event_dates):,}")

if len(future_events):
    print("\nEvents dated after the anchor:")
    display(
        future_events[
            [
                "_Ledger_Row", "Effective_Date",
                "_Role", "_Symbol", "Source_URL"
            ]
        ].head(50)
    )

# ------------------------------------------------
# 7. WALK BACKWARD THROUGH EVENT DATES
# ------------------------------------------------

current_symbols = set(anchor_symbols)

mismatch_records = []
daily_records = []

for raw_date in event_dates:

    date = pd.Timestamp(raw_date)

    day_events = eligible_events.loc[
        eligible_events["_Effective"].eq(date)
    ].copy()

    entry_events = day_events.loc[
        day_events["_Role"].eq("ENTRY")
    ]

    exit_events = day_events.loc[
        day_events["_Role"].eq("EXIT")
    ]

    entry_symbols = set(entry_events["_Symbol"].unique())
    exit_symbols = set(exit_events["_Symbol"].unique())

    same_day_conflicts = entry_symbols & exit_symbols

    if same_day_conflicts:
        raise ValueError(
            f"ENTRY/EXIT conflict on {date.date()}: "
            f"{sorted(same_day_conflicts)}"
        )

    # Current state represents membership immediately AFTER this date's
    # events, because we began at the post-change anchor and walk backward.
    state_after = set(current_symbols)
    count_after = len(state_after)

    date_issues = []

    # Every entry should be present in the roster after its effective date.
    for _, row in entry_events.iterrows():
        symbol = row["_Symbol"]

        if symbol not in state_after:
            record = {
                "Effective_Date": date.strftime("%Y-%m-%d"),
                "Symbol": symbol,
                "Role": "ENTRY",
                "Issue": "ENTRY_SYMBOL_ABSENT_AFTER_EVENT",
                "Ledger_Row": int(row["_Ledger_Row"]),
                "Release_Title": row["Release_Title"],
                "Source_URL": row["Source_URL"],
                "Count_After": count_after,
            }
            mismatch_records.append(record)
            date_issues.append(record)

    # Every exit should be absent from the roster after its effective date.
    for _, row in exit_events.iterrows():
        symbol = row["_Symbol"]

        if symbol in state_after:
            record = {
                "Effective_Date": date.strftime("%Y-%m-%d"),
                "Symbol": symbol,
                "Role": "EXIT",
                "Issue": "EXIT_SYMBOL_STILL_PRESENT_AFTER_EVENT",
                "Ledger_Row": int(row["_Ledger_Row"]),
                "Release_Title": row["Release_Title"],
                "Source_URL": row["Source_URL"],
                "Count_After": count_after,
            }
            mismatch_records.append(record)
            date_issues.append(record)

    # Reverse the events:
    # entries are removed, exits are restored.
    state_before = (
        state_after - entry_symbols
    ) | exit_symbols

    # Replay forward, checking that the transformation is reversible.
    replayed_after = (
        state_before - exit_symbols
    ) | entry_symbols

    count_before = len(state_before)

    daily_records.append({
        "Effective_Date": date.strftime("%Y-%m-%d"),
        "Entry_Count": len(entry_symbols),
        "Exit_Count": len(exit_symbols),
        "Count_After": count_after,
        "Count_Before": count_before,
        "Mismatch_Count": len(date_issues),
        "Mismatch_Symbols": ", ".join(
            sorted({r["Symbol"] for r in date_issues})
        ),
        "State_Replay_Consistent": replayed_after == state_after,
        "State_Checks_Pass": len(date_issues) == 0,
    })

    current_symbols = state_before

# ------------------------------------------------
# 8. CREATE REPORT DATAFRAMES
# ------------------------------------------------

mismatch_columns = [
    "Effective_Date",
    "Symbol",
    "Role",
    "Issue",
    "Ledger_Row",
    "Release_Title",
    "Source_URL",
    "Count_After",
]

mismatches = pd.DataFrame(
    mismatch_records,
    columns=mismatch_columns
)

daily_summary = pd.DataFrame(daily_records)

if not mismatches.empty:
    mismatches = mismatches.sort_values(
        ["Effective_Date", "Issue", "Symbol"]
    ).reset_index(drop=True)

if not daily_summary.empty:
    daily_summary = daily_summary.sort_values(
        "Effective_Date"
    ).reset_index(drop=True)

# ------------------------------------------------
# 9. OVERALL RESULTS
# ------------------------------------------------

dates_with_issues = (
    daily_summary["Mismatch_Count"] > 0
)

print("\n" + "=" * 85)
print("V12 OVERALL RESULTS")
print("=" * 85)
print(f"Effective dates processed:           {len(daily_summary):,}")
print(f"Event-state mismatch rows:            {len(mismatches):,}")
print(f"Dates with mismatches:                {int(dates_with_issues.sum()):,}")
print(
    f"Dates with all event checks passing:  "
    f"{int(daily_summary['State_Checks_Pass'].sum()):,}"
)
print(
    f"Minimum reconstructed roster size:   "
    f"{int(daily_summary['Count_Before'].min()):,}"
)
print(
    f"Maximum reconstructed roster size:   "
    f"{int(daily_summary['Count_Before'].max()):,}"
)

# ------------------------------------------------
# 10. DISPLAY DATES WITH MISMATCHES
# ------------------------------------------------

print("\n" + "=" * 85)
print("DATE SUMMARY — DATES WITH MISMATCHES")
print("=" * 85)

with pd.option_context(
    "display.max_rows", 150,
    "display.max_columns", 20,
    "display.max_colwidth", 140,
    "display.width", 220,
):
    display(
        daily_summary.loc[dates_with_issues].reset_index(drop=True)
    )

# ------------------------------------------------
# 11. DISPLAY ALL MISMATCHES
# ------------------------------------------------

print("\n" + "=" * 85)
print("MISMATCH DETAILS")
print("=" * 85)

with pd.option_context(
    "display.max_rows", 100,
    "display.max_columns", 20,
    "display.max_colwidth", 140,
    "display.width", 220,
):
    display(mismatches)

# ------------------------------------------------
# 12. IMPORTANT DATE CHECKS
# ------------------------------------------------

selected_dates = [
    "2015-03-24",
    "2015-03-27",
    "2015-09-28",
    "2016-04-01",
    "2016-09-30",
    "2017-03-16",
    "2017-09-05",
    "2018-03-13",
    "2019-12-27",
    "2020-03-19",
    "2020-06-26",
    "2020-09-25",
    "2024-03-28",
    "2024-09-30",
    "2026-03-30",
    "2026-09-30",
]

print("\n" + "=" * 85)
print("SELECTED HISTORICAL DATES")
print("=" * 85)

display(
    daily_summary.loc[
        daily_summary["Effective_Date"].isin(selected_dates)
    ].reset_index(drop=True)
)

# ------------------------------------------------
# 13. SAVE FRESH V3 REPORTS
# ------------------------------------------------

mismatches.to_csv(MISMATCH_OUT, index=False)
daily_summary.to_csv(SUMMARY_OUT, index=False)

print("\n" + "=" * 85)
print("FRESH V12 REPORTS SAVED")
print("=" * 85)
print(f"Mismatch report: {MISMATCH_OUT}")
print(f"Date summary:    {SUMMARY_OUT}")
print("\nV12 ledger and staged intervals remain unchanged.")
print("These are diagnostic reports, not final membership flags.")

V12 BACKWARD MEMBERSHIP RECONSTRUCTION
Anchor date:                         2026-09-30
Active interval rows at anchor:      501
Unique symbols in anchor roster:     501
V12 event rows:                      1,427
Events after anchor:                 0
Events on/before anchor:             1,427
Distinct effective dates:            76

V12 OVERALL RESULTS
Effective dates processed:           76
Event-state mismatch rows:            37
Dates with mismatches:                23
Dates with all event checks passing:  53
Minimum reconstructed roster size:   501
Maximum reconstructed roster size:   535

DATE SUMMARY — DATES WITH MISMATCHES


,Effective_Date,Entry_Count,Exit_Count,Count_After,Count_Before,Mismatch_Count,Mismatch_Symbols,State_Replay_Consistent,State_Checks_Pass
0,2015-03-27,15,15,533,535,2,"FCEL, MCDOWELLN",False,False
1,2015-09-28,16,16,532,533,1,SITICABLE,False,False
2,2016-04-01,76,75,528,532,5,"ADANITRANS, AEGISCHEM, HITACHIHOM, STYABS, TATAMTRDVR",False,False
3,2016-09-30,25,25,526,528,2,"CROMPGREAV, IDFCBANK",False,False
4,2017-03-31,22,22,523,526,3,"LTI, MINDAIND, STRTECH",False,False
5,2017-09-29,24,24,522,523,1,IBVENTURES,False,False
6,2018-02-05,4,4,521,522,1,TIFIN,False,False
7,2018-04-02,24,24,520,521,1,HEG,False,False
8,2018-09-28,26,26,518,520,2,"PHILIPCARB, RNAM",False,False
9,2019-03-29,25,25,517,518,1,ESSELPACK,False,False



MISMATCH DETAILS


,Effective_Date,Symbol,Role,Issue,Ledger_Row,Release_Title,Source_URL,Count_After
0,2015-03-27,FCEL,ENTRY,ENTRY_SYMBOL_ABSENT_AFTER_EVENT,12,"Change in indices w.e.f March 27, 2015.",https://www.niftyindices.com/Press_Release/ind_prs20022015.pdf,533
1,2015-03-27,MCDOWELLN,ENTRY,ENTRY_SYMBOL_ABSENT_AFTER_EVENT,22,"Change in indices w.e.f March 27, 2015.",https://www.niftyindices.com/Press_Release/ind_prs20022015.pdf,533
2,2015-09-28,SITICABLE,ENTRY,ENTRY_SYMBOL_ABSENT_AFTER_EVENT,54,"Change in Indices and Methodology w.e.f September 28, 2015",https://www.niftyindices.com/Press_Release/ind_prs12082015.pdf,532
3,2016-04-01,ADANITRANS,ENTRY,ENTRY_SYMBOL_ABSENT_AFTER_EVENT,148,"Change in Indices w.e.f. April 01, 2016.",https://www.niftyindices.com/Press_Release/ind_prs22022016_2.pdf,528
4,2016-04-01,AEGISCHEM,ENTRY,ENTRY_SYMBOL_ABSENT_AFTER_EVENT,150,"Change in Indices w.e.f. April 01, 2016.",https://www.niftyindices.com/Press_Release/ind_prs22022016_2.pdf,528
5,2016-04-01,HITACHIHOM,ENTRY,ENTRY_SYMBOL_ABSENT_AFTER_EVENT,170,"Change in Indices w.e.f. April 01, 2016.",https://www.niftyindices.com/Press_Release/ind_prs22022016_2.pdf,528
6,2016-04-01,STYABS,ENTRY,ENTRY_SYMBOL_ABSENT_AFTER_EVENT,207,"Change in Indices w.e.f. April 01, 2016.",https://www.niftyindices.com/Press_Release/ind_prs22022016_2.pdf,528
7,2016-04-01,TATAMTRDVR,ENTRY,ENTRY_SYMBOL_ABSENT_AFTER_EVENT,210,"Change in Indices w.e.f. April 01, 2016.",https://www.niftyindices.com/Press_Release/ind_prs22022016_2.pdf,528
8,2016-09-30,CROMPGREAV,ENTRY,ENTRY_SYMBOL_ABSENT_AFTER_EVENT,257,"Change in Indices w.e.f. September 30, 2016.",https://www.niftyindices.com/Press_Release/ind_prs12082016.pdf,526
9,2016-09-30,IDFCBANK,ENTRY,ENTRY_SYMBOL_ABSENT_AFTER_EVENT,260,"Change in Indices w.e.f. September 30, 2016.",https://www.niftyindices.com/Press_Release/ind_prs12082016.pdf,526



SELECTED HISTORICAL DATES


,Effective_Date,Entry_Count,Exit_Count,Count_After,Count_Before,Mismatch_Count,Mismatch_Symbols,State_Replay_Consistent,State_Checks_Pass
0,2015-03-24,1,1,535,535,0,,True,True
1,2015-03-27,15,15,533,535,2,"FCEL, MCDOWELLN",False,False
2,2015-09-28,16,16,532,533,1,SITICABLE,False,False
3,2016-04-01,76,75,528,532,5,"ADANITRANS, AEGISCHEM, HITACHIHOM, STYABS, TAT...",False,False
4,2016-09-30,25,25,526,528,2,"CROMPGREAV, IDFCBANK",False,False
5,2017-03-16,3,3,526,526,0,,True,True
6,2017-09-05,1,1,523,523,0,,True,True
7,2018-03-13,1,1,521,521,0,,True,True
8,2019-12-27,9,9,515,515,0,,True,True
9,2020-03-19,1,1,515,515,0,,True,True



FRESH V12 REPORTS SAVED
Mismatch report: /content/nifty500_pit_data/membership_backward_walk_state_mismatches_v3.csv
Date summary:    /content/nifty500_pit_data/membership_backward_walk_date_summary_v3.csv

V12 ledger and staged intervals remain unchanged.
These are diagnostic reports, not final membership flags.


In [77]:
# ================================================================
# V12 MISSING-LATER-ACTION INVESTIGATION
#
# Search earlier event-candidate versions v3-v11 for actions
# involving currently mismatched symbols that are absent from v12.
#
# READ-ONLY for all existing inputs.
#
# Creates:
#   v12_prior_version_later_action_candidates_v1.csv
#   v12_prior_version_action_summary_v1.csv
# ================================================================

from pathlib import Path
from urllib.parse import urlsplit
import re
import pandas as pd
import numpy as np
from IPython.display import display

BASE_DIR = Path("/content/nifty500_pit_data")

V12_FILE = BASE_DIR / "nifty500_official_event_candidates_v12.csv"
MISMATCH_FILE = BASE_DIR / "membership_backward_walk_state_mismatches_v3.csv"

CANDIDATES_OUT = (
    BASE_DIR / "v12_prior_version_later_action_candidates_v1.csv"
)
SUMMARY_OUT = (
    BASE_DIR / "v12_prior_version_action_summary_v1.csv"
)

for path in [V12_FILE, MISMATCH_FILE]:
    if not path.exists():
        raise FileNotFoundError(f"Required input missing: {path}")


# ------------------------------------------------
# 1. HELPERS
# ------------------------------------------------

def norm_symbol(value):
    return re.sub(r"[^A-Z0-9]", "", str(value).upper().strip())

def source_filename(value):
    """Extract PDF filename from a stored source URL."""
    value = str(value).strip()
    if not value:
        return ""
    try:
        return urlsplit(value).path.rsplit("/", 1)[-1].lower()
    except Exception:
        return value.lower().rsplit("/", 1)[-1]


# ------------------------------------------------
# 2. LOAD V12 AND MISMATCHES
# ------------------------------------------------

v12 = pd.read_csv(V12_FILE, dtype=str, keep_default_na=False)
mismatches = pd.read_csv(
    MISMATCH_FILE, dtype=str, keep_default_na=False
)

symbol_col = (
    "Canonical_Symbol" if "Canonical_Symbol" in v12.columns
    else "Symbol" if "Symbol" in v12.columns
    else None
)

if symbol_col is None:
    raise ValueError("Could not identify the symbol column in v12.")

required_mismatch = {"Effective_Date", "Symbol", "Role", "Issue"}
if not required_mismatch.issubset(mismatches.columns):
    raise ValueError(
        "Mismatch report lacks required columns: "
        f"{sorted(required_mismatch - set(mismatches.columns))}"
    )

v12["_Date"] = pd.to_datetime(
    v12["Effective_Date"], errors="coerce"
).dt.normalize()
v12["_Role"] = v12["Role"].astype(str).str.upper().str.strip()
v12["_Symbol_Key"] = v12[symbol_col].map(norm_symbol)
v12["_Source_File"] = v12["Source_URL"].map(source_filename)

mismatches["_Mismatch_Date"] = pd.to_datetime(
    mismatches["Effective_Date"], errors="coerce"
).dt.normalize()
mismatches["_Symbol_Key"] = mismatches["Symbol"].map(norm_symbol)

if mismatches["_Mismatch_Date"].isna().any():
    raise ValueError("Mismatch report contains invalid dates.")

if mismatches["_Symbol_Key"].eq("").any():
    raise ValueError("Mismatch report contains blank symbols.")


# ------------------------------------------------
# 3. DEFINE V12 EVENT SIGNATURES
# ------------------------------------------------

# A candidate is considered already present when its date, role,
# normalized symbol, and source PDF filename match v12.
v12_signatures = set(
    zip(
        v12["_Date"].dt.strftime("%Y-%m-%d"),
        v12["_Role"],
        v12["_Symbol_Key"],
        v12["_Source_File"],
    )
)

mismatch_items = (
    mismatches[
        ["Effective_Date", "Symbol", "Role", "Issue",
         "_Mismatch_Date", "_Symbol_Key"]
    ]
    .drop_duplicates(
        ["_Mismatch_Date", "_Symbol_Key", "Issue"]
    )
    .copy()
)

mismatch_symbols = set(mismatch_items["_Symbol_Key"])
print("V12 ledger rows:", f"{len(v12):,}")
print("Distinct mismatched symbol/date/issue combinations:",
      f"{len(mismatch_items):,}")
print("Distinct mismatched normalized symbols:",
      f"{len(mismatch_symbols):,}")


# ------------------------------------------------
# 4. DISCOVER EARLIER CANDIDATE LEDGERS
# ------------------------------------------------

version_pattern = re.compile(
    r"^nifty500_official_event_candidates_v(\d+)\.csv$",
    flags=re.IGNORECASE,
)

prior_files = []

for path in BASE_DIR.glob("nifty500_official_event_candidates_v*.csv"):
    match = version_pattern.match(path.name)
    if not match:
        continue

    version = int(match.group(1))

    if 3 <= version < 12:
        prior_files.append((version, path))

prior_files.sort(key=lambda item: item[0])

if not prior_files:
    raise FileNotFoundError(
        "No earlier event-candidate CSVs v3-v11 were found in "
        f"{BASE_DIR}. No conclusions can be drawn from this audit."
    )

print("\nEarlier candidate versions found:")
print(", ".join(f"v{version}" for version, _ in prior_files))


# ------------------------------------------------
# 5. SEARCH EACH PRIOR VERSION FOR LATER ACTIONS
# ------------------------------------------------

candidate_records = []

# Cache the unique mismatch dates for each symbol.
mismatch_lookup = {}

for _, row in mismatch_items.iterrows():
    key = row["_Symbol_Key"]
    mismatch_lookup.setdefault(key, []).append({
        "Mismatch_Date": row["_Mismatch_Date"],
        "Mismatch_Symbol": row["Symbol"],
        "Mismatch_Role": row["Role"],
        "Mismatch_Issue": row["Issue"],
    })

for version, path in prior_files:

    try:
        old = pd.read_csv(path, dtype=str, keep_default_na=False)
    except Exception as exc:
        print(f"WARNING: Could not read {path.name}: {exc}")
        continue

    old_symbol_col = (
        "Canonical_Symbol" if "Canonical_Symbol" in old.columns
        else "Symbol" if "Symbol" in old.columns
        else None
    )

    if old_symbol_col is None:
        print(f"WARNING: Skipping {path.name}: no symbol column.")
        continue

    required_old = {"Effective_Date", "Role", "Source_URL"}
    if not required_old.issubset(old.columns):
        print(f"WARNING: Skipping {path.name}: missing required columns.")
        continue

    if "Release_Title" not in old.columns:
        old["Release_Title"] = ""

    old["_Date"] = pd.to_datetime(
        old["Effective_Date"], errors="coerce"
    ).dt.normalize()

    old["_Role"] = old["Role"].astype(str).str.upper().str.strip()
    old["_Symbol_Raw"] = old[old_symbol_col].astype(str).str.strip()
    old["_Symbol_Key"] = old["_Symbol_Raw"].map(norm_symbol)
    old["_Source_File"] = old["Source_URL"].map(source_filename)

    old = old.loc[
        old["_Symbol_Key"].isin(mismatch_symbols)
        & old["_Role"].isin(["ENTRY", "EXIT"])
        & old["_Date"].notna()
    ].copy()

    for _, candidate in old.iterrows():

        sym_key = candidate["_Symbol_Key"]
        candidate_date = candidate["_Date"]

        # Compare to each mismatch occurrence for this symbol.
        for mismatch_item in mismatch_lookup.get(sym_key, []):

            mismatch_date = mismatch_item["Mismatch_Date"]

            # Only actions later than the event that triggered the mismatch.
            if candidate_date <= mismatch_date:
                continue

            signature = (
                candidate_date.strftime("%Y-%m-%d"),
                candidate["_Role"],
                sym_key,
                candidate["_Source_File"],
            )

            # If it is already present in v12, it is not a missing action.
            if signature in v12_signatures:
                continue

            candidate_records.append({
                "Mismatch_Date": mismatch_date.strftime("%Y-%m-%d"),
                "Mismatch_Symbol": mismatch_item["Mismatch_Symbol"],
                "Mismatch_Issue": mismatch_item["Mismatch_Issue"],
                "Candidate_Date": candidate_date.strftime("%Y-%m-%d"),
                "Candidate_Role": candidate["_Role"],
                "Candidate_Symbol": candidate["_Symbol_Raw"],
                "Candidate_Source_File": candidate["_Source_File"],
                "Candidate_Release_Title": candidate["Release_Title"],
                "Candidate_Source_URL": candidate["Source_URL"],
                "Found_In_Version": f"v{version}",
                "Candidate_File": path.name,
            })


# ------------------------------------------------
# 6. COMBINE REPEATED CANDIDATES ACROSS VERSIONS
# ------------------------------------------------

candidate_df = pd.DataFrame(candidate_records)

candidate_key_cols = [
    "Mismatch_Date",
    "Mismatch_Symbol",
    "Mismatch_Issue",
    "Candidate_Date",
    "Candidate_Role",
    "Candidate_Symbol",
    "Candidate_Source_File",
    "Candidate_Release_Title",
    "Candidate_Source_URL",
]

if not candidate_df.empty:

    candidates = (
        candidate_df.groupby(
            candidate_key_cols,
            dropna=False,
            as_index=False,
        )
        .agg(
            Versions_Found=(
                "Found_In_Version",
                lambda values: ", ".join(
                    sorted(
                        set(values),
                        key=lambda v: int(v.lstrip("v"))
                    )
                ),
            ),
            Version_Count=("Found_In_Version", "nunique"),
        )
        .sort_values(
            ["Mismatch_Date", "Mismatch_Symbol",
             "Candidate_Date", "Candidate_Role"]
        )
        .reset_index(drop=True)
    )

else:
    candidates = pd.DataFrame(columns=[
        *candidate_key_cols, "Versions_Found", "Version_Count"
    ])


# ------------------------------------------------
# 7. BUILD SUMMARY PER MISMATCH
# ------------------------------------------------

summary_rows = []

for _, item in mismatch_items.iterrows():

    mismatch_date = item["_Mismatch_Date"]
    sym_key = item["_Symbol_Key"]

    current_symbol_events = v12.loc[
        v12["_Symbol_Key"].eq(sym_key)
        & v12["_Date"].gt(mismatch_date)
    ].copy()

    current_later_exits = current_symbol_events.loc[
        current_symbol_events["_Role"].eq("EXIT")
    ]

    matching_candidates = candidates.loc[
        candidates["Mismatch_Date"].eq(
            mismatch_date.strftime("%Y-%m-%d")
        )
        & candidates["Mismatch_Symbol"].map(norm_symbol).eq(sym_key)
    ] if not candidates.empty else pd.DataFrame()

    candidate_exits = (
        matching_candidates.loc[
            matching_candidates["Candidate_Role"].eq("EXIT")
        ]
        if not matching_candidates.empty
        else pd.DataFrame()
    )

    summary_rows.append({
        "Mismatch_Date": mismatch_date.strftime("%Y-%m-%d"),
        "Mismatch_Symbol": item["Symbol"],
        "Mismatch_Issue": item["Issue"],
        "V12_Later_Event_Count": len(current_symbol_events),
        "V12_Later_Exit_Count": len(current_later_exits),
        "V12_Later_Exit_Dates": ", ".join(
            current_later_exits["_Date"].dt.strftime("%Y-%m-%d").tolist()
        ),
        "Earlier_Version_Later_Action_Count": len(matching_candidates),
        "Earlier_Version_Later_Exit_Count": len(candidate_exits),
        "Earlier_Version_Exit_Dates": ", ".join(
            candidate_exits["Candidate_Date"].astype(str).tolist()
        ),
    })

summary = pd.DataFrame(summary_rows).sort_values(
    ["Mismatch_Date", "Mismatch_Symbol"]
).reset_index(drop=True)


# ------------------------------------------------
# 8. SAVE NEW INVESTIGATION REPORTS
# ------------------------------------------------

candidates.to_csv(CANDIDATES_OUT, index=False)
summary.to_csv(SUMMARY_OUT, index=False)


# ------------------------------------------------
# 9. DISPLAY RESULTS
# ------------------------------------------------

print("\n" + "=" * 90)
print("SECTION A — MISMATCHES WITH LATER-ACTION SEARCH RESULTS")
print("=" * 90)

with pd.option_context(
    "display.max_rows", 100,
    "display.max_columns", 20,
    "display.max_colwidth", 180,
    "display.width", 260,
):
    display(summary)

print("\n" + "=" * 90)
print("SECTION B — EARLIER-VERSION ACTION CANDIDATES NOT IN V12")
print("=" * 90)

if candidates.empty:
    print(
        "No later actions absent from v12 were found in the earlier "
        "candidate versions. This does NOT prove no official exit exists; "
        "it means this extractor/version history does not contain one."
    )
else:
    with pd.option_context(
        "display.max_rows", 200,
        "display.max_columns", 20,
        "display.max_colwidth", 180,
        "display.width", 260,
    ):
        display(candidates)

print("\n" + "=" * 90)
print("OUTPUT FILES")
print("=" * 90)
print(f"Summary: {SUMMARY_OUT}")
print(f"Candidate events: {CANDIDATES_OUT}")
print("\nNo existing event ledger or interval file was changed.")
print(
    "Do not automatically re-add any candidate. Previously removed "
    "events may have been void, superseded, or assigned to another index."
)

V12 ledger rows: 1,427
Distinct mismatched symbol/date/issue combinations: 37
Distinct mismatched normalized symbols: 37

Earlier candidate versions found:
v3, v4, v5, v6, v7, v8, v9, v10, v11


KeyError: 'Candidate_Date'

In [1]:
# ------------------------------------------------
# 7. BUILD SUMMARY PER MISMATCH — EMPTY-RESULT SAFE
# ------------------------------------------------

summary_rows = []

for _, item in mismatch_items.iterrows():

    mismatch_date = item["_Mismatch_Date"]
    sym_key = item["_Symbol_Key"]
    mismatch_date_str = mismatch_date.strftime("%Y-%m-%d")

    # Find later events for this symbol in v12.
    current_symbol_events = v12.loc[
        v12["_Symbol_Key"].eq(sym_key)
        & v12["_Date"].gt(mismatch_date)
    ].copy()

    current_later_exits = current_symbol_events.loc[
        current_symbol_events["_Role"].eq("EXIT")
    ]

    # Find earlier-version candidates associated with this mismatch.
    if not candidates.empty:
        matching_candidates = candidates.loc[
            candidates["Mismatch_Date"].eq(mismatch_date_str)
            & candidates["Mismatch_Symbol"].map(norm_symbol).eq(sym_key)
        ].copy()
    else:
        matching_candidates = pd.DataFrame(
            columns=[
                "Candidate_Date",
                "Candidate_Role",
                "Candidate_Symbol",
            ]
        )

    # Always preserve the expected columns, even when there are no rows.
    candidate_exits = matching_candidates.loc[
        matching_candidates["Candidate_Role"].eq("EXIT")
    ].copy()

    current_exit_dates = (
        current_later_exits["_Date"]
        .dropna()
        .dt.strftime("%Y-%m-%d")
        .tolist()
    )

    candidate_exit_dates = (
        candidate_exits["Candidate_Date"]
        .dropna()
        .astype(str)
        .tolist()
    )

    summary_rows.append({
        "Mismatch_Date": mismatch_date_str,
        "Mismatch_Symbol": item["Symbol"],
        "Mismatch_Issue": item["Issue"],

        "V12_Later_Event_Count": len(current_symbol_events),
        "V12_Later_Exit_Count": len(current_later_exits),
        "V12_Later_Exit_Dates": ", ".join(current_exit_dates),

        "Earlier_Version_Later_Action_Count": len(matching_candidates),
        "Earlier_Version_Later_Exit_Count": len(candidate_exits),
        "Earlier_Version_Exit_Dates": ", ".join(candidate_exit_dates),
    })

summary = pd.DataFrame(summary_rows)

if not summary.empty:
    summary = summary.sort_values(
        ["Mismatch_Date", "Mismatch_Symbol"]
    ).reset_index(drop=True)

print("Mismatch summary rows:", len(summary))
print("Candidate action rows:", len(candidates))

display(summary)

NameError: name 'mismatch_items' is not defined

In [2]:
# ================================================================
# V12 MISSING-LATER-ACTION INVESTIGATION — COMPLETE FIXED CELL
#
# Inputs:
#   nifty500_official_event_candidates_v12.csv
#   membership_backward_walk_state_mismatches_v3.csv
#   Earlier event-candidate ledgers v3-v11
#
# Outputs:
#   v12_prior_version_later_action_candidates_v1.csv
#   v12_prior_version_action_summary_v1.csv
#
# This is read-only with respect to all existing input files.
# ================================================================

from pathlib import Path
from urllib.parse import urlsplit
import re
import pandas as pd
from IPython.display import display

# ------------------------------------------------
# 1. FILE PATHS
# ------------------------------------------------

BASE_DIR = Path("/content/nifty500_pit_data")

V12_FILE = (
    BASE_DIR / "nifty500_official_event_candidates_v12.csv"
)

MISMATCH_FILE = (
    BASE_DIR / "membership_backward_walk_state_mismatches_v3.csv"
)

CANDIDATES_OUT = (
    BASE_DIR / "v12_prior_version_later_action_candidates_v1.csv"
)

SUMMARY_OUT = (
    BASE_DIR / "v12_prior_version_action_summary_v1.csv"
)

for path in [V12_FILE, MISMATCH_FILE]:
    if not path.exists():
        raise FileNotFoundError(f"Required input file not found: {path}")


# ------------------------------------------------
# 2. HELPER FUNCTIONS
# ------------------------------------------------

def norm_symbol(value):
    """Normalize symbol punctuation and case for matching only."""
    return re.sub(
        r"[^A-Z0-9]",
        "",
        str(value).upper().strip()
    )


def source_filename(value):
    """Extract the PDF filename from a source URL."""
    value = str(value).strip()

    if not value:
        return ""

    try:
        return urlsplit(value).path.rsplit("/", 1)[-1].lower()
    except Exception:
        return value.lower().rsplit("/", 1)[-1]


# ------------------------------------------------
# 3. LOAD V12 AND THE FRESH MISMATCH REPORT
# ------------------------------------------------

v12 = pd.read_csv(
    V12_FILE,
    dtype=str,
    keep_default_na=False
)

mismatches = pd.read_csv(
    MISMATCH_FILE,
    dtype=str,
    keep_default_na=False
)

symbol_col = (
    "Canonical_Symbol"
    if "Canonical_Symbol" in v12.columns
    else "Symbol"
    if "Symbol" in v12.columns
    else None
)

if symbol_col is None:
    raise ValueError(
        "Could not find Canonical_Symbol or Symbol in v12."
    )

required_v12_columns = {
    "Effective_Date", "Role", "Source_URL"
}

if not required_v12_columns.issubset(v12.columns):
    raise ValueError(
        "V12 ledger is missing columns: "
        f"{sorted(required_v12_columns - set(v12.columns))}"
    )

required_mismatch_columns = {
    "Effective_Date", "Symbol", "Role", "Issue"
}

if not required_mismatch_columns.issubset(mismatches.columns):
    raise ValueError(
        "Mismatch report is missing columns: "
        f"{sorted(required_mismatch_columns - set(mismatches.columns))}"
    )

if "Release_Title" not in v12.columns:
    v12["Release_Title"] = ""

v12["_Date"] = pd.to_datetime(
    v12["Effective_Date"],
    errors="coerce"
).dt.normalize()

v12["_Role"] = (
    v12["Role"].astype(str).str.upper().str.strip()
)

v12["_Symbol_Key"] = (
    v12[symbol_col].map(norm_symbol)
)

v12["_Source_File"] = (
    v12["Source_URL"].map(source_filename)
)

mismatches["_Mismatch_Date"] = pd.to_datetime(
    mismatches["Effective_Date"],
    errors="coerce"
).dt.normalize()

mismatches["_Symbol_Key"] = (
    mismatches["Symbol"].map(norm_symbol)
)

if v12["_Date"].isna().any():
    raise ValueError("V12 contains invalid effective dates.")

if mismatches["_Mismatch_Date"].isna().any():
    raise ValueError("Mismatch report contains invalid effective dates.")

if mismatches["_Symbol_Key"].eq("").any():
    raise ValueError("Mismatch report contains blank symbols.")


# ------------------------------------------------
# 4. BUILD DISTINCT MISMATCH ITEMS
# ------------------------------------------------

mismatch_items = (
    mismatches[
        [
            "Effective_Date",
            "Symbol",
            "Role",
            "Issue",
            "_Mismatch_Date",
            "_Symbol_Key",
        ]
    ]
    .drop_duplicates(
        subset=["_Mismatch_Date", "_Symbol_Key", "Issue"]
    )
    .copy()
    .reset_index(drop=True)
)

mismatch_symbols = set(
    mismatch_items["_Symbol_Key"]
)

print("=" * 90)
print("V12 MISSING-LATER-ACTION INVESTIGATION")
print("=" * 90)
print(f"V12 ledger rows:                       {len(v12):,}")
print(f"Mismatch records in report:            {len(mismatches):,}")
print(f"Distinct mismatch items:                {len(mismatch_items):,}")
print(f"Distinct mismatched symbols:            {len(mismatch_symbols):,}")


# ------------------------------------------------
# 5. BUILD V12 EVENT SIGNATURES
# ------------------------------------------------

# An action is considered present if its date, role, symbol and source
# PDF filename match a row already in v12.
v12_signatures = set(
    zip(
        v12["_Date"].dt.strftime("%Y-%m-%d"),
        v12["_Role"],
        v12["_Symbol_Key"],
        v12["_Source_File"],
    )
)


# ------------------------------------------------
# 6. FIND EARLIER EVENT-CANDIDATE LEDGERS
# ------------------------------------------------

version_pattern = re.compile(
    r"^nifty500_official_event_candidates_v(\d+)\.csv$",
    flags=re.IGNORECASE
)

prior_files = []

for path in BASE_DIR.glob(
    "nifty500_official_event_candidates_v*.csv"
):
    match = version_pattern.match(path.name)

    if not match:
        continue

    version = int(match.group(1))

    if 3 <= version < 12:
        prior_files.append((version, path))

prior_files.sort(key=lambda item: item[0])

if not prior_files:
    raise FileNotFoundError(
        "No earlier event-candidate versions v3-v11 were found."
    )

print("\nEarlier versions found:")
print(", ".join(f"v{version}" for version, _ in prior_files))


# ------------------------------------------------
# 7. SEARCH OLDER VERSIONS FOR LATER ACTIONS
# ------------------------------------------------

mismatch_lookup = {}

for _, row in mismatch_items.iterrows():
    symbol_key = row["_Symbol_Key"]

    mismatch_lookup.setdefault(
        symbol_key, []
    ).append({
        "Mismatch_Date": row["_Mismatch_Date"],
        "Mismatch_Symbol": row["Symbol"],
        "Mismatch_Issue": row["Issue"],
    })


candidate_records = []

for version, path in prior_files:

    print(f"Scanning {path.name} ...")

    try:
        old = pd.read_csv(
            path,
            dtype=str,
            keep_default_na=False
        )
    except Exception as exc:
        print(f"WARNING: Could not read {path.name}: {exc}")
        continue

    old_symbol_col = (
        "Canonical_Symbol"
        if "Canonical_Symbol" in old.columns
        else "Symbol"
        if "Symbol" in old.columns
        else None
    )

    required_old_columns = {
        "Effective_Date", "Role", "Source_URL"
    }

    if (
        old_symbol_col is None
        or not required_old_columns.issubset(old.columns)
    ):
        print(
            f"WARNING: Skipping {path.name}; "
            "required columns are missing."
        )
        continue

    if "Release_Title" not in old.columns:
        old["Release_Title"] = ""

    old["_Date"] = pd.to_datetime(
        old["Effective_Date"],
        errors="coerce"
    ).dt.normalize()

    old["_Role"] = (
        old["Role"].astype(str).str.upper().str.strip()
    )

    old["_Symbol_Raw"] = (
        old[old_symbol_col].astype(str).str.strip()
    )

    old["_Symbol_Key"] = (
        old["_Symbol_Raw"].map(norm_symbol)
    )

    old["_Source_File"] = (
        old["Source_URL"].map(source_filename)
    )

    # Only investigate symbols already flagged by the v12 walk.
    old = old.loc[
        old["_Symbol_Key"].isin(mismatch_symbols)
        & old["_Role"].isin(["ENTRY", "EXIT"])
        & old["_Date"].notna()
    ].copy()

    for _, candidate in old.iterrows():

        sym_key = candidate["_Symbol_Key"]
        candidate_date = candidate["_Date"]

        for mismatch_item in mismatch_lookup.get(sym_key, []):

            mismatch_date = mismatch_item["Mismatch_Date"]

            # Only search for events later than the flagged action.
            if candidate_date <= mismatch_date:
                continue

            signature = (
                candidate_date.strftime("%Y-%m-%d"),
                candidate["_Role"],
                sym_key,
                candidate["_Source_File"],
            )

            # Do not report actions already present in v12.
            if signature in v12_signatures:
                continue

            candidate_records.append({
                "Mismatch_Date": mismatch_date.strftime("%Y-%m-%d"),
                "Mismatch_Symbol": mismatch_item["Mismatch_Symbol"],
                "Mismatch_Issue": mismatch_item["Mismatch_Issue"],
                "Candidate_Date": candidate_date.strftime("%Y-%m-%d"),
                "Candidate_Role": candidate["_Role"],
                "Candidate_Symbol": candidate["_Symbol_Raw"],
                "Candidate_Source_File": candidate["_Source_File"],
                "Candidate_Release_Title": candidate["Release_Title"],
                "Candidate_Source_URL": candidate["Source_URL"],
                "Found_In_Version": f"v{version}",
                "Candidate_File": path.name,
            })


# ------------------------------------------------
# 8. COMBINE REPEATED CANDIDATES — EMPTY-SAFE
# ------------------------------------------------

candidate_columns = [
    "Mismatch_Date",
    "Mismatch_Symbol",
    "Mismatch_Issue",
    "Candidate_Date",
    "Candidate_Role",
    "Candidate_Symbol",
    "Candidate_Source_File",
    "Candidate_Release_Title",
    "Candidate_Source_URL",
    "Found_In_Version",
    "Candidate_File",
]

candidate_df = pd.DataFrame(
    candidate_records,
    columns=candidate_columns
)

candidate_key_columns = [
    "Mismatch_Date",
    "Mismatch_Symbol",
    "Mismatch_Issue",
    "Candidate_Date",
    "Candidate_Role",
    "Candidate_Symbol",
    "Candidate_Source_File",
    "Candidate_Release_Title",
    "Candidate_Source_URL",
]

if candidate_df.empty:

    # Preserve the columns required by the summary below.
    candidates = pd.DataFrame(
        columns=[
            *candidate_key_columns,
            "Versions_Found",
            "Version_Count",
        ]
    )

else:

    def sort_versions(values):
        unique_values = set(values)
        return ", ".join(
            sorted(
                unique_values,
                key=lambda value: int(value.lstrip("v"))
            )
        )

    candidates = (
        candidate_df.groupby(
            candidate_key_columns,
            dropna=False,
            as_index=False,
        )
        .agg(
            Versions_Found=(
                "Found_In_Version",
                sort_versions
            ),
            Version_Count=(
                "Found_In_Version",
                "nunique"
            ),
        )
        .sort_values(
            [
                "Mismatch_Date",
                "Mismatch_Symbol",
                "Candidate_Date",
                "Candidate_Role",
            ]
        )
        .reset_index(drop=True)
    )


# ------------------------------------------------
# 9. BUILD SUMMARY — EMPTY-SAFE
# ------------------------------------------------

summary_rows = []

for _, item in mismatch_items.iterrows():

    mismatch_date = item["_Mismatch_Date"]
    mismatch_date_str = mismatch_date.strftime("%Y-%m-%d")
    sym_key = item["_Symbol_Key"]

    # Later events already present in v12.
    current_symbol_events = v12.loc[
        v12["_Symbol_Key"].eq(sym_key)
        & v12["_Date"].gt(mismatch_date)
    ].copy()

    current_later_exits = current_symbol_events.loc[
        current_symbol_events["_Role"].eq("EXIT")
    ]

    # Candidate events recovered from older versions.
    if candidates.empty:
        matching_candidates = candidates.copy()
    else:
        matching_candidates = candidates.loc[
            candidates["Mismatch_Date"].eq(mismatch_date_str)
            & candidates["Mismatch_Symbol"].map(norm_symbol).eq(sym_key)
        ].copy()

    candidate_exits = matching_candidates.loc[
        matching_candidates["Candidate_Role"].eq("EXIT")
    ].copy()

    current_exit_dates = (
        current_later_exits["_Date"]
        .dt.strftime("%Y-%m-%d")
        .tolist()
    )

    candidate_exit_dates = (
        candidate_exits["Candidate_Date"]
        .astype(str)
        .tolist()
    )

    summary_rows.append({
        "Mismatch_Date": mismatch_date_str,
        "Mismatch_Symbol": item["Symbol"],
        "Mismatch_Issue": item["Issue"],
        "V12_Later_Event_Count": len(current_symbol_events),
        "V12_Later_Exit_Count": len(current_later_exits),
        "V12_Later_Exit_Dates": ", ".join(current_exit_dates),
        "Earlier_Version_Later_Action_Count": len(matching_candidates),
        "Earlier_Version_Later_Exit_Count": len(candidate_exits),
        "Earlier_Version_Exit_Dates": ", ".join(candidate_exit_dates),
    })

summary = pd.DataFrame(summary_rows)

if not summary.empty:
    summary = summary.sort_values(
        ["Mismatch_Date", "Mismatch_Symbol"]
    ).reset_index(drop=True)


# ------------------------------------------------
# 10. SAVE NEW DIAGNOSTIC REPORTS
# ------------------------------------------------

candidates.to_csv(CANDIDATES_OUT, index=False)
summary.to_csv(SUMMARY_OUT, index=False)


# ------------------------------------------------
# 11. DISPLAY RESULTS
# ------------------------------------------------

print("\n" + "=" * 90)
print("SECTION A — MISMATCHES AND LATER-ACTION SEARCH RESULTS")
print("=" * 90)

with pd.option_context(
    "display.max_rows", 100,
    "display.max_columns", 20,
    "display.max_colwidth", 180,
    "display.width", 260,
):
    display(summary)


print("\n" + "=" * 90)
print("SECTION B — EARLIER-VERSION ACTION CANDIDATES NOT IN V12")
print("=" * 90)

if candidates.empty:

    print(
        "No later actions absent from v12 were found in the earlier "
        "candidate files searched."
    )
    print(
        "This does not prove that no official exit exists; "
        "the action might be missing from every extracted version."
    )

else:

    with pd.option_context(
        "display.max_rows", 200,
        "display.max_columns", 20,
        "display.max_colwidth", 180,
        "display.width", 260,
    ):
        display(candidates)


print("\n" + "=" * 90)
print("OUTPUT FILES")
print("=" * 90)
print(f"Summary:          {SUMMARY_OUT}")
print(f"Candidate events: {CANDIDATES_OUT}")

print("\nInvestigation complete. Existing input files remain unchanged.")
print(
    "Do not automatically restore candidate events: an earlier version "
    "may contain events that were void, superseded, or assigned to "
    "another index."
)

FileNotFoundError: Required input file not found: /content/nifty500_pit_data/nifty500_official_event_candidates_v12.csv

In [3]:
# ================================================================
# RECOVER NIFTY 500 EVENT LEDGER V12
# Safe recovery if v12 is missing from /content.
# ================================================================

from pathlib import Path
import pandas as pd
import re

BASE_DIR = Path("/content/nifty500_pit_data")
DRIVE_DIR = Path("/content/drive/MyDrive/nifty500_pit_data")

V11_NAME = "nifty500_official_event_candidates_v11.csv"
V12_NAME = "nifty500_official_event_candidates_v12.csv"

OUTPUT_FILE = BASE_DIR / V12_NAME
CHANGELOG_FILE = BASE_DIR / "nifty500_event_ledger_change_log_v7.csv"
QA_FILE = BASE_DIR / "nifty500_2015_sep28_scope_QA_v2.csv"

# ------------------------------------------------
# 1. CHECK WHETHER V12 ALREADY EXISTS
# ------------------------------------------------

v12_candidates = [
    BASE_DIR / V12_NAME,
    DRIVE_DIR / V12_NAME,
]

existing_v12 = next(
    (p for p in v12_candidates if p.exists()),
    None
)

if existing_v12 is not None:
    BASE_DIR.mkdir(parents=True, exist_ok=True)

    if existing_v12.resolve() != OUTPUT_FILE.resolve():
        import shutil
        shutil.copy2(existing_v12, OUTPUT_FILE)

    print(f"Existing v12 found: {existing_v12}")
    print(f"Available at:       {OUTPUT_FILE}")

else:
    # ------------------------------------------------------------
    # 2. FIND V11 TO REBUILD V12
    # ------------------------------------------------------------

    v11_candidates = [
        BASE_DIR / V11_NAME,
        DRIVE_DIR / V11_NAME,
    ]

    v11_file = next(
        (p for p in v11_candidates if p.exists()),
        None
    )

    if v11_file is None:
        print("Could not find v11 in either expected location.")
        print("\nFiles currently in the project directory:")

        if BASE_DIR.exists():
            for p in sorted(BASE_DIR.iterdir()):
                print(" ", p.name)
        else:
            print(f"  Directory does not exist: {BASE_DIR}")

        print("\nFiles currently in the Drive project directory:")

        if DRIVE_DIR.exists():
            for p in sorted(DRIVE_DIR.iterdir()):
                print(" ", p.name)
        else:
            print(f"  Directory does not exist: {DRIVE_DIR}")

        raise FileNotFoundError(
            "Neither v12 nor v11 was found. Restore the v11 CSV "
            "from your backup, or rerun the earlier ledger-building "
            "steps before continuing."
        )

    print(f"Rebuilding v12 from: {v11_file}")

    v11 = pd.read_csv(
        v11_file,
        dtype=str,
        keep_default_na=False
    )

    symbol_col = (
        "Canonical_Symbol"
        if "Canonical_Symbol" in v11.columns
        else "Symbol"
        if "Symbol" in v11.columns
        else None
    )

    if symbol_col is None:
        raise ValueError("V11 has no recognized symbol column.")

    def normalize_symbol(value):
        return re.sub(
            r"[^A-Z0-9]",
            "",
            str(value).upper().strip()
        )

    dates = pd.to_datetime(
        v11["Effective_Date"],
        errors="coerce"
    ).dt.strftime("%Y-%m-%d")

    roles = v11["Role"].str.upper().str.strip()
    symbols = v11[symbol_col].map(normalize_symbol)
    urls = v11["Source_URL"].str.lower()

    # The single verified false-positive event:
    # HAVELLS, ENTRY, effective 2015-09-28, from the Aug 24 release.
    remove_mask = (
        urls.str.contains(
            "ind_prs24082015.pdf",
            regex=False,
            na=False
        )
        & dates.eq("2015-09-28")
        & roles.eq("ENTRY")
        & symbols.eq("HAVELLS")
    )

    matched = v11.loc[remove_mask].copy()

    if len(matched) != 1:
        print(f"Expected one HAVELLS row; found {len(matched)}.")
        display(matched)
        raise ValueError(
            "Cannot safely rebuild v12; no output was written."
        )

    removed = matched.iloc[0].to_dict()
    removed["Change_Action"] = "REMOVE"
    removed["Correction_Reason"] = (
        "Wrong index scope: HAVELLS appears in the LIX15 Midcap "
        "section, not the CNX 500 section, of the 2015-08-24 release."
    )

    v12 = v11.loc[~remove_mask].copy().reset_index(drop=True)

    if len(v12) != len(v11) - 1:
        raise AssertionError("Unexpected v12 row count.")

    # ------------------------------------------------------------
    # 3. QA AGAINST OFFICIAL LISTS
    # ------------------------------------------------------------

    official_lists = [
        {
            "pdf": "ind_prs12082015.pdf",
            "role": "ENTRY",
            "expected": {
                "ADANIPOWER", "BALKRISIND", "CAMLINFINE", "FLFL",
                "ILFSENGG", "INOXWIND", "KRBL", "METALFORGE",
                "ORIENTCEM", "RICOAUTO", "SITICABLE", "SRIPIPES",
                "SYMPHONY", "VRLLOG",
            },
        },
        {
            "pdf": "ind_prs12082015.pdf",
            "role": "EXIT",
            "expected": {
                "ANSALAPI", "BGRENERGY", "GAMMONIND", "GUJNRECOKE",
                "HINDOILEXP", "HOTELEELA", "IFBIND", "KESORAMIND",
                "MANGCHEFER", "MONNETISPA", "ORCHIDCHEM", "RAMCOIND",
                "RASOYPR", "RTNPOWER",
            },
        },
        {
            "pdf": "ind_prs24082015.pdf",
            "role": "ENTRY",
            "expected": {"ADLABS", "EVEREADY"},
        },
        {
            "pdf": "ind_prs24082015.pdf",
            "role": "EXIT",
            "expected": {"CROMPGREAV", "MAX"},
        },
    ]

    v12_dates = pd.to_datetime(
        v12["Effective_Date"],
        errors="coerce"
    ).dt.strftime("%Y-%m-%d")

    v12_roles = v12["Role"].str.upper().str.strip()
    v12_symbols = v12[symbol_col].map(normalize_symbol)
    v12_urls = v12["Source_URL"].str.lower()

    qa_rows = []

    for item in official_lists:
        scope = (
            v12_urls.str.contains(
                item["pdf"],
                regex=False,
                na=False
            )
            & v12_dates.eq("2015-09-28")
            & v12_roles.eq(item["role"])
        )

        actual = set(v12_symbols.loc[scope])
        expected = item["expected"]

        qa_rows.append({
            "Source_PDF": item["pdf"],
            "Effective_Date": "2015-09-28",
            "Role": item["role"],
            "Expected_Count": len(expected),
            "Actual_Count": len(actual),
            "Missing_Symbols": ", ".join(sorted(expected - actual)),
            "Unexpected_Symbols": ", ".join(sorted(actual - expected)),
            "Pass": actual == expected,
        })

    qa = pd.DataFrame(qa_rows)

    print("\nOfficial source-specific QA:")
    display(qa)

    if not qa["Pass"].all():
        raise AssertionError(
            "Official-list QA failed. v12 was not saved."
        )

    # ------------------------------------------------------------
    # 4. SAVE ONLY AFTER QA PASSES
    # ------------------------------------------------------------

    BASE_DIR.mkdir(parents=True, exist_ok=True)

    v12.to_csv(OUTPUT_FILE, index=False)
    pd.DataFrame([removed]).to_csv(
        CHANGELOG_FILE,
        index=False
    )
    qa.to_csv(QA_FILE, index=False)

    print("\nV12 RECOVERED")
    print(f"V11 rows:       {len(v11):,}")
    print("Removed rows:   1")
    print(f"V12 rows:       {len(v12):,}")
    print(f"Ledger:         {OUTPUT_FILE}")
    print(f"Change log:     {CHANGELOG_FILE}")
    print(f"QA report:      {QA_FILE}")
    print("V11 remains unchanged.")

# ------------------------------------------------
# 5. CHECK THE MISMATCH REPORT NEEDED NEXT
# ------------------------------------------------

mismatch_file = (
    BASE_DIR / "membership_backward_walk_state_mismatches_v3.csv"
)

print("\nNEXT-STEP INPUT CHECK")
print(f"V12 exists:             {OUTPUT_FILE.exists()}")
print(f"V12 mismatch report exists: {mismatch_file.exists()}")

if not mismatch_file.exists():
    print(
        "\nThe v12 ledger is available, but the fresh mismatch report "
        "is missing. Rerun the V12 backward membership reconstruction "
        "cell before rerunning the missing-later-action investigation."
    )


Could not find v11 in either expected location.

Files currently in the project directory:
  Directory does not exist: /content/nifty500_pit_data

Files currently in the Drive project directory:
  Directory does not exist: /content/drive/MyDrive/nifty500_pit_data


FileNotFoundError: Neither v12 nor v11 was found. Restore the v11 CSV from your backup, or rerun the earlier ledger-building steps before continuing.